# F1 Pantheon — Validation & Development Archive

This notebook preserves diagnostics, sensitivity studies, forensic audits and recovery utilities that were intentionally removed from the clean production path.

It is **not** the normal execution path for a new LIVE release. Many cells expect objects created by the production notebook and are retained here primarily for provenance and targeted re-use.


## Frozen v2.1 production and scoring provenance


In [ ]:
# ==========================================================
# STAGE 13 — 1982–2025 NUTPIE/JAX SCREEN
# ==========================================================
#
# Full data:
#   2 chains
#   200 tune
#   200 posterior draws
#
# Checkpoint-aware screen fit.
#
# Uses h5netcdf because the PyMC result is an xarray DataTree
# with NetCDF groups. An existing in-memory trace or valid
# checkpoint is reused rather than resampled.
# ==========================================================

import time
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd
import xarray as xr

# ==========================================================
# CHECKPOINT BACKEND
# ==========================================================

CHECKPOINT_ENGINE = "h5netcdf"


def ensure_checkpoint_backend():

    try:

        import h5netcdf  # noqa: F401
        import h5py  # noqa: F401

    except ImportError as exc:

        raise RuntimeError(
            "The grouped xarray DataTree checkpoint requires "
            "h5netcdf + h5py.\n\n"
            "Install them with:\n\n"
            "    %pip install h5netcdf h5py\n\n"
            "Then rerun this cell. The completed posterior "
            "will remain in memory as long as the kernel "
            "has not been restarted."
        ) from exc


ensure_checkpoint_backend()


# ==========================================================
# SCREEN CONFIGURATION
# ==========================================================

SCREEN_DRAWS = 200
SCREEN_TUNE = 200
SCREEN_CHAINS = 2
SCREEN_SEED = 42
SCREEN_TARGET_ACCEPT = 0.92


SCREEN_TRACE_PATH = Path(CFG.output_dir) / (
    "joint_driver_car_v2_1_" "1982_2025_" "nutpie_jax_screen_" "200x200_2chain.nc"
)


# ==========================================================
# DETERMINE WHETHER WE NEED TO SAMPLE
# ==========================================================

need_to_sample = False


# ----------------------------------------------------------
# CASE 1: posterior already exists in memory.
# ----------------------------------------------------------

if "screen_trace_2025" in globals():

    print("Using existing in-memory " "1982–2025 screen posterior.")

    print("NO RESAMPLING will be performed.")

    # Rewrite the checkpoint from the in-memory posterior so
    # the on-disk file is guaranteed to use the grouped
    # h5netcdf backend.
    if SCREEN_TRACE_PATH.exists():

        print("Removing partial/old checkpoint ->")

        print(SCREEN_TRACE_PATH)

        SCREEN_TRACE_PATH.unlink()

    screen_trace_2025.to_netcdf(
        SCREEN_TRACE_PATH,
        engine=CHECKPOINT_ENGINE,
        mode="w",
    )

    print("\nSaved existing posterior ->")

    print(SCREEN_TRACE_PATH)


# ----------------------------------------------------------
# CASE 2:
# No posterior in memory, but a checkpoint exists.
# Try loading it.
# ----------------------------------------------------------

elif SCREEN_TRACE_PATH.exists():

    print("Found existing screen checkpoint ->")

    print(SCREEN_TRACE_PATH)

    try:

        screen_trace_2025 = xr.open_datatree(
            SCREEN_TRACE_PATH,
            engine=CHECKPOINT_ENGINE,
        )

        print("Checkpoint loaded successfully.")

    except Exception as exc:

        print("\nExisting checkpoint is not readable:")

        print(repr(exc))

        print("\nRemoving invalid checkpoint.")

        SCREEN_TRACE_PATH.unlink()

        need_to_sample = True


# ----------------------------------------------------------
# CASE 3:
# Fresh run.
# ----------------------------------------------------------

else:

    need_to_sample = True


# ==========================================================
# SAMPLE ONLY IF NECESSARY
# ==========================================================

if need_to_sample:

    print("\nRunning 1982–2025 " "nutpie/JAX screen...")

    print(f"{SCREEN_CHAINS} chains × " f"({SCREEN_TUNE} tune + " f"{SCREEN_DRAWS} draws)")

    t0 = time.perf_counter()

    with model_1982_2025:

        screen_trace_2025 = pm.sample(
            draws=SCREEN_DRAWS,
            tune=SCREEN_TUNE,
            chains=SCREEN_CHAINS,
            random_seed=SCREEN_SEED,
            target_accept=SCREEN_TARGET_ACCEPT,
            nuts_sampler="nutpie",
            nuts_sampler_kwargs={
                "backend": "jax",
            },
            return_inferencedata=True,
            progressbar=True,
        )

    elapsed = time.perf_counter() - t0

    print(
        "\nWall time:",
        f"{elapsed / 60.0:.2f} min",
    )

    screen_trace_2025.to_netcdf(
        SCREEN_TRACE_PATH,
        engine=CHECKPOINT_ENGINE,
        mode="w",
    )

    print("\nSaved screen trace ->")

    print(SCREEN_TRACE_PATH)


# ==========================================================
# EXTRACT ORDINARY XARRAY DATASETS
#
# Current PyMC/xarray returns a DataTree.
# Working explicitly with the posterior Dataset avoids the
# DataTree issues we hit previously in ArviZ diagnostics.
# ==========================================================

posterior_screen = screen_trace_2025["posterior"].to_dataset()


sample_stats_screen = screen_trace_2025["sample_stats"].to_dataset()


print("\nPosterior dimensions:")

print(dict(posterior_screen.sizes))


# ==========================================================
# GLOBAL PARAMETERS
# ==========================================================

GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]


print("\n========================================")

print("GLOBAL POSTERIOR — 1982–2025 SCREEN")

print("========================================")


global_summary_2025 = az.summary(
    posterior_screen,
    var_names=GLOBAL_VARS,
    kind="all",
)


display(global_summary_2025)

# ==========================================================
# HAND OFF TO STAGE 13A
# ==========================================================

print("\nStage 13 screen fit/checkpoint complete.")

print("Run Stage 13A for the safe variable-by-variable " "screen diagnostics.")

### Stage 13A — Screening diagnostics

Computes convergence diagnostics variable-by-variable to avoid xarray broadcasting
variables with unrelated dimensions into an enormous dense array.

For a 2×200 screen these diagnostics are interpreted as a sanity check rather than a
production convergence gate.


In [ ]:
# ==========================================================
# STAGE 13A — SCREEN CONVERGENCE / ENDPOINT DIAGNOSTICS
# ==========================================================
#
# Compute diagnostics one variable at a time.
#
# xarray datasets contain parameters with unrelated dimensions;
# flattening each variable independently avoids accidental
# cross-variable broadcasting.
# ==========================================================

import numpy as np
import pandas as pd
import arviz as az

# ==========================================================
# 1. GET ORDINARY DATASETS
# ==========================================================

posterior_screen = screen_trace_2025["posterior"].to_dataset()


sample_stats_screen = screen_trace_2025["sample_stats"].to_dataset()


print(
    "Posterior dimensions:",
    dict(posterior_screen.sizes),
)


# ==========================================================
# 2. GLOBAL PARAMETER SUMMARY
# ==========================================================

GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]


print("\n========================================")

print("GLOBAL POSTERIOR — 1982–2025 SCREEN")

print("========================================")


global_summary_2025 = az.summary(
    posterior_screen,
    var_names=GLOBAL_VARS,
    kind="all",
)


display(global_summary_2025)


# ==========================================================
# 3. RHAT / ESS
#
# IMPORTANT:
# NEVER call Dataset.to_array() here.
#
# Different model variables have different dimensions.
# xarray would broadcast all of them together before
# stacking, which is what caused the attempted PiB-sized
# allocation.
# ==========================================================

rhat_ds = az.rhat(posterior_screen)


ess_ds = az.ess(
    posterior_screen,
    method="bulk",
)


def flatten_dataset_variables(
    dataset,
):
    """
    Flatten each variable independently.

    This avoids broadcasting variables with unrelated
    parameter dimensions.
    """

    arrays = []

    for variable_name in dataset.data_vars:

        values = np.asarray(
            dataset[variable_name].values,
            dtype=float,
        ).ravel()

        values = values[np.isfinite(values)]

        if values.size:

            arrays.append(values)

    if not arrays:

        return np.array(
            [],
            dtype=float,
        )

    return np.concatenate(arrays)


rhat_values = flatten_dataset_variables(rhat_ds)


ess_values = flatten_dataset_variables(ess_ds)


if not rhat_values.size:

    raise RuntimeError("No finite Rhat values found.")


if not ess_values.size:

    raise RuntimeError("No finite ESS values found.")


max_rhat = float(np.max(rhat_values))


n_rhat_101 = int(np.sum(rhat_values > 1.01))


n_rhat_105 = int(np.sum(rhat_values > 1.05))


min_ess = float(np.min(ess_values))


median_ess = float(np.median(ess_values))


# ==========================================================
# 4. DIVERGENCES
# ==========================================================

if "diverging" in sample_stats_screen:

    divergences = int(np.asarray(sample_stats_screen["diverging"].values).sum())

else:

    divergences = 0


print("\n========================================")

print("SCREEN HEALTH")

print("========================================")


print(
    "max Rhat:",
    max_rhat,
)

print(
    "n Rhat > 1.01:",
    n_rhat_101,
)

print(
    "n Rhat > 1.05:",
    n_rhat_105,
)

print(
    "min ESS bulk:",
    min_ess,
)

print(
    "median ESS bulk:",
    median_ess,
)

print(
    "divergences:",
    divergences,
)


# ==========================================================
# 5. PER-VARIABLE DIAGNOSTIC SUMMARY
#
# Useful for identifying what causes the worst Rhat/ESS
# rather than only seeing the global extrema.
# ==========================================================

diagnostic_rows = []


for variable_name in rhat_ds.data_vars:

    rhat = np.asarray(
        rhat_ds[variable_name].values,
        dtype=float,
    ).ravel()

    ess = np.asarray(
        ess_ds[variable_name].values,
        dtype=float,
    ).ravel()

    rhat = rhat[np.isfinite(rhat)]

    ess = ess[np.isfinite(ess)]

    diagnostic_rows.append(
        {
            "variable": variable_name,
            "n_parameters": int(len(rhat)),
            "max_rhat": (float(rhat.max()) if len(rhat) else np.nan),
            "n_rhat_gt_1.01": (int(np.sum(rhat > 1.01)) if len(rhat) else 0),
            "n_rhat_gt_1.05": (int(np.sum(rhat > 1.05)) if len(rhat) else 0),
            "min_ess_bulk": (float(ess.min()) if len(ess) else np.nan),
            "median_ess_bulk": (float(np.median(ess)) if len(ess) else np.nan),
        }
    )


diagnostic_table = (
    pd.DataFrame(diagnostic_rows)
    .sort_values(
        [
            "max_rhat",
            "min_ess_bulk",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print("\nPer-variable diagnostics:")


display(diagnostic_table)


# ==========================================================
# 6. HISTORICAL TEMPERATURES
# ==========================================================

tau_q = posterior_screen["tau_hist_quali"].values.reshape(-1)


tau_r = posterior_screen["tau_hist_race"].values.reshape(-1)


print("\nHistorical temperatures:")


print(
    "median tau qualifying:",
    float(np.median(tau_q)),
)


print(
    "median tau race:",
    float(np.median(tau_r)),
)


print(
    "P(tau_race > tau_quali):",
    float(np.mean(tau_r > tau_q)),
)


# ==========================================================
# 7. ALPHA DRAWS
# ==========================================================

alpha_da = posterior_screen["alpha_dy"].stack(
    sample=(
        "chain",
        "draw",
    )
)


alpha_other_dims = [dim for dim in alpha_da.dims if dim != "sample"]


alpha_screen = alpha_da.transpose(
    "sample",
    *alpha_other_dims,
).values


alpha_screen = alpha_screen.reshape(
    alpha_screen.shape[0],
    -1,
)


print(
    "\nAlpha posterior array:",
    alpha_screen.shape,
)


# ==========================================================
# 8. 2025 DRIVER POSTERIOR
# ==========================================================

driver_2025_rows = []


for driver in driver_states_2025:

    state_id = idx_all.dy_map[
        (
            driver,
            2025,
        )
    ]

    draws = alpha_screen[
        :,
        state_id,
    ]

    driver_2025_rows.append(
        {
            "driver": driver,
            "mean": float(np.mean(draws)),
            "sd": float(
                np.std(
                    draws,
                    ddof=1,
                )
            ),
            "q055": float(
                np.quantile(
                    draws,
                    0.055,
                )
            ),
            "median": float(np.median(draws)),
            "q945": float(
                np.quantile(
                    draws,
                    0.945,
                )
            ),
        }
    )


driver_2025_summary = (
    pd.DataFrame(driver_2025_rows)
    .sort_values(
        "mean",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n========================================")

print("2025 DRIVER-SEASON ALPHA")

print("========================================")


display(driver_2025_summary)


# ==========================================================
# 9. CAR DRAWS
# ==========================================================

car_da = posterior_screen["car_csy"].stack(
    sample=(
        "chain",
        "draw",
    )
)


car_other_dims = [dim for dim in car_da.dims if dim != "sample"]


car_screen = car_da.transpose(
    "sample",
    *car_other_dims,
).values


car_screen = car_screen.reshape(
    car_screen.shape[0],
    -1,
)


print(
    "\nCar posterior array:",
    car_screen.shape,
)


# ==========================================================
# 10. 2025 CONSTRUCTOR POSTERIOR
# ==========================================================

car_2025_rows = []


for constructor_id in constructor_states_2025:

    state_id = idx_all.csy_map[
        (
            constructor_id,
            2025,
        )
    ]

    draws = car_screen[
        :,
        state_id,
    ]

    car_2025_rows.append(
        {
            "constructorId": constructor_id,
            "constructor": constructor_names.get(
                constructor_id,
                str(constructor_id),
            ),
            "mean": float(np.mean(draws)),
            "sd": float(
                np.std(
                    draws,
                    ddof=1,
                )
            ),
            "q055": float(
                np.quantile(
                    draws,
                    0.055,
                )
            ),
            "median": float(np.median(draws)),
            "q945": float(
                np.quantile(
                    draws,
                    0.945,
                )
            ),
        }
    )


car_2025_summary = (
    pd.DataFrame(car_2025_rows)
    .sort_values(
        "mean",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n========================================")

print("2025 CONSTRUCTOR-SEASON EFFECT")

print("========================================")


display(car_2025_summary)


# ==========================================================
# 11. 2024 -> 2025 DRIVER CONTINUITY
# ==========================================================

continuity_drivers = [
    "max_verstappen",
    "norris",
    "piastri",
    "leclerc",
    "russell",
    "hamilton",
    "alonso",
    "sainz",
]


continuity_rows = []


for driver in continuity_drivers:

    if (
        driver,
        2024,
    ) not in idx_all.dy_map or (
        driver,
        2025,
    ) not in idx_all.dy_map:

        continue

    i24 = idx_all.dy_map[
        (
            driver,
            2024,
        )
    ]

    i25 = idx_all.dy_map[
        (
            driver,
            2025,
        )
    ]

    a24 = alpha_screen[
        :,
        i24,
    ]

    a25 = alpha_screen[
        :,
        i25,
    ]

    delta = a25 - a24

    continuity_rows.append(
        {
            "driver": driver,
            "alpha_2024": float(np.mean(a24)),
            "alpha_2025": float(np.mean(a25)),
            "delta_mean": float(np.mean(delta)),
            "delta_sd": float(
                np.std(
                    delta,
                    ddof=1,
                )
            ),
        }
    )


continuity_2024_2025 = (
    pd.DataFrame(continuity_rows)
    .sort_values(
        "alpha_2025",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n========================================")

print("2024 -> 2025 DRIVER CONTINUITY")

print("========================================")


display(continuity_2024_2025)


print("\nStage 13A recovery diagnostics complete.")

## Stage 14 — Final 1982–2025 production fit

Runs or loads the official frozen production posterior:

- 4 chains;
- 1000 tuning iterations per chain;
- 1000 posterior draws per chain;
- `target_accept = 0.92`;
- nutpie + JAX.

The checkpoint is stored separately from the earlier 1982–2024 final fit.


In [ ]:
# ==========================================================
# STAGE 14 — FINAL 1982–2025 PRODUCTION FIT
# ==========================================================
#
# FROZEN MODEL SPECIFICATION
#
#   4 chains
#   1000 tune
#   1000 posterior draws
#   target_accept = 0.92
#   seed = 42
#   nutpie + JAX
#
# This is the official 1982–2025 production posterior.
# Re-running this cell reuses a valid checkpoint when available.
#
# The previous 1982–2024 final trace is NOT touched.
# ==========================================================

from pathlib import Path

import time
import xarray as xr

# ==========================================================
# PRODUCTION CONFIG
# ==========================================================

FINAL_DRAWS = 1000
FINAL_TUNE = 1000
FINAL_CHAINS = 4
FINAL_TARGET_ACCEPT = 0.92
FINAL_SEED = 42


FINAL_TRACE_PATH = Path(CFG.output_dir) / (
    "joint_driver_car_v2_1_" "1982_2025_final_" "nutpie_jax_" "1000x1000_4chain.nc"
)


FINAL_TRACE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


CHECKPOINT_ENGINE = "h5netcdf"


def ensure_checkpoint_backend():

    try:

        import h5netcdf  # noqa: F401
        import h5py  # noqa: F401

    except ImportError as exc:

        raise RuntimeError(
            "Final checkpoint requires h5netcdf + h5py.\n\n"
            "Install with:\n"
            "    %pip install h5netcdf h5py"
        ) from exc


ensure_checkpoint_backend()


# ==========================================================
# LOAD EXISTING VALID FINAL TRACE OR RUN
# ==========================================================

if FINAL_TRACE_PATH.exists():

    print("Existing 1982–2025 final trace found:")

    print(FINAL_TRACE_PATH)

    try:

        final_trace_2025 = xr.open_datatree(
            FINAL_TRACE_PATH,
            engine=CHECKPOINT_ENGINE,
        )

        print("\nLoaded existing final trace.")

        print("NO RESAMPLING performed.")

    except Exception as exc:

        raise RuntimeError(
            "A file exists at the final production path "
            "but cannot be read as a valid checkpoint.\n\n"
            f"{FINAL_TRACE_PATH}\n\n"
            "Do not overwrite it automatically. Inspect or "
            "rename/delete the invalid file deliberately."
        ) from exc


else:

    print("========================================")

    print("FINAL 1982–2025 PRODUCTION FIT")

    print("========================================")

    print(f"{FINAL_CHAINS} chains × " f"({FINAL_TUNE} tune + " f"{FINAL_DRAWS} posterior)")

    print("Sampler: nutpie / JAX")

    print(
        "target_accept:",
        FINAL_TARGET_ACCEPT,
    )

    print(
        "seed:",
        FINAL_SEED,
    )

    print("\nCheckpoint:")

    print(FINAL_TRACE_PATH)

    t0 = time.perf_counter()

    with model_1982_2025:

        final_trace_2025 = pm.sample(
            draws=FINAL_DRAWS,
            tune=FINAL_TUNE,
            chains=FINAL_CHAINS,
            random_seed=FINAL_SEED,
            target_accept=FINAL_TARGET_ACCEPT,
            nuts_sampler="nutpie",
            nuts_sampler_kwargs={
                "backend": "jax",
            },
            return_inferencedata=True,
            progressbar=True,
        )

    elapsed = time.perf_counter() - t0

    print(
        "\nSampling wall time:",
        f"{elapsed / 60.0:.2f} minutes",
    )

    # ------------------------------------------------------
    # Save grouped DataTree using HDF5-backed NetCDF.
    # ------------------------------------------------------

    final_trace_2025.to_netcdf(
        FINAL_TRACE_PATH,
        engine=CHECKPOINT_ENGINE,
        mode="w",
    )

    print("\nSaved final production trace ->")

    print(FINAL_TRACE_PATH)


print("\nStage 14 complete.")

## Stage 15 — Final production health and convergence gate

Evaluates global summaries, per-variable R-hat/ESS diagnostics, divergences, and the
historical temperature relationship from the 4,000-draw final posterior.

The final production gate is **R-hat < 1.01 for every monitored parameter/state and
zero divergences**.


In [ ]:
# ==========================================================
# STAGE 15 — FINAL 1982–2025 PRODUCTION HEALTH
# ==========================================================
# Canonical production convergence gate. Do not use xarray
# .to_array() across heterogeneous parameter dimensions.

import arviz as az
import numpy as np
import pandas as pd

# ==========================================================
# EXTRACT DATASETS
# ==========================================================

posterior_final = final_trace_2025["posterior"].to_dataset()


sample_stats_final = final_trace_2025["sample_stats"].to_dataset()


print("Posterior dimensions:")

print(dict(posterior_final.sizes))


# ==========================================================
# GLOBAL PARAMETER SUMMARY
# ==========================================================

GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]


print("\n========================================")

print("FINAL GLOBAL POSTERIOR — 1982–2025")

print("========================================")


global_final_summary = az.summary(
    posterior_final,
    var_names=GLOBAL_VARS,
    kind="all",
)


display(global_final_summary)


# ==========================================================
# FULL RHAT / ESS
# ==========================================================

rhat_final_ds = az.rhat(posterior_final)


ess_bulk_final_ds = az.ess(
    posterior_final,
    method="bulk",
)


ess_tail_final_ds = az.ess(
    posterior_final,
    method="tail",
)


def flatten_dataset_variables(
    dataset,
):

    values_all = []

    for var_name in dataset.data_vars:

        values = np.asarray(
            dataset[var_name].values,
            dtype=float,
        ).ravel()

        values = values[np.isfinite(values)]

        if values.size:

            values_all.append(values)

    if not values_all:

        return np.array(
            [],
            dtype=float,
        )

    return np.concatenate(values_all)


rhat_values = flatten_dataset_variables(rhat_final_ds)


ess_bulk_values = flatten_dataset_variables(ess_bulk_final_ds)


ess_tail_values = flatten_dataset_variables(ess_tail_final_ds)


max_rhat = float(np.max(rhat_values))


n_rhat_gt_101 = int(np.sum(rhat_values > 1.01))


n_rhat_gt_105 = int(np.sum(rhat_values > 1.05))


min_ess_bulk = float(np.min(ess_bulk_values))


median_ess_bulk = float(np.median(ess_bulk_values))


min_ess_tail = float(np.min(ess_tail_values))


# ==========================================================
# DIVERGENCES
# ==========================================================

if "diverging" in sample_stats_final:

    divergences = int(np.asarray(sample_stats_final["diverging"].values).sum())

else:

    divergences = 0


print("\n========================================")

print("FINAL FULL HEALTH")

print("========================================")


print(
    "max Rhat:",
    max_rhat,
)

print(
    "n Rhat > 1.01:",
    n_rhat_gt_101,
)

print(
    "n Rhat > 1.05:",
    n_rhat_gt_105,
)

print(
    "min ESS bulk:",
    min_ess_bulk,
)

print(
    "median ESS bulk:",
    median_ess_bulk,
)

print(
    "min ESS tail:",
    min_ess_tail,
)

print(
    "divergences:",
    divergences,
)


# ==========================================================
# PER-VARIABLE HEALTH
# ==========================================================

health_rows = []


for var_name in rhat_final_ds.data_vars:

    rhat = np.asarray(
        rhat_final_ds[var_name].values,
        dtype=float,
    ).ravel()

    bulk = np.asarray(
        ess_bulk_final_ds[var_name].values,
        dtype=float,
    ).ravel()

    tail = np.asarray(
        ess_tail_final_ds[var_name].values,
        dtype=float,
    ).ravel()

    rhat = rhat[np.isfinite(rhat)]

    bulk = bulk[np.isfinite(bulk)]

    tail = tail[np.isfinite(tail)]

    health_rows.append(
        {
            "variable": var_name,
            "n_parameters": len(rhat),
            "max_rhat": (float(rhat.max()) if len(rhat) else np.nan),
            "n_rhat_gt_1.01": (int(np.sum(rhat > 1.01)) if len(rhat) else 0),
            "min_ess_bulk": (float(bulk.min()) if len(bulk) else np.nan),
            "median_ess_bulk": (float(np.median(bulk)) if len(bulk) else np.nan),
            "min_ess_tail": (float(tail.min()) if len(tail) else np.nan),
        }
    )


final_health_table = (
    pd.DataFrame(health_rows)
    .sort_values(
        [
            "max_rhat",
            "min_ess_bulk",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print("\nPer-variable final health:")


display(final_health_table)


# ==========================================================
# HISTORICAL TEMPERATURE CHECK
# ==========================================================

tau_q_final = posterior_final["tau_hist_quali"].values.reshape(-1)


tau_r_final = posterior_final["tau_hist_race"].values.reshape(-1)


print("\nHistorical temperatures:")


print(
    "P(tau_race > tau_quali):",
    float(np.mean(tau_r_final > tau_q_final)),
)


print(
    "median tau qualifying:",
    float(np.median(tau_q_final)),
)


print(
    "median tau race:",
    float(np.median(tau_r_final)),
)


print(
    "median race/quali ratio:",
    float(np.median(tau_r_final / tau_q_final)),
)


# ==========================================================
# FORMAL PRODUCTION GATE
# ==========================================================

health_pass = (max_rhat < 1.01) and (n_rhat_gt_101 == 0) and (divergences == 0)


print("\n========================================")


if health_pass:

    print("FINAL PRODUCTION HEALTH: PASS")

else:

    print("FINAL PRODUCTION HEALTH: REVIEW")


print("========================================")

# Part IV — Final Pantheon scoring and robustness

## Stage 16 — Final 1982–2025 Pantheon / JAWS leaderboard

Applies the locked final scoring definition to the production posterior.

Actual-start eligibility is computed from race results/statuses rather than from the
subset of races that happen to yield usable pace observations. The JAWS normalisation
pool contains every driver with at least five modelled seasons; the 35-start and
3-teammate gates are applied afterwards.


In [ ]:
# ==========================================================
# STAGE 16 — FINAL 1982–2025 PANTHEON / JAWS LEADERBOARD
# ==========================================================
#
# FINAL LOCKED DEFINITION
#
# Eligibility:
#   >= 35 actual F1 race starts
#   >= 3 unique actual-start teammates
#   >= 5 modelled seasons
#
# Score:
#   peak = mean(best 5 alpha seasons)
#   surplus = sum(max(alpha, 0))
#   longevity = sqrt(surplus)
#
# Per posterior draw:
#   peak max      -> 100
#   longevity max -> 100
#
# JAWS:
#   0.60 * peak
# + 0.40 * longevity
#
# Normalisation pool:
#   ALL drivers with >=5 modelled seasons,
#   before applying starts / teammate gates.
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd

# ==========================================================
# 0. LOCKED CONSTANTS
# ==========================================================

PANTHEON_MIN_STARTS = 35
PANTHEON_MIN_TEAMMATES = 3
PANTHEON_MIN_MODELLED_SEASONS = 5

PANTHEON_PEAK_WEIGHT = 0.60
PANTHEON_LONGEVITY_WEIGHT = 0.40

PANTHEON_START_YEAR = 1982
PANTHEON_END_YEAR = 2025


# ==========================================================
# 1. ACTUAL RACE STARTS
#
# Eligibility must depend on actual race participation,
# not whether a race generated a usable pace observation.
# ==========================================================

career_results = raw.results.merge(
    raw.races[
        [
            "raceId",
            "year",
        ]
    ],
    on="raceId",
    how="left",
    validate="many_to_one",
).copy()


career_results = career_results[
    career_results["year"].between(
        PANTHEON_START_YEAR,
        PANTHEON_END_YEAR,
    )
].copy()


career_results["driver"] = career_results["driverId"].map(raw.driver_ref)


career_results["status_str"] = (
    career_results["statusId"].map(raw.status_map).fillna("").astype(str).str.lower()
)


# Statuses indicating that the entered driver never
# actually started the Grand Prix.
#
# Do NOT exclude retirement, accident, collision,
# disqualification, not classified, etc. Those drivers
# started the race.
NON_START_PATTERN = (
    r"did not start"
    r"|did not qualify"
    r"|did not prequalify"
    r"|withdrawn"
    r"|withdrew"
    r"|injured"
    r"|illness"
    r"|unwell"
    r"|not permitted to start"
)


career_results["is_actual_start"] = ~(
    career_results["status_str"].str.contains(
        NON_START_PATTERN,
        regex=True,
        na=False,
    )
)


actual_starts = (
    career_results[career_results["is_actual_start"]][
        [
            "raceId",
            "year",
            "driverId",
            "driver",
            "constructorId",
            "status_str",
        ]
    ]
    .drop_duplicates(
        subset=[
            "raceId",
            "driverId",
            "constructorId",
        ]
    )
    .copy()
)


starts_lookup = actual_starts.groupby("driver")["raceId"].nunique().to_dict()


print("Actual-start rows:", len(actual_starts))


print("\nStatuses excluded from start counts:")


display(
    career_results.loc[
        ~career_results["is_actual_start"],
        "status_str",
    ]
    .value_counts()
    .rename_axis("status")
    .to_frame("rows")
)


# ==========================================================
# 2. UNIQUE TEAMMATES
#
# A teammate is another driver who actually started for
# the same constructor in the same race.
# ==========================================================

teammate_sets = {driver: set() for driver in idx_all.unique_drivers}


for (
    race_id,
    constructor_id,
), grp in actual_starts.groupby(
    [
        "raceId",
        "constructorId",
    ]
):

    drivers_here = grp["driver"].dropna().unique().tolist()

    for driver in drivers_here:

        if driver not in teammate_sets:

            teammate_sets[driver] = set()

        teammate_sets[driver].update(other for other in drivers_here if other != driver)


teammates_lookup = {
    driver: len(
        teammate_sets.get(
            driver,
            set(),
        )
    )
    for driver in idx_all.unique_drivers
}


# ==========================================================
# 3. FINAL DRIVER-SEASON POSTERIOR
# ==========================================================

if "posterior_final" in globals():

    alpha_post = posterior_final["alpha_dy"].values

else:

    alpha_post = final_trace_2025["posterior"].to_dataset()["alpha_dy"].values


alpha_post = alpha_post.reshape(
    -1,
    alpha_post.shape[-1],
)


n_draws = alpha_post.shape[0]


print("\nPosterior draws:", n_draws)


print("Driver-season states:", alpha_post.shape[1])


assert n_draws == 4000


assert alpha_post.shape[1] == len(idx_all.active_driver_years)


# ==========================================================
# 4. DRIVER -> MODELLED SEASONS
# ==========================================================

driver_years = {}


for driver, year in idx_all.active_driver_years:

    driver_years.setdefault(
        driver,
        [],
    ).append(int(year))


for driver in driver_years:

    driver_years[driver] = sorted(set(driver_years[driver]))


# ==========================================================
# NORMALISATION POOL
#
# This is intentionally >=5 seasons.
#
# Therefore every driver contributing to the peak and
# longevity denominators has a literal five-season peak.
# ==========================================================

scored_drivers = [
    driver
    for driver in idx_all.unique_drivers
    if (
        len(
            driver_years.get(
                driver,
                [],
            )
        )
        >= PANTHEON_MIN_MODELLED_SEASONS
    )
]


print("\nDrivers in JAWS normalization pool:", len(scored_drivers))


assert all(len(driver_years[driver]) >= 5 for driver in scored_drivers)


# ==========================================================
# 5. RAW JAWS COMPONENTS
# ==========================================================

n_scored = len(scored_drivers)


surplus_mat = np.zeros(
    (
        n_draws,
        n_scored,
    ),
    dtype=float,
)


top5_mat = np.zeros(
    (
        n_draws,
        n_scored,
    ),
    dtype=float,
)


season_count_lookup = {}


for j, driver in enumerate(scored_drivers):

    years = driver_years[driver]

    season_count_lookup[driver] = len(years)

    dy_idx = np.array(
        [
            idx_all.dy_map[
                (
                    driver,
                    year,
                )
            ]
            for year in years
        ],
        dtype=int,
    )

    states = alpha_post[
        :,
        dy_idx,
    ]

    # ------------------------------------------------------
    # CAREER POSITIVE SURPLUS
    # ------------------------------------------------------

    surplus_mat[
        :,
        j,
    ] = np.maximum(
        states,
        0.0,
    ).sum(axis=1)

    # ------------------------------------------------------
    # LITERAL BEST-FIVE-SEASON PEAK
    # ------------------------------------------------------

    top5_mat[
        :,
        j,
    ] = np.sort(
        states,
        axis=1,
    )[
        :,
        -5:,
    ].mean(axis=1)


# ==========================================================
# 6. PER-DRAW NORMALISATION
# ==========================================================

damped_surplus_mat = np.sqrt(surplus_mat)


peak_denominator = top5_mat.max(
    axis=1,
    keepdims=True,
)


longevity_denominator = damped_surplus_mat.max(
    axis=1,
    keepdims=True,
)


if np.any(peak_denominator <= 0):

    raise RuntimeError("Non-positive peak normalization denominator.")


if np.any(longevity_denominator <= 0):

    raise RuntimeError("Non-positive longevity normalization denominator.")


peak_score_mat = top5_mat / peak_denominator * 100.0


longevity_score_mat = damped_surplus_mat / longevity_denominator * 100.0


jaws_mat = (
    PANTHEON_PEAK_WEIGHT * peak_score_mat + PANTHEON_LONGEVITY_WEIGHT * longevity_score_mat
)


# ==========================================================
# 7. FINAL PANTHEON ELIGIBILITY
#
# >=35 actual starts
# >=3 unique teammates
# >=5 modelled seasons already enforced by scored_drivers
# ==========================================================

eligible_drivers = [
    driver
    for driver in scored_drivers
    if (
        starts_lookup.get(
            driver,
            0,
        )
        >= PANTHEON_MIN_STARTS
    )
    and (
        teammates_lookup.get(
            driver,
            0,
        )
        >= PANTHEON_MIN_TEAMMATES
    )
]


eligible_indices = np.array(
    [scored_drivers.index(driver) for driver in eligible_drivers],
    dtype=int,
)


eligible_jaws_mat = jaws_mat[
    :,
    eligible_indices,
]


print("\nPantheon-eligible drivers:", len(eligible_drivers))


# ==========================================================
# 8. POSTERIOR RANK DISTRIBUTIONS
# ==========================================================

n_eligible = len(eligible_drivers)


order = np.argsort(
    -eligible_jaws_mat,
    axis=1,
)


rank_mat = np.empty_like(
    order,
    dtype=int,
)


row_ids = np.arange(n_draws)[:, None]


rank_mat[
    row_ids,
    order,
] = np.arange(
    1,
    n_eligible + 1,
)[None, :]


# ==========================================================
# 9. BUILD FINAL LEADERBOARD
# ==========================================================

rows = []


for eligible_j, driver in enumerate(eligible_drivers):

    scored_j = eligible_indices[eligible_j]

    jaws_draws = eligible_jaws_mat[
        :,
        eligible_j,
    ]

    ranks = rank_mat[
        :,
        eligible_j,
    ]

    peak_raw = top5_mat[
        :,
        scored_j,
    ]

    surplus_raw = surplus_mat[
        :,
        scored_j,
    ]

    peak_scaled = peak_score_mat[
        :,
        scored_j,
    ]

    longevity_scaled = longevity_score_mat[
        :,
        scored_j,
    ]

    rows.append(
        {
            "driver": driver,
            "jaws_mean": float(jaws_draws.mean()),
            "jaws_median": float(np.median(jaws_draws)),
            "jaws_lb": float(
                np.quantile(
                    jaws_draws,
                    0.055,
                )
            ),
            "jaws_ub": float(
                np.quantile(
                    jaws_draws,
                    0.945,
                )
            ),
            "ci_width": float(
                np.quantile(
                    jaws_draws,
                    0.945,
                )
                - np.quantile(
                    jaws_draws,
                    0.055,
                )
            ),
            "top5_peak_alpha_mean": float(peak_raw.mean()),
            "career_surplus_mean": float(surplus_raw.mean()),
            "peak_score_mean": float(peak_scaled.mean()),
            "longevity_score_mean": float(longevity_scaled.mean()),
            "rank_mean": float(ranks.mean()),
            "rank_median": float(np.median(ranks)),
            "rank_lb": float(
                np.quantile(
                    ranks,
                    0.055,
                )
            ),
            "rank_ub": float(
                np.quantile(
                    ranks,
                    0.945,
                )
            ),
            "p_rank_1": float(np.mean(ranks == 1)),
            "p_top_5": float(np.mean(ranks <= 5)),
            "p_top_10": float(np.mean(ranks <= 10)),
            "seasons": int(season_count_lookup[driver]),
            "starts": int(
                starts_lookup.get(
                    driver,
                    0,
                )
            ),
            "unique_teammates": int(
                teammates_lookup.get(
                    driver,
                    0,
                )
            ),
        }
    )


pantheon_2025 = (
    pd.DataFrame(rows)
    .sort_values(
        "jaws_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)


pantheon_2025.insert(
    0,
    "rank",
    np.arange(
        1,
        len(pantheon_2025) + 1,
    ),
)


# ==========================================================
# 10. SAVE FINAL RESULTS
# ==========================================================

PANTHEON_CSV_PATH = Path(CFG.output_dir) / ("pantheon_jaws_" "1982_2025_FINAL.csv")


PANTHEON_DRAWS_PATH = Path(CFG.output_dir) / ("pantheon_jaws_" "1982_2025_FINAL_draws.npz")


pantheon_2025.to_csv(
    PANTHEON_CSV_PATH,
    index=False,
)


np.savez_compressed(
    PANTHEON_DRAWS_PATH,
    drivers=np.asarray(
        eligible_drivers,
        dtype=str,
    ),
    jaws=eligible_jaws_mat,
    ranks=rank_mat,
    peak_raw=top5_mat[
        :,
        eligible_indices,
    ],
    surplus_raw=surplus_mat[
        :,
        eligible_indices,
    ],
    peak_score=peak_score_mat[
        :,
        eligible_indices,
    ],
    longevity_score=longevity_score_mat[
        :,
        eligible_indices,
    ],
)


# ==========================================================
# 11. DISPLAY
# ==========================================================

print("\n========================================")

print("FINAL 1982–2025 PANTHEON LEADERBOARD")

print("========================================")


display(pantheon_2025.head(30))


print("\nSaved FINAL Pantheon ->", PANTHEON_CSV_PATH)


print("Saved FINAL posterior draws ->", PANTHEON_DRAWS_PATH)


# ==========================================================
# 12. ELIGIBILITY SANITY
# ==========================================================

assert pantheon_2025["seasons"].min() >= 5


assert pantheon_2025["starts"].min() >= 35


assert pantheon_2025["unique_teammates"].min() >= 3


print("\nAll eligible drivers satisfy:")

print("  seasons >= 5")

print("  actual starts >= 35")

print("  unique teammates >= 3")


# ==========================================================
# 13. PAIRWISE POSTERIOR HELPER
# ==========================================================


def p_jaws_greater(
    driver_a,
    driver_b,
):

    if driver_a not in eligible_drivers:

        raise ValueError(f"{driver_a} is not Pantheon-eligible.")

    if driver_b not in eligible_drivers:

        raise ValueError(f"{driver_b} is not Pantheon-eligible.")

    ia = eligible_drivers.index(driver_a)

    ib = eligible_drivers.index(driver_b)

    return float(
        np.mean(
            eligible_jaws_mat[
                :,
                ia,
            ]
            > eligible_jaws_mat[
                :,
                ib,
            ]
        )
    )


print("\nStage 16 complete.")

## Stage 17 — Optional targeted case study: Eddie Irvine

A post-hoc interpretability audit motivated by Irvine's unexpectedly high posterior
mean rank. It inspects his season-by-season `alpha`, top-five inclusion probabilities,
and nearby comparator drivers.

This section is **diagnostic only** and does not alter the model or official score.


In [ ]:
# ==========================================================
# STAGE 17 — EDDIE IRVINE PEAK / CAREER AUDIT
# ==========================================================

AUDIT_DRIVER = "irvine"

COMPARATORS = [
    "massa",
    "berger",
    "hakkinen",
]


# ==========================================================
# 1. POSTERIOR ALPHA BY SEASON
# ==========================================================


def driver_season_summary(driver):

    years = driver_years[driver]

    rows = []

    for year in years:

        idx = idx_all.dy_map[
            (
                driver,
                year,
            )
        ]

        draws = alpha_post[
            :,
            idx,
        ]

        rows.append(
            {
                "driver": driver,
                "year": year,
                "alpha_mean": float(draws.mean()),
                "alpha_sd": float(draws.std(ddof=1)),
                "alpha_lb": float(
                    np.quantile(
                        draws,
                        0.055,
                    )
                ),
                "alpha_median": float(np.median(draws)),
                "alpha_ub": float(
                    np.quantile(
                        draws,
                        0.945,
                    )
                ),
                "p_alpha_gt_0": float(np.mean(draws > 0)),
            }
        )

    return pd.DataFrame(rows)


irvine_seasons = driver_season_summary(AUDIT_DRIVER).sort_values("year").reset_index(drop=True)


print("========================================")

print("EDDIE IRVINE — SEASON ALPHA")

print("========================================")

display(irvine_seasons)


# ==========================================================
# 2. WHICH SEASONS ENTER HIS TOP FIVE?
#
# Because peak is calculated per posterior draw, the
# identity of the top five seasons can change between draws.
# ==========================================================

irvine_years = np.array(
    driver_years[AUDIT_DRIVER],
    dtype=int,
)


irvine_indices = np.array(
    [
        idx_all.dy_map[
            (
                AUDIT_DRIVER,
                year,
            )
        ]
        for year in irvine_years
    ],
    dtype=int,
)


irvine_states = alpha_post[
    :,
    irvine_indices,
]


top5_position_indices = np.argsort(
    irvine_states,
    axis=1,
)[
    :,
    -5:,
]


top5_inclusion_rows = []


for season_j, year in enumerate(irvine_years):

    included = (top5_position_indices == season_j).any(axis=1)

    top5_inclusion_rows.append(
        {
            "year": int(year),
            "alpha_mean": float(
                irvine_states[
                    :,
                    season_j,
                ].mean()
            ),
            "p_in_top5": float(included.mean()),
        }
    )


irvine_top5 = (
    pd.DataFrame(top5_inclusion_rows)
    .sort_values(
        [
            "p_in_top5",
            "alpha_mean",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n========================================")

print("IRVINE — PROBABILITY EACH SEASON IS TOP FIVE")

print("========================================")

display(irvine_top5)


# ==========================================================
# 3. POSTERIOR-MEAN TOP FIVE
#
# Descriptive only.
# The official JAWS peak remains computed per draw.
# ==========================================================

mean_top5 = irvine_seasons.sort_values(
    "alpha_mean",
    ascending=False,
).head(5)


print("\nTop five seasons by posterior mean alpha:")

display(mean_top5)


print(
    "\nMean of those five posterior means:",
    mean_top5["alpha_mean"].mean(),
)


# ==========================================================
# 4. COMPARE CAREER SHAPES
# ==========================================================

comparison_frames = []


for driver in [
    AUDIT_DRIVER,
    *COMPARATORS,
]:

    g = driver_season_summary(driver)

    g["mean_rank_within_career"] = (
        g["alpha_mean"]
        .rank(
            ascending=False,
            method="min",
        )
        .astype(int)
    )

    comparison_frames.append(g)


comparison_seasons = pd.concat(
    comparison_frames,
    ignore_index=True,
)


print("\n========================================")

print("IRVINE / MASSA / BERGER / HAKKINEN")

print("========================================")

display(
    comparison_seasons.sort_values(
        [
            "driver",
            "year",
        ]
    )
)


# ==========================================================
# 5. PEAK + SURPLUS SUMMARY
# ==========================================================

cluster = pantheon_2025[
    pantheon_2025["driver"].isin(
        [
            "irvine",
            "massa",
            "berger",
            "hakkinen",
        ]
    )
][
    [
        "rank",
        "driver",
        "jaws_mean",
        "top5_peak_alpha_mean",
        "career_surplus_mean",
        "peak_score_mean",
        "longevity_score_mean",
        "rank_mean",
        "rank_lb",
        "rank_ub",
        "p_top_10",
    ]
].sort_values(
    "jaws_mean",
    ascending=False,
)


print("\nCluster-level JAWS decomposition:")

display(cluster)


# ==========================================================
# 6. PAIRWISE JAWS PROBABILITIES
# ==========================================================

pairs = [
    ("irvine", "massa"),
    ("irvine", "berger"),
    ("irvine", "hakkinen"),
]


pair_rows = []


for a, b in pairs:

    pair_rows.append(
        {
            "comparison": f"{a} > {b}",
            "probability": p_jaws_greater(
                a,
                b,
            ),
        }
    )


print("\nPairwise posterior JAWS probabilities:")

display(pd.DataFrame(pair_rows))


print("\nStage 17 complete.")

## Stage 18 — Top-five posterior selection-bonus diagnostic

Quantifies the order-statistic effect created by selecting the five best seasons
inside each posterior draw. It compares the official draw-wise peak with a plug-in
peak obtained by first averaging each season and then selecting five.

This is a robustness/interpretation diagnostic; the official JAWS definition remains
unchanged.


In [ ]:
# ==========================================================
# STAGE 18 — TOP-FIVE POSTERIOR SELECTION-BONUS DIAGNOSTIC
# ==========================================================
#
# Purpose:
#
# Compare:
#
#   A) official Bayesian peak:
#      E[ mean(top 5 alpha seasons in each posterior draw) ]
#
# against:
#
#   B) plug-in peak:
#      mean(top 5 posterior-mean alpha seasons)
#
# Difference:
#
#   selection_bonus_alpha = A - B
#
# This is DIAGNOSTIC ONLY.
# It does NOT alter the official Pantheon / JAWS score.
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. POSTERIOR MEAN / SD OF EVERY DRIVER-SEASON STATE
# ==========================================================

alpha_state_mean = alpha_post.mean(axis=0)


alpha_state_sd = alpha_post.std(
    axis=0,
    ddof=1,
)


# ==========================================================
# 2. DRIVER-LEVEL SELECTION BONUS
# ==========================================================

selection_rows = []


for j, driver in enumerate(scored_drivers):

    years = driver_years[driver]

    dy_idx = np.array(
        [
            idx_all.dy_map[
                (
                    driver,
                    year,
                )
            ]
            for year in years
        ],
        dtype=int,
    )

    states = alpha_post[
        :,
        dy_idx,
    ]

    season_means = alpha_state_mean[dy_idx]

    season_sds = alpha_state_sd[dy_idx]

    n_seasons = len(years)

    # ------------------------------------------------------
    # OFFICIAL RAW PEAK
    #
    # This is already calculated in Stage 16:
    #
    # for each posterior draw:
    #     select best 5 seasons
    #     average them
    #
    # then average that posterior quantity.
    # ------------------------------------------------------

    official_drawwise_peak = float(
        top5_mat[
            :,
            j,
        ].mean()
    )

    # ------------------------------------------------------
    # PLUG-IN PEAK
    #
    # First average each season over posterior draws.
    # Then select the five largest means.
    # ------------------------------------------------------

    sorted_mean_idx = np.argsort(season_means)

    top5_mean_idx = sorted_mean_idx[-5:]

    plugin_peak = float(season_means[top5_mean_idx].mean())

    # ------------------------------------------------------
    # SELECTION BONUS
    # ------------------------------------------------------

    selection_bonus = official_drawwise_peak - plugin_peak

    # ------------------------------------------------------
    # RELATIVE BONUS
    # ------------------------------------------------------

    if abs(plugin_peak) > 1e-12:

        bonus_fraction = selection_bonus / abs(plugin_peak)

    else:

        bonus_fraction = np.nan

    # ------------------------------------------------------
    # 5th / 6th POSTERIOR-MEAN BOUNDARY
    #
    # Small gap => many seasons are plausible candidates
    # for the final top-five position.
    # ------------------------------------------------------

    sorted_means = np.sort(season_means)

    if n_seasons >= 6:

        fifth_best = sorted_means[-5]

        sixth_best = sorted_means[-6]

        boundary_gap = float(fifth_best - sixth_best)

    else:

        boundary_gap = np.nan

    # ------------------------------------------------------
    # CAREER-WIDE POSTERIOR SEASON CORRELATION
    #
    # If seasons move almost perfectly together across
    # posterior draws, there should be less opportunity for
    # top-five membership to switch.
    # ------------------------------------------------------

    if n_seasons >= 2:

        corr = np.corrcoef(
            states,
            rowvar=False,
        )

        upper = corr[
            np.triu_indices(
                n_seasons,
                k=1,
            )
        ]

        upper = upper[np.isfinite(upper)]

        mean_pairwise_corr = float(upper.mean()) if len(upper) else np.nan

    else:

        mean_pairwise_corr = np.nan

    # ------------------------------------------------------
    # IDENTIFY FIVE BEST POSTERIOR-MEAN SEASONS
    # ------------------------------------------------------

    plugin_top5_years = [years[k] for k in top5_mean_idx[::-1]]

    # ------------------------------------------------------
    # OFFICIAL LEADERBOARD INFO IF ELIGIBLE
    # ------------------------------------------------------

    pantheon_match = pantheon_2025[pantheon_2025["driver"] == driver]

    if len(pantheon_match):

        official_rank = int(pantheon_match.iloc[0]["rank"])

        official_jaws = float(pantheon_match.iloc[0]["jaws_mean"])

        pantheon_eligible = True

    else:

        official_rank = np.nan
        official_jaws = np.nan
        pantheon_eligible = False

    selection_rows.append(
        {
            "driver": driver,
            "n_seasons": n_seasons,
            "official_drawwise_peak": official_drawwise_peak,
            "plugin_top5_peak": plugin_peak,
            "selection_bonus_alpha": selection_bonus,
            "selection_bonus_pct": (100.0 * bonus_fraction),
            "mean_season_sd": float(season_sds.mean()),
            "median_season_sd": float(np.median(season_sds)),
            "mean_pairwise_season_corr": mean_pairwise_corr,
            "top5_boundary_gap": boundary_gap,
            "plugin_top5_years": ", ".join(str(year) for year in plugin_top5_years),
            "pantheon_eligible": pantheon_eligible,
            "official_rank": official_rank,
            "official_jaws": official_jaws,
        }
    )


selection_bonus_df = pd.DataFrame(selection_rows)


# ==========================================================
# 3. BONUS RANK / PERCENTILE
# ==========================================================

selection_bonus_df["bonus_rank"] = (
    selection_bonus_df["selection_bonus_alpha"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)


selection_bonus_df["bonus_percentile"] = (
    selection_bonus_df["selection_bonus_alpha"].rank(
        pct=True,
        ascending=True,
    )
    * 100.0
)


selection_bonus_df = selection_bonus_df.sort_values(
    "selection_bonus_alpha",
    ascending=False,
).reset_index(drop=True)


# ==========================================================
# 4. OVERALL DISTRIBUTION
# ==========================================================

print("========================================")

print("TOP-FIVE SELECTION BONUS — ALL DRIVERS")

print("========================================")


print("Drivers:", len(selection_bonus_df))


print("\nSelection bonus alpha:")


print("mean:", selection_bonus_df["selection_bonus_alpha"].mean())


print("median:", selection_bonus_df["selection_bonus_alpha"].median())


print("90th percentile:", selection_bonus_df["selection_bonus_alpha"].quantile(0.90))


print("95th percentile:", selection_bonus_df["selection_bonus_alpha"].quantile(0.95))


print("max:", selection_bonus_df["selection_bonus_alpha"].max())


# ==========================================================
# 5. BIGGEST BENEFICIARIES
# ==========================================================

print("\n========================================")

print("TOP 25 SELECTION BONUSES")

print("========================================")


display(
    selection_bonus_df[
        [
            "bonus_rank",
            "driver",
            "selection_bonus_alpha",
            "selection_bonus_pct",
            "official_drawwise_peak",
            "plugin_top5_peak",
            "n_seasons",
            "mean_season_sd",
            "mean_pairwise_season_corr",
            "top5_boundary_gap",
            "plugin_top5_years",
            "official_rank",
        ]
    ].head(25)
)


# ==========================================================
# 6. IRVINE SPECIFIC RESULT
# ==========================================================

irvine_bonus = selection_bonus_df[selection_bonus_df["driver"] == "irvine"]


print("\n========================================")

print("EDDIE IRVINE SELECTION BONUS")

print("========================================")


display(
    irvine_bonus[
        [
            "driver",
            "bonus_rank",
            "bonus_percentile",
            "official_drawwise_peak",
            "plugin_top5_peak",
            "selection_bonus_alpha",
            "selection_bonus_pct",
            "n_seasons",
            "mean_season_sd",
            "mean_pairwise_season_corr",
            "top5_boundary_gap",
            "plugin_top5_years",
            "official_rank",
            "official_jaws",
        ]
    ]
)


# ==========================================================
# 7. COMPARISON CLUSTER
# ==========================================================

cluster_drivers = [
    "irvine",
    "massa",
    "berger",
    "hakkinen",
]


print("\n========================================")

print("IRVINE / MASSA / BERGER / HAKKINEN")

print("========================================")


display(
    selection_bonus_df[selection_bonus_df["driver"].isin(cluster_drivers)][
        [
            "driver",
            "selection_bonus_alpha",
            "selection_bonus_pct",
            "bonus_rank",
            "bonus_percentile",
            "official_drawwise_peak",
            "plugin_top5_peak",
            "n_seasons",
            "mean_season_sd",
            "mean_pairwise_season_corr",
            "top5_boundary_gap",
            "official_rank",
        ]
    ].sort_values("official_rank")
)


# ==========================================================
# 8. CORRELATION DIAGNOSTICS
# ==========================================================

diagnostic_variables = [
    "n_seasons",
    "mean_season_sd",
    "median_season_sd",
    "mean_pairwise_season_corr",
    "top5_boundary_gap",
]


correlation_rows = []


for variable in diagnostic_variables:

    sub = selection_bonus_df[
        [
            "selection_bonus_alpha",
            variable,
        ]
    ].dropna()

    pearson = sub["selection_bonus_alpha"].corr(
        sub[variable],
        method="pearson",
    )

    spearman = sub["selection_bonus_alpha"].corr(
        sub[variable],
        method="spearman",
    )

    correlation_rows.append(
        {
            "variable": variable,
            "n": len(sub),
            "pearson_corr": pearson,
            "spearman_corr": spearman,
        }
    )


selection_correlations = pd.DataFrame(correlation_rows)


print("\n========================================")

print("CORRELATION WITH SELECTION BONUS")

print("========================================")


display(selection_correlations)


# ==========================================================
# 9. BONUS BY CAREER LENGTH BAND
# ==========================================================

selection_bonus_df["career_length_band"] = pd.cut(
    selection_bonus_df["n_seasons"],
    bins=[
        4,
        7,
        10,
        14,
        100,
    ],
    labels=[
        "5–7",
        "8–10",
        "11–14",
        "15+",
    ],
)


career_band_summary = (
    selection_bonus_df.groupby(
        "career_length_band",
        observed=False,
    )
    .agg(
        drivers=(
            "driver",
            "size",
        ),
        mean_bonus=(
            "selection_bonus_alpha",
            "mean",
        ),
        median_bonus=(
            "selection_bonus_alpha",
            "median",
        ),
        mean_season_sd=(
            "mean_season_sd",
            "mean",
        ),
    )
    .reset_index()
)


print("\n========================================")

print("SELECTION BONUS BY CAREER LENGTH")

print("========================================")


display(career_band_summary)


# ==========================================================
# 10. PLOT — BONUS VS NUMBER OF SEASONS
# ==========================================================

plt.figure(
    figsize=(
        8,
        5,
    )
)


plt.scatter(
    selection_bonus_df["n_seasons"],
    selection_bonus_df["selection_bonus_alpha"],
    alpha=0.7,
)


irvine_row = selection_bonus_df[selection_bonus_df["driver"] == "irvine"].iloc[0]


plt.scatter(
    [irvine_row["n_seasons"]],
    [irvine_row["selection_bonus_alpha"]],
    s=100,
)


plt.annotate(
    "Irvine",
    (
        irvine_row["n_seasons"],
        irvine_row["selection_bonus_alpha"],
    ),
    xytext=(
        6,
        6,
    ),
    textcoords="offset points",
)


plt.xlabel("Modelled seasons")

plt.ylabel("Top-five selection bonus in alpha")

plt.title("Posterior top-five selection bonus vs career length")

plt.grid(alpha=0.2)

plt.show()


# ==========================================================
# 11. PLOT — BONUS VS MEAN SEASON UNCERTAINTY
# ==========================================================

plt.figure(
    figsize=(
        8,
        5,
    )
)


plt.scatter(
    selection_bonus_df["mean_season_sd"],
    selection_bonus_df["selection_bonus_alpha"],
    alpha=0.7,
)


plt.scatter(
    [irvine_row["mean_season_sd"]],
    [irvine_row["selection_bonus_alpha"]],
    s=100,
)


plt.annotate(
    "Irvine",
    (
        irvine_row["mean_season_sd"],
        irvine_row["selection_bonus_alpha"],
    ),
    xytext=(
        6,
        6,
    ),
    textcoords="offset points",
)


plt.xlabel("Mean posterior SD of season alpha")

plt.ylabel("Top-five selection bonus in alpha")

plt.title("Posterior top-five selection bonus vs season uncertainty")

plt.grid(alpha=0.2)

plt.show()


# ==========================================================
# 12. PLOT — BONUS VS 5th/6th BOUNDARY GAP
# ==========================================================

boundary_plot = selection_bonus_df[selection_bonus_df["top5_boundary_gap"].notna()]


plt.figure(
    figsize=(
        8,
        5,
    )
)


plt.scatter(
    boundary_plot["top5_boundary_gap"],
    boundary_plot["selection_bonus_alpha"],
    alpha=0.7,
)


if np.isfinite(irvine_row["top5_boundary_gap"]):

    plt.scatter(
        [irvine_row["top5_boundary_gap"]],
        [irvine_row["selection_bonus_alpha"]],
        s=100,
    )

    plt.annotate(
        "Irvine",
        (
            irvine_row["top5_boundary_gap"],
            irvine_row["selection_bonus_alpha"],
        ),
        xytext=(
            6,
            6,
        ),
        textcoords="offset points",
    )


plt.xlabel("Posterior-mean alpha gap: 5th-best minus 6th-best")

plt.ylabel("Top-five selection bonus in alpha")

plt.title("Selection bonus vs ambiguity at the top-five boundary")

plt.grid(alpha=0.2)

plt.show()


print("\nStage 18 complete.")

## Stage 19 — Fixed-top-five ranking robustness check

Constructs a counterfactual leaderboard in which each driver's five peak seasons are
fixed according to posterior-mean `alpha`, while posterior uncertainty in those five
season values is still propagated.

The official and fixed rankings are compared only to test sensitivity to uncertainty
in top-five membership. The draw-wise top-five Pantheon remains the primary result.


In [ ]:
# ==========================================================
# STAGE 19 — FIXED TOP-FIVE ROBUSTNESS CHECK
# ==========================================================
#
# OFFICIAL PANTHEON:
#
#   For every posterior draw:
#       choose that driver's best 5 seasons
#       average them
#
# COUNTERFACTUAL ROBUSTNESS VERSION:
#
#   1. Calculate posterior-mean alpha for each season.
#   2. Choose the driver's best 5 seasons ONCE.
#   3. Keep those five seasons fixed in every posterior draw.
#   4. Propagate posterior uncertainty in their alpha values.
#   5. Re-normalise peak per draw.
#   6. Leave longevity completely unchanged.
#
# This isolates the effect of uncertainty in WHICH seasons
# enter the top five.
#
# OFFICIAL PANTHEON REMAINS THE PRIMARY RESULT.
# ==========================================================

import numpy as np
import pandas as pd

# ==========================================================
# 1. POSTERIOR MEAN ALPHA FOR EACH DRIVER-SEASON
# ==========================================================

alpha_state_mean = alpha_post.mean(axis=0)


# ==========================================================
# 2. FIXED TOP-FIVE RAW PEAK MATRIX
#
# Same dimensions as top5_mat:
#
#   posterior draws x scored drivers
# ==========================================================

fixed_top5_mat = np.zeros_like(top5_mat)


fixed_top5_years = {}


for j, driver in enumerate(scored_drivers):

    years = driver_years[driver]

    dy_idx = np.array(
        [
            idx_all.dy_map[
                (
                    driver,
                    year,
                )
            ]
            for year in years
        ],
        dtype=int,
    )

    season_means = alpha_state_mean[dy_idx]

    # ------------------------------------------------------
    # Select the five seasons with highest posterior-mean
    # alpha.
    # ------------------------------------------------------

    top5_local_idx = np.argsort(season_means)[-5:]

    # Store years in descending posterior-mean order.
    ordered_top5_local_idx = top5_local_idx[np.argsort(season_means[top5_local_idx])[::-1]]

    fixed_top5_years[driver] = [int(years[k]) for k in ordered_top5_local_idx]

    # ------------------------------------------------------
    # Convert local season positions into global alpha_dy
    # indices.
    # ------------------------------------------------------

    fixed_dy_idx = dy_idx[top5_local_idx]

    # ------------------------------------------------------
    # Same five seasons used in EVERY posterior draw.
    # ------------------------------------------------------

    fixed_top5_mat[
        :,
        j,
    ] = alpha_post[
        :,
        fixed_dy_idx,
    ].mean(axis=1)


# ==========================================================
# 3. SANITY CHECK
#
# Drivers with exactly five modelled seasons must have
# identical official and fixed raw peak values.
# ==========================================================

exactly_five = [
    j for j, driver in enumerate(scored_drivers) if (len(driver_years[driver]) == 5)
]


if exactly_five:

    exact5_difference = np.abs(
        fixed_top5_mat[
            :,
            exactly_five,
        ]
        - top5_mat[
            :,
            exactly_five,
        ]
    )

    print(
        "Max official-vs-fixed difference " "for exactly-five-season drivers:",
        float(exact5_difference.max()),
    )

    assert exact5_difference.max() < 1e-10


# ==========================================================
# 4. FIXED-TOP5 PER-DRAW NORMALISATION
#
# The counterfactual peak metric gets its own denominator,
# using the SAME >=5-season normalization pool.
# ==========================================================

fixed_peak_denominator = fixed_top5_mat.max(
    axis=1,
    keepdims=True,
)


if np.any(fixed_peak_denominator <= 0):

    raise RuntimeError("Non-positive fixed-top5 peak denominator.")


fixed_peak_score_mat = fixed_top5_mat / fixed_peak_denominator * 100.0


# ==========================================================
# 5. COUNTERFACTUAL JAWS
#
# Longevity is intentionally IDENTICAL to official JAWS.
# ==========================================================

fixed_jaws_mat = (
    PANTHEON_PEAK_WEIGHT * fixed_peak_score_mat
    + PANTHEON_LONGEVITY_WEIGHT * longevity_score_mat
)


# ==========================================================
# 6. APPLY EXACT SAME ELIGIBILITY FILTER
# ==========================================================

eligible_fixed_jaws_mat = fixed_jaws_mat[
    :,
    eligible_indices,
]


eligible_fixed_peak_mat = fixed_top5_mat[
    :,
    eligible_indices,
]


# ==========================================================
# 7. COUNTERFACTUAL POSTERIOR RANKS
# ==========================================================

fixed_order = np.argsort(
    -eligible_fixed_jaws_mat,
    axis=1,
)


fixed_rank_mat = np.empty_like(
    fixed_order,
    dtype=int,
)


fixed_rank_mat[
    row_ids,
    fixed_order,
] = np.arange(
    1,
    n_eligible + 1,
)[None, :]


# ==========================================================
# 8. DRIVER-BY-DRIVER COMPARISON
# ==========================================================

robustness_rows = []


for eligible_j, driver in enumerate(eligible_drivers):

    official_jaws_draws = eligible_jaws_mat[
        :,
        eligible_j,
    ]

    fixed_jaws_draws = eligible_fixed_jaws_mat[
        :,
        eligible_j,
    ]

    official_rank_draws = rank_mat[
        :,
        eligible_j,
    ]

    fixed_rank_draws = fixed_rank_mat[
        :,
        eligible_j,
    ]

    scored_j = eligible_indices[eligible_j]

    official_peak_draws = top5_mat[
        :,
        scored_j,
    ]

    fixed_peak_draws = fixed_top5_mat[
        :,
        scored_j,
    ]

    robustness_rows.append(
        {
            "driver": driver,
            # ----------------------------------------------
            # OFFICIAL RESULT
            # ----------------------------------------------
            "official_jaws": float(official_jaws_draws.mean()),
            "official_rank": int(
                pantheon_2025.loc[
                    pantheon_2025["driver"] == driver,
                    "rank",
                ].iloc[0]
            ),
            "official_rank_mean": float(official_rank_draws.mean()),
            "official_p_top10": float(np.mean(official_rank_draws <= 10)),
            # ----------------------------------------------
            # FIXED-TOP5 RESULT
            # ----------------------------------------------
            "fixed_jaws": float(fixed_jaws_draws.mean()),
            "fixed_rank_mean": float(fixed_rank_draws.mean()),
            "fixed_rank_median": float(np.median(fixed_rank_draws)),
            "fixed_rank_lb": float(
                np.quantile(
                    fixed_rank_draws,
                    0.055,
                )
            ),
            "fixed_rank_ub": float(
                np.quantile(
                    fixed_rank_draws,
                    0.945,
                )
            ),
            "fixed_p_top10": float(np.mean(fixed_rank_draws <= 10)),
            # ----------------------------------------------
            # PEAK COMPARISON
            # ----------------------------------------------
            "official_peak": float(official_peak_draws.mean()),
            "fixed_peak": float(fixed_peak_draws.mean()),
            "peak_selection_bonus": float(
                official_peak_draws.mean() - fixed_peak_draws.mean()
            ),
            # ----------------------------------------------
            # SCORE/RANK CHANGE
            # ----------------------------------------------
            "jaws_change_fixed_minus_official": float(
                fixed_jaws_draws.mean() - official_jaws_draws.mean()
            ),
            "rank_mean_change": float(fixed_rank_draws.mean() - official_rank_draws.mean()),
            "p_top10_change": float(
                np.mean(fixed_rank_draws <= 10) - np.mean(official_rank_draws <= 10)
            ),
            "n_seasons": len(driver_years[driver]),
            "fixed_top5_years": ", ".join(str(year) for year in fixed_top5_years[driver]),
        }
    )


robustness_df = pd.DataFrame(robustness_rows)


# ==========================================================
# 9. POINT-ESTIMATE RANK UNDER FIXED VERSION
#
# Important:
# rank by posterior MEAN fixed JAWS, exactly analogous to
# the official leaderboard.
# ==========================================================

fixed_point_order = (
    robustness_df["fixed_jaws"]
    .rank(
        ascending=False,
        method="first",
    )
    .astype(int)
)


robustness_df["fixed_rank"] = fixed_point_order


robustness_df["point_rank_change"] = (
    robustness_df["fixed_rank"] - robustness_df["official_rank"]
)


# Reorder for readability.

robustness_df = robustness_df[
    [
        "driver",
        "official_rank",
        "fixed_rank",
        "point_rank_change",
        "official_jaws",
        "fixed_jaws",
        "jaws_change_fixed_minus_official",
        "official_peak",
        "fixed_peak",
        "peak_selection_bonus",
        "official_rank_mean",
        "fixed_rank_mean",
        "rank_mean_change",
        "official_p_top10",
        "fixed_p_top10",
        "p_top10_change",
        "n_seasons",
        "fixed_top5_years",
    ]
]


# ==========================================================
# 10. FIXED-TOP5 LEADERBOARD
# ==========================================================

fixed_leaderboard = robustness_df.sort_values(
    "fixed_jaws",
    ascending=False,
).reset_index(drop=True)


print("\n========================================")

print("FIXED-TOP5 COUNTERFACTUAL LEADERBOARD")

print("========================================")


display(
    fixed_leaderboard[
        [
            "fixed_rank",
            "driver",
            "fixed_jaws",
            "official_rank",
            "point_rank_change",
            "official_jaws",
            "fixed_rank_mean",
            "fixed_p_top10",
            "fixed_top5_years",
        ]
    ].head(30)
)


# ==========================================================
# 11. LARGEST POINT-RANK MOVEMENTS
# ==========================================================

rank_movement = robustness_df.assign(
    abs_point_rank_change=robustness_df["point_rank_change"].abs()
).sort_values(
    [
        "abs_point_rank_change",
        "official_rank",
    ],
    ascending=[
        False,
        True,
    ],
)


print("\n========================================")

print("LARGEST POINT-RANK CHANGES")

print("========================================")


display(
    rank_movement[
        [
            "driver",
            "official_rank",
            "fixed_rank",
            "point_rank_change",
            "official_jaws",
            "fixed_jaws",
            "peak_selection_bonus",
            "n_seasons",
        ]
    ].head(25)
)


# ==========================================================
# 12. TOP-30 STABILITY
# ==========================================================

official_top30 = set(pantheon_2025.head(30)["driver"])


fixed_top30 = set(fixed_leaderboard.head(30)["driver"])


entered_top30 = sorted(fixed_top30 - official_top30)


left_top30 = sorted(official_top30 - fixed_top30)


print("\n========================================")

print("TOP-30 STABILITY")

print("========================================")


print("Entered top 30 under fixed-top5:", entered_top30)


print("Left top 30 under fixed-top5:", left_top30)


print(
    "Top-30 membership overlap:",
    len(official_top30 & fixed_top30),
    "/ 30",
)


# ==========================================================
# 13. TOP-10 STABILITY
# ==========================================================

official_top10 = set(pantheon_2025.head(10)["driver"])


fixed_top10 = set(fixed_leaderboard.head(10)["driver"])


print("\n========================================")

print("TOP-10 STABILITY")

print("========================================")


print("Entered top 10:", sorted(fixed_top10 - official_top10))


print("Left top 10:", sorted(official_top10 - fixed_top10))


print(
    "Top-10 membership overlap:",
    len(official_top10 & fixed_top10),
    "/ 10",
)


# ==========================================================
# 14. IRVINE + COMPARATOR CLUSTER
# ==========================================================

cluster_drivers = [
    "irvine",
    "massa",
    "berger",
    "hakkinen",
]


print("\n========================================")

print("IRVINE CLUSTER — OFFICIAL VS FIXED")

print("========================================")


display(
    robustness_df[robustness_df["driver"].isin(cluster_drivers)][
        [
            "driver",
            "official_rank",
            "fixed_rank",
            "point_rank_change",
            "official_jaws",
            "fixed_jaws",
            "official_peak",
            "fixed_peak",
            "peak_selection_bonus",
            "official_rank_mean",
            "fixed_rank_mean",
            "official_p_top10",
            "fixed_p_top10",
            "fixed_top5_years",
        ]
    ].sort_values("official_rank")
)


# ==========================================================
# 15. SELECTED HEADLINE DRIVERS
# ==========================================================

headline_drivers = [
    "senna",
    "michael_schumacher",
    "max_verstappen",
    "alonso",
    "mansell",
    "prost",
    "hamilton",
    "rosberg",
    "barrichello",
    "raikkonen",
]


print("\n========================================")

print("HEADLINE DRIVER ROBUSTNESS")

print("========================================")


display(
    robustness_df[robustness_df["driver"].isin(headline_drivers)][
        [
            "driver",
            "official_rank",
            "fixed_rank",
            "point_rank_change",
            "official_jaws",
            "fixed_jaws",
            "peak_selection_bonus",
            "official_rank_mean",
            "fixed_rank_mean",
            "official_p_top10",
            "fixed_p_top10",
        ]
    ].sort_values("official_rank")
)


# ==========================================================
# 16. PAIRWISE IRVINE PROBABILITIES UNDER FIXED-TOP5
# ==========================================================


def p_fixed_jaws_greater(
    driver_a,
    driver_b,
):

    ia = eligible_drivers.index(driver_a)

    ib = eligible_drivers.index(driver_b)

    return float(
        np.mean(
            eligible_fixed_jaws_mat[
                :,
                ia,
            ]
            > eligible_fixed_jaws_mat[
                :,
                ib,
            ]
        )
    )


irvine_pair_rows = []


for other in [
    "massa",
    "berger",
    "hakkinen",
]:

    irvine_pair_rows.append(
        {
            "comparison": f"irvine > {other}",
            "official_probability": p_jaws_greater(
                "irvine",
                other,
            ),
            "fixed_top5_probability": p_fixed_jaws_greater(
                "irvine",
                other,
            ),
        }
    )


irvine_pairwise_fixed = pd.DataFrame(irvine_pair_rows)


print("\n========================================")

print("IRVINE PAIRWISE ROBUSTNESS")

print("========================================")


display(irvine_pairwise_fixed)


# ==========================================================
# 17. OVERALL RANK AGREEMENT
# ==========================================================

point_rank_corr = robustness_df["official_rank"].corr(
    robustness_df["fixed_rank"],
    method="spearman",
)


mean_rank_corr = robustness_df["official_rank_mean"].corr(
    robustness_df["fixed_rank_mean"],
    method="spearman",
)


mean_abs_point_change = robustness_df["point_rank_change"].abs().mean()


median_abs_point_change = robustness_df["point_rank_change"].abs().median()


max_abs_point_change = robustness_df["point_rank_change"].abs().max()


print("\n========================================")

print("OVERALL RANKING ROBUSTNESS")

print("========================================")


print("Spearman correlation, point ranks:", point_rank_corr)


print("Spearman correlation, posterior mean ranks:", mean_rank_corr)


print("Mean absolute point-rank change:", mean_abs_point_change)


print("Median absolute point-rank change:", median_abs_point_change)


print("Maximum absolute point-rank change:", max_abs_point_change)


# ==========================================================
# 18. SAVE ROBUSTNESS TABLE
# ==========================================================

FIXED_TOP5_ROBUSTNESS_PATH = Path(CFG.output_dir) / (
    "pantheon_jaws_" "1982_2025_" "fixed_top5_robustness.csv"
)


robustness_df.to_csv(
    FIXED_TOP5_ROBUSTNESS_PATH,
    index=False,
)


print("\nSaved robustness diagnostic ->", FIXED_TOP5_ROBUSTNESS_PATH)


print("\nStage 19 complete.")

# Part V — Interactive final output

## Stage 20 — Build the final 1982–2025 Pantheon dashboard

Creates the self-contained analysis HTML (with Plotly loaded from its CDN) containing:

- searchable/sortable Pantheon leaderboard;
- 89% JAWS and posterior-rank intervals;
- driver-season `alpha` trajectories;
- posterior rank distributions;
- peak-vs-longevity explorer;
- pairwise posterior JAWS comparisons;
- fixed-top-five robustness diagnostics;
- model/eligibility notes and the pre-1982 truncation caveat.


## Post-fit validation suite


In [ ]:
# ==========================================================
# STAGE 21 — SCHUMACHER CLUSTER NETWORK / EVIDENCE AUDIT
# ==========================================================
#
# Diagnostic only — does not refit or alter the final model.
#
# Focal cluster:
#   Schumacher / Barrichello / Irvine / Herbert
#
# Main questions:
#   1. Which teammate edges connect this cluster to the grid?
#   2. What direct pace / qualifying / historical evidence is
#      carried by those edges?
#   3. In each season, how much of team performance is assigned
#      to driver alpha versus constructor beta?
#   4. Are cluster-driver posteriors strongly correlated because
#      of common team-level uncertainty?
#
# Sign convention throughout:
#   positive => focal driver better than teammate.
# ==========================================================

from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

try:
    import networkx as nx
except ImportError:
    nx = None


# ----------------------------------------------------------
# 0. SETTINGS
# ----------------------------------------------------------

CLUSTER = ["warwick", "patrese", "cheever", "boutsen"]
FOCAL_YEARS = (1982, 1993)
MIN_SHARED_STARTS_GRAPH = 5

AUDIT_DIR = Path(CFG.output_dir) / "network_audit_schumacher_cluster"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

required = [
    "idx_all", "actual_starts", "driver_years", "pantheon_2025",
    "pace_contrast_all", "pace_level_all",
    "q_contrast_all", "q_level_all_model",
    "hist_quali", "hist_race", "raw",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError("Run the final model/scoring stages first. Missing: " + ", ".join(missing))

if any(d not in idx_all.unique_drivers for d in CLUSTER):
    raise RuntimeError("One or more cluster driverRefs are not in idx_all.unique_drivers.")


# ----------------------------------------------------------
# 1. FINAL POSTERIOR
# ----------------------------------------------------------

post = posterior_final if "posterior_final" in globals() else final_trace_2025["posterior"].to_dataset()

def flatten_states(da):
    x = np.asarray(da.values)
    return x.reshape(-1, x.shape[-1])

alpha_draws = flatten_states(post["alpha_dy"])
car_draws = flatten_states(post["car_csy"])
gamma_draws = np.asarray(post["gamma_tenure"].values).reshape(-1)

alpha_mean = alpha_draws.mean(0)
alpha_lb = np.quantile(alpha_draws, 0.055, axis=0)
alpha_ub = np.quantile(alpha_draws, 0.945, axis=0)

car_mean = car_draws.mean(0)
car_lb = np.quantile(car_draws, 0.055, axis=0)
car_ub = np.quantile(car_draws, 0.945, axis=0)

gamma_mean = float(gamma_draws.mean())

assert alpha_draws.shape[1] == len(idx_all.active_driver_years)
assert car_draws.shape[1] == len(idx_all.active_constructor_years)


# ----------------------------------------------------------
# 2. READABLE NAMES + POSTERIOR HELPERS
# ----------------------------------------------------------

driver_names = {
    str(r.driverRef): f"{r.forename} {r.surname}".strip()
    for r in raw.drivers.itertuples(index=False)
} if {"driverRef", "forename", "surname"}.issubset(raw.drivers.columns) else {}

constructor_names = {
    int(r.constructorId): str(r.constructorRef)
    for r in raw.constructors.itertuples(index=False)
} if (
    hasattr(raw, "constructors")
    and raw.constructors is not None
    and {"constructorId", "constructorRef"}.issubset(raw.constructors.columns)
) else {}

def dname(d):
    return driver_names.get(str(d), str(d).replace("_", " ").title())

def cname(c):
    return constructor_names.get(int(c), str(c))

def didx(d, y):
    return idx_all.dy_map.get((d, int(y)))

def cidx(c, y):
    return idx_all.csy_map.get((int(c), int(y)))

def state_summary(mean, lb, ub, idx):
    if idx is None:
        return np.nan, np.nan, np.nan
    return float(mean[idx]), float(lb[idx]), float(ub[idx])

def driver_summary(d, y):
    return state_summary(alpha_mean, alpha_lb, alpha_ub, didx(d, y))

def car_summary(c, y):
    return state_summary(car_mean, car_lb, car_ub, cidx(c, y))

def delta_summary(a, b, years):
    years = [int(y) for y in years if didx(a, y) is not None and didx(b, y) is not None]
    if not years:
        return (np.nan,) * 5

    ds, corrs = [], []
    for y in years:
        ad = alpha_draws[:, didx(a, y)]
        bd = alpha_draws[:, didx(b, y)]
        ds.append(ad - bd)
        corrs.append(np.corrcoef(ad, bd)[0, 1])

    d = np.column_stack(ds).mean(1)

    return (
        float(d.mean()),
        float(np.quantile(d, 0.055)),
        float(np.quantile(d, 0.945)),
        float(np.mean(d > 0)),
        float(np.mean(corrs)),
    )


# ----------------------------------------------------------
# 3. DESCRIPTIVE WITHIN-SEASON DRIVER / CAR RANKS
# ----------------------------------------------------------

driver_rank = {}
for year in sorted({int(y) for _, y in idx_all.active_driver_years}):
    states = [(d, idx_all.dy_map[(d, year)]) for d, y in idx_all.active_driver_years if int(y) == year]
    states.sort(key=lambda x: alpha_mean[x[1]], reverse=True)
    for rank, (d, _) in enumerate(states, 1):
        driver_rank[(d, year)] = rank

car_rank = {}
for year in sorted({int(y) for _, y in idx_all.active_constructor_years}):
    states = [(int(c), idx_all.csy_map[(int(c), year)]) for c, y in idx_all.active_constructor_years if int(y) == year]
    states.sort(key=lambda x: car_mean[x[1]], reverse=True)
    for rank, (c, _) in enumerate(states, 1):
        car_rank[(c, year)] = rank


# ----------------------------------------------------------
# 4. BUILD ACTUAL-START TEAMMATE EDGES
# ----------------------------------------------------------

acc = {}

for (race_id, year, constructor_id), grp in actual_starts.groupby(["raceId", "year", "constructorId"]):
    drivers = sorted(set(grp["driver"].dropna().astype(str)))

    for a, b in combinations(drivers, 2):
        key = tuple(sorted((a, b)))
        rec = acc.setdefault(
            key,
            dict(a=key[0], b=key[1], starts=0, years=set(), constructors=Counter()),
        )
        rec["starts"] += 1
        rec["years"].add(int(year))
        rec["constructors"][int(constructor_id)] += 1

edge_rows = []
for rec in acc.values():
    edge_rows.append(
        {
            "driver_a": rec["a"],
            "driver_b": rec["b"],
            "shared_starts": rec["starts"],
            "shared_seasons": ", ".join(map(str, sorted(rec["years"]))),
            "constructors": ", ".join(
                f"{cname(cid)} ({n})" for cid, n in rec["constructors"].most_common()
            ),
            "_years": sorted(rec["years"]),
        }
    )

teammate_edges = pd.DataFrame(edge_rows)


# ----------------------------------------------------------
# 5. EVIDENCE HELPERS
# ----------------------------------------------------------

def modern_contrast(df, focal, teammate, year=None, constructor_id=None):
    mask = (
        ((df["driver_a"] == focal) & (df["driver_b"] == teammate))
        | ((df["driver_a"] == teammate) & (df["driver_b"] == focal))
    )
    if year is not None:
        mask &= df["year"].eq(int(year))
    if constructor_id is not None:
        mask &= df["constructorId"].eq(int(constructor_id))

    x = df.loc[mask].copy()
    if x.empty:
        return dict(n=0, weekends=0, y=np.nan, pos=np.nan, tenure=np.nan)

    sign = np.where(x["driver_a"].to_numpy() == focal, 1.0, -1.0)
    y = sign * x["y"].to_numpy(float)
    w = x["weight"].to_numpy(float)

    tenure = np.nan
    if "tenure_diff" in x.columns:
        t = sign * x["tenure_diff"].to_numpy(float)
        tenure = gamma_mean * float(np.average(t, weights=w))

    return dict(
        n=len(x),
        weekends=x["raceId"].nunique(),
        y=float(np.average(y, weights=w)),
        pos=float(np.average((y > 0).astype(float), weights=w)),
        tenure=float(tenure),
    )

def historical_pair(df, focal, teammate, year=None, constructor_id=None):
    cols = ["raceId", "year", "constructorId", "driver", "rank_value"]
    a = df.loc[df["driver"].eq(focal), cols].copy()
    b = df.loc[df["driver"].eq(teammate), cols].copy()

    if year is not None:
        a, b = a[a["year"].eq(int(year))], b[b["year"].eq(int(year))]
    if constructor_id is not None:
        a = a[a["constructorId"].eq(int(constructor_id))]
        b = b[b["constructorId"].eq(int(constructor_id))]

    a = a.rename(columns={"rank_value": "a_rank"})
    b = b.rename(columns={"rank_value": "b_rank"})

    m = a.merge(
        b[["raceId", "year", "constructorId", "b_rank"]],
        on=["raceId", "year", "constructorId"],
        how="inner",
        validate="one_to_one",
    )

    if m.empty:
        return dict(n=0, ahead=np.nan, gap=np.nan)

    gap = m["b_rank"].to_numpy(float) - m["a_rank"].to_numpy(float)

    return dict(
        n=len(m),
        ahead=float(np.mean(gap > 0)),
        gap=float(np.mean(gap)),
    )

def team_level(df, year, constructor_id):
    x = df.loc[
        df["year"].eq(int(year))
        & df["constructorId"].eq(int(constructor_id))
    ].copy()

    if x.empty:
        return dict(obs=np.nan, driver=np.nan, car=np.nan, tenure=np.nan, pred=np.nan)

    w = x["weight"].to_numpy(float)
    w = w / w.sum()

    drv = 0.5 * (
        alpha_mean[x["dy_a"].to_numpy(int)]
        + alpha_mean[x["dy_b"].to_numpy(int)]
    )
    car = car_mean[x["csy"].to_numpy(int)]
    ten = gamma_mean * x["tenure_mean"].to_numpy(float)
    pred = drv + car + ten

    return dict(
        obs=float(np.sum(w * x["y"].to_numpy(float))),
        driver=float(np.sum(w * drv)),
        car=float(np.sum(w * car)),
        tenure=float(np.sum(w * ten)),
        pred=float(np.sum(w * pred)),
    )


# ----------------------------------------------------------
# 6. INTERNAL CLUSTER EDGE EVIDENCE
# ----------------------------------------------------------

rows = []

for focal, teammate in combinations(CLUSTER, 2):
    ka, kb = sorted((focal, teammate))
    m = teammate_edges[
        teammate_edges["driver_a"].eq(ka)
        & teammate_edges["driver_b"].eq(kb)
    ]
    if m.empty:
        continue

    e = m.iloc[0]
    dm, dl, du, pgt, corr = delta_summary(focal, teammate, e["_years"])
    pace = modern_contrast(pace_contrast_all, focal, teammate)
    quali = modern_contrast(q_contrast_all, focal, teammate)
    hq = historical_pair(hist_quali, focal, teammate)
    hr = historical_pair(hist_race, focal, teammate)

    rows.append(
        {
            "focal": dname(focal),
            "teammate": dname(teammate),
            "shared_starts": int(e["shared_starts"]),
            "shared_seasons": e["shared_seasons"],
            "constructors": e["constructors"],
            "delta_alpha_mean": dm,
            "delta_alpha_lb": dl,
            "delta_alpha_ub": du,
            "p_focal_gt_teammate": pgt,
            "alpha_corr": corr,
            "pace_weekends": pace["weekends"],
            "pace_mean_y": pace["y"],
            "pace_positive_share": pace["pos"],
            "pace_tenure_effect": pace["tenure"],
            "quali_weekends": quali["weekends"],
            "quali_rows": quali["n"],
            "quali_mean_y": quali["y"],
            "quali_positive_share": quali["pos"],
            "quali_tenure_effect": quali["tenure"],
            "hist_quali_pairs": hq["n"],
            "hist_quali_ahead_share": hq["ahead"],
            "hist_quali_mean_rank_gap": hq["gap"],
            "hist_race_pairs": hr["n"],
            "hist_race_ahead_share": hr["ahead"],
            "hist_race_mean_rank_gap": hr["gap"],
        }
    )

cluster_internal_edges = (
    pd.DataFrame(rows)
    .sort_values("shared_starts", ascending=False)
    .reset_index(drop=True)
)

print("\n========================================")
print("INTERNAL CLUSTER EDGE EVIDENCE")
print("========================================")
display(cluster_internal_edges)


# ----------------------------------------------------------
# 7. DIRECT NETWORK ANCHORS
# ----------------------------------------------------------

rows = []

for focal in CLUSTER:
    touching = teammate_edges[
        teammate_edges["driver_a"].eq(focal)
        | teammate_edges["driver_b"].eq(focal)
    ]

    for _, e in touching.iterrows():
        teammate = e["driver_b"] if e["driver_a"] == focal else e["driver_a"]
        dm, dl, du, pgt, corr = delta_summary(focal, teammate, e["_years"])

        rows.append(
            {
                "focal": dname(focal),
                "teammate": dname(teammate),
                "internal_cluster_edge": teammate in CLUSTER,
                "shared_starts": int(e["shared_starts"]),
                "shared_seasons": e["shared_seasons"],
                "constructors": e["constructors"],
                "delta_alpha_mean": dm,
                "delta_alpha_lb": dl,
                "delta_alpha_ub": du,
                "p_focal_gt_teammate": pgt,
                "alpha_corr": corr,
            }
        )

cluster_anchor_edges = (
    pd.DataFrame(rows)
    .sort_values(["focal", "shared_starts"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n========================================")
print("DIRECT NETWORK ANCHORS")
print("========================================")
display(cluster_anchor_edges)


# ----------------------------------------------------------
# 8. SEASON-BY-SEASON SUPPORT
# ----------------------------------------------------------

rows = []

for focal in CLUSTER:
    stints = (
        actual_starts.loc[
            actual_starts["driver"].eq(focal),
            ["raceId", "year", "constructorId"],
        ]
        .drop_duplicates()
    )

    for (year, constructor_id), g in stints.groupby(["year", "constructorId"]):
        year, constructor_id = int(year), int(constructor_id)
        focal_races = set(g["raceId"].astype(int))

        teammates = sorted(
            set(
                actual_starts.loc[
                    actual_starts["year"].eq(year)
                    & actual_starts["constructorId"].eq(constructor_id)
                    & actual_starts["raceId"].isin(focal_races)
                    & ~actual_starts["driver"].eq(focal),
                    "driver",
                ].dropna().astype(str)
            )
        ) or [None]

        am, al, au = driver_summary(focal, year)
        bm, bl, bu = car_summary(constructor_id, year)

        pace_level = team_level(pace_level_all, year, constructor_id)
        quali_level = team_level(q_level_all_model, year, constructor_id)

        years_focal = sorted(driver_years.get(focal, []))
        prevs = [y for y in years_focal if y < year]
        nexts = [y for y in years_focal if y > year]
        prev_year = max(prevs) if prevs else None
        next_year = min(nexts) if nexts else None

        for teammate in teammates:
            if teammate is None:
                shared = 0
                tm = dm = dl = du = pgt = corr = np.nan
                pace = dict(weekends=0, y=np.nan, pos=np.nan, tenure=np.nan)
                quali = dict(weekends=0, y=np.nan, pos=np.nan, tenure=np.nan)
                hq = hr = dict(n=0, ahead=np.nan, gap=np.nan)
            else:
                teammate_races = set(
                    actual_starts.loc[
                        actual_starts["year"].eq(year)
                        & actual_starts["constructorId"].eq(constructor_id)
                        & actual_starts["driver"].eq(teammate),
                        "raceId",
                    ].astype(int)
                )
                shared = len(focal_races & teammate_races)
                tm = driver_summary(teammate, year)[0]
                dm, dl, du, pgt, corr = delta_summary(focal, teammate, [year])

                pace = modern_contrast(
                    pace_contrast_all, focal, teammate, year, constructor_id
                )
                quali = modern_contrast(
                    q_contrast_all, focal, teammate, year, constructor_id
                )
                hq = historical_pair(
                    hist_quali, focal, teammate, year, constructor_id
                )
                hr = historical_pair(
                    hist_race, focal, teammate, year, constructor_id
                )

            rows.append(
                {
                    "driver": dname(focal),
                    "driver_ref": focal,
                    "year": year,
                    "constructor": cname(constructor_id),
                    "constructorId": constructor_id,
                    "teammate": dname(teammate) if teammate else "",
                    "teammate_ref": teammate,
                    "shared_starts": shared,

                    "alpha_mean": am,
                    "alpha_lb": al,
                    "alpha_ub": au,
                    "driver_season_rank": driver_rank.get((focal, year), np.nan),

                    "teammate_alpha_mean": tm,
                    "delta_alpha_mean": dm,
                    "delta_alpha_lb": dl,
                    "delta_alpha_ub": du,
                    "p_driver_gt_teammate": pgt,
                    "alpha_corr": corr,

                    "car_beta_mean": bm,
                    "car_beta_lb": bl,
                    "car_beta_ub": bu,
                    "car_season_rank": car_rank.get((constructor_id, year), np.nan),

                    "pace_mean_y": pace["y"],
                    "pace_positive_share": pace["pos"],
                    "pace_tenure_effect": pace["tenure"],

                    "quali_mean_y": quali["y"],
                    "quali_positive_share": quali["pos"],
                    "quali_tenure_effect": quali["tenure"],

                    "hist_quali_pairs": hq["n"],
                    "hist_quali_ahead_share": hq["ahead"],
                    "hist_quali_rank_gap": hq["gap"],

                    "hist_race_pairs": hr["n"],
                    "hist_race_ahead_share": hr["ahead"],
                    "hist_race_rank_gap": hr["gap"],

                    "pace_level_observed": pace_level["obs"],
                    "pace_level_driver_component": pace_level["driver"],
                    "pace_level_car_component": pace_level["car"],
                    "pace_level_tenure_component": pace_level["tenure"],
                    "pace_level_predicted": pace_level["pred"],

                    "q1_level_observed": quali_level["obs"],
                    "q1_level_driver_component": quali_level["driver"],
                    "q1_level_car_component": quali_level["car"],
                    "q1_level_tenure_component": quali_level["tenure"],
                    "q1_level_predicted": quali_level["pred"],

                    "previous_modelled_year": prev_year,
                    "previous_alpha_mean": (
                        driver_summary(focal, prev_year)[0]
                        if prev_year is not None else np.nan
                    ),
                    "next_modelled_year": next_year,
                    "next_alpha_mean": (
                        driver_summary(focal, next_year)[0]
                        if next_year is not None else np.nan
                    ),
                }
            )

cluster_season_support = (
    pd.DataFrame(rows)
    .sort_values(["year", "driver", "constructor", "teammate"])
    .reset_index(drop=True)
)

focus = cluster_season_support[
    cluster_season_support["year"].between(*FOCAL_YEARS)
]

print("\n========================================")
print(f"SEASON SUPPORT — {FOCAL_YEARS[0]}–{FOCAL_YEARS[1]}")
print("========================================")

display(
    focus[
        [
            "driver", "year", "constructor", "teammate", "shared_starts",
            "alpha_mean", "driver_season_rank",
            "teammate_alpha_mean",
            "delta_alpha_mean", "delta_alpha_lb", "delta_alpha_ub",
            "p_driver_gt_teammate", "alpha_corr",
            "car_beta_mean", "car_season_rank",
            "pace_mean_y", "quali_mean_y",
            "hist_quali_ahead_share", "hist_race_ahead_share",
            "pace_level_observed", "pace_level_driver_component",
            "pace_level_car_component",
            "q1_level_observed", "q1_level_driver_component",
            "q1_level_car_component",
            "previous_alpha_mean", "next_alpha_mean",
        ]
    ]
)


# ----------------------------------------------------------
# 9. CLEAN TEAM-LEVEL DECOMPOSITION TABLE
# ----------------------------------------------------------

team_decomposition = (
    cluster_season_support[
        [
            "year", "constructorId", "constructor",
            "car_beta_mean", "car_beta_lb", "car_beta_ub", "car_season_rank",
            "pace_level_observed", "pace_level_driver_component",
            "pace_level_car_component", "pace_level_tenure_component",
            "pace_level_predicted",
            "q1_level_observed", "q1_level_driver_component",
            "q1_level_car_component", "q1_level_tenure_component",
            "q1_level_predicted",
        ]
    ]
    .drop_duplicates()
    .sort_values(["year", "constructor"])
    .reset_index(drop=True)
)

print("\n========================================")
print(f"TEAM-LEVEL DECOMPOSITION — {FOCAL_YEARS[0]}–{FOCAL_YEARS[1]}")
print("========================================")

display(
    team_decomposition[
        team_decomposition["year"].between(*FOCAL_YEARS)
    ]
)


# ----------------------------------------------------------
# 10. TEAMMATE EGO-NETWORK
# ----------------------------------------------------------

def career_mean_alpha(driver):
    ids = [didx(driver, y) for y in driver_years.get(driver, [])]
    ids = [i for i in ids if i is not None]
    return float(alpha_mean[ids].mean()) if ids else np.nan

if nx is None:
    print("\nnetworkx not installed — graph skipped.")
    print("Install with: pip install networkx")
else:
    retained = set(CLUSTER)

    for e in teammate_edges.itertuples(index=False):
        if (
            (e.driver_a in CLUSTER or e.driver_b in CLUSTER)
            and e.shared_starts >= MIN_SHARED_STARTS_GRAPH
        ):
            retained.update([e.driver_a, e.driver_b])

    G = nx.Graph()

    for d in retained:
        G.add_node(
            d,
            alpha=career_mean_alpha(d),
            cluster=d in CLUSTER,
            starts=int(starts_lookup.get(d, 0)),
        )

    for e in teammate_edges.itertuples(index=False):
        if e.driver_a not in retained or e.driver_b not in retained:
            continue

        internal = e.driver_a in CLUSTER and e.driver_b in CLUSTER

        if e.shared_starts >= MIN_SHARED_STARTS_GRAPH or internal:
            G.add_edge(
                e.driver_a,
                e.driver_b,
                shared_starts=int(e.shared_starts),
            )

    plt.figure(figsize=(16, 12))

    pos = nx.spring_layout(
        G,
        seed=42,
        k=1.15,
        iterations=300,
        weight="shared_starts",
    )

    values = np.array([G.nodes[n]["alpha"] for n in G.nodes], dtype=float)
    finite = values[np.isfinite(values)]
    vmax = max(0.25, float(np.max(np.abs(finite)))) if finite.size else 1.0

    nx.draw_networkx_nodes(
        G,
        pos,
        node_size=[
            2500 if G.nodes[n]["cluster"]
            else 500 + 12 * np.sqrt(G.nodes[n]["starts"] + 1)
            for n in G.nodes
        ],
        node_color=values,
        cmap=plt.cm.coolwarm,
        vmin=-vmax,
        vmax=vmax,
        edgecolors=[
            "black" if G.nodes[n]["cluster"] else "#555555"
            for n in G.nodes
        ],
        linewidths=[
            3 if G.nodes[n]["cluster"] else 1
            for n in G.nodes
        ],
        alpha=0.95,
    )

    nx.draw_networkx_edges(
        G,
        pos,
        width=[
            0.8 + 0.45 * np.sqrt(G.edges[e]["shared_starts"])
            for e in G.edges
        ],
        alpha=0.45,
    )

    nx.draw_networkx_labels(
        G,
        pos,
        labels={n: dname(n) for n in G.nodes},
        font_size=9,
    )

    edge_labels = {
        e: G.edges[e]["shared_starts"]
        for e in G.edges
        if (
            G.edges[e]["shared_starts"] >= 10
            or (e[0] in CLUSTER and e[1] in CLUSTER)
        )
    }

    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=edge_labels,
        font_size=7,
        rotate=False,
    )

    sm = plt.cm.ScalarMappable(
        norm=Normalize(vmin=-vmax, vmax=vmax),
        cmap=plt.cm.coolwarm,
    )
    sm.set_array([])

    cbar = plt.colorbar(sm, ax=plt.gca(), shrink=0.75)
    cbar.set_label("Mean modelled-season posterior alpha")

    plt.title(
        "Schumacher / Barrichello / Irvine / Herbert teammate network\n"
        "edge labels = shared actual race starts"
    )
    plt.axis("off")
    plt.tight_layout()

    network_path = AUDIT_DIR / "schumacher_cluster_teammate_network.png"
    plt.savefig(network_path, dpi=180, bbox_inches="tight")
    plt.show()


# ----------------------------------------------------------
# 11. SAVE AUDIT TABLES
# ----------------------------------------------------------

teammate_edges.drop(columns="_years").to_csv(
    AUDIT_DIR / "all_teammate_edges.csv",
    index=False,
)

cluster_internal_edges.to_csv(
    AUDIT_DIR / "cluster_internal_edge_evidence.csv",
    index=False,
)

cluster_anchor_edges.to_csv(
    AUDIT_DIR / "cluster_direct_network_anchors.csv",
    index=False,
)

cluster_season_support.to_csv(
    AUDIT_DIR / "cluster_season_support.csv",
    index=False,
)

team_decomposition.to_csv(
    AUDIT_DIR / "cluster_team_level_decomposition.csv",
    index=False,
)


# ----------------------------------------------------------
# 12. QUICK INTERPRETATION GUIDE
# ----------------------------------------------------------

print("\n========================================")
print("NETWORK AUDIT SUMMARY")
print("========================================")

for d in CLUSTER:
    x = pantheon_2025[pantheon_2025["driver"].eq(d)]

    if x.empty:
        print(f"{dname(d):22s} not Pantheon-eligible")
    else:
        r = x.iloc[0]
        print(
            f"{dname(d):22s} "
            f"JAWS #{int(r['rank']):2d}  "
            f"score={r['jaws_mean']:.2f}"
        )

print("\nInterpretation:")
print("  delta_alpha_mean > 0        : focal latent alpha above teammate")
print("  pace/quali mean_y > 0       : direct modern evidence favours focal")
print("  historical ahead_share > .5 : focal more often ranks ahead")
print("  car_beta_mean > 0            : above-average constructor effect")
print("  alpha_corr high              : common level uncertainty; not itself inflation")
print("  team-level components        : how team performance splits into driver/car/tenure")

print("\nSaved audit outputs ->")
print(AUDIT_DIR.resolve())

print("\nStage 21 complete.")


In [ ]:
# ==========================================================
# STAGE 22 — K-FOLD LEAVE-TEAMMATE-EDGE-OUT CALIBRATION
# ==========================================================
#
# PURPOSE
# -------
# Test whether the rest of the Pantheon model/network can predict
# teammate evidence that it has NOT been allowed to see directly.
#
# This is a modern-era (1996–2025) validation because the modern
# likelihood contains explicit teammate-contrast observations.
#
# Validation unit:
#   one unordered driver-pair edge across ALL of the seasons in
#   which that pair appears in the modern contrast likelihood.
#
# Each candidate edge is assigned to exactly one fold. In that fold:
#
#   - ALL pace-contrast rows for that driver pair are removed;
#   - ALL qualifying-contrast rows for that driver pair are removed;
#   - the symmetric team-level observations are RETAINED;
#   - all historical likelihood information is retained;
#   - all other teammate edges are retained.
#
# Retaining team-level rows is intentional: those rows contain the
# pair average + constructor context, but not which teammate was
# better. Therefore this specifically removes DIRECT within-team
# ordering information while preserving field/car information.
#
# This is a grouped K-fold approximation to leave-one-edge-out CV:
# every tested edge is unseen directly once, but several independent
# edges are held out in each refit so we only need K refits.
#
# OUTPUTS
# -------
# 1. Held-out predictive ELPD vs zero-signal null.
# 2. Correlation / RMSE / sign accuracy for unseen contrasts.
# 3. Posterior-predictive probability calibration.
# 4. Edge-season latent-alpha calibration against aggregated,
#    tenure-adjusted held-out teammate evidence.
# 5. Performance split by whether an alternative teammate path
#    remains after the direct edge is removed.
#
# IMPORTANT
# ---------
# This cell NEVER overwrites the official production trace.
# ==========================================================

from collections import defaultdict
from itertools import combinations
from pathlib import Path
import gc
import time

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import scipy.stats as st
from scipy.special import logsumexp
import xarray as xr


# ==========================================================
# 0. VALIDATION CONFIGURATION
# ==========================================================

EDGE_CV_FOLDS = 5

# Only validate edges with enough direct modern race-weekend evidence
# to make their held-out result interpretable.
EDGE_CV_MIN_RACES = 5

EDGE_CV_DRAWS = 300
EDGE_CV_TUNE = 300
EDGE_CV_CHAINS = 2
EDGE_CV_TARGET_ACCEPT = 0.92
EDGE_CV_SEED = 22042

CHECKPOINT_ENGINE_EDGE_CV = "h5netcdf"

EDGE_CV_DIR = (
    Path(CFG.output_dir)
    / "validation_leave_teammate_edge_out"
)

EDGE_CV_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==========================================================
# 1. REQUIRE FINAL MODEL OBJECTS
# ==========================================================

required_objects = [
    "build_joint_model_1982_2025",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "hist_q_pack",
    "hist_race_pack",
    "idx_all",
    "actual_starts",
    "raw",
    "CFG",
]


missing = [
    name
    for name in required_objects
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Run the final model/scoring stages first. Missing: "
        + ", ".join(missing)
    )


# ==========================================================
# 2. SMALL HELPERS
# ==========================================================

def add_edge_id(
    df,
):
    """
    Add a deterministic unordered driver-pair identifier.
    """

    out = df.copy()

    edge_a = []
    edge_b = []
    edge_id = []

    for a, b in zip(
        out["driver_a"].astype(str),
        out["driver_b"].astype(str),
    ):

        aa, bb = sorted(
            (
                a,
                b,
            )
        )

        edge_a.append(aa)
        edge_b.append(bb)
        edge_id.append(f"{aa}|||{bb}")

    out["_edge_a"] = edge_a
    out["_edge_b"] = edge_b
    out["_edge_id"] = edge_id

    return out


def stack_state_draws(
    posterior,
    var_name,
):
    """
    Return posterior state variable as
        n_draws x n_states
    without assuming raw dimension order beyond chain/draw.
    """

    da = posterior[var_name].stack(
        sample=(
            "chain",
            "draw",
        )
    )

    other_dims = [
        dim
        for dim in da.dims
        if dim != "sample"
    ]

    x = da.transpose(
        "sample",
        *other_dims,
    ).values

    return np.asarray(
        x,
        dtype=float,
    ).reshape(
        x.shape[0],
        -1,
    )


def stack_scalar_draws(
    posterior,
    var_name,
):

    return np.asarray(
        posterior[var_name].values,
        dtype=float,
    ).reshape(-1)


def safe_corr(
    x,
    y,
    method="pearson",
):

    x = pd.Series(
        np.asarray(
            x,
            dtype=float,
        )
    )

    y = pd.Series(
        np.asarray(
            y,
            dtype=float,
        )
    )

    ok = x.notna() & y.notna()

    if ok.sum() < 3:

        return np.nan

    if x[ok].nunique() < 2 or y[ok].nunique() < 2:

        return np.nan

    return float(
        x[ok].corr(
            y[ok],
            method=method,
        )
    )


def weighted_average(
    x,
    w,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    w = np.asarray(
        w,
        dtype=float,
    )

    ok = (
        np.isfinite(x)
        &
        np.isfinite(w)
        &
        (w > 0)
    )

    if not ok.any():

        return np.nan

    return float(
        np.average(
            x[ok],
            weights=w[ok],
        )
    )


def weighted_rmse(
    observed,
    predicted,
    weights,
):

    observed = np.asarray(
        observed,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    ok = (
        np.isfinite(observed)
        &
        np.isfinite(predicted)
        &
        np.isfinite(weights)
        &
        (weights > 0)
    )

    if not ok.any():

        return np.nan

    return float(
        np.sqrt(
            np.average(
                (
                    observed[ok]
                    -
                    predicted[ok]
                )
                ** 2,
                weights=weights[ok],
            )
        )
    )


# ==========================================================
# 3. ADD EDGE IDS TO THE TWO CONTRAST TABLES
# ==========================================================

pace_cv = add_edge_id(
    pace_contrast_all
)

q_cv = add_edge_id(
    q_contrast_all
)


# ==========================================================
# 4. BUILD THE MODERN EDGE TABLE
# ==========================================================

pace_meta = (
    pace_cv[
        [
            "_edge_id",
            "_edge_a",
            "_edge_b",
            "raceId",
            "year",
        ]
    ]
    .copy()
)

pace_meta["channel"] = "pace"


q_meta = (
    q_cv[
        [
            "_edge_id",
            "_edge_a",
            "_edge_b",
            "raceId",
            "year",
        ]
    ]
    .copy()
)

q_meta["channel"] = "quali"


edge_meta = pd.concat(
    [
        pace_meta,
        q_meta,
    ],
    ignore_index=True,
)


edge_counts = (
    edge_meta.groupby(
        [
            "_edge_id",
            "_edge_a",
            "_edge_b",
        ],
        as_index=False,
    )
    .agg(
        n_rows=(
            "channel",
            "size",
        ),
        n_races=(
            "raceId",
            "nunique",
        ),
        first_year=(
            "year",
            "min",
        ),
        last_year=(
            "year",
            "max",
        ),
    )
)


pace_counts = (
    pace_meta.groupby(
        "_edge_id"
    )
    .size()
    .rename(
        "pace_rows"
    )
)


q_counts = (
    q_meta.groupby(
        "_edge_id"
    )
    .size()
    .rename(
        "quali_rows"
    )
)


edge_counts = (
    edge_counts
    .merge(
        pace_counts,
        on="_edge_id",
        how="left",
    )
    .merge(
        q_counts,
        on="_edge_id",
        how="left",
    )
)


edge_counts[
    [
        "pace_rows",
        "quali_rows",
    ]
] = (
    edge_counts[
        [
            "pace_rows",
            "quali_rows",
        ]
    ]
    .fillna(0)
    .astype(int)
)


candidate_edges = (
    edge_counts[
        edge_counts[
            "n_races"
        ]
        >= EDGE_CV_MIN_RACES
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if len(candidate_edges) < 2:

    raise RuntimeError(
        "Too few candidate teammate edges for cross-validation. "
        "Lower EDGE_CV_MIN_RACES."
    )


N_EDGE_FOLDS = min(
    EDGE_CV_FOLDS,
    len(candidate_edges),
)


print(
    "========================================"
)

print(
    "STAGE 22 — LEAVE-TEAMMATE-EDGE-OUT CV"
)

print(
    "========================================"
)


print(
    "Candidate edges:",
    len(candidate_edges),
)

print(
    "Minimum shared modern races:",
    EDGE_CV_MIN_RACES,
)

print(
    "Folds:",
    N_EDGE_FOLDS,
)

print(
    "Validation sampler:",
    f"{EDGE_CV_CHAINS} chains × "
    f"({EDGE_CV_TUNE} tune + {EDGE_CV_DRAWS} draws)",
)


# ==========================================================
# 5. NETWORK TOPOLOGY:
#    DOES AN ALTERNATIVE TEAMMATE PATH EXIST?
# ==========================================================
#
# Use the actual-start teammate graph across the full 1982–2025
# coverage as a descriptive topology diagnostic.
#
# For each tested modern edge A-B:
#   remove A-B
#   ask whether another teammate chain still connects A to B.
#
# This is not used in fitting; it only lets us stratify the
# validation afterwards.
# ==========================================================

graph_adjacency = defaultdict(set)


for (
    _race_id,
    _year,
    _constructor_id,
), grp in actual_starts.groupby(
    [
        "raceId",
        "year",
        "constructorId",
    ]
):

    drivers = sorted(
        set(
            grp[
                "driver"
            ]
            .dropna()
            .astype(str)
        )
    )

    for a, b in combinations(
        drivers,
        2,
    ):

        graph_adjacency[a].add(b)
        graph_adjacency[b].add(a)


def shortest_alternative_path_length(
    start,
    target,
):
    """
    Shortest teammate-path length after deleting the direct
    start-target edge. Returns NaN if no alternative path exists.
    """

    if start == target:

        return 0

    queue = [
        (
            start,
            0,
        )
    ]

    visited = {
        start
    }

    qpos = 0

    while qpos < len(queue):

        node, depth = queue[qpos]
        qpos += 1

        for nxt in graph_adjacency.get(
            node,
            set(),
        ):

            # Delete the tested direct edge in both orientations.
            if (
                (
                    node == start
                    and nxt == target
                )
                or
                (
                    node == target
                    and nxt == start
                )
            ):

                continue

            if nxt in visited:

                continue

            if nxt == target:

                return depth + 1

            visited.add(nxt)

            queue.append(
                (
                    nxt,
                    depth + 1,
                )
            )

    return np.nan


candidate_edges[
    "alternative_path_hops"
] = [
    shortest_alternative_path_length(
        a,
        b,
    )
    for a, b in zip(
        candidate_edges[
            "_edge_a"
        ],
        candidate_edges[
            "_edge_b"
        ],
    )
]


candidate_edges[
    "has_alternative_path"
] = candidate_edges[
    "alternative_path_hops"
].notna()


# ==========================================================
# 6. ASSIGN WHOLE EDGES TO BALANCED FOLDS
# ==========================================================
#
# Greedy row-count balancing:
# largest evidence edges are allocated first to the currently
# lightest fold.
# ==========================================================

rng = np.random.default_rng(
    EDGE_CV_SEED
)


assignment_order = (
    candidate_edges
    .assign(
        _tie=
            rng.random(
                len(
                    candidate_edges
                )
            )
    )
    .sort_values(
        [
            "n_rows",
            "_tie",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .copy()
)


fold_loads = np.zeros(
    N_EDGE_FOLDS,
    dtype=int,
)


fold_ids = {}


for _, row in assignment_order.iterrows():

    fold = int(
        np.argmin(
            fold_loads
        )
    )

    fold_ids[
        row[
            "_edge_id"
        ]
    ] = fold

    fold_loads[
        fold
    ] += int(
        row[
            "n_rows"
        ]
    )


candidate_edges[
    "fold"
] = candidate_edges[
    "_edge_id"
].map(
    fold_ids
).astype(int)


candidate_edges = (
    candidate_edges
    .sort_values(
        [
            "fold",
            "n_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


FOLD_ASSIGNMENT_PATH = (
    EDGE_CV_DIR
    / (
        f"edge_assignments_"
        f"{N_EDGE_FOLDS}fold_"
        f"min{EDGE_CV_MIN_RACES}races.csv"
    )
)


candidate_edges.to_csv(
    FOLD_ASSIGNMENT_PATH,
    index=False,
)


print(
    "\nFold evidence loads:"
)

for fold in range(
    N_EDGE_FOLDS
):

    f = candidate_edges[
        candidate_edges[
            "fold"
        ]
        ==
        fold
    ]

    print(
        f"  fold {fold + 1}: "
        f"{len(f)} edges, "
        f"{int(f['n_rows'].sum())} held-out rows"
    )


# ==========================================================
# 7. POSTERIOR SCORING FUNCTION
# ==========================================================

def score_heldout_channel(
    heldout,
    posterior,
    *,
    channel,
    sigma_var,
    fold,
):
    """
    Score held-out contrast rows under the fold posterior.

    Returns one row per held-out likelihood observation.

    Predictive probability includes Student-t observation noise.
    Latent probability refers only to alpha_a - alpha_b.
    """

    if heldout.empty:

        return pd.DataFrame()


    alpha = stack_state_draws(
        posterior,
        "alpha_dy",
    )


    gamma = stack_scalar_draws(
        posterior,
        "gamma_tenure",
    )


    sigma = stack_scalar_draws(
        posterior,
        sigma_var,
    )


    dy_a = heldout[
        "dy_a"
    ].to_numpy(
        dtype=int
    )


    dy_b = heldout[
        "dy_b"
    ].to_numpy(
        dtype=int
    )


    tenure = heldout[
        "tenure_diff"
    ].to_numpy(
        dtype=float
    )


    weight = heldout[
        "weight"
    ].to_numpy(
        dtype=float
    )


    y = heldout[
        "y"
    ].to_numpy(
        dtype=float
    )


    # ------------------------------------------------------
    # Latent teammate ability difference.
    # ------------------------------------------------------

    latent_delta = (
        alpha[
            :,
            dy_a,
        ]
        -
        alpha[
            :,
            dy_b,
        ]
    )


    # ------------------------------------------------------
    # Expected observed teammate contrast.
    # ------------------------------------------------------

    mu = (
        latent_delta
        +
        gamma[
            :,
            None,
        ]
        *
        tenure[
            None,
            :,
        ]
    )


    scale = (
        sigma[
            :,
            None,
        ]
        /
        np.sqrt(
            weight[
                None,
                :,
            ]
        )
    )


    # ------------------------------------------------------
    # Posterior predictive log density.
    # ------------------------------------------------------

    logp_model_draw = st.t.logpdf(
        y[
            None,
            :,
        ],
        df=CFG.student_t_nu,
        loc=mu,
        scale=scale,
    )


    elpd_model = (
        logsumexp(
            logp_model_draw,
            axis=0,
        )
        -
        np.log(
            logp_model_draw.shape[
                0
            ]
        )
    )


    # Zero-signal null with the SAME posterior residual scale.
    logp_null_draw = st.t.logpdf(
        y[
            None,
            :,
        ],
        df=CFG.student_t_nu,
        loc=0.0,
        scale=scale,
    )


    elpd_null = (
        logsumexp(
            logp_null_draw,
            axis=0,
        )
        -
        np.log(
            logp_null_draw.shape[
                0
            ]
        )
    )


    # ------------------------------------------------------
    # P(Y_a - Y_b > 0 | training data)
    #
    # This includes observation noise and is therefore the
    # correct probability for calibration against an actual
    # held-out contrast sign.
    # ------------------------------------------------------

    p_y_positive = np.mean(
        st.t.sf(
            (
                0.0
                -
                mu
            )
            /
            scale,
            df=CFG.student_t_nu,
        ),
        axis=0,
    )


    # ------------------------------------------------------
    # P(alpha_a > alpha_b | training data)
    #
    # This is the latent ability comparison, excluding tenure
    # and observation noise.
    # ------------------------------------------------------

    p_latent_positive = np.mean(
        latent_delta
        >
        0.0,
        axis=0,
    )


    pred_mean = mu.mean(
        axis=0
    )


    latent_mean = latent_delta.mean(
        axis=0
    )


    latent_lb = np.quantile(
        latent_delta,
        0.055,
        axis=0,
    )


    latent_ub = np.quantile(
        latent_delta,
        0.945,
        axis=0,
    )


    gamma_mean_fold = float(
        gamma.mean()
    )


    sigma_mean_fold = float(
        sigma.mean()
    )


    obs_tenure_adjusted = (
        y
        -
        gamma_mean_fold
        *
        tenure
    )


    actual_positive = (
        y
        >
        0.0
    )


    predicted_positive = (
        p_y_positive
        >=
        0.5
    )


    predictive_confidence = np.maximum(
        p_y_positive,
        1.0
        -
        p_y_positive,
    )


    predictive_correct = (
        predicted_positive
        ==
        actual_positive
    )


    result = heldout[
        [
            "_edge_id",
            "_edge_a",
            "_edge_b",
            "raceId",
            "year",
            "constructorId",
            "driver_a",
            "driver_b",
            "weight",
            "y",
            "tenure_diff",
            "dy_a",
            "dy_b",
        ]
    ].copy()


    result[
        "fold"
    ] = int(
        fold
    )


    result[
        "channel"
    ] = channel


    result[
        "pred_mean"
    ] = pred_mean


    result[
        "latent_delta_mean"
    ] = latent_mean


    result[
        "latent_delta_lb"
    ] = latent_lb


    result[
        "latent_delta_ub"
    ] = latent_ub


    result[
        "p_predictive_a_better"
    ] = p_y_positive


    result[
        "p_latent_a_better"
    ] = p_latent_positive


    result[
        "predictive_confidence"
    ] = predictive_confidence


    result[
        "predictive_correct"
    ] = predictive_correct.astype(
        int
    )


    result[
        "actual_positive"
    ] = actual_positive.astype(
        int
    )


    result[
        "brier"
    ] = (
        p_y_positive
        -
        actual_positive.astype(float)
    ) ** 2


    result[
        "elpd_model"
    ] = elpd_model


    result[
        "elpd_null"
    ] = elpd_null


    result[
        "delta_elpd"
    ] = (
        elpd_model
        -
        elpd_null
    )


    result[
        "obs_tenure_adjusted"
    ] = obs_tenure_adjusted


    result[
        "sigma_mean"
    ] = sigma_mean_fold


    result[
        "effective_precision"
    ] = (
        result[
            "weight"
        ].to_numpy(
            dtype=float
        )
        /
        (
            sigma_mean_fold
            ** 2
        )
    )


    return result


# ==========================================================
# 8. RUN / LOAD THE K FOLD REFITS
# ==========================================================

all_score_frames = []

fold_health_rows = []


GLOBAL_EDGE_CV_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]


for fold in range(
    N_EDGE_FOLDS
):

    print(
        "\n========================================"
    )

    print(
        f"EDGE CV FOLD {fold + 1}/{N_EDGE_FOLDS}"
    )

    print(
        "========================================"
    )


    fold_edge_ids = set(
        candidate_edges.loc[
            candidate_edges[
                "fold"
            ]
            ==
            fold,
            "_edge_id",
        ]
    )


    hold_pc = pace_cv[
        pace_cv[
            "_edge_id"
        ].isin(
            fold_edge_ids
        )
    ].copy()


    hold_qc = q_cv[
        q_cv[
            "_edge_id"
        ].isin(
            fold_edge_ids
        )
    ].copy()


    train_pc = pace_cv[
        ~pace_cv[
            "_edge_id"
        ].isin(
            fold_edge_ids
        )
    ].copy()


    train_qc = q_cv[
        ~q_cv[
            "_edge_id"
        ].isin(
            fold_edge_ids
        )
    ].copy()


    print(
        "Held-out edges:",
        len(
            fold_edge_ids
        ),
    )

    print(
        "Pace contrast:",
        f"{len(train_pc):,} train / {len(hold_pc):,} test",
    )

    print(
        "Quali contrast:",
        f"{len(train_qc):,} train / {len(hold_qc):,} test",
    )


    # Extra helper columns are harmless, but strip them from
    # the model inputs to keep the builder inputs clean.
    helper_cols = [
        "_edge_id",
        "_edge_a",
        "_edge_b",
    ]


    train_pc_model = train_pc.drop(
        columns=helper_cols,
        errors="ignore",
    )


    train_qc_model = train_qc.drop(
        columns=helper_cols,
        errors="ignore",
    )


    fold_model = build_joint_model_1982_2025(
        train_pc_model,
        pace_level_all,
        train_qc_model,
        q_level_all_model,
        hist_q_pack,
        hist_race_pack,
        idx_all,
        cfg=CFG,
    )


    fold_trace_path = (
        EDGE_CV_DIR
        /
        (
            f"edge_cv_"
            f"{N_EDGE_FOLDS}fold_"
            f"min{EDGE_CV_MIN_RACES}races_"
            f"fold{fold + 1}_"
            f"{EDGE_CV_DRAWS}x{EDGE_CV_DRAWS}_"
            f"{EDGE_CV_CHAINS}chain.nc"
        )
    )


    if fold_trace_path.exists():

        print(
            "Loading checkpoint ->",
            fold_trace_path,
        )

        fold_trace = xr.open_datatree(
            fold_trace_path,
            engine=CHECKPOINT_ENGINE_EDGE_CV,
        )


    else:

        print(
            "Sampling fold..."
        )

        t0 = time.perf_counter()


        with fold_model:

            fold_trace = pm.sample(
                draws=EDGE_CV_DRAWS,
                tune=EDGE_CV_TUNE,
                chains=EDGE_CV_CHAINS,
                random_seed=(
                    EDGE_CV_SEED
                    +
                    fold
                ),
                target_accept=EDGE_CV_TARGET_ACCEPT,
                nuts_sampler="nutpie",
                nuts_sampler_kwargs={
                    "backend": "jax",
                },
                return_inferencedata=True,
                progressbar=True,
            )


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        print(
            "Fold wall time:",
            f"{elapsed / 60.0:.2f} min",
        )


        fold_trace.to_netcdf(
            fold_trace_path,
            engine=CHECKPOINT_ENGINE_EDGE_CV,
            mode="w",
        )


        print(
            "Saved ->",
            fold_trace_path,
        )


    posterior_fold = fold_trace[
        "posterior"
    ].to_dataset()


    sample_stats_fold = fold_trace[
        "sample_stats"
    ].to_dataset()


    # ------------------------------------------------------
    # Lightweight validation-fit health.
    # ------------------------------------------------------

    fold_summary = az.summary(
        posterior_fold,
        var_names=GLOBAL_EDGE_CV_VARS,
        kind="diagnostics",
    )


    fold_max_rhat = float(
        fold_summary[
            "r_hat"
        ].max()
    )


    fold_min_ess = float(
        fold_summary[
            "ess_bulk"
        ].min()
    )


    if "diverging" in sample_stats_fold:

        fold_divergences = int(
            np.asarray(
                sample_stats_fold[
                    "diverging"
                ].values
            ).sum()
        )

    else:

        fold_divergences = 0


    fold_health_rows.append(
        {
            "fold": fold + 1,
            "heldout_edges": len(
                fold_edge_ids
            ),
            "heldout_pace_rows": len(
                hold_pc
            ),
            "heldout_quali_rows": len(
                hold_qc
            ),
            "max_global_rhat": fold_max_rhat,
            "min_global_ess_bulk": fold_min_ess,
            "divergences": fold_divergences,
        }
    )


    print(
        "Validation-fit health:",
        f"max global Rhat={fold_max_rhat:.4f},",
        f"min global ESS={fold_min_ess:.1f},",
        f"divergences={fold_divergences}",
    )


    # ------------------------------------------------------
    # Score unseen pace and qualifying contrasts.
    # ------------------------------------------------------

    if len(
        hold_pc
    ):

        pace_scores_fold = score_heldout_channel(
            hold_pc,
            posterior_fold,
            channel="pace",
            sigma_var="sigma_pace_contrast",
            fold=fold + 1,
        )

        all_score_frames.append(
            pace_scores_fold
        )


    if len(
        hold_qc
    ):

        q_scores_fold = score_heldout_channel(
            hold_qc,
            posterior_fold,
            channel="quali",
            sigma_var="sigma_q_contrast",
            fold=fold + 1,
        )

        all_score_frames.append(
            q_scores_fold
        )


    # Avoid retaining all fold posteriors in memory.
    try:

        fold_trace.close()

    except Exception:

        pass


    del fold_model
    del fold_trace
    del posterior_fold
    del sample_stats_fold

    gc.collect()


# ==========================================================
# 9. COLLECT ALL HELD-OUT PREDICTIONS
# ==========================================================

heldout_edge_scores = pd.concat(
    all_score_frames,
    ignore_index=True,
)


fold_health = pd.DataFrame(
    fold_health_rows
)


# Join topology info.
topology_cols = candidate_edges[
    [
        "_edge_id",
        "n_races",
        "first_year",
        "last_year",
        "alternative_path_hops",
        "has_alternative_path",
    ]
].drop_duplicates(
    "_edge_id"
)


heldout_edge_scores = heldout_edge_scores.merge(
    topology_cols,
    on="_edge_id",
    how="left",
    validate="many_to_one",
)


print(
    "\n========================================"
)

print(
    "VALIDATION FIT HEALTH"
)

print(
    "========================================"
)


display(
    fold_health
)


# ==========================================================
# 10. ROW-LEVEL PREDICTIVE METRICS
# ==========================================================

def summarise_predictive_rows(
    df,
    label,
):

    if df.empty:

        return None


    obs = df[
        "y"
    ].to_numpy(
        dtype=float
    )


    pred = df[
        "pred_mean"
    ].to_numpy(
        dtype=float
    )


    w = df[
        "weight"
    ].to_numpy(
        dtype=float
    )


    correct = df[
        "predictive_correct"
    ].to_numpy(
        dtype=float
    )


    return {
        "group": label,
        "rows": len(df),
        "edges": df[
            "_edge_id"
        ].nunique(),
        "pearson_y_vs_pred": safe_corr(
            obs,
            pred,
            "pearson",
        ),
        "spearman_y_vs_pred": safe_corr(
            obs,
            pred,
            "spearman",
        ),
        "rmse": float(
            np.sqrt(
                np.mean(
                    (
                        obs
                        -
                        pred
                    )
                    ** 2
                )
            )
        ),
        "weighted_rmse": weighted_rmse(
            obs,
            pred,
            w,
        ),
        "sign_accuracy": float(
            np.mean(
                correct
            )
        ),
        "weighted_sign_accuracy": weighted_average(
            correct,
            w,
        ),
        "brier": float(
            df[
                "brier"
            ].mean()
        ),
        "model_elpd": float(
            df[
                "elpd_model"
            ].sum()
        ),
        "null_elpd": float(
            df[
                "elpd_null"
            ].sum()
        ),
        "delta_elpd": float(
            df[
                "delta_elpd"
            ].sum()
        ),
        "mean_delta_elpd": float(
            df[
                "delta_elpd"
            ].mean()
        ),
    }


metric_rows = []


for channel in [
    "pace",
    "quali",
]:

    row = summarise_predictive_rows(
        heldout_edge_scores[
            heldout_edge_scores[
                "channel"
            ]
            ==
            channel
        ],
        channel,
    )

    if row is not None:

        metric_rows.append(
            row
        )


metric_rows.append(
    summarise_predictive_rows(
        heldout_edge_scores,
        "pooled",
    )
)


predictive_metrics = pd.DataFrame(
    metric_rows
)


print(
    "\n========================================"
)

print(
    "HELD-OUT CONTRAST PREDICTIVE METRICS"
)

print(
    "========================================"
)


display(
    predictive_metrics
)


# ==========================================================
# 11. PREDICTIVE CALIBRATION:
#     DOES AN X% PREDICTION WIN ABOUT X% OF THE TIME?
# ==========================================================

CAL_BINS = np.array(
    [
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
        1.000001,
    ]
)


CAL_LABELS = [
    "50–60%",
    "60–70%",
    "70–80%",
    "80–90%",
    "90–100%",
]


def make_calibration_table(
    df,
    *,
    confidence_col,
    correct_col,
    group_name,
):

    x = df[
        [
            confidence_col,
            correct_col,
        ]
    ].copy()


    x[
        "confidence_bin"
    ] = pd.cut(
        x[
            confidence_col
        ],
        bins=CAL_BINS,
        labels=CAL_LABELS,
        include_lowest=True,
        right=False,
    )


    out = (
        x.groupby(
            "confidence_bin",
            observed=False,
        )
        .agg(
            n=(
                correct_col,
                "size",
            ),
            mean_confidence=(
                confidence_col,
                "mean",
            ),
            observed_accuracy=(
                correct_col,
                "mean",
            ),
        )
        .reset_index()
    )


    out.insert(
        0,
        "group",
        group_name,
    )


    return out


predictive_calibration_frames = []


for channel in [
    "pace",
    "quali",
]:

    x = heldout_edge_scores[
        heldout_edge_scores[
            "channel"
        ]
        ==
        channel
    ]


    predictive_calibration_frames.append(
        make_calibration_table(
            x,
            confidence_col="predictive_confidence",
            correct_col="predictive_correct",
            group_name=channel,
        )
    )


predictive_calibration = pd.concat(
    predictive_calibration_frames,
    ignore_index=True,
)


print(
    "\n========================================"
)

print(
    "POSTERIOR-PREDICTIVE SIGN CALIBRATION"
)

print(
    "========================================"
)


display(
    predictive_calibration
)


# Plot predictive calibration.
plt.figure(
    figsize=(
        7,
        6,
    )
)


for channel, grp in predictive_calibration.groupby(
    "group"
):

    g = grp[
        grp[
            "n"
        ]
        >
        0
    ]


    plt.plot(
        g[
            "mean_confidence"
        ],
        g[
            "observed_accuracy"
        ],
        marker="o",
        label=channel,
    )


plt.plot(
    [
        0.5,
        1.0,
    ],
    [
        0.5,
        1.0,
    ],
    linestyle="--",
    label="ideal calibration",
)


plt.xlim(
    0.5,
    1.0,
)

plt.ylim(
    0.5,
    1.0,
)

plt.xlabel(
    "Mean predicted probability of predicted winner"
)

plt.ylabel(
    "Observed fraction predicted winner was better"
)

plt.title(
    "Leave-edge-out posterior-predictive calibration"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ==========================================================
# 12. CANONICAL ORIENTATION FOR EDGE-SEASON VALIDATION
# ==========================================================
#
# The contrast tables are stored as driver_a - driver_b, but the
# driver_a ordering is an implementation detail.
#
# For edge-season summaries we orient every row as:
#     edge_a (lexicographically first ref) - edge_b
# so every observation for the same edge has one sign convention.
# ==========================================================

canon_sign = np.where(
    heldout_edge_scores[
        "driver_a"
    ].astype(str)
    ==
    heldout_edge_scores[
        "_edge_a"
    ].astype(str),
    1.0,
    -1.0,
)


heldout_edge_scores[
    "canonical_obs_adjusted"
] = (
    canon_sign
    *
    heldout_edge_scores[
        "obs_tenure_adjusted"
    ].to_numpy(
        dtype=float
    )
)


heldout_edge_scores[
    "canonical_pred_mean"
] = (
    canon_sign
    *
    heldout_edge_scores[
        "pred_mean"
    ].to_numpy(
        dtype=float
    )
)


heldout_edge_scores[
    "canonical_latent_delta_mean"
] = (
    canon_sign
    *
    heldout_edge_scores[
        "latent_delta_mean"
    ].to_numpy(
        dtype=float
    )
)


heldout_edge_scores[
    "canonical_p_latent_positive"
] = np.where(
    canon_sign
    >
    0,
    heldout_edge_scores[
        "p_latent_a_better"
    ].to_numpy(
        dtype=float
    ),
    1.0
    -
    heldout_edge_scores[
        "p_latent_a_better"
    ].to_numpy(
        dtype=float
    ),
)


# ==========================================================
# 13. EDGE-SEASON LATENT ABILITY VALIDATION
# ==========================================================
#
# Combine all held-out pace/quali evidence for the same
# driver-pair season using approximate observation precision:
#
#     effective_precision = weight / sigma_channel^2
#
# We compare:
#
#   posterior latent delta alpha
#
# against:
#
#   held-out contrast evidence after subtracting the posterior
#   mean tenure effect.
#
# This observed aggregate is NOT ground-truth alpha. It is a noisy
# out-of-sample proxy for the latent teammate difference.
# ==========================================================

edge_season_rows = []


for (
    edge_id,
    year,
), grp in heldout_edge_scores.groupby(
    [
        "_edge_id",
        "year",
    ]
):

    w = grp[
        "effective_precision"
    ].to_numpy(
        dtype=float
    )


    obs_adj = weighted_average(
        grp[
            "canonical_obs_adjusted"
        ],
        w,
    )


    pred_obs = weighted_average(
        grp[
            "canonical_pred_mean"
        ],
        w,
    )


    latent_delta = weighted_average(
        grp[
            "canonical_latent_delta_mean"
        ],
        w,
    )


    p_latent = weighted_average(
        grp[
            "canonical_p_latent_positive"
        ],
        w,
    )


    actual_positive = bool(
        obs_adj
        >
        0.0
    )


    latent_predicted_positive = bool(
        p_latent
        >=
        0.5
    )


    edge_season_rows.append(
        {
            "_edge_id": edge_id,
            "edge_a": grp[
                "_edge_a"
            ].iloc[
                0
            ],
            "edge_b": grp[
                "_edge_b"
            ].iloc[
                0
            ],
            "year": int(
                year
            ),
            "fold": int(
                grp[
                    "fold"
                ].iloc[
                    0
                ]
            ),
            "pace_rows": int(
                np.sum(
                    grp[
                        "channel"
                    ]
                    ==
                    "pace"
                )
            ),
            "quali_rows": int(
                np.sum(
                    grp[
                        "channel"
                    ]
                    ==
                    "quali"
                )
            ),
            "races": int(
                grp[
                    "raceId"
                ].nunique()
            ),
            "heldout_adjusted_evidence": obs_adj,
            "predicted_observation_mean": pred_obs,
            "latent_delta_mean": latent_delta,
            "p_latent_edge_a_better": p_latent,
            "latent_confidence": max(
                p_latent,
                1.0
                -
                p_latent,
            ),
            "observed_edge_a_better": int(
                actual_positive
            ),
            "latent_sign_correct": int(
                latent_predicted_positive
                ==
                actual_positive
            ),
            "latent_brier": (
                p_latent
                -
                float(
                    actual_positive
                )
            )
            ** 2,
            "has_alternative_path": bool(
                grp[
                    "has_alternative_path"
                ].iloc[
                    0
                ]
            ),
            "alternative_path_hops": grp[
                "alternative_path_hops"
            ].iloc[
                0
            ],
        }
    )


edge_season_validation = pd.DataFrame(
    edge_season_rows
)


print(
    "\n========================================"
)

print(
    "EDGE-SEASON LATENT ABILITY VALIDATION"
)

print(
    "========================================"
)


edge_season_metrics = pd.DataFrame(
    [
        {
            "edge_seasons": len(
                edge_season_validation
            ),
            "unique_edges": edge_season_validation[
                "_edge_id"
            ].nunique(),
            "pearson_latent_vs_heldout": safe_corr(
                edge_season_validation[
                    "latent_delta_mean"
                ],
                edge_season_validation[
                    "heldout_adjusted_evidence"
                ],
                "pearson",
            ),
            "spearman_latent_vs_heldout": safe_corr(
                edge_season_validation[
                    "latent_delta_mean"
                ],
                edge_season_validation[
                    "heldout_adjusted_evidence"
                ],
                "spearman",
            ),
            "latent_sign_accuracy": float(
                edge_season_validation[
                    "latent_sign_correct"
                ].mean()
            ),
            "latent_brier": float(
                edge_season_validation[
                    "latent_brier"
                ].mean()
            ),
        }
    ]
)


display(
    edge_season_metrics
)


# ==========================================================
# 14. LATENT-ABILITY CALIBRATION
# ==========================================================

latent_calibration = make_calibration_table(
    edge_season_validation,
    confidence_col="latent_confidence",
    correct_col="latent_sign_correct",
    group_name="edge-season latent alpha",
)


print(
    "\n========================================"
)

print(
    "LATENT TEAMMATE-ABILITY CALIBRATION"
)

print(
    "========================================"
)


display(
    latent_calibration
)


plt.figure(
    figsize=(
        7,
        6,
    )
)


g = latent_calibration[
    latent_calibration[
        "n"
    ]
    >
    0
]


plt.plot(
    g[
        "mean_confidence"
    ],
    g[
        "observed_accuracy"
    ],
    marker="o",
    label="edge-season latent alpha",
)


plt.plot(
    [
        0.5,
        1.0,
    ],
    [
        0.5,
        1.0,
    ],
    linestyle="--",
    label="ideal calibration",
)


plt.xlim(
    0.5,
    1.0,
)

plt.ylim(
    0.5,
    1.0,
)

plt.xlabel(
    "Mean posterior confidence in predicted better driver"
)

plt.ylabel(
    "Observed fraction prediction agreed with held-out evidence"
)

plt.title(
    "Leave-edge-out latent teammate-ability calibration"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ==========================================================
# 15. NETWORK-REDUNDANCY STRATIFICATION
# ==========================================================
#
# This is particularly useful for interpreting the teammate graph:
#
#   alternative path = after deleting the direct teammate edge,
#                      another teammate chain still joins the pair.
#
# If performance remains strong in this subset, the network is
# genuinely propagating relative ability through independent paths.
# ==========================================================

network_predictive_rows = []


for has_path, grp in heldout_edge_scores.groupby(
    "has_alternative_path",
    dropna=False,
):

    label = (
        "alternative path exists"
        if bool(
            has_path
        )
        else "no alternative path"
    )


    row = summarise_predictive_rows(
        grp,
        label,
    )

    if row is not None:

        network_predictive_rows.append(
            row
        )


network_predictive_metrics = pd.DataFrame(
    network_predictive_rows
)


print(
    "\n========================================"
)

print(
    "PREDICTIVE PERFORMANCE BY NETWORK REDUNDANCY"
)

print(
    "========================================"
)


display(
    network_predictive_metrics
)


network_latent_metrics = (
    edge_season_validation
    .groupby(
        "has_alternative_path",
        dropna=False,
    )
    .agg(
        edge_seasons=(
            "latent_sign_correct",
            "size",
        ),
        sign_accuracy=(
            "latent_sign_correct",
            "mean",
        ),
        brier=(
            "latent_brier",
            "mean",
        ),
    )
    .reset_index()
)


network_latent_metrics[
    "group"
] = np.where(
    network_latent_metrics[
        "has_alternative_path"
    ],
    "alternative path exists",
    "no alternative path",
)


network_latent_metrics[
    "pearson_latent_vs_heldout"
] = [
    safe_corr(
        edge_season_validation.loc[
            edge_season_validation[
                "has_alternative_path"
            ]
            ==
            value,
            "latent_delta_mean",
        ],
        edge_season_validation.loc[
            edge_season_validation[
                "has_alternative_path"
            ]
            ==
            value,
            "heldout_adjusted_evidence",
        ],
    )
    for value in network_latent_metrics[
        "has_alternative_path"
    ]
]


print(
    "\n========================================"
)

print(
    "LATENT ABILITY BY NETWORK REDUNDANCY"
)

print(
    "========================================"
)


display(
    network_latent_metrics[
        [
            "group",
            "edge_seasons",
            "pearson_latent_vs_heldout",
            "sign_accuracy",
            "brier",
        ]
    ]
)


# ==========================================================
# 16. HARDEST / MOST SURPRISING HELD-OUT EDGE-SEASONS
# ==========================================================

edge_season_validation[
    "absolute_error"
] = np.abs(
    edge_season_validation[
        "latent_delta_mean"
    ]
    -
    edge_season_validation[
        "heldout_adjusted_evidence"
    ]
)


edge_season_validation[
    "surprise_score"
] = np.where(
    edge_season_validation[
        "latent_sign_correct"
    ]
    ==
    1,
    0.0,
    edge_season_validation[
        "latent_confidence"
    ],
)


hardest_edge_seasons = (
    edge_season_validation
    .sort_values(
        [
            "surprise_score",
            "absolute_error",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========================================"
)

print(
    "MOST SURPRISING HELD-OUT EDGE-SEASONS"
)

print(
    "========================================"
)


display(
    hardest_edge_seasons[
        [
            "edge_a",
            "edge_b",
            "year",
            "races",
            "pace_rows",
            "quali_rows",
            "latent_delta_mean",
            "heldout_adjusted_evidence",
            "p_latent_edge_a_better",
            "latent_confidence",
            "latent_sign_correct",
            "has_alternative_path",
            "alternative_path_hops",
        ]
    ]
    .head(
        30
    )
)


# ==========================================================
# 17. SAVE ALL VALIDATION OUTPUTS
# ==========================================================

HELDOUT_ROWS_PATH = (
    EDGE_CV_DIR
    / "heldout_contrast_predictions.csv"
)


EDGE_SEASON_PATH = (
    EDGE_CV_DIR
    / "edge_season_latent_validation.csv"
)


PREDICTIVE_METRICS_PATH = (
    EDGE_CV_DIR
    / "predictive_metrics.csv"
)


PREDICTIVE_CALIBRATION_PATH = (
    EDGE_CV_DIR
    / "predictive_calibration.csv"
)


LATENT_CALIBRATION_PATH = (
    EDGE_CV_DIR
    / "latent_calibration.csv"
)


FOLD_HEALTH_PATH = (
    EDGE_CV_DIR
    / "fold_health.csv"
)


heldout_edge_scores.to_csv(
    HELDOUT_ROWS_PATH,
    index=False,
)


edge_season_validation.to_csv(
    EDGE_SEASON_PATH,
    index=False,
)


predictive_metrics.to_csv(
    PREDICTIVE_METRICS_PATH,
    index=False,
)


predictive_calibration.to_csv(
    PREDICTIVE_CALIBRATION_PATH,
    index=False,
)


latent_calibration.to_csv(
    LATENT_CALIBRATION_PATH,
    index=False,
)


fold_health.to_csv(
    FOLD_HEALTH_PATH,
    index=False,
)


# ==========================================================
# 18. FINAL SUMMARY
# ==========================================================

pooled = predictive_metrics[
    predictive_metrics[
        "group"
    ]
    ==
    "pooled"
].iloc[
    0
]


latent = edge_season_metrics.iloc[
    0
]


print(
    "\n========================================"
)

print(
    "STAGE 22 SUMMARY"
)

print(
    "========================================"
)


print(
    f"Validated modern teammate edges: "
    f"{candidate_edges['_edge_id'].nunique()}"
)


print(
    f"Held-out contrast rows: "
    f"{len(heldout_edge_scores):,}"
)


print(
    f"Pooled held-out correlation: "
    f"{pooled['pearson_y_vs_pred']:.3f}"
)


print(
    f"Pooled held-out sign accuracy: "
    f"{pooled['sign_accuracy']:.3f}"
)


print(
    f"Pooled ΔELPD vs zero-signal null: "
    f"{pooled['delta_elpd']:+.2f}"
)


print(
    f"Edge-season latent-vs-heldout correlation: "
    f"{latent['pearson_latent_vs_heldout']:.3f}"
)


print(
    f"Edge-season latent sign accuracy: "
    f"{latent['latent_sign_accuracy']:.3f}"
)


print(
    "\nInterpretation:"
)


print(
    "  ΔELPD > 0:"
    " unseen teammate evidence is predicted better than a zero-signal null."
)


print(
    "  Calibration near the diagonal:"
    " posterior probabilities have realistic uncertainty."
)


print(
    "  Strong performance when an alternative path exists:"
    " independent teammate-network routes genuinely carry information."
)


print(
    "  Weak calibration / negative ΔELPD:"
    " the latent teammate network is overconfident or not generalising."
)


print(
    "\nSaved validation outputs ->"
)


print(
    EDGE_CV_DIR.resolve()
)


print(
    "\nStage 22 complete."
)


In [ ]:
# ==========================================================
# STAGE 23 — K-FOLD DRIVER-SEASON HOLDOUT VALIDATION
# ==========================================================
#
# Goal:
#   Remove ALL modern likelihood rows that directly contain a selected
#   driver-season state, refit, then ask whether that season's alpha can
#   be reconstructed from the remaining career/network/context.
#
# Modern-only (1996–2025):
#   - remove pace contrasts involving held-out (driver, year)
#   - remove pace team-level rows involving held-out (driver, year)
#   - remove qualifying contrasts involving held-out (driver, year)
#   - remove Q1 team-level rows involving held-out (driver, year)
#
# This is deliberately harsher than Stage 22.
#
# Real data do not reveal "true alpha", so we construct an OUT-OF-SAMPLE
# alpha proxy by algebraically inverting the held-out contrast/level
# equations after the fit. That proxy is noisy; Stage 24 simulation
# recovery will be the genuine truth-known coverage/bias test.
#
# Official production trace is NEVER overwritten.
# ==========================================================

from collections import Counter, defaultdict
from pathlib import Path
import gc, time

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import scipy.stats as st
from scipy.special import logsumexp
import xarray as xr


# ==========================================================
# 0. CONFIG
# ==========================================================

DS_CV_FOLDS = 5
DS_CV_MIN_RACES = 6

# Primary random validation panel.
DS_CV_RANDOM_STATES = 120
DS_CV_MAX_RANDOM_STATES_PER_DRIVER = 3

DS_CV_DRAWS = 300
DS_CV_TUNE = 300
DS_CV_CHAINS = 2
DS_CV_TARGET_ACCEPT = 0.92
DS_CV_SEED = 23042

CHECKPOINT_ENGINE_DS_CV = "h5netcdf"

DS_CV_DIR = Path(CFG.output_dir) / "validation_driver_season_holdout"
DS_CV_DIR.mkdir(parents=True, exist_ok=True)

# Extra case studies. These are excluded from the PRIMARY random-only
# metrics so they do not bias the validation sample.
#
# Schumacher 1995 is historical, so it is not eligible for this
# modern-only Stage 23.
FORCED_DRIVER_SEASONS = [
    ("michael_schumacher", 1996),
    ("irvine", 1996),
    ("barrichello", 2000),
    ("herbert", 2000),
    ("alonso", 2012),
    ("hamilton", 2012),
    ("vettel", 2013),
    ("ricciardo", 2014),
    ("max_verstappen", 2025),
    ("norris", 2025),
]


# ==========================================================
# 1. REQUIRE OBJECTS
# ==========================================================

required = [
    "build_joint_model_1982_2025",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "hist_q_pack",
    "hist_race_pack",
    "idx_all",
    "CFG",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError("Missing required objects: " + ", ".join(missing))

if "posterior_final" not in globals() and "final_trace_2025" not in globals():
    raise RuntimeError(
        "Need the accepted final posterior in posterior_final or final_trace_2025."
    )


# ==========================================================
# 2. HELPERS
# ==========================================================

def state_key(driver, year):
    return f"{str(driver)}|||{int(year)}"

def split_state_key(key):
    d, y = key.rsplit("|||", 1)
    return d, int(y)

def add_state_keys(df):
    out = df.copy()
    out["_state_a"] = [
        state_key(d, y)
        for d, y in zip(out["driver_a"].astype(str), out["year"].astype(int))
    ]
    out["_state_b"] = [
        state_key(d, y)
        for d, y in zip(out["driver_b"].astype(str), out["year"].astype(int))
    ]
    return out

def stack_state_draws(posterior, var_name):
    da = posterior[var_name].stack(sample=("chain", "draw"))
    other_dims = [d for d in da.dims if d != "sample"]
    x = da.transpose("sample", *other_dims).values
    return np.asarray(x, dtype=float).reshape(x.shape[0], -1)

def stack_scalar_draws(posterior, var_name):
    return np.asarray(posterior[var_name].values, dtype=float).reshape(-1)

def safe_corr(x, y, method="pearson"):
    x = pd.Series(np.asarray(x, dtype=float))
    y = pd.Series(np.asarray(y, dtype=float))
    ok = x.notna() & y.notna()
    if ok.sum() < 3 or x[ok].nunique() < 2 or y[ok].nunique() < 2:
        return np.nan
    return float(x[ok].corr(y[ok], method=method))

def weighted_average(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not ok.any():
        return np.nan
    return float(np.average(x[ok], weights=w[ok]))

def weighted_rmse(obs, pred, w):
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(obs) & np.isfinite(pred) & np.isfinite(w) & (w > 0)
    if not ok.any():
        return np.nan
    return float(np.sqrt(np.average((obs[ok] - pred[ok])**2, weights=w[ok])))

def involvement_flags(df, heldout_keys):
    a = df["_state_a"].isin(heldout_keys).to_numpy()
    b = df["_state_b"].isin(heldout_keys).to_numpy()
    return a, b


# ==========================================================
# 3. ADD STATE KEYS TO MODERN TABLES
# ==========================================================

pace_contrast_cv = add_state_keys(pace_contrast_all)
pace_level_cv = add_state_keys(pace_level_all)
q_contrast_cv = add_state_keys(q_contrast_all)
q_level_cv = add_state_keys(q_level_all_model)


# ==========================================================
# 4. BUILD DRIVER-SEASON CANDIDATES
# ==========================================================

def endpoint_rows(df, channel):
    a = pd.DataFrame({
        "state_key": df["_state_a"].to_numpy(),
        "driver": df["driver_a"].astype(str).to_numpy(),
        "year": df["year"].astype(int).to_numpy(),
        "raceId": df["raceId"].astype(int).to_numpy(),
        "channel": channel,
    })
    b = pd.DataFrame({
        "state_key": df["_state_b"].to_numpy(),
        "driver": df["driver_b"].astype(str).to_numpy(),
        "year": df["year"].astype(int).to_numpy(),
        "raceId": df["raceId"].astype(int).to_numpy(),
        "channel": channel,
    })
    return pd.concat([a, b], ignore_index=True)

endpoint_meta = pd.concat(
    [
        endpoint_rows(pace_contrast_cv, "pace"),
        endpoint_rows(q_contrast_cv, "quali"),
    ],
    ignore_index=True,
)

candidate_states = (
    endpoint_meta
    .groupby(["state_key", "driver", "year"], as_index=False)
    .agg(
        contrast_rows=("channel", "size"),
        n_races=("raceId", "nunique"),
    )
)

for channel in ["pace", "quali"]:
    tmp = (
        endpoint_meta[endpoint_meta["channel"] == channel]
        .groupby("state_key")
        .agg(
            **{
                f"{channel}_rows": ("channel", "size"),
                f"{channel}_races": ("raceId", "nunique"),
            }
        )
    )
    candidate_states = candidate_states.merge(
        tmp, left_on="state_key", right_index=True, how="left"
    )

for col in ["pace_rows", "pace_races", "quali_rows", "quali_races"]:
    candidate_states[col] = candidate_states[col].fillna(0).astype(int)

# Career-state support.
years_by_driver = defaultdict(list)
for d, y in idx_all.active_driver_years:
    years_by_driver[str(d)].append(int(y))
for d in years_by_driver:
    years_by_driver[d] = sorted(set(years_by_driver[d]))

def support_info(driver, year):
    years = years_by_driver.get(str(driver), [])
    prevs = [y for y in years if y < year]
    nexts = [y for y in years if y > year]

    if prevs and nexts:
        support = "both sides"
    elif prevs or nexts:
        support = "one side"
    else:
        support = "no other season"

    return {
        "modelled_career_seasons": len(years),
        "other_modelled_seasons": max(0, len(years) - 1),
        "previous_modelled_year": max(prevs) if prevs else np.nan,
        "next_modelled_year": min(nexts) if nexts else np.nan,
        "temporal_support": support,
    }

support = [
    support_info(d, int(y))
    for d, y in zip(candidate_states["driver"], candidate_states["year"])
]

for col in support[0]:
    candidate_states[col] = [x[col] for x in support]

candidate_states["era"] = pd.cut(
    candidate_states["year"],
    bins=[1995, 2005, 2015, 2025],
    labels=["1996–2005", "2006–2015", "2016–2025"],
    include_lowest=True,
)

candidate_states = (
    candidate_states[candidate_states["n_races"] >= DS_CV_MIN_RACES]
    .copy()
    .reset_index(drop=True)
)

print("========================================")
print("STAGE 23 — DRIVER-SEASON HOLDOUT CV")
print("========================================")
print("Candidate modern driver-seasons:", len(candidate_states))
print("Minimum direct contrast races:", DS_CV_MIN_RACES)
print("Requested random states:", DS_CV_RANDOM_STATES)
print("Folds:", DS_CV_FOLDS)
print(
    "Validation sampler:",
    f"{DS_CV_CHAINS} chains × ({DS_CV_TUNE} tune + {DS_CV_DRAWS} draws)",
)


# ==========================================================
# 5. RANDOM PANEL + FORCED CASES
# ==========================================================

rng = np.random.default_rng(DS_CV_SEED)

forced_requested = {
    state_key(d, y)
    for d, y in FORCED_DRIVER_SEASONS
}

forced_states = candidate_states[
    candidate_states["state_key"].isin(forced_requested)
].copy()
forced_states["selection_source"] = "forced case"

missing_forced = forced_requested - set(forced_states["state_key"])
if missing_forced:
    print("\nForced cases unavailable under the candidate rules:")
    for x in sorted(missing_forced):
        print(" ", x)

random_pool = candidate_states[
    ~candidate_states["state_key"].isin(forced_states["state_key"])
].copy()

random_pool["_random_priority"] = rng.random(len(random_pool))

era_names = ["1996–2005", "2006–2015", "2016–2025"]
base = DS_CV_RANDOM_STATES // len(era_names)
rem = DS_CV_RANDOM_STATES % len(era_names)
quota = {
    era: base + (1 if i < rem else 0)
    for i, era in enumerate(era_names)
}

selected_rows = []
driver_counts = Counter()

for era in era_names:
    grp = (
        random_pool[random_pool["era"].astype(str) == era]
        .sort_values("_random_priority")
    )
    taken = 0
    for _, row in grp.iterrows():
        d = str(row["driver"])
        if driver_counts[d] >= DS_CV_MAX_RANDOM_STATES_PER_DRIVER:
            continue
        selected_rows.append(row)
        driver_counts[d] += 1
        taken += 1
        if taken >= quota[era]:
            break

random_states = pd.DataFrame(selected_rows)

# Backfill if the cap/era split leaves us short.
if len(random_states) < DS_CV_RANDOM_STATES:
    selected_keys = set(random_states["state_key"])
    remaining = random_pool[
        ~random_pool["state_key"].isin(selected_keys)
    ].sort_values("_random_priority")

    for _, row in remaining.iterrows():
        if len(random_states) >= DS_CV_RANDOM_STATES:
            break
        d = str(row["driver"])
        if driver_counts[d] >= DS_CV_MAX_RANDOM_STATES_PER_DRIVER:
            continue
        random_states = pd.concat(
            [random_states, row.to_frame().T],
            ignore_index=True,
        )
        driver_counts[d] += 1

random_states["selection_source"] = "random"

validation_states = (
    pd.concat([random_states, forced_states], ignore_index=True)
    .drop_duplicates("state_key")
    .reset_index(drop=True)
)

print("\nRandom states:", (validation_states["selection_source"] == "random").sum())
print("Forced states:", (validation_states["selection_source"] == "forced case").sum())
print("Total validation states:", len(validation_states))
print("\nRandom panel by era:")
print(
    validation_states.loc[
        validation_states["selection_source"] == "random", "era"
    ].value_counts().sort_index()
)


# ==========================================================
# 6. BALANCED FOLD ASSIGNMENT
# ==========================================================
#
# Avoid, where possible:
#   - two direct teammate states in the same fold;
#   - two seasons from the same driver in the same fold.
# ==========================================================

selected_keys = set(validation_states["state_key"])
conflicts = defaultdict(set)

for df in [pace_contrast_cv, q_contrast_cv]:
    rel = df[
        df["_state_a"].isin(selected_keys)
        & df["_state_b"].isin(selected_keys)
    ]
    for a, b in zip(rel["_state_a"], rel["_state_b"]):
        if a == b:
            continue
        conflicts[a].add(b)
        conflicts[b].add(a)

assignment = validation_states.copy()
assignment["_tie"] = rng.random(len(assignment))
assignment = assignment.sort_values(
    ["contrast_rows", "_tie"],
    ascending=[False, True],
)

fold_load = np.zeros(DS_CV_FOLDS, dtype=int)
fold_states = {f: set() for f in range(DS_CV_FOLDS)}
fold_drivers = {f: set() for f in range(DS_CV_FOLDS)}
state_to_fold = {}

for _, row in assignment.iterrows():
    key = row["state_key"]
    driver = str(row["driver"])

    valid_folds = []
    for f in np.argsort(fold_load):
        f = int(f)
        same_driver = driver in fold_drivers[f]
        teammate_conflict = any(x in fold_states[f] for x in conflicts.get(key, set()))
        if not same_driver and not teammate_conflict:
            valid_folds.append(f)

    if valid_folds:
        f = valid_folds[0]
    else:
        # Rare fallback: minimum conflicts, then minimum load.
        scores = []
        for ff in range(DS_CV_FOLDS):
            nconf = int(driver in fold_drivers[ff])
            nconf += sum(x in fold_states[ff] for x in conflicts.get(key, set()))
            scores.append((nconf, fold_load[ff], ff))
        f = min(scores)[2]

    state_to_fold[key] = f
    fold_states[f].add(key)
    fold_drivers[f].add(driver)
    fold_load[f] += int(row["contrast_rows"])

validation_states["fold"] = (
    validation_states["state_key"].map(state_to_fold).astype(int)
)

assignment_path = DS_CV_DIR / (
    f"driver_season_assignments_{DS_CV_FOLDS}fold_"
    f"random{DS_CV_RANDOM_STATES}_min{DS_CV_MIN_RACES}races.csv"
)
validation_states.to_csv(assignment_path, index=False)

print("\nFold loads:")
for f in range(DS_CV_FOLDS):
    x = validation_states[validation_states["fold"] == f]
    print(
        f"  fold {f+1}: {len(x)} states, "
        f"{int(x['contrast_rows'].sum())} endpoint contrast rows"
    )


# ==========================================================
# 7. OFFICIAL FULL-DATA ALPHA
# ==========================================================

if "posterior_final" in globals():
    full_post = posterior_final
else:
    full_post = final_trace_2025["posterior"].to_dataset()

full_alpha = stack_state_draws(full_post, "alpha_dy")
full_alpha_mean = full_alpha.mean(axis=0)
full_alpha_sd = full_alpha.std(axis=0, ddof=1)
full_alpha_lb = np.quantile(full_alpha, 0.055, axis=0)
full_alpha_ub = np.quantile(full_alpha, 0.945, axis=0)


# ==========================================================
# 8. SCORING: CONTRAST ROWS
# ==========================================================

def score_contrasts(heldout, posterior, heldout_keys, channel, sigma_var, fold):
    if heldout.empty:
        return pd.DataFrame()

    alpha = stack_state_draws(posterior, "alpha_dy")
    gamma = stack_scalar_draws(posterior, "gamma_tenure")
    sigma = stack_scalar_draws(posterior, sigma_var)

    aheld, bheld = involvement_flags(heldout, heldout_keys)
    valid = np.logical_xor(aheld, bheld)
    x = heldout.loc[valid].copy()
    if x.empty:
        return pd.DataFrame()

    focal_is_a = aheld[valid]
    sign = np.where(focal_is_a, 1.0, -1.0)

    dy_a = x["dy_a"].to_numpy(int)
    dy_b = x["dy_b"].to_numpy(int)
    focal_idx = np.where(focal_is_a, dy_a, dy_b)
    tm_idx = np.where(focal_is_a, dy_b, dy_a)

    focal_driver = np.where(
        focal_is_a,
        x["driver_a"].astype(str).to_numpy(),
        x["driver_b"].astype(str).to_numpy(),
    )
    teammate = np.where(
        focal_is_a,
        x["driver_b"].astype(str).to_numpy(),
        x["driver_a"].astype(str).to_numpy(),
    )
    focal_state = np.where(
        focal_is_a,
        x["_state_a"].to_numpy(),
        x["_state_b"].to_numpy(),
    )

    y = sign * x["y"].to_numpy(float)
    tenure = sign * x["tenure_diff"].to_numpy(float)
    weight = x["weight"].to_numpy(float)

    focal_a = alpha[:, focal_idx]
    tm_a = alpha[:, tm_idx]

    mu = (
        focal_a
        - tm_a
        + gamma[:, None] * tenure[None, :]
    )

    scale = sigma[:, None] / np.sqrt(weight[None, :])

    lp_model = st.t.logpdf(
        y[None, :],
        df=CFG.student_t_nu,
        loc=mu,
        scale=scale,
    )
    lp_null = st.t.logpdf(
        y[None, :],
        df=CFG.student_t_nu,
        loc=0.0,
        scale=scale,
    )

    elpd_model = logsumexp(lp_model, axis=0) - np.log(lp_model.shape[0])
    elpd_null = logsumexp(lp_null, axis=0) - np.log(lp_null.shape[0])

    p_y_pos = np.mean(
        st.t.sf((0.0 - mu) / scale, df=CFG.student_t_nu),
        axis=0,
    )

    focal_mean = focal_a.mean(axis=0)
    focal_sd = focal_a.std(axis=0, ddof=1)
    focal_lb = np.quantile(focal_a, 0.055, axis=0)
    focal_ub = np.quantile(focal_a, 0.945, axis=0)
    p_alpha_gt0 = np.mean(focal_a > 0.0, axis=0)

    gamma_mean = float(gamma.mean())
    sigma_mean = float(sigma.mean())
    tm_mean = tm_a.mean(axis=0)

    # y = alpha_focal - alpha_tm + gamma*tenure + noise
    implied_alpha = y - gamma_mean * tenure + tm_mean

    return pd.DataFrame({
        "fold": fold,
        "channel": channel,
        "observation_type": "contrast",
        "focal_state": focal_state,
        "focal_driver": focal_driver,
        "year": x["year"].astype(int).to_numpy(),
        "raceId": x["raceId"].astype(int).to_numpy(),
        "constructorId": x["constructorId"].astype(int).to_numpy(),
        "teammate": teammate,
        "weight": weight,
        "observed_y": y,
        "predicted_y": mu.mean(axis=0),
        "elpd_model": elpd_model,
        "elpd_null": elpd_null,
        "delta_elpd": elpd_model - elpd_null,
        "p_predictive_positive": p_y_pos,
        "predictive_confidence": np.maximum(p_y_pos, 1.0 - p_y_pos),
        "predictive_correct": (
            (p_y_pos >= 0.5) == (y > 0.0)
        ).astype(int),
        "focal_alpha_mean": focal_mean,
        "focal_alpha_sd": focal_sd,
        "focal_alpha_lb": focal_lb,
        "focal_alpha_ub": focal_ub,
        "p_alpha_gt_zero": p_alpha_gt0,
        "teammate_alpha_mean": tm_mean,
        "implied_alpha_proxy": implied_alpha,
        "effective_precision": weight / (sigma_mean**2),
    })


# ==========================================================
# 9. SCORING: TEAM-LEVEL ROWS
# ==========================================================

def score_levels(heldout, posterior, heldout_keys, channel, sigma_var, fold):
    if heldout.empty:
        return pd.DataFrame()

    alpha = stack_state_draws(posterior, "alpha_dy")
    car = stack_state_draws(posterior, "car_csy")
    gamma = stack_scalar_draws(posterior, "gamma_tenure")
    sigma = stack_scalar_draws(posterior, sigma_var)

    aheld, bheld = involvement_flags(heldout, heldout_keys)
    valid = np.logical_xor(aheld, bheld)
    x = heldout.loc[valid].copy()
    if x.empty:
        return pd.DataFrame()

    focal_is_a = aheld[valid]

    dy_a = x["dy_a"].to_numpy(int)
    dy_b = x["dy_b"].to_numpy(int)
    focal_idx = np.where(focal_is_a, dy_a, dy_b)
    tm_idx = np.where(focal_is_a, dy_b, dy_a)

    focal_driver = np.where(
        focal_is_a,
        x["driver_a"].astype(str).to_numpy(),
        x["driver_b"].astype(str).to_numpy(),
    )
    teammate = np.where(
        focal_is_a,
        x["driver_b"].astype(str).to_numpy(),
        x["driver_a"].astype(str).to_numpy(),
    )
    focal_state = np.where(
        focal_is_a,
        x["_state_a"].to_numpy(),
        x["_state_b"].to_numpy(),
    )

    y = x["y"].to_numpy(float)
    weight = x["weight"].to_numpy(float)
    tenure = x["tenure_mean"].to_numpy(float)
    csy = x["csy"].to_numpy(int)

    focal_a = alpha[:, focal_idx]
    tm_a = alpha[:, tm_idx]
    car_eff = car[:, csy]

    mu = (
        0.5 * (focal_a + tm_a)
        + car_eff
        + gamma[:, None] * tenure[None, :]
    )

    scale = sigma[:, None] / np.sqrt(weight[None, :])

    lp_model = st.t.logpdf(
        y[None, :],
        df=CFG.student_t_nu,
        loc=mu,
        scale=scale,
    )
    lp_null = st.t.logpdf(
        y[None, :],
        df=CFG.student_t_nu,
        loc=0.0,
        scale=scale,
    )

    elpd_model = logsumexp(lp_model, axis=0) - np.log(lp_model.shape[0])
    elpd_null = logsumexp(lp_null, axis=0) - np.log(lp_null.shape[0])

    focal_mean = focal_a.mean(axis=0)
    focal_sd = focal_a.std(axis=0, ddof=1)
    focal_lb = np.quantile(focal_a, 0.055, axis=0)
    focal_ub = np.quantile(focal_a, 0.945, axis=0)
    p_alpha_gt0 = np.mean(focal_a > 0.0, axis=0)

    tm_mean = tm_a.mean(axis=0)
    car_mean = car_eff.mean(axis=0)
    gamma_mean = float(gamma.mean())
    sigma_mean = float(sigma.mean())

    # y = .5(alpha_focal + alpha_tm) + car + gamma*tenure + noise
    implied_alpha = (
        2.0 * (y - car_mean - gamma_mean * tenure)
        - tm_mean
    )

    # Inverting the .5 coefficient doubles the observation noise.
    eff_precision = weight / (4.0 * sigma_mean**2)

    return pd.DataFrame({
        "fold": fold,
        "channel": channel,
        "observation_type": "level",
        "focal_state": focal_state,
        "focal_driver": focal_driver,
        "year": x["year"].astype(int).to_numpy(),
        "raceId": x["raceId"].astype(int).to_numpy(),
        "constructorId": x["constructorId"].astype(int).to_numpy(),
        "teammate": teammate,
        "weight": weight,
        "observed_y": y,
        "predicted_y": mu.mean(axis=0),
        "elpd_model": elpd_model,
        "elpd_null": elpd_null,
        "delta_elpd": elpd_model - elpd_null,
        "p_predictive_positive": np.nan,
        "predictive_confidence": np.nan,
        "predictive_correct": np.nan,
        "focal_alpha_mean": focal_mean,
        "focal_alpha_sd": focal_sd,
        "focal_alpha_lb": focal_lb,
        "focal_alpha_ub": focal_ub,
        "p_alpha_gt_zero": p_alpha_gt0,
        "teammate_alpha_mean": tm_mean,
        "implied_alpha_proxy": implied_alpha,
        "effective_precision": eff_precision,
    })


# ==========================================================
# 10. RUN / LOAD FIVE REFITS
# ==========================================================

score_frames = []
health_rows = []

GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]

helper_cols = ["_state_a", "_state_b"]

for fold in range(DS_CV_FOLDS):

    print("\n========================================")
    print(f"DRIVER-SEASON CV FOLD {fold+1}/{DS_CV_FOLDS}")
    print("========================================")

    fold_meta = validation_states[validation_states["fold"] == fold]
    held_keys = set(fold_meta["state_key"])

    pc_a, pc_b = involvement_flags(pace_contrast_cv, held_keys)
    pl_a, pl_b = involvement_flags(pace_level_cv, held_keys)
    qc_a, qc_b = involvement_flags(q_contrast_cv, held_keys)
    ql_a, ql_b = involvement_flags(q_level_cv, held_keys)

    pc_mask = pc_a | pc_b
    pl_mask = pl_a | pl_b
    qc_mask = qc_a | qc_b
    ql_mask = ql_a | ql_b

    hold_pc = pace_contrast_cv.loc[pc_mask].copy()
    hold_pl = pace_level_cv.loc[pl_mask].copy()
    hold_qc = q_contrast_cv.loc[qc_mask].copy()
    hold_ql = q_level_cv.loc[ql_mask].copy()

    train_pc = pace_contrast_cv.loc[~pc_mask].copy()
    train_pl = pace_level_cv.loc[~pl_mask].copy()
    train_qc = q_contrast_cv.loc[~qc_mask].copy()
    train_ql = q_level_cv.loc[~ql_mask].copy()

    double_counts = {
        "pace_contrast": int(np.sum(pc_a & pc_b)),
        "pace_level": int(np.sum(pl_a & pl_b)),
        "quali_contrast": int(np.sum(qc_a & qc_b)),
        "q1_level": int(np.sum(ql_a & ql_b)),
    }

    print("Held-out driver-seasons:", len(fold_meta))
    print("Pace contrast:", f"{len(train_pc):,} train / {len(hold_pc):,} test")
    print("Pace level:", f"{len(train_pl):,} train / {len(hold_pl):,} test")
    print("Quali contrast:", f"{len(train_qc):,} train / {len(hold_qc):,} test")
    print("Q1 level:", f"{len(train_ql):,} train / {len(hold_ql):,} test")
    print("Rows with BOTH endpoints held out:", double_counts)

    model = build_joint_model_1982_2025(
        train_pc.drop(columns=helper_cols, errors="ignore"),
        train_pl.drop(columns=helper_cols, errors="ignore"),
        train_qc.drop(columns=helper_cols, errors="ignore"),
        train_ql.drop(columns=helper_cols, errors="ignore"),
        hist_q_pack,
        hist_race_pack,
        idx_all,
        cfg=CFG,
    )

    trace_path = DS_CV_DIR / (
        f"driver_season_cv_{DS_CV_FOLDS}fold_"
        f"random{DS_CV_RANDOM_STATES}_min{DS_CV_MIN_RACES}races_"
        f"fold{fold+1}_{DS_CV_DRAWS}draw_{DS_CV_CHAINS}chain.nc"
    )

    if trace_path.exists():
        print("Loading checkpoint ->", trace_path)
        trace = xr.open_datatree(
            trace_path,
            engine=CHECKPOINT_ENGINE_DS_CV,
        )
    else:
        print("Sampling fold...")
        t0 = time.perf_counter()

        with model:
            trace = pm.sample(
                draws=DS_CV_DRAWS,
                tune=DS_CV_TUNE,
                chains=DS_CV_CHAINS,
                random_seed=DS_CV_SEED + fold,
                target_accept=DS_CV_TARGET_ACCEPT,
                nuts_sampler="nutpie",
                nuts_sampler_kwargs={"backend": "jax"},
                return_inferencedata=True,
                progressbar=True,
            )

        print(f"Fold wall time: {(time.perf_counter()-t0)/60:.2f} min")

        trace.to_netcdf(
            trace_path,
            engine=CHECKPOINT_ENGINE_DS_CV,
            mode="w",
        )
        print("Saved ->", trace_path)

    post = trace["posterior"].to_dataset()
    sample_stats = trace["sample_stats"].to_dataset()

    diag = az.summary(
        post,
        var_names=GLOBAL_VARS,
        kind="diagnostics",
    )

    max_rhat = float(diag["r_hat"].max())
    min_ess = float(diag["ess_bulk"].min())
    div = (
        int(np.asarray(sample_stats["diverging"].values).sum())
        if "diverging" in sample_stats
        else 0
    )

    health_rows.append({
        "fold": fold + 1,
        "heldout_states": len(fold_meta),
        "heldout_pace_contrast": len(hold_pc),
        "heldout_pace_level": len(hold_pl),
        "heldout_quali_contrast": len(hold_qc),
        "heldout_q1_level": len(hold_ql),
        "double_heldout_pace_contrast": double_counts["pace_contrast"],
        "double_heldout_quali_contrast": double_counts["quali_contrast"],
        "max_global_rhat": max_rhat,
        "min_global_ess_bulk": min_ess,
        "divergences": div,
    })

    print(
        "Validation-fit health:",
        f"max global Rhat={max_rhat:.4f},",
        f"min global ESS={min_ess:.1f},",
        f"divergences={div}",
    )

    score_frames += [
        score_contrasts(
            hold_pc, post, held_keys,
            "pace_contrast", "sigma_pace_contrast", fold + 1,
        ),
        score_levels(
            hold_pl, post, held_keys,
            "pace_level", "sigma_pace_level", fold + 1,
        ),
        score_contrasts(
            hold_qc, post, held_keys,
            "quali_contrast", "sigma_q_contrast", fold + 1,
        ),
        score_levels(
            hold_ql, post, held_keys,
            "q1_level", "sigma_q_level", fold + 1,
        ),
    ]

    try:
        trace.close()
    except Exception:
        pass

    del model, trace, post, sample_stats
    gc.collect()


# ==========================================================
# 11. COLLECT ROW PREDICTIONS
# ==========================================================

score_frames = [x for x in score_frames if x is not None and not x.empty]
heldout_rows = pd.concat(score_frames, ignore_index=True)
fold_health = pd.DataFrame(health_rows)

meta_cols = validation_states[
    [
        "state_key",
        "selection_source",
        "n_races",
        "contrast_rows",
        "pace_races",
        "quali_races",
        "modelled_career_seasons",
        "other_modelled_seasons",
        "previous_modelled_year",
        "next_modelled_year",
        "temporal_support",
        "era",
    ]
].rename(columns={"state_key": "focal_state"})

heldout_rows = heldout_rows.merge(
    meta_cols,
    on="focal_state",
    how="left",
    validate="many_to_one",
)

print("\n========================================")
print("VALIDATION FIT HEALTH")
print("========================================")
display(fold_health)


# ==========================================================
# 12. RANDOM-PANEL ROW-LEVEL PREDICTIVE METRICS
# ==========================================================

def summarise_rows(df, label):
    if df.empty:
        return None

    obs = df["observed_y"].to_numpy(float)
    pred = df["predicted_y"].to_numpy(float)
    w = df["weight"].to_numpy(float)

    out = {
        "group": label,
        "rows": len(df),
        "driver_seasons": df["focal_state"].nunique(),
        "pearson": safe_corr(obs, pred),
        "spearman": safe_corr(obs, pred, "spearman"),
        "rmse": float(np.sqrt(np.mean((obs - pred)**2))),
        "weighted_rmse": weighted_rmse(obs, pred, w),
        "model_elpd": float(df["elpd_model"].sum()),
        "null_elpd": float(df["elpd_null"].sum()),
        "delta_elpd": float(df["delta_elpd"].sum()),
        "mean_delta_elpd": float(df["delta_elpd"].mean()),
    }

    contrast = df[df["observation_type"] == "contrast"]
    out["sign_accuracy"] = (
        float(contrast["predictive_correct"].mean())
        if len(contrast) else np.nan
    )
    return out

random_rows = heldout_rows[
    heldout_rows["selection_source"] == "random"
].copy()

row_metric_rows = []
for channel in [
    "pace_contrast",
    "pace_level",
    "quali_contrast",
    "q1_level",
]:
    row = summarise_rows(
        random_rows[random_rows["channel"] == channel],
        channel,
    )
    if row is not None:
        row_metric_rows.append(row)

row_metric_rows.append(
    summarise_rows(random_rows, "all held-out modern rows")
)

row_predictive_metrics = pd.DataFrame(row_metric_rows)

print("\n========================================")
print("RANDOM-PANEL HELD-OUT PREDICTIVE METRICS")
print("========================================")
display(row_predictive_metrics)


# ==========================================================
# 13. ONE HELD-OUT ALPHA PROXY PER DRIVER-SEASON
# ==========================================================

state_rows = []

for state, grp in heldout_rows.groupby("focal_state"):

    precision = grp["effective_precision"].to_numpy(float)
    proxy = weighted_average(grp["implied_alpha_proxy"], precision)

    # Same focal posterior summary across every row for this state/fold.
    hold_mean = float(grp["focal_alpha_mean"].iloc[0])
    hold_sd = float(grp["focal_alpha_sd"].iloc[0])
    hold_lb = float(grp["focal_alpha_lb"].iloc[0])
    hold_ub = float(grp["focal_alpha_ub"].iloc[0])
    p_gt0 = float(grp["p_alpha_gt_zero"].iloc[0])

    driver, year = split_state_key(state)
    idx = idx_all.dy_map[(driver, year)]

    fmean = float(full_alpha_mean[idx])
    fsd = float(full_alpha_sd[idx])
    flb = float(full_alpha_lb[idx])
    fub = float(full_alpha_ub[idx])

    observed_positive = proxy > 0.0
    predicted_positive = p_gt0 >= 0.5

    shift = hold_mean - fmean
    pooled_sd = np.sqrt(hold_sd**2 + fsd**2)

    state_rows.append({
        "state_key": state,
        "driver": driver,
        "year": year,
        "fold": int(grp["fold"].iloc[0]),
        "selection_source": grp["selection_source"].iloc[0],
        "era": str(grp["era"].iloc[0]),
        "n_races": int(grp["n_races"].iloc[0]),
        "heldout_proxy_rows": len(grp),
        "heldout_contrast_rows": int((grp["observation_type"] == "contrast").sum()),
        "heldout_level_rows": int((grp["observation_type"] == "level").sum()),
        "other_modelled_seasons": int(grp["other_modelled_seasons"].iloc[0]),
        "temporal_support": grp["temporal_support"].iloc[0],
        "holdout_alpha_mean": hold_mean,
        "holdout_alpha_sd": hold_sd,
        "holdout_alpha_lb": hold_lb,
        "holdout_alpha_ub": hold_ub,
        "p_holdout_alpha_gt_zero": p_gt0,
        "heldout_alpha_proxy": proxy,
        "proxy_sign_correct": int(predicted_positive == observed_positive),
        "proxy_brier": (p_gt0 - float(observed_positive))**2,
        # NOT literal coverage: proxy is noisy.
        "proxy_inside_holdout_89": int(hold_lb <= proxy <= hold_ub),
        "full_alpha_mean": fmean,
        "full_alpha_sd": fsd,
        "full_alpha_lb": flb,
        "full_alpha_ub": fub,
        "holdout_minus_full": shift,
        "absolute_holdout_full_shift": abs(shift),
        "holdout_full_shift_pooled_sd": shift / pooled_sd if pooled_sd > 0 else np.nan,
        "full_mean_inside_holdout_89": int(hold_lb <= fmean <= hold_ub),
    })

driver_season_validation = pd.DataFrame(state_rows)

driver_season_validation["evidence_band"] = pd.cut(
    driver_season_validation["n_races"],
    bins=[0, 9, 14, np.inf],
    labels=["6–9 races", "10–14 races", "15+ races"],
)

driver_season_validation["career_support_band"] = pd.cut(
    driver_season_validation["other_modelled_seasons"],
    bins=[-1, 1, 4, np.inf],
    labels=["0–1 other seasons", "2–4 other seasons", "5+ other seasons"],
)


# ==========================================================
# 14. DRIVER-SEASON RECONSTRUCTION METRICS
# ==========================================================

def summarise_states(df, label):
    if len(df) < 2:
        return None

    return {
        "group": label,
        "driver_seasons": len(df),
        "pearson_holdout_vs_proxy": safe_corr(
            df["holdout_alpha_mean"], df["heldout_alpha_proxy"]
        ),
        "spearman_holdout_vs_proxy": safe_corr(
            df["holdout_alpha_mean"], df["heldout_alpha_proxy"], "spearman"
        ),
        "rmse_holdout_vs_proxy": float(
            np.sqrt(np.mean(
                (df["holdout_alpha_mean"] - df["heldout_alpha_proxy"])**2
            ))
        ),
        "mean_bias_holdout_minus_proxy": float(
            (df["holdout_alpha_mean"] - df["heldout_alpha_proxy"]).mean()
        ),
        "above_average_sign_accuracy": float(df["proxy_sign_correct"].mean()),
        "proxy_brier": float(df["proxy_brier"].mean()),
        "proxy_inside_holdout_89": float(df["proxy_inside_holdout_89"].mean()),
        "pearson_holdout_vs_full": safe_corr(
            df["holdout_alpha_mean"], df["full_alpha_mean"]
        ),
        "spearman_holdout_vs_full": safe_corr(
            df["holdout_alpha_mean"], df["full_alpha_mean"], "spearman"
        ),
        "rmse_holdout_vs_full": float(
            np.sqrt(np.mean(df["holdout_minus_full"]**2))
        ),
        "median_abs_holdout_full_shift": float(
            df["absolute_holdout_full_shift"].median()
        ),
        "full_mean_inside_holdout_89": float(
            df["full_mean_inside_holdout_89"].mean()
        ),
    }

random_states_result = driver_season_validation[
    driver_season_validation["selection_source"] == "random"
].copy()

primary_state_metrics = pd.DataFrame([
    summarise_states(random_states_result, "random validation panel"),
    summarise_states(driver_season_validation, "random + forced cases"),
])

print("\n========================================")
print("DRIVER-SEASON ALPHA RECONSTRUCTION")
print("========================================")
display(primary_state_metrics)


# ==========================================================
# 15. SCATTER: HOLDOUT ALPHA VS HELD-OUT EVIDENCE
# ==========================================================

plt.figure(figsize=(7, 6))
plt.scatter(
    random_states_result["heldout_alpha_proxy"],
    random_states_result["holdout_alpha_mean"],
    alpha=0.70,
)

lims = [
    min(
        random_states_result["heldout_alpha_proxy"].min(),
        random_states_result["holdout_alpha_mean"].min(),
    ),
    max(
        random_states_result["heldout_alpha_proxy"].max(),
        random_states_result["holdout_alpha_mean"].max(),
    ),
]

plt.plot(lims, lims, linestyle="--")
plt.xlabel("Held-out evidence alpha proxy")
plt.ylabel("Reconstructed posterior mean alpha")
plt.title("Driver-season holdout reconstruction vs unseen evidence")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# ==========================================================
# 16. SCATTER: HOLDOUT ALPHA VS OFFICIAL FULL-DATA ALPHA
# ==========================================================

plt.figure(figsize=(7, 6))
plt.scatter(
    random_states_result["full_alpha_mean"],
    random_states_result["holdout_alpha_mean"],
    alpha=0.70,
)

lims = [
    min(
        random_states_result["full_alpha_mean"].min(),
        random_states_result["holdout_alpha_mean"].min(),
    ),
    max(
        random_states_result["full_alpha_mean"].max(),
        random_states_result["holdout_alpha_mean"].max(),
    ),
]

plt.plot(lims, lims, linestyle="--")
plt.xlabel("Official full-data posterior mean alpha")
plt.ylabel("Holdout reconstructed posterior mean alpha")
plt.title("Driver-season score stability after removing direct evidence")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# ==========================================================
# 17. ABOVE-AVERAGE ALPHA PROXY CALIBRATION
# ==========================================================
#
# This is NOT a literal posterior coverage test because the held-out
# alpha proxy is itself noisy.
# ==========================================================

random_states_result["alpha_confidence"] = np.maximum(
    random_states_result["p_holdout_alpha_gt_zero"],
    1.0 - random_states_result["p_holdout_alpha_gt_zero"],
)

bins = np.array([0.50, 0.60, 0.70, 0.80, 0.90, 1.000001])
labels = ["50–60%", "60–70%", "70–80%", "80–90%", "90–100%"]

random_states_result["confidence_bin"] = pd.cut(
    random_states_result["alpha_confidence"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False,
)

alpha_proxy_calibration = (
    random_states_result
    .groupby("confidence_bin", observed=False)
    .agg(
        n=("proxy_sign_correct", "size"),
        mean_confidence=("alpha_confidence", "mean"),
        observed_proxy_accuracy=("proxy_sign_correct", "mean"),
    )
    .reset_index()
)

print("\n========================================")
print("DRIVER-SEASON ABOVE-AVERAGE PROXY CALIBRATION")
print("========================================")
display(alpha_proxy_calibration)

plt.figure(figsize=(7, 6))
cal = alpha_proxy_calibration[alpha_proxy_calibration["n"] > 0]
plt.plot(
    cal["mean_confidence"],
    cal["observed_proxy_accuracy"],
    marker="o",
    label="driver-season holdout",
)
plt.plot([0.5, 1.0], [0.5, 1.0], linestyle="--", label="ideal")
plt.xlim(0.5, 1.0)
plt.ylim(0.5, 1.0)
plt.xlabel("Posterior confidence alpha is on predicted side of zero")
plt.ylabel("Fraction agreeing with held-out evidence proxy")
plt.title("Driver-season holdout alpha sign calibration")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# ==========================================================
# 18. STRATIFIED PERFORMANCE
# ==========================================================

def grouped_metrics(df, col):
    rows = []
    for value, grp in df.groupby(col, observed=False, dropna=False):
        x = summarise_states(grp, str(value))
        if x is not None:
            rows.append(x)
    return pd.DataFrame(rows)

temporal_metrics = grouped_metrics(random_states_result, "temporal_support")
era_metrics = grouped_metrics(random_states_result, "era")
evidence_metrics = grouped_metrics(random_states_result, "evidence_band")
career_metrics = grouped_metrics(random_states_result, "career_support_band")

print("\n========================================")
print("ALPHA RECONSTRUCTION BY TEMPORAL SUPPORT")
print("========================================")
display(temporal_metrics)

print("\n========================================")
print("ALPHA RECONSTRUCTION BY ERA")
print("========================================")
display(era_metrics)

print("\n========================================")
print("ALPHA RECONSTRUCTION BY HELD-OUT DATA RICHNESS")
print("========================================")
display(evidence_metrics)

print("\n========================================")
print("ALPHA RECONSTRUCTION BY CAREER SUPPORT")
print("========================================")
display(career_metrics)


# ==========================================================
# 19. FORCED CASE STUDIES
# ==========================================================

forced_results = (
    driver_season_validation[
        driver_season_validation["selection_source"] == "forced case"
    ]
    .sort_values(["year", "driver"])
    .copy()
)

print("\n========================================")
print("FORCED DRIVER-SEASON CASE STUDIES")
print("========================================")
display(
    forced_results[
        [
            "driver",
            "year",
            "n_races",
            "temporal_support",
            "other_modelled_seasons",
            "holdout_alpha_mean",
            "holdout_alpha_lb",
            "holdout_alpha_ub",
            "heldout_alpha_proxy",
            "full_alpha_mean",
            "holdout_minus_full",
            "holdout_full_shift_pooled_sd",
            "p_holdout_alpha_gt_zero",
            "proxy_sign_correct",
        ]
    ]
)


# ==========================================================
# 20. LARGEST SCORE SHIFTS / LARGEST PROXY DISAGREEMENTS
# ==========================================================

largest_shifts = (
    driver_season_validation
    .sort_values("absolute_holdout_full_shift", ascending=False)
    .reset_index(drop=True)
)

print("\n========================================")
print("LARGEST DRIVER-SEASON SCORE SHIFTS")
print("========================================")
display(
    largest_shifts[
        [
            "driver",
            "year",
            "selection_source",
            "n_races",
            "temporal_support",
            "other_modelled_seasons",
            "full_alpha_mean",
            "holdout_alpha_mean",
            "holdout_minus_full",
            "holdout_full_shift_pooled_sd",
            "heldout_alpha_proxy",
            "p_holdout_alpha_gt_zero",
        ]
    ].head(30)
)

driver_season_validation["absolute_proxy_error"] = np.abs(
    driver_season_validation["holdout_alpha_mean"]
    - driver_season_validation["heldout_alpha_proxy"]
)

largest_proxy_errors = (
    driver_season_validation
    .sort_values("absolute_proxy_error", ascending=False)
    .reset_index(drop=True)
)

print("\n========================================")
print("LARGEST DISAGREEMENTS WITH HELD-OUT ALPHA PROXY")
print("========================================")
display(
    largest_proxy_errors[
        [
            "driver",
            "year",
            "selection_source",
            "n_races",
            "holdout_alpha_mean",
            "holdout_alpha_lb",
            "holdout_alpha_ub",
            "heldout_alpha_proxy",
            "absolute_proxy_error",
            "full_alpha_mean",
            "temporal_support",
            "other_modelled_seasons",
        ]
    ].head(30)
)


# ==========================================================
# 21. SAVE OUTPUTS
# ==========================================================

heldout_rows.to_csv(
    DS_CV_DIR / "heldout_driver_season_row_predictions.csv",
    index=False,
)

driver_season_validation.to_csv(
    DS_CV_DIR / "driver_season_alpha_validation.csv",
    index=False,
)

row_predictive_metrics.to_csv(
    DS_CV_DIR / "row_predictive_metrics.csv",
    index=False,
)

primary_state_metrics.to_csv(
    DS_CV_DIR / "driver_season_reconstruction_metrics.csv",
    index=False,
)

alpha_proxy_calibration.to_csv(
    DS_CV_DIR / "driver_season_alpha_proxy_calibration.csv",
    index=False,
)

fold_health.to_csv(
    DS_CV_DIR / "fold_health.csv",
    index=False,
)


# ==========================================================
# 22. FINAL SUMMARY
# ==========================================================

primary = primary_state_metrics[
    primary_state_metrics["group"] == "random validation panel"
].iloc[0]

contrast_random = random_rows[random_rows["observation_type"] == "contrast"]
level_random = random_rows[random_rows["observation_type"] == "level"]

print("\n========================================")
print("STAGE 23 SUMMARY")
print("========================================")
print("Random driver-seasons validated:", len(random_states_result))
print("Forced case studies:", len(forced_results))
print(
    f"Holdout alpha vs unseen proxy correlation: "
    f"{primary['pearson_holdout_vs_proxy']:.3f}"
)
print(
    f"Holdout alpha vs unseen proxy sign accuracy: "
    f"{primary['above_average_sign_accuracy']:.3f}"
)
print(
    f"Holdout alpha vs official full-data alpha correlation: "
    f"{primary['pearson_holdout_vs_full']:.3f}"
)
print(
    f"Median |holdout - full alpha|: "
    f"{primary['median_abs_holdout_full_shift']:.3f}"
)
print(
    f"Official full-data alpha mean inside holdout 89% interval: "
    f"{primary['full_mean_inside_holdout_89']:.3f}"
)

if len(contrast_random):
    print(
        f"Held-out contrast ΔELPD vs zero null: "
        f"{contrast_random['delta_elpd'].sum():+.2f}"
    )

if len(level_random):
    print(
        f"Held-out team-level ΔELPD vs zero null: "
        f"{level_random['delta_elpd'].sum():+.2f}"
    )

print("\nInterpretation:")
print(
    "  High holdout-vs-proxy correlation => the latent season score "
    "predicts unseen direct evidence."
)
print(
    "  High holdout-vs-full correlation + small shifts => official season "
    "scores are stable rather than being driven only by that season's data."
)
print(
    "  Strong dependence on temporal support => the state-space is doing "
    "meaningful reconstruction work."
)
print(
    "  Proxy calibration is only approximate because the held-out alpha "
    "proxy is noisy."
)
print(
    "  Stage 24 simulation recovery is the decisive truth-known test of "
    "alpha/beta bias and posterior interval coverage."
)

print("\nSaved validation outputs ->")
print(DS_CV_DIR.resolve())
print("\nStage 23 complete.")


In [ ]:
# ==========================================================
# STAGE 24 — SIMULATION-BASED LATENT RECOVERY
# ==========================================================
#
# PURPOSE
# -------
# This is the first validation stage where the "true" latent values
# are known.
#
# We:
#
#   1. take realistic joint truth states from DIFFERENT posterior
#      draws of the accepted full-data Pantheon fit;
#   2. generate synthetic modern pace / qualifying observations from
#      the frozen Student-t likelihood;
#   3. generate synthetic historical qualifying / race orders from
#      the frozen Plackett-Luce likelihood;
#   4. refit the EXACT same frozen model to each synthetic dataset;
#   5. compare recovered posterior alpha, constructor beta, teammate
#      deltas, ranks and global parameters with the known truth.
#
# WHY POSTERIOR-DRAW TRUTHS?
# --------------------------
# This is a realistic recovery experiment rather than formal SBC.
# Drawing truth from the accepted posterior keeps simulated seasons,
# cars and global scales in the realistic part of parameter space
# supported by actual F1 data.
#
# It therefore answers:
#
#   "If the real latent F1 structure looked like a plausible Pantheon
#    world, how accurately could this model recover it from data with
#    the same schedule / teammate / constructor design?"
#
# It does NOT test model misspecification, because the synthetic data
# are generated by the same model that is then fitted.
#
# It also conditions on the observed design:
#   - race participation;
#   - constructor assignments;
#   - qualifying-session participation;
#   - historical classified-driver sets.
#
# Formal prior-SBC would be a separate, stronger computational test.
#
# PRIMARY OUTPUTS
# ---------------
# - alpha recovery correlation / RMSE / bias;
# - constructor beta recovery;
# - 50%, 80%, 89%, 95% empirical interval coverage;
# - within-season driver and constructor rank recovery;
# - teammate delta-alpha recovery and interval coverage;
# - driver-vs-car allocation error correlation;
# - global-parameter recovery;
# - optional JAWS recovery when Stage 16 scoring objects are in memory.
#
# IMPORTANT
# ---------
# This cell NEVER overwrites the official production trace.
# Every synthetic replicate has its own checkpoint directory.
# ==========================================================

from itertools import combinations
from pathlib import Path
import gc
import json
import time

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import scipy.stats as st
import xarray as xr


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

SIM_RECOVERY_REPS = 3

SIM_RECOVERY_DRAWS = 300
SIM_RECOVERY_TUNE = 300
SIM_RECOVERY_CHAINS = 2
SIM_RECOVERY_TARGET_ACCEPT = 0.92

SIM_RECOVERY_SEED = 24042

CHECKPOINT_ENGINE_SIM_RECOVERY = "h5netcdf"

SIM_RECOVERY_DIR = (
    Path(CFG.output_dir)
    / "validation_simulation_latent_recovery"
)

SIM_RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==========================================================
# 1. REQUIRED OBJECTS
# ==========================================================

required_objects = [
    "build_joint_model_1982_2025",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "hist_q_pack",
    "hist_race_pack",
    "idx_all",
    "actual_starts",
    "CFG",
]


missing = [
    name
    for name in required_objects
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Run the final model/scoring stages first. Missing: "
        + ", ".join(missing)
    )


if (
    "posterior_final" not in globals()
    and "final_trace_2025" not in globals()
):

    raise RuntimeError(
        "Could not find the accepted full-data posterior. "
        "Expected posterior_final or final_trace_2025."
    )


# ==========================================================
# 2. POSTERIOR HELPERS
# ==========================================================

def stack_state_draws(
    posterior,
    var_name,
):
    """
    Return an array:
        posterior sample x flattened state dimension
    """

    da = posterior[var_name].stack(
        sample=(
            "chain",
            "draw",
        )
    )

    other_dims = [
        dim
        for dim in da.dims
        if dim != "sample"
    ]

    x = da.transpose(
        "sample",
        *other_dims,
    ).values

    return np.asarray(
        x,
        dtype=float,
    ).reshape(
        x.shape[0],
        -1,
    )


def stack_scalar_draws(
    posterior,
    var_name,
):

    return np.asarray(
        posterior[var_name].values,
        dtype=float,
    ).reshape(-1)


def safe_corr(
    x,
    y,
    method="pearson",
):

    x = pd.Series(
        np.asarray(
            x,
            dtype=float,
        )
    )

    y = pd.Series(
        np.asarray(
            y,
            dtype=float,
        )
    )

    ok = x.notna() & y.notna()

    if ok.sum() < 3:

        return np.nan

    if (
        x[ok].nunique() < 2
        or y[ok].nunique() < 2
    ):

        return np.nan

    return float(
        x[ok].corr(
            y[ok],
            method=method,
        )
    )


# ==========================================================
# 3. ACCEPTED FULL-DATA POSTERIOR
# ==========================================================

if (
    "posterior_final"
    in globals()
):

    full_posterior = posterior_final

else:

    full_posterior = final_trace_2025[
        "posterior"
    ].to_dataset()


TRUTH_GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]


full_alpha_samples = stack_state_draws(
    full_posterior,
    "alpha_dy",
)


full_car_samples = stack_state_draws(
    full_posterior,
    "car_csy",
)


full_global_samples = {
    var:
        stack_scalar_draws(
            full_posterior,
            var,
        )
    for var in TRUTH_GLOBAL_VARS
}


n_full_draws = full_alpha_samples.shape[
    0
]


assert (
    full_car_samples.shape[
        0
    ]
    ==
    n_full_draws
)


for var, values in full_global_samples.items():

    assert (
        len(values)
        ==
        n_full_draws
    ), (
        var,
        len(values),
        n_full_draws,
    )


# ==========================================================
# 4. CHOOSE REALISTIC TRUTH DRAWS
# ==========================================================

truth_rng = np.random.default_rng(
    SIM_RECOVERY_SEED
)


truth_draw_indices = truth_rng.choice(
    n_full_draws,
    size=SIM_RECOVERY_REPS,
    replace=False,
)


print(
    "========================================"
)

print(
    "STAGE 24 — SIMULATION-BASED LATENT RECOVERY"
)

print(
    "========================================"
)


print(
    "Synthetic replicates:",
    SIM_RECOVERY_REPS,
)


print(
    "Truth source:",
    "distinct draws from accepted full-data posterior",
)


print(
    "Truth draw indices:",
    truth_draw_indices.tolist(),
)


print(
    "Sampler per replicate:",
    f"{SIM_RECOVERY_CHAINS} chains × "
    f"({SIM_RECOVERY_TUNE} tune + {SIM_RECOVERY_DRAWS} draws)",
)


print(
    "Driver-season states:",
    len(
        idx_all.active_driver_years
    ),
)


print(
    "Constructor-season states:",
    len(
        idx_all.active_constructor_years
    ),
)


# ==========================================================
# 5. GENERATIVE HELPERS — MODERN LIKELIHOODS
# ==========================================================

def simulate_modern_contrast(
    template,
    *,
    alpha_true,
    gamma_true,
    sigma_true,
    rng,
):
    """
    y =
        alpha_a - alpha_b
        + gamma * tenure_diff
        + Student-t noise
    """

    out = template.copy()

    dy_a = out[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = out[
        "dy_b"
    ].to_numpy(
        dtype=int
    )

    tenure = out[
        "tenure_diff"
    ].to_numpy(
        dtype=float
    )

    weight = out[
        "weight"
    ].to_numpy(
        dtype=float
    )

    mu = (
        alpha_true[
            dy_a
        ]
        -
        alpha_true[
            dy_b
        ]
        +
        gamma_true
        *
        tenure
    )

    scale = (
        sigma_true
        /
        np.sqrt(
            weight
        )
    )

    out[
        "y"
    ] = (
        mu
        +
        scale
        *
        rng.standard_t(
            df=CFG.student_t_nu,
            size=len(
                out
            ),
        )
    )

    return out


def simulate_modern_level(
    template,
    *,
    alpha_true,
    car_true,
    gamma_true,
    sigma_true,
    rng,
):
    """
    y =
        .5(alpha_a + alpha_b)
        + car
        + gamma * tenure_mean
        + Student-t noise
    """

    out = template.copy()

    dy_a = out[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = out[
        "dy_b"
    ].to_numpy(
        dtype=int
    )

    csy = out[
        "csy"
    ].to_numpy(
        dtype=int
    )

    tenure = out[
        "tenure_mean"
    ].to_numpy(
        dtype=float
    )

    weight = out[
        "weight"
    ].to_numpy(
        dtype=float
    )

    mu = (
        0.5
        *
        (
            alpha_true[
                dy_a
            ]
            +
            alpha_true[
                dy_b
            ]
        )
        +
        car_true[
            csy
        ]
        +
        gamma_true
        *
        tenure
    )

    scale = (
        sigma_true
        /
        np.sqrt(
            weight
        )
    )

    out[
        "y"
    ] = (
        mu
        +
        scale
        *
        rng.standard_t(
            df=CFG.student_t_nu,
            size=len(
                out
            ),
        )
    )

    return out


# ==========================================================
# 6. GENERATIVE HELPERS — HISTORICAL PLACKETT-LUCE
# ==========================================================

def softmax_stable(
    x,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    z = (
        x
        -
        np.max(
            x
        )
    )

    p = np.exp(
        z
    )

    return (
        p
        /
        p.sum()
    )


def simulate_plackett_luce_pack(
    template_pack,
    *,
    alpha_true,
    car_true,
    gamma_true,
    temperature_true,
    rng,
):
    """
    Generate a new historical ordering for every packed race while
    keeping the observed competitor set, constructor assignment,
    tenure covariate, raceId and year fixed.

    The returned dictionary has exactly the same structure expected
    by plackett_luce_logp_vectorized().
    """

    out = {
        key:
            (
                value.copy()
                if isinstance(
                    value,
                    np.ndarray,
                )
                else value
            )
        for key, value in template_pack.items()
    }

    dy_out = out[
        "dy"
    ]

    csy_out = out[
        "csy"
    ]

    tenure_out = out[
        "tenure"
    ]

    valid = template_pack[
        "valid"
    ]

    n_races = int(
        template_pack[
            "n_races"
        ]
    )

    for i in range(
        n_races
    ):

        n = int(
            valid[
                i
            ].sum()
        )

        dy = template_pack[
            "dy"
        ][
            i,
            :n,
        ].copy()

        csy = template_pack[
            "csy"
        ][
            i,
            :n,
        ].copy()

        tenure = template_pack[
            "tenure"
        ][
            i,
            :n,
        ].copy()

        score = (
            alpha_true[
                dy
            ]
            +
            car_true[
                csy
            ]
            +
            gamma_true
            *
            tenure
        )

        remaining = list(
            range(
                n
            )
        )

        order = []

        while len(
            remaining
        ) > 1:

            utilities = (
                score[
                    remaining
                ]
                /
                temperature_true
            )

            p = softmax_stable(
                utilities
            )

            local_choice = int(
                rng.choice(
                    len(
                        remaining
                    ),
                    p=p,
                )
            )

            chosen = remaining.pop(
                local_choice
            )

            order.append(
                chosen
            )

        order.extend(
            remaining
        )

        order = np.asarray(
            order,
            dtype=int,
        )

        dy_out[
            i,
            :n,
        ] = dy[
            order
        ]

        csy_out[
            i,
            :n,
        ] = csy[
            order
        ]

        tenure_out[
            i,
            :n,
        ] = tenure[
            order
        ]

    return out


# ==========================================================
# 7. SAVE / LOAD PACK HELPERS
# ==========================================================

PACK_ARRAY_KEYS = [
    "dy",
    "csy",
    "tenure",
    "valid",
    "choice",
    "candidate_mask",
    "raceId",
    "year",
]


def save_pack_npz(
    pack,
    path,
):

    np.savez_compressed(
        path,
        **{
            key:
                pack[
                    key
                ]
            for key in PACK_ARRAY_KEYS
        },
        n_races=np.asarray(
            [
                pack[
                    "n_races"
                ]
            ],
            dtype=int,
        ),
        max_field=np.asarray(
            [
                pack[
                    "max_field"
                ]
            ],
            dtype=int,
        ),
    )


def load_pack_npz(
    path,
):

    x = np.load(
        path,
        allow_pickle=False,
    )

    return {
        **{
            key:
                x[
                    key
                ]
            for key in PACK_ARRAY_KEYS
        },
        "n_races":
            int(
                x[
                    "n_races"
                ][
                    0
                ]
            ),
        "max_field":
            int(
                x[
                    "max_field"
                ][
                    0
                ]
            ),
    }


# ==========================================================
# 8. RECOVERY METRIC HELPERS
# ==========================================================

INTERVAL_SPECS = {
    "50":
        (
            0.25,
            0.75,
        ),
    "80":
        (
            0.10,
            0.90,
        ),
    "89":
        (
            0.055,
            0.945,
        ),
    "95":
        (
            0.025,
            0.975,
        ),
}


def recovery_table(
    draws,
    truth,
    *,
    labels,
    replicate,
    quantity,
):
    """
    One row per latent state.
    """

    draws = np.asarray(
        draws,
        dtype=float,
    )

    truth = np.asarray(
        truth,
        dtype=float,
    )

    mean = draws.mean(
        axis=0
    )

    sd = draws.std(
        axis=0,
        ddof=1,
    )

    out = pd.DataFrame(
        labels
    ).copy()

    out[
        "replicate"
    ] = int(
        replicate
    )

    out[
        "quantity"
    ] = quantity

    out[
        "truth"
    ] = truth

    out[
        "posterior_mean"
    ] = mean

    out[
        "posterior_sd"
    ] = sd

    out[
        "error"
    ] = (
        mean
        -
        truth
    )

    out[
        "absolute_error"
    ] = np.abs(
        out[
            "error"
        ]
    )

    out[
        "z_error"
    ] = np.where(
        sd > 0,
        (
            mean
            -
            truth
        )
        /
        sd,
        np.nan,
    )

    for name, (
        qlo,
        qhi,
    ) in INTERVAL_SPECS.items():

        lo = np.quantile(
            draws,
            qlo,
            axis=0,
        )

        hi = np.quantile(
            draws,
            qhi,
            axis=0,
        )

        out[
            f"lb_{name}"
        ] = lo

        out[
            f"ub_{name}"
        ] = hi

        out[
            f"covered_{name}"
        ] = (
            (
                truth
                >=
                lo
            )
            &
            (
                truth
                <=
                hi
            )
        ).astype(
            int
        )

    return out


def summarise_recovery(
    table,
    *,
    group_label,
):

    return {
        "group":
            group_label,
        "n":
            len(
                table
            ),
        "pearson":
            safe_corr(
                table[
                    "truth"
                ],
                table[
                    "posterior_mean"
                ],
                "pearson",
            ),
        "spearman":
            safe_corr(
                table[
                    "truth"
                ],
                table[
                    "posterior_mean"
                ],
                "spearman",
            ),
        "bias":
            float(
                table[
                    "error"
                ].mean()
            ),
        "rmse":
            float(
                np.sqrt(
                    np.mean(
                        table[
                            "error"
                        ]
                        ** 2
                    )
                )
            ),
        "mae":
            float(
                table[
                    "absolute_error"
                ].mean()
            ),
        "median_abs_error":
            float(
                table[
                    "absolute_error"
                ].median()
            ),
        "mean_posterior_sd":
            float(
                table[
                    "posterior_sd"
                ].mean()
            ),
        "mean_z_error":
            float(
                table[
                    "z_error"
                ].mean()
            ),
        "sd_z_error":
            float(
                table[
                    "z_error"
                ].std(
                    ddof=1
                )
            ),
        "coverage_50":
            float(
                table[
                    "covered_50"
                ].mean()
            ),
        "coverage_80":
            float(
                table[
                    "covered_80"
                ].mean()
            ),
        "coverage_89":
            float(
                table[
                    "covered_89"
                ].mean()
            ),
        "coverage_95":
            float(
                table[
                    "covered_95"
                ].mean()
            ),
    }


# ==========================================================
# 9. STATE LABELS
# ==========================================================

driver_state_labels = pd.DataFrame(
    [
        {
            "driver":
                str(
                    driver
                ),
            "year":
                int(
                    year
                ),
        }
        for driver, year
        in idx_all.active_driver_years
    ]
)


driver_state_labels[
    "era"
] = pd.cut(
    driver_state_labels[
        "year"
    ],
    bins=[
        1981,
        1995,
        2005,
        2015,
        2025,
    ],
    labels=[
        "1982–1995",
        "1996–2005",
        "2006–2015",
        "2016–2025",
    ],
)


car_state_labels = pd.DataFrame(
    [
        {
            "constructorId":
                int(
                    constructor_id
                ),
            "year":
                int(
                    year
                ),
        }
        for constructor_id, year
        in idx_all.active_constructor_years
    ]
)


car_state_labels[
    "era"
] = pd.cut(
    car_state_labels[
        "year"
    ],
    bins=[
        1981,
        1995,
        2005,
        2015,
        2025,
    ],
    labels=[
        "1982–1995",
        "1996–2005",
        "2006–2015",
        "2016–2025",
    ],
)


# ==========================================================
# 10. TEAMMATE EDGE-SEASON DESIGN
# ==========================================================
#
# Use actual-start teammate relationships to evaluate recovery of
# alpha_i,t - alpha_j,t directly.
# ==========================================================

teammate_pair_rows = []


for (
    year,
    constructor_id,
), grp in actual_starts.groupby(
    [
        "year",
        "constructorId",
    ]
):

    drivers = sorted(
        set(
            grp[
                "driver"
            ]
            .dropna()
            .astype(str)
        )
    )

    for a, b in combinations(
        drivers,
        2,
    ):

        if (
            (
                a,
                int(
                    year
                ),
            )
            not in idx_all.dy_map
            or
            (
                b,
                int(
                    year
                ),
            )
            not in idx_all.dy_map
        ):

            continue

        shared_races = len(
            set(
                grp.loc[
                    grp[
                        "driver"
                    ].eq(
                        a
                    ),
                    "raceId",
                ]
            )
            &
            set(
                grp.loc[
                    grp[
                        "driver"
                    ].eq(
                        b
                    ),
                    "raceId",
                ]
            )
        )

        if shared_races <= 0:

            continue

        teammate_pair_rows.append(
            {
                "driver_a":
                    a,
                "driver_b":
                    b,
                "year":
                    int(
                        year
                    ),
                "constructorId":
                    int(
                        constructor_id
                    ),
                "shared_starts":
                    int(
                        shared_races
                    ),
                "dy_a":
                    int(
                        idx_all.dy_map[
                            (
                                a,
                                int(
                                    year
                                ),
                            )
                        ]
                    ),
                "dy_b":
                    int(
                        idx_all.dy_map[
                            (
                                b,
                                int(
                                    year
                                ),
                            )
                        ]
                    ),
            }
        )


teammate_pair_design = (
    pd.DataFrame(
        teammate_pair_rows
    )
    .drop_duplicates(
        [
            "driver_a",
            "driver_b",
            "year",
            "constructorId",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==========================================================
# 11. WITHIN-SEASON RANK RECOVERY
# ==========================================================

def within_season_rank_metrics(
    recovery,
    *,
    entity_cols,
    quantity_name,
):
    """
    Calculate ranking recovery separately within every season.
    """

    rows = []

    for (
        replicate,
        year,
    ), grp in recovery.groupby(
        [
            "replicate",
            "year",
        ]
    ):

        if len(
            grp
        ) < 2:

            continue

        truth = grp[
            "truth"
        ].to_numpy(
            dtype=float
        )

        estimate = grp[
            "posterior_mean"
        ].to_numpy(
            dtype=float
        )

        true_order = np.argsort(
            -truth
        )

        estimate_order = np.argsort(
            -estimate
        )

        true_top = int(
            true_order[
                0
            ]
        )

        estimate_top = int(
            estimate_order[
                0
            ]
        )

        true_top3 = set(
            true_order[
                : min(
                    3,
                    len(
                        grp
                    ),
                )
            ].tolist()
        )

        estimate_top3 = set(
            estimate_order[
                : min(
                    3,
                    len(
                        grp
                    ),
                )
            ].tolist()
        )

        rows.append(
            {
                "replicate":
                    int(
                        replicate
                    ),
                "year":
                    int(
                        year
                    ),
                "quantity":
                    quantity_name,
                "n_states":
                    len(
                        grp
                    ),
                "rank_spearman":
                    safe_corr(
                        truth,
                        estimate,
                        "spearman",
                    ),
                "top1_correct":
                    int(
                        true_top
                        ==
                        estimate_top
                    ),
                "top3_overlap":
                    len(
                        true_top3
                        &
                        estimate_top3
                    )
                    /
                    len(
                        true_top3
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ==========================================================
# 12. DRIVER-vs-CAR ALLOCATION DIAGNOSTIC
# ==========================================================

def allocation_diagnostic(
    alpha_hat,
    car_hat,
    alpha_true,
    car_true,
):
    """
    For each constructor-season, compare:
      error in the average driver contribution
    with
      error in constructor beta.

    A strong negative correlation means the model often gets total
    team strength right while trading driver credit against car credit.
    """

    rows = []

    for channel, df in [
        (
            "pace_level",
            pace_level_all,
        ),
        (
            "q1_level",
            q_level_all_model,
        ),
    ]:

        driver_error = (
            0.5
            *
            (
                (
                    alpha_hat[
                        df[
                            "dy_a"
                        ].to_numpy(
                            dtype=int
                        )
                    ]
                    -
                    alpha_true[
                        df[
                            "dy_a"
                        ].to_numpy(
                            dtype=int
                        )
                    ]
                )
                +
                (
                    alpha_hat[
                        df[
                            "dy_b"
                        ].to_numpy(
                            dtype=int
                        )
                    ]
                    -
                    alpha_true[
                        df[
                            "dy_b"
                        ].to_numpy(
                            dtype=int
                        )
                    ]
                )
            )
        )

        csy = df[
            "csy"
        ].to_numpy(
            dtype=int
        )

        car_error = (
            car_hat[
                csy
            ]
            -
            car_true[
                csy
            ]
        )

        weight = df[
            "weight"
        ].to_numpy(
            dtype=float
        )

        tmp = pd.DataFrame(
            {
                "channel":
                    channel,
                "csy":
                    csy,
                "driver_common_error":
                    driver_error,
                "car_error":
                    car_error,
                "weight":
                    weight,
            }
        )

        rows.append(
            tmp
        )


    x = pd.concat(
        rows,
        ignore_index=True,
    )


    aggregated = []


    for csy, grp in x.groupby(
        "csy"
    ):

        w = grp[
            "weight"
        ].to_numpy(
            dtype=float
        )

        driver_common_error = float(
            np.average(
                grp[
                    "driver_common_error"
                ],
                weights=w,
            )
        )

        car_error = float(
            np.average(
                grp[
                    "car_error"
                ],
                weights=w,
            )
        )

        aggregated.append(
            {
                "csy":
                    int(
                        csy
                    ),
                "driver_common_error":
                    driver_common_error,
                "car_error":
                    car_error,
                "total_team_signal_error":
                    driver_common_error
                    +
                    car_error,
            }
        )


    aggregated = pd.DataFrame(
        aggregated
    )


    summary = {
        "constructor_seasons":
            len(
                aggregated
            ),
        "corr_driver_error_vs_car_error":
            safe_corr(
                aggregated[
                    "driver_common_error"
                ],
                aggregated[
                    "car_error"
                ],
                "pearson",
            ),
        "rmse_driver_common_error":
            float(
                np.sqrt(
                    np.mean(
                        aggregated[
                            "driver_common_error"
                        ]
                        ** 2
                    )
                )
            ),
        "rmse_car_error":
            float(
                np.sqrt(
                    np.mean(
                        aggregated[
                            "car_error"
                        ]
                        ** 2
                    )
                )
            ),
        "rmse_total_team_signal_error":
            float(
                np.sqrt(
                    np.mean(
                        aggregated[
                            "total_team_signal_error"
                        ]
                        ** 2
                    )
                )
            ),
    }


    return (
        aggregated,
        summary,
    )


# ==========================================================
# 13. OPTIONAL JAWS RECOVERY
# ==========================================================

def compute_jaws_from_alpha_draws(
    alpha_draws,
    drivers,
    driver_year_map,
):
    """
    Reproduce the official draw-wise JAWS functional:
      60% peak + 40% sqrt-surplus longevity.

    `drivers` defines the normalization pool.
    """

    n_draws = alpha_draws.shape[
        0
    ]

    n_drivers = len(
        drivers
    )

    peak = np.zeros(
        (
            n_draws,
            n_drivers,
        ),
        dtype=float,
    )

    longevity_raw = np.zeros_like(
        peak
    )

    for j, driver in enumerate(
        drivers
    ):

        years = sorted(
            driver_year_map[
                driver
            ]
        )

        indices = [
            idx_all.dy_map[
                (
                    driver,
                    int(
                        year
                    ),
                )
            ]
            for year in years
        ]

        a = alpha_draws[
            :,
            indices,
        ]

        top_n = min(
            5,
            a.shape[
                1
            ],
        )

        top = np.sort(
            a,
            axis=1,
        )[
            :,
            -top_n:
        ]

        peak[
            :,
            j
        ] = top.mean(
            axis=1
        )

        surplus = np.maximum(
            a,
            0.0,
        ).sum(
            axis=1
        )

        longevity_raw[
            :,
            j
        ] = np.sqrt(
            surplus
        )


    peak_max = np.max(
        peak,
        axis=1,
        keepdims=True,
    )

    long_max = np.max(
        longevity_raw,
        axis=1,
        keepdims=True,
    )


    peak_score = (
        100.0
        *
        peak
        /
        peak_max
    )


    longevity_score = (
        100.0
        *
        longevity_raw
        /
        long_max
    )


    jaws = (
        0.60
        *
        peak_score
        +
        0.40
        *
        longevity_score
    )


    return jaws


# ==========================================================
# 14. RESULTS COLLECTORS
# ==========================================================

alpha_recovery_frames = []

car_recovery_frames = []

teammate_recovery_frames = []

global_recovery_frames = []

allocation_frames = []

allocation_summary_rows = []

fit_health_rows = []

jaws_recovery_frames = []


# ==========================================================
# 15. RUN SYNTHETIC RECOVERY REPLICATES
# ==========================================================

for rep_idx, truth_draw_idx in enumerate(
    truth_draw_indices,
    start=1,
):

    print(
        "\n========================================"
    )

    print(
        f"SYNTHETIC RECOVERY REPLICATE "
        f"{rep_idx}/{SIM_RECOVERY_REPS}"
    )

    print(
        "========================================"
    )


    rep_dir = (
        SIM_RECOVERY_DIR
        /
        f"replicate_{rep_idx:02d}"
    )

    rep_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ------------------------------------------------------
    # Known truth for this replicate.
    # ------------------------------------------------------

    alpha_true = full_alpha_samples[
        truth_draw_idx
    ].copy()

    car_true = full_car_samples[
        truth_draw_idx
    ].copy()


    truth_globals = {
        var:
            float(
                full_global_samples[
                    var
                ][
                    truth_draw_idx
                ]
            )
        for var in TRUTH_GLOBAL_VARS
    }


    truth_npz_path = (
        rep_dir
        /
        "truth_states.npz"
    )


    np.savez_compressed(
        truth_npz_path,
        truth_draw_index=np.asarray(
            [
                truth_draw_idx
            ],
            dtype=int,
        ),
        alpha_true=alpha_true,
        car_true=car_true,
        **{
            var:
                np.asarray(
                    [
                        value
                    ],
                    dtype=float,
                )
            for var, value
            in truth_globals.items()
        },
    )


    # ------------------------------------------------------
    # Simulate or reload synthetic likelihood data.
    # ------------------------------------------------------

    sim_pc_path = (
        rep_dir
        /
        "sim_pace_contrast.csv"
    )

    sim_pl_path = (
        rep_dir
        /
        "sim_pace_level.csv"
    )

    sim_qc_path = (
        rep_dir
        /
        "sim_q_contrast.csv"
    )

    sim_ql_path = (
        rep_dir
        /
        "sim_q_level.csv"
    )

    sim_hq_path = (
        rep_dir
        /
        "sim_hist_quali_pack.npz"
    )

    sim_hr_path = (
        rep_dir
        /
        "sim_hist_race_pack.npz"
    )


    all_sim_files_exist = all(
        path.exists()
        for path in [
            sim_pc_path,
            sim_pl_path,
            sim_qc_path,
            sim_ql_path,
            sim_hq_path,
            sim_hr_path,
        ]
    )


    if all_sim_files_exist:

        print(
            "Loading existing synthetic dataset."
        )

        sim_pc = pd.read_csv(
            sim_pc_path
        )

        sim_pl = pd.read_csv(
            sim_pl_path
        )

        sim_qc = pd.read_csv(
            sim_qc_path
        )

        sim_ql = pd.read_csv(
            sim_ql_path
        )

        sim_hq = load_pack_npz(
            sim_hq_path
        )

        sim_hr = load_pack_npz(
            sim_hr_path
        )


    else:

        print(
            "Generating synthetic dataset..."
        )


        sim_rng = np.random.default_rng(
            SIM_RECOVERY_SEED
            +
            1000
            *
            rep_idx
        )


        sim_pc = simulate_modern_contrast(
            pace_contrast_all,
            alpha_true=alpha_true,
            gamma_true=truth_globals[
                "gamma_tenure"
            ],
            sigma_true=truth_globals[
                "sigma_pace_contrast"
            ],
            rng=sim_rng,
        )


        sim_pl = simulate_modern_level(
            pace_level_all,
            alpha_true=alpha_true,
            car_true=car_true,
            gamma_true=truth_globals[
                "gamma_tenure"
            ],
            sigma_true=truth_globals[
                "sigma_pace_level"
            ],
            rng=sim_rng,
        )


        sim_qc = simulate_modern_contrast(
            q_contrast_all,
            alpha_true=alpha_true,
            gamma_true=truth_globals[
                "gamma_tenure"
            ],
            sigma_true=truth_globals[
                "sigma_q_contrast"
            ],
            rng=sim_rng,
        )


        sim_ql = simulate_modern_level(
            q_level_all_model,
            alpha_true=alpha_true,
            car_true=car_true,
            gamma_true=truth_globals[
                "gamma_tenure"
            ],
            sigma_true=truth_globals[
                "sigma_q_level"
            ],
            rng=sim_rng,
        )


        sim_hq = simulate_plackett_luce_pack(
            hist_q_pack,
            alpha_true=alpha_true,
            car_true=car_true,
            gamma_true=truth_globals[
                "gamma_tenure"
            ],
            temperature_true=truth_globals[
                "tau_hist_quali"
            ],
            rng=sim_rng,
        )


        sim_hr = simulate_plackett_luce_pack(
            hist_race_pack,
            alpha_true=alpha_true,
            car_true=car_true,
            gamma_true=truth_globals[
                "gamma_tenure"
            ],
            temperature_true=truth_globals[
                "tau_hist_race"
            ],
            rng=sim_rng,
        )


        sim_pc.to_csv(
            sim_pc_path,
            index=False,
        )

        sim_pl.to_csv(
            sim_pl_path,
            index=False,
        )

        sim_qc.to_csv(
            sim_qc_path,
            index=False,
        )

        sim_ql.to_csv(
            sim_ql_path,
            index=False,
        )

        save_pack_npz(
            sim_hq,
            sim_hq_path,
        )

        save_pack_npz(
            sim_hr,
            sim_hr_path,
        )


    # ------------------------------------------------------
    # Build the exact frozen model on synthetic data.
    # ------------------------------------------------------

    synthetic_model = build_joint_model_1982_2025(
        sim_pc,
        sim_pl,
        sim_qc,
        sim_ql,
        sim_hq,
        sim_hr,
        idx_all,
        cfg=CFG,
    )


    trace_path = (
        rep_dir
        /
        (
            f"simulation_recovery_trace_"
            f"{SIM_RECOVERY_DRAWS}draw_"
            f"{SIM_RECOVERY_CHAINS}chain.nc"
        )
    )


    if trace_path.exists():

        print(
            "Loading synthetic-fit checkpoint ->",
            trace_path,
        )

        sim_trace = xr.open_datatree(
            trace_path,
            engine=CHECKPOINT_ENGINE_SIM_RECOVERY,
        )


    else:

        print(
            "Sampling synthetic recovery fit..."
        )

        t0 = time.perf_counter()


        with synthetic_model:

            sim_trace = pm.sample(
                draws=SIM_RECOVERY_DRAWS,
                tune=SIM_RECOVERY_TUNE,
                chains=SIM_RECOVERY_CHAINS,
                random_seed=(
                    SIM_RECOVERY_SEED
                    +
                    100
                    *
                    rep_idx
                ),
                target_accept=SIM_RECOVERY_TARGET_ACCEPT,
                nuts_sampler="nutpie",
                nuts_sampler_kwargs={
                    "backend":
                        "jax",
                },
                return_inferencedata=True,
                progressbar=True,
            )


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        print(
            "Synthetic fit wall time:",
            f"{elapsed / 60.0:.2f} min",
        )


        sim_trace.to_netcdf(
            trace_path,
            engine=CHECKPOINT_ENGINE_SIM_RECOVERY,
            mode="w",
        )


        print(
            "Saved ->",
            trace_path,
        )


    sim_posterior = sim_trace[
        "posterior"
    ].to_dataset()


    sim_sample_stats = sim_trace[
        "sample_stats"
    ].to_dataset()


    # ------------------------------------------------------
    # Lightweight fit health.
    # ------------------------------------------------------

    health_summary = az.summary(
        sim_posterior,
        var_names=TRUTH_GLOBAL_VARS,
        kind="diagnostics",
    )


    max_global_rhat = float(
        health_summary[
            "r_hat"
        ].max()
    )


    min_global_ess = float(
        health_summary[
            "ess_bulk"
        ].min()
    )


    if (
        "diverging"
        in sim_sample_stats
    ):

        divergences = int(
            np.asarray(
                sim_sample_stats[
                    "diverging"
                ].values
            ).sum()
        )

    else:

        divergences = 0


    fit_health_rows.append(
        {
            "replicate":
                rep_idx,
            "truth_draw_index":
                int(
                    truth_draw_idx
                ),
            "max_global_rhat":
                max_global_rhat,
            "min_global_ess_bulk":
                min_global_ess,
            "divergences":
                divergences,
        }
    )


    print(
        "Synthetic-fit health:",
        f"max global Rhat={max_global_rhat:.4f},",
        f"min global ESS={min_global_ess:.1f},",
        f"divergences={divergences}",
    )


    # ------------------------------------------------------
    # Recover alpha and constructor beta.
    # ------------------------------------------------------

    alpha_fit_draws = stack_state_draws(
        sim_posterior,
        "alpha_dy",
    )


    car_fit_draws = stack_state_draws(
        sim_posterior,
        "car_csy",
    )


    alpha_table = recovery_table(
        alpha_fit_draws,
        alpha_true,
        labels=driver_state_labels,
        replicate=rep_idx,
        quantity="driver_alpha",
    )


    car_table = recovery_table(
        car_fit_draws,
        car_true,
        labels=car_state_labels,
        replicate=rep_idx,
        quantity="constructor_beta",
    )


    alpha_recovery_frames.append(
        alpha_table
    )


    car_recovery_frames.append(
        car_table
    )


    # ------------------------------------------------------
    # Global parameter recovery.
    # ------------------------------------------------------

    global_rows = []


    for var in TRUTH_GLOBAL_VARS:

        draws = stack_scalar_draws(
            sim_posterior,
            var,
        )

        truth = truth_globals[
            var
        ]

        mean = float(
            draws.mean()
        )

        sd = float(
            draws.std(
                ddof=1
            )
        )

        lb89 = float(
            np.quantile(
                draws,
                0.055,
            )
        )

        ub89 = float(
            np.quantile(
                draws,
                0.945,
            )
        )

        global_rows.append(
            {
                "replicate":
                    rep_idx,
                "parameter":
                    var,
                "truth":
                    truth,
                "posterior_mean":
                    mean,
                "posterior_sd":
                    sd,
                "error":
                    mean
                    -
                    truth,
                "lb_89":
                    lb89,
                "ub_89":
                    ub89,
                "covered_89":
                    int(
                        truth
                        >=
                        lb89
                        and
                        truth
                        <=
                        ub89
                    ),
            }
        )


    global_recovery_frames.append(
        pd.DataFrame(
            global_rows
        )
    )


    # ------------------------------------------------------
    # Teammate delta-alpha recovery.
    # ------------------------------------------------------

    dy_a = teammate_pair_design[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = teammate_pair_design[
        "dy_b"
    ].to_numpy(
        dtype=int
    )


    delta_draws = (
        alpha_fit_draws[
            :,
            dy_a,
        ]
        -
        alpha_fit_draws[
            :,
            dy_b,
        ]
    )


    delta_truth = (
        alpha_true[
            dy_a
        ]
        -
        alpha_true[
            dy_b
        ]
    )


    delta_mean = delta_draws.mean(
        axis=0
    )


    delta_sd = delta_draws.std(
        axis=0,
        ddof=1,
    )


    delta_lb89 = np.quantile(
        delta_draws,
        0.055,
        axis=0,
    )


    delta_ub89 = np.quantile(
        delta_draws,
        0.945,
        axis=0,
    )


    teammate_table = teammate_pair_design[
        [
            "driver_a",
            "driver_b",
            "year",
            "constructorId",
            "shared_starts",
        ]
    ].copy()


    teammate_table[
        "replicate"
    ] = rep_idx


    teammate_table[
        "truth_delta_alpha"
    ] = delta_truth


    teammate_table[
        "posterior_mean_delta_alpha"
    ] = delta_mean


    teammate_table[
        "posterior_sd_delta_alpha"
    ] = delta_sd


    teammate_table[
        "lb_89"
    ] = delta_lb89


    teammate_table[
        "ub_89"
    ] = delta_ub89


    teammate_table[
        "error"
    ] = (
        delta_mean
        -
        delta_truth
    )


    teammate_table[
        "covered_89"
    ] = (
        (
            delta_truth
            >=
            delta_lb89
        )
        &
        (
            delta_truth
            <=
            delta_ub89
        )
    ).astype(
        int
    )


    teammate_table[
        "sign_correct"
    ] = (
        np.sign(
            delta_mean
        )
        ==
        np.sign(
            delta_truth
        )
    ).astype(
        int
    )


    teammate_table[
        "sign_correct_abs_truth_gt_0p1"
    ] = np.where(
        np.abs(
            delta_truth
        )
        >
        0.10,
        teammate_table[
            "sign_correct"
        ],
        np.nan,
    )


    teammate_recovery_frames.append(
        teammate_table
    )


    # ------------------------------------------------------
    # Driver / car allocation diagnostic.
    # ------------------------------------------------------

    alpha_hat = alpha_fit_draws.mean(
        axis=0
    )

    car_hat = car_fit_draws.mean(
        axis=0
    )


    (
        allocation_table,
        allocation_summary,
    ) = allocation_diagnostic(
        alpha_hat,
        car_hat,
        alpha_true,
        car_true,
    )


    allocation_table[
        "replicate"
    ] = rep_idx


    allocation_frames.append(
        allocation_table
    )


    allocation_summary[
        "replicate"
    ] = rep_idx


    allocation_summary_rows.append(
        allocation_summary
    )


    # ------------------------------------------------------
    # Optional JAWS recovery.
    # ------------------------------------------------------

    if (
        "scored_drivers"
        in globals()
        and
        "driver_years"
        in globals()
        and
        "pantheon_2025"
        in globals()
    ):

        jaws_pool = [
            str(
                d
            )
            for d in scored_drivers
            if str(
                d
            )
            in driver_years
        ]


        if len(
            jaws_pool
        ):

            truth_jaws = compute_jaws_from_alpha_draws(
                alpha_true[
                    None,
                    :
                ],
                jaws_pool,
                driver_years,
            )[
                0
            ]


            fit_jaws_draws = compute_jaws_from_alpha_draws(
                alpha_fit_draws,
                jaws_pool,
                driver_years,
            )


            fit_jaws_mean = fit_jaws_draws.mean(
                axis=0
            )


            fit_jaws_lb89 = np.quantile(
                fit_jaws_draws,
                0.055,
                axis=0,
            )


            fit_jaws_ub89 = np.quantile(
                fit_jaws_draws,
                0.945,
                axis=0,
            )


            jaws_table = pd.DataFrame(
                {
                    "replicate":
                        rep_idx,
                    "driver":
                        jaws_pool,
                    "truth_jaws":
                        truth_jaws,
                    "posterior_mean_jaws":
                        fit_jaws_mean,
                    "lb_89":
                        fit_jaws_lb89,
                    "ub_89":
                        fit_jaws_ub89,
                }
            )


            jaws_table[
                "error"
            ] = (
                jaws_table[
                    "posterior_mean_jaws"
                ]
                -
                jaws_table[
                    "truth_jaws"
                ]
            )


            jaws_table[
                "covered_89"
            ] = (
                (
                    jaws_table[
                        "truth_jaws"
                    ]
                    >=
                    jaws_table[
                        "lb_89"
                    ]
                )
                &
                (
                    jaws_table[
                        "truth_jaws"
                    ]
                    <=
                    jaws_table[
                        "ub_89"
                    ]
                )
            ).astype(
                int
            )


            jaws_recovery_frames.append(
                jaws_table
            )


    # ------------------------------------------------------
    # Save replicate-specific recovery tables.
    # ------------------------------------------------------

    alpha_table.to_csv(
        rep_dir
        /
        "alpha_recovery.csv",
        index=False,
    )


    car_table.to_csv(
        rep_dir
        /
        "car_recovery.csv",
        index=False,
    )


    teammate_table.to_csv(
        rep_dir
        /
        "teammate_delta_recovery.csv",
        index=False,
    )


    pd.DataFrame(
        global_rows
    ).to_csv(
        rep_dir
        /
        "global_parameter_recovery.csv",
        index=False,
    )


    allocation_table.to_csv(
        rep_dir
        /
        "driver_car_allocation_errors.csv",
        index=False,
    )


    # ------------------------------------------------------
    # Free memory before next replicate.
    # ------------------------------------------------------

    try:

        sim_trace.close()

    except Exception:

        pass


    del synthetic_model
    del sim_trace
    del sim_posterior
    del sim_sample_stats
    del alpha_fit_draws
    del car_fit_draws

    gc.collect()


# ==========================================================
# 16. COMBINE REPLICATES
# ==========================================================

alpha_recovery = pd.concat(
    alpha_recovery_frames,
    ignore_index=True,
)


car_recovery = pd.concat(
    car_recovery_frames,
    ignore_index=True,
)


teammate_recovery = pd.concat(
    teammate_recovery_frames,
    ignore_index=True,
)


global_recovery = pd.concat(
    global_recovery_frames,
    ignore_index=True,
)


allocation_errors = pd.concat(
    allocation_frames,
    ignore_index=True,
)


allocation_summary = pd.DataFrame(
    allocation_summary_rows
)


fit_health = pd.DataFrame(
    fit_health_rows
)


if jaws_recovery_frames:

    jaws_recovery = pd.concat(
        jaws_recovery_frames,
        ignore_index=True,
    )

else:

    jaws_recovery = pd.DataFrame()


# ==========================================================
# 17. ALPHA / BETA RECOVERY SUMMARY
# ==========================================================

recovery_summary_rows = []


for rep in sorted(
    alpha_recovery[
        "replicate"
    ].unique()
):

    recovery_summary_rows.append(
        {
            "replicate":
                int(
                    rep
                ),
            **summarise_recovery(
                alpha_recovery[
                    alpha_recovery[
                        "replicate"
                    ]
                    ==
                    rep
                ],
                group_label="driver alpha",
            ),
        }
    )


    recovery_summary_rows.append(
        {
            "replicate":
                int(
                    rep
                ),
            **summarise_recovery(
                car_recovery[
                    car_recovery[
                        "replicate"
                    ]
                    ==
                    rep
                ],
                group_label="constructor beta",
            ),
        }
    )


# Pooled state-level summary.
recovery_summary_rows.append(
    {
        "replicate":
            "pooled",
        **summarise_recovery(
            alpha_recovery,
            group_label="driver alpha",
        ),
    }
)


recovery_summary_rows.append(
    {
        "replicate":
            "pooled",
        **summarise_recovery(
            car_recovery,
            group_label="constructor beta",
        ),
    }
)


latent_recovery_summary = pd.DataFrame(
    recovery_summary_rows
)


print(
    "\n========================================"
)

print(
    "LATENT STATE RECOVERY"
)

print(
    "========================================"
)


display(
    latent_recovery_summary
)


# ==========================================================
# 18. ALPHA RECOVERY BY ERA
# ==========================================================

alpha_era_rows = []


for era, grp in alpha_recovery.groupby(
    "era",
    observed=False,
):

    alpha_era_rows.append(
        summarise_recovery(
            grp,
            group_label=str(
                era
            ),
        )
    )


alpha_recovery_by_era = pd.DataFrame(
    alpha_era_rows
)


print(
    "\n========================================"
)

print(
    "DRIVER ALPHA RECOVERY BY ERA"
)

print(
    "========================================"
)


display(
    alpha_recovery_by_era
)


# ==========================================================
# 19. WITHIN-SEASON RANK RECOVERY
# ==========================================================

driver_rank_recovery = within_season_rank_metrics(
    alpha_recovery,
    entity_cols=[
        "driver",
    ],
    quantity_name="driver alpha",
)


car_rank_recovery = within_season_rank_metrics(
    car_recovery,
    entity_cols=[
        "constructorId",
    ],
    quantity_name="constructor beta",
)


rank_recovery_summary = pd.DataFrame(
    [
        {
            "quantity":
                "driver alpha",
            "season_replicates":
                len(
                    driver_rank_recovery
                ),
            "mean_rank_spearman":
                driver_rank_recovery[
                    "rank_spearman"
                ].mean(),
            "median_rank_spearman":
                driver_rank_recovery[
                    "rank_spearman"
                ].median(),
            "top1_accuracy":
                driver_rank_recovery[
                    "top1_correct"
                ].mean(),
            "mean_top3_overlap":
                driver_rank_recovery[
                    "top3_overlap"
                ].mean(),
        },
        {
            "quantity":
                "constructor beta",
            "season_replicates":
                len(
                    car_rank_recovery
                ),
            "mean_rank_spearman":
                car_rank_recovery[
                    "rank_spearman"
                ].mean(),
            "median_rank_spearman":
                car_rank_recovery[
                    "rank_spearman"
                ].median(),
            "top1_accuracy":
                car_rank_recovery[
                    "top1_correct"
                ].mean(),
            "mean_top3_overlap":
                car_rank_recovery[
                    "top3_overlap"
                ].mean(),
        },
    ]
)


print(
    "\n========================================"
)

print(
    "WITHIN-SEASON RANK RECOVERY"
)

print(
    "========================================"
)


display(
    rank_recovery_summary
)


# ==========================================================
# 20. TEAMMATE DELTA-ALPHA RECOVERY
# ==========================================================

teammate_recovery_summary = pd.DataFrame(
    [
        {
            "pair_seasons":
                len(
                    teammate_recovery
                ),
            "pearson":
                safe_corr(
                    teammate_recovery[
                        "truth_delta_alpha"
                    ],
                    teammate_recovery[
                        "posterior_mean_delta_alpha"
                    ],
                    "pearson",
                ),
            "spearman":
                safe_corr(
                    teammate_recovery[
                        "truth_delta_alpha"
                    ],
                    teammate_recovery[
                        "posterior_mean_delta_alpha"
                    ],
                    "spearman",
                ),
            "bias":
                float(
                    teammate_recovery[
                        "error"
                    ].mean()
                ),
            "rmse":
                float(
                    np.sqrt(
                        np.mean(
                            teammate_recovery[
                                "error"
                            ]
                            ** 2
                        )
                    )
                ),
            "coverage_89":
                float(
                    teammate_recovery[
                        "covered_89"
                    ].mean()
                ),
            "sign_accuracy_all":
                float(
                    teammate_recovery[
                        "sign_correct"
                    ].mean()
                ),
            "sign_accuracy_abs_truth_gt_0p1":
                float(
                    teammate_recovery[
                        "sign_correct_abs_truth_gt_0p1"
                    ].mean()
                ),
        }
    ]
)


print(
    "\n========================================"
)

print(
    "TEAMMATE DELTA-ALPHA RECOVERY"
)

print(
    "========================================"
)


display(
    teammate_recovery_summary
)


# ==========================================================
# 21. DRIVER-vs-CAR ALLOCATION
# ==========================================================

print(
    "\n========================================"
)

print(
    "DRIVER-vs-CAR ALLOCATION ERROR"
)

print(
    "========================================"
)


display(
    allocation_summary
)


# ==========================================================
# 22. GLOBAL PARAMETER RECOVERY
# ==========================================================

global_parameter_summary = (
    global_recovery.groupby(
        "parameter",
        as_index=False,
    )
    .agg(
        mean_truth=(
            "truth",
            "mean",
        ),
        mean_posterior=(
            "posterior_mean",
            "mean",
        ),
        mean_error=(
            "error",
            "mean",
        ),
        rmse=(
            "error",
            lambda x:
                float(
                    np.sqrt(
                        np.mean(
                            np.asarray(
                                x,
                                dtype=float,
                            )
                            ** 2
                        )
                    )
                ),
        ),
        coverage_89=(
            "covered_89",
            "mean",
        ),
    )
)


print(
    "\n========================================"
)

print(
    "GLOBAL PARAMETER RECOVERY"
)

print(
    "========================================"
)


display(
    global_parameter_summary
)


# ==========================================================
# 23. OPTIONAL JAWS RECOVERY
# ==========================================================

if not jaws_recovery.empty:

    jaws_recovery_summary = pd.DataFrame(
        [
            {
                "driver_replicates":
                    len(
                        jaws_recovery
                    ),
                "pearson":
                    safe_corr(
                        jaws_recovery[
                            "truth_jaws"
                        ],
                        jaws_recovery[
                            "posterior_mean_jaws"
                        ],
                        "pearson",
                    ),
                "spearman":
                    safe_corr(
                        jaws_recovery[
                            "truth_jaws"
                        ],
                        jaws_recovery[
                            "posterior_mean_jaws"
                        ],
                        "spearman",
                    ),
                "bias":
                    float(
                        jaws_recovery[
                            "error"
                        ].mean()
                    ),
                "rmse":
                    float(
                        np.sqrt(
                            np.mean(
                                jaws_recovery[
                                    "error"
                                ]
                                ** 2
                            )
                        )
                    ),
                "coverage_89":
                    float(
                        jaws_recovery[
                            "covered_89"
                        ].mean()
                    ),
            }
        ]
    )


    print(
        "\n========================================"
    )

    print(
        "OPTIONAL JAWS RECOVERY"
    )

    print(
        "========================================"
    )


    display(
        jaws_recovery_summary
    )


# ==========================================================
# 24. FIT HEALTH
# ==========================================================

print(
    "\n========================================"
)

print(
    "SYNTHETIC FIT HEALTH"
)

print(
    "========================================"
)


display(
    fit_health
)


# ==========================================================
# 25. RECOVERY PLOTS
# ==========================================================

plt.figure(
    figsize=(
        7,
        6,
    )
)


plt.scatter(
    alpha_recovery[
        "truth"
    ],
    alpha_recovery[
        "posterior_mean"
    ],
    alpha=0.35,
)


lo = min(
    alpha_recovery[
        "truth"
    ].min(),
    alpha_recovery[
        "posterior_mean"
    ].min(),
)


hi = max(
    alpha_recovery[
        "truth"
    ].max(),
    alpha_recovery[
        "posterior_mean"
    ].max(),
)


plt.plot(
    [
        lo,
        hi,
    ],
    [
        lo,
        hi,
    ],
    linestyle="--",
)


plt.xlabel(
    "True simulated driver-season alpha"
)


plt.ylabel(
    "Recovered posterior mean alpha"
)


plt.title(
    "Simulation recovery — driver-season alpha"
)


plt.grid(
    alpha=0.25
)


plt.tight_layout()

plt.show()


plt.figure(
    figsize=(
        7,
        6,
    )
)


plt.scatter(
    car_recovery[
        "truth"
    ],
    car_recovery[
        "posterior_mean"
    ],
    alpha=0.45,
)


lo = min(
    car_recovery[
        "truth"
    ].min(),
    car_recovery[
        "posterior_mean"
    ].min(),
)


hi = max(
    car_recovery[
        "truth"
    ].max(),
    car_recovery[
        "posterior_mean"
    ].max(),
)


plt.plot(
    [
        lo,
        hi,
    ],
    [
        lo,
        hi,
    ],
    linestyle="--",
)


plt.xlabel(
    "True simulated constructor-season beta"
)


plt.ylabel(
    "Recovered posterior mean beta"
)


plt.title(
    "Simulation recovery — constructor-season beta"
)


plt.grid(
    alpha=0.25
)


plt.tight_layout()

plt.show()


plt.figure(
    figsize=(
        7,
        6,
    )
)


coverage_labels = [
    "50%",
    "80%",
    "89%",
    "95%",
]


nominal = np.array(
    [
        0.50,
        0.80,
        0.89,
        0.95,
    ]
)


alpha_cov = np.array(
    [
        alpha_recovery[
            "covered_50"
        ].mean(),
        alpha_recovery[
            "covered_80"
        ].mean(),
        alpha_recovery[
            "covered_89"
        ].mean(),
        alpha_recovery[
            "covered_95"
        ].mean(),
    ]
)


car_cov = np.array(
    [
        car_recovery[
            "covered_50"
        ].mean(),
        car_recovery[
            "covered_80"
        ].mean(),
        car_recovery[
            "covered_89"
        ].mean(),
        car_recovery[
            "covered_95"
        ].mean(),
    ]
)


plt.plot(
    nominal,
    alpha_cov,
    marker="o",
    label="driver alpha",
)


plt.plot(
    nominal,
    car_cov,
    marker="o",
    label="constructor beta",
)


plt.plot(
    [
        0.45,
        1.0,
    ],
    [
        0.45,
        1.0,
    ],
    linestyle="--",
    label="nominal",
)


plt.xticks(
    nominal,
    coverage_labels,
)


plt.xlabel(
    "Nominal posterior interval"
)


plt.ylabel(
    "Empirical truth coverage"
)


plt.title(
    "Simulation recovery — posterior interval coverage"
)


plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ==========================================================
# 26. SAVE COMBINED OUTPUTS
# ==========================================================

alpha_recovery.to_csv(
    SIM_RECOVERY_DIR
    /
    "alpha_recovery_all_replicates.csv",
    index=False,
)


car_recovery.to_csv(
    SIM_RECOVERY_DIR
    /
    "car_recovery_all_replicates.csv",
    index=False,
)


teammate_recovery.to_csv(
    SIM_RECOVERY_DIR
    /
    "teammate_delta_recovery_all_replicates.csv",
    index=False,
)


global_recovery.to_csv(
    SIM_RECOVERY_DIR
    /
    "global_parameter_recovery_all_replicates.csv",
    index=False,
)


allocation_errors.to_csv(
    SIM_RECOVERY_DIR
    /
    "driver_car_allocation_errors_all_replicates.csv",
    index=False,
)


allocation_summary.to_csv(
    SIM_RECOVERY_DIR
    /
    "driver_car_allocation_summary.csv",
    index=False,
)


fit_health.to_csv(
    SIM_RECOVERY_DIR
    /
    "synthetic_fit_health.csv",
    index=False,
)


latent_recovery_summary.to_csv(
    SIM_RECOVERY_DIR
    /
    "latent_recovery_summary.csv",
    index=False,
)


alpha_recovery_by_era.to_csv(
    SIM_RECOVERY_DIR
    /
    "alpha_recovery_by_era.csv",
    index=False,
)


driver_rank_recovery.to_csv(
    SIM_RECOVERY_DIR
    /
    "driver_rank_recovery_by_season.csv",
    index=False,
)


car_rank_recovery.to_csv(
    SIM_RECOVERY_DIR
    /
    "car_rank_recovery_by_season.csv",
    index=False,
)


rank_recovery_summary.to_csv(
    SIM_RECOVERY_DIR
    /
    "rank_recovery_summary.csv",
    index=False,
)


teammate_recovery_summary.to_csv(
    SIM_RECOVERY_DIR
    /
    "teammate_delta_recovery_summary.csv",
    index=False,
)


global_parameter_summary.to_csv(
    SIM_RECOVERY_DIR
    /
    "global_parameter_recovery_summary.csv",
    index=False,
)


if not jaws_recovery.empty:

    jaws_recovery.to_csv(
        SIM_RECOVERY_DIR
        /
        "jaws_recovery_all_replicates.csv",
        index=False,
    )


    jaws_recovery_summary.to_csv(
        SIM_RECOVERY_DIR
        /
        "jaws_recovery_summary.csv",
        index=False,
    )


# ==========================================================
# 27. FINAL SUMMARY
# ==========================================================

alpha_pooled = latent_recovery_summary[
    (
        latent_recovery_summary[
            "replicate"
        ]
        ==
        "pooled"
    )
    &
    (
        latent_recovery_summary[
            "group"
        ]
        ==
        "driver alpha"
    )
].iloc[
    0
]


car_pooled = latent_recovery_summary[
    (
        latent_recovery_summary[
            "replicate"
        ]
        ==
        "pooled"
    )
    &
    (
        latent_recovery_summary[
            "group"
        ]
        ==
        "constructor beta"
    )
].iloc[
    0
]


tm = teammate_recovery_summary.iloc[
    0
]


print(
    "\n========================================"
)

print(
    "STAGE 24 SUMMARY"
)

print(
    "========================================"
)


print(
    f"Driver alpha recovery correlation: "
    f"{alpha_pooled['pearson']:.3f}"
)


print(
    f"Driver alpha RMSE: "
    f"{alpha_pooled['rmse']:.3f}"
)


print(
    f"Driver alpha 89% empirical coverage: "
    f"{alpha_pooled['coverage_89']:.3f}"
)


print(
    f"Constructor beta recovery correlation: "
    f"{car_pooled['pearson']:.3f}"
)


print(
    f"Constructor beta RMSE: "
    f"{car_pooled['rmse']:.3f}"
)


print(
    f"Constructor beta 89% empirical coverage: "
    f"{car_pooled['coverage_89']:.3f}"
)


print(
    f"Teammate delta-alpha recovery correlation: "
    f"{tm['pearson']:.3f}"
)


print(
    f"Teammate delta-alpha 89% coverage: "
    f"{tm['coverage_89']:.3f}"
)


print(
    f"Teammate ordering accuracy "
    f"(|true delta| > 0.1): "
    f"{tm['sign_accuracy_abs_truth_gt_0p1']:.3f}"
)


print(
    "\nDriver/car allocation error by replicate:"
)


display(
    allocation_summary[
        [
            "replicate",
            "corr_driver_error_vs_car_error",
            "rmse_driver_common_error",
            "rmse_car_error",
            "rmse_total_team_signal_error",
        ]
    ]
)


print(
    "\nInterpretation:"
)


print(
    "  High alpha/beta correlation + small RMSE:"
    " the model numerically recovers latent ability and car strength."
)


print(
    "  Empirical 89% coverage near 0.89:"
    " posterior state uncertainty is well calibrated in this"
    " realistic synthetic setting."
)


print(
    "  Coverage materially below 0.89:"
    " latent posterior intervals are too narrow even when the model"
    " is correctly specified."
)


print(
    "  Strong negative driver/car error correlation:"
    " residual driver-vs-car allocation ambiguity remains."
)


print(
    "  Small total-team-signal RMSE but larger component RMSEs:"
    " team performance is identified better than the exact split"
    " between driver and constructor."
)


print(
    "\nCaveat:"
)


print(
    "  This is realistic posterior-draw recovery, not formal prior SBC."
)


print(
    "  It tests inference/identification under the frozen model,"
    " not whether real F1 data violate that model."
)


print(
    "\nSaved validation outputs ->"
)


print(
    SIM_RECOVERY_DIR.resolve()
)


print(
    "\nStage 24 complete."
)


In [ ]:
# ==========================================================
# STAGE 25 — AUTOMATED WHOLE-NETWORK COMMUNITY / ANCHORING AUDIT
# ==========================================================
#
# PURPOSE
# -------
# Search the ENTIRE teammate graph for local communities that could
# be vulnerable to "cluster-level" anchoring problems.
#
# The concern is not whether teammates inside a cluster are ordered
# correctly. A weakly connected cluster can have good INTERNAL
# relative comparisons while its whole absolute alpha level is more
# dependent on one or two external bridges.
#
# This stage is diagnostic only. It does NOT change the model,
# the production posterior, alpha, JAWS, or the leaderboard.
#
# PRIMARY GRAPH
# -------------
# Nodes:
#   all modelled 1982–2025 drivers
#
# Edges:
#   actual shared race starts for the same constructor
#
# Community detection:
#   weighted Louvain on edges with >= 5 shared starts
#   (greedy modularity fallback if Louvain is unavailable)
#
# WHY 5+?
# -------
# A single substitute / one-off shared weekend should not make a
# historical pocket look strongly anchored. We still calculate
# threshold sensitivity at 1+, 5+, 10+, and 20+ shared starts.
#
# COMMUNITY FRAGILITY METRICS
# ---------------------------
# 1. weighted conductance
#      low = community is weakly attached relative to its internal volume
#
# 2. external edges per driver
#      low = few independent exits from the community
#
# 3. boundary-driver concentration
#      high = most external shared-start weight passes through one driver
#
# 4. boundary bridge-edge fraction
#      high = removing individual graph edges disconnects paths
#
# 5. posterior common-mode SD
#      high = community-year absolute alpha level is uncertain
#
# These are combined into a RELATIVE "anchoring fragility percentile".
# It is only a triage heuristic, not a statistical probability.
#
# ADDITIONAL CHECKS
# -----------------
# - community detection stability across Louvain seeds;
# - historical-state share;
# - internal posterior alpha correlation;
# - articulation points / graph bridges;
# - top-50 Pantheon drivers inside each community;
# - Stage 24 known-truth simulation recovery by community, if available.
#
# If a community is structurally fragile AND has worse simulation
# recovery, that is a strong candidate for a targeted stress-test refit.
# ==========================================================

from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

COMMUNITY_MIN_SHARED_STARTS = 5
CONNECTIVITY_THRESHOLDS = [1, 5, 10, 20]
COMMUNITY_SEED = 25042
N_COMMUNITY_STABILITY_RUNS = 20

N_COMMUNITIES_TO_SHOW = 12
N_TOP50_DRIVERS_TO_SHOW = 40

AUDIT_DIR = (
    Path(CFG.output_dir)
    / "network_community_anchoring_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==========================================================
# 1. REQUIRED OBJECTS
# ==========================================================

required_objects = [
    "actual_starts",
    "idx_all",
    "CFG",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the final model/scoring stages first. Missing: "
        + ", ".join(missing)
    )

if (
    "posterior_final" not in globals()
    and "final_trace_2025" not in globals()
):
    raise RuntimeError(
        "Could not find the accepted final posterior. "
        "Expected posterior_final or final_trace_2025."
    )


# ==========================================================
# 2. HELPERS
# ==========================================================

def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def stack_state_draws(posterior, var_name):
    da = posterior[var_name].stack(
        sample=("chain", "draw")
    )

    other_dims = [
        dim
        for dim in da.dims
        if dim != "sample"
    ]

    x = da.transpose(
        "sample",
        *other_dims,
    ).values

    return np.asarray(
        x,
        dtype=float,
    ).reshape(
        x.shape[0],
        -1,
    )


def safe_corr(x, y, method="pearson"):
    x = pd.Series(
        np.asarray(
            x,
            dtype=float,
        )
    )

    y = pd.Series(
        np.asarray(
            y,
            dtype=float,
        )
    )

    ok = x.notna() & y.notna()

    if ok.sum() < 3:
        return np.nan

    if x[ok].nunique() < 2 or y[ok].nunique() < 2:
        return np.nan

    return float(
        x[ok].corr(
            y[ok],
            method=method,
        )
    )


def pretty_driver(driver):
    return (
        str(driver)
        .replace("_", " ")
        .title()
    )


def weighted_mean_or_nan(values, weights):
    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    ok = (
        np.isfinite(values)
        &
        np.isfinite(weights)
        &
        (weights > 0)
    )

    if not ok.any():
        return np.nan

    return float(
        np.average(
            values[ok],
            weights=weights[ok],
        )
    )


# ==========================================================
# 3. ACCEPTED FULL-DATA POSTERIOR
# ==========================================================

if "posterior_final" in globals():
    final_posterior = posterior_final
else:
    final_posterior = final_trace_2025[
        "posterior"
    ].to_dataset()

alpha_draws = stack_state_draws(
    final_posterior,
    "alpha_dy",
)

alpha_state_mean = alpha_draws.mean(
    axis=0
)

alpha_state_sd = alpha_draws.std(
    axis=0,
    ddof=1,
)


# ==========================================================
# 4. DRIVER DISPLAY / PANTHEON METADATA
# ==========================================================

all_modelled_drivers = sorted(
    {
        str(driver)
        for driver, _year
        in idx_all.active_driver_years
    }
)

display_name_map = {
    driver: pretty_driver(driver)
    for driver in all_modelled_drivers
}

if (
    "raw" in globals()
    and hasattr(raw, "drivers")
):
    raw_drivers = raw.drivers.copy()

    if all(
        col in raw_drivers.columns
        for col in [
            "driverRef",
            "forename",
            "surname",
        ]
    ):
        for row in raw_drivers[
            [
                "driverRef",
                "forename",
                "surname",
            ]
        ].dropna(
            subset=["driverRef"]
        ).itertuples(
            index=False
        ):
            display_name_map[
                str(row.driverRef)
            ] = (
                f"{row.forename} {row.surname}"
            ).strip()


pantheon_meta = pd.DataFrame(
    {
        "driver": all_modelled_drivers,
    }
)

pantheon_meta["pantheon_rank"] = np.nan
pantheon_meta["jaws_mean"] = np.nan

if "pantheon_2025" in globals():
    p = pantheon_2025.copy()

    p_driver_col = first_existing_column(
        p,
        [
            "driver",
            "driver_ref",
            "driverRef",
        ],
    )

    p_rank_col = first_existing_column(
        p,
        [
            "rank",
            "official_rank",
            "jaws_rank",
        ],
    )

    p_jaws_col = first_existing_column(
        p,
        [
            "jaws_mean",
            "official_jaws",
            "jaws",
        ],
    )

    p_display_col = first_existing_column(
        p,
        [
            "display_name",
            "name",
            "driver_name",
        ],
    )

    if p_driver_col is not None:
        tmp = pd.DataFrame(
            {
                "driver":
                    p[p_driver_col].astype(str),
            }
        )

        tmp["pantheon_rank"] = (
            p[p_rank_col].to_numpy(dtype=float)
            if p_rank_col is not None
            else np.nan
        )

        tmp["jaws_mean"] = (
            p[p_jaws_col].to_numpy(dtype=float)
            if p_jaws_col is not None
            else np.nan
        )

        pantheon_meta = (
            pantheon_meta[["driver"]]
            .merge(
                tmp,
                on="driver",
                how="left",
                validate="one_to_one",
            )
        )

        if p_display_col is not None:
            for driver, name in zip(
                p[p_driver_col].astype(str),
                p[p_display_col],
            ):
                if pd.notna(name):
                    display_name_map[
                        driver
                    ] = str(name)


# ==========================================================
# 5. BUILD ACTUAL-START TEAMMATE EDGE TABLE
# ==========================================================

starts = (
    actual_starts[
        [
            "raceId",
            "year",
            "constructorId",
            "driver",
        ]
    ]
    .drop_duplicates()
    .copy()
)

starts["driver"] = starts[
    "driver"
].astype(str)

edge_accumulator = {}

for (
    race_id,
    year,
    constructor_id,
), grp in starts.groupby(
    [
        "raceId",
        "year",
        "constructorId",
    ]
):

    drivers = sorted(
        set(grp["driver"])
    )

    for a, b in combinations(
        drivers,
        2,
    ):
        key = (a, b)

        if key not in edge_accumulator:
            edge_accumulator[key] = {
                "driver_a": a,
                "driver_b": b,
                "shared_starts": 0,
                "years": set(),
                "constructors": set(),
            }

        edge_accumulator[key][
            "shared_starts"
        ] += 1

        edge_accumulator[key][
            "years"
        ].add(
            int(year)
        )

        edge_accumulator[key][
            "constructors"
        ].add(
            int(constructor_id)
        )


edge_rows = []

for value in edge_accumulator.values():
    edge_rows.append(
        {
            "driver_a":
                value["driver_a"],
            "driver_b":
                value["driver_b"],
            "shared_starts":
                int(
                    value[
                        "shared_starts"
                    ]
                ),
            "shared_seasons":
                len(
                    value["years"]
                ),
            "first_year":
                min(
                    value["years"]
                ),
            "last_year":
                max(
                    value["years"]
                ),
            "n_constructors":
                len(
                    value["constructors"]
                ),
        }
    )

edge_table = pd.DataFrame(
    edge_rows
)


print(
    "========================================"
)
print(
    "STAGE 25 — WHOLE-NETWORK COMMUNITY AUDIT"
)
print(
    "========================================"
)

print(
    "Modelled drivers:",
    len(all_modelled_drivers),
)

print(
    "All actual-start teammate edges:",
    len(edge_table),
)


# ==========================================================
# 6. BUILD THRESHOLDED GRAPHS
# ==========================================================

def build_graph(min_shared_starts):
    G = nx.Graph()

    G.add_nodes_from(
        all_modelled_drivers
    )

    x = edge_table[
        edge_table[
            "shared_starts"
        ]
        >=
        min_shared_starts
    ]

    for row in x.itertuples(
        index=False
    ):
        weight = float(
            row.shared_starts
        )

        G.add_edge(
            row.driver_a,
            row.driver_b,
            weight=weight,
            shared_starts=int(
                row.shared_starts
            ),
            distance=1.0 / weight,
            first_year=int(
                row.first_year
            ),
            last_year=int(
                row.last_year
            ),
        )

    return G


graphs = {
    threshold:
        build_graph(threshold)
    for threshold
    in CONNECTIVITY_THRESHOLDS
}

G = graphs[
    COMMUNITY_MIN_SHARED_STARTS
]

print(
    f"Primary {COMMUNITY_MIN_SHARED_STARTS}+ graph edges:",
    G.number_of_edges(),
)

print(
    f"Primary {COMMUNITY_MIN_SHARED_STARTS}+ connected components:",
    nx.number_connected_components(G),
)

component_sizes = sorted(
    [
        len(component)
        for component
        in nx.connected_components(G)
    ],
    reverse=True,
)

print(
    "Largest component sizes:",
    component_sizes[:10],
)


# ==========================================================
# 7. COMMUNITY DETECTION
# ==========================================================

def detect_communities(graph, seed):
    louvain = getattr(
        nx.community,
        "louvain_communities",
        None,
    )

    if louvain is not None:
        communities = louvain(
            graph,
            weight="weight",
            seed=seed,
            resolution=1.0,
        )

        method = "weighted Louvain"

    else:
        communities = list(
            nx.community.greedy_modularity_communities(
                graph,
                weight="weight",
            )
        )

        assigned = (
            set().union(*communities)
            if communities
            else set()
        )

        for node in graph.nodes:
            if node not in assigned:
                communities.append(
                    frozenset([node])
                )

        method = "weighted greedy modularity"

    return (
        [
            set(community)
            for community in communities
        ],
        method,
    )


primary_communities, community_method = (
    detect_communities(
        G,
        COMMUNITY_SEED,
    )
)

assigned = (
    set().union(*primary_communities)
    if primary_communities
    else set()
)

for node in G.nodes:
    if node not in assigned:
        primary_communities.append(
            {node}
        )

rank_lookup = dict(
    zip(
        pantheon_meta["driver"],
        pantheon_meta[
            "pantheon_rank"
        ],
    )
)


def community_best_rank(members):
    ranks = [
        rank_lookup.get(
            driver,
            np.nan,
        )
        for driver in members
    ]

    ranks = [
        float(x)
        for x in ranks
        if pd.notna(x)
    ]

    return (
        min(ranks)
        if ranks
        else np.inf
    )


primary_communities = sorted(
    primary_communities,
    key=lambda members: (
        -len(members),
        community_best_rank(members),
        sorted(members)[0],
    ),
)

driver_to_community = {}

for i, members in enumerate(
    primary_communities,
    start=1,
):
    cid = f"C{i:02d}"

    for driver in members:
        driver_to_community[
            driver
        ] = cid


print(
    "\nCommunity detection:",
    community_method,
)

print(
    "Communities:",
    len(primary_communities),
)

print(
    "Community sizes:",
    [
        len(c)
        for c in primary_communities
    ],
)


# ==========================================================
# 8. COMMUNITY STABILITY ACROSS LOUVAIN SEEDS
# ==========================================================

community_stability = {
    f"C{i:02d}": np.nan
    for i in range(
        1,
        len(primary_communities) + 1
    )
}

if (
    getattr(
        nx.community,
        "louvain_communities",
        None,
    )
    is not None
    and N_COMMUNITY_STABILITY_RUNS > 1
):

    restart_maps = []

    for run in range(
        N_COMMUNITY_STABILITY_RUNS
    ):
        communities_run = (
            nx.community.louvain_communities(
                G,
                weight="weight",
                seed=(
                    COMMUNITY_SEED
                    + 100
                    + run
                ),
                resolution=1.0,
            )
        )

        mapping = {}

        for j, members in enumerate(
            communities_run
        ):
            for driver in members:
                mapping[driver] = j

        for j, driver in enumerate(
            G.nodes
        ):
            if driver not in mapping:
                mapping[driver] = (
                    100000 + j
                )

        restart_maps.append(
            mapping
        )

    for i, members in enumerate(
        primary_communities,
        start=1,
    ):
        cid = f"C{i:02d}"

        members = sorted(
            members
        )

        if len(members) <= 1:
            community_stability[cid] = 1.0
            continue

        pair_scores = []

        for a, b in combinations(
            members,
            2,
        ):
            pair_scores.append(
                np.mean(
                    [
                        int(
                            mapping[a]
                            ==
                            mapping[b]
                        )
                        for mapping
                        in restart_maps
                    ]
                )
            )

        community_stability[cid] = float(
            np.mean(pair_scores)
        )


# ==========================================================
# 9. GLOBAL BRIDGES / ARTICULATION POINTS / CENTRALITY
# ==========================================================

bridge_edges = {
    tuple(sorted(edge))
    for edge in nx.bridges(G)
}

articulation_points = set(
    nx.articulation_points(G)
)

betweenness = nx.betweenness_centrality(
    G,
    weight="distance",
    normalized=True,
)


# ==========================================================
# 10. DRIVER POSTERIOR / CAREER SUMMARIES
# ==========================================================

states_by_driver = defaultdict(
    list
)

for state_idx, (
    driver,
    year,
) in enumerate(
    idx_all.active_driver_years
):
    states_by_driver[
        str(driver)
    ].append(
        (
            int(year),
            int(state_idx),
        )
    )


driver_posterior_rows = []

for driver in all_modelled_drivers:
    states = sorted(
        states_by_driver.get(
            driver,
            [],
        )
    )

    indices = [
        idx
        for _year, idx
        in states
    ]

    years = [
        year
        for year, _idx
        in states
    ]

    if indices:
        career_draws = alpha_draws[
            :,
            indices,
        ].mean(
            axis=1
        )

        career_mean = float(
            career_draws.mean()
        )

        career_sd = float(
            career_draws.std(
                ddof=1
            )
        )

        mean_season_sd = float(
            np.mean(
                alpha_state_sd[
                    indices
                ]
            )
        )

    else:
        career_mean = np.nan
        career_sd = np.nan
        mean_season_sd = np.nan

    driver_posterior_rows.append(
        {
            "driver":
                driver,
            "display_name":
                display_name_map[
                    driver
                ],
            "first_modelled_year":
                min(years)
                if years
                else np.nan,
            "last_modelled_year":
                max(years)
                if years
                else np.nan,
            "modelled_seasons":
                len(years),
            "historical_seasons":
                int(
                    np.sum(
                        np.asarray(years)
                        <=
                        1995
                    )
                )
                if years
                else 0,
            "career_mean_alpha":
                career_mean,
            "career_mean_alpha_sd":
                career_sd,
            "mean_season_alpha_sd":
                mean_season_sd,
        }
    )


driver_posterior = pd.DataFrame(
    driver_posterior_rows
)

starts_by_driver = (
    starts.groupby(
        "driver"
    )["raceId"]
    .nunique()
)

driver_posterior[
    "actual_starts"
] = (
    driver_posterior[
        "driver"
    ]
    .map(starts_by_driver)
    .fillna(0)
    .astype(int)
)


# ==========================================================
# 11. INTERNAL TEAMMATE PAIR-SEASON POSTERIOR CORRELATIONS
# ==========================================================

pair_season_rows = []

for (
    year,
    constructor_id,
), grp in starts.groupby(
    [
        "year",
        "constructorId",
    ]
):

    drivers = sorted(
        set(
            grp["driver"]
        )
    )

    for a, b in combinations(
        drivers,
        2,
    ):
        state_a = (
            a,
            int(year),
        )

        state_b = (
            b,
            int(year),
        )

        if (
            state_a not in idx_all.dy_map
            or state_b not in idx_all.dy_map
        ):
            continue

        races_a = set(
            grp.loc[
                grp["driver"] == a,
                "raceId",
            ]
        )

        races_b = set(
            grp.loc[
                grp["driver"] == b,
                "raceId",
            ]
        )

        shared_starts = len(
            races_a & races_b
        )

        if shared_starts <= 0:
            continue

        idx_a = idx_all.dy_map[
            state_a
        ]

        idx_b = idx_all.dy_map[
            state_b
        ]

        corr = float(
            np.corrcoef(
                alpha_draws[
                    :,
                    idx_a,
                ],
                alpha_draws[
                    :,
                    idx_b,
                ],
            )[0, 1]
        )

        pair_season_rows.append(
            {
                "driver_a": a,
                "driver_b": b,
                "year": int(year),
                "constructorId":
                    int(constructor_id),
                "shared_starts":
                    int(shared_starts),
                "alpha_corr": corr,
                "community_a":
                    driver_to_community[a],
                "community_b":
                    driver_to_community[b],
            }
        )


pair_season_corr = pd.DataFrame(
    pair_season_rows
)


# ==========================================================
# 12. THRESHOLD CONNECTIVITY HELPERS
# ==========================================================

largest_components = {}

for threshold, graph in graphs.items():
    components = list(
        nx.connected_components(
            graph
        )
    )

    largest_components[
        threshold
    ] = (
        max(
            components,
            key=len,
        )
        if components
        else set()
    )


# ==========================================================
# 13. COMMUNITY COMMON-MODE POSTERIOR UNCERTAINTY
# ==========================================================

def community_common_mode_stats(members):
    by_year = defaultdict(
        list
    )

    for driver in members:
        for year, state_idx in (
            states_by_driver.get(
                driver,
                [],
            )
        ):
            by_year[
                int(year)
            ].append(
                int(state_idx)
            )

    yearly_sds = []

    for year, indices in by_year.items():
        if len(indices) < 2:
            continue

        common_draw = alpha_draws[
            :,
            indices,
        ].mean(
            axis=1
        )

        yearly_sds.append(
            float(
                common_draw.std(
                    ddof=1
                )
            )
        )

    if not yearly_sds:
        return (
            np.nan,
            np.nan,
            0,
        )

    return (
        float(
            np.mean(yearly_sds)
        ),
        float(
            np.max(yearly_sds)
        ),
        len(yearly_sds),
    )


# ==========================================================
# 14. COMMUNITY STRUCTURAL METRICS
# ==========================================================

total_graph_volume = sum(
    dict(
        G.degree(
            weight="weight",
        )
    ).values()
)

largest_component_5 = (
    largest_components[
        COMMUNITY_MIN_SHARED_STARTS
    ]
)

community_rows = []

for i, members in enumerate(
    primary_communities,
    start=1,
):

    cid = f"C{i:02d}"
    members = set(members)

    internal_edges = []
    boundary_edges = []

    for a, b, data in G.edges(
        data=True
    ):
        a_in = a in members
        b_in = b in members

        if a_in and b_in:
            internal_edges.append(
                (a, b, data)
            )

        elif a_in != b_in:
            boundary_edges.append(
                (a, b, data)
            )

    internal_weight = float(
        sum(
            data["weight"]
            for _a, _b, data
            in internal_edges
        )
    )

    boundary_weight = float(
        sum(
            data["weight"]
            for _a, _b, data
            in boundary_edges
        )
    )

    weighted_degree = dict(
        G.degree(
            members,
            weight="weight",
        )
    )

    volume_s = float(
        sum(
            weighted_degree.values()
        )
    )

    volume_t = float(
        total_graph_volume
        -
        volume_s
    )

    conductance_denominator = min(
        volume_s,
        volume_t,
    )

    conductance = (
        boundary_weight
        /
        conductance_denominator
        if conductance_denominator > 0
        else 0.0
    )

    boundary_internal_weight_by_driver = defaultdict(
        float
    )

    boundary_external_neighbors = set()
    boundary_internal_drivers = set()

    for a, b, data in boundary_edges:
        if a in members:
            inside = a
            outside = b
        else:
            inside = b
            outside = a

        boundary_internal_weight_by_driver[
            inside
        ] += float(
            data["weight"]
        )

        boundary_internal_drivers.add(
            inside
        )

        boundary_external_neighbors.add(
            outside
        )

    if (
        boundary_weight > 0
        and boundary_internal_weight_by_driver
    ):
        boundary_driver_concentration = (
            max(
                boundary_internal_weight_by_driver.values()
            )
            /
            boundary_weight
        )
    else:
        boundary_driver_concentration = 1.0

    boundary_bridge_count = sum(
        tuple(
            sorted(
                (a, b)
            )
        )
        in bridge_edges
        for a, b, _data
        in boundary_edges
    )

    boundary_bridge_fraction = (
        boundary_bridge_count
        /
        len(boundary_edges)
        if boundary_edges
        else 1.0
    )

    historical_state_count = 0
    total_state_count = 0

    for driver in members:
        states_driver = (
            states_by_driver.get(
                driver,
                [],
            )
        )

        total_state_count += len(
            states_driver
        )

        historical_state_count += sum(
            year <= 1995
            for year, _idx
            in states_driver
        )

    historical_share = (
        historical_state_count
        /
        total_state_count
        if total_state_count > 0
        else np.nan
    )

    (
        common_mode_mean_sd,
        common_mode_max_sd,
        common_mode_years,
    ) = community_common_mode_stats(
        members
    )

    internal_pair_rows = pair_season_corr[
        (
            pair_season_corr[
                "community_a"
            ]
            ==
            cid
        )
        &
        (
            pair_season_corr[
                "community_b"
            ]
            ==
            cid
        )
    ]

    mean_internal_alpha_corr = (
        weighted_mean_or_nan(
            internal_pair_rows[
                "alpha_corr"
            ],
            internal_pair_rows[
                "shared_starts"
            ],
        )
        if len(internal_pair_rows)
        else np.nan
    )

    member_ranks = pantheon_meta[
        pantheon_meta[
            "driver"
        ].isin(
            members
        )
    ].copy()

    ranked_members = member_ranks[
        member_ranks[
            "pantheon_rank"
        ].notna()
    ].sort_values(
        "pantheon_rank"
    )

    top_rank = (
        float(
            ranked_members[
                "pantheon_rank"
            ].iloc[0]
        )
        if len(ranked_members)
        else np.nan
    )

    top_rank_driver = (
        ranked_members[
            "driver"
        ].iloc[0]
        if len(ranked_members)
        else None
    )

    top50_members = ranked_members[
        ranked_members[
            "pantheon_rank"
        ]
        <=
        50
    ]

    top50_label = ", ".join(
        [
            (
                f"#{int(row.pantheon_rank)} "
                f"{display_name_map[row.driver]}"
            )
            for row
            in top50_members.itertuples(
                index=False
            )
        ][:8]
    )

    component_nodes = (
        nx.node_connected_component(
            G,
            next(iter(members)),
        )
        if members
        else set()
    )

    row = {
        "community": cid,
        "n_drivers":
            len(members),
        "drivers":
            ", ".join(
                sorted(
                    [
                        display_name_map[
                            driver
                        ]
                        for driver
                        in members
                    ]
                )
            ),
        "top_pantheon_rank":
            top_rank,
        "top_rank_driver":
            (
                display_name_map[
                    top_rank_driver
                ]
                if top_rank_driver
                is not None
                else ""
            ),
        "n_top50":
            len(top50_members),
        "top50_members":
            top50_label,
        "internal_edges":
            len(internal_edges),
        "internal_shared_start_weight":
            internal_weight,
        "external_edges":
            len(boundary_edges),
        "external_shared_start_weight":
            boundary_weight,
        "external_edges_per_driver":
            (
                len(boundary_edges)
                /
                len(members)
                if members
                else np.nan
            ),
        "external_neighbors":
            len(
                boundary_external_neighbors
            ),
        "boundary_drivers":
            len(
                boundary_internal_drivers
            ),
        "boundary_driver_fraction":
            (
                len(
                    boundary_internal_drivers
                )
                /
                len(members)
                if members
                else np.nan
            ),
        "boundary_driver_concentration":
            boundary_driver_concentration,
        "boundary_bridge_edges":
            boundary_bridge_count,
        "boundary_bridge_fraction":
            boundary_bridge_fraction,
        "weighted_conductance":
            conductance,
        "historical_state_share":
            historical_share,
        "posterior_common_mode_mean_sd":
            common_mode_mean_sd,
        "posterior_common_mode_max_sd":
            common_mode_max_sd,
        "common_mode_years":
            common_mode_years,
        "mean_internal_alpha_corr":
            mean_internal_alpha_corr,
        "community_stability":
            community_stability[
                cid
            ],
        "component_size_5":
            len(component_nodes),
        "in_largest_component_5":
            int(
                bool(
                    members
                    &
                    largest_component_5
                )
            ),
    }

    for threshold in (
        CONNECTIVITY_THRESHOLDS
    ):
        graph_t = graphs[
            threshold
        ]

        largest_t = largest_components[
            threshold
        ]

        row[
            f"member_fraction_in_lcc_{threshold}"
        ] = (
            len(
                members
                &
                largest_t
            )
            /
            len(members)
            if members
            else np.nan
        )

        subgraph_t = graph_t.subgraph(
            members
        )

        row[
            f"internal_components_{threshold}"
        ] = (
            nx.number_connected_components(
                subgraph_t
            )
            if len(members)
            else 0
        )

    community_rows.append(
        row
    )


community_summary = pd.DataFrame(
    community_rows
)


# ==========================================================
# 15. RELATIVE ANCHORING FRAGILITY INDEX
# ==========================================================
#
# Higher = more structurally fragile.
#
#   35% weak conductance
#   25% few external edges per driver
#   20% external weight concentrated through one driver
#   10% boundary edges that are graph bridges
#   10% posterior common-mode uncertainty
#
# Historical share is reported but NOT included.
# ==========================================================

risk_population_mask = (
    community_summary[
        "n_drivers"
    ]
    >=
    2
)

risk_population = (
    community_summary.loc[
        risk_population_mask
    ]
    .copy()
)

risk_population[
    "risk_weak_conductance"
] = (
    risk_population[
        "weighted_conductance"
    ]
    .rank(
        pct=True,
        ascending=False,
        method="average",
    )
)

risk_population[
    "risk_few_external_edges"
] = (
    risk_population[
        "external_edges_per_driver"
    ]
    .rank(
        pct=True,
        ascending=False,
        method="average",
    )
)

risk_population[
    "risk_boundary_concentration"
] = (
    risk_population[
        "boundary_driver_concentration"
    ]
    .rank(
        pct=True,
        ascending=True,
        method="average",
    )
)

risk_population[
    "risk_bridge_fraction"
] = (
    risk_population[
        "boundary_bridge_fraction"
    ]
    .rank(
        pct=True,
        ascending=True,
        method="average",
    )
)

risk_population[
    "risk_common_mode_sd"
] = (
    risk_population[
        "posterior_common_mode_mean_sd"
    ]
    .rank(
        pct=True,
        ascending=True,
        method="average",
    )
)

risk_population[
    "anchoring_fragility_index"
] = (
    100.0
    *
    (
        0.35
        *
        risk_population[
            "risk_weak_conductance"
        ]
        +
        0.25
        *
        risk_population[
            "risk_few_external_edges"
        ]
        +
        0.20
        *
        risk_population[
            "risk_boundary_concentration"
        ]
        +
        0.10
        *
        risk_population[
            "risk_bridge_fraction"
        ]
        +
        0.10
        *
        risk_population[
            "risk_common_mode_sd"
        ]
    )
)

risk_population[
    "anchoring_fragility_percentile"
] = (
    100.0
    *
    risk_population[
        "anchoring_fragility_index"
    ]
    .rank(
        pct=True,
        ascending=True,
        method="average",
    )
)

risk_population[
    "fragility_band"
] = pd.cut(
    risk_population[
        "anchoring_fragility_percentile"
    ],
    bins=[
        -np.inf,
        65,
        85,
        np.inf,
    ],
    labels=[
        "LOW",
        "WATCH",
        "HIGH",
    ],
)

community_summary = (
    community_summary
    .merge(
        risk_population[
            [
                "community",
                "risk_weak_conductance",
                "risk_few_external_edges",
                "risk_boundary_concentration",
                "risk_bridge_fraction",
                "risk_common_mode_sd",
                "anchoring_fragility_index",
                "anchoring_fragility_percentile",
                "fragility_band",
            ]
        ],
        on="community",
        how="left",
        validate="one_to_one",
    )
)

community_summary = (
    community_summary
    .sort_values(
        [
            "anchoring_fragility_percentile",
            "top_pantheon_rank",
        ],
        ascending=[
            False,
            True,
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)


# ==========================================================
# 16. DRIVER-LEVEL ANCHOR / BRIDGE DIAGNOSTICS
# ==========================================================

driver_rows = []

for driver in all_modelled_drivers:
    cid = driver_to_community[
        driver
    ]

    incident = list(
        G.edges(
            driver,
            data=True,
        )
    )

    weighted_degree_5 = float(
        sum(
            data["weight"]
            for _a, _b, data
            in incident
        )
    )

    external_incident = []
    bridge_incident = 0

    for a, b, data in incident:
        other = (
            b
            if a == driver
            else a
        )

        if (
            driver_to_community[
                other
            ]
            !=
            cid
        ):
            external_incident.append(
                (other, data)
            )

        if (
            tuple(
                sorted(
                    (a, b)
                )
            )
            in bridge_edges
        ):
            bridge_incident += 1

    external_weight = float(
        sum(
            data["weight"]
            for _other, data
            in external_incident
        )
    )

    driver_rows.append(
        {
            "driver":
                driver,
            "display_name":
                display_name_map[
                    driver
                ],
            "community":
                cid,
            "degree_5":
                int(
                    G.degree(driver)
                ),
            "weighted_degree_5":
                weighted_degree_5,
            "external_degree_5":
                len(
                    external_incident
                ),
            "external_weight_5":
                external_weight,
            "external_weight_fraction_5":
                (
                    external_weight
                    /
                    weighted_degree_5
                    if weighted_degree_5 > 0
                    else 0.0
                ),
            "is_articulation_point_5":
                int(
                    driver
                    in articulation_points
                ),
            "incident_bridge_fraction_5":
                (
                    bridge_incident
                    /
                    len(incident)
                    if incident
                    else np.nan
                ),
            "betweenness_5":
                float(
                    betweenness.get(
                        driver,
                        0.0,
                    )
                ),
        }
    )


driver_network_audit = (
    pd.DataFrame(
        driver_rows
    )
    .merge(
        driver_posterior,
        on=[
            "driver",
            "display_name",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        pantheon_meta,
        on="driver",
        how="left",
        validate="one_to_one",
    )
    .merge(
        community_summary[
            [
                "community",
                "n_drivers",
                "weighted_conductance",
                "historical_state_share",
                "posterior_common_mode_mean_sd",
                "mean_internal_alpha_corr",
                "community_stability",
                "anchoring_fragility_index",
                "anchoring_fragility_percentile",
                "fragility_band",
            ]
        ],
        on="community",
        how="left",
        validate="many_to_one",
    )
)


# ==========================================================
# 17. OPTIONAL STAGE 24 KNOWN-TRUTH RECOVERY BY COMMUNITY
# ==========================================================

stage24_path = (
    Path(CFG.output_dir)
    /
    "validation_simulation_latent_recovery"
    /
    "alpha_recovery_all_replicates.csv"
)

simulation_community_recovery = (
    pd.DataFrame()
)

if stage24_path.exists():
    sim_alpha = pd.read_csv(
        stage24_path
    )

    if all(
        col in sim_alpha.columns
        for col in [
            "driver",
            "error",
            "covered_89",
        ]
    ):
        sim_alpha["driver"] = (
            sim_alpha["driver"]
            .astype(str)
        )

        sim_alpha[
            "community"
        ] = sim_alpha[
            "driver"
        ].map(
            driver_to_community
        )

        sim_alpha = sim_alpha[
            sim_alpha[
                "community"
            ].notna()
        ].copy()

        simulation_community_recovery = (
            sim_alpha.groupby(
                "community",
                as_index=False,
            )
            .agg(
                sim_states=(
                    "error",
                    "size",
                ),
                sim_alpha_bias=(
                    "error",
                    "mean",
                ),
                sim_alpha_mae=(
                    "error",
                    lambda x:
                        float(
                            np.mean(
                                np.abs(
                                    np.asarray(
                                        x,
                                        dtype=float,
                                    )
                                )
                            )
                        ),
                ),
                sim_alpha_rmse=(
                    "error",
                    lambda x:
                        float(
                            np.sqrt(
                                np.mean(
                                    np.asarray(
                                        x,
                                        dtype=float,
                                    )
                                    ** 2
                                )
                            )
                        ),
                ),
                sim_alpha_coverage89=(
                    "covered_89",
                    "mean",
                ),
            )
        )

        community_summary = (
            community_summary
            .merge(
                simulation_community_recovery,
                on="community",
                how="left",
                validate="one_to_one",
            )
        )

        driver_network_audit = (
            driver_network_audit
            .merge(
                simulation_community_recovery[
                    [
                        "community",
                        "sim_alpha_rmse",
                        "sim_alpha_coverage89",
                    ]
                ],
                on="community",
                how="left",
                validate="many_to_one",
            )
        )


# ==========================================================
# 18. COMMUNITY META-GRAPH
# ==========================================================

community_graph = nx.Graph()

for row in community_summary.itertuples(
    index=False
):
    community_graph.add_node(
        row.community,
        n_drivers=int(
            row.n_drivers
        ),
        fragility=(
            float(
                row.anchoring_fragility_percentile
            )
            if pd.notna(
                row.anchoring_fragility_percentile
            )
            else 0.0
        ),
    )


community_edge_weights = defaultdict(
    float
)

for row in edge_table[
    edge_table[
        "shared_starts"
    ]
    >=
    COMMUNITY_MIN_SHARED_STARTS
].itertuples(
    index=False
):
    ca = driver_to_community[
        row.driver_a
    ]

    cb = driver_to_community[
        row.driver_b
    ]

    if ca == cb:
        continue

    key = tuple(
        sorted(
            (ca, cb)
        )
    )

    community_edge_weights[
        key
    ] += float(
        row.shared_starts
    )


for (
    ca,
    cb,
), weight in (
    community_edge_weights.items()
):
    community_graph.add_edge(
        ca,
        cb,
        weight=weight,
        distance=1.0 / weight,
    )


# ==========================================================
# 19. PRINT COMMUNITY TABLE
# ==========================================================

community_display_cols = [
    "community",
    "n_drivers",
    "top_pantheon_rank",
    "top_rank_driver",
    "n_top50",
    "top50_members",
    "weighted_conductance",
    "external_edges",
    "external_neighbors",
    "boundary_drivers",
    "boundary_driver_concentration",
    "boundary_bridge_fraction",
    "historical_state_share",
    "posterior_common_mode_mean_sd",
    "mean_internal_alpha_corr",
    "community_stability",
    "anchoring_fragility_percentile",
    "fragility_band",
]

if (
    "sim_alpha_rmse"
    in community_summary.columns
):
    community_display_cols.extend(
        [
            "sim_alpha_rmse",
            "sim_alpha_coverage89",
        ]
    )


print(
    "\n========================================"
)
print(
    "COMMUNITIES — HIGHEST ANCHORING FRAGILITY"
)
print(
    "========================================"
)

display(
    community_summary[
        community_display_cols
    ]
    .head(
        N_COMMUNITIES_TO_SHOW
    )
)


# ==========================================================
# 20. TOP-50 PANTHEON DRIVERS IN FRAGILE COMMUNITIES
# ==========================================================

top50_risk = (
    driver_network_audit[
        driver_network_audit[
            "pantheon_rank"
        ]
        <=
        50
    ]
    .copy()
    .sort_values(
        [
            "anchoring_fragility_percentile",
            "pantheon_rank",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

driver_display_cols = [
    "pantheon_rank",
    "display_name",
    "jaws_mean",
    "community",
    "fragility_band",
    "anchoring_fragility_percentile",
    "degree_5",
    "weighted_degree_5",
    "external_degree_5",
    "external_weight_fraction_5",
    "is_articulation_point_5",
    "incident_bridge_fraction_5",
    "betweenness_5",
    "career_mean_alpha",
    "career_mean_alpha_sd",
    "historical_seasons",
    "modelled_seasons",
]

if (
    "sim_alpha_rmse"
    in top50_risk.columns
):
    driver_display_cols.extend(
        [
            "sim_alpha_rmse",
            "sim_alpha_coverage89",
        ]
    )


print(
    "\n========================================"
)
print(
    "PANTHEON TOP-50 — MOST STRUCTURALLY FRAGILE FIRST"
)
print(
    "========================================"
)

display(
    top50_risk[
        driver_display_cols
    ]
    .head(
        N_TOP50_DRIVERS_TO_SHOW
    )
)


# ==========================================================
# 21. EXTERNAL BRIDGE EDGES FOR HIGH-RISK COMMUNITIES
# ==========================================================

high_risk_communities = set(
    community_summary.loc[
        community_summary[
            "fragility_band"
        ]
        ==
        "HIGH",
        "community",
    ]
)

boundary_rows = []

for row in edge_table[
    edge_table[
        "shared_starts"
    ]
    >=
    COMMUNITY_MIN_SHARED_STARTS
].itertuples(
    index=False
):
    ca = driver_to_community[
        row.driver_a
    ]

    cb = driver_to_community[
        row.driver_b
    ]

    if ca == cb:
        continue

    if (
        ca not in high_risk_communities
        and cb not in high_risk_communities
    ):
        continue

    boundary_rows.append(
        {
            "driver_a":
                display_name_map[
                    row.driver_a
                ],
            "community_a":
                ca,
            "driver_b":
                display_name_map[
                    row.driver_b
                ],
            "community_b":
                cb,
            "shared_starts":
                int(
                    row.shared_starts
                ),
            "first_year":
                int(
                    row.first_year
                ),
            "last_year":
                int(
                    row.last_year
                ),
            "is_global_bridge_5":
                int(
                    tuple(
                        sorted(
                            (
                                row.driver_a,
                                row.driver_b,
                            )
                        )
                    )
                    in bridge_edges
                ),
        }
    )


boundary_edge_audit = (
    pd.DataFrame(
        boundary_rows
    )
    if boundary_rows
    else pd.DataFrame(
        columns=[
            "driver_a",
            "community_a",
            "driver_b",
            "community_b",
            "shared_starts",
            "first_year",
            "last_year",
            "is_global_bridge_5",
        ]
    )
)

if len(
    boundary_edge_audit
):
    boundary_edge_audit = (
        boundary_edge_audit
        .sort_values(
            [
                "is_global_bridge_5",
                "shared_starts",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )

print(
    "\n========================================"
)
print(
    "EXTERNAL EDGES OF HIGH-FRAGILITY COMMUNITIES"
)
print(
    "========================================"
)

display(
    boundary_edge_audit.head(
        50
    )
)


# ==========================================================
# 22. DOES STRUCTURAL FRAGILITY PREDICT STAGE-24 RECOVERY ERROR?
# ==========================================================

if (
    "sim_alpha_rmse"
    in community_summary.columns
):
    valid_sim = (
        community_summary[
            community_summary[
                "anchoring_fragility_percentile"
            ].notna()
            &
            community_summary[
                "sim_alpha_rmse"
            ].notna()
        ]
        .copy()
    )

    print(
        "\n========================================"
    )
    print(
        "STRUCTURAL FRAGILITY vs KNOWN-TRUTH SIMULATION RECOVERY"
    )
    print(
        "========================================"
    )

    print(
        "Spearman corr(fragility percentile, simulation alpha RMSE):",
        safe_corr(
            valid_sim[
                "anchoring_fragility_percentile"
            ],
            valid_sim[
                "sim_alpha_rmse"
            ],
            "spearman",
        ),
    )

    print(
        "Spearman corr(fragility percentile, simulation 89% coverage):",
        safe_corr(
            valid_sim[
                "anchoring_fragility_percentile"
            ],
            valid_sim[
                "sim_alpha_coverage89"
            ],
            "spearman",
        ),
    )

    risk_q75 = valid_sim[
        "anchoring_fragility_percentile"
    ].quantile(
        0.75
    )

    sim_risk_comparison = pd.DataFrame(
        [
            {
                "group":
                    "top fragility quartile",
                "communities":
                    int(
                        (
                            valid_sim[
                                "anchoring_fragility_percentile"
                            ]
                            >=
                            risk_q75
                        ).sum()
                    ),
                "mean_sim_alpha_rmse":
                    valid_sim.loc[
                        valid_sim[
                            "anchoring_fragility_percentile"
                        ]
                        >=
                        risk_q75,
                        "sim_alpha_rmse",
                    ].mean(),
                "mean_sim_coverage89":
                    valid_sim.loc[
                        valid_sim[
                            "anchoring_fragility_percentile"
                        ]
                        >=
                        risk_q75,
                        "sim_alpha_coverage89",
                    ].mean(),
            },
            {
                "group":
                    "lower three quartiles",
                "communities":
                    int(
                        (
                            valid_sim[
                                "anchoring_fragility_percentile"
                            ]
                            <
                            risk_q75
                        ).sum()
                    ),
                "mean_sim_alpha_rmse":
                    valid_sim.loc[
                        valid_sim[
                            "anchoring_fragility_percentile"
                        ]
                        <
                        risk_q75,
                        "sim_alpha_rmse",
                    ].mean(),
                "mean_sim_coverage89":
                    valid_sim.loc[
                        valid_sim[
                            "anchoring_fragility_percentile"
                        ]
                        <
                        risk_q75,
                        "sim_alpha_coverage89",
                    ].mean(),
            },
        ]
    )

    display(
        sim_risk_comparison
    )


# ==========================================================
# 23. COMMUNITY META-GRAPH PLOT
# ==========================================================

plt.figure(
    figsize=(
        12,
        9,
    )
)

if (
    community_graph.number_of_nodes()
    >
    0
):
    pos = nx.spring_layout(
        community_graph,
        seed=COMMUNITY_SEED,
        weight="weight",
        k=1.2,
    )

    node_ids = list(
        community_graph.nodes()
    )

    node_sizes = [
        500
        +
        140
        *
        community_graph.nodes[
            node
        ][
            "n_drivers"
        ]
        for node in node_ids
    ]

    node_fragility = [
        community_graph.nodes[
            node
        ][
            "fragility"
        ]
        for node in node_ids
    ]

    edge_widths = [
        0.8
        +
        1.2
        *
        np.log1p(
            community_graph[
                a
            ][
                b
            ][
                "weight"
            ]
        )
        for a, b
        in community_graph.edges()
    ]

    nodes_artist = nx.draw_networkx_nodes(
        community_graph,
        pos,
        nodelist=node_ids,
        node_size=node_sizes,
        node_color=node_fragility,
        cmap="viridis",
        vmin=0,
        vmax=100,
        alpha=0.90,
    )

    nx.draw_networkx_edges(
        community_graph,
        pos,
        width=edge_widths,
        alpha=0.45,
    )

    meta_labels = {}

    for cid in node_ids:
        row = community_summary[
            community_summary[
                "community"
            ]
            ==
            cid
        ].iloc[0]

        top_driver = (
            row["top_rank_driver"]
            if row["top_rank_driver"]
            else ""
        )

        meta_labels[cid] = (
            f"{cid}\n"
            f"{top_driver}\n"
            f"n={int(row['n_drivers'])}"
        )

    nx.draw_networkx_labels(
        community_graph,
        pos,
        labels=meta_labels,
        font_size=8,
    )

    cbar = plt.colorbar(
        nodes_artist,
        shrink=0.75,
    )

    cbar.set_label(
        "Anchoring fragility percentile"
    )

plt.title(
    "Pantheon teammate communities — structural anchoring audit\n"
    "node size = drivers, node colour = relative fragility, "
    "edge width = shared-start weight between communities"
)

plt.axis("off")
plt.tight_layout()

COMMUNITY_META_PLOT_PATH = (
    AUDIT_DIR
    /
    "community_meta_graph.png"
)

plt.savefig(
    COMMUNITY_META_PLOT_PATH,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ==========================================================
# 24. SAVE GRAPH / TABLE OUTPUTS
# ==========================================================

COMMUNITY_SUMMARY_PATH = (
    AUDIT_DIR
    /
    "community_summary.csv"
)

DRIVER_AUDIT_PATH = (
    AUDIT_DIR
    /
    "driver_network_audit.csv"
)

EDGE_TABLE_PATH = (
    AUDIT_DIR
    /
    "teammate_edge_table.csv"
)

PAIR_CORR_PATH = (
    AUDIT_DIR
    /
    "pair_season_alpha_correlations.csv"
)

BOUNDARY_EDGE_PATH = (
    AUDIT_DIR
    /
    "high_fragility_boundary_edges.csv"
)

community_summary.to_csv(
    COMMUNITY_SUMMARY_PATH,
    index=False,
)

driver_network_audit.to_csv(
    DRIVER_AUDIT_PATH,
    index=False,
)

edge_table.to_csv(
    EDGE_TABLE_PATH,
    index=False,
)

pair_season_corr.to_csv(
    PAIR_CORR_PATH,
    index=False,
)

boundary_edge_audit.to_csv(
    BOUNDARY_EDGE_PATH,
    index=False,
)


# GraphML export for optional external exploration.
for node in G.nodes:
    cid = driver_to_community[
        node
    ]

    driver_row = (
        driver_network_audit[
            driver_network_audit[
                "driver"
            ]
            ==
            node
        ]
        .iloc[0]
    )

    G.nodes[node][
        "display_name"
    ] = display_name_map[
        node
    ]

    G.nodes[node][
        "community"
    ] = cid

    G.nodes[node][
        "pantheon_rank"
    ] = (
        float(
            driver_row[
                "pantheon_rank"
            ]
        )
        if pd.notna(
            driver_row[
                "pantheon_rank"
            ]
        )
        else -1.0
    )

    G.nodes[node][
        "jaws_mean"
    ] = (
        float(
            driver_row[
                "jaws_mean"
            ]
        )
        if pd.notna(
            driver_row[
                "jaws_mean"
            ]
        )
        else -999.0
    )

    G.nodes[node][
        "career_mean_alpha"
    ] = (
        float(
            driver_row[
                "career_mean_alpha"
            ]
        )
        if pd.notna(
            driver_row[
                "career_mean_alpha"
            ]
        )
        else -999.0
    )

    G.nodes[node][
        "fragility_percentile"
    ] = (
        float(
            driver_row[
                "anchoring_fragility_percentile"
            ]
        )
        if pd.notna(
            driver_row[
                "anchoring_fragility_percentile"
            ]
        )
        else -1.0
    )

    G.nodes[node][
        "articulation_point"
    ] = int(
        driver_row[
            "is_articulation_point_5"
        ]
    )

GRAPHML_PATH = (
    AUDIT_DIR
    /
    (
        f"teammate_network_"
        f"{COMMUNITY_MIN_SHARED_STARTS}plus.graphml"
    )
)

nx.write_graphml(
    G,
    GRAPHML_PATH,
)


# ==========================================================
# 25. FINAL TRIAGE SUMMARY
# ==========================================================

top_fragile = (
    community_summary[
        community_summary[
            "anchoring_fragility_percentile"
        ].notna()
    ]
    .head(5)
)

print(
    "\n========================================"
)
print(
    "STAGE 25 TRIAGE SUMMARY"
)
print(
    "========================================"
)

print(
    "Primary graph threshold:",
    f"{COMMUNITY_MIN_SHARED_STARTS}+ shared starts",
)

print(
    "Community method:",
    community_method,
)

print(
    "High-fragility communities:",
    int(
        (
            community_summary[
                "fragility_band"
            ]
            ==
            "HIGH"
        ).sum()
    ),
)

print(
    "\nHighest-fragility communities:"
)

for row in top_fragile.itertuples(
    index=False
):
    sim_text = ""

    if (
        hasattr(
            row,
            "sim_alpha_rmse",
        )
        and pd.notna(
            row.sim_alpha_rmse
        )
    ):
        sim_text = (
            f", sim RMSE={row.sim_alpha_rmse:.3f}, "
            f"sim cov89={row.sim_alpha_coverage89:.3f}"
        )

    print(
        f"  {row.community}: "
        f"risk pctl={row.anchoring_fragility_percentile:.1f}, "
        f"n={row.n_drivers}, "
        f"conductance={row.weighted_conductance:.3f}, "
        f"external edges={row.external_edges}, "
        f"boundary concentration={row.boundary_driver_concentration:.3f}, "
        f"historical share={row.historical_state_share:.3f}"
        f"{sim_text}"
    )

    if row.top50_members:
        print(
            "      Top-50:",
            row.top50_members,
        )

print(
    "\nInterpretation:"
)

print(
    "  HIGH fragility does NOT mean a community is biased."
)

print(
    "  It means its absolute level depends more heavily on a small"
    " number of external anchors and deserves targeted inspection."
)

print(
    "  Low conductance + high boundary concentration is the clearest"
    " structural warning sign."
)

print(
    "  If Stage 24 simulation RMSE is also high / coverage low for the"
    " same community, that community becomes a strong stress-test target."
)

print(
    "  Community stability near 1 means the same grouping appears across"
    " Louvain seeds; low stability means the boundary is algorithmically"
    " ambiguous rather than a clearly isolated racing cohort."
)

print(
    "\nSaved outputs ->"
)

print(
    AUDIT_DIR.resolve()
)

print(
    "\nStage 25 complete."
)


In [ ]:
# ==========================================================
# STAGE 26 — REAL-DATA HETEROSKEDASTICITY / LOW-CAR VARIANCE AUDIT
# ==========================================================
#
# QUESTION
# --------
# Do weaker constructor-seasons produce systematically noisier
# observations than stronger constructor-seasons?
#
# This directly tests the criticism:
#
#   "The model assumes one residual scale per channel, but lower-
#    performance cars may have higher performance variance."
#
# IMPORTANT DISTINCTION
# ---------------------
# Team-level channels test common constructor/weekend variability:
#
#   pace_level
#   q1_level
#
# Teammate-contrast channels test whether the *driver comparison*
# itself becomes noisier in weaker cars:
#
#   pace_contrast
#   quali_contrast
#
# If heteroskedasticity appears only in LEVEL channels, that is much
# less worrying for relative driver comparisons because the common
# constructor effect cancels from the contrast likelihood.
#
# METHOD
# ------
# For every modern likelihood row:
#
#   residual = observed y - fitted posterior-mean mu
#
# then remove the model's existing exposure/weight scaling:
#
#   z = residual * sqrt(weight) / fitted channel sigma
#
# Under the frozen homoskedastic Student-t likelihood, the distribution
# of z should not systematically depend on constructor beta.
#
# We then:
#
#   1. aggregate residual scale by constructor-season;
#   2. regress log residual scale on constructor beta;
#   3. use YEAR-FIXED-EFFECT slopes so the test is within-season;
#   4. cluster-bootstrap whole years for uncertainty;
#   5. compare bottom-car vs top-car quartiles;
#   6. report Student-t tail rates by car-strength quartile;
#   7. separate LEVEL from CONTRAST evidence.
#
# INTERPRETATION
# --------------
# Negative slope:
#   lower-beta / weaker cars have larger residual scale.
#
# Bottom/top scale ratio > 1:
#   bottom-quartile constructor-seasons are noisier.
#
# This is an IN-SAMPLE residual diagnostic. It is appropriate for
# detecting variance structure, but if we find a strong effect the
# next step should be a held-out heteroskedasticity check before
# modifying the frozen model.
# ==========================================================

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

HETERO_BOOTSTRAPS = 3000
HETERO_SEED = 26042

MIN_ROWS_PER_CONSTRUCTOR_SEASON = 6

HETERO_OUTPUT_DIR = (
    Path(CFG.output_dir)
    / "validation_real_data_heteroskedasticity"
)

HETERO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==========================================================
# 1. REQUIRED OBJECTS
# ==========================================================

required_objects = [
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "idx_all",
    "CFG",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the final model stages first. Missing: "
        + ", ".join(missing)
    )

if (
    "posterior_final" not in globals()
    and
    "final_trace_2025" not in globals()
):
    raise RuntimeError(
        "Could not find accepted final posterior. "
        "Expected posterior_final or final_trace_2025."
    )


# ==========================================================
# 2. POSTERIOR HELPERS
# ==========================================================

def stack_state_draws(
    posterior,
    var_name,
):
    da = posterior[var_name].stack(
        sample=("chain", "draw")
    )

    other_dims = [
        dim
        for dim in da.dims
        if dim != "sample"
    ]

    x = da.transpose(
        "sample",
        *other_dims,
    ).values

    return np.asarray(
        x,
        dtype=float,
    ).reshape(
        x.shape[0],
        -1,
    )


def stack_scalar_draws(
    posterior,
    var_name,
):
    return np.asarray(
        posterior[var_name].values,
        dtype=float,
    ).reshape(-1)


def safe_spearman(
    x,
    y,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    ok = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    if ok.sum() < 3:
        return np.nan

    if (
        np.unique(x[ok]).size < 2
        or
        np.unique(y[ok]).size < 2
    ):
        return np.nan

    return float(
        st.spearmanr(
            x[ok],
            y[ok],
        ).statistic
    )


# ==========================================================
# 3. ACCEPTED FINAL POSTERIOR MEANS
# ==========================================================

if "posterior_final" in globals():
    final_posterior = posterior_final
else:
    final_posterior = final_trace_2025[
        "posterior"
    ].to_dataset()


alpha_mean = stack_state_draws(
    final_posterior,
    "alpha_dy",
).mean(
    axis=0
)


car_mean = stack_state_draws(
    final_posterior,
    "car_csy",
).mean(
    axis=0
)


gamma_mean = float(
    stack_scalar_draws(
        final_posterior,
        "gamma_tenure",
    ).mean()
)


sigma_means = {
    "pace_contrast":
        float(
            stack_scalar_draws(
                final_posterior,
                "sigma_pace_contrast",
            ).mean()
        ),
    "pace_level":
        float(
            stack_scalar_draws(
                final_posterior,
                "sigma_pace_level",
            ).mean()
        ),
    "quali_contrast":
        float(
            stack_scalar_draws(
                final_posterior,
                "sigma_q_contrast",
            ).mean()
        ),
    "q1_level":
        float(
            stack_scalar_draws(
                final_posterior,
                "sigma_q_level",
            ).mean()
        ),
}


# ==========================================================
# 4. CONSTRUCTOR-SEASON METADATA
# ==========================================================

csy_meta_rows = []

for csy, (
    constructor_id,
    year,
) in enumerate(
    idx_all.active_constructor_years
):
    csy_meta_rows.append(
        {
            "csy":
                int(csy),
            "constructorId":
                int(constructor_id),
            "year":
                int(year),
            "beta":
                float(
                    car_mean[
                        csy
                    ]
                ),
        }
    )


csy_meta = pd.DataFrame(
    csy_meta_rows
)


# Optional readable constructor labels.
constructor_name_map = {}

if (
    "raw" in globals()
    and hasattr(raw, "constructors")
):
    rc = raw.constructors.copy()

    if "constructorId" in rc.columns:
        name_col = None

        for candidate in [
            "name",
            "constructorRef",
        ]:
            if candidate in rc.columns:
                name_col = candidate
                break

        if name_col is not None:
            constructor_name_map = dict(
                zip(
                    rc["constructorId"].astype(int),
                    rc[name_col].astype(str),
                )
            )


csy_meta[
    "constructor"
] = (
    csy_meta[
        "constructorId"
    ]
    .map(
        constructor_name_map
    )
    .fillna(
        csy_meta[
            "constructorId"
        ].astype(str)
    )
)


# ==========================================================
# 5. ROBUST CSY INFERENCE
# ==========================================================

csy_map = getattr(
    idx_all,
    "csy_map",
    None,
)


def ensure_csy(
    df,
    channel,
):
    """
    Return an integer csy array if the channel can be associated with
    a constructor-season. Otherwise return None.
    """

    if "csy" in df.columns:
        return df[
            "csy"
        ].to_numpy(
            dtype=int
        )

    if (
        csy_map is not None
        and
        "constructorId" in df.columns
        and
        "year" in df.columns
    ):
        values = []

        for constructor_id, year in zip(
            df["constructorId"],
            df["year"],
        ):
            key = (
                int(constructor_id),
                int(year),
            )

            if key not in csy_map:
                return None

            values.append(
                int(
                    csy_map[
                        key
                    ]
                )
            )

        return np.asarray(
            values,
            dtype=int,
        )

    warnings.warn(
        f"{channel}: no csy / constructorId+year mapping found. "
        "This channel will be skipped in the car-strength variance test."
    )

    return None


# ==========================================================
# 6. BUILD POSTERIOR-MEAN RESIDUAL ROWS
# ==========================================================

def build_contrast_residuals(
    df,
    *,
    channel,
    sigma,
):
    csy = ensure_csy(
        df,
        channel,
    )

    if csy is None:
        return pd.DataFrame()

    dy_a = df[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = df[
        "dy_b"
    ].to_numpy(
        dtype=int
    )

    tenure = df[
        "tenure_diff"
    ].to_numpy(
        dtype=float
    )

    y = df[
        "y"
    ].to_numpy(
        dtype=float
    )

    weight = (
        df[
            "weight"
        ].to_numpy(
            dtype=float
        )
        if "weight" in df.columns
        else np.ones(
            len(df),
            dtype=float,
        )
    )

    mu = (
        alpha_mean[
            dy_a
        ]
        -
        alpha_mean[
            dy_b
        ]
        +
        gamma_mean
        *
        tenure
    )

    residual = (
        y
        -
        mu
    )

    z = (
        residual
        *
        np.sqrt(
            weight
        )
        /
        sigma
    )

    out = pd.DataFrame(
        {
            "channel":
                channel,
            "channel_type":
                "contrast",
            "csy":
                csy,
            "y":
                y,
            "mu":
                mu,
            "residual":
                residual,
            "weight":
                weight,
            "z":
                z,
            "abs_z":
                np.abs(
                    z
                ),
            "z2":
                z ** 2,
        }
    )

    return out.merge(
        csy_meta[
            [
                "csy",
                "constructorId",
                "constructor",
                "year",
                "beta",
            ]
        ],
        on="csy",
        how="left",
        validate="many_to_one",
    )


def build_level_residuals(
    df,
    *,
    channel,
    sigma,
):
    csy = ensure_csy(
        df,
        channel,
    )

    if csy is None:
        return pd.DataFrame()

    dy_a = df[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = df[
        "dy_b"
    ].to_numpy(
        dtype=int
    )

    tenure = df[
        "tenure_mean"
    ].to_numpy(
        dtype=float
    )

    y = df[
        "y"
    ].to_numpy(
        dtype=float
    )

    weight = (
        df[
            "weight"
        ].to_numpy(
            dtype=float
        )
        if "weight" in df.columns
        else np.ones(
            len(df),
            dtype=float,
        )
    )

    mu = (
        0.5
        *
        (
            alpha_mean[
                dy_a
            ]
            +
            alpha_mean[
                dy_b
            ]
        )
        +
        car_mean[
            csy
        ]
        +
        gamma_mean
        *
        tenure
    )

    residual = (
        y
        -
        mu
    )

    z = (
        residual
        *
        np.sqrt(
            weight
        )
        /
        sigma
    )

    out = pd.DataFrame(
        {
            "channel":
                channel,
            "channel_type":
                "level",
            "csy":
                csy,
            "y":
                y,
            "mu":
                mu,
            "residual":
                residual,
            "weight":
                weight,
            "z":
                z,
            "abs_z":
                np.abs(
                    z
                ),
            "z2":
                z ** 2,
        }
    )

    return out.merge(
        csy_meta[
            [
                "csy",
                "constructorId",
                "constructor",
                "year",
                "beta",
            ]
        ],
        on="csy",
        how="left",
        validate="many_to_one",
    )


residual_frames = [
    build_level_residuals(
        pace_level_all,
        channel="pace_level",
        sigma=sigma_means[
            "pace_level"
        ],
    ),
    build_level_residuals(
        q_level_all_model,
        channel="q1_level",
        sigma=sigma_means[
            "q1_level"
        ],
    ),
    build_contrast_residuals(
        pace_contrast_all,
        channel="pace_contrast",
        sigma=sigma_means[
            "pace_contrast"
        ],
    ),
    build_contrast_residuals(
        q_contrast_all,
        channel="quali_contrast",
        sigma=sigma_means[
            "quali_contrast"
        ],
    ),
]


residual_rows = pd.concat(
    [
        x
        for x in residual_frames
        if len(x)
    ],
    ignore_index=True,
)


if residual_rows.empty:
    raise RuntimeError(
        "No residual rows could be associated with constructor-season beta."
    )


# ==========================================================
# 7. WITHIN-YEAR CAR-STRENGTH QUARTILES
# ==========================================================

csy_beta_rank = csy_meta.copy()

csy_beta_rank[
    "beta_percentile_within_year"
] = (
    csy_beta_rank.groupby(
        "year"
    )[
        "beta"
    ]
    .rank(
        pct=True,
        method="average",
    )
)


def beta_band(
    p,
):
    if p <= 0.25:
        return "Q1 weakest"
    elif p <= 0.50:
        return "Q2"
    elif p <= 0.75:
        return "Q3"
    else:
        return "Q4 strongest"


csy_beta_rank[
    "car_strength_quartile"
] = csy_beta_rank[
    "beta_percentile_within_year"
].map(
    beta_band
)


residual_rows = residual_rows.drop(
    columns=[
        "car_strength_quartile",
        "beta_percentile_within_year",
    ],
    errors="ignore",
).merge(
    csy_beta_rank[
        [
            "csy",
            "beta_percentile_within_year",
            "car_strength_quartile",
        ]
    ],
    on="csy",
    how="left",
    validate="many_to_one",
)


# ==========================================================
# 8. CONSTRUCTOR-SEASON RESIDUAL SCALE
# ==========================================================

def robust_scale_mad(
    x,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    if len(x) < 2:
        return np.nan

    med = np.median(
        x
    )

    mad = np.median(
        np.abs(
            x
            -
            med
        )
    )

    return float(
        1.4826
        *
        mad
    )


def build_csy_scale_table(
    frame,
    label,
    channel_type,
):
    rows = []

    for (
        csy,
        year,
        constructor_id,
        constructor,
        beta,
        quartile,
    ), grp in frame.groupby(
        [
            "csy",
            "year",
            "constructorId",
            "constructor",
            "beta",
            "car_strength_quartile",
        ],
        observed=False,
        dropna=False,
    ):
        if (
            len(grp)
            <
            MIN_ROWS_PER_CONSTRUCTOR_SEASON
        ):
            continue

        z = grp[
            "z"
        ].to_numpy(
            dtype=float
        )

        rows.append(
            {
                "group":
                    label,
                "channel_type":
                    channel_type,
                "csy":
                    int(csy),
                "year":
                    int(year),
                "constructorId":
                    int(constructor_id),
                "constructor":
                    str(constructor),
                "beta":
                    float(beta),
                "car_strength_quartile":
                    quartile,
                "rows":
                    len(grp),
                "mean_z":
                    float(
                        np.mean(
                            z
                        )
                    ),
                "mean_abs_z":
                    float(
                        np.mean(
                            np.abs(
                                z
                            )
                        )
                    ),
                "median_abs_z":
                    float(
                        np.median(
                            np.abs(
                                z
                            )
                        )
                    ),
                "robust_scale":
                    robust_scale_mad(
                        z
                    ),
                "rms_z":
                    float(
                        np.sqrt(
                            np.mean(
                                z ** 2
                            )
                        )
                    ),
                "tail_gt_2":
                    float(
                        np.mean(
                            np.abs(
                                z
                            )
                            >
                            2.0
                        )
                    ),
                "tail_gt_3":
                    float(
                        np.mean(
                            np.abs(
                                z
                            )
                            >
                            3.0
                        )
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


csy_scale_frames = []


for channel in [
    "pace_level",
    "q1_level",
    "pace_contrast",
    "quali_contrast",
]:
    x = residual_rows[
        residual_rows[
            "channel"
        ]
        ==
        channel
    ]

    if len(x):
        channel_type = x[
            "channel_type"
        ].iloc[0]

        csy_scale_frames.append(
            build_csy_scale_table(
                x,
                label=channel,
                channel_type=channel_type,
            )
        )


level_rows = residual_rows[
    residual_rows[
        "channel_type"
    ]
    ==
    "level"
]

if len(level_rows):
    csy_scale_frames.append(
        build_csy_scale_table(
            level_rows,
            label="LEVEL POOLED",
            channel_type="level",
        )
    )


contrast_rows = residual_rows[
    residual_rows[
        "channel_type"
    ]
    ==
    "contrast"
]

if len(contrast_rows):
    csy_scale_frames.append(
        build_csy_scale_table(
            contrast_rows,
            label="CONTRAST POOLED",
            channel_type="contrast",
        )
    )


csy_scale_frames.append(
    build_csy_scale_table(
        residual_rows,
        label="ALL POOLED",
        channel_type="all",
    )
)


csy_scales = pd.concat(
    [
        x
        for x in csy_scale_frames
        if len(x)
    ],
    ignore_index=True,
)


# ==========================================================
# 9. YEAR-FIXED-EFFECT SLOPE
# ==========================================================

def fixed_effect_slope(
    df,
    *,
    x_col="beta",
    y_col="log_scale",
):
    x = df[
        [
            "year",
            x_col,
            y_col,
        ]
    ].dropna().copy()

    if len(x) < 3:
        return np.nan

    x[
        "x_dm"
    ] = (
        x[
            x_col
        ]
        -
        x.groupby(
            "year"
        )[
            x_col
        ].transform(
            "mean"
        )
    )

    x[
        "y_dm"
    ] = (
        x[
            y_col
        ]
        -
        x.groupby(
            "year"
        )[
            y_col
        ].transform(
            "mean"
        )
    )

    denominator = float(
        np.sum(
            x[
                "x_dm"
            ]
            ** 2
        )
    )

    if denominator <= 0:
        return np.nan

    return float(
        np.sum(
            x[
                "x_dm"
            ]
            *
            x[
                "y_dm"
            ]
        )
        /
        denominator
    )


# ==========================================================
# 10. CLUSTER BOOTSTRAP BY YEAR
# ==========================================================

bootstrap_rng = np.random.default_rng(
    HETERO_SEED
)


def bootstrap_year_stat(
    df,
    stat_func,
    *,
    n_boot=HETERO_BOOTSTRAPS,
):
    years = np.asarray(
        sorted(
            df[
                "year"
            ].dropna().unique()
        )
    )

    if len(years) < 3:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    observed = float(
        stat_func(
            df
        )
    )

    boot = []

    for _ in range(
        n_boot
    ):
        sampled_years = bootstrap_rng.choice(
            years,
            size=len(years),
            replace=True,
        )

        pieces = []

        for bootstrap_year_id, year in enumerate(
            sampled_years
        ):
            piece = df[
                df[
                    "year"
                ]
                ==
                year
            ].copy()

            piece[
                "year"
            ] = bootstrap_year_id

            pieces.append(
                piece
            )

        sample = pd.concat(
            pieces,
            ignore_index=True,
        )

        value = stat_func(
            sample
        )

        if np.isfinite(
            value
        ):
            boot.append(
                float(
                    value
                )
            )

    if not boot:
        return (
            observed,
            np.nan,
            np.nan,
        )

    boot = np.asarray(
        boot,
        dtype=float,
    )

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


def bottom_top_scale_ratio(
    df,
):
    bottom = df[
        df[
            "car_strength_quartile"
        ]
        ==
        "Q1 weakest"
    ][
        "robust_scale"
    ].dropna()

    top = df[
        df[
            "car_strength_quartile"
        ]
        ==
        "Q4 strongest"
    ][
        "robust_scale"
    ].dropna()

    if (
        len(bottom) == 0
        or len(top) == 0
    ):
        return np.nan

    top_median = float(
        np.median(
            top
        )
    )

    if top_median <= 0:
        return np.nan

    return float(
        np.median(
            bottom
        )
        /
        top_median
    )


# ==========================================================
# 11. PRIMARY HETEROSKEDASTICITY TEST TABLE
# ==========================================================

test_rows = []


for group, grp in csy_scales.groupby(
    "group",
    sort=False,
):
    grp = grp[
        grp[
            "robust_scale"
        ].notna()
        &
        (
            grp[
                "robust_scale"
            ]
            >
            0
        )
    ].copy()

    if len(grp) < 5:
        continue

    grp[
        "log_scale"
    ] = np.log(
        grp[
            "robust_scale"
        ]
    )

    fe_slope, fe_lb, fe_ub = bootstrap_year_stat(
        grp,
        lambda x:
            fixed_effect_slope(
                x,
                x_col="beta",
                y_col="log_scale",
            ),
    )

    ratio, ratio_lb, ratio_ub = bootstrap_year_stat(
        grp,
        bottom_top_scale_ratio,
    )

    test_rows.append(
        {
            "group":
                group,
            "channel_type":
                grp[
                    "channel_type"
                ].iloc[0],
            "constructor_seasons":
                len(grp),
            "years":
                grp[
                    "year"
                ].nunique(),
            "spearman_beta_vs_scale":
                safe_spearman(
                    grp[
                        "beta"
                    ],
                    grp[
                        "robust_scale"
                    ],
                ),
            "year_FE_logscale_slope":
                fe_slope,
            "slope_95_lb":
                fe_lb,
            "slope_95_ub":
                fe_ub,
            "bottom_vs_top_scale_ratio":
                ratio,
            "ratio_95_lb":
                ratio_lb,
            "ratio_95_ub":
                ratio_ub,
            "mean_robust_scale":
                float(
                    grp[
                        "robust_scale"
                    ].mean()
                ),
        }
    )


hetero_tests = pd.DataFrame(
    test_rows
)


print(
    "========================================"
)
print(
    "STAGE 26 — REAL-DATA HETEROSKEDASTICITY AUDIT"
)
print(
    "========================================"
)

print(
    "Posterior mean gamma tenure:",
    gamma_mean,
)

print(
    "Residual rows analysed:",
    len(
        residual_rows
    ),
)

print(
    "Constructor-season scale rows:",
    len(
        csy_scales
    ),
)


print(
    "\n========================================"
)
print(
    "PRIMARY TEST — DOES RESIDUAL SCALE FALL AS CAR BETA RISES?"
)
print(
    "========================================"
)

display(
    hetero_tests
)


# ==========================================================
# 12. QUARTILE SUMMARY
# ==========================================================

quartile_order = [
    "Q1 weakest",
    "Q2",
    "Q3",
    "Q4 strongest",
]


quartile_summary = (
    csy_scales.groupby(
        [
            "group",
            "channel_type",
            "car_strength_quartile",
        ],
        observed=False,
        as_index=False,
    )
    .agg(
        constructor_seasons=(
            "csy",
            "nunique",
        ),
        median_beta=(
            "beta",
            "median",
        ),
        median_robust_scale=(
            "robust_scale",
            "median",
        ),
        median_rms_z=(
            "rms_z",
            "median",
        ),
        mean_tail_gt_2=(
            "tail_gt_2",
            "mean",
        ),
        mean_tail_gt_3=(
            "tail_gt_3",
            "mean",
        ),
    )
)


quartile_summary[
    "car_strength_quartile"
] = pd.Categorical(
    quartile_summary[
        "car_strength_quartile"
    ],
    categories=quartile_order,
    ordered=True,
)


quartile_summary = quartile_summary.sort_values(
    [
        "group",
        "car_strength_quartile",
    ]
)


print(
    "\n========================================"
)
print(
    "RESIDUAL SCALE BY WITHIN-SEASON CAR-STRENGTH QUARTILE"
)
print(
    "========================================"
)

display(
    quartile_summary
)


# ==========================================================
# 13. ROW-LEVEL TAIL CALIBRATION BY CAR STRENGTH
# ==========================================================

nu = CFG.student_t_nu

theory_tail_2 = float(
    2.0
    *
    st.t.sf(
        2.0,
        df=nu,
    )
)

theory_tail_3 = float(
    2.0
    *
    st.t.sf(
        3.0,
        df=nu,
    )
)


row_tail_summary = (
    residual_rows.groupby(
        [
            "channel_type",
            "car_strength_quartile",
        ],
        observed=False,
        as_index=False,
    )
    .agg(
        rows=(
            "z",
            "size",
        ),
        mean_abs_z=(
            "abs_z",
            "mean",
        ),
        observed_tail_gt_2=(
            "abs_z",
            lambda x:
                float(
                    np.mean(
                        np.asarray(
                            x,
                            dtype=float,
                        )
                        >
                        2.0
                    )
                ),
        ),
        observed_tail_gt_3=(
            "abs_z",
            lambda x:
                float(
                    np.mean(
                        np.asarray(
                            x,
                            dtype=float,
                        )
                        >
                        3.0
                    )
                ),
        ),
    )
)


row_tail_summary[
    "theory_t4_tail_gt_2"
] = theory_tail_2

row_tail_summary[
    "theory_t4_tail_gt_3"
] = theory_tail_3


row_tail_summary[
    "car_strength_quartile"
] = pd.Categorical(
    row_tail_summary[
        "car_strength_quartile"
    ],
    categories=quartile_order,
    ordered=True,
)


row_tail_summary = row_tail_summary.sort_values(
    [
        "channel_type",
        "car_strength_quartile",
    ]
)


print(
    "\n========================================"
)
print(
    "STUDENT-t TAIL RATE BY CAR-STRENGTH QUARTILE"
)
print(
    "========================================"
)

print(
    f"Theoretical t({nu}) P(|z| > 2): "
    f"{theory_tail_2:.4f}"
)

print(
    f"Theoretical t({nu}) P(|z| > 3): "
    f"{theory_tail_3:.4f}"
)

display(
    row_tail_summary
)


# ==========================================================
# 14. MOST / LEAST VARIABLE CONSTRUCTOR-SEASONS
# ==========================================================

pooled_scales = csy_scales[
    csy_scales[
        "group"
    ]
    ==
    "ALL POOLED"
].copy()


pooled_scales = pooled_scales.sort_values(
    "robust_scale",
    ascending=False,
)


print(
    "\n========================================"
)
print(
    "MOST VARIABLE CONSTRUCTOR-SEASONS — ALL MODERN CHANNELS"
)
print(
    "========================================"
)

display(
    pooled_scales[
        [
            "year",
            "constructor",
            "beta",
            "car_strength_quartile",
            "rows",
            "robust_scale",
            "rms_z",
            "tail_gt_2",
            "tail_gt_3",
        ]
    ]
    .head(
        20
    )
)


print(
    "\n========================================"
)
print(
    "LEAST VARIABLE CONSTRUCTOR-SEASONS — ALL MODERN CHANNELS"
)
print(
    "========================================"
)

display(
    pooled_scales[
        [
            "year",
            "constructor",
            "beta",
            "car_strength_quartile",
            "rows",
            "robust_scale",
            "rms_z",
            "tail_gt_2",
            "tail_gt_3",
        ]
    ]
    .tail(
        20
    )
    .sort_values(
        "robust_scale"
    )
)


# ==========================================================
# 15. PLOTS
# ==========================================================

plot_groups = [
    "LEVEL POOLED",
    "CONTRAST POOLED",
    "ALL POOLED",
]


for plot_group in plot_groups:
    plot_df = csy_scales[
        csy_scales[
            "group"
        ]
        ==
        plot_group
    ].dropna(
        subset=[
            "beta",
            "robust_scale",
        ]
    )

    if len(plot_df) < 5:
        continue

    plt.figure(
        figsize=(
            8,
            6,
        )
    )

    plt.scatter(
        plot_df[
            "beta"
        ],
        plot_df[
            "robust_scale"
        ],
        alpha=0.55,
    )

    x = plot_df[
        "beta"
    ].to_numpy(
        dtype=float
    )

    y = np.log(
        plot_df[
            "robust_scale"
        ].to_numpy(
            dtype=float
        )
    )

    if (
        np.isfinite(x).sum() >= 3
        and
        np.isfinite(y).sum() >= 3
    ):
        slope, intercept = np.polyfit(
            x,
            y,
            1,
        )

        xx = np.linspace(
            np.nanmin(x),
            np.nanmax(x),
            200,
        )

        yy = np.exp(
            intercept
            +
            slope
            *
            xx
        )

        plt.plot(
            xx,
            yy,
            linestyle="--",
        )

    plt.axvline(
        0.0,
        linestyle=":",
        linewidth=1.0,
    )

    plt.xlabel(
        "Posterior mean constructor-season beta"
    )

    plt.ylabel(
        "Robust residual scale in fitted Student-t units"
    )

    plt.title(
        f"Residual variability vs constructor strength — {plot_group}"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.show()


quartile_plot = quartile_summary[
    quartile_summary[
        "group"
    ].isin(
        [
            "LEVEL POOLED",
            "CONTRAST POOLED",
        ]
    )
].copy()


if len(
    quartile_plot
):
    for group in [
        "LEVEL POOLED",
        "CONTRAST POOLED",
    ]:
        x = quartile_plot[
            quartile_plot[
                "group"
            ]
            ==
            group
        ]

        if x.empty:
            continue

        plt.figure(
            figsize=(
                8,
                5,
            )
        )

        plt.plot(
            x[
                "car_strength_quartile"
            ].astype(str),
            x[
                "median_robust_scale"
            ],
            marker="o",
        )

        plt.xlabel(
            "Constructor strength quartile within season"
        )

        plt.ylabel(
            "Median constructor-season robust residual scale"
        )

        plt.title(
            f"Residual variability across car-strength quartiles — {group}"
        )

        plt.grid(
            alpha=0.25
        )

        plt.tight_layout()

        plt.show()


# ==========================================================
# 16. SAVE OUTPUTS
# ==========================================================

residual_rows.to_csv(
    HETERO_OUTPUT_DIR
    /
    "modern_residual_rows.csv",
    index=False,
)


csy_scales.to_csv(
    HETERO_OUTPUT_DIR
    /
    "constructor_season_residual_scales.csv",
    index=False,
)


hetero_tests.to_csv(
    HETERO_OUTPUT_DIR
    /
    "heteroskedasticity_primary_tests.csv",
    index=False,
)


quartile_summary.to_csv(
    HETERO_OUTPUT_DIR
    /
    "heteroskedasticity_quartile_summary.csv",
    index=False,
)


row_tail_summary.to_csv(
    HETERO_OUTPUT_DIR
    /
    "student_t_tail_calibration_by_car_strength.csv",
    index=False,
)


# ==========================================================
# 17. FINAL TRIAGE SUMMARY
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 26 TRIAGE SUMMARY"
)
print(
    "========================================"
)


def print_group_summary(
    group,
):
    x = hetero_tests[
        hetero_tests[
            "group"
        ]
        ==
        group
    ]

    if x.empty:
        print(
            f"{group}: not available"
        )
        return

    row = x.iloc[0]

    print(
        f"{group}:"
    )

    print(
        f"  FE slope log(scale) vs beta = "
        f"{row['year_FE_logscale_slope']:.4f} "
        f"[95% bootstrap "
        f"{row['slope_95_lb']:.4f}, "
        f"{row['slope_95_ub']:.4f}]"
    )

    print(
        f"  Bottom/top robust-scale ratio = "
        f"{row['bottom_vs_top_scale_ratio']:.3f} "
        f"[95% bootstrap "
        f"{row['ratio_95_lb']:.3f}, "
        f"{row['ratio_95_ub']:.3f}]"
    )

    print(
        f"  Spearman(beta, residual scale) = "
        f"{row['spearman_beta_vs_scale']:.3f}"
    )


print_group_summary(
    "LEVEL POOLED"
)

print_group_summary(
    "CONTRAST POOLED"
)

print_group_summary(
    "ALL POOLED"
)


print(
    "\nDecision guide:"
)

print(
    "  Little evidence of problematic heteroskedasticity:"
)

print(
    "    FE slope near 0, bootstrap interval spans 0,"
)

print(
    "    bottom/top ratio near 1, and no monotonic quartile pattern."
)

print(
    "  Evidence that weak cars are noisier:"
)

print(
    "    FE slope clearly < 0 AND bottom/top ratio > 1."
)

print(
    "  If that appears mainly in LEVEL evidence but not CONTRASTS,"
)

print(
    "    the consequence is primarily constructor/team-level uncertainty,"
)

print(
    "    not direct teammate ordering."
)

print(
    "  If it also appears strongly in CONTRAST evidence,"
)

print(
    "    weaker cars may make driver-vs-driver comparisons themselves"
)

print(
    "    more variable; that is the more important ranking concern."
)

print(
    "\nImportant caveat:"
)

print(
    "  These are production-fit residuals. A strong detected pattern"
)

print(
    "  should be confirmed on held-out observations before changing"
)

print(
    "  the frozen likelihood."
)

print(
    "\nSaved outputs ->"
)

print(
    HETERO_OUTPUT_DIR.resolve()
)

print(
    "\nStage 26 complete."
)


In [ ]:
# ==========================================================
# STAGE 27 — HELD-OUT HETEROSKEDASTICITY + QUALIFYING-SESSION AUDIT
# ==========================================================
#
# PURPOSE
# -------
# Stage 26 found real-data evidence that weaker constructor-seasons
# are noisier, especially in qualifying. Stage 27 asks whether that
# pattern survives on genuinely UNSEEN modern race weekends.
#
# It also answers the crucial follow-up:
#
#   Is the qualifying effect really associated with weak cars,
#   or is it mainly caused by mixing Q1 / Q2 / Q3?
#
# DESIGN
# ------
# 1. Hold out ~20% of MODERN race weekends, stratified by season.
# 2. Remove those raceIds from ALL FOUR modern likelihood channels:
#
#       pace contrast
#       pace team level
#       qualifying contrast
#       Q1 team level
#
# 3. Keep the full historical 1982–1995 likelihood in training.
# 4. Fit the FROZEN model once to the remaining modern data.
# 5. Predict the unseen modern observations.
# 6. Standardise held-out residuals by the model's fitted Student-t
#    scale and the existing row reliability weights.
# 7. Test residual scale against constructor beta.
# 8. Repeat the qualifying-contrast test separately for:
#
#       Q1 only
#       Q2 only
#       Q3 only
#
# KEY INTERPRETATION
# ------------------
# If Q1-only teammate contrasts remain more variable in weak cars,
# the Stage 26 result is NOT merely a Q1/Q2/Q3 session-mix artefact.
#
# If the effect disappears in Q1-only contrasts but appears when
# sessions are pooled, then session composition / advancement is a
# much more plausible explanation.
#
# IMPORTANT
# ---------
# This is a VALIDATION fit only.
# It does NOT alter or overwrite the accepted production posterior.
#
# The checkpoint is cached. Re-running the cell after a successful
# fit should load the saved validation trace instead of resampling.
#
# The training model intentionally retains the already-built full-data
# centring matrices. Those matrices depend on participation/reliability
# weights, not held-out y values. This matches the earlier modern
# holdout-validation convention and does not leak held-out outcomes.
# ==========================================================

from pathlib import Path
import time

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import scipy.stats as st
import xarray as xr


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

H27_HOLDOUT_FRACTION = 0.20
H27_SPLIT_SEED = 27042

H27_CHAINS = 2
H27_TUNE = 400
H27_DRAWS = 400
H27_TARGET_ACCEPT = 0.92
H27_SAMPLE_SEED = 27043

H27_BOOTSTRAPS = 3000
H27_BOOTSTRAP_SEED = 27044

# Held-out constructor-seasons only have ~20% of their weekends,
# so the minimum must be lower than in Stage 26.
H27_MIN_ROWS_PER_CSY = 3

H27_OUTPUT_DIR = (
    Path(CFG.output_dir)
    / "validation_heldout_heteroskedasticity"
)

H27_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

H27_TRACE_PATH = (
    H27_OUTPUT_DIR
    / (
        "heldout_weekend_20pct_"
        f"{H27_TUNE}tune_{H27_DRAWS}draw_"
        f"{H27_CHAINS}chain_nutpie_jax.nc"
    )
)

H27_SPLIT_PATH = (
    H27_OUTPUT_DIR
    / "heldout_race_split.csv"
)

H27_CHECKPOINT_ENGINE = "h5netcdf"


# ==========================================================
# 1. REQUIRED OBJECTS
# ==========================================================

required_objects = [
    "CFG",
    "idx_all",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "hist_q_pack",
    "hist_race_pack",
    "build_joint_model_1982_2025",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the final model-construction stages first. Missing: "
        + ", ".join(missing)
    )


# ==========================================================
# 2. HELPERS
# ==========================================================

def stack_state_draws(
    posterior,
    var_name,
):
    da = posterior[var_name].stack(
        sample=("chain", "draw")
    )

    other_dims = [
        dim
        for dim in da.dims
        if dim != "sample"
    ]

    x = da.transpose(
        "sample",
        *other_dims,
    ).values

    return np.asarray(
        x,
        dtype=float,
    ).reshape(
        x.shape[0],
        -1,
    )


def stack_scalar_draws(
    posterior,
    var_name,
):
    return np.asarray(
        posterior[var_name].values,
        dtype=float,
    ).reshape(-1)


def safe_spearman(
    x,
    y,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    ok = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    if ok.sum() < 3:
        return np.nan

    if (
        np.unique(x[ok]).size < 2
        or
        np.unique(y[ok]).size < 2
    ):
        return np.nan

    return float(
        st.spearmanr(
            x[ok],
            y[ok],
        ).statistic
    )


def robust_scale_mad(
    x,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
    ]

    if len(x) < 2:
        return np.nan

    median = np.median(
        x
    )

    mad = np.median(
        np.abs(
            x
            -
            median
        )
    )

    return float(
        1.4826
        *
        mad
    )


# ==========================================================
# 3. DETERMINISTIC STRATIFIED WEEKEND HOLDOUT
# ==========================================================

race_year_frames = []

for frame in [
    pace_contrast_all,
    pace_level_all,
    q_contrast_all,
    q_level_all_model,
]:
    race_year_frames.append(
        frame[
            [
                "raceId",
                "year",
            ]
        ].drop_duplicates()
    )

modern_race_year = (
    pd.concat(
        race_year_frames,
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values(
        [
            "year",
            "raceId",
        ]
    )
    .reset_index(
        drop=True
    )
)

race_year_nunique = (
    modern_race_year
    .groupby("raceId")["year"]
    .nunique()
)

if (
    race_year_nunique.max()
    >
    1
):
    raise RuntimeError(
        "A modern raceId maps to multiple years."
    )


split_rows = []

for year, grp in modern_race_year.groupby(
    "year",
    sort=True,
):
    race_ids = np.asarray(
        sorted(
            grp[
                "raceId"
            ].astype(int)
        ),
        dtype=int,
    )

    n_races = len(
        race_ids
    )

    n_holdout = max(
        1,
        int(
            round(
                H27_HOLDOUT_FRACTION
                *
                n_races
            )
        ),
    )

    # Never remove an entire season.
    n_holdout = min(
        n_holdout,
        n_races - 1,
    )

    year_rng = np.random.default_rng(
        H27_SPLIT_SEED
        +
        int(year)
    )

    heldout = set(
        year_rng.choice(
            race_ids,
            size=n_holdout,
            replace=False,
        ).tolist()
    )

    for race_id in race_ids:
        split_rows.append(
            {
                "raceId":
                    int(race_id),
                "year":
                    int(year),
                "is_holdout":
                    int(
                        race_id
                        in heldout
                    ),
            }
        )


h27_split = pd.DataFrame(
    split_rows
)

h27_split.to_csv(
    H27_SPLIT_PATH,
    index=False,
)


heldout_race_ids = set(
    h27_split.loc[
        h27_split[
            "is_holdout"
        ]
        ==
        1,
        "raceId",
    ].astype(int)
)


print(
    "========================================"
)
print(
    "STAGE 27 — HELD-OUT HETEROSKEDASTICITY"
)
print(
    "========================================"
)

print(
    "Modern weekends:",
    len(
        h27_split
    ),
)

print(
    "Held-out weekends:",
    int(
        h27_split[
            "is_holdout"
        ].sum()
    ),
)

print(
    "Held-out fraction:",
    float(
        h27_split[
            "is_holdout"
        ].mean()
    ),
)

print(
    "Years represented:",
    h27_split[
        "year"
    ].nunique(),
)


# ==========================================================
# 4. TRAIN / TEST LIKELIHOOD TABLES
# ==========================================================

def split_modern_frame(
    df,
):
    mask = df[
        "raceId"
    ].isin(
        heldout_race_ids
    )

    train = (
        df.loc[
            ~mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    test = (
        df.loc[
            mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    return (
        train,
        test,
    )


(
    h27_pc_train,
    h27_pc_test,
) = split_modern_frame(
    pace_contrast_all
)

(
    h27_pl_train,
    h27_pl_test,
) = split_modern_frame(
    pace_level_all
)

(
    h27_qc_train,
    h27_qc_test,
) = split_modern_frame(
    q_contrast_all
)

(
    h27_ql_train,
    h27_ql_test,
) = split_modern_frame(
    q_level_all_model
)


split_count_table = pd.DataFrame(
    {
        "channel": [
            "pace_contrast",
            "pace_level",
            "quali_contrast",
            "q1_level",
        ],
        "train_rows": [
            len(
                h27_pc_train
            ),
            len(
                h27_pl_train
            ),
            len(
                h27_qc_train
            ),
            len(
                h27_ql_train
            ),
        ],
        "heldout_rows": [
            len(
                h27_pc_test
            ),
            len(
                h27_pl_test
            ),
            len(
                h27_qc_test
            ),
            len(
                h27_ql_test
            ),
        ],
    }
)


print(
    "\nTrain / held-out rows:"
)

display(
    split_count_table
)


print(
    "\nHeld-out qualifying contrast rows by session:"
)

display(
    h27_qc_test.groupby(
        "session"
    )
    .agg(
        rows=(
            "y",
            "size",
        ),
        races=(
            "raceId",
            "nunique",
        ),
        constructor_seasons=(
            "csy",
            "nunique",
        ),
    )
    .reset_index()
)


# ==========================================================
# 5. BUILD THE VALIDATION MODEL
# ==========================================================
#
# Same frozen likelihood and priors.
# Only the held-out modern raceIds are removed from the likelihood.
# Historical data remain fully included.
# ==========================================================

h27_model = (
    build_joint_model_1982_2025(
        h27_pc_train,
        h27_pl_train,
        h27_qc_train,
        h27_ql_train,
        hist_q_pack,
        hist_race_pack,
        idx_all,
        cfg=CFG,
    )
)


# ==========================================================
# 6. SAMPLE OR LOAD CACHED VALIDATION TRACE
# ==========================================================

try:
    import h5netcdf  # noqa: F401
    import h5py      # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Stage 27 checkpointing requires h5netcdf + h5py."
    ) from exc


if H27_TRACE_PATH.exists():

    print(
        "\nExisting Stage 27 validation trace found:"
    )

    print(
        H27_TRACE_PATH
    )

    try:
        h27_trace = xr.open_datatree(
            H27_TRACE_PATH,
            engine=
                H27_CHECKPOINT_ENGINE,
        )

        print(
            "Loaded checkpoint. "
            "NO RESAMPLING performed."
        )

    except Exception as exc:
        raise RuntimeError(
            "Stage 27 checkpoint exists but could not be read.\n"
            "Do not overwrite it automatically.\n\n"
            f"{H27_TRACE_PATH}\n\n"
            f"{repr(exc)}"
        ) from exc


else:

    print(
        "\nSampling Stage 27 validation fit..."
    )

    print(
        f"{H27_CHAINS} chains × "
        f"({H27_TUNE} tune + "
        f"{H27_DRAWS} posterior draws)"
    )

    print(
        "Sampler: nutpie / JAX"
    )

    t0 = time.perf_counter()

    with h27_model:

        h27_trace = pm.sample(
            draws=
                H27_DRAWS,
            tune=
                H27_TUNE,
            chains=
                H27_CHAINS,
            random_seed=
                H27_SAMPLE_SEED,
            target_accept=
                H27_TARGET_ACCEPT,
            nuts_sampler=
                "nutpie",
            nuts_sampler_kwargs={
                "backend":
                    "jax",
            },
            return_inferencedata=
                True,
            progressbar=
                True,
        )

    elapsed = (
        time.perf_counter()
        -
        t0
    )

    print(
        "\nSampling wall time:",
        f"{elapsed / 60.0:.2f} min",
    )

    h27_trace.to_netcdf(
        H27_TRACE_PATH,
        engine=
            H27_CHECKPOINT_ENGINE,
        mode="w",
    )

    print(
        "Saved Stage 27 trace ->"
    )

    print(
        H27_TRACE_PATH
    )


# ==========================================================
# 7. VALIDATION-FIT HEALTH
# ==========================================================

h27_posterior = (
    h27_trace[
        "posterior"
    ].to_dataset()
)

h27_sample_stats = (
    h27_trace[
        "sample_stats"
    ].to_dataset()
)


H27_GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]


health_rows = []

for var in H27_GLOBAL_VARS:

    rhat = np.asarray(
        az.rhat(
            h27_posterior[
                [
                    var
                ]
            ]
        )
        .to_array()
        .values,
        dtype=float,
    ).reshape(-1)

    ess = np.asarray(
        az.ess(
            h27_posterior[
                [
                    var
                ]
            ],
            method="bulk",
        )
        .to_array()
        .values,
        dtype=float,
    ).reshape(-1)

    health_rows.append(
        {
            "parameter":
                var,
            "max_rhat":
                float(
                    np.nanmax(
                        rhat
                    )
                ),
            "min_ess_bulk":
                float(
                    np.nanmin(
                        ess
                    )
                ),
        }
    )


h27_health = pd.DataFrame(
    health_rows
)

h27_divergences = int(
    np.asarray(
        h27_sample_stats[
            "diverging"
        ].values
    ).sum()
)


print(
    "\n========================================"
)
print(
    "VALIDATION-FIT HEALTH"
)
print(
    "========================================"
)

display(
    h27_health
)

print(
    "Max global Rhat:",
    h27_health[
        "max_rhat"
    ].max(),
)

print(
    "Min global ESS bulk:",
    h27_health[
        "min_ess_bulk"
    ].min(),
)

print(
    "Divergences:",
    h27_divergences,
)


# ==========================================================
# 8. POSTERIOR MEANS FOR HELD-OUT PREDICTION
# ==========================================================

h27_alpha_mean = (
    stack_state_draws(
        h27_posterior,
        "alpha_dy",
    )
    .mean(
        axis=0
    )
)

h27_car_mean = (
    stack_state_draws(
        h27_posterior,
        "car_csy",
    )
    .mean(
        axis=0
    )
)

h27_gamma_mean = float(
    stack_scalar_draws(
        h27_posterior,
        "gamma_tenure",
    ).mean()
)

h27_sigma = {
    "pace_contrast":
        float(
            stack_scalar_draws(
                h27_posterior,
                "sigma_pace_contrast",
            ).mean()
        ),
    "pace_level":
        float(
            stack_scalar_draws(
                h27_posterior,
                "sigma_pace_level",
            ).mean()
        ),
    "quali_contrast":
        float(
            stack_scalar_draws(
                h27_posterior,
                "sigma_q_contrast",
            ).mean()
        ),
    "q1_level":
        float(
            stack_scalar_draws(
                h27_posterior,
                "sigma_q_level",
            ).mean()
        ),
}


# ==========================================================
# 9. CONSTRUCTOR-SEASON BETA METADATA FROM TRAINING FIT
# ==========================================================

constructor_name_map = {}

if (
    "raw" in globals()
    and hasattr(
        raw,
        "constructors",
    )
):

    rc = raw.constructors.copy()

    if "constructorId" in rc.columns:

        name_col = None

        for candidate in [
            "name",
            "constructorRef",
        ]:

            if candidate in rc.columns:
                name_col = candidate
                break

        if name_col is not None:

            constructor_name_map = dict(
                zip(
                    rc[
                        "constructorId"
                    ].astype(int),
                    rc[
                        name_col
                    ].astype(str),
                )
            )


h27_csy_meta = pd.DataFrame(
    [
        {
            "csy":
                int(csy),
            "constructorId":
                int(
                    constructor_id
                ),
            "year":
                int(year),
            "beta":
                float(
                    h27_car_mean[
                        csy
                    ]
                ),
        }
        for csy, (
            constructor_id,
            year,
        ) in enumerate(
            idx_all.active_constructor_years
        )
    ]
)


h27_csy_meta[
    "constructor"
] = (
    h27_csy_meta[
        "constructorId"
    ]
    .map(
        constructor_name_map
    )
    .fillna(
        h27_csy_meta[
            "constructorId"
        ].astype(str)
    )
)


# Rank cars WITHIN EACH SEASON using the training-fit beta.
h27_csy_meta[
    "beta_percentile_within_year"
] = (
    h27_csy_meta.groupby(
        "year"
    )[
        "beta"
    ]
    .rank(
        pct=True,
        method="average",
    )
)


def h27_beta_band(
    p,
):

    if p <= 0.25:
        return "Q1 weakest"

    if p <= 0.50:
        return "Q2"

    if p <= 0.75:
        return "Q3"

    return "Q4 strongest"


h27_csy_meta[
    "car_strength_quartile"
] = (
    h27_csy_meta[
        "beta_percentile_within_year"
    ]
    .map(
        h27_beta_band
    )
)


# ==========================================================
# 10. HELD-OUT STANDARDISED RESIDUALS
# ==========================================================

h27_nu = CFG.student_t_nu

h27_abs_z_89 = float(
    st.t.ppf(
        0.5
        +
        0.89
        /
        2.0,
        df=
            h27_nu,
    )
)

h27_abs_z_95 = float(
    st.t.ppf(
        0.975,
        df=
            h27_nu,
    )
)


def h27_build_contrast_residuals(
    df,
    *,
    channel,
    sigma,
):

    dy_a = df[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = df[
        "dy_b"
    ].to_numpy(
        dtype=int
    )

    csy = df[
        "csy"
    ].to_numpy(
        dtype=int
    )

    tenure = df[
        "tenure_diff"
    ].to_numpy(
        dtype=float
    )

    y = df[
        "y"
    ].to_numpy(
        dtype=float
    )

    weight = df[
        "weight"
    ].to_numpy(
        dtype=float
    )

    mu = (
        h27_alpha_mean[
            dy_a
        ]
        -
        h27_alpha_mean[
            dy_b
        ]
        +
        h27_gamma_mean
        *
        tenure
    )

    residual = (
        y
        -
        mu
    )

    z = (
        residual
        *
        np.sqrt(
            weight
        )
        /
        sigma
    )

    out = pd.DataFrame(
        {
            "raceId":
                df[
                    "raceId"
                ].to_numpy(
                    dtype=int
                ),
            "channel":
                channel,
            "channel_type":
                "contrast",
            "csy":
                csy,
            "y":
                y,
            "mu":
                mu,
            "residual":
                residual,
            "weight":
                weight,
            "z":
                z,
            "abs_z":
                np.abs(
                    z
                ),
            "inside_conditional_89":
                (
                    np.abs(z)
                    <=
                    h27_abs_z_89
                ).astype(int),
            "inside_conditional_95":
                (
                    np.abs(z)
                    <=
                    h27_abs_z_95
                ).astype(int),
        }
    )

    if "session" in df.columns:
        out[
            "session"
        ] = (
            df[
                "session"
            ]
            .astype(str)
            .str.lower()
            .to_numpy()
        )

    return out.merge(
        h27_csy_meta[
            [
                "csy",
                "constructorId",
                "constructor",
                "year",
                "beta",
                "beta_percentile_within_year",
                "car_strength_quartile",
            ]
        ],
        on="csy",
        how="left",
        validate="many_to_one",
    )


def h27_build_level_residuals(
    df,
    *,
    channel,
    sigma,
):

    dy_a = df[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = df[
        "dy_b"
    ].to_numpy(
        dtype=int
    )

    csy = df[
        "csy"
    ].to_numpy(
        dtype=int
    )

    tenure = df[
        "tenure_mean"
    ].to_numpy(
        dtype=float
    )

    y = df[
        "y"
    ].to_numpy(
        dtype=float
    )

    weight = df[
        "weight"
    ].to_numpy(
        dtype=float
    )

    mu = (
        0.5
        *
        (
            h27_alpha_mean[
                dy_a
            ]
            +
            h27_alpha_mean[
                dy_b
            ]
        )
        +
        h27_car_mean[
            csy
        ]
        +
        h27_gamma_mean
        *
        tenure
    )

    residual = (
        y
        -
        mu
    )

    z = (
        residual
        *
        np.sqrt(
            weight
        )
        /
        sigma
    )

    out = pd.DataFrame(
        {
            "raceId":
                df[
                    "raceId"
                ].to_numpy(
                    dtype=int
                ),
            "channel":
                channel,
            "channel_type":
                "level",
            "csy":
                csy,
            "y":
                y,
            "mu":
                mu,
            "residual":
                residual,
            "weight":
                weight,
            "z":
                z,
            "abs_z":
                np.abs(
                    z
                ),
            "inside_conditional_89":
                (
                    np.abs(z)
                    <=
                    h27_abs_z_89
                ).astype(int),
            "inside_conditional_95":
                (
                    np.abs(z)
                    <=
                    h27_abs_z_95
                ).astype(int),
        }
    )

    return out.merge(
        h27_csy_meta[
            [
                "csy",
                "constructorId",
                "constructor",
                "year",
                "beta",
                "beta_percentile_within_year",
                "car_strength_quartile",
            ]
        ],
        on="csy",
        how="left",
        validate="many_to_one",
    )


h27_pc_resid = (
    h27_build_contrast_residuals(
        h27_pc_test,
        channel=
            "pace_contrast",
        sigma=
            h27_sigma[
                "pace_contrast"
            ],
    )
)

h27_pl_resid = (
    h27_build_level_residuals(
        h27_pl_test,
        channel=
            "pace_level",
        sigma=
            h27_sigma[
                "pace_level"
            ],
    )
)

h27_qc_resid = (
    h27_build_contrast_residuals(
        h27_qc_test,
        channel=
            "quali_contrast",
        sigma=
            h27_sigma[
                "quali_contrast"
            ],
    )
)

h27_ql_resid = (
    h27_build_level_residuals(
        h27_ql_test,
        channel=
            "q1_level",
        sigma=
            h27_sigma[
                "q1_level"
            ],
    )
)


h27_residual_rows = pd.concat(
    [
        h27_pc_resid,
        h27_pl_resid,
        h27_qc_resid,
        h27_ql_resid,
    ],
    ignore_index=True,
)


# ==========================================================
# 11. ANALYSIS GROUPS
# ==========================================================
#
# Qualifying is analysed both:
#
#   all common sessions together
#   Q1 only
#   Q2 only
#   Q3 only
#
# Q1-only is the most important session-control diagnostic.
# ==========================================================

h27_analysis_frames = {
    "pace_contrast":
        h27_pc_resid.copy(),

    "pace_level":
        h27_pl_resid.copy(),

    "quali_contrast":
        h27_qc_resid.copy(),

    "q1_level":
        h27_ql_resid.copy(),

    "q1_contrast":
        h27_qc_resid[
            h27_qc_resid[
                "session"
            ]
            ==
            "q1"
        ].copy(),

    "q2_contrast":
        h27_qc_resid[
            h27_qc_resid[
                "session"
            ]
            ==
            "q2"
        ].copy(),

    "q3_contrast":
        h27_qc_resid[
            h27_qc_resid[
                "session"
            ]
            ==
            "q3"
        ].copy(),

    "LEVEL POOLED":
        pd.concat(
            [
                h27_pl_resid,
                h27_ql_resid,
            ],
            ignore_index=True,
        ),

    "CONTRAST POOLED":
        pd.concat(
            [
                h27_pc_resid,
                h27_qc_resid,
            ],
            ignore_index=True,
        ),

    "ALL POOLED":
        h27_residual_rows.copy(),
}


# ==========================================================
# 12. CONSTRUCTOR-SEASON HELD-OUT RESIDUAL SCALES
# ==========================================================

def h27_build_csy_scale_table(
    frame,
    *,
    analysis_group,
):

    rows = []

    grouped = frame.groupby(
        [
            "csy",
            "year",
            "constructorId",
            "constructor",
            "beta",
            "car_strength_quartile",
        ],
        observed=False,
        dropna=False,
    )

    for (
        csy,
        year,
        constructor_id,
        constructor,
        beta,
        quartile,
    ), grp in grouped:

        if (
            len(grp)
            <
            H27_MIN_ROWS_PER_CSY
        ):
            continue

        z = grp[
            "z"
        ].to_numpy(
            dtype=float
        )

        rows.append(
            {
                "analysis_group":
                    analysis_group,
                "csy":
                    int(csy),
                "year":
                    int(year),
                "constructorId":
                    int(
                        constructor_id
                    ),
                "constructor":
                    str(
                        constructor
                    ),
                "beta":
                    float(beta),
                "car_strength_quartile":
                    quartile,
                "heldout_rows":
                    int(
                        len(grp)
                    ),
                "robust_scale":
                    robust_scale_mad(
                        z
                    ),
                "median_abs_z":
                    float(
                        np.median(
                            np.abs(z)
                        )
                    ),
                "mean_abs_z":
                    float(
                        np.mean(
                            np.abs(z)
                        )
                    ),
                "rms_z":
                    float(
                        np.sqrt(
                            np.mean(
                                z ** 2
                            )
                        )
                    ),
                "tail_gt_2":
                    float(
                        np.mean(
                            np.abs(z)
                            >
                            2.0
                        )
                    ),
                "tail_gt_3":
                    float(
                        np.mean(
                            np.abs(z)
                            >
                            3.0
                        )
                    ),
                "coverage_conditional_89":
                    float(
                        grp[
                            "inside_conditional_89"
                        ].mean()
                    ),
                "coverage_conditional_95":
                    float(
                        grp[
                            "inside_conditional_95"
                        ].mean()
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


h27_scale_tables = []

for (
    analysis_group,
    frame,
) in h27_analysis_frames.items():

    if len(
        frame
    ) == 0:
        continue

    scale_table = (
        h27_build_csy_scale_table(
            frame,
            analysis_group=
                analysis_group,
        )
    )

    if len(
        scale_table
    ):
        h27_scale_tables.append(
            scale_table
        )


h27_csy_scales = pd.concat(
    h27_scale_tables,
    ignore_index=True,
)


# ==========================================================
# 13. FAST YEAR-CLUSTER BOOTSTRAP
# ==========================================================

h27_boot_rng = np.random.default_rng(
    H27_BOOTSTRAP_SEED
)


def h27_fast_fe_bootstrap(
    df,
):
    """
    Year-fixed-effect slope of log(residual scale) on beta.

    Bootstrap resamples entire seasons using precomputed within-year
    sufficient statistics.
    """

    x = df[
        [
            "year",
            "beta",
            "robust_scale",
        ]
    ].dropna().copy()

    x = x[
        x[
            "robust_scale"
        ]
        >
        0
    ].copy()

    per_year = []

    for year, grp in x.groupby(
        "year"
    ):

        beta = grp[
            "beta"
        ].to_numpy(
            dtype=float
        )

        log_scale = np.log(
            grp[
                "robust_scale"
            ].to_numpy(
                dtype=float
            )
        )

        if len(
            beta
        ) < 2:
            continue

        beta_dm = (
            beta
            -
            beta.mean()
        )

        scale_dm = (
            log_scale
            -
            log_scale.mean()
        )

        sxx = float(
            np.sum(
                beta_dm
                ** 2
            )
        )

        sxy = float(
            np.sum(
                beta_dm
                *
                scale_dm
            )
        )

        if (
            sxx
            >
            0
        ):
            per_year.append(
                (
                    int(year),
                    sxx,
                    sxy,
                )
            )

    if len(
        per_year
    ) < 3:

        return (
            np.nan,
            np.nan,
            np.nan,
        )

    sxx = np.asarray(
        [
            row[1]
            for row
            in per_year
        ],
        dtype=float,
    )

    sxy = np.asarray(
        [
            row[2]
            for row
            in per_year
        ],
        dtype=float,
    )

    observed = float(
        sxy.sum()
        /
        sxx.sum()
    )

    n_years = len(
        per_year
    )

    sampled = (
        h27_boot_rng.integers(
            0,
            n_years,
            size=(
                H27_BOOTSTRAPS,
                n_years,
            ),
        )
    )

    boot_sxx = (
        sxx[
            sampled
        ]
        .sum(
            axis=1
        )
    )

    boot_sxy = (
        sxy[
            sampled
        ]
        .sum(
            axis=1
        )
    )

    valid = (
        boot_sxx
        >
        0
    )

    boot = (
        boot_sxy[
            valid
        ]
        /
        boot_sxx[
            valid
        ]
    )

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


def h27_observed_bottom_top_ratio(
    df,
):

    bottom = df.loc[
        df[
            "car_strength_quartile"
        ]
        ==
        "Q1 weakest",
        "robust_scale",
    ].dropna().to_numpy(
        dtype=float
    )

    top = df.loc[
        df[
            "car_strength_quartile"
        ]
        ==
        "Q4 strongest",
        "robust_scale",
    ].dropna().to_numpy(
        dtype=float
    )

    if (
        len(bottom) == 0
        or
        len(top) == 0
    ):
        return np.nan

    top_median = float(
        np.median(
            top
        )
    )

    if (
        top_median
        <=
        0
    ):
        return np.nan

    return float(
        np.median(
            bottom
        )
        /
        top_median
    )


def h27_fast_ratio_bootstrap(
    df,
):
    """
    Resample whole seasons. Within each sampled season preserve all
    constructor-season residual-scale estimates.
    """

    years = np.asarray(
        sorted(
            df[
                "year"
            ]
            .dropna()
            .unique()
        ),
        dtype=int,
    )

    if len(
        years
    ) < 3:

        return (
            np.nan,
            np.nan,
            np.nan,
        )

    bottom_by_year = {}
    top_by_year = {}

    for year in years:

        year_df = df[
            df[
                "year"
            ]
            ==
            year
        ]

        bottom_by_year[
            year
        ] = (
            year_df.loc[
                year_df[
                    "car_strength_quartile"
                ]
                ==
                "Q1 weakest",
                "robust_scale",
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

        top_by_year[
            year
        ] = (
            year_df.loc[
                year_df[
                    "car_strength_quartile"
                ]
                ==
                "Q4 strongest",
                "robust_scale",
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

    observed = (
        h27_observed_bottom_top_ratio(
            df
        )
    )

    n_years = len(
        years
    )

    sampled_year_indices = (
        h27_boot_rng.integers(
            0,
            n_years,
            size=(
                H27_BOOTSTRAPS,
                n_years,
            ),
        )
    )

    boot = np.full(
        H27_BOOTSTRAPS,
        np.nan,
        dtype=float,
    )

    for b in range(
        H27_BOOTSTRAPS
    ):

        sampled_years = (
            years[
                sampled_year_indices[
                    b
                ]
            ]
        )

        bottom_parts = [
            bottom_by_year[
                year
            ]
            for year
            in sampled_years
            if len(
                bottom_by_year[
                    year
                ]
            )
        ]

        top_parts = [
            top_by_year[
                year
            ]
            for year
            in sampled_years
            if len(
                top_by_year[
                    year
                ]
            )
        ]

        if (
            not bottom_parts
            or
            not top_parts
        ):
            continue

        bottom = np.concatenate(
            bottom_parts
        )

        top = np.concatenate(
            top_parts
        )

        top_median = float(
            np.median(
                top
            )
        )

        if (
            top_median
            <=
            0
        ):
            continue

        boot[
            b
        ] = (
            float(
                np.median(
                    bottom
                )
            )
            /
            top_median
        )

    boot = boot[
        np.isfinite(
            boot
        )
    ]

    if len(
        boot
    ) == 0:

        return (
            observed,
            np.nan,
            np.nan,
        )

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


# ==========================================================
# 14. PRIMARY HELD-OUT HETEROSKEDASTICITY TESTS
# ==========================================================

h27_test_order = [
    "pace_level",
    "q1_level",
    "LEVEL POOLED",
    "pace_contrast",
    "quali_contrast",
    "q1_contrast",
    "q2_contrast",
    "q3_contrast",
    "CONTRAST POOLED",
    "ALL POOLED",
]


h27_test_rows = []

for analysis_group in h27_test_order:

    grp = h27_csy_scales[
        h27_csy_scales[
            "analysis_group"
        ]
        ==
        analysis_group
    ].copy()

    grp = grp[
        grp[
            "robust_scale"
        ].notna()
        &
        (
            grp[
                "robust_scale"
            ]
            >
            0
        )
    ]

    if len(
        grp
    ) < 5:
        continue

    (
        fe_slope,
        fe_lb,
        fe_ub,
    ) = h27_fast_fe_bootstrap(
        grp
    )

    (
        ratio,
        ratio_lb,
        ratio_ub,
    ) = h27_fast_ratio_bootstrap(
        grp
    )

    h27_test_rows.append(
        {
            "analysis_group":
                analysis_group,
            "constructor_seasons":
                int(
                    len(grp)
                ),
            "years":
                int(
                    grp[
                        "year"
                    ].nunique()
                ),
            "spearman_beta_vs_scale":
                safe_spearman(
                    grp[
                        "beta"
                    ],
                    grp[
                        "robust_scale"
                    ],
                ),
            "year_FE_logscale_slope":
                fe_slope,
            "slope_95_lb":
                fe_lb,
            "slope_95_ub":
                fe_ub,
            "bottom_vs_top_scale_ratio":
                ratio,
            "ratio_95_lb":
                ratio_lb,
            "ratio_95_ub":
                ratio_ub,
            "mean_robust_scale":
                float(
                    grp[
                        "robust_scale"
                    ].mean()
                ),
        }
    )


h27_primary_tests = pd.DataFrame(
    h27_test_rows
)


print(
    "\n========================================"
)
print(
    "PRIMARY HELD-OUT HETEROSKEDASTICITY TEST"
)
print(
    "========================================"
)

display(
    h27_primary_tests
)


# ==========================================================
# 15. ROW-LEVEL HELD-OUT TAIL / COVERAGE SUMMARY
# ==========================================================

h27_quartile_order = [
    "Q1 weakest",
    "Q2",
    "Q3",
    "Q4 strongest",
]


row_summary_frames = []

for (
    analysis_group,
    frame,
) in h27_analysis_frames.items():

    if len(
        frame
    ) == 0:
        continue

    summary = (
        frame.groupby(
            "car_strength_quartile",
            observed=False,
            as_index=False,
        )
        .agg(
            heldout_rows=(
                "z",
                "size",
            ),
            constructor_seasons=(
                "csy",
                "nunique",
            ),
            median_beta=(
                "beta",
                "median",
            ),
            mean_abs_z=(
                "abs_z",
                "mean",
            ),
            tail_gt_2=(
                "abs_z",
                lambda x:
                    float(
                        np.mean(
                            np.asarray(
                                x,
                                dtype=float,
                            )
                            >
                            2.0
                        )
                    ),
            ),
            tail_gt_3=(
                "abs_z",
                lambda x:
                    float(
                        np.mean(
                            np.asarray(
                                x,
                                dtype=float,
                            )
                            >
                            3.0
                        )
                    ),
            ),
            conditional_89_coverage=(
                "inside_conditional_89",
                "mean",
            ),
            conditional_95_coverage=(
                "inside_conditional_95",
                "mean",
            ),
        )
    )

    summary[
        "analysis_group"
    ] = analysis_group

    row_summary_frames.append(
        summary
    )


h27_quartile_summary = pd.concat(
    row_summary_frames,
    ignore_index=True,
)


h27_quartile_summary[
    "car_strength_quartile"
] = pd.Categorical(
    h27_quartile_summary[
        "car_strength_quartile"
    ],
    categories=
        h27_quartile_order,
    ordered=True,
)


h27_quartile_summary = (
    h27_quartile_summary
    .sort_values(
        [
            "analysis_group",
            "car_strength_quartile",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========================================"
)
print(
    "HELD-OUT RESIDUALS BY CAR-STRENGTH QUARTILE"
)
print(
    "========================================"
)

display(
    h27_quartile_summary[
        h27_quartile_summary[
            "analysis_group"
        ].isin(
            [
                "pace_contrast",
                "quali_contrast",
                "q1_contrast",
                "q2_contrast",
                "q3_contrast",
                "q1_level",
            ]
        )
    ]
)


# ==========================================================
# 16. QUALIFYING-SESSION DECOMPOSITION
# ==========================================================

h27_session_tests = (
    h27_primary_tests[
        h27_primary_tests[
            "analysis_group"
        ].isin(
            [
                "quali_contrast",
                "q1_contrast",
                "q2_contrast",
                "q3_contrast",
            ]
        )
    ]
    .copy()
)


print(
    "\n========================================"
)
print(
    "QUALIFYING SESSION DECOMPOSITION"
)
print(
    "========================================"
)

display(
    h27_session_tests
)


# ==========================================================
# 17. COMPARE WITH STAGE 26 IN-SAMPLE RESULT, IF AVAILABLE
# ==========================================================

stage26_primary_path = (
    Path(CFG.output_dir)
    /
    "validation_real_data_heteroskedasticity"
    /
    "heteroskedasticity_primary_tests.csv"
)


if stage26_primary_path.exists():

    h27_stage26 = pd.read_csv(
        stage26_primary_path
    )

    stage26_group_col = (
        "group"
        if "group"
        in h27_stage26.columns
        else "analysis_group"
    )

    common_groups = [
        "pace_level",
        "q1_level",
        "LEVEL POOLED",
        "pace_contrast",
        "quali_contrast",
        "CONTRAST POOLED",
        "ALL POOLED",
    ]

    stage26_compare = (
        h27_stage26[
            h27_stage26[
                stage26_group_col
            ].isin(
                common_groups
            )
        ][
            [
                stage26_group_col,
                "year_FE_logscale_slope",
                "slope_95_lb",
                "slope_95_ub",
                "bottom_vs_top_scale_ratio",
                "ratio_95_lb",
                "ratio_95_ub",
            ]
        ]
        .rename(
            columns={
                stage26_group_col:
                    "analysis_group",
                "year_FE_logscale_slope":
                    "stage26_in_sample_slope",
                "slope_95_lb":
                    "stage26_slope_lb",
                "slope_95_ub":
                    "stage26_slope_ub",
                "bottom_vs_top_scale_ratio":
                    "stage26_in_sample_ratio",
                "ratio_95_lb":
                    "stage26_ratio_lb",
                "ratio_95_ub":
                    "stage26_ratio_ub",
            }
        )
        .merge(
            h27_primary_tests[
                [
                    "analysis_group",
                    "year_FE_logscale_slope",
                    "slope_95_lb",
                    "slope_95_ub",
                    "bottom_vs_top_scale_ratio",
                    "ratio_95_lb",
                    "ratio_95_ub",
                ]
            ]
            .rename(
                columns={
                    "year_FE_logscale_slope":
                        "stage27_heldout_slope",
                    "slope_95_lb":
                        "stage27_slope_lb",
                    "slope_95_ub":
                        "stage27_slope_ub",
                    "bottom_vs_top_scale_ratio":
                        "stage27_heldout_ratio",
                    "ratio_95_lb":
                        "stage27_ratio_lb",
                    "ratio_95_ub":
                        "stage27_ratio_ub",
                }
            ),
            on=
                "analysis_group",
            how=
                "inner",
            validate=
                "one_to_one",
        )
    )

    print(
        "\n========================================"
    )
    print(
        "STAGE 26 IN-SAMPLE vs STAGE 27 HELD-OUT"
    )
    print(
        "========================================"
    )

    display(
        stage26_compare
    )

else:

    stage26_compare = pd.DataFrame()

    print(
        "\nStage 26 primary-test CSV not found; "
        "skipping direct Stage 26/27 comparison."
    )


# ==========================================================
# 18. PLOTS
# ==========================================================

# ----------------------------------------------------------
# A. Bottom-vs-top scale ratios with bootstrap intervals
# ----------------------------------------------------------

ratio_plot_groups = [
    "pace_contrast",
    "quali_contrast",
    "q1_contrast",
    "q2_contrast",
    "q3_contrast",
    "q1_level",
]

ratio_plot = (
    h27_primary_tests[
        h27_primary_tests[
            "analysis_group"
        ].isin(
            ratio_plot_groups
        )
    ]
    .copy()
)

ratio_plot[
    "_order"
] = ratio_plot[
    "analysis_group"
].map(
    {
        name: i
        for i, name
        in enumerate(
            ratio_plot_groups
        )
    }
)

ratio_plot = ratio_plot.sort_values(
    "_order"
)


if len(
    ratio_plot
):

    x = np.arange(
        len(
            ratio_plot
        )
    )

    y = ratio_plot[
        "bottom_vs_top_scale_ratio"
    ].to_numpy(
        dtype=float
    )

    yerr = np.vstack(
        [
            y
            -
            ratio_plot[
                "ratio_95_lb"
            ].to_numpy(
                dtype=float
            ),
            ratio_plot[
                "ratio_95_ub"
            ].to_numpy(
                dtype=float
            )
            -
            y,
        ]
    )

    plt.figure(
        figsize=(
            10,
            6,
        )
    )

    plt.errorbar(
        x,
        y,
        yerr=
            yerr,
        fmt="o",
        capsize=4,
    )

    plt.axhline(
        1.0,
        linestyle="--",
        linewidth=1.2,
    )

    plt.xticks(
        x,
        ratio_plot[
            "analysis_group"
        ],
        rotation=30,
        ha="right",
    )

    plt.ylabel(
        "Weakest / strongest constructor residual-scale ratio"
    )

    plt.title(
        "Held-out heteroskedasticity by evidence channel"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------
# B. Q1 contrast vs pace contrast across car quartiles
# ----------------------------------------------------------

plt.figure(
    figsize=(
        9,
        6,
    )
)

for analysis_group in [
    "pace_contrast",
    "q1_contrast",
]:

    x = h27_quartile_summary[
        h27_quartile_summary[
            "analysis_group"
        ]
        ==
        analysis_group
    ].copy()

    if x.empty:
        continue

    x = x.sort_values(
        "car_strength_quartile"
    )

    plt.plot(
        x[
            "car_strength_quartile"
        ].astype(str),
        x[
            "mean_abs_z"
        ],
        marker="o",
        label=
            analysis_group,
    )

plt.xlabel(
    "Constructor strength quartile within season"
)

plt.ylabel(
    "Mean |held-out standardised residual|"
)

plt.title(
    "Held-out race pace vs Q1 teammate-contrast variability"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# C. Conditional 89% coverage in Q1 contrast
# ----------------------------------------------------------

q1_cov_plot = (
    h27_quartile_summary[
        h27_quartile_summary[
            "analysis_group"
        ]
        ==
        "q1_contrast"
    ]
    .copy()
    .sort_values(
        "car_strength_quartile"
    )
)

if len(
    q1_cov_plot
):

    plt.figure(
        figsize=(
            8,
            5,
        )
    )

    plt.plot(
        q1_cov_plot[
            "car_strength_quartile"
        ].astype(str),
        q1_cov_plot[
            "conditional_89_coverage"
        ],
        marker="o",
    )

    plt.axhline(
        0.89,
        linestyle="--",
        linewidth=1.2,
    )

    plt.ylim(
        0.0,
        1.0,
    )

    plt.xlabel(
        "Constructor strength quartile within season"
    )

    plt.ylabel(
        "Held-out conditional 89% coverage"
    )

    plt.title(
        "Q1 teammate-contrast coverage by constructor strength"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()


# ==========================================================
# 19. SAVE OUTPUTS
# ==========================================================

split_count_table.to_csv(
    H27_OUTPUT_DIR
    /
    "train_test_row_counts.csv",
    index=False,
)

h27_health.to_csv(
    H27_OUTPUT_DIR
    /
    "validation_fit_health.csv",
    index=False,
)

h27_residual_rows.to_csv(
    H27_OUTPUT_DIR
    /
    "heldout_standardised_residual_rows.csv",
    index=False,
)

h27_csy_scales.to_csv(
    H27_OUTPUT_DIR
    /
    "heldout_constructor_season_residual_scales.csv",
    index=False,
)

h27_primary_tests.to_csv(
    H27_OUTPUT_DIR
    /
    "heldout_heteroskedasticity_primary_tests.csv",
    index=False,
)

h27_quartile_summary.to_csv(
    H27_OUTPUT_DIR
    /
    "heldout_residual_quartile_summary.csv",
    index=False,
)

h27_session_tests.to_csv(
    H27_OUTPUT_DIR
    /
    "heldout_qualifying_session_decomposition.csv",
    index=False,
)

if len(
    stage26_compare
):
    stage26_compare.to_csv(
        H27_OUTPUT_DIR
        /
        "stage26_vs_stage27_comparison.csv",
        index=False,
    )


# ==========================================================
# 20. FINAL TRIAGE
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 27 TRIAGE SUMMARY"
)
print(
    "========================================"
)


def h27_print_result(
    analysis_group,
):

    x = h27_primary_tests[
        h27_primary_tests[
            "analysis_group"
        ]
        ==
        analysis_group
    ]

    if x.empty:

        print(
            f"{analysis_group}: "
            "insufficient held-out constructor-season data"
        )

        return

    row = x.iloc[0]

    print(
        f"{analysis_group}:"
    )

    print(
        f"  FE log-scale slope vs beta = "
        f"{row['year_FE_logscale_slope']:.4f} "
        f"[95% "
        f"{row['slope_95_lb']:.4f}, "
        f"{row['slope_95_ub']:.4f}]"
    )

    print(
        f"  Weakest/strongest scale ratio = "
        f"{row['bottom_vs_top_scale_ratio']:.3f} "
        f"[95% "
        f"{row['ratio_95_lb']:.3f}, "
        f"{row['ratio_95_ub']:.3f}]"
    )

    print(
        f"  Spearman(beta, scale) = "
        f"{row['spearman_beta_vs_scale']:.3f}"
    )


h27_print_result(
    "pace_contrast"
)

h27_print_result(
    "quali_contrast"
)

h27_print_result(
    "q1_contrast"
)

h27_print_result(
    "q2_contrast"
)

h27_print_result(
    "q3_contrast"
)

h27_print_result(
    "q1_level"
)


# ----------------------------------------------------------
# Simple evidence labels.
# ----------------------------------------------------------

def h27_evidence_label(
    analysis_group,
):

    x = h27_primary_tests[
        h27_primary_tests[
            "analysis_group"
        ]
        ==
        analysis_group
    ]

    if x.empty:
        return "INSUFFICIENT DATA"

    row = x.iloc[0]

    ratio_support = (
        pd.notna(
            row[
                "ratio_95_lb"
            ]
        )
        and
        row[
            "ratio_95_lb"
        ]
        >
        1.0
    )

    slope_support = (
        pd.notna(
            row[
                "slope_95_ub"
            ]
        )
        and
        row[
            "slope_95_ub"
        ]
        <
        0.0
    )

    point_direction = (
        pd.notna(
            row[
                "bottom_vs_top_scale_ratio"
            ]
        )
        and
        row[
            "bottom_vs_top_scale_ratio"
        ]
        >
        1.0
        and
        pd.notna(
            row[
                "year_FE_logscale_slope"
            ]
        )
        and
        row[
            "year_FE_logscale_slope"
        ]
        <
        0.0
    )

    if (
        ratio_support
        and
        slope_support
    ):
        return "STRONG HELD-OUT SUPPORT"

    if (
        ratio_support
        or
        slope_support
    ):
        return "HELD-OUT SUPPORT"

    if point_direction:
        return "DIRECTIONALLY CONSISTENT, UNCERTAIN"

    return "NO CLEAR HELD-OUT SUPPORT"


print(
    "\nEvidence labels:"
)

for group in [
    "pace_contrast",
    "quali_contrast",
    "q1_contrast",
    "q2_contrast",
    "q3_contrast",
    "q1_level",
]:
    print(
        f"  {group:18s}: "
        f"{h27_evidence_label(group)}"
    )


print(
    "\nHow to interpret the key comparison:"
)

print(
    "  1. If q1_contrast still shows a weakest/strongest ratio > 1"
)

print(
    "     with held-out support, the qualifying effect survives after"
)

print(
    "     controlling for session. It is then genuinely associated"
)

print(
    "     with weaker constructor-seasons, not just Q1/Q2/Q3 mixing."
)

print(
    "  2. If pooled qualifying is noisy but q1_contrast is null,"
)

print(
    "     session advancement/composition is the leading explanation."
)

print(
    "  3. If pace_contrast remains null while q1_contrast is positive,"
)

print(
    "     the misspecification is qualifying-specific rather than a"
)

print(
    "     general 'bad cars are noisier everywhere' effect."
)

print(
    "  4. Conditional 89% coverage below 0.89 in the weakest-car"
)

print(
    "     quartile is direct evidence that the common qualifying scale"
)

print(
    "     understates weak-car observation uncertainty."
)

print(
    "\nNo production-model change should be made from Stage 27 alone."
)

print(
    "If the held-out Q1 effect survives, the next step is to compare"
)

print(
    "the frozen model against a deliberately minimal heteroskedastic"
)

print(
    "qualifying extension on held-out predictive performance."
)

print(
    "\nSaved outputs ->"
)

print(
    H27_OUTPUT_DIR.resolve()
)

print(
    "\nStage 27 complete."
)


In [ ]:
# ==========================================================
# STAGE 28 — Q1 QUALIFYING-WEIGHT CONFOUND AUDIT
# ==========================================================
#
# PURPOSE
# -------
# Stage 27 found:
#
#   * no held-out heteroskedasticity in race-pace teammate contrasts;
#   * suggestive extra Q1 teammate-contrast variability in weak cars;
#   * clear held-out heteroskedasticity in pooled team-level evidence.
#
# But qualifying teammate contrasts use:
#
#       weight = 1 / n_common_sessions
#
# so a pair that only shares Q1 has weight 1, while a pair that also
# shares Q2/Q3 has weight 1/2 or 1/3. Because weak cars are more likely
# to disappear after Q1, car strength and this weight can be associated.
#
# Stage 28 asks:
#
#   1. How strongly is Q1 weight / later-session progression associated
#      with constructor strength?
#
#   2. Does the weak-car Q1 variance pattern remain if we REMOVE the
#      model's qualifying information-weight scaling from the residual
#      diagnostic?
#
#   3. Does it remain after explicitly controlling for
#      n_common_sessions?
#
#   4. Within exact n_common_sessions strata (1, 2, 3), is there still
#      a car-strength / Q1-variance relationship?
#
#   5. Does the unweighted Q1 TEAM-LEVEL evidence still show the
#      weak-car variance pattern?
#
# NO MCMC IS RUN HERE.
# This cell uses only the held-out residuals from Stage 27.
#
# PRIMARY DEFINITIONS
# -------------------
# Stage-27 likelihood-standardised Q1 contrast residual:
#
#       z_current = residual * sqrt(weight) / sigma_q
#
# Weight-neutral diagnostic residual:
#
#       z_raw = residual / sigma_q
#             = z_current / sqrt(weight)
#
# If the Stage-27 Q1 pattern disappears for z_raw and after controlling
# for n_common_sessions, the apparent heteroskedasticity was largely
# induced by the qualifying information-weight scheme / progression mix.
#
# If it survives those checks, then weak-car Q1 heteroskedasticity is
# much more likely to be genuine.
#
# NOTE
# ----
# Conditioning on n_common_sessions is a diagnostic control, not a
# causal estimand: advancement itself depends on performance. The most
# persuasive evidence is agreement across:
#
#   * weight-neutral constructor-season scales,
#   * controlled row-level regression,
#   * exact-weight strata,
#   * unweighted Q1 team-level benchmark.
# ==========================================================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

H28_BOOTSTRAPS = 3000
H28_BOOTSTRAP_SEED = 28042

# Same minimum as Stage 27 so the weighted/raw constructor-season
# comparison is apples-to-apples.
H28_MIN_ROWS_PER_CSY = 3

H28_OUTPUT_DIR = (
    Path(CFG.output_dir)
    / "validation_q1_weight_confound"
)

H28_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==========================================================
# 1. LOAD STAGE-27 HELD-OUT RESIDUALS
# ==========================================================

stage27_residual_path = (
    Path(CFG.output_dir)
    / "validation_heldout_heteroskedasticity"
    / "heldout_standardised_residual_rows.csv"
)


if (
    "h27_residual_rows"
    in globals()
):
    h28_source = (
        h27_residual_rows
        .copy()
    )

elif stage27_residual_path.exists():
    h28_source = pd.read_csv(
        stage27_residual_path
    )

else:
    raise RuntimeError(
        "Stage 28 needs the held-out residual rows from Stage 27.\n"
        "Run Stage 27 first, or ensure this file exists:\n"
        f"{stage27_residual_path}"
    )


required_columns = [
    "raceId",
    "channel",
    "csy",
    "year",
    "beta",
    "car_strength_quartile",
    "weight",
    "z",
]

missing = [
    col
    for col in required_columns
    if col not in h28_source.columns
]

if missing:
    raise RuntimeError(
        "Stage-27 residual table is missing required columns: "
        + ", ".join(missing)
    )


if (
    "session"
    not in h28_source.columns
):
    raise RuntimeError(
        "Stage-27 residual table has no `session` column, so Q1 "
        "qualifying contrasts cannot be isolated."
    )


# ==========================================================
# 2. EXTRACT Q1 TEAMMATE CONTRASTS + Q1 TEAM LEVEL
# ==========================================================

h28_session = (
    h28_source[
        "session"
    ]
    .astype(str)
    .str.lower()
)


h28_q1 = (
    h28_source[
        (
            h28_source[
                "channel"
            ]
            ==
            "quali_contrast"
        )
        &
        (
            h28_session
            ==
            "q1"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


h28_q1_level = (
    h28_source[
        h28_source[
            "channel"
        ]
        ==
        "q1_level"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if h28_q1.empty:
    raise RuntimeError(
        "No held-out Q1 qualifying-contrast residual rows were found."
    )


if h28_q1_level.empty:
    raise RuntimeError(
        "No held-out Q1 team-level residual rows were found."
    )


# ==========================================================
# 3. RECOVER n_common_sessions FROM THE MODEL WEIGHT
# ==========================================================
#
# Frozen qualifying contrast construction:
#
#       weight = 1 / n_common_sessions
#
# Validate this instead of silently assuming it.
# ==========================================================

if (
    h28_q1[
        "weight"
    ]
    <=
    0
).any():
    raise RuntimeError(
        "Q1 qualifying contrast contains non-positive weights."
    )


h28_q1[
    "n_common_sessions_float"
] = (
    1.0
    /
    h28_q1[
        "weight"
    ]
)


h28_q1[
    "n_common_sessions"
] = np.rint(
    h28_q1[
        "n_common_sessions_float"
    ]
).astype(int)


h28_q1[
    "expected_weight"
] = (
    1.0
    /
    h28_q1[
        "n_common_sessions"
    ]
)


h28_max_weight_deviation = float(
    np.max(
        np.abs(
            h28_q1[
                "weight"
            ]
            -
            h28_q1[
                "expected_weight"
            ]
        )
    )
)


print(
    "========================================"
)
print(
    "STAGE 28 — Q1 QUALIFYING-WEIGHT CONFOUND AUDIT"
)
print(
    "========================================"
)

print(
    "Held-out Q1 contrast rows:",
    len(
        h28_q1
    ),
)

print(
    "Held-out Q1-level rows:",
    len(
        h28_q1_level
    ),
)

print(
    "Recovered n_common_sessions values:",
    sorted(
        h28_q1[
            "n_common_sessions"
        ].unique()
    ),
)

print(
    "Max |observed weight - 1/n_common_sessions|:",
    h28_max_weight_deviation,
)


if (
    h28_max_weight_deviation
    >
    1e-8
):
    print(
        "WARNING: weights are not exactly 1/n_common_sessions. "
        "Inspect before treating the recovered strata as exact."
    )


# ==========================================================
# 4. CONSTRUCT CURRENT VS WEIGHT-NEUTRAL RESIDUALS
# ==========================================================
#
# Stage 27:
#
#     z_current = residual * sqrt(weight) / sigma
#
# Therefore:
#
#     z_raw = z_current / sqrt(weight)
#           = residual / sigma
#
# `z_raw` removes ONLY the information-weight rescaling. It does not
# change the fitted mean or refit the model.
# ==========================================================

h28_q1[
    "z_current"
] = h28_q1[
    "z"
].astype(float)


h28_q1[
    "z_raw"
] = (
    h28_q1[
        "z_current"
    ]
    /
    np.sqrt(
        h28_q1[
            "weight"
        ].astype(float)
    )
)


h28_q1[
    "abs_z_current"
] = np.abs(
    h28_q1[
        "z_current"
    ]
)


h28_q1[
    "abs_z_raw"
] = np.abs(
    h28_q1[
        "z_raw"
    ]
)


# Q1 level is our benchmark because its construction is Q1-only.
# Check rather than assume that its row weights are all 1.
h28_q1_level[
    "z_level"
] = h28_q1_level[
    "z"
].astype(float)


h28_q1_level[
    "abs_z_level"
] = np.abs(
    h28_q1_level[
        "z_level"
    ]
)


h28_level_weight_summary = (
    h28_q1_level[
        "weight"
    ]
    .astype(float)
    .describe()
)


# ==========================================================
# 5. HOW STRONGLY IS QUALIFYING WEIGHT ASSOCIATED WITH CAR STRENGTH?
# ==========================================================

quartile_order = [
    "Q1 weakest",
    "Q2",
    "Q3",
    "Q4 strongest",
]


weight_mix = (
    h28_q1.groupby(
        [
            "car_strength_quartile",
            "n_common_sessions",
        ],
        observed=False,
    )
    .size()
    .rename(
        "rows"
    )
    .reset_index()
)


weight_mix[
    "share_within_car_quartile"
] = (
    weight_mix[
        "rows"
    ]
    /
    weight_mix.groupby(
        "car_strength_quartile",
        observed=False,
    )[
        "rows"
    ]
    .transform(
        "sum"
    )
)


weight_mix[
    "car_strength_quartile"
] = pd.Categorical(
    weight_mix[
        "car_strength_quartile"
    ],
    categories=
        quartile_order,
    ordered=True,
)


weight_mix = (
    weight_mix
    .sort_values(
        [
            "car_strength_quartile",
            "n_common_sessions",
        ]
    )
    .reset_index(
        drop=True
    )
)


weight_pivot = (
    weight_mix
    .pivot(
        index=
            "car_strength_quartile",
        columns=
            "n_common_sessions",
        values=
            "share_within_car_quartile",
    )
    .fillna(0.0)
)


print(
    "\n========================================"
)
print(
    "Q1 QUALIFYING WEIGHT / PROGRESSION MIX BY CAR STRENGTH"
)
print(
    "========================================"
)

display(
    weight_mix
)

print(
    "\nShares within each car-strength quartile:"
)

display(
    weight_pivot
)


# Direct descriptive association.
h28_weight_assoc_spearman = float(
    st.spearmanr(
        h28_q1[
            "beta"
        ].astype(float),
        h28_q1[
            "n_common_sessions"
        ].astype(float),
    ).statistic
)


print(
    "Spearman(beta, n_common_sessions):",
    h28_weight_assoc_spearman,
)


# ==========================================================
# 6. ROBUST CONSTRUCTOR-SEASON SCALE TABLES
# ==========================================================

def h28_robust_scale_mad(
    values,
):
    x = np.asarray(
        values,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
    ]

    if len(x) < 2:
        return np.nan

    median = np.median(
        x
    )

    mad = np.median(
        np.abs(
            x
            -
            median
        )
    )

    return float(
        1.4826
        *
        mad
    )


def h28_build_csy_scales(
    frame,
    *,
    z_col,
    label,
):
    rows = []

    grouping_columns = [
        "csy",
        "year",
        "beta",
        "car_strength_quartile",
    ]

    optional_columns = [
        "constructorId",
        "constructor",
    ]

    for col in optional_columns:
        if col in frame.columns:
            grouping_columns.append(
                col
            )

    for keys, grp in frame.groupby(
        grouping_columns,
        observed=False,
        dropna=False,
    ):
        if (
            len(grp)
            <
            H28_MIN_ROWS_PER_CSY
        ):
            continue

        key_map = dict(
            zip(
                grouping_columns,
                (
                    keys
                    if isinstance(
                        keys,
                        tuple,
                    )
                    else (
                        keys,
                    )
                ),
            )
        )

        z = grp[
            z_col
        ].to_numpy(
            dtype=float
        )

        row = {
            "diagnostic":
                label,
            "csy":
                int(
                    key_map[
                        "csy"
                    ]
                ),
            "year":
                int(
                    key_map[
                        "year"
                    ]
                ),
            "beta":
                float(
                    key_map[
                        "beta"
                    ]
                ),
            "car_strength_quartile":
                key_map[
                    "car_strength_quartile"
                ],
            "heldout_rows":
                int(
                    len(grp)
                ),
            "robust_scale":
                h28_robust_scale_mad(
                    z
                ),
            "mean_abs_z":
                float(
                    np.mean(
                        np.abs(z)
                    )
                ),
            "median_abs_z":
                float(
                    np.median(
                        np.abs(z)
                    )
                ),
            "rms_z":
                float(
                    np.sqrt(
                        np.mean(
                            z ** 2
                        )
                    )
                ),
        }

        if (
            "constructorId"
            in key_map
        ):
            row[
                "constructorId"
            ] = key_map[
                "constructorId"
            ]

        if (
            "constructor"
            in key_map
        ):
            row[
                "constructor"
            ] = key_map[
                "constructor"
            ]

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    )


h28_current_scales = (
    h28_build_csy_scales(
        h28_q1,
        z_col=
            "z_current",
        label=
            "Q1 contrast — current weighted z",
    )
)


h28_raw_scales = (
    h28_build_csy_scales(
        h28_q1,
        z_col=
            "z_raw",
        label=
            "Q1 contrast — weight-neutral z",
    )
)


h28_level_scales = (
    h28_build_csy_scales(
        h28_q1_level,
        z_col=
            "z_level",
        label=
            "Q1 team level — benchmark",
    )
)


h28_csy_scales = pd.concat(
    [
        h28_current_scales,
        h28_raw_scales,
        h28_level_scales,
    ],
    ignore_index=True,
)


# ==========================================================
# 7. FAST YEAR-CLUSTER BOOTSTRAP FOR CSY-SCALE TESTS
# ==========================================================

h28_rng = np.random.default_rng(
    H28_BOOTSTRAP_SEED
)


def h28_fast_fe_scale_bootstrap(
    df,
):
    """
    Year-fixed-effect slope:
        log(robust_scale_csy) ~ beta_csy + year FE

    Bootstrap resamples whole years via precomputed sufficient stats.
    """

    x = df[
        [
            "year",
            "beta",
            "robust_scale",
        ]
    ].dropna().copy()

    x = x[
        x[
            "robust_scale"
        ]
        >
        0
    ].copy()

    per_year = []

    for year, grp in x.groupby(
        "year"
    ):
        beta = grp[
            "beta"
        ].to_numpy(
            dtype=float
        )

        log_scale = np.log(
            grp[
                "robust_scale"
            ].to_numpy(
                dtype=float
            )
        )

        if len(beta) < 2:
            continue

        beta_dm = (
            beta
            -
            beta.mean()
        )

        scale_dm = (
            log_scale
            -
            log_scale.mean()
        )

        sxx = float(
            np.sum(
                beta_dm
                ** 2
            )
        )

        sxy = float(
            np.sum(
                beta_dm
                *
                scale_dm
            )
        )

        if sxx > 0:
            per_year.append(
                (
                    int(year),
                    sxx,
                    sxy,
                )
            )

    if len(per_year) < 3:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    sxx = np.asarray(
        [
            row[1]
            for row in per_year
        ],
        dtype=float,
    )

    sxy = np.asarray(
        [
            row[2]
            for row in per_year
        ],
        dtype=float,
    )

    observed = float(
        sxy.sum()
        /
        sxx.sum()
    )

    n_years = len(
        per_year
    )

    sampled = h28_rng.integers(
        0,
        n_years,
        size=(
            H28_BOOTSTRAPS,
            n_years,
        ),
    )

    boot_sxx = sxx[
        sampled
    ].sum(
        axis=1
    )

    boot_sxy = sxy[
        sampled
    ].sum(
        axis=1
    )

    valid = (
        boot_sxx
        >
        0
    )

    boot = (
        boot_sxy[
            valid
        ]
        /
        boot_sxx[
            valid
        ]
    )

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


def h28_observed_bottom_top_ratio(
    df,
):
    bottom = df.loc[
        df[
            "car_strength_quartile"
        ]
        ==
        "Q1 weakest",
        "robust_scale",
    ].dropna().to_numpy(
        dtype=float
    )

    top = df.loc[
        df[
            "car_strength_quartile"
        ]
        ==
        "Q4 strongest",
        "robust_scale",
    ].dropna().to_numpy(
        dtype=float
    )

    if (
        len(bottom) == 0
        or
        len(top) == 0
    ):
        return np.nan

    top_median = float(
        np.median(
            top
        )
    )

    if top_median <= 0:
        return np.nan

    return float(
        np.median(
            bottom
        )
        /
        top_median
    )


def h28_fast_ratio_bootstrap(
    df,
):
    years = np.asarray(
        sorted(
            df[
                "year"
            ]
            .dropna()
            .unique()
        ),
        dtype=int,
    )

    if len(years) < 3:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    bottom_by_year = {}
    top_by_year = {}

    for year in years:
        year_df = df[
            df[
                "year"
            ]
            ==
            year
        ]

        bottom_by_year[
            year
        ] = (
            year_df.loc[
                year_df[
                    "car_strength_quartile"
                ]
                ==
                "Q1 weakest",
                "robust_scale",
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

        top_by_year[
            year
        ] = (
            year_df.loc[
                year_df[
                    "car_strength_quartile"
                ]
                ==
                "Q4 strongest",
                "robust_scale",
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

    observed = (
        h28_observed_bottom_top_ratio(
            df
        )
    )

    n_years = len(
        years
    )

    sampled_year_indices = (
        h28_rng.integers(
            0,
            n_years,
            size=(
                H28_BOOTSTRAPS,
                n_years,
            ),
        )
    )

    boot = np.full(
        H28_BOOTSTRAPS,
        np.nan,
        dtype=float,
    )

    for b in range(
        H28_BOOTSTRAPS
    ):
        sampled_years = years[
            sampled_year_indices[
                b
            ]
        ]

        bottom_parts = [
            bottom_by_year[
                year
            ]
            for year
            in sampled_years
            if len(
                bottom_by_year[
                    year
                ]
            )
        ]

        top_parts = [
            top_by_year[
                year
            ]
            for year
            in sampled_years
            if len(
                top_by_year[
                    year
                ]
            )
        ]

        if (
            not bottom_parts
            or
            not top_parts
        ):
            continue

        bottom = np.concatenate(
            bottom_parts
        )

        top = np.concatenate(
            top_parts
        )

        top_median = float(
            np.median(
                top
            )
        )

        if top_median <= 0:
            continue

        boot[
            b
        ] = (
            float(
                np.median(
                    bottom
                )
            )
            /
            top_median
        )

    boot = boot[
        np.isfinite(
            boot
        )
    ]

    if len(boot) == 0:
        return (
            observed,
            np.nan,
            np.nan,
        )

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


# ==========================================================
# 8. PRIMARY CURRENT-vs-RAW CONSTRUCTOR-SEASON TEST
# ==========================================================

h28_primary_rows = []

for diagnostic, grp in h28_csy_scales.groupby(
    "diagnostic",
    sort=False,
):
    grp = grp[
        grp[
            "robust_scale"
        ].notna()
        &
        (
            grp[
                "robust_scale"
            ]
            >
            0
        )
    ].copy()

    if len(grp) < 5:
        continue

    (
        slope,
        slope_lb,
        slope_ub,
    ) = h28_fast_fe_scale_bootstrap(
        grp
    )

    (
        ratio,
        ratio_lb,
        ratio_ub,
    ) = h28_fast_ratio_bootstrap(
        grp
    )

    h28_primary_rows.append(
        {
            "diagnostic":
                diagnostic,
            "constructor_seasons":
                int(
                    len(grp)
                ),
            "years":
                int(
                    grp[
                        "year"
                    ].nunique()
                ),
            "spearman_beta_vs_scale":
                float(
                    st.spearmanr(
                        grp[
                            "beta"
                        ].astype(float),
                        grp[
                            "robust_scale"
                        ].astype(float),
                    ).statistic
                ),
            "year_FE_logscale_slope":
                slope,
            "slope_95_lb":
                slope_lb,
            "slope_95_ub":
                slope_ub,
            "weakest_vs_strongest_scale_ratio":
                ratio,
            "ratio_95_lb":
                ratio_lb,
            "ratio_95_ub":
                ratio_ub,
            "mean_robust_scale":
                float(
                    grp[
                        "robust_scale"
                    ].mean()
                ),
        }
    )


h28_primary_tests = pd.DataFrame(
    h28_primary_rows
)


print(
    "\n========================================"
)
print(
    "PRIMARY TEST — CURRENT WEIGHTED vs WEIGHT-NEUTRAL Q1"
)
print(
    "========================================"
)

display(
    h28_primary_tests
)


# ==========================================================
# 9. ROW-LEVEL QUARTILE SUMMARY
# ==========================================================

def h28_row_quartile_summary(
    frame,
    *,
    abs_col,
    label,
):
    out = (
        frame.groupby(
            "car_strength_quartile",
            observed=False,
            as_index=False,
        )
        .agg(
            heldout_rows=(
                abs_col,
                "size",
            ),
            constructor_seasons=(
                "csy",
                "nunique",
            ),
            median_beta=(
                "beta",
                "median",
            ),
            mean_abs_z=(
                abs_col,
                "mean",
            ),
            median_abs_z=(
                abs_col,
                "median",
            ),
        )
    )

    out[
        "diagnostic"
    ] = label

    return out


h28_row_summary = pd.concat(
    [
        h28_row_quartile_summary(
            h28_q1,
            abs_col=
                "abs_z_current",
            label=
                "Q1 contrast — current weighted z",
        ),
        h28_row_quartile_summary(
            h28_q1,
            abs_col=
                "abs_z_raw",
            label=
                "Q1 contrast — weight-neutral z",
        ),
        h28_row_quartile_summary(
            h28_q1_level,
            abs_col=
                "abs_z_level",
            label=
                "Q1 team level — benchmark",
        ),
    ],
    ignore_index=True,
)


h28_row_summary[
    "car_strength_quartile"
] = pd.Categorical(
    h28_row_summary[
        "car_strength_quartile"
    ],
    categories=
        quartile_order,
    ordered=True,
)


h28_row_summary = (
    h28_row_summary
    .sort_values(
        [
            "diagnostic",
            "car_strength_quartile",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n========================================"
)
print(
    "ROW-LEVEL Q1 DISPERSION BY CAR-STRENGTH QUARTILE"
)
print(
    "========================================"
)

display(
    h28_row_summary
)


# ==========================================================
# 10. FAST ROW-LEVEL YEAR-FE REGRESSION
# ==========================================================
#
# Dispersion proxy:
#
#       log(1 + |z_raw|)
#
# This is finite at zero and less tail-sensitive than |z| itself.
#
# Models:
#
#   A. year FE only
#   B. year FE + n_common_sessions categorical controls
#
# If beta remains negative in B, the weak-car Q1 variance relationship
# survives explicit control for the qualifying-weight/progression class.
# ==========================================================

h28_q1[
    "log1p_abs_z_raw"
] = np.log1p(
    h28_q1[
        "abs_z_raw"
    ].astype(float)
)


def h28_prepare_within_year_crossproducts(
    frame,
    *,
    y_col,
    include_ncommon_controls,
):
    """
    Build per-year X'X and X'y after demeaning within year.

    Regressors:
        beta
        optionally I(n_common_sessions=2), I(n_common_sessions=3)

    Demeaning absorbs year fixed effects.
    """

    work = frame[
        [
            "year",
            "beta",
            "n_common_sessions",
            y_col,
        ]
    ].dropna().copy()

    unique_n = sorted(
        work[
            "n_common_sessions"
        ].astype(int).unique()
    )

    control_levels = [
        n
        for n in unique_n
        if (
            include_ncommon_controls
            and
            n
            !=
            unique_n[0]
        )
    ]

    parameter_names = [
        "beta"
    ] + [
        f"I(n_common_sessions={n})"
        for n
        in control_levels
    ]

    per_year = []

    for year, grp in work.groupby(
        "year"
    ):
        columns = [
            grp[
                "beta"
            ].to_numpy(
                dtype=float
            )
        ]

        for n in control_levels:
            columns.append(
                (
                    grp[
                        "n_common_sessions"
                    ].to_numpy(
                        dtype=int
                    )
                    ==
                    n
                ).astype(float)
            )

        X = np.column_stack(
            columns
        )

        y = grp[
            y_col
        ].to_numpy(
            dtype=float
        )

        # Remove year fixed effect.
        X = (
            X
            -
            X.mean(
                axis=0,
                keepdims=True,
            )
        )

        y = (
            y
            -
            y.mean()
        )

        xtx = (
            X.T
            @
            X
        )

        xty = (
            X.T
            @
            y
        )

        per_year.append(
            {
                "year":
                    int(year),
                "xtx":
                    xtx,
                "xty":
                    xty,
            }
        )

    return (
        per_year,
        parameter_names,
    )


def h28_fit_from_crossproducts(
    xtx,
    xty,
):
    return (
        np.linalg.pinv(
            xtx
        )
        @
        xty
    )


def h28_row_fe_bootstrap(
    frame,
    *,
    y_col,
    include_ncommon_controls,
):
    (
        per_year,
        parameter_names,
    ) = h28_prepare_within_year_crossproducts(
        frame,
        y_col=
            y_col,
        include_ncommon_controls=
            include_ncommon_controls,
    )

    if len(per_year) < 3:
        return (
            parameter_names,
            np.full(
                len(
                    parameter_names
                ),
                np.nan,
            ),
            np.full(
                len(
                    parameter_names
                ),
                np.nan,
            ),
            np.full(
                len(
                    parameter_names
                ),
                np.nan,
            ),
        )

    xtx_stack = np.stack(
        [
            item[
                "xtx"
            ]
            for item
            in per_year
        ]
    )

    xty_stack = np.stack(
        [
            item[
                "xty"
            ]
            for item
            in per_year
        ]
    )

    observed = (
        h28_fit_from_crossproducts(
            xtx_stack.sum(
                axis=0
            ),
            xty_stack.sum(
                axis=0
            ),
        )
    )

    n_years = len(
        per_year
    )

    sampled = h28_rng.integers(
        0,
        n_years,
        size=(
            H28_BOOTSTRAPS,
            n_years,
        ),
    )

    boot = np.full(
        (
            H28_BOOTSTRAPS,
            len(
                parameter_names
            ),
        ),
        np.nan,
        dtype=float,
    )

    for b in range(
        H28_BOOTSTRAPS
    ):
        ids = sampled[
            b
        ]

        xtx = xtx_stack[
            ids
        ].sum(
            axis=0
        )

        xty = xty_stack[
            ids
        ].sum(
            axis=0
        )

        boot[
            b
        ] = (
            h28_fit_from_crossproducts(
                xtx,
                xty,
            )
        )

    lb = np.nanquantile(
        boot,
        0.025,
        axis=0,
    )

    ub = np.nanquantile(
        boot,
        0.975,
        axis=0,
    )

    return (
        parameter_names,
        observed,
        lb,
        ub,
    )


row_regression_rows = []


for model_name, controls in [
    (
        "Raw Q1 dispersion ~ beta + year FE",
        False,
    ),
    (
        "Raw Q1 dispersion ~ beta + year FE + progression controls",
        True,
    ),
]:
    (
        names,
        estimates,
        lbs,
        ubs,
    ) = h28_row_fe_bootstrap(
        h28_q1,
        y_col=
            "log1p_abs_z_raw",
        include_ncommon_controls=
            controls,
    )

    for (
        parameter,
        estimate,
        lb,
        ub,
    ) in zip(
        names,
        estimates,
        lbs,
        ubs,
    ):
        row_regression_rows.append(
            {
                "model":
                    model_name,
                "parameter":
                    parameter,
                "estimate":
                    float(
                        estimate
                    ),
                "bootstrap_95_lb":
                    float(
                        lb
                    ),
                "bootstrap_95_ub":
                    float(
                        ub
                    ),
            }
        )


h28_controlled_regression = pd.DataFrame(
    row_regression_rows
)


print(
    "\n========================================"
)
print(
    "ROW-LEVEL WEIGHT-CONTROLLED Q1 DISPERSION REGRESSION"
)
print(
    "========================================"
)

display(
    h28_controlled_regression
)


# ==========================================================
# 11. EXACT n_common_sessions STRATA
# ==========================================================
#
# Within each exact stratum the likelihood weight is constant, so it
# cannot mechanically create a beta/dispersion association.
#
# We use:
#
#       log(1 + |z_raw|) ~ beta + year FE
#
# separately for n_common_sessions = 1, 2, 3.
# ==========================================================

def h28_simple_row_fe_bootstrap(
    frame,
    *,
    y_col,
):
    per_year = []

    for year, grp in frame.groupby(
        "year"
    ):
        if len(grp) < 2:
            continue

        x = grp[
            "beta"
        ].to_numpy(
            dtype=float
        )

        y = grp[
            y_col
        ].to_numpy(
            dtype=float
        )

        x_dm = (
            x
            -
            x.mean()
        )

        y_dm = (
            y
            -
            y.mean()
        )

        sxx = float(
            np.sum(
                x_dm
                ** 2
            )
        )

        sxy = float(
            np.sum(
                x_dm
                *
                y_dm
            )
        )

        if sxx > 0:
            per_year.append(
                (
                    int(year),
                    sxx,
                    sxy,
                )
            )

    if len(per_year) < 3:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    sxx = np.asarray(
        [
            row[1]
            for row
            in per_year
        ],
        dtype=float,
    )

    sxy = np.asarray(
        [
            row[2]
            for row
            in per_year
        ],
        dtype=float,
    )

    observed = float(
        sxy.sum()
        /
        sxx.sum()
    )

    n_years = len(
        per_year
    )

    sampled = h28_rng.integers(
        0,
        n_years,
        size=(
            H28_BOOTSTRAPS,
            n_years,
        ),
    )

    boot_sxx = sxx[
        sampled
    ].sum(
        axis=1
    )

    boot_sxy = sxy[
        sampled
    ].sum(
        axis=1
    )

    valid = (
        boot_sxx
        >
        0
    )

    boot = (
        boot_sxy[
            valid
        ]
        /
        boot_sxx[
            valid
        ]
    )

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


stratum_rows = []


for n_common in sorted(
    h28_q1[
        "n_common_sessions"
    ].unique()
):
    stratum = h28_q1[
        h28_q1[
            "n_common_sessions"
        ]
        ==
        n_common
    ].copy()

    (
        slope,
        lb,
        ub,
    ) = h28_simple_row_fe_bootstrap(
        stratum,
        y_col=
            "log1p_abs_z_raw",
    )

    quartile_counts = (
        stratum[
            "car_strength_quartile"
        ]
        .value_counts()
    )

    stratum_rows.append(
        {
            "n_common_sessions":
                int(
                    n_common
                ),
            "weight":
                float(
                    1.0
                    /
                    n_common
                ),
            "heldout_rows":
                int(
                    len(stratum)
                ),
            "years":
                int(
                    stratum[
                        "year"
                    ].nunique()
                ),
            "constructor_seasons":
                int(
                    stratum[
                        "csy"
                    ].nunique()
                ),
            "beta_min":
                float(
                    stratum[
                        "beta"
                    ].min()
                ),
            "beta_median":
                float(
                    stratum[
                        "beta"
                    ].median()
                ),
            "beta_max":
                float(
                    stratum[
                        "beta"
                    ].max()
                ),
            "q1_weakest_rows":
                int(
                    quartile_counts.get(
                        "Q1 weakest",
                        0,
                    )
                ),
            "q4_strongest_rows":
                int(
                    quartile_counts.get(
                        "Q4 strongest",
                        0,
                    )
                ),
            "year_FE_beta_slope":
                slope,
            "slope_95_lb":
                lb,
            "slope_95_ub":
                ub,
        }
    )


h28_exact_strata = pd.DataFrame(
    stratum_rows
)


print(
    "\n========================================"
)
print(
    "EXACT QUALIFYING-WEIGHT STRATA"
)
print(
    "========================================"
)

display(
    h28_exact_strata
)


# ==========================================================
# 12. Q1 TEAM-LEVEL BENCHMARK WEIGHT CHECK
# ==========================================================

print(
    "\n========================================"
)
print(
    "Q1 TEAM-LEVEL BENCHMARK WEIGHTS"
)
print(
    "========================================"
)

display(
    h28_level_weight_summary.to_frame(
        name=
            "q1_level_weight"
    )
)


print(
    "Max |q1_level weight - 1|:",
    float(
        np.max(
            np.abs(
                h28_q1_level[
                    "weight"
                ].astype(float)
                -
                1.0
            )
        )
    ),
)


# ==========================================================
# 13. PLOTS
# ==========================================================

# ----------------------------------------------------------
# A. Progression mix by car-strength quartile
# ----------------------------------------------------------

plt.figure(
    figsize=(
        9,
        6,
    )
)

bottom = np.zeros(
    len(
        weight_pivot.index
    ),
    dtype=float,
)

for n_common in sorted(
    weight_pivot.columns
):
    values = weight_pivot[
        n_common
    ].to_numpy(
        dtype=float
    )

    plt.bar(
        weight_pivot.index.astype(str),
        values,
        bottom=
            bottom,
        label=
            f"{int(n_common)} common session"
            +
            (
                ""
                if int(n_common) == 1
                else "s"
            ),
    )

    bottom = (
        bottom
        +
        values
    )


plt.ylabel(
    "Share of held-out Q1 teammate contrasts"
)

plt.xlabel(
    "Constructor strength quartile within season"
)

plt.title(
    "Q1 qualifying progression mix by constructor strength"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# B. Current weighted vs weight-neutral Q1 residual dispersion
# ----------------------------------------------------------

plt.figure(
    figsize=(
        9,
        6,
    )
)

for diagnostic in [
    "Q1 contrast — current weighted z",
    "Q1 contrast — weight-neutral z",
    "Q1 team level — benchmark",
]:
    x = h28_row_summary[
        h28_row_summary[
            "diagnostic"
        ]
        ==
        diagnostic
    ].copy()

    if x.empty:
        continue

    x = x.sort_values(
        "car_strength_quartile"
    )

    plt.plot(
        x[
            "car_strength_quartile"
        ].astype(str),
        x[
            "mean_abs_z"
        ],
        marker="o",
        label=
            diagnostic,
    )


plt.xlabel(
    "Constructor strength quartile within season"
)

plt.ylabel(
    "Mean absolute held-out residual"
)

plt.title(
    "Q1 residual dispersion before and after removing qualifying weight"
)

plt.legend()

plt.grid(
    alpha=0.25,
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# C. Constructor-season robust scale vs beta:
#    current weighted and weight-neutral
# ----------------------------------------------------------

for diagnostic in [
    "Q1 contrast — current weighted z",
    "Q1 contrast — weight-neutral z",
]:
    plot_df = h28_csy_scales[
        h28_csy_scales[
            "diagnostic"
        ]
        ==
        diagnostic
    ].dropna(
        subset=[
            "beta",
            "robust_scale",
        ]
    )

    if len(plot_df) < 5:
        continue

    plt.figure(
        figsize=(
            8,
            6,
        )
    )

    plt.scatter(
        plot_df[
            "beta"
        ],
        plot_df[
            "robust_scale"
        ],
        alpha=0.55,
    )

    plt.axvline(
        0.0,
        linestyle=":",
        linewidth=1.0,
    )

    plt.xlabel(
        "Training-fit constructor-season beta"
    )

    plt.ylabel(
        "Held-out constructor-season robust residual scale"
    )

    plt.title(
        diagnostic
    )

    plt.grid(
        alpha=0.25,
    )

    plt.tight_layout()
    plt.show()


# ==========================================================
# 14. SAVE OUTPUTS
# ==========================================================

h28_q1.to_csv(
    H28_OUTPUT_DIR
    /
    "heldout_q1_contrast_weight_audit_rows.csv",
    index=False,
)


weight_mix.to_csv(
    H28_OUTPUT_DIR
    /
    "q1_progression_mix_by_car_strength.csv",
    index=False,
)


h28_csy_scales.to_csv(
    H28_OUTPUT_DIR
    /
    "q1_constructor_season_scale_comparison.csv",
    index=False,
)


h28_primary_tests.to_csv(
    H28_OUTPUT_DIR
    /
    "q1_weight_audit_primary_tests.csv",
    index=False,
)


h28_row_summary.to_csv(
    H28_OUTPUT_DIR
    /
    "q1_row_dispersion_by_car_strength.csv",
    index=False,
)


h28_controlled_regression.to_csv(
    H28_OUTPUT_DIR
    /
    "q1_weight_controlled_regression.csv",
    index=False,
)


h28_exact_strata.to_csv(
    H28_OUTPUT_DIR
    /
    "q1_exact_weight_strata.csv",
    index=False,
)


# ==========================================================
# 15. FINAL TRIAGE SUMMARY
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 28 TRIAGE SUMMARY"
)
print(
    "========================================"
)


def h28_print_primary(
    diagnostic,
):
    x = h28_primary_tests[
        h28_primary_tests[
            "diagnostic"
        ]
        ==
        diagnostic
    ]

    if x.empty:
        print(
            diagnostic,
            ": insufficient data",
        )
        return

    row = x.iloc[0]

    print(
        diagnostic
    )

    print(
        f"  FE log-scale slope vs beta = "
        f"{row['year_FE_logscale_slope']:.4f} "
        f"[95% "
        f"{row['slope_95_lb']:.4f}, "
        f"{row['slope_95_ub']:.4f}]"
    )

    print(
        f"  Weakest/strongest robust-scale ratio = "
        f"{row['weakest_vs_strongest_scale_ratio']:.3f} "
        f"[95% "
        f"{row['ratio_95_lb']:.3f}, "
        f"{row['ratio_95_ub']:.3f}]"
    )

    print(
        f"  Spearman(beta, scale) = "
        f"{row['spearman_beta_vs_scale']:.3f}"
    )


h28_print_primary(
    "Q1 contrast — current weighted z"
)

h28_print_primary(
    "Q1 contrast — weight-neutral z"
)

h28_print_primary(
    "Q1 team level — benchmark"
)


controlled_beta = (
    h28_controlled_regression[
        (
            h28_controlled_regression[
                "model"
            ]
            ==
            "Raw Q1 dispersion ~ beta + year FE + progression controls"
        )
        &
        (
            h28_controlled_regression[
                "parameter"
            ]
            ==
            "beta"
        )
    ]
)


print(
    "\nWeight / progression association:"
)

print(
    f"  Spearman(beta, n_common_sessions) = "
    f"{h28_weight_assoc_spearman:.3f}"
)


if len(
    controlled_beta
):
    row = controlled_beta.iloc[0]

    print(
        "\nWeight-controlled row-level beta effect:"
    )

    print(
        f"  beta coefficient = "
        f"{row['estimate']:.4f} "
        f"[95% "
        f"{row['bootstrap_95_lb']:.4f}, "
        f"{row['bootstrap_95_ub']:.4f}]"
    )


print(
    "\nDecision guide:"
)

print(
    "  A. If CURRENT weighted Q1 shows weak-car excess variance but"
)

print(
    "     WEIGHT-NEUTRAL Q1 moves close to ratio 1 / slope 0, then"
)

print(
    "     much of Stage 27's Q1 effect came from the information-weight"
)

print(
    "     / progression mixture."
)

print(
    "  B. If WEIGHT-NEUTRAL Q1 still has ratio > 1 and negative slope,"
)

print(
    "     and the progression-controlled beta coefficient is also"
)

print(
    "     negative, genuine weak-car Q1 heteroskedasticity remains."
)

print(
    "  C. Exact n_common_sessions strata are the strongest mechanical"
)

print(
    "     check because weight is constant inside each stratum."
)

print(
    "  D. Q1 TEAM LEVEL is an independent benchmark: its evidence cannot"
)

print(
    "     be explained by the teammate contrast 1/n_common_sessions"
)

print(
    "     weighting scheme."
)

print(
    "\nDo not modify the production likelihood until this diagnostic is"
)

print(
    "interpreted. If genuine Q1 heteroskedasticity survives, the next"
)

print(
    "step is a minimal heteroskedastic qualifying extension evaluated"
)

print(
    "against the frozen model on held-out predictive performance."
)

print(
    "\nSaved outputs ->"
)

print(
    H28_OUTPUT_DIR.resolve()
)

print(
    "\nStage 28 complete."
)


In [ ]:
# ==========================================================
# STAGE 29 — DRIVER / CAR ALLOCATION + DOMINANT-TEAM LIFT AUDIT
# ==========================================================
#
# MOTIVATION
# ----------
# Pérez's latent alpha rises markedly after joining Red Bull even though
# the direct teammate evidence says Verstappen beat him very heavily.
#
# This audit tests the specific concern:
#
#     Is the model good at estimating teammate DIFFERENCES, but less
#     well identified in deciding how much of a team's common level
#     belongs to the drivers versus the constructor?
#
# This is deliberately an audit of the accepted production posterior.
# NO MCMC / REFITTING is performed here.
#
# It does five things:
#
#   A. Reconstruct Pérez's season-by-season decomposition:
#        alpha, car beta, teammate alpha/delta, observed pace/quali
#        contrasts, observed/predicted team-level evidence.
#
#   B. Quantify the posterior allocation ridge for every constructor-
#      season:
#
#        common driver level  <---->  constructor beta
#
#      If those two quantities are strongly negatively correlated while
#      their SUM is much more stable, the posterior knows the total team
#      performance much better than the driver/car split.
#
#   C. Show the local likelihood invariance explicitly:
#
#        alpha_A -> alpha_A - delta
#        alpha_B -> alpha_B - delta
#        beta    -> beta    + delta
#
#      leaves teammate contrasts AND team-level predictions unchanged
#      for that constructor-season. The temporal priors, global centring
#      and constructor prior are therefore what locate the pair
#      vertically.
#
#   D. Compare several dominant-team cases:
#        Pérez / Red Bull
#        Bottas / Mercedes
#        Barrichello / Ferrari
#        Irvine / Ferrari
#        Webber / Red Bull
#
#   E. Test globally whether year-to-year alpha changes track changes in
#      constructor beta, separately for team stayers and team switchers.
#
# IMPORTANT INTERPRETATION
# ------------------------
# A strong negative posterior car-vs-common-driver correlation is not
# automatically a "bug": it is the expected signature of an allocation
# ridge. The important questions are:
#
#   * Is the ridge large enough to drive counter-intuitive career shapes?
#   * Do drivers systematically jump when their constructor beta jumps?
#   * Is total team signal much better identified than the split?
#
# If YES, the next decisive test is a targeted refit that removes or
# perturbs the team-level anchor for selected dominant constructor-
# seasons and checks how alpha moves.
# ==========================================================

from pathlib import Path
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

S29_TARGET_DRIVER_QUERY = "perez"

S29_PEREZ_START_YEAR = 2011
S29_PEREZ_END_YEAR = 2024

S29_BOOTSTRAPS = 3000
S29_BOOTSTRAP_SEED = 29042

S29_OUTPUT_DIR = (
    Path(CFG.output_dir)
    / "validation_driver_car_allocation"
)

S29_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==========================================================
# 1. REQUIRED OBJECTS
# ==========================================================

required_objects = [
    "CFG",
    "idx_all",
    "final_trace_2025",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "actual_starts",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the final production/model/scoring stages first. Missing: "
        + ", ".join(missing)
    )


# ==========================================================
# 2. POSTERIOR EXTRACTION
# ==========================================================

if (
    "posterior_final"
    in globals()
):
    s29_post = posterior_final
else:
    s29_post = (
        final_trace_2025[
            "posterior"
        ]
        .to_dataset()
    )


def s29_stack_state(
    var_name,
):
    arr = np.asarray(
        s29_post[
            var_name
        ].values,
        dtype=float,
    )

    return arr.reshape(
        -1,
        arr.shape[-1],
    )


def s29_stack_scalar(
    var_name,
):
    return np.asarray(
        s29_post[
            var_name
        ].values,
        dtype=float,
    ).reshape(-1)


s29_alpha = s29_stack_state(
    "alpha_dy"
)

s29_car = s29_stack_state(
    "car_csy"
)

s29_gamma = s29_stack_scalar(
    "gamma_tenure"
)


n_draws = s29_alpha.shape[0]

if (
    s29_car.shape[0]
    !=
    n_draws
    or
    s29_gamma.shape[0]
    !=
    n_draws
):
    raise RuntimeError(
        "Posterior draw counts do not match."
    )


s29_alpha_mean = s29_alpha.mean(
    axis=0
)

s29_car_mean = s29_car.mean(
    axis=0
)

s29_gamma_mean = float(
    s29_gamma.mean()
)


print(
    "========================================"
)
print(
    "STAGE 29 — DRIVER / CAR ALLOCATION AUDIT"
)
print(
    "========================================"
)

print(
    "Posterior draws:",
    n_draws,
)

print(
    "Driver-season states:",
    s29_alpha.shape[1],
)

print(
    "Constructor-season states:",
    s29_car.shape[1],
)

print(
    "Posterior mean tenure coefficient:",
    s29_gamma_mean,
)


# ==========================================================
# 3. NAME HELPERS
# ==========================================================

def s29_ascii(
    text,
):
    return (
        unicodedata.normalize(
            "NFKD",
            str(text),
        )
        .encode(
            "ascii",
            "ignore",
        )
        .decode(
            "ascii"
        )
        .lower()
    )


all_model_drivers = sorted(
    set(
        driver
        for driver, _
        in idx_all.active_driver_years
    )
)


def s29_resolve_driver(
    query,
):
    q = s29_ascii(
        query
    )

    exact = [
        driver
        for driver
        in all_model_drivers
        if s29_ascii(
            driver
        )
        ==
        q
    ]

    if len(exact) == 1:
        return exact[0]

    partial = [
        driver
        for driver
        in all_model_drivers
        if q
        in s29_ascii(
            driver
        )
    ]

    if len(partial) == 1:
        return partial[0]

    if not partial:
        raise KeyError(
            f"No modelled driver matches {query!r}"
        )

    raise KeyError(
        f"Ambiguous driver query {query!r}: {partial}"
    )


# Constructor names.
constructor_name_map = {}

if (
    "raw"
    in globals()
    and hasattr(
        raw,
        "constructors",
    )
):
    cdf = raw.constructors.copy()

    if "constructorId" in cdf.columns:

        name_col = None

        for candidate in [
            "name",
            "constructorRef",
        ]:
            if candidate in cdf.columns:
                name_col = candidate
                break

        if name_col is not None:
            constructor_name_map = dict(
                zip(
                    cdf[
                        "constructorId"
                    ].astype(int),
                    cdf[
                        name_col
                    ].astype(str),
                )
            )


def s29_constructor_name(
    constructor_id,
):
    return constructor_name_map.get(
        int(
            constructor_id
        ),
        str(
            int(
                constructor_id
            )
        ),
    )


# ==========================================================
# 4. CONSTRUCTOR-SEASON POSTERIOR MEAN RANKS
# ==========================================================

s29_csy_meta = pd.DataFrame(
    [
        {
            "csy":
                int(i),
            "constructorId":
                int(
                    constructor_id
                ),
            "year":
                int(year),
            "constructor":
                s29_constructor_name(
                    constructor_id
                ),
            "beta_mean":
                float(
                    s29_car_mean[
                        i
                    ]
                ),
        }
        for i, (
            constructor_id,
            year,
        ) in enumerate(
            idx_all.active_constructor_years
        )
    ]
)


s29_csy_meta[
    "beta_rank_mean_based"
] = (
    s29_csy_meta.groupby(
        "year"
    )[
        "beta_mean"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)


s29_csy_meta[
    "field_size"
] = (
    s29_csy_meta.groupby(
        "year"
    )[
        "csy"
    ]
    .transform(
        "size"
    )
)


s29_csy_lookup = (
    s29_csy_meta
    .set_index(
        [
            "constructorId",
            "year",
        ]
    )
    .to_dict(
        "index"
    )
)


# ==========================================================
# 5. ACTUAL-START TEAM / DRIVER TABLE
# ==========================================================

s29_starts = (
    actual_starts[
        actual_starts[
            "year"
        ].astype(int)
        >=
        int(
            CFG.modern_floor_year
        )
    ]
    .copy()
)


s29_team_driver_starts = (
    s29_starts.groupby(
        [
            "year",
            "constructorId",
            "driver",
        ],
        as_index=False,
    )
    .agg(
        starts=(
            "raceId",
            "nunique",
        )
    )
)


# ==========================================================
# 6. POSTERIOR ALLOCATION RIDGE BY CONSTRUCTOR-SEASON
# ==========================================================
#
# common_driver_draws = race-start-weighted mean alpha of the drivers
# who actually started for that constructor-season.
#
# total_common_draws = common_driver_draws + car_draws
#
# We deliberately leave tenure out of this ridge quantity because the
# question is specifically the alpha-vs-beta allocation. Tenure can be
# added to the observation prediction later and is not constructor-
# season specific latent performance.
# ==========================================================

allocation_rows = []
allocation_draw_cache = {}


for (
    year,
    constructor_id,
), grp in s29_team_driver_starts.groupby(
    [
        "year",
        "constructorId",
    ]
):

    year = int(
        year
    )

    constructor_id = int(
        constructor_id
    )

    csy_key = (
        constructor_id,
        year,
    )

    if csy_key not in idx_all.csy_map:
        continue

    driver_ids = []
    driver_weights = []
    driver_names = []

    for row in grp.itertuples(
        index=False
    ):

        dy_key = (
            row.driver,
            year,
        )

        if dy_key not in idx_all.dy_map:
            continue

        driver_ids.append(
            idx_all.dy_map[
                dy_key
            ]
        )

        driver_weights.append(
            float(
                row.starts
            )
        )

        driver_names.append(
            row.driver
        )

    if not driver_ids:
        continue

    driver_ids = np.asarray(
        driver_ids,
        dtype=int,
    )

    driver_weights = np.asarray(
        driver_weights,
        dtype=float,
    )

    driver_weights = (
        driver_weights
        /
        driver_weights.sum()
    )

    common_driver_draws = np.sum(
        s29_alpha[
            :,
            driver_ids,
        ]
        *
        driver_weights[
            None,
            :
        ],
        axis=1,
    )

    csy = idx_all.csy_map[
        csy_key
    ]

    car_draws = s29_car[
        :,
        csy,
    ]

    total_draws = (
        common_driver_draws
        +
        car_draws
    )

    if (
        np.std(
            common_driver_draws
        )
        >
        0
        and
        np.std(
            car_draws
        )
        >
        0
    ):
        corr = float(
            np.corrcoef(
                common_driver_draws,
                car_draws,
            )[
                0,
                1,
            ]
        )
    else:
        corr = np.nan

    sd_driver = float(
        np.std(
            common_driver_draws,
            ddof=1,
        )
    )

    sd_car = float(
        np.std(
            car_draws,
            ddof=1,
        )
    )

    sd_total = float(
        np.std(
            total_draws,
            ddof=1,
        )
    )

    independent_sd = float(
        np.sqrt(
            sd_driver ** 2
            +
            sd_car ** 2
        )
    )

    cancellation_ratio = (
        independent_sd
        /
        sd_total
        if sd_total
        >
        0
        else np.nan
    )

    cmeta = s29_csy_lookup.get(
        csy_key,
        {},
    )

    allocation_rows.append(
        {
            "year":
                year,
            "constructorId":
                constructor_id,
            "constructor":
                s29_constructor_name(
                    constructor_id
                ),
            "drivers":
                " / ".join(
                    driver_names
                ),
            "n_drivers":
                int(
                    len(
                        driver_ids
                    )
                ),
            "team_actual_starts":
                int(
                    grp[
                        "starts"
                    ].sum()
                ),
            "car_beta_mean":
                float(
                    car_draws.mean()
                ),
            "car_rank":
                cmeta.get(
                    "beta_rank_mean_based",
                    np.nan,
                ),
            "common_driver_mean":
                float(
                    common_driver_draws.mean()
                ),
            "total_common_mean":
                float(
                    total_draws.mean()
                ),
            "corr_car_vs_common_driver":
                corr,
            "sd_common_driver":
                sd_driver,
            "sd_car":
                sd_car,
            "sd_driver_plus_car":
                sd_total,
            "allocation_cancellation_ratio":
                cancellation_ratio,
        }
    )

    allocation_draw_cache[
        csy_key
    ] = {
        "common_driver":
            common_driver_draws,
        "car":
            car_draws,
        "total":
            total_draws,
        "driver_ids":
            driver_ids,
        "driver_names":
            driver_names,
        "weights":
            driver_weights,
    }


s29_allocation = pd.DataFrame(
    allocation_rows
)


print(
    "\n========================================"
)
print(
    "WHOLE-MODEL DRIVER/CAR ALLOCATION RIDGE"
)
print(
    "========================================"
)

print(
    "Constructor-seasons audited:",
    len(
        s29_allocation
    ),
)

print(
    "Median corr(car, common driver level):",
    float(
        s29_allocation[
            "corr_car_vs_common_driver"
        ].median()
    ),
)

print(
    "10th percentile corr:",
    float(
        s29_allocation[
            "corr_car_vs_common_driver"
        ].quantile(
            0.10
        )
    ),
)

print(
    "Share with corr < -0.5:",
    float(
        np.mean(
            s29_allocation[
                "corr_car_vs_common_driver"
            ]
            <
            -0.5
        )
    ),
)

print(
    "Median allocation cancellation ratio:",
    float(
        s29_allocation[
            "allocation_cancellation_ratio"
        ].median()
    ),
)


display(
    s29_allocation.sort_values(
        "corr_car_vs_common_driver"
    ).head(
        20
    )
)


# ==========================================================
# 7. EVIDENCE HELPERS
# ==========================================================

def s29_oriented_contrast_summary(
    table,
    driver,
    teammate,
    year,
    constructor_id,
):
    sub = table[
        (
            table[
                "year"
            ].astype(int)
            ==
            int(year)
        )
        &
        (
            table[
                "constructorId"
            ].astype(int)
            ==
            int(
                constructor_id
            )
        )
        &
        (
            (
                (
                    table[
                        "driver_a"
                    ]
                    ==
                    driver
                )
                &
                (
                    table[
                        "driver_b"
                    ]
                    ==
                    teammate
                )
            )
            |
            (
                (
                    table[
                        "driver_a"
                    ]
                    ==
                    teammate
                )
                &
                (
                    table[
                        "driver_b"
                    ]
                    ==
                    driver
                )
            )
        )
    ].copy()

    if sub.empty:
        return {
            "mean":
                np.nan,
            "n_rows":
                0,
            "n_weekends":
                0,
        }

    oriented = np.where(
        sub[
            "driver_a"
        ].to_numpy()
        ==
        driver,
        sub[
            "y"
        ].to_numpy(
            dtype=float
        ),
        -sub[
            "y"
        ].to_numpy(
            dtype=float
        ),
    )

    weights = sub[
        "weight"
    ].to_numpy(
        dtype=float
    )

    return {
        "mean":
            float(
                np.average(
                    oriented,
                    weights=
                        weights,
                )
            ),
        "n_rows":
            int(
                len(
                    sub
                )
            ),
        "n_weekends":
            int(
                sub[
                    "raceId"
                ].nunique()
            ),
    }


def s29_team_level_summary(
    table,
    *,
    year,
    constructor_id,
    channel,
):
    sub = table[
        (
            table[
                "year"
            ].astype(int)
            ==
            int(year)
        )
        &
        (
            table[
                "constructorId"
            ].astype(int)
            ==
            int(
                constructor_id
            )
        )
    ].copy()

    if sub.empty:
        return {
            "observed":
                np.nan,
            "predicted":
                np.nan,
            "residual":
                np.nan,
            "n_rows":
                0,
            "n_weekends":
                0,
        }

    weights = sub[
        "weight"
    ].to_numpy(
        dtype=float
    )

    observed = sub[
        "y"
    ].to_numpy(
        dtype=float
    )

    dy_a = sub[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = sub[
        "dy_b"
    ].to_numpy(
        dtype=int
    )

    csy = sub[
        "csy"
    ].to_numpy(
        dtype=int
    )

    tenure_mean = sub[
        "tenure_mean"
    ].to_numpy(
        dtype=float
    )

    predicted = (
        0.5
        *
        (
            s29_alpha_mean[
                dy_a
            ]
            +
            s29_alpha_mean[
                dy_b
            ]
        )
        +
        s29_car_mean[
            csy
        ]
        +
        s29_gamma_mean
        *
        tenure_mean
    )

    return {
        "observed":
            float(
                np.average(
                    observed,
                    weights=
                        weights,
                )
            ),
        "predicted":
            float(
                np.average(
                    predicted,
                    weights=
                        weights,
                )
            ),
        "residual":
            float(
                np.average(
                    observed
                    -
                    predicted,
                    weights=
                        weights,
                )
            ),
        "n_rows":
            int(
                len(
                    sub
                )
            ),
        "n_weekends":
            int(
                sub[
                    "raceId"
                ].nunique()
            ),
        "channel":
            channel,
    }


# ==========================================================
# 8. PRIMARY CONSTRUCTOR + MAIN TEAMMATE FOR EACH DRIVER-YEAR
# ==========================================================

primary_team_rows = []


for (
    year,
    driver,
), grp in s29_team_driver_starts.groupby(
    [
        "year",
        "driver",
    ]
):
    grp = grp.sort_values(
        [
            "starts",
            "constructorId",
        ],
        ascending=[
            False,
            True,
        ],
    )

    top = grp.iloc[0]

    primary_team_rows.append(
        {
            "driver":
                driver,
            "year":
                int(year),
            "constructorId":
                int(
                    top[
                        "constructorId"
                    ]
                ),
            "constructor":
                s29_constructor_name(
                    top[
                        "constructorId"
                    ]
                ),
            "starts":
                int(
                    top[
                        "starts"
                    ]
                ),
        }
    )


s29_primary_team = pd.DataFrame(
    primary_team_rows
)


def s29_main_teammate(
    driver,
    year,
    constructor_id,
):
    driver_races = set(
        s29_starts.loc[
            (
                s29_starts[
                    "driver"
                ]
                ==
                driver
            )
            &
            (
                s29_starts[
                    "year"
                ].astype(int)
                ==
                int(year)
            )
            &
            (
                s29_starts[
                    "constructorId"
                ].astype(int)
                ==
                int(
                    constructor_id
                )
            ),
            "raceId",
        ].astype(int)
    )

    if not driver_races:
        return (
            None,
            0,
        )

    others = s29_starts[
        (
            s29_starts[
                "year"
            ].astype(int)
            ==
            int(year)
        )
        &
        (
            s29_starts[
                "constructorId"
            ].astype(int)
            ==
            int(
                constructor_id
            )
        )
        &
        (
            s29_starts[
                "driver"
            ]
            !=
            driver
        )
        &
        (
            s29_starts[
                "raceId"
            ].astype(int)
            .isin(
                driver_races
            )
        )
    ].copy()

    if others.empty:
        return (
            None,
            0,
        )

    counts = (
        others.groupby(
            "driver"
        )[
            "raceId"
        ]
        .nunique()
        .sort_values(
            ascending=False
        )
    )

    return (
        counts.index[0],
        int(
            counts.iloc[0]
        ),
    )


# ==========================================================
# 9. BUILD DRIVER-SEASON DECOMPOSITION TABLE
# ==========================================================

driver_season_rows = []


for row in s29_primary_team.itertuples(
    index=False
):
    driver = row.driver
    year = int(
        row.year
    )

    dy_key = (
        driver,
        year,
    )

    csy_key = (
        int(
            row.constructorId
        ),
        year,
    )

    if (
        dy_key
        not in idx_all.dy_map
        or
        csy_key
        not in idx_all.csy_map
    ):
        continue

    dy = idx_all.dy_map[
        dy_key
    ]

    csy = idx_all.csy_map[
        csy_key
    ]

    alpha_draws = s29_alpha[
        :,
        dy,
    ]

    car_draws = s29_car[
        :,
        csy,
    ]

    teammate, shared_starts = (
        s29_main_teammate(
            driver,
            year,
            row.constructorId,
        )
    )

    teammate_alpha_mean = np.nan
    delta_alpha_mean = np.nan
    delta_alpha_lb = np.nan
    delta_alpha_ub = np.nan
    p_driver_better = np.nan
    corr_car_driver = float(
        np.corrcoef(
            car_draws,
            alpha_draws,
        )[
            0,
            1,
        ]
    )

    pace_delta = np.nan
    pace_weekends = 0
    quali_delta = np.nan
    quali_weekends = 0

    if (
        teammate is not None
        and
        (
            teammate,
            year,
        )
        in idx_all.dy_map
    ):
        teammate_draws = s29_alpha[
            :,
            idx_all.dy_map[
                (
                    teammate,
                    year,
                )
            ],
        ]

        delta_draws = (
            alpha_draws
            -
            teammate_draws
        )

        teammate_alpha_mean = float(
            teammate_draws.mean()
        )

        delta_alpha_mean = float(
            delta_draws.mean()
        )

        delta_alpha_lb = float(
            np.quantile(
                delta_draws,
                0.055,
            )
        )

        delta_alpha_ub = float(
            np.quantile(
                delta_draws,
                0.945,
            )
        )

        p_driver_better = float(
            np.mean(
                delta_draws
                >
                0
            )
        )

        pace_evidence = (
            s29_oriented_contrast_summary(
                pace_contrast_all,
                driver,
                teammate,
                year,
                row.constructorId,
            )
        )

        quali_evidence = (
            s29_oriented_contrast_summary(
                q_contrast_all,
                driver,
                teammate,
                year,
                row.constructorId,
            )
        )

        pace_delta = (
            pace_evidence[
                "mean"
            ]
        )

        pace_weekends = (
            pace_evidence[
                "n_weekends"
            ]
        )

        quali_delta = (
            quali_evidence[
                "mean"
            ]
        )

        quali_weekends = (
            quali_evidence[
                "n_weekends"
            ]
        )

    pace_level = (
        s29_team_level_summary(
            pace_level_all,
            year=
                year,
            constructor_id=
                row.constructorId,
            channel=
                "pace",
        )
    )

    q1_level = (
        s29_team_level_summary(
            q_level_all_model,
            year=
                year,
            constructor_id=
                row.constructorId,
            channel=
                "q1",
        )
    )

    ridge = allocation_draw_cache.get(
        csy_key
    )

    if ridge is not None:

        common_driver_draws = ridge[
            "common_driver"
        ]

        total_common_draws = ridge[
            "total"
        ]

        corr_car_common = float(
            np.corrcoef(
                ridge[
                    "car"
                ],
                common_driver_draws,
            )[
                0,
                1,
            ]
        )

        common_driver_mean = float(
            common_driver_draws.mean()
        )

        total_common_mean = float(
            total_common_draws.mean()
        )

        sd_common_driver = float(
            np.std(
                common_driver_draws,
                ddof=1,
            )
        )

        sd_car = float(
            np.std(
                ridge[
                    "car"
                ],
                ddof=1,
            )
        )

        sd_total = float(
            np.std(
                total_common_draws,
                ddof=1,
            )
        )

    else:

        corr_car_common = np.nan
        common_driver_mean = np.nan
        total_common_mean = np.nan
        sd_common_driver = np.nan
        sd_car = np.nan
        sd_total = np.nan

    cmeta = s29_csy_lookup[
        csy_key
    ]

    driver_season_rows.append(
        {
            "driver":
                driver,
            "year":
                year,
            "constructorId":
                int(
                    row.constructorId
                ),
            "constructor":
                row.constructor,
            "starts":
                int(
                    row.starts
                ),

            "alpha_mean":
                float(
                    alpha_draws.mean()
                ),
            "alpha_89_lb":
                float(
                    np.quantile(
                        alpha_draws,
                        0.055,
                    )
                ),
            "alpha_89_ub":
                float(
                    np.quantile(
                        alpha_draws,
                        0.945,
                    )
                ),
            "p_alpha_gt_0":
                float(
                    np.mean(
                        alpha_draws
                        >
                        0
                    )
                ),

            "car_beta_mean":
                float(
                    car_draws.mean()
                ),
            "car_rank":
                int(
                    cmeta[
                        "beta_rank_mean_based"
                    ]
                ),
            "constructor_field_size":
                int(
                    cmeta[
                        "field_size"
                    ]
                ),

            "main_teammate":
                teammate,
            "shared_starts":
                int(
                    shared_starts
                ),
            "teammate_alpha_mean":
                teammate_alpha_mean,
            "delta_alpha_mean":
                delta_alpha_mean,
            "delta_alpha_89_lb":
                delta_alpha_lb,
            "delta_alpha_89_ub":
                delta_alpha_ub,
            "p_driver_better":
                p_driver_better,

            "observed_pace_delta":
                pace_delta,
            "pace_weekends":
                pace_weekends,
            "observed_quali_delta":
                quali_delta,
            "quali_weekends":
                quali_weekends,

            "pace_team_level_observed":
                pace_level[
                    "observed"
                ],
            "pace_team_level_predicted":
                pace_level[
                    "predicted"
                ],
            "pace_team_level_residual":
                pace_level[
                    "residual"
                ],

            "q1_team_level_observed":
                q1_level[
                    "observed"
                ],
            "q1_team_level_predicted":
                q1_level[
                    "predicted"
                ],
            "q1_team_level_residual":
                q1_level[
                    "residual"
                ],

            "common_driver_level_mean":
                common_driver_mean,
            "driver_plus_car_mean":
                total_common_mean,

            "corr_car_vs_driver_alpha":
                corr_car_driver,
            "corr_car_vs_common_driver":
                corr_car_common,

            "sd_common_driver":
                sd_common_driver,
            "sd_car":
                sd_car,
            "sd_driver_plus_car":
                sd_total,
        }
    )


s29_driver_seasons = pd.DataFrame(
    driver_season_rows
)


# Year-to-year changes within a driver's consecutive seasons.
s29_driver_seasons = (
    s29_driver_seasons
    .sort_values(
        [
            "driver",
            "year",
        ]
    )
    .reset_index(
        drop=True
    )
)


for col, new_col in [
    (
        "alpha_mean",
        "alpha_change",
    ),
    (
        "car_beta_mean",
        "car_beta_change",
    ),
]:
    prev = (
        s29_driver_seasons.groupby(
            "driver"
        )[
            col
        ]
        .shift(1)
    )

    prev_year = (
        s29_driver_seasons.groupby(
            "driver"
        )[
            "year"
        ]
        .shift(1)
    )

    s29_driver_seasons[
        new_col
    ] = np.where(
        (
            s29_driver_seasons[
                "year"
            ]
            -
            prev_year
        )
        ==
        1,
        s29_driver_seasons[
            col
        ]
        -
        prev,
        np.nan,
    )


s29_driver_seasons[
    "previous_constructor"
] = (
    s29_driver_seasons.groupby(
        "driver"
    )[
        "constructor"
    ]
    .shift(1)
)


s29_driver_seasons[
    "previous_year"
] = (
    s29_driver_seasons.groupby(
        "driver"
    )[
        "year"
    ]
    .shift(1)
)


s29_driver_seasons[
    "is_consecutive"
] = (
    s29_driver_seasons[
        "year"
    ]
    -
    s29_driver_seasons[
        "previous_year"
    ]
    ==
    1
)


s29_driver_seasons[
    "switched_constructor"
] = (
    s29_driver_seasons[
        "is_consecutive"
    ]
    &
    (
        s29_driver_seasons[
            "constructor"
        ]
        !=
        s29_driver_seasons[
            "previous_constructor"
        ]
    )
)


# ==========================================================
# 10. PÉREZ DECOMPOSITION
# ==========================================================

s29_target_driver = s29_resolve_driver(
    S29_TARGET_DRIVER_QUERY
)


s29_perez = (
    s29_driver_seasons[
        (
            s29_driver_seasons[
                "driver"
            ]
            ==
            s29_target_driver
        )
        &
        (
            s29_driver_seasons[
                "year"
            ].between(
                S29_PEREZ_START_YEAR,
                S29_PEREZ_END_YEAR,
            )
        )
    ]
    .copy()
    .sort_values(
        "year"
    )
)


print(
    "\n========================================"
)
print(
    f"{s29_target_driver.upper()} — SEASON DECOMPOSITION"
)
print(
    "========================================"
)


perez_display_cols = [
    "year",
    "constructor",
    "starts",
    "alpha_mean",
    "alpha_89_lb",
    "alpha_89_ub",
    "alpha_change",
    "car_beta_mean",
    "car_beta_change",
    "car_rank",
    "main_teammate",
    "shared_starts",
    "teammate_alpha_mean",
    "delta_alpha_mean",
    "p_driver_better",
    "observed_pace_delta",
    "observed_quali_delta",
    "pace_team_level_observed",
    "pace_team_level_predicted",
    "q1_team_level_observed",
    "q1_team_level_predicted",
    "common_driver_level_mean",
    "driver_plus_car_mean",
    "corr_car_vs_driver_alpha",
    "corr_car_vs_common_driver",
    "sd_common_driver",
    "sd_car",
    "sd_driver_plus_car",
]


display(
    s29_perez[
        perez_display_cols
    ]
)


# ==========================================================
# 11. PÉREZ 2022 RIDGE DETAIL
# ==========================================================

S29_FOCUS_YEAR = 2022


focus_row_df = s29_perez[
    s29_perez[
        "year"
    ]
    ==
    S29_FOCUS_YEAR
]


if focus_row_df.empty:
    raise RuntimeError(
        f"No {s29_target_driver} {S29_FOCUS_YEAR} row found."
    )


focus_row = focus_row_df.iloc[0]

focus_csy_key = (
    int(
        focus_row[
            "constructorId"
        ]
    ),
    int(
        S29_FOCUS_YEAR
    ),
)

focus_ridge = allocation_draw_cache[
    focus_csy_key
]


print(
    "\n========================================"
)
print(
    f"{s29_target_driver} {S29_FOCUS_YEAR} — ALLOCATION RIDGE"
)
print(
    "========================================"
)

print(
    "Constructor:",
    focus_row[
        "constructor"
    ],
)

print(
    "Driver alpha:",
    focus_row[
        "alpha_mean"
    ],
)

print(
    "Main teammate:",
    focus_row[
        "main_teammate"
    ],
)

print(
    "Posterior alpha difference:",
    focus_row[
        "delta_alpha_mean"
    ],
)

print(
    "Observed pace contrast:",
    focus_row[
        "observed_pace_delta"
    ],
)

print(
    "Observed qualifying contrast:",
    focus_row[
        "observed_quali_delta"
    ],
)

print(
    "Car beta:",
    focus_row[
        "car_beta_mean"
    ],
)

print(
    "Common driver level:",
    focus_row[
        "common_driver_level_mean"
    ],
)

print(
    "Driver + car total:",
    focus_row[
        "driver_plus_car_mean"
    ],
)

print(
    "corr(car, Pérez alpha):",
    focus_row[
        "corr_car_vs_driver_alpha"
    ],
)

print(
    "corr(car, common Red Bull driver level):",
    focus_row[
        "corr_car_vs_common_driver"
    ],
)

print(
    "SD common driver level:",
    focus_row[
        "sd_common_driver"
    ],
)

print(
    "SD car beta:",
    focus_row[
        "sd_car"
    ],
)

print(
    "SD driver + car:",
    focus_row[
        "sd_driver_plus_car"
    ],
)


# ==========================================================
# 12. LOCAL LIKELIHOOD-INVARIANCE DEMONSTRATION
# ==========================================================
#
# This is NOT a valid posterior transformation because it ignores the
# global centring constraints and temporal/prior structure.
#
# It demonstrates the observation-level identification problem:
#
#   contrast:
#       (alpha_A - delta) - (alpha_B - delta)
#       = alpha_A - alpha_B
#
#   level:
#       0.5[(alpha_A-delta)+(alpha_B-delta)] + (beta+delta)
#       = 0.5(alpha_A+alpha_B) + beta
#
# So the modern observations for that constructor-season cannot,
# by themselves, locate the common driver level separately from beta.
# ==========================================================

focus_driver = s29_target_driver
focus_teammate = focus_row[
    "main_teammate"
]

focus_alpha_driver = float(
    focus_row[
        "alpha_mean"
    ]
)

focus_alpha_teammate = float(
    focus_row[
        "teammate_alpha_mean"
    ]
)

focus_beta = float(
    focus_row[
        "car_beta_mean"
    ]
)


shift_values = np.array(
    [
        -0.40,
        -0.30,
        -0.20,
        -0.10,
        0.00,
        0.10,
        0.20,
        0.30,
        0.40,
    ],
    dtype=float,
)


local_shift_rows = []


baseline_contrast = (
    focus_alpha_driver
    -
    focus_alpha_teammate
)

baseline_level = (
    0.5
    *
    (
        focus_alpha_driver
        +
        focus_alpha_teammate
    )
    +
    focus_beta
)


for shift in shift_values:

    shifted_driver = (
        focus_alpha_driver
        -
        shift
    )

    shifted_teammate = (
        focus_alpha_teammate
        -
        shift
    )

    shifted_beta = (
        focus_beta
        +
        shift
    )

    shifted_contrast = (
        shifted_driver
        -
        shifted_teammate
    )

    shifted_level = (
        0.5
        *
        (
            shifted_driver
            +
            shifted_teammate
        )
        +
        shifted_beta
    )

    local_shift_rows.append(
        {
            "shift_from_drivers_to_car":
                float(
                    shift
                ),
            "driver_alpha":
                shifted_driver,
            "teammate_alpha":
                shifted_teammate,
            "car_beta":
                shifted_beta,
            "teammate_contrast_prediction":
                shifted_contrast,
            "team_level_prediction_ex_tenure":
                shifted_level,
            "contrast_change":
                shifted_contrast
                -
                baseline_contrast,
            "level_change":
                shifted_level
                -
                baseline_level,
        }
    )


s29_local_invariance = pd.DataFrame(
    local_shift_rows
)


print(
    "\n========================================"
)
print(
    "LOCAL LIKELIHOOD-EQUIVALENT SHIFT"
)
print(
    "========================================"
)

display(
    s29_local_invariance
)


print(
    "Max absolute contrast-prediction change:",
    float(
        s29_local_invariance[
            "contrast_change"
        ].abs().max()
    ),
)

print(
    "Max absolute team-level-prediction change:",
    float(
        s29_local_invariance[
            "level_change"
        ].abs().max()
    ),
)


# ==========================================================
# 13. SELECTED DOMINANT-TEAM CASES
# ==========================================================

case_specs = [
    {
        "label":
            "Pérez -> Red Bull",
        "driver_query":
            "perez",
        "entry_year":
            2021,
    },
    {
        "label":
            "Bottas -> Mercedes",
        "driver_query":
            "bottas",
        "entry_year":
            2017,
    },
    {
        "label":
            "Barrichello -> Ferrari",
        "driver_query":
            "barrichello",
        "entry_year":
            2000,
    },
    {
        "label":
            "Irvine -> Ferrari",
        "driver_query":
            "irvine",
        "entry_year":
            1996,
    },
    {
        "label":
            "Webber -> Red Bull",
        "driver_query":
            "webber",
        "entry_year":
            2007,
    },
]


case_rows = []


for spec in case_specs:

    driver = s29_resolve_driver(
        spec[
            "driver_query"
        ]
    )

    entry_year = int(
        spec[
            "entry_year"
        ]
    )

    current = s29_driver_seasons[
        (
            s29_driver_seasons[
                "driver"
            ]
            ==
            driver
        )
        &
        (
            s29_driver_seasons[
                "year"
            ]
            ==
            entry_year
        )
    ]

    previous = s29_driver_seasons[
        (
            s29_driver_seasons[
                "driver"
            ]
            ==
            driver
        )
        &
        (
            s29_driver_seasons[
                "year"
            ]
            ==
            entry_year
            -
            1
        )
    ]

    if current.empty:
        continue

    cur = current.iloc[0]

    prev = (
        previous.iloc[0]
        if not previous.empty
        else None
    )

    case_rows.append(
        {
            "case":
                spec[
                    "label"
                ],
            "driver":
                driver,
            "entry_year":
                entry_year,
            "constructor":
                cur[
                    "constructor"
                ],

            "alpha_before":
                (
                    prev[
                        "alpha_mean"
                    ]
                    if prev is not None
                    else np.nan
                ),
            "alpha_entry":
                cur[
                    "alpha_mean"
                ],
            "alpha_jump":
                (
                    cur[
                        "alpha_mean"
                    ]
                    -
                    prev[
                        "alpha_mean"
                    ]
                    if prev is not None
                    else np.nan
                ),

            "car_before":
                (
                    prev[
                        "car_beta_mean"
                    ]
                    if prev is not None
                    else np.nan
                ),
            "car_entry":
                cur[
                    "car_beta_mean"
                ],
            "car_jump":
                (
                    cur[
                        "car_beta_mean"
                    ]
                    -
                    prev[
                        "car_beta_mean"
                    ]
                    if prev is not None
                    else np.nan
                ),

            "car_rank_entry":
                cur[
                    "car_rank"
                ],

            "main_teammate":
                cur[
                    "main_teammate"
                ],
            "delta_alpha_entry":
                cur[
                    "delta_alpha_mean"
                ],
            "pace_delta_entry":
                cur[
                    "observed_pace_delta"
                ],
            "quali_delta_entry":
                cur[
                    "observed_quali_delta"
                ],

            "corr_car_vs_driver_alpha":
                cur[
                    "corr_car_vs_driver_alpha"
                ],
            "corr_car_vs_common_driver":
                cur[
                    "corr_car_vs_common_driver"
                ],

            "sd_common_driver":
                cur[
                    "sd_common_driver"
                ],
            "sd_car":
                cur[
                    "sd_car"
                ],
            "sd_driver_plus_car":
                cur[
                    "sd_driver_plus_car"
                ],
        }
    )


s29_selected_cases = pd.DataFrame(
    case_rows
)


print(
    "\n========================================"
)
print(
    "SELECTED DOMINANT-TEAM / STRONG-TEAM ENTRIES"
)
print(
    "========================================"
)

display(
    s29_selected_cases
)


# ==========================================================
# 14. GLOBAL YEAR-TO-YEAR ALPHA-vs-CAR MOVEMENT
# ==========================================================
#
# This is a diagnostic, NOT a causal estimate. Better teams recruit
# better/improving drivers, so switchers have selection effects.
#
# The SAME-CONSTRUCTOR subset is particularly informative because it
# removes the "better team hired a better driver" mechanism.
# ==========================================================

s29_changes = s29_driver_seasons[
    s29_driver_seasons[
        "is_consecutive"
    ]
    &
    s29_driver_seasons[
        "alpha_change"
    ].notna()
    &
    s29_driver_seasons[
        "car_beta_change"
    ].notna()
].copy()


def s29_safe_corr(
    x,
    y,
    method="pearson",
):
    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    ok = (
        np.isfinite(
            x
        )
        &
        np.isfinite(
            y
        )
    )

    x = x[
        ok
    ]

    y = y[
        ok
    ]

    if (
        len(
            x
        )
        <
        3
        or
        np.std(
            x
        )
        ==
        0
        or
        np.std(
            y
        )
        ==
        0
    ):
        return np.nan

    if method == "pearson":
        return float(
            st.pearsonr(
                x,
                y,
            ).statistic
        )

    return float(
        st.spearmanr(
            x,
            y,
        ).statistic
    )


def s29_year_fe_slope(
    frame,
):
    sxx = []
    sxy = []

    for year, grp in frame.groupby(
        "year"
    ):

        if len(grp) < 2:
            continue

        x = grp[
            "car_beta_change"
        ].to_numpy(
            dtype=float
        )

        y = grp[
            "alpha_change"
        ].to_numpy(
            dtype=float
        )

        x = (
            x
            -
            x.mean()
        )

        y = (
            y
            -
            y.mean()
        )

        this_sxx = float(
            np.sum(
                x
                **
                2
            )
        )

        this_sxy = float(
            np.sum(
                x
                *
                y
            )
        )

        if this_sxx > 0:
            sxx.append(
                this_sxx
            )
            sxy.append(
                this_sxy
            )

    if not sxx:
        return np.nan

    return float(
        np.sum(
            sxy
        )
        /
        np.sum(
            sxx
        )
    )


s29_rng = np.random.default_rng(
    S29_BOOTSTRAP_SEED
)


def s29_year_cluster_bootstrap_slope(
    frame,
):
    year_stats = []

    for year, grp in frame.groupby(
        "year"
    ):

        if len(grp) < 2:
            continue

        x = grp[
            "car_beta_change"
        ].to_numpy(
            dtype=float
        )

        y = grp[
            "alpha_change"
        ].to_numpy(
            dtype=float
        )

        x = (
            x
            -
            x.mean()
        )

        y = (
            y
            -
            y.mean()
        )

        sxx = float(
            np.sum(
                x
                **
                2
            )
        )

        sxy = float(
            np.sum(
                x
                *
                y
            )
        )

        if sxx > 0:
            year_stats.append(
                (
                    sxx,
                    sxy,
                )
            )

    if len(
        year_stats
    ) < 3:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    sxx = np.asarray(
        [
            row[0]
            for row in year_stats
        ],
        dtype=float,
    )

    sxy = np.asarray(
        [
            row[1]
            for row in year_stats
        ],
        dtype=float,
    )

    observed = float(
        sxy.sum()
        /
        sxx.sum()
    )

    n_years = len(
        sxx
    )

    ids = s29_rng.integers(
        0,
        n_years,
        size=(
            S29_BOOTSTRAPS,
            n_years,
        ),
    )

    boot_xx = sxx[
        ids
    ].sum(
        axis=1
    )

    boot_xy = sxy[
        ids
    ].sum(
        axis=1
    )

    good = (
        boot_xx
        >
        0
    )

    boot = (
        boot_xy[
            good
        ]
        /
        boot_xx[
            good
        ]
    )

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


change_groups = {
    "ALL consecutive driver-seasons":
        s29_changes,

    "SAME constructor":
        s29_changes[
            ~s29_changes[
                "switched_constructor"
            ]
        ],

    "SWITCHED constructor":
        s29_changes[
            s29_changes[
                "switched_constructor"
            ]
        ],

    "Large car improvement (delta beta >= +0.5)":
        s29_changes[
            s29_changes[
                "car_beta_change"
            ]
            >=
            0.5
        ],

    "Large car decline (delta beta <= -0.5)":
        s29_changes[
            s29_changes[
                "car_beta_change"
            ]
            <=
            -0.5
        ],
}


change_summary_rows = []


for label, grp in change_groups.items():

    (
        slope,
        slope_lb,
        slope_ub,
    ) = s29_year_cluster_bootstrap_slope(
        grp
    )

    change_summary_rows.append(
        {
            "group":
                label,
            "rows":
                int(
                    len(
                        grp
                    )
                ),
            "drivers":
                int(
                    grp[
                        "driver"
                    ].nunique()
                ),
            "years":
                int(
                    grp[
                        "year"
                    ].nunique()
                ),
            "pearson_delta_car_vs_delta_alpha":
                s29_safe_corr(
                    grp[
                        "car_beta_change"
                    ],
                    grp[
                        "alpha_change"
                    ],
                    method=
                        "pearson",
                ),
            "spearman_delta_car_vs_delta_alpha":
                s29_safe_corr(
                    grp[
                        "car_beta_change"
                    ],
                    grp[
                        "alpha_change"
                    ],
                    method=
                        "spearman",
                ),
            "year_FE_slope":
                slope,
            "slope_95_lb":
                slope_lb,
            "slope_95_ub":
                slope_ub,
            "mean_alpha_change":
                (
                    float(
                        grp[
                            "alpha_change"
                        ].mean()
                    )
                    if len(grp)
                    else np.nan
                ),
            "mean_car_change":
                (
                    float(
                        grp[
                            "car_beta_change"
                        ].mean()
                    )
                    if len(grp)
                    else np.nan
                ),
        }
    )


s29_change_summary = pd.DataFrame(
    change_summary_rows
)


print(
    "\n========================================"
)
print(
    "GLOBAL YEAR-TO-YEAR ALPHA vs CAR-BETA MOVEMENT"
)
print(
    "========================================"
)

display(
    s29_change_summary
)


# ==========================================================
# 15. TOP-2 CAR ENTRY AUDIT
# ==========================================================
#
# Among consecutive seasons where a driver changed constructor,
# inspect moves from a non-top-2 constructor to a top-2 constructor.
# This is descriptive and subject to selection effects, but useful for
# finding Pérez/Bottas-style cases automatically.
# ==========================================================

prev_rank = (
    s29_driver_seasons.groupby(
        "driver"
    )[
        "car_rank"
    ]
    .shift(1)
)


s29_driver_seasons[
    "previous_car_rank"
] = prev_rank


s29_top2_entries = s29_driver_seasons[
    s29_driver_seasons[
        "is_consecutive"
    ]
    &
    s29_driver_seasons[
        "switched_constructor"
    ]
    &
    (
        s29_driver_seasons[
            "car_rank"
        ]
        <=
        2
    )
    &
    (
        s29_driver_seasons[
            "previous_car_rank"
        ]
        >
        2
    )
].copy()


s29_top2_entries = (
    s29_top2_entries[
        [
            "driver",
            "year",
            "previous_constructor",
            "constructor",
            "previous_car_rank",
            "car_rank",
            "alpha_change",
            "car_beta_change",
            "alpha_mean",
            "car_beta_mean",
            "main_teammate",
            "delta_alpha_mean",
            "observed_pace_delta",
            "observed_quali_delta",
            "corr_car_vs_driver_alpha",
            "corr_car_vs_common_driver",
        ]
    ]
    .sort_values(
        "alpha_change",
        ascending=False,
    )
)


print(
    "\n========================================"
)
print(
    "DRIVERS ENTERING A TOP-2 CAR"
)
print(
    "========================================"
)

display(
    s29_top2_entries
)


# ==========================================================
# 16. PLOTS
# ==========================================================

# ----------------------------------------------------------
# A. Pérez alpha and primary-constructor beta
# ----------------------------------------------------------

plt.figure(
    figsize=(
        11,
        6,
    )
)

plt.plot(
    s29_perez[
        "year"
    ],
    s29_perez[
        "alpha_mean"
    ],
    marker="o",
    label=
        f"{s29_target_driver} alpha",
)

plt.fill_between(
    s29_perez[
        "year"
    ],
    s29_perez[
        "alpha_89_lb"
    ],
    s29_perez[
        "alpha_89_ub"
    ],
    alpha=0.15,
)

plt.plot(
    s29_perez[
        "year"
    ],
    s29_perez[
        "car_beta_mean"
    ],
    marker="s",
    label=
        "Primary-constructor beta",
)

plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.xlabel(
    "Season"
)

plt.ylabel(
    "Latent model units"
)

plt.title(
    f"{s29_target_driver}: driver alpha vs constructor beta"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# B. Pérez direct teammate evidence vs posterior delta alpha
# ----------------------------------------------------------

plt.figure(
    figsize=(
        11,
        6,
    )
)

plt.plot(
    s29_perez[
        "year"
    ],
    s29_perez[
        "delta_alpha_mean"
    ],
    marker="o",
    label=
        "Posterior Δalpha",
)

plt.plot(
    s29_perez[
        "year"
    ],
    s29_perez[
        "observed_pace_delta"
    ],
    marker="s",
    label=
        "Observed pace Δ",
)

plt.plot(
    s29_perez[
        "year"
    ],
    s29_perez[
        "observed_quali_delta"
    ],
    marker="^",
    label=
        "Observed qualifying Δ",
)

plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.xlabel(
    "Season"
)

plt.ylabel(
    "Selected driver minus main teammate"
)

plt.title(
    f"{s29_target_driver}: direct teammate evidence"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# C. 2022 Red Bull posterior allocation ridge
# ----------------------------------------------------------

plt.figure(
    figsize=(
        8,
        6,
    )
)

plt.scatter(
    focus_ridge[
        "car"
    ],
    focus_ridge[
        "common_driver"
    ],
    alpha=0.18,
    s=12,
)

plt.xlabel(
    f"{focus_row['constructor']} {S29_FOCUS_YEAR} beta"
)

plt.ylabel(
    "Race-start-weighted common driver alpha"
)

plt.title(
    (
        f"{focus_row['constructor']} {S29_FOCUS_YEAR}: "
        "posterior driver/car allocation ridge\n"
        f"corr = {focus_row['corr_car_vs_common_driver']:.3f}"
    )
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# D. Global alpha change vs car-beta change
# ----------------------------------------------------------

plt.figure(
    figsize=(
        9,
        7,
    )
)

same_team = s29_changes[
    ~s29_changes[
        "switched_constructor"
    ]
]

switchers = s29_changes[
    s29_changes[
        "switched_constructor"
    ]
]

plt.scatter(
    same_team[
        "car_beta_change"
    ],
    same_team[
        "alpha_change"
    ],
    alpha=0.35,
    s=22,
    label=
        "Same constructor",
)

plt.scatter(
    switchers[
        "car_beta_change"
    ],
    switchers[
        "alpha_change"
    ],
    alpha=0.55,
    s=28,
    marker="x",
    label=
        "Changed constructor",
)

plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.axvline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.xlabel(
    "Year-to-year change in constructor beta"
)

plt.ylabel(
    "Year-to-year change in driver alpha"
)

plt.title(
    "Do driver alpha changes track changes in car strength?"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ==========================================================
# 17. SAVE OUTPUTS
# ==========================================================

s29_allocation.to_csv(
    S29_OUTPUT_DIR
    /
    "constructor_season_driver_car_allocation_ridge.csv",
    index=False,
)


s29_driver_seasons.to_csv(
    S29_OUTPUT_DIR
    /
    "modern_driver_season_decomposition.csv",
    index=False,
)


s29_perez.to_csv(
    S29_OUTPUT_DIR
    /
    "perez_season_decomposition.csv",
    index=False,
)


s29_local_invariance.to_csv(
    S29_OUTPUT_DIR
    /
    "perez_2022_local_likelihood_invariance.csv",
    index=False,
)


s29_selected_cases.to_csv(
    S29_OUTPUT_DIR
    /
    "selected_dominant_team_entries.csv",
    index=False,
)


s29_change_summary.to_csv(
    S29_OUTPUT_DIR
    /
    "global_alpha_vs_car_change_summary.csv",
    index=False,
)


s29_top2_entries.to_csv(
    S29_OUTPUT_DIR
    /
    "top2_car_entries.csv",
    index=False,
)


# ==========================================================
# 18. TRIAGE SUMMARY
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 29 TRIAGE SUMMARY"
)
print(
    "========================================"
)


p22 = focus_row


print(
    f"{s29_target_driver} {S29_FOCUS_YEAR}:"
)

print(
    f"  alpha = {p22['alpha_mean']:+.3f} "
    f"[89% {p22['alpha_89_lb']:+.3f}, "
    f"{p22['alpha_89_ub']:+.3f}]"
)

print(
    f"  car beta = {p22['car_beta_mean']:+.3f} "
    f"(rank #{int(p22['car_rank'])})"
)

print(
    f"  main teammate = {p22['main_teammate']}"
)

print(
    f"  posterior Δalpha = {p22['delta_alpha_mean']:+.3f}"
)

print(
    f"  observed pace Δ = {p22['observed_pace_delta']:+.3f}"
)

print(
    f"  observed quali Δ = {p22['observed_quali_delta']:+.3f}"
)

print(
    f"  corr(car, driver alpha) = "
    f"{p22['corr_car_vs_driver_alpha']:+.3f}"
)

print(
    f"  corr(car, common team driver level) = "
    f"{p22['corr_car_vs_common_driver']:+.3f}"
)

print(
    f"  SD common driver = {p22['sd_common_driver']:.3f}"
)

print(
    f"  SD car = {p22['sd_car']:.3f}"
)

print(
    f"  SD driver + car = {p22['sd_driver_plus_car']:.3f}"
)


same_summary = s29_change_summary[
    s29_change_summary[
        "group"
    ]
    ==
    "SAME constructor"
]

switch_summary = s29_change_summary[
    s29_change_summary[
        "group"
    ]
    ==
    "SWITCHED constructor"
]


if not same_summary.empty:

    r = same_summary.iloc[0]

    print(
        "\nSame-constructor year-to-year diagnostic:"
    )

    print(
        f"  Pearson corr(Δbeta, Δalpha) = "
        f"{r['pearson_delta_car_vs_delta_alpha']:+.3f}"
    )

    print(
        f"  Year-FE slope = "
        f"{r['year_FE_slope']:+.3f} "
        f"[95% {r['slope_95_lb']:+.3f}, "
        f"{r['slope_95_ub']:+.3f}]"
    )


if not switch_summary.empty:

    r = switch_summary.iloc[0]

    print(
        "\nConstructor-switch year-to-year diagnostic:"
    )

    print(
        f"  Pearson corr(Δbeta, Δalpha) = "
        f"{r['pearson_delta_car_vs_delta_alpha']:+.3f}"
    )

    print(
        f"  Year-FE slope = "
        f"{r['year_FE_slope']:+.3f} "
        f"[95% {r['slope_95_lb']:+.3f}, "
        f"{r['slope_95_ub']:+.3f}]"
    )


print(
    "\nHow to read this audit:"
)

print(
    "  1. A strongly negative corr(car, common-driver alpha) plus a"
)

print(
    "     much smaller SD for (driver + car) than for either component"
)

print(
    "     is direct evidence of a driver/car allocation ridge."
)

print(
    "  2. The local-shift table should show exactly zero change in the"
)

print(
    "     modern teammate-contrast and team-level predictions. That"
)

print(
    "     demonstrates why the observations alone cannot locate the"
)

print(
    "     pair's common driver level separately from the constructor."
)

print(
    "  3. If alpha systematically rises when beta rises even for SAME-"
)

print(
    "     constructor year-to-year changes, that is stronger evidence"
)

print(
    "     of common team performance leaking into driver alpha."
)

print(
    "  4. Switchers are less decisive because good/improving drivers"
)

print(
    "     are selected by better teams, but repeated Pérez/Bottas-style"
)

print(
    "     jumps are still an important stress-test signal."
)

print(
    "  5. If Pérez's Red Bull alpha jump coincides with a huge beta jump"
)

print(
    "     while his direct teammate evidence gets worse, the next test"
)

print(
    "     should be a TARGETED REFIT rather than changing the model by"
)

print(
    "     intuition."
)

print(
    "\nSaved outputs ->"
)

print(
    S29_OUTPUT_DIR.resolve()
)

print(
    "\nStage 29 complete."
)


In [ ]:
# ==========================================================
# STAGE 30 — PRIOR / ALLOCATION SENSITIVITY REFITS
# ==========================================================
#
# PURPOSE
# -------
# Stage 29 established two things:
#
#   1. Teammate DIFFERENCES are well identified.
#   2. Common driver level and constructor beta have a strong
#      negative posterior allocation ridge.
#
# Pérez 2022 is the clearest example:
#
#     alpha_Perez - alpha_Max
#
# is tightly informed by direct teammate evidence, while
#
#     0.5 * (alpha_Perez + alpha_Max) + beta_RedBull
#
# is tightly informed by the team-level likelihood. The split of
# that common level between drivers and car is less directly
# identified.
#
# Stage 30 asks:
#
#     How sensitive are absolute alpha, constructor beta and JAWS
#     to plausible / deliberately stressful prior changes that
#     locate that allocation ridge?
#
# THIS IS A SENSITIVITY ANALYSIS, NOT A NEW PRODUCTION MODEL.
#
# The accepted frozen production posterior is never overwritten.
#
# ==========================================================
# SENSITIVITY VARIANTS
# ==========================================================
#
# FROZEN REFERENCE
#   sigma_car prior scale          = 0.75
#   sigma_driver_base prior scale  = 0.75
#   sigma_driver_drift prior scale = 0.25
#
# Variant A — CAR WIDE
#   sigma_car prior scale          = 1.50
#   everything else frozen
#
# This asks whether the original car prior was preventing the
# constructor from absorbing more of extreme team performance.
#
# Variant B — CAR TIGHT
#   sigma_car prior scale          = 0.375
#   everything else frozen
#
# This is deliberately a stress test in the opposite direction:
# if car effects are constrained, does common team performance
# migrate into driver alpha?
#
# Variant C — DRIVER TIGHT
#   sigma_driver_base prior scale  = 0.375
#   sigma_driver_drift prior scale = 0.125
#   car prior unchanged
#
# This asks whether the absolute vertical location of drivers is
# sensitive to the state-space prior that anchors career level and
# year-to-year movement.
#
# ==========================================================
# INTERPRETATION
# ==========================================================
#
# Robust outcome:
#   - Red Bull 2022 beta / Perez alpha barely move;
#   - teammate delta remains fixed;
#   - JAWS / rank ordering barely changes;
#   - observable predictions remain essentially unchanged.
#
# Prior-sensitive allocation:
#   - Perez / Max common alpha and Red Bull beta trade materially;
#   - teammate delta and observable predictions remain similar;
#   - absolute alpha / JAWS move appreciably.
#
# If the latter occurs, the model is telling us that relative
# ability is robust but absolute driver-vs-car allocation is
# materially prior-dependent.
#
# ==========================================================
# COMPUTATION
# ==========================================================
#
# Three validation/sensitivity refits:
#
#   2 chains x (400 tune + 400 posterior)
#   nutpie / JAX
#   target_accept = 0.92
#
# Each is checkpointed independently.
#
# If a checkpoint already exists, it is loaded and NO resampling
# is performed.
#
# ==========================================================

from dataclasses import replace
from pathlib import Path
import json
import time

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

S30_CHAINS = 2
S30_TUNE = 400
S30_DRAWS = 400
S30_TARGET_ACCEPT = 0.92

S30_SEEDS = {
    "car_wide":
        30041,
    "car_tight":
        30042,
    "driver_tight":
        30043,
}

S30_OUTPUT_DIR = (
    Path(CFG.output_dir)
    /
    "validation_prior_allocation_sensitivity"
)

S30_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

S30_CHECKPOINT_ENGINE = "h5netcdf"


# ==========================================================
# 1. REQUIRED OBJECTS
# ==========================================================

required_objects = [
    "CFG",
    "idx_all",
    "build_joint_model_1982_2025",
    "final_trace_2025",

    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",

    "hist_q_pack",
    "hist_race_pack",

    "actual_starts",

    # Exact frozen Stage-16 scoring definitions.
    "scored_drivers",
    "eligible_drivers",
    "driver_years",
    "starts_lookup",
    "teammates_lookup",

    "PANTHEON_PEAK_WEIGHT",
    "PANTHEON_LONGEVITY_WEIGHT",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Stage 30 requires the final model/scoring environment. "
        "Missing:\n  "
        + "\n  ".join(
            missing
        )
    )


# ==========================================================
# 2. CHECKPOINT BACKEND
# ==========================================================

try:
    import h5netcdf  # noqa: F401
    import h5py      # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Stage 30 requires h5netcdf + h5py for grouped "
        "checkpoint files.\n\n"
        "Install with:\n"
        "    %pip install h5netcdf h5py"
    ) from exc


# ==========================================================
# 3. SENSITIVITY CONFIGURATIONS
# ==========================================================

S30_VARIANTS = {
    "car_wide":
        {
            "label":
                "CAR WIDE",
            "description":
                "double constructor-scale prior",
            "cfg":
                replace(
                    CFG,
                    sigma_car_prior=
                        1.50,
                ),
        },

    "car_tight":
        {
            "label":
                "CAR TIGHT",
            "description":
                "halve constructor-scale prior",
            "cfg":
                replace(
                    CFG,
                    sigma_car_prior=
                        0.375,
                ),
        },

    "driver_tight":
        {
            "label":
                "DRIVER TIGHT",
            "description":
                "halve driver baseline + drift prior scales",
            "cfg":
                replace(
                    CFG,
                    sigma_driver_base_prior=
                        0.375,
                    sigma_driver_drift_prior=
                        0.125,
                ),
        },
}


print(
    "========================================"
)
print(
    "STAGE 30 — PRIOR / ALLOCATION SENSITIVITY"
)
print(
    "========================================"
)

print(
    "Frozen prior scales:"
)

print(
    "  sigma_car_prior =",
    CFG.sigma_car_prior,
)

print(
    "  sigma_driver_base_prior =",
    CFG.sigma_driver_base_prior,
)

print(
    "  sigma_driver_drift_prior =",
    CFG.sigma_driver_drift_prior,
)


print(
    "\nSensitivity variants:"
)

for key, spec in S30_VARIANTS.items():

    cfg = spec[
        "cfg"
    ]

    print(
        f"  {key:14s} | "
        f"car={cfg.sigma_car_prior:.3f} | "
        f"driver_base={cfg.sigma_driver_base_prior:.3f} | "
        f"driver_drift={cfg.sigma_driver_drift_prior:.3f}"
    )


# Save configuration metadata.
with open(
    S30_OUTPUT_DIR
    /
    "sensitivity_config.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "chains":
                S30_CHAINS,
            "tune":
                S30_TUNE,
            "draws":
                S30_DRAWS,
            "target_accept":
                S30_TARGET_ACCEPT,

            "frozen":
                {
                    "sigma_car_prior":
                        CFG.sigma_car_prior,
                    "sigma_driver_base_prior":
                        CFG.sigma_driver_base_prior,
                    "sigma_driver_drift_prior":
                        CFG.sigma_driver_drift_prior,
                },

            "variants":
                {
                    key:
                        {
                            "sigma_car_prior":
                                spec[
                                    "cfg"
                                ].sigma_car_prior,
                            "sigma_driver_base_prior":
                                spec[
                                    "cfg"
                                ].sigma_driver_base_prior,
                            "sigma_driver_drift_prior":
                                spec[
                                    "cfg"
                                ].sigma_driver_drift_prior,
                        }
                    for key, spec
                    in S30_VARIANTS.items()
                },
        },
        f,
        indent=2,
    )


# ==========================================================
# 4. BUILD + SAMPLE / LOAD EACH SENSITIVITY FIT
# ==========================================================

s30_traces = {}


for key, spec in S30_VARIANTS.items():

    cfg_variant = spec[
        "cfg"
    ]

    trace_path = (
        S30_OUTPUT_DIR
        /
        (
            f"{key}_"
            f"{S30_TUNE}tune_"
            f"{S30_DRAWS}draw_"
            f"{S30_CHAINS}chain_"
            "nutpie_jax.nc"
        )
    )

    print(
        "\n========================================"
    )

    print(
        spec[
            "label"
        ]
    )

    print(
        spec[
            "description"
        ]
    )

    print(
        "========================================"
    )

    print(
        "Checkpoint:"
    )

    print(
        trace_path
    )


    # ------------------------------------------------------
    # Load existing valid checkpoint.
    # ------------------------------------------------------

    if trace_path.exists():

        print(
            "\nExisting checkpoint found."
        )

        try:

            trace = xr.open_datatree(
                trace_path,
                engine=
                    S30_CHECKPOINT_ENGINE,
            )

            print(
                "Loaded successfully."
            )

            print(
                "NO RESAMPLING performed."
            )

        except Exception as exc:

            raise RuntimeError(
                "A Stage-30 checkpoint exists but cannot be read.\n"
                "Do NOT overwrite it automatically.\n\n"
                f"{trace_path}\n\n"
                f"{repr(exc)}"
            ) from exc


    # ------------------------------------------------------
    # Otherwise build and sample the variant.
    # ------------------------------------------------------

    else:

        print(
            "\nBuilding sensitivity model..."
        )

        model_variant = (
            build_joint_model_1982_2025(
                pace_contrast_all,
                pace_level_all,
                q_contrast_all,
                q_level_all_model,
                hist_q_pack,
                hist_race_pack,
                idx_all,
                cfg=
                    cfg_variant,
            )
        )

        print(
            f"Sampling {S30_CHAINS} chains x "
            f"({S30_TUNE} tune + {S30_DRAWS} posterior)"
        )

        print(
            "Sampler: nutpie / JAX"
        )

        t0 = time.perf_counter()

        with model_variant:

            trace = pm.sample(
                draws=
                    S30_DRAWS,

                tune=
                    S30_TUNE,

                chains=
                    S30_CHAINS,

                random_seed=
                    S30_SEEDS[
                        key
                    ],

                target_accept=
                    S30_TARGET_ACCEPT,

                nuts_sampler=
                    "nutpie",

                nuts_sampler_kwargs={
                    "backend":
                        "jax",
                },

                return_inferencedata=
                    True,

                progressbar=
                    True,
            )

        elapsed = (
            time.perf_counter()
            -
            t0
        )

        print(
            "\nSampling wall time:",
            f"{elapsed / 60.0:.2f} min",
        )

        trace.to_netcdf(
            trace_path,
            engine=
                S30_CHECKPOINT_ENGINE,
            mode="w",
        )

        print(
            "Saved ->"
        )

        print(
            trace_path
        )

    s30_traces[
        key
    ] = trace


# ==========================================================
# 5. TRACE HELPERS
# ==========================================================

def s30_posterior_dataset(
    trace,
):
    # Current PyMC checkpoint is an xarray DataTree.
    if hasattr(
        trace,
        "children",
    ):
        return (
            trace[
                "posterior"
            ]
            .to_dataset()
        )

    # Fallback for an InferenceData-style object.
    if hasattr(
        trace,
        "posterior",
    ):
        return trace.posterior

    raise TypeError(
        "Could not locate posterior Dataset in trace."
    )


def s30_sample_stats_dataset(
    trace,
):
    if hasattr(
        trace,
        "children",
    ):
        return (
            trace[
                "sample_stats"
            ]
            .to_dataset()
        )

    if hasattr(
        trace,
        "sample_stats",
    ):
        return trace.sample_stats

    raise TypeError(
        "Could not locate sample_stats Dataset in trace."
    )


def s30_stack_state(
    posterior,
    var_name,
):
    arr = np.asarray(
        posterior[
            var_name
        ].values,
        dtype=float,
    )

    return arr.reshape(
        -1,
        arr.shape[-1],
    )


def s30_stack_scalar(
    posterior,
    var_name,
):
    return np.asarray(
        posterior[
            var_name
        ].values,
        dtype=float,
    ).reshape(-1)


s30_posteriors = {
    "frozen":
        s30_posterior_dataset(
            final_trace_2025
        ),
}


for key, trace in s30_traces.items():

    s30_posteriors[
        key
    ] = s30_posterior_dataset(
        trace
    )


# ==========================================================
# 6. FIT HEALTH
# ==========================================================

S30_GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",

    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",

    "tau_hist_quali",
    "tau_hist_race",
]


health_rows = []


for key, trace in s30_traces.items():

    posterior = s30_posteriors[
        key
    ]

    stats = s30_sample_stats_dataset(
        trace
    )

    rhats = []
    esses = []

    for var in S30_GLOBAL_VARS:

        r = np.asarray(
            az.rhat(
                posterior[
                    [
                        var
                    ]
                ]
            )
            .to_array()
            .values,
            dtype=float,
        ).reshape(-1)

        e = np.asarray(
            az.ess(
                posterior[
                    [
                        var
                    ]
                ],
                method=
                    "bulk",
            )
            .to_array()
            .values,
            dtype=float,
        ).reshape(-1)

        rhats.extend(
            r.tolist()
        )

        esses.extend(
            e.tolist()
        )

    health_rows.append(
        {
            "variant":
                key,
            "max_global_rhat":
                float(
                    np.nanmax(
                        rhats
                    )
                ),
            "min_global_ess_bulk":
                float(
                    np.nanmin(
                        esses
                    )
                ),
            "divergences":
                int(
                    np.asarray(
                        stats[
                            "diverging"
                        ].values
                    ).sum()
                ),
        }
    )


s30_health = pd.DataFrame(
    health_rows
)


print(
    "\n========================================"
)
print(
    "SENSITIVITY-FIT HEALTH"
)
print(
    "========================================"
)

display(
    s30_health
)


# ==========================================================
# 7. GLOBAL HYPERPARAMETER POSTERIORS
# ==========================================================

global_rows = []


for variant, posterior in s30_posteriors.items():

    for var in [
        "sigma_driver_base",
        "sigma_driver_drift",
        "sigma_car",
        "gamma_tenure",
    ]:

        draws = s30_stack_scalar(
            posterior,
            var,
        )

        global_rows.append(
            {
                "variant":
                    variant,
                "parameter":
                    var,
                "mean":
                    float(
                        draws.mean()
                    ),
                "sd":
                    float(
                        draws.std(
                            ddof=1
                        )
                    ),
                "q055":
                    float(
                        np.quantile(
                            draws,
                            0.055,
                        )
                    ),
                "q945":
                    float(
                        np.quantile(
                            draws,
                            0.945,
                        )
                    ),
            }
        )


s30_global_summary = pd.DataFrame(
    global_rows
)


print(
    "\n========================================"
)
print(
    "GLOBAL POSTERIOR SENSITIVITY"
)
print(
    "========================================"
)

display(
    s30_global_summary
)


# ==========================================================
# 8. EXACT STAGE-16 JAWS SCORING HELPER
# ==========================================================

s30_scored_index = {
    driver:
        j
    for j, driver
    in enumerate(
        scored_drivers
    )
}


def s30_score_alpha_draws(
    alpha_draws,
):
    """
    Reproduce the locked Stage-16 JAWS definition exactly:

        peak      = mean(best 5 alpha seasons)
        surplus   = sum(max(alpha, 0))
        longevity = sqrt(surplus)

        per-draw normalisation across ALL >=5-season drivers

        JAWS = 0.60 peak + 0.40 longevity
    """

    n_draws = alpha_draws.shape[0]

    n_scored = len(
        scored_drivers
    )

    surplus = np.zeros(
        (
            n_draws,
            n_scored,
        ),
        dtype=float,
    )

    top5 = np.zeros(
        (
            n_draws,
            n_scored,
        ),
        dtype=float,
    )

    for j, driver in enumerate(
        scored_drivers
    ):

        dy_idx = np.asarray(
            [
                idx_all.dy_map[
                    (
                        driver,
                        year,
                    )
                ]
                for year
                in driver_years[
                    driver
                ]
            ],
            dtype=int,
        )

        states = alpha_draws[
            :,
            dy_idx,
        ]

        surplus[
            :,
            j,
        ] = (
            np.maximum(
                states,
                0.0,
            )
            .sum(
                axis=1
            )
        )

        top5[
            :,
            j,
        ] = (
            np.sort(
                states,
                axis=1,
            )[
                :,
                -5:,
            ]
            .mean(
                axis=1
            )
        )

    longevity_raw = np.sqrt(
        surplus
    )

    peak_denominator = top5.max(
        axis=1,
        keepdims=True,
    )

    longevity_denominator = (
        longevity_raw.max(
            axis=1,
            keepdims=True,
        )
    )

    if np.any(
        peak_denominator
        <=
        0
    ):
        raise RuntimeError(
            "Non-positive peak denominator."
        )

    if np.any(
        longevity_denominator
        <=
        0
    ):
        raise RuntimeError(
            "Non-positive longevity denominator."
        )

    peak_score = (
        top5
        /
        peak_denominator
        *
        100.0
    )

    longevity_score = (
        longevity_raw
        /
        longevity_denominator
        *
        100.0
    )

    jaws = (
        PANTHEON_PEAK_WEIGHT
        *
        peak_score

        +
        PANTHEON_LONGEVITY_WEIGHT
        *
        longevity_score
    )

    eligible_idx = np.asarray(
        [
            s30_scored_index[
                driver
            ]
            for driver
            in eligible_drivers
        ],
        dtype=int,
    )

    eligible_jaws = jaws[
        :,
        eligible_idx,
    ]

    order = np.argsort(
        -eligible_jaws,
        axis=1,
    )

    ranks = np.empty_like(
        order,
        dtype=int,
    )

    row_ids = np.arange(
        n_draws
    )[
        :,
        None,
    ]

    ranks[
        row_ids,
        order,
    ] = np.arange(
        1,
        len(
            eligible_drivers
        )
        +
        1,
    )[
        None,
        :
    ]

    rows = []

    for j, driver in enumerate(
        eligible_drivers
    ):

        scored_j = eligible_idx[
            j
        ]

        rows.append(
            {
                "driver":
                    driver,
                "jaws_mean":
                    float(
                        eligible_jaws[
                            :,
                            j,
                        ].mean()
                    ),
                "jaws_median":
                    float(
                        np.median(
                            eligible_jaws[
                                :,
                                j,
                            ]
                        )
                    ),
                "jaws_lb":
                    float(
                        np.quantile(
                            eligible_jaws[
                                :,
                                j,
                            ],
                            0.055,
                        )
                    ),
                "jaws_ub":
                    float(
                        np.quantile(
                            eligible_jaws[
                                :,
                                j,
                            ],
                            0.945,
                        )
                    ),
                "rank_mean":
                    float(
                        ranks[
                            :,
                            j,
                        ].mean()
                    ),
                "rank_median":
                    float(
                        np.median(
                            ranks[
                                :,
                                j,
                            ]
                        )
                    ),
                "peak_raw_mean":
                    float(
                        top5[
                            :,
                            scored_j,
                        ].mean()
                    ),
                "surplus_raw_mean":
                    float(
                        surplus[
                            :,
                            scored_j,
                        ].mean()
                    ),
            }
        )

    table = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "jaws_mean",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    table.insert(
        0,
        "point_rank",
        np.arange(
            1,
            len(
                table
            )
            +
            1,
        ),
    )

    return {
        "table":
            table,
        "jaws":
            eligible_jaws,
        "ranks":
            ranks,
        "top5":
            top5,
        "surplus":
            surplus,
    }


s30_scores = {}


for variant, posterior in s30_posteriors.items():

    alpha_draws = s30_stack_state(
        posterior,
        "alpha_dy",
    )

    s30_scores[
        variant
    ] = s30_score_alpha_draws(
        alpha_draws
    )


# ==========================================================
# 9. DRIVER-SEASON POSTERIOR SENSITIVITY
# ==========================================================

selected_driver_years = [
    # Pérez / Verstappen stress case
    ("perez", 2020),
    ("perez", 2021),
    ("perez", 2022),
    ("perez", 2023),
    ("perez", 2024),

    ("max_verstappen", 2020),
    ("max_verstappen", 2021),
    ("max_verstappen", 2022),
    ("max_verstappen", 2023),
    ("max_verstappen", 2024),

    # Mercedes comparison
    ("bottas", 2016),
    ("bottas", 2017),
    ("bottas", 2018),

    ("hamilton", 2016),
    ("hamilton", 2017),
    ("hamilton", 2018),

    # Ferrari comparison
    ("barrichello", 1999),
    ("barrichello", 2000),
    ("barrichello", 2001),

    ("michael_schumacher", 1999),
    ("michael_schumacher", 2000),
    ("michael_schumacher", 2001),
]


season_rows = []


for variant, posterior in s30_posteriors.items():

    alpha_draws = s30_stack_state(
        posterior,
        "alpha_dy",
    )

    for driver, year in selected_driver_years:

        key = (
            driver,
            year,
        )

        if key not in idx_all.dy_map:
            continue

        draws = alpha_draws[
            :,
            idx_all.dy_map[
                key
            ],
        ]

        season_rows.append(
            {
                "variant":
                    variant,
                "driver":
                    driver,
                "year":
                    year,
                "alpha_mean":
                    float(
                        draws.mean()
                    ),
                "alpha_sd":
                    float(
                        draws.std(
                            ddof=1
                        )
                    ),
                "alpha_89_lb":
                    float(
                        np.quantile(
                            draws,
                            0.055,
                        )
                    ),
                "alpha_89_ub":
                    float(
                        np.quantile(
                            draws,
                            0.945,
                        )
                    ),
            }
        )


s30_season_sensitivity = pd.DataFrame(
    season_rows
)


frozen_alpha_lookup = (
    s30_season_sensitivity[
        s30_season_sensitivity[
            "variant"
        ]
        ==
        "frozen"
    ]
    .set_index(
        [
            "driver",
            "year",
        ]
    )[
        "alpha_mean"
    ]
    .to_dict()
)


s30_season_sensitivity[
    "delta_alpha_vs_frozen"
] = [
    row.alpha_mean
    -
    frozen_alpha_lookup[
        (
            row.driver,
            row.year,
        )
    ]
    for row
    in s30_season_sensitivity.itertuples(
        index=False
    )
]


print(
    "\n========================================"
)
print(
    "SELECTED DRIVER-SEASON ALPHA SENSITIVITY"
)
print(
    "========================================"
)

display(
    s30_season_sensitivity
    .sort_values(
        [
            "driver",
            "year",
            "variant",
        ]
    )
)


# ==========================================================
# 10. CONSTRUCTOR-SEASON SENSITIVITY
# ==========================================================

def s30_constructor_id_from_name(
    constructor_name,
):
    if (
        "raw"
        not in globals()
        or
        not hasattr(
            raw,
            "constructors",
        )
    ):
        raise RuntimeError(
            "raw.constructors is required for constructor lookup."
        )

    df = raw.constructors.copy()

    if "name" not in df.columns:
        raise RuntimeError(
            "raw.constructors has no `name` column."
        )

    matches = df[
        df[
            "name"
        ]
        .astype(str)
        .str.lower()
        ==
        str(
            constructor_name
        ).lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Constructor lookup for {constructor_name!r} "
            f"returned {len(matches)} rows."
        )

    return int(
        matches.iloc[
            0
        ][
            "constructorId"
        ]
    )


selected_constructor_years = [
    ("Red Bull", 2021),
    ("Red Bull", 2022),
    ("Red Bull", 2023),
    ("Red Bull", 2024),

    ("Mercedes", 2017),

    ("Ferrari", 2000),
]


constructor_rows = []


for constructor_name, year in selected_constructor_years:

    constructor_id = (
        s30_constructor_id_from_name(
            constructor_name
        )
    )

    csy_key = (
        constructor_id,
        year,
    )

    if csy_key not in idx_all.csy_map:
        continue

    csy = idx_all.csy_map[
        csy_key
    ]

    for variant, posterior in s30_posteriors.items():

        car_draws = s30_stack_state(
            posterior,
            "car_csy",
        )[
            :,
            csy,
        ]

        constructor_rows.append(
            {
                "variant":
                    variant,
                "constructor":
                    constructor_name,
                "year":
                    year,
                "beta_mean":
                    float(
                        car_draws.mean()
                    ),
                "beta_sd":
                    float(
                        car_draws.std(
                            ddof=1
                        )
                    ),
                "beta_89_lb":
                    float(
                        np.quantile(
                            car_draws,
                            0.055,
                        )
                    ),
                "beta_89_ub":
                    float(
                        np.quantile(
                            car_draws,
                            0.945,
                        )
                    ),
            }
        )


s30_constructor_sensitivity = pd.DataFrame(
    constructor_rows
)


frozen_beta_lookup = (
    s30_constructor_sensitivity[
        s30_constructor_sensitivity[
            "variant"
        ]
        ==
        "frozen"
    ]
    .set_index(
        [
            "constructor",
            "year",
        ]
    )[
        "beta_mean"
    ]
    .to_dict()
)


s30_constructor_sensitivity[
    "delta_beta_vs_frozen"
] = [
    row.beta_mean
    -
    frozen_beta_lookup[
        (
            row.constructor,
            row.year,
        )
    ]
    for row
    in s30_constructor_sensitivity.itertuples(
        index=False
    )
]


print(
    "\n========================================"
)
print(
    "SELECTED CONSTRUCTOR-SEASON BETA SENSITIVITY"
)
print(
    "========================================"
)

display(
    s30_constructor_sensitivity
    .sort_values(
        [
            "constructor",
            "year",
            "variant",
        ]
    )
)


# ==========================================================
# 11. RED BULL 2022 ALLOCATION RIDGE UNDER EACH PRIOR
# ==========================================================

red_bull_id = s30_constructor_id_from_name(
    "Red Bull"
)

rb22_csy = idx_all.csy_map[
    (
        red_bull_id,
        2022,
    )
]

perez22_dy = idx_all.dy_map[
    (
        "perez",
        2022,
    )
]

max22_dy = idx_all.dy_map[
    (
        "max_verstappen",
        2022,
    )
]


rb22_rows = []


for variant, posterior in s30_posteriors.items():

    alpha = s30_stack_state(
        posterior,
        "alpha_dy",
    )

    car = s30_stack_state(
        posterior,
        "car_csy",
    )

    perez = alpha[
        :,
        perez22_dy,
    ]

    maxv = alpha[
        :,
        max22_dy,
    ]

    beta = car[
        :,
        rb22_csy,
    ]

    common = (
        0.5
        *
        (
            perez
            +
            maxv
        )
    )

    total = (
        common
        +
        beta
    )

    delta = (
        perez
        -
        maxv
    )

    rb22_rows.append(
        {
            "variant":
                variant,

            "perez_alpha":
                float(
                    perez.mean()
                ),

            "max_alpha":
                float(
                    maxv.mean()
                ),

            "delta_alpha":
                float(
                    delta.mean()
                ),

            "red_bull_beta":
                float(
                    beta.mean()
                ),

            "common_driver_level":
                float(
                    common.mean()
                ),

            "driver_plus_car_total":
                float(
                    total.mean()
                ),

            "corr_beta_common_driver":
                float(
                    np.corrcoef(
                        beta,
                        common,
                    )[
                        0,
                        1,
                    ]
                ),

            "sd_perez":
                float(
                    perez.std(
                        ddof=1
                    )
                ),

            "sd_max":
                float(
                    maxv.std(
                        ddof=1
                    )
                ),

            "sd_common_driver":
                float(
                    common.std(
                        ddof=1
                    )
                ),

            "sd_beta":
                float(
                    beta.std(
                        ddof=1
                    )
                ),

            "sd_total":
                float(
                    total.std(
                        ddof=1
                    )
                ),
        }
    )


s30_rb22 = pd.DataFrame(
    rb22_rows
)


frozen_rb = (
    s30_rb22[
        s30_rb22[
            "variant"
        ]
        ==
        "frozen"
    ]
    .iloc[
        0
    ]
)


for col in [
    "perez_alpha",
    "max_alpha",
    "delta_alpha",
    "red_bull_beta",
    "common_driver_level",
    "driver_plus_car_total",
]:

    s30_rb22[
        f"delta_{col}_vs_frozen"
    ] = (
        s30_rb22[
            col
        ]
        -
        float(
            frozen_rb[
                col
            ]
        )
    )


print(
    "\n========================================"
)
print(
    "RED BULL 2022 — PRIOR SENSITIVITY OF THE ALLOCATION RIDGE"
)
print(
    "========================================"
)

display(
    s30_rb22
)


# ==========================================================
# 12. OBSERVABLE MODERN PREDICTION SENSITIVITY
# ==========================================================
#
# If alpha and beta move along the allocation ridge while these
# predictions barely change, that is exactly the prior-sensitivity
# mechanism we are looking for.
# ==========================================================

def s30_prediction_vectors(
    posterior,
):
    alpha = s30_stack_state(
        posterior,
        "alpha_dy",
    ).mean(
        axis=0
    )

    car = s30_stack_state(
        posterior,
        "car_csy",
    ).mean(
        axis=0
    )

    gamma = float(
        s30_stack_scalar(
            posterior,
            "gamma_tenure",
        ).mean()
    )

    out = {}

    # Pace teammate contrast.
    out[
        "pace_contrast"
    ] = (
        alpha[
            pace_contrast_all[
                "dy_a"
            ].to_numpy(
                dtype=int
            )
        ]
        -
        alpha[
            pace_contrast_all[
                "dy_b"
            ].to_numpy(
                dtype=int
            )
        ]
        +
        gamma
        *
        pace_contrast_all[
            "tenure_diff"
        ].to_numpy(
            dtype=float
        )
    )

    # Pace team level.
    out[
        "pace_level"
    ] = (
        0.5
        *
        (
            alpha[
                pace_level_all[
                    "dy_a"
                ].to_numpy(
                    dtype=int
                )
            ]
            +
            alpha[
                pace_level_all[
                    "dy_b"
                ].to_numpy(
                    dtype=int
                )
            ]
        )
        +
        car[
            pace_level_all[
                "csy"
            ].to_numpy(
                dtype=int
            )
        ]
        +
        gamma
        *
        pace_level_all[
            "tenure_mean"
        ].to_numpy(
            dtype=float
        )
    )

    # Qualifying teammate contrast.
    out[
        "q_contrast"
    ] = (
        alpha[
            q_contrast_all[
                "dy_a"
            ].to_numpy(
                dtype=int
            )
        ]
        -
        alpha[
            q_contrast_all[
                "dy_b"
            ].to_numpy(
                dtype=int
            )
        ]
        +
        gamma
        *
        q_contrast_all[
            "tenure_diff"
        ].to_numpy(
            dtype=float
        )
    )

    # Q1 team level.
    out[
        "q_level"
    ] = (
        0.5
        *
        (
            alpha[
                q_level_all_model[
                    "dy_a"
                ].to_numpy(
                    dtype=int
                )
            ]
            +
            alpha[
                q_level_all_model[
                    "dy_b"
                ].to_numpy(
                    dtype=int
                )
            ]
        )
        +
        car[
            q_level_all_model[
                "csy"
            ].to_numpy(
                dtype=int
            )
        ]
        +
        gamma
        *
        q_level_all_model[
            "tenure_mean"
        ].to_numpy(
            dtype=float
        )
    )

    return out


s30_predictions = {
    variant:
        s30_prediction_vectors(
            posterior
        )
    for variant, posterior
    in s30_posteriors.items()
}


s30_observed = {
    "pace_contrast":
        pace_contrast_all[
            "y"
        ].to_numpy(
            dtype=float
        ),

    "pace_level":
        pace_level_all[
            "y"
        ].to_numpy(
            dtype=float
        ),

    "q_contrast":
        q_contrast_all[
            "y"
        ].to_numpy(
            dtype=float
        ),

    "q_level":
        q_level_all_model[
            "y"
        ].to_numpy(
            dtype=float
        ),
}


prediction_rows = []


for variant, pred_dict in s30_predictions.items():

    for channel, mu in pred_dict.items():

        y = s30_observed[
            channel
        ]

        frozen_mu = s30_predictions[
            "frozen"
        ][
            channel
        ]

        prediction_rows.append(
            {
                "variant":
                    variant,
                "channel":
                    channel,
                "rows":
                    int(
                        len(
                            y
                        )
                    ),

                "rmse_to_observed":
                    float(
                        np.sqrt(
                            np.mean(
                                (
                                    y
                                    -
                                    mu
                                )
                                **
                                2
                            )
                        )
                    ),

                "mae_to_observed":
                    float(
                        np.mean(
                            np.abs(
                                y
                                -
                                mu
                            )
                        )
                    ),

                "rmse_prediction_change_vs_frozen":
                    float(
                        np.sqrt(
                            np.mean(
                                (
                                    mu
                                    -
                                    frozen_mu
                                )
                                **
                                2
                            )
                        )
                    ),

                "max_abs_prediction_change_vs_frozen":
                    float(
                        np.max(
                            np.abs(
                                mu
                                -
                                frozen_mu
                            )
                        )
                    ),
            }
        )


s30_prediction_sensitivity = pd.DataFrame(
    prediction_rows
)


print(
    "\n========================================"
)
print(
    "MODERN OBSERVABLE-PREDICTION SENSITIVITY"
)
print(
    "========================================"
)

display(
    s30_prediction_sensitivity
)


# ==========================================================
# 13. JAWS / RANK SENSITIVITY
# ==========================================================

frozen_table = (
    s30_scores[
        "frozen"
    ][
        "table"
    ]
    .set_index(
        "driver"
    )
)


jaws_compare_frames = []


for variant, scored in s30_scores.items():

    table = (
        scored[
            "table"
        ]
        .set_index(
            "driver"
        )
    )

    compare = pd.DataFrame(
        {
            "driver":
                eligible_drivers,

            "variant":
                variant,

            "jaws_mean":
                [
                    float(
                        table.loc[
                            driver,
                            "jaws_mean"
                        ]
                    )
                    for driver
                    in eligible_drivers
                ],

            "point_rank":
                [
                    int(
                        table.loc[
                            driver,
                            "point_rank"
                        ]
                    )
                    for driver
                    in eligible_drivers
                ],

            "rank_mean":
                [
                    float(
                        table.loc[
                            driver,
                            "rank_mean"
                        ]
                    )
                    for driver
                    in eligible_drivers
                ],
        }
    )

    compare[
        "frozen_jaws_mean"
    ] = [
        float(
            frozen_table.loc[
                driver,
                "jaws_mean"
            ]
        )
        for driver
        in eligible_drivers
    ]

    compare[
        "frozen_point_rank"
    ] = [
        int(
            frozen_table.loc[
                driver,
                "point_rank"
            ]
        )
        for driver
        in eligible_drivers
    ]

    compare[
        "frozen_rank_mean"
    ] = [
        float(
            frozen_table.loc[
                driver,
                "rank_mean"
            ]
        )
        for driver
        in eligible_drivers
    ]

    compare[
        "delta_jaws"
    ] = (
        compare[
            "jaws_mean"
        ]
        -
        compare[
            "frozen_jaws_mean"
        ]
    )

    compare[
        "point_rank_move"
    ] = (
        compare[
            "frozen_point_rank"
        ]
        -
        compare[
            "point_rank"
        ]
    )

    compare[
        "delta_rank_mean"
    ] = (
        compare[
            "rank_mean"
        ]
        -
        compare[
            "frozen_rank_mean"
        ]
    )

    jaws_compare_frames.append(
        compare
    )


s30_jaws_compare = pd.concat(
    jaws_compare_frames,
    ignore_index=True,
)


print(
    "\n========================================"
)
print(
    "JAWS / RANK SENSITIVITY — SELECTED DRIVERS"
)
print(
    "========================================"
)


selected_jaws_drivers = [
    "perez",
    "max_verstappen",
    "bottas",
    "hamilton",
    "barrichello",
    "michael_schumacher",
    "irvine",
    "webber",
    "alonso",
    "senna",
]


display(
    s30_jaws_compare[
        s30_jaws_compare[
            "driver"
        ].isin(
            selected_jaws_drivers
        )
    ]
    .sort_values(
        [
            "driver",
            "variant",
        ]
    )
)


print(
    "\nLargest absolute JAWS changes:"
)

display(
    s30_jaws_compare[
        s30_jaws_compare[
            "variant"
        ]
        !=
        "frozen"
    ]
    .assign(
        abs_delta_jaws=
            lambda x:
                x[
                    "delta_jaws"
                ].abs()
    )
    .sort_values(
        "abs_delta_jaws",
        ascending=False,
    )
    .head(
        30
    )
)


print(
    "\nLargest point-rank movements:"
)

display(
    s30_jaws_compare[
        s30_jaws_compare[
            "variant"
        ]
        !=
        "frozen"
    ]
    .assign(
        abs_rank_move=
            lambda x:
                x[
                    "point_rank_move"
                ].abs()
    )
    .sort_values(
        [
            "abs_rank_move",
            "delta_jaws",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(
        30
    )
)


# ==========================================================
# 14. WHOLE-LEADERBOARD STABILITY SUMMARY
# ==========================================================

stability_rows = []


frozen_rank_by_driver = (
    frozen_table[
        "point_rank"
    ]
    .to_dict()
)


frozen_jaws_by_driver = (
    frozen_table[
        "jaws_mean"
    ]
    .to_dict()
)


for variant in S30_VARIANTS:

    table = s30_scores[
        variant
    ][
        "table"
    ]

    rank_variant = (
        table
        .set_index(
            "driver"
        )[
            "point_rank"
        ]
        .to_dict()
    )

    jaws_variant = (
        table
        .set_index(
            "driver"
        )[
            "jaws_mean"
        ]
        .to_dict()
    )

    rank_moves = np.asarray(
        [
            rank_variant[
                driver
            ]
            -
            frozen_rank_by_driver[
                driver
            ]
            for driver
            in eligible_drivers
        ],
        dtype=float,
    )

    jaws_changes = np.asarray(
        [
            jaws_variant[
                driver
            ]
            -
            frozen_jaws_by_driver[
                driver
            ]
            for driver
            in eligible_drivers
        ],
        dtype=float,
    )

    frozen_rank_vec = np.asarray(
        [
            frozen_rank_by_driver[
                driver
            ]
            for driver
            in eligible_drivers
        ],
        dtype=float,
    )

    variant_rank_vec = np.asarray(
        [
            rank_variant[
                driver
            ]
            for driver
            in eligible_drivers
        ],
        dtype=float,
    )

    frozen_top10 = set(
        frozen_table.sort_values(
            "point_rank"
        )
        .head(
            10
        )
        .index
    )

    variant_top10 = set(
        table.sort_values(
            "point_rank"
        )
        .head(
            10
        )[
            "driver"
        ]
    )

    frozen_top30 = set(
        frozen_table.sort_values(
            "point_rank"
        )
        .head(
            30
        )
        .index
    )

    variant_top30 = set(
        table.sort_values(
            "point_rank"
        )
        .head(
            30
        )[
            "driver"
        ]
    )

    stability_rows.append(
        {
            "variant":
                variant,

            "spearman_point_rank":
                float(
                    st.spearmanr(
                        frozen_rank_vec,
                        variant_rank_vec,
                    ).statistic
                ),

            "mean_abs_point_rank_move":
                float(
                    np.mean(
                        np.abs(
                            rank_moves
                        )
                    )
                ),

            "max_abs_point_rank_move":
                int(
                    np.max(
                        np.abs(
                            rank_moves
                        )
                    )
                ),

            "mean_abs_jaws_change":
                float(
                    np.mean(
                        np.abs(
                            jaws_changes
                        )
                    )
                ),

            "max_abs_jaws_change":
                float(
                    np.max(
                        np.abs(
                            jaws_changes
                        )
                    )
                ),

            "top10_membership_overlap":
                int(
                    len(
                        frozen_top10
                        &
                        variant_top10
                    )
                ),

            "top30_membership_overlap":
                int(
                    len(
                        frozen_top30
                        &
                        variant_top30
                    )
                ),
        }
    )


s30_stability = pd.DataFrame(
    stability_rows
)


print(
    "\n========================================"
)
print(
    "WHOLE-LEADERBOARD STABILITY"
)
print(
    "========================================"
)

display(
    s30_stability
)


# ==========================================================
# 15. PLOTS
# ==========================================================

variant_order = [
    "frozen",
    "car_wide",
    "car_tight",
    "driver_tight",
]


# ----------------------------------------------------------
# A. Pérez 2020–2024 alpha under each prior
# ----------------------------------------------------------

perez_plot = s30_season_sensitivity[
    (
        s30_season_sensitivity[
            "driver"
        ]
        ==
        "perez"
    )
    &
    (
        s30_season_sensitivity[
            "year"
        ].between(
            2020,
            2024,
        )
    )
].copy()


plt.figure(
    figsize=(
        10,
        6,
    )
)


for variant in variant_order:

    x = perez_plot[
        perez_plot[
            "variant"
        ]
        ==
        variant
    ].sort_values(
        "year"
    )

    if x.empty:
        continue

    plt.plot(
        x[
            "year"
        ],
        x[
            "alpha_mean"
        ],
        marker="o",
        label=
            variant,
    )


plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.xlabel(
    "Season"
)

plt.ylabel(
    "Posterior mean alpha"
)

plt.title(
    "Pérez alpha sensitivity to allocation priors"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# B. Red Bull 2022 alpha / beta decomposition
# ----------------------------------------------------------

rb_plot = (
    s30_rb22
    .set_index(
        "variant"
    )
    .loc[
        [
            v
            for v
            in variant_order
            if v
            in s30_rb22[
                "variant"
            ].values
        ]
    ]
    .reset_index()
)


x = np.arange(
    len(
        rb_plot
    )
)


plt.figure(
    figsize=(
        10,
        6,
    )
)

plt.plot(
    x,
    rb_plot[
        "perez_alpha"
    ],
    marker="o",
    label=
        "Pérez alpha",
)

plt.plot(
    x,
    rb_plot[
        "max_alpha"
    ],
    marker="o",
    label=
        "Verstappen alpha",
)

plt.plot(
    x,
    rb_plot[
        "red_bull_beta"
    ],
    marker="s",
    label=
        "Red Bull beta",
)

plt.plot(
    x,
    rb_plot[
        "driver_plus_car_total"
    ],
    marker="^",
    label=
        "Common driver + car total",
)

plt.xticks(
    x,
    rb_plot[
        "variant"
    ],
    rotation=20,
    ha="right",
)

plt.ylabel(
    "Latent model units"
)

plt.title(
    "Red Bull 2022 allocation under prior sensitivity fits"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# C. Pérez JAWS sensitivity
# ----------------------------------------------------------

perez_jaws = (
    s30_jaws_compare[
        s30_jaws_compare[
            "driver"
        ]
        ==
        "perez"
    ]
    .copy()
)


perez_jaws[
    "_order"
] = perez_jaws[
    "variant"
].map(
    {
        v:
            i
        for i, v
        in enumerate(
            variant_order
        )
    }
)


perez_jaws = perez_jaws.sort_values(
    "_order"
)


plt.figure(
    figsize=(
        9,
        5,
    )
)

plt.plot(
    perez_jaws[
        "variant"
    ],
    perez_jaws[
        "jaws_mean"
    ],
    marker="o",
)

plt.ylabel(
    "Mean JAWS"
)

plt.title(
    "Pérez career score sensitivity"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# D. JAWS frozen vs each sensitivity variant
# ----------------------------------------------------------

for variant in S30_VARIANTS:

    table = (
        s30_jaws_compare[
            s30_jaws_compare[
                "variant"
            ]
            ==
            variant
        ]
    )

    plt.figure(
        figsize=(
            7,
            7,
        )
    )

    plt.scatter(
        table[
            "frozen_jaws_mean"
        ],
        table[
            "jaws_mean"
        ],
        alpha=0.65,
    )

    lo = float(
        min(
            table[
                "frozen_jaws_mean"
            ].min(),
            table[
                "jaws_mean"
            ].min(),
        )
    )

    hi = float(
        max(
            table[
                "frozen_jaws_mean"
            ].max(),
            table[
                "jaws_mean"
            ].max(),
        )
    )

    plt.plot(
        [
            lo,
            hi,
        ],
        [
            lo,
            hi,
        ],
        linestyle="--",
        linewidth=1.0,
    )

    plt.xlabel(
        "Frozen JAWS mean"
    )

    plt.ylabel(
        f"{variant} JAWS mean"
    )

    plt.title(
        f"JAWS sensitivity — {variant}"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()


# ==========================================================
# 16. SAVE OUTPUTS
# ==========================================================

s30_health.to_csv(
    S30_OUTPUT_DIR
    /
    "sensitivity_fit_health.csv",
    index=False,
)


s30_global_summary.to_csv(
    S30_OUTPUT_DIR
    /
    "global_parameter_sensitivity.csv",
    index=False,
)


s30_season_sensitivity.to_csv(
    S30_OUTPUT_DIR
    /
    "selected_driver_season_alpha_sensitivity.csv",
    index=False,
)


s30_constructor_sensitivity.to_csv(
    S30_OUTPUT_DIR
    /
    "selected_constructor_beta_sensitivity.csv",
    index=False,
)


s30_rb22.to_csv(
    S30_OUTPUT_DIR
    /
    "red_bull_2022_allocation_sensitivity.csv",
    index=False,
)


s30_prediction_sensitivity.to_csv(
    S30_OUTPUT_DIR
    /
    "modern_prediction_sensitivity.csv",
    index=False,
)


s30_jaws_compare.to_csv(
    S30_OUTPUT_DIR
    /
    "jaws_rank_sensitivity_all_drivers.csv",
    index=False,
)


s30_stability.to_csv(
    S30_OUTPUT_DIR
    /
    "leaderboard_stability_summary.csv",
    index=False,
)


for variant, scored in s30_scores.items():

    scored[
        "table"
    ].to_csv(
        S30_OUTPUT_DIR
        /
        f"leaderboard_{variant}.csv",
        index=False,
    )


# ==========================================================
# 17. FINAL TRIAGE
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 30 TRIAGE SUMMARY"
)
print(
    "========================================"
)


print(
    "\nRed Bull 2022:"
)


triage_cols = [
    "variant",
    "perez_alpha",
    "max_alpha",
    "delta_alpha",
    "red_bull_beta",
    "common_driver_level",
    "driver_plus_car_total",
    "corr_beta_common_driver",
    "sd_common_driver",
    "sd_beta",
    "sd_total",
    "delta_perez_alpha_vs_frozen",
    "delta_max_alpha_vs_frozen",
    "delta_red_bull_beta_vs_frozen",
]


display(
    s30_rb22[
        triage_cols
    ]
)


print(
    "\nPérez JAWS:"
)


display(
    s30_jaws_compare[
        s30_jaws_compare[
            "driver"
        ]
        ==
        "perez"
    ][
        [
            "variant",
            "jaws_mean",
            "delta_jaws",
            "point_rank",
            "frozen_point_rank",
            "point_rank_move",
            "rank_mean",
            "delta_rank_mean",
        ]
    ]
)


print(
    "\nLeaderboard stability:"
)

display(
    s30_stability
)


print(
    "\nDecision guide:"
)

print(
    "  ROBUST allocation:"
)

print(
    "    - car_wide / car_tight barely move Perez 2022 alpha;"
)

print(
    "    - Red Bull beta barely moves;"
)

print(
    "    - teammate delta remains essentially unchanged;"
)

print(
    "    - JAWS/ranks are highly stable."
)

print(
    "\n  CAR-PRIOR SENSITIVE allocation:"
)

print(
    "    - car_wide pushes beta materially upward and both driver"
)

print(
    "      alphas downward while delta_alpha stays stable;"
)

print(
    "    - car_tight does the reverse;"
)

print(
    "    - modern observable predictions barely change."
)

print(
    "\n  DRIVER-PRIOR SENSITIVE allocation:"
)

print(
    "    - driver_tight materially changes common driver alpha / beta"
)

print(
    "      while teammate differences and predictions remain stable."
)

print(
    "\n  The key distinction is RELATIVE vs ABSOLUTE robustness:"
)

print(
    "    stable delta_alpha + unstable absolute alpha means the model"
)

print(
    "    is trustworthy about who beat whom, but the absolute vertical"
)

print(
    "    driver/car split is assumption-sensitive."
)

print(
    "\n  If all three sensitivity fits leave Perez / Max / Red Bull and"
)

print(
    "  the Pantheon leaderboard broadly unchanged, the strange Perez"
)

print(
    "  trajectory is much harder to dismiss as prior-driven allocation."
)


print(
    "\nSaved outputs ->"
)

print(
    S30_OUTPUT_DIR.resolve()
)

print(
    "\nStage 30 complete."
)


In [ ]:
# ==========================================================
# STAGE 31 — PÉREZ / VERSTAPPEN 2021–22 EVIDENCE-ABLATION AUDIT
# ==========================================================
#
# PURPOSE
# -------
# Stage 29 showed a strong driver/car allocation ridge.
# Stage 30 showed that Pérez's Red Bull peak is NOT materially driven
# by plausible changes to the driver/car prior scales.
#
# Stage 31 now asks the more direct explanatory question:
#
#     WHAT EVIDENCE is actually holding Pérez at ~+0.26/+0.30
#     in 2021–22 despite the very negative Verstappen teammate
#     contrasts?
#
# We fit three deliberately targeted ablations, always keeping the
# frozen model structure and priors unchanged.
#
# ----------------------------------------------------------
# A. NO_RB_TEAM_LEVEL_21_22
# ----------------------------------------------------------
# Remove Red Bull 2021–22 absolute/team-level evidence only:
#
#   - pace_level rows
#   - Q1 team-level rows
#
# Keep ALL teammate contrasts, including Pérez vs Verstappen.
#
# Interpretation:
#   If Pérez alpha FALLS when this evidence is removed, Red Bull's
#   absolute team performance had been pulling the common driver
#   level upward.
#
# ----------------------------------------------------------
# B. NO_PV_CONTRAST_21_22
# ----------------------------------------------------------
# Remove Pérez-vs-Verstappen 2021–22 teammate contrasts only:
#
#   - race-pace contrasts
#   - all qualifying-session contrasts
#
# Keep Red Bull team-level evidence.
#
# Interpretation:
#   If Pérez alpha RISES when these contrasts are removed, the direct
#   Verstappen comparison is pulling him downward, as expected.
#
# ----------------------------------------------------------
# C. NO_PV_ALL_DIRECT_21_22
# ----------------------------------------------------------
# Remove ALL modern observations involving the Pérez 2021/2022 or
# Verstappen 2021/2022 driver-season states from all four channels.
#
# The latent states remain in the model and are predicted from:
#
#   - their surrounding seasons,
#   - the rest of their teammate networks,
#   - the state-space prior,
#   - global centring / population structure.
#
# This is the "career/network expectation before seeing those
# seasons' direct modern evidence" benchmark.
#
# ----------------------------------------------------------
# IMPORTANT
# ----------------------------------------------------------
# These are EXPLANATORY ABLATION fits, not candidate production models.
# The production posterior is never overwritten.
#
# The combined index / centring matrices remain the full-data versions.
# As in our earlier holdout validation, they contain participation /
# reliability structure but no held-out outcome values.
#
# ==========================================================
# COMPUTATION
# ==========================================================
#
# Three refits:
#   2 chains x (500 tune + 500 posterior)
#   nutpie / JAX
#   target_accept = 0.92
#
# Every fit is independently checkpointed.
# Existing valid checkpoints are loaded rather than resampled.
# ==========================================================

from pathlib import Path
import time

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import xarray as xr


# ==========================================================
# 0. CONFIGURATION
# ==========================================================

S31_YEARS = (2021, 2022)

S31_DRIVER_A = "perez"
S31_DRIVER_B = "max_verstappen"
S31_CONSTRUCTOR_NAME = "Red Bull"

S31_CHAINS = 2
S31_TUNE = 500
S31_DRAWS = 500
S31_TARGET_ACCEPT = 0.92

S31_SEEDS = {
    "no_rb_team_level_21_22":
        31041,

    "no_pv_contrast_21_22":
        31042,

    "no_pv_all_direct_21_22":
        31043,
}

S31_OUTPUT_DIR = (
    Path(CFG.output_dir)
    /
    "validation_perez_verstappen_evidence_ablation"
)

S31_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

S31_CHECKPOINT_ENGINE = "h5netcdf"


# ==========================================================
# 1. REQUIRED OBJECTS
# ==========================================================

required_objects = [
    "CFG",
    "idx_all",
    "build_joint_model_1982_2025",
    "final_trace_2025",

    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",

    "hist_q_pack",
    "hist_race_pack",

    "raw",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Stage 31 requires the model-building environment. Missing:\n  "
        + "\n  ".join(
            missing
        )
    )


# ==========================================================
# 2. CHECKPOINT BACKEND
# ==========================================================

try:
    import h5netcdf  # noqa: F401
    import h5py      # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Stage 31 requires h5netcdf + h5py for grouped checkpoints.\n\n"
        "Install with:\n"
        "    %pip install h5netcdf h5py"
    ) from exc


# ==========================================================
# 3. IDENTIFY RED BULL + TARGET DRIVER-SEASON STATES
# ==========================================================

constructors = raw.constructors.copy()

if "constructorId" not in constructors.columns:
    raise RuntimeError(
        "raw.constructors has no constructorId column."
    )

constructor_name_col = None

for candidate in [
    "name",
    "constructorRef",
]:
    if candidate in constructors.columns:
        constructor_name_col = candidate
        break

if constructor_name_col is None:
    raise RuntimeError(
        "Could not find constructor name column."
    )


rb_matches = constructors[
    constructors[
        constructor_name_col
    ]
    .astype(str)
    .str.lower()
    ==
    S31_CONSTRUCTOR_NAME.lower()
]


if len(rb_matches) != 1:
    raise RuntimeError(
        f"Constructor lookup for {S31_CONSTRUCTOR_NAME!r} "
        f"returned {len(rb_matches)} rows."
    )


S31_RB_ID = int(
    rb_matches.iloc[
        0
    ][
        "constructorId"
    ]
)


S31_TARGET_DY = set()

for driver in [
    S31_DRIVER_A,
    S31_DRIVER_B,
]:
    for year in S31_YEARS:

        key = (
            driver,
            int(year),
        )

        if key not in idx_all.dy_map:
            raise RuntimeError(
                f"Missing driver-season state {key}"
            )

        S31_TARGET_DY.add(
            int(
                idx_all.dy_map[
                    key
                ]
            )
        )


print(
    "========================================"
)
print(
    "STAGE 31 — PÉREZ / VERSTAPPEN EVIDENCE ABLATION"
)
print(
    "========================================"
)

print(
    "Constructor:",
    S31_CONSTRUCTOR_NAME,
    "| constructorId =",
    S31_RB_ID,
)

print(
    "Years:",
    S31_YEARS,
)

print(
    "Target driver-season indices:",
    sorted(
        S31_TARGET_DY
    ),
)


# ==========================================================
# 4. FILTER HELPERS
# ==========================================================

def s31_is_rb_21_22(
    frame,
):
    return (
        frame[
            "year"
        ].astype(int)
        .isin(
            S31_YEARS
        )
        &
        (
            frame[
                "constructorId"
            ].astype(int)
            ==
            S31_RB_ID
        )
    )


def s31_is_exact_pv_pair(
    frame,
):
    a = frame[
        "driver_a"
    ].astype(str)

    b = frame[
        "driver_b"
    ].astype(str)

    return (
        (
            (a == S31_DRIVER_A)
            &
            (b == S31_DRIVER_B)
        )
        |
        (
            (a == S31_DRIVER_B)
            &
            (b == S31_DRIVER_A)
        )
    )


def s31_involves_target_state(
    frame,
):
    return (
        frame[
            "dy_a"
        ].astype(int)
        .isin(
            S31_TARGET_DY
        )
        |
        frame[
            "dy_b"
        ].astype(int)
        .isin(
            S31_TARGET_DY
        )
    )


# ==========================================================
# 5. DEFINE THE THREE ABLATIONS
# ==========================================================

# ----------------------------------------------------------
# A. Remove Red Bull 2021–22 TEAM-LEVEL evidence only.
# ----------------------------------------------------------

mask_a_pc = np.zeros(
    len(
        pace_contrast_all
    ),
    dtype=bool,
)

mask_a_pl = s31_is_rb_21_22(
    pace_level_all
).to_numpy(
    dtype=bool
)

mask_a_qc = np.zeros(
    len(
        q_contrast_all
    ),
    dtype=bool,
)

mask_a_ql = s31_is_rb_21_22(
    q_level_all_model
).to_numpy(
    dtype=bool
)


# ----------------------------------------------------------
# B. Remove Pérez-vs-Verstappen 2021–22 CONTRASTS only.
# ----------------------------------------------------------

mask_b_pc = (
    s31_is_rb_21_22(
        pace_contrast_all
    )
    &
    s31_is_exact_pv_pair(
        pace_contrast_all
    )
).to_numpy(
    dtype=bool
)

mask_b_pl = np.zeros(
    len(
        pace_level_all
    ),
    dtype=bool,
)

mask_b_qc = (
    s31_is_rb_21_22(
        q_contrast_all
    )
    &
    s31_is_exact_pv_pair(
        q_contrast_all
    )
).to_numpy(
    dtype=bool
)

mask_b_ql = np.zeros(
    len(
        q_level_all_model
    ),
    dtype=bool,
)


# ----------------------------------------------------------
# C. Remove ALL direct modern evidence involving either
#    Pérez 2021/22 or Verstappen 2021/22.
# ----------------------------------------------------------

mask_c_pc = s31_involves_target_state(
    pace_contrast_all
).to_numpy(
    dtype=bool
)

mask_c_pl = s31_involves_target_state(
    pace_level_all
).to_numpy(
    dtype=bool
)

mask_c_qc = s31_involves_target_state(
    q_contrast_all
).to_numpy(
    dtype=bool
)

mask_c_ql = s31_involves_target_state(
    q_level_all_model
).to_numpy(
    dtype=bool
)


S31_ABLATIONS = {
    "no_rb_team_level_21_22":
        {
            "label":
                "NO RB TEAM LEVEL 2021–22",

            "description":
                (
                    "remove Red Bull 2021–22 pace/Q1 team-level "
                    "evidence; retain Pérez–Verstappen contrasts"
                ),

            "masks":
                {
                    "pace_contrast":
                        mask_a_pc,
                    "pace_level":
                        mask_a_pl,
                    "q_contrast":
                        mask_a_qc,
                    "q_level":
                        mask_a_ql,
                },
        },

    "no_pv_contrast_21_22":
        {
            "label":
                "NO PÉREZ–VERSTAPPEN CONTRAST 2021–22",

            "description":
                (
                    "remove Pérez–Verstappen pace/quali contrasts; "
                    "retain Red Bull team-level evidence"
                ),

            "masks":
                {
                    "pace_contrast":
                        mask_b_pc,
                    "pace_level":
                        mask_b_pl,
                    "q_contrast":
                        mask_b_qc,
                    "q_level":
                        mask_b_ql,
                },
        },

    "no_pv_all_direct_21_22":
        {
            "label":
                "NO PÉREZ/MAX DIRECT EVIDENCE 2021–22",

            "description":
                (
                    "remove every modern observation involving "
                    "Pérez 2021/22 or Verstappen 2021/22"
                ),

            "masks":
                {
                    "pace_contrast":
                        mask_c_pc,
                    "pace_level":
                        mask_c_pl,
                    "q_contrast":
                        mask_c_qc,
                    "q_level":
                        mask_c_ql,
                },
        },
}


# ==========================================================
# 6. AUDIT EXACTLY WHAT WILL BE REMOVED
# ==========================================================

removal_rows = []


for key, spec in S31_ABLATIONS.items():

    masks = spec[
        "masks"
    ]

    removal_rows.append(
        {
            "ablation":
                key,
            "pace_contrast_removed":
                int(
                    masks[
                        "pace_contrast"
                    ].sum()
                ),
            "pace_level_removed":
                int(
                    masks[
                        "pace_level"
                    ].sum()
                ),
            "q_contrast_removed":
                int(
                    masks[
                        "q_contrast"
                    ].sum()
                ),
            "q_level_removed":
                int(
                    masks[
                        "q_level"
                    ].sum()
                ),
            "total_removed":
                int(
                    masks[
                        "pace_contrast"
                    ].sum()
                    +
                    masks[
                        "pace_level"
                    ].sum()
                    +
                    masks[
                        "q_contrast"
                    ].sum()
                    +
                    masks[
                        "q_level"
                    ].sum()
                ),
        }
    )


s31_removal_summary = pd.DataFrame(
    removal_rows
)


print(
    "\n========================================"
)
print(
    "ROWS REMOVED BY EACH ABLATION"
)
print(
    "========================================"
)

display(
    s31_removal_summary
)


# Sanity checks: the targeted ablations should actually remove data.
if int(
    mask_a_pl.sum()
    +
    mask_a_ql.sum()
) == 0:
    raise RuntimeError(
        "Team-level ablation removed zero rows."
    )

if int(
    mask_b_pc.sum()
    +
    mask_b_qc.sum()
) == 0:
    raise RuntimeError(
        "Pérez–Verstappen contrast ablation removed zero rows."
    )

if int(
    mask_c_pc.sum()
    +
    mask_c_pl.sum()
    +
    mask_c_qc.sum()
    +
    mask_c_ql.sum()
) == 0:
    raise RuntimeError(
        "All-direct ablation removed zero rows."
    )


# ==========================================================
# 7. BUILD TRAIN / HELD-OUT TABLES FOR EACH ABLATION
# ==========================================================

def s31_split_tables(
    masks,
):
    train = {
        "pace_contrast":
            pace_contrast_all.loc[
                ~masks[
                    "pace_contrast"
                ]
            ].copy(),

        "pace_level":
            pace_level_all.loc[
                ~masks[
                    "pace_level"
                ]
            ].copy(),

        "q_contrast":
            q_contrast_all.loc[
                ~masks[
                    "q_contrast"
                ]
            ].copy(),

        "q_level":
            q_level_all_model.loc[
                ~masks[
                    "q_level"
                ]
            ].copy(),
    }

    hold = {
        "pace_contrast":
            pace_contrast_all.loc[
                masks[
                    "pace_contrast"
                ]
            ].copy(),

        "pace_level":
            pace_level_all.loc[
                masks[
                    "pace_level"
                ]
            ].copy(),

        "q_contrast":
            q_contrast_all.loc[
                masks[
                    "q_contrast"
                ]
            ].copy(),

        "q_level":
            q_level_all_model.loc[
                masks[
                    "q_level"
                ]
            ].copy(),
    }

    return (
        train,
        hold,
    )


s31_data = {}


for key, spec in S31_ABLATIONS.items():

    train, hold = s31_split_tables(
        spec[
            "masks"
        ]
    )

    s31_data[
        key
    ] = {
        "train":
            train,
        "hold":
            hold,
    }


# ==========================================================
# 8. BUILD + SAMPLE / LOAD THE THREE REFITS
# ==========================================================

s31_traces = {}


for key, spec in S31_ABLATIONS.items():

    print(
        "\n========================================"
    )

    print(
        spec[
            "label"
        ]
    )

    print(
        spec[
            "description"
        ]
    )

    print(
        "========================================"
    )

    train = s31_data[
        key
    ][
        "train"
    ]

    hold = s31_data[
        key
    ][
        "hold"
    ]

    print(
        "TRAIN / HELD-OUT:"
    )

    for channel in [
        "pace_contrast",
        "pace_level",
        "q_contrast",
        "q_level",
    ]:

        print(
            f"  {channel:14s}: "
            f"{len(train[channel]):5d} train / "
            f"{len(hold[channel]):4d} removed"
        )

    trace_path = (
        S31_OUTPUT_DIR
        /
        (
            f"{key}_"
            f"{S31_TUNE}tune_"
            f"{S31_DRAWS}draw_"
            f"{S31_CHAINS}chain_"
            "nutpie_jax.nc"
        )
    )

    print(
        "\nCheckpoint ->"
    )

    print(
        trace_path
    )


    if trace_path.exists():

        print(
            "Existing checkpoint found."
        )

        try:

            trace = xr.open_datatree(
                trace_path,
                engine=
                    S31_CHECKPOINT_ENGINE,
            )

            print(
                "Loaded successfully."
            )

            print(
                "NO RESAMPLING performed."
            )

        except Exception as exc:

            raise RuntimeError(
                "A Stage-31 checkpoint exists but cannot be read.\n"
                "Do NOT overwrite it automatically.\n\n"
                f"{trace_path}\n\n"
                f"{repr(exc)}"
            ) from exc


    else:

        print(
            "\nBuilding ablation model..."
        )

        model = build_joint_model_1982_2025(
            train[
                "pace_contrast"
            ],
            train[
                "pace_level"
            ],
            train[
                "q_contrast"
            ],
            train[
                "q_level"
            ],
            hist_q_pack,
            hist_race_pack,
            idx_all,
            cfg=
                CFG,
        )

        print(
            f"Sampling {S31_CHAINS} chains x "
            f"({S31_TUNE} tune + {S31_DRAWS} posterior)"
        )

        print(
            "Sampler: nutpie / JAX"
        )

        t0 = time.perf_counter()

        with model:

            trace = pm.sample(
                draws=
                    S31_DRAWS,

                tune=
                    S31_TUNE,

                chains=
                    S31_CHAINS,

                random_seed=
                    S31_SEEDS[
                        key
                    ],

                target_accept=
                    S31_TARGET_ACCEPT,

                nuts_sampler=
                    "nutpie",

                nuts_sampler_kwargs={
                    "backend":
                        "jax",
                },

                return_inferencedata=
                    True,

                progressbar=
                    True,
            )

        elapsed = (
            time.perf_counter()
            -
            t0
        )

        print(
            "\nSampling wall time:",
            f"{elapsed / 60.0:.2f} min",
        )

        trace.to_netcdf(
            trace_path,
            engine=
                S31_CHECKPOINT_ENGINE,
            mode="w",
        )

        print(
            "Saved ->"
        )

        print(
            trace_path
        )

    s31_traces[
        key
    ] = trace


# ==========================================================
# 9. TRACE HELPERS
# ==========================================================

def s31_posterior_dataset(
    trace,
):
    if hasattr(
        trace,
        "children",
    ):
        return (
            trace[
                "posterior"
            ]
            .to_dataset()
        )

    if hasattr(
        trace,
        "posterior",
    ):
        return trace.posterior

    raise TypeError(
        "Could not locate posterior dataset."
    )


def s31_sample_stats_dataset(
    trace,
):
    if hasattr(
        trace,
        "children",
    ):
        return (
            trace[
                "sample_stats"
            ]
            .to_dataset()
        )

    if hasattr(
        trace,
        "sample_stats",
    ):
        return trace.sample_stats

    raise TypeError(
        "Could not locate sample_stats dataset."
    )


def s31_stack_state(
    posterior,
    var_name,
):
    arr = np.asarray(
        posterior[
            var_name
        ].values,
        dtype=float,
    )

    return arr.reshape(
        -1,
        arr.shape[-1],
    )


def s31_stack_scalar(
    posterior,
    var_name,
):
    return np.asarray(
        posterior[
            var_name
        ].values,
        dtype=float,
    ).reshape(-1)


s31_posteriors = {
    "frozen":
        s31_posterior_dataset(
            final_trace_2025
        )
}


for key, trace in s31_traces.items():

    s31_posteriors[
        key
    ] = s31_posterior_dataset(
        trace
    )


# ==========================================================
# 10. FIT HEALTH
# ==========================================================

S31_GLOBAL_VARS = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",

    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",

    "tau_hist_quali",
    "tau_hist_race",
]


health_rows = []


for key, trace in s31_traces.items():

    posterior = s31_posteriors[
        key
    ]

    sample_stats = s31_sample_stats_dataset(
        trace
    )

    diag = az.summary(
        posterior,
        var_names=
            S31_GLOBAL_VARS,
        kind=
            "diagnostics",
    )

    health_rows.append(
        {
            "ablation":
                key,

            "max_global_rhat":
                float(
                    diag[
                        "r_hat"
                    ].max()
                ),

            "min_global_ess_bulk":
                float(
                    diag[
                        "ess_bulk"
                    ].min()
                ),

            "divergences":
                int(
                    np.asarray(
                        sample_stats[
                            "diverging"
                        ].values
                    ).sum()
                ),
        }
    )


s31_health = pd.DataFrame(
    health_rows
)


print(
    "\n========================================"
)
print(
    "ABLATION-FIT HEALTH"
)
print(
    "========================================"
)

display(
    s31_health
)


# ==========================================================
# 11. RED BULL CONSTRUCTOR-SEASON IDS
# ==========================================================

s31_rb_csy = {}


for year in S31_YEARS:

    key = (
        S31_RB_ID,
        int(year),
    )

    if key not in idx_all.csy_map:
        raise RuntimeError(
            f"Missing constructor-season state {key}"
        )

    s31_rb_csy[
        int(year)
    ] = int(
        idx_all.csy_map[
            key
        ]
    )


# ==========================================================
# 12. CORE STATE DECOMPOSITION — 2021 + 2022
# ==========================================================

state_rows = []


for variant, posterior in s31_posteriors.items():

    alpha = s31_stack_state(
        posterior,
        "alpha_dy",
    )

    car = s31_stack_state(
        posterior,
        "car_csy",
    )

    for year in S31_YEARS:

        perez = alpha[
            :,
            idx_all.dy_map[
                (
                    S31_DRIVER_A,
                    int(year),
                )
            ],
        ]

        maxv = alpha[
            :,
            idx_all.dy_map[
                (
                    S31_DRIVER_B,
                    int(year),
                )
            ],
        ]

        beta = car[
            :,
            s31_rb_csy[
                int(year)
            ],
        ]

        delta = (
            perez
            -
            maxv
        )

        common = (
            0.5
            *
            (
                perez
                +
                maxv
            )
        )

        total = (
            common
            +
            beta
        )

        state_rows.append(
            {
                "variant":
                    variant,
                "year":
                    int(year),

                "perez_alpha":
                    float(
                        perez.mean()
                    ),
                "perez_alpha_sd":
                    float(
                        perez.std(
                            ddof=1
                        )
                    ),
                "perez_alpha_89_lb":
                    float(
                        np.quantile(
                            perez,
                            0.055,
                        )
                    ),
                "perez_alpha_89_ub":
                    float(
                        np.quantile(
                            perez,
                            0.945,
                        )
                    ),

                "max_alpha":
                    float(
                        maxv.mean()
                    ),
                "max_alpha_sd":
                    float(
                        maxv.std(
                            ddof=1
                        )
                    ),

                "delta_alpha":
                    float(
                        delta.mean()
                    ),
                "delta_alpha_sd":
                    float(
                        delta.std(
                            ddof=1
                        )
                    ),
                "delta_alpha_89_lb":
                    float(
                        np.quantile(
                            delta,
                            0.055,
                        )
                    ),
                "delta_alpha_89_ub":
                    float(
                        np.quantile(
                            delta,
                            0.945,
                        )
                    ),

                "red_bull_beta":
                    float(
                        beta.mean()
                    ),
                "red_bull_beta_sd":
                    float(
                        beta.std(
                            ddof=1
                        )
                    ),

                "common_driver_level":
                    float(
                        common.mean()
                    ),
                "common_driver_sd":
                    float(
                        common.std(
                            ddof=1
                        )
                    ),

                "driver_plus_car_total":
                    float(
                        total.mean()
                    ),
                "driver_plus_car_sd":
                    float(
                        total.std(
                            ddof=1
                        )
                    ),

                "corr_beta_common_driver":
                    float(
                        np.corrcoef(
                            beta,
                            common,
                        )[
                            0,
                            1,
                        ]
                    ),
            }
        )


s31_state = pd.DataFrame(
    state_rows
)


# Frozen lookup for delta columns.
frozen_state_lookup = (
    s31_state[
        s31_state[
            "variant"
        ]
        ==
        "frozen"
    ]
    .set_index(
        "year"
    )
)


for metric in [
    "perez_alpha",
    "max_alpha",
    "delta_alpha",
    "red_bull_beta",
    "common_driver_level",
    "driver_plus_car_total",
]:

    s31_state[
        f"delta_{metric}_vs_frozen"
    ] = [
        row[
            metric
        ]
        -
        float(
            frozen_state_lookup.loc[
                int(
                    row[
                        "year"
                    ]
                ),
                metric,
            ]
        )
        for _, row
        in s31_state.iterrows()
    ]


print(
    "\n========================================"
)
print(
    "RED BULL 2021–22 — STATE DECOMPOSITION"
)
print(
    "========================================"
)

display(
    s31_state.sort_values(
        [
            "year",
            "variant",
        ]
    )
)


# ==========================================================
# 13. FULL PÉREZ / MAX TRAJECTORY AROUND THE ABLATION
# ==========================================================

trajectory_rows = []


for variant, posterior in s31_posteriors.items():

    alpha = s31_stack_state(
        posterior,
        "alpha_dy",
    )

    for driver in [
        S31_DRIVER_A,
        S31_DRIVER_B,
    ]:

        for year in range(
            2019,
            2025,
        ):

            key = (
                driver,
                year,
            )

            if key not in idx_all.dy_map:
                continue

            draws = alpha[
                :,
                idx_all.dy_map[
                    key
                ],
            ]

            trajectory_rows.append(
                {
                    "variant":
                        variant,
                    "driver":
                        driver,
                    "year":
                        year,
                    "alpha_mean":
                        float(
                            draws.mean()
                        ),
                    "alpha_sd":
                        float(
                            draws.std(
                                ddof=1
                            )
                        ),
                    "alpha_89_lb":
                        float(
                            np.quantile(
                                draws,
                                0.055,
                            )
                        ),
                    "alpha_89_ub":
                        float(
                            np.quantile(
                                draws,
                                0.945,
                            )
                        ),
                }
            )


s31_trajectory = pd.DataFrame(
    trajectory_rows
)


# ==========================================================
# 14. POSTERIOR-MEAN PREDICTION OF THE REMOVED EVIDENCE
# ==========================================================
#
# This asks:
#
#   "If this evidence had NOT been used to estimate the states,
#    what would the remaining career/network evidence have predicted?"
#
# For each removed channel we report:
#
#   observed y
#   posterior-mean predicted mu
#   weighted residual
#   RMSE
#   mean |z| using the fitted Student-t scale
#
# ==========================================================

S31_SIGMA_VAR = {
    "pace_contrast":
        "sigma_pace_contrast",
    "pace_level":
        "sigma_pace_level",
    "q_contrast":
        "sigma_q_contrast",
    "q_level":
        "sigma_q_level",
}


def s31_predict_rows(
    posterior,
    frame,
    channel,
):
    if frame.empty:
        return {
            "mu_draws":
                np.empty(
                    (
                        0,
                        0,
                    )
                ),
            "mu_mean":
                np.asarray(
                    [],
                    dtype=float,
                ),
            "sigma_mean":
                np.nan,
        }

    alpha = s31_stack_state(
        posterior,
        "alpha_dy",
    )

    gamma = s31_stack_scalar(
        posterior,
        "gamma_tenure",
    )

    dy_a = frame[
        "dy_a"
    ].to_numpy(
        dtype=int
    )

    dy_b = frame[
        "dy_b"
    ].to_numpy(
        dtype=int
    )


    if channel in [
        "pace_contrast",
        "q_contrast",
    ]:

        tenure = frame[
            "tenure_diff"
        ].to_numpy(
            dtype=float
        )

        mu_draws = (
            alpha[
                :,
                dy_a,
            ]
            -
            alpha[
                :,
                dy_b,
            ]
            +
            gamma[
                :,
                None,
            ]
            *
            tenure[
                None,
                :
            ]
        )


    else:

        car = s31_stack_state(
            posterior,
            "car_csy",
        )

        csy = frame[
            "csy"
        ].to_numpy(
            dtype=int
        )

        tenure = frame[
            "tenure_mean"
        ].to_numpy(
            dtype=float
        )

        mu_draws = (
            0.5
            *
            (
                alpha[
                    :,
                    dy_a,
                ]
                +
                alpha[
                    :,
                    dy_b,
                ]
            )
            +
            car[
                :,
                csy,
            ]
            +
            gamma[
                :,
                None,
            ]
            *
            tenure[
                None,
                :
            ]
        )


    sigma_mean = float(
        s31_stack_scalar(
            posterior,
            S31_SIGMA_VAR[
                channel
            ],
        ).mean()
    )

    return {
        "mu_draws":
            mu_draws,
        "mu_mean":
            mu_draws.mean(
                axis=0
            ),
        "sigma_mean":
            sigma_mean,
    }


prediction_rows = []
prediction_detail_frames = []


for variant in S31_ABLATIONS:

    posterior = s31_posteriors[
        variant
    ]

    hold = s31_data[
        variant
    ][
        "hold"
    ]

    for channel in [
        "pace_contrast",
        "pace_level",
        "q_contrast",
        "q_level",
    ]:

        frame = hold[
            channel
        ].copy()

        if frame.empty:
            continue

        pred = s31_predict_rows(
            posterior,
            frame,
            channel,
        )

        y = frame[
            "y"
        ].to_numpy(
            dtype=float
        )

        w = frame[
            "weight"
        ].to_numpy(
            dtype=float
        )

        mu = pred[
            "mu_mean"
        ]

        residual = (
            y
            -
            mu
        )

        z = (
            residual
            *
            np.sqrt(
                w
            )
            /
            pred[
                "sigma_mean"
            ]
        )

        prediction_rows.append(
            {
                "ablation":
                    variant,
                "channel":
                    channel,
                "removed_rows":
                    int(
                        len(
                            frame
                        )
                    ),
                "observed_mean":
                    float(
                        np.average(
                            y,
                            weights=
                                w,
                        )
                    ),
                "predicted_mean":
                    float(
                        np.average(
                            mu,
                            weights=
                                w,
                        )
                    ),
                "weighted_mean_residual":
                    float(
                        np.average(
                            residual,
                            weights=
                                w,
                        )
                    ),
                "weighted_rmse":
                    float(
                        np.sqrt(
                            np.average(
                                residual
                                **
                                2,
                                weights=
                                    w,
                            )
                        )
                    ),
                "mean_abs_standardised_residual":
                    float(
                        np.mean(
                            np.abs(
                                z
                            )
                        )
                    ),
                "tail_abs_z_gt_2":
                    float(
                        np.mean(
                            np.abs(
                                z
                            )
                            >
                            2.0
                        )
                    ),
            }
        )


        detail = frame.copy()

        detail[
            "ablation"
        ] = variant

        detail[
            "channel"
        ] = channel

        detail[
            "predicted_mu"
        ] = mu

        detail[
            "residual"
        ] = residual

        detail[
            "standardised_residual"
        ] = z

        prediction_detail_frames.append(
            detail
        )


s31_removed_prediction_summary = pd.DataFrame(
    prediction_rows
)


s31_removed_prediction_detail = pd.concat(
    prediction_detail_frames,
    ignore_index=True,
) if prediction_detail_frames else pd.DataFrame()


print(
    "\n========================================"
)
print(
    "HOW THE REMAINING NETWORK PREDICTS THE REMOVED EVIDENCE"
)
print(
    "========================================"
)

display(
    s31_removed_prediction_summary
)


# ==========================================================
# 15. DIRECT PÉREZ-vs-MAX CONTRAST EXPECTATION
# ==========================================================
#
# For the contrast-removed fits, orient every removed contrast as:
#
#     Pérez minus Verstappen
#
# This makes the comparison directly interpretable.
# ==========================================================

oriented_rows = []


for variant in [
    "no_pv_contrast_21_22",
    "no_pv_all_direct_21_22",
]:

    posterior = s31_posteriors[
        variant
    ]

    hold = s31_data[
        variant
    ][
        "hold"
    ]

    for channel in [
        "pace_contrast",
        "q_contrast",
    ]:

        frame = hold[
            channel
        ].copy()

        if frame.empty:
            continue

        # Keep only exact Pérez/Max rows for the all-direct fit.
        exact_pair = s31_is_exact_pv_pair(
            frame
        ).to_numpy(
            dtype=bool
        )

        frame = frame.loc[
            exact_pair
        ].copy()

        if frame.empty:
            continue

        pred = s31_predict_rows(
            posterior,
            frame,
            channel,
        )

        y = frame[
            "y"
        ].to_numpy(
            dtype=float
        )

        mu = pred[
            "mu_mean"
        ]

        sign = np.where(
            frame[
                "driver_a"
            ].astype(str)
            .to_numpy()
            ==
            S31_DRIVER_A,
            1.0,
            -1.0,
        )

        y_oriented = (
            sign
            *
            y
        )

        mu_oriented = (
            sign
            *
            mu
        )

        w = frame[
            "weight"
        ].to_numpy(
            dtype=float
        )

        for year in sorted(
            frame[
                "year"
            ].astype(int)
            .unique()
        ):

            mask = (
                frame[
                    "year"
                ].astype(int)
                .to_numpy()
                ==
                int(year)
            )

            oriented_rows.append(
                {
                    "ablation":
                        variant,
                    "channel":
                        channel,
                    "year":
                        int(year),
                    "rows":
                        int(
                            mask.sum()
                        ),
                    "observed_perez_minus_max":
                        float(
                            np.average(
                                y_oriented[
                                    mask
                                ],
                                weights=
                                    w[
                                        mask
                                    ],
                            )
                        ),
                    "predicted_perez_minus_max":
                        float(
                            np.average(
                                mu_oriented[
                                    mask
                                ],
                                weights=
                                    w[
                                        mask
                                    ],
                            )
                        ),
                }
            )


s31_oriented_contrast_prediction = pd.DataFrame(
    oriented_rows
)


print(
    "\n========================================"
)
print(
    "PÉREZ MINUS VERSTAPPEN — OBSERVED vs NETWORK-PREDICTED"
)
print(
    "========================================"
)

display(
    s31_oriented_contrast_prediction
)


# ==========================================================
# 16. EVIDENCE-INFLUENCE TABLE
# ==========================================================
#
# Sign convention:
#
#   removal_effect = ablated posterior mean - frozen posterior mean
#
# Examples:
#
#   NO TEAM LEVEL gives negative removal_effect:
#       team-level evidence had been pulling alpha UP.
#
#   NO CONTRAST gives positive removal_effect:
#       direct teammate contrasts had been pulling alpha DOWN.
#
# ==========================================================

influence_rows = []


for year in S31_YEARS:

    frozen = s31_state[
        (
            s31_state[
                "variant"
            ]
            ==
            "frozen"
        )
        &
        (
            s31_state[
                "year"
            ]
            ==
            int(year)
        )
    ].iloc[
        0
    ]

    for variant in S31_ABLATIONS:

        row = s31_state[
            (
                s31_state[
                    "variant"
                ]
                ==
                variant
            )
            &
            (
                s31_state[
                    "year"
                ]
                ==
                int(year)
            )
        ].iloc[
            0
        ]

        influence_rows.append(
            {
                "year":
                    int(year),
                "ablation":
                    variant,

                "perez_removal_effect":
                    float(
                        row[
                            "perez_alpha"
                        ]
                        -
                        frozen[
                            "perez_alpha"
                        ]
                    ),

                "max_removal_effect":
                    float(
                        row[
                            "max_alpha"
                        ]
                        -
                        frozen[
                            "max_alpha"
                        ]
                    ),

                "delta_alpha_removal_effect":
                    float(
                        row[
                            "delta_alpha"
                        ]
                        -
                        frozen[
                            "delta_alpha"
                        ]
                    ),

                "red_bull_beta_removal_effect":
                    float(
                        row[
                            "red_bull_beta"
                        ]
                        -
                        frozen[
                            "red_bull_beta"
                        ]
                    ),

                "common_driver_removal_effect":
                    float(
                        row[
                            "common_driver_level"
                        ]
                        -
                        frozen[
                            "common_driver_level"
                        ]
                    ),

                "team_total_removal_effect":
                    float(
                        row[
                            "driver_plus_car_total"
                        ]
                        -
                        frozen[
                            "driver_plus_car_total"
                        ]
                    ),
            }
        )


s31_influence = pd.DataFrame(
    influence_rows
)


print(
    "\n========================================"
)
print(
    "ABLATION EFFECT RELATIVE TO FROZEN POSTERIOR"
)
print(
    "========================================"
)

display(
    s31_influence
)


# ==========================================================
# 17. OPTIONAL PÉREZ JAWS SENSITIVITY
# ==========================================================
#
# If Stage 30's exact scoring helper is still in memory, reuse it.
# This is deliberately optional because Stage 31's primary purpose
# is explaining the season-level alpha, not rebuilding the ranking.
# ==========================================================

s31_perez_jaws = pd.DataFrame()


if (
    "s30_score_alpha_draws"
    in globals()
    and
    "eligible_drivers"
    in globals()
):

    jaws_rows = []

    for variant, posterior in s31_posteriors.items():

        alpha = s31_stack_state(
            posterior,
            "alpha_dy",
        )

        scored = s30_score_alpha_draws(
            alpha
        )

        table = scored[
            "table"
        ]

        p = table[
            table[
                "driver"
            ]
            ==
            S31_DRIVER_A
        ]

        if p.empty:
            continue

        p = p.iloc[
            0
        ]

        jaws_rows.append(
            {
                "variant":
                    variant,
                "perez_jaws_mean":
                    float(
                        p[
                            "jaws_mean"
                        ]
                    ),
                "perez_point_rank":
                    int(
                        p[
                            "point_rank"
                        ]
                    ),
                "perez_rank_mean":
                    float(
                        p[
                            "rank_mean"
                        ]
                    ),
            }
        )

    s31_perez_jaws = pd.DataFrame(
        jaws_rows
    )

    print(
        "\n========================================"
    )
    print(
        "OPTIONAL PÉREZ JAWS SENSITIVITY"
    )
    print(
        "========================================"
    )

    display(
        s31_perez_jaws
    )


# ==========================================================
# 18. PLOTS
# ==========================================================

variant_order = [
    "frozen",
    "no_rb_team_level_21_22",
    "no_pv_contrast_21_22",
    "no_pv_all_direct_21_22",
]


variant_labels = {
    "frozen":
        "Frozen",

    "no_rb_team_level_21_22":
        "No RB team level",

    "no_pv_contrast_21_22":
        "No Pérez–Max contrast",

    "no_pv_all_direct_21_22":
        "No direct 21–22 evidence",
}


# ----------------------------------------------------------
# A. Pérez 2019–2024 trajectory
# ----------------------------------------------------------

plt.figure(
    figsize=(
        11,
        6,
    )
)


for variant in variant_order:

    x = s31_trajectory[
        (
            s31_trajectory[
                "driver"
            ]
            ==
            S31_DRIVER_A
        )
        &
        (
            s31_trajectory[
                "variant"
            ]
            ==
            variant
        )
    ].sort_values(
        "year"
    )

    plt.plot(
        x[
            "year"
        ],
        x[
            "alpha_mean"
        ],
        marker="o",
        label=
            variant_labels[
                variant
            ],
    )


plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.xlabel(
    "Season"
)

plt.ylabel(
    "Posterior mean alpha"
)

plt.title(
    "Pérez career trajectory under 2021–22 evidence ablations"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# B. Verstappen 2019–2024 trajectory
# ----------------------------------------------------------

plt.figure(
    figsize=(
        11,
        6,
    )
)


for variant in variant_order:

    x = s31_trajectory[
        (
            s31_trajectory[
                "driver"
            ]
            ==
            S31_DRIVER_B
        )
        &
        (
            s31_trajectory[
                "variant"
            ]
            ==
            variant
        )
    ].sort_values(
        "year"
    )

    plt.plot(
        x[
            "year"
        ],
        x[
            "alpha_mean"
        ],
        marker="o",
        label=
            variant_labels[
                variant
            ],
    )


plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.xlabel(
    "Season"
)

plt.ylabel(
    "Posterior mean alpha"
)

plt.title(
    "Verstappen career trajectory under 2021–22 evidence ablations"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# C. Pérez 2021 / 2022 alpha by ablation
# ----------------------------------------------------------

plot_state = s31_state.copy()

plot_state[
    "_variant_order"
] = plot_state[
    "variant"
].map(
    {
        v:
            i
        for i, v
        in enumerate(
            variant_order
        )
    }
)

plot_state = plot_state.sort_values(
    [
        "year",
        "_variant_order",
    ]
)


x = np.arange(
    len(
        variant_order
    )
)

width = 0.35


plt.figure(
    figsize=(
        11,
        6,
    )
)


for j, year in enumerate(
    S31_YEARS
):

    y = (
        plot_state[
            plot_state[
                "year"
            ]
            ==
            year
        ]
        .set_index(
            "variant"
        )
        .loc[
            variant_order,
            "perez_alpha"
        ]
        .to_numpy(
            dtype=float
        )
    )

    plt.bar(
        x
        +
        (
            j
            -
            0.5
        )
        *
        width,
        y,
        width=
            width,
        label=
            str(
                year
            ),
    )


plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

plt.xticks(
    x,
    [
        variant_labels[
            v
        ]
        for v in variant_order
    ],
    rotation=20,
    ha="right",
)

plt.ylabel(
    "Posterior mean Pérez alpha"
)

plt.title(
    "Which evidence holds Pérez's 2021–22 alpha up or down?"
)

plt.legend(
    title=
        "Season"
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# D. Red Bull 2022 allocation
# ----------------------------------------------------------

rb22 = (
    s31_state[
        s31_state[
            "year"
        ]
        ==
        2022
    ]
    .set_index(
        "variant"
    )
    .loc[
        variant_order
    ]
    .reset_index()
)


x = np.arange(
    len(
        rb22
    )
)


plt.figure(
    figsize=(
        11,
        6,
    )
)


plt.plot(
    x,
    rb22[
        "perez_alpha"
    ],
    marker="o",
    label=
        "Pérez alpha",
)

plt.plot(
    x,
    rb22[
        "max_alpha"
    ],
    marker="o",
    label=
        "Verstappen alpha",
)

plt.plot(
    x,
    rb22[
        "red_bull_beta"
    ],
    marker="s",
    label=
        "Red Bull beta",
)

plt.plot(
    x,
    rb22[
        "driver_plus_car_total"
    ],
    marker="^",
    label=
        "Common driver + car total",
)


plt.xticks(
    x,
    [
        variant_labels[
            v
        ]
        for v in rb22[
            "variant"
        ]
    ],
    rotation=20,
    ha="right",
)

plt.ylabel(
    "Latent model units"
)

plt.title(
    "Red Bull 2022 decomposition under evidence ablation"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ==========================================================
# 19. SAVE OUTPUTS
# ==========================================================

s31_removal_summary.to_csv(
    S31_OUTPUT_DIR
    /
    "rows_removed_by_ablation.csv",
    index=False,
)


s31_health.to_csv(
    S31_OUTPUT_DIR
    /
    "ablation_fit_health.csv",
    index=False,
)


s31_state.to_csv(
    S31_OUTPUT_DIR
    /
    "red_bull_2021_2022_state_decomposition.csv",
    index=False,
)


s31_trajectory.to_csv(
    S31_OUTPUT_DIR
    /
    "perez_verstappen_2019_2024_trajectory.csv",
    index=False,
)


s31_removed_prediction_summary.to_csv(
    S31_OUTPUT_DIR
    /
    "removed_evidence_prediction_summary.csv",
    index=False,
)


if not s31_removed_prediction_detail.empty:

    s31_removed_prediction_detail.to_csv(
        S31_OUTPUT_DIR
        /
        "removed_evidence_prediction_detail.csv",
        index=False,
    )


s31_oriented_contrast_prediction.to_csv(
    S31_OUTPUT_DIR
    /
    "perez_minus_verstappen_network_prediction.csv",
    index=False,
)


s31_influence.to_csv(
    S31_OUTPUT_DIR
    /
    "ablation_influence_vs_frozen.csv",
    index=False,
)


if not s31_perez_jaws.empty:

    s31_perez_jaws.to_csv(
        S31_OUTPUT_DIR
        /
        "perez_jaws_ablation_sensitivity.csv",
        index=False,
    )


# ==========================================================
# 20. TRIAGE SUMMARY
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 31 TRIAGE SUMMARY"
)
print(
    "========================================"
)


for year in S31_YEARS:

    print(
        f"\n{year}:"
    )

    frozen = s31_state[
        (
            s31_state[
                "variant"
            ]
            ==
            "frozen"
        )
        &
        (
            s31_state[
                "year"
            ]
            ==
            year
        )
    ].iloc[
        0
    ]

    print(
        f"  Frozen Pérez alpha = "
        f"{frozen['perez_alpha']:+.3f}"
    )

    print(
        f"  Frozen Verstappen alpha = "
        f"{frozen['max_alpha']:+.3f}"
    )

    print(
        f"  Frozen Δalpha (Pérez-Max) = "
        f"{frozen['delta_alpha']:+.3f}"
    )

    print(
        f"  Frozen Red Bull beta = "
        f"{frozen['red_bull_beta']:+.3f}"
    )


    for variant in S31_ABLATIONS:

        r = s31_state[
            (
                s31_state[
                    "variant"
                ]
                ==
                variant
            )
            &
            (
                s31_state[
                    "year"
                ]
                ==
                year
            )
        ].iloc[
            0
        ]

        print(
            f"\n  {variant}:"
        )

        print(
            f"    Pérez alpha = "
            f"{r['perez_alpha']:+.3f} "
            f"(change {r['delta_perez_alpha_vs_frozen']:+.3f})"
        )

        print(
            f"    Verstappen alpha = "
            f"{r['max_alpha']:+.3f} "
            f"(change {r['delta_max_alpha_vs_frozen']:+.3f})"
        )

        print(
            f"    Δalpha = "
            f"{r['delta_alpha']:+.3f} "
            f"(change {r['delta_delta_alpha_vs_frozen']:+.3f})"
        )

        print(
            f"    Red Bull beta = "
            f"{r['red_bull_beta']:+.3f} "
            f"(change {r['delta_red_bull_beta_vs_frozen']:+.3f})"
        )


print(
    "\nDecision guide:"
)

print(
    "  1. NO RB TEAM LEVEL:"
)

print(
    "       Pérez alpha drops -> Red Bull absolute/team-level evidence"
)

print(
    "       was pulling the common driver level upward."
)

print(
    "       Pérez alpha barely moves -> that evidence is NOT the main"
)

print(
    "       explanation for his high 2021–22 alpha."
)


print(
    "\n  2. NO PÉREZ–MAX CONTRAST:"
)

print(
    "       Pérez alpha rises -> the brutal Verstappen comparison was"
)

print(
    "       correctly pulling Pérez downward."
)

print(
    "       The size of the rise quantifies that direct downward pull."
)


print(
    "\n  3. NO DIRECT 2021–22 EVIDENCE:"
)

print(
    "       This is the surrounding-career/network expectation."
)

print(
    "       If Pérez is already near +0.25 here, his Red Bull peak is"
)

print(
    "       largely inherited from connected career/state-space evidence."
)

print(
    "       If he falls near his 2020 level (~+0.10), then the 2021–22"
)

print(
    "       observations themselves are what lift his absolute level."
)


print(
    "\n  4. Read the three ablations together:"
)

print(
    "       frozen vs no-team-level  -> upward pull from absolute RB level"
)

print(
    "       frozen vs no-contrast    -> downward pull from Verstappen gap"
)

print(
    "       no-all-direct            -> prior/network baseline"
)


print(
    "\nSaved outputs ->"
)

print(
    S31_OUTPUT_DIR.resolve()
)

print(
    "\nStage 31 complete."
)


## 2026 / familiarity development audits


In [ ]:
# ==========================================================
# STAGE 32B — LIVE 2026 MODERN-EVIDENCE AUDIT
# ==========================================================
#
# RUN THIS:
#   after Stage 3, Stage 4, Stage 5 and Stage 6
#   after the new live-2026 augmentation cell.
#
# DO NOT BUILD / SAMPLE THE 1982–2026 MODEL YET.
#
# PURPOSE
# -------
# Verify that the partial 2026 season has entered the modern
# likelihood exactly as intended:
#
#   1. all completed 2026 races represented;
#   2. sensible pace and qualifying coverage;
#   3. correct two-driver teammate pairings;
#   4. no duplicate observations;
#   5. qualifying weights = 1 / number of common sessions;
#   6. Q1 level weights = 1;
#   7. constructor-tenure values are sensible;
#   8. new/rebranded teams are visible explicitly;
#   9. no suspicious 2026 clipping / scale pathology;
#  10. identify exactly which team-races are absent from any
#      channel and WHY before fitting.
#
# This cell changes NOTHING. It is read-only diagnostics.
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


AUDIT_YEAR = 2026


# ==========================================================
# 0. REQUIRED OBJECTS
# ==========================================================

required = [
    "CFG",
    "raw",
    "driver_event_cov",
    "df_pace",
    "df_quali",
    "pace_contrast",
    "pace_level",
    "q_contrast",
    "q_level",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Stage 32B requires Stages 3–6. Missing:\n  "
        + "\n  ".join(missing)
    )


# ==========================================================
# 1. BASIC 2026 RACE / RESULT UNIVERSE
# ==========================================================

races_2026 = (
    raw.races[
        pd.to_numeric(
            raw.races["year"],
            errors="coerce",
        )
        ==
        AUDIT_YEAR
    ]
    .copy()
)

if races_2026.empty:
    raise RuntimeError(
        "No 2026 races are present in raw.races."
    )

race_ids_2026 = set(
    pd.to_numeric(
        races_2026["raceId"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)

results_2026 = (
    raw.results[
        pd.to_numeric(
            raw.results["raceId"],
            errors="coerce",
        ).isin(
            race_ids_2026
        )
    ]
    .copy()
)

qualifying_raw_2026 = (
    raw.qualifying[
        pd.to_numeric(
            raw.qualifying["raceId"],
            errors="coerce",
        ).isin(
            race_ids_2026
        )
    ]
    .copy()
)

laps_raw_2026 = (
    raw.laps[
        pd.to_numeric(
            raw.laps["raceId"],
            errors="coerce",
        ).isin(
            race_ids_2026
        )
    ]
    .copy()
)


race_meta_cols = [
    col
    for col in [
        "raceId",
        "round",
        "name",
        "date",
    ]
    if col in races_2026.columns
]

race_meta = (
    races_2026[
        race_meta_cols
    ]
    .copy()
)

race_meta["raceId"] = pd.to_numeric(
    race_meta["raceId"],
    errors="coerce",
).astype(int)

if "round" in race_meta.columns:
    race_meta["round"] = pd.to_numeric(
        race_meta["round"],
        errors="coerce",
    ).astype("Int64")


# ==========================================================
# 2. CONSTRUCTOR + DRIVER NAME HELPERS
# ==========================================================

constructor_name_col = None

for candidate in [
    "name",
    "constructorRef",
]:
    if candidate in raw.constructors.columns:
        constructor_name_col = candidate
        break

if constructor_name_col is None:
    raise RuntimeError(
        "Could not find constructor name/ref column."
    )

constructor_name_map = dict(
    zip(
        pd.to_numeric(
            raw.constructors["constructorId"],
            errors="coerce",
        ).astype("Int64"),
        raw.constructors[
            constructor_name_col
        ].astype(str),
    )
)

constructor_ref_map = {}

if "constructorRef" in raw.constructors.columns:
    constructor_ref_map = dict(
        zip(
            pd.to_numeric(
                raw.constructors["constructorId"],
                errors="coerce",
            ).astype("Int64"),
            raw.constructors[
                "constructorRef"
            ].astype(str),
        )
    )


def constructor_name(cid):
    try:
        cid = int(cid)
    except Exception:
        return str(cid)

    return constructor_name_map.get(
        cid,
        str(cid),
    )


def constructor_ref(cid):
    try:
        cid = int(cid)
    except Exception:
        return str(cid)

    return constructor_ref_map.get(
        cid,
        str(cid),
    )


driver_name_map = dict(
    zip(
        pd.to_numeric(
            raw.drivers["driverId"],
            errors="coerce",
        ).astype("Int64"),
        raw.drivers["driverRef"].astype(str),
    )
)


def driver_name(did):
    try:
        did = int(did)
    except Exception:
        return str(did)

    return driver_name_map.get(
        did,
        str(did),
    )


# ==========================================================
# 3. 2026-ONLY MODERN TABLES
# ==========================================================

pace_2026 = (
    df_pace[
        pd.to_numeric(
            df_pace["year"],
            errors="coerce",
        )
        ==
        AUDIT_YEAR
    ]
    .copy()
)

quali_2026 = (
    df_quali[
        pd.to_numeric(
            df_quali["year"],
            errors="coerce",
        )
        ==
        AUDIT_YEAR
    ]
    .copy()
)

pc_2026 = (
    pace_contrast[
        pd.to_numeric(
            pace_contrast["year"],
            errors="coerce",
        )
        ==
        AUDIT_YEAR
    ]
    .copy()
)

pl_2026 = (
    pace_level[
        pd.to_numeric(
            pace_level["year"],
            errors="coerce",
        )
        ==
        AUDIT_YEAR
    ]
    .copy()
)

qc_2026 = (
    q_contrast[
        pd.to_numeric(
            q_contrast["year"],
            errors="coerce",
        )
        ==
        AUDIT_YEAR
    ]
    .copy()
)

ql_2026 = (
    q_level[
        pd.to_numeric(
            q_level["year"],
            errors="coerce",
        )
        ==
        AUDIT_YEAR
    ]
    .copy()
)


print(
    "========================================"
)
print(
    "STAGE 32B — LIVE 2026 MODERN-EVIDENCE AUDIT"
)
print(
    "========================================"
)

print(
    "CFG.ceiling_year:",
    CFG.ceiling_year,
)

print(
    "2026 races in raw.races:",
    races_2026["raceId"].nunique(),
)

print(
    "2026 result rows:",
    len(results_2026),
)

print(
    "2026 raw qualifying rows:",
    len(qualifying_raw_2026),
)

print(
    "2026 raw lap rows:",
    len(laps_raw_2026),
)

print(
    "\n2026 MODERN OBSERVATIONS:"
)

print(
    "  driver-race pace rows:",
    len(pace_2026),
)

print(
    "  timed qualifying driver-session rows:",
    len(quali_2026),
)

print(
    "  pace contrasts:",
    len(pc_2026),
)

print(
    "  pace levels:",
    len(pl_2026),
)

print(
    "  qualifying contrasts:",
    len(qc_2026),
)

print(
    "  Q1 levels:",
    len(ql_2026),
)


# ==========================================================
# 4. RACE-BY-RACE COVERAGE
# ==========================================================

coverage_rows = []


for race in race_meta.sort_values(
    [
        col
        for col in [
            "round",
            "raceId",
        ]
        if col in race_meta.columns
    ]
).itertuples(
    index=False
):

    race_id = int(
        race.raceId
    )

    res = results_2026[
        pd.to_numeric(
            results_2026["raceId"],
            errors="coerce",
        )
        ==
        race_id
    ]

    constructors_result = (
        pd.to_numeric(
            res["constructorId"],
            errors="coerce",
        )
        .dropna()
        .astype(int)
        .nunique()
    )

    p = pace_2026[
        pace_2026["raceId"]
        ==
        race_id
    ]

    q = quali_2026[
        quali_2026["raceId"]
        ==
        race_id
    ]

    pc = pc_2026[
        pc_2026["raceId"]
        ==
        race_id
    ]

    qc = qc_2026[
        qc_2026["raceId"]
        ==
        race_id
    ]

    ql = ql_2026[
        ql_2026["raceId"]
        ==
        race_id
    ]

    row = {
        "raceId":
            race_id,

        "round":
            (
                int(race.round)
                if (
                    hasattr(
                        race,
                        "round"
                    )
                    and
                    not pd.isna(
                        race.round
                    )
                )
                else np.nan
            ),

        "name":
            (
                race.name
                if hasattr(
                    race,
                    "name"
                )
                else
                str(
                    race_id
                )
            ),

        "result_drivers":
            res["driverId"].nunique(),

        "result_constructors":
            constructors_result,

        "pace_drivers":
            p["driverId"].nunique(),

        "pace_pairs":
            len(pc),

        "q1_drivers":
            q.loc[
                q["session"]
                ==
                "q1",
                "driverId",
            ].nunique(),

        "q2_drivers":
            q.loc[
                q["session"]
                ==
                "q2",
                "driverId",
            ].nunique(),

        "q3_drivers":
            q.loc[
                q["session"]
                ==
                "q3",
                "driverId",
            ].nunique(),

        "q_common_session_contrasts":
            len(qc),

        "q1_team_levels":
            len(ql),
    }

    row[
        "all_result_teams_have_pace_pair"
    ] = (
        row[
            "pace_pairs"
        ]
        ==
        row[
            "result_constructors"
        ]
    )

    row[
        "all_result_teams_have_q1_level"
    ] = (
        row[
            "q1_team_levels"
        ]
        ==
        row[
            "result_constructors"
        ]
    )

    coverage_rows.append(
        row
    )


coverage_2026 = pd.DataFrame(
    coverage_rows
)


print(
    "\n========================================"
)
print(
    "RACE-BY-RACE 2026 COVERAGE"
)
print(
    "========================================"
)

display(
    coverage_2026
)


# ==========================================================
# 5. 2026 GRID — ACTUAL RESULT CONSTRUCTOR/DRIVER ASSIGNMENTS
# ==========================================================

grid_2026 = (
    results_2026[
        [
            "raceId",
            "driverId",
            "constructorId",
        ]
    ]
    .drop_duplicates()
    .merge(
        race_meta[
            [
                col
                for col in [
                    "raceId",
                    "round",
                    "name",
                ]
                if col in race_meta.columns
            ]
        ],
        on="raceId",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

grid_2026["driver"] = (
    grid_2026["driverId"]
    .map(
        driver_name
    )
)

grid_2026["constructor"] = (
    grid_2026["constructorId"]
    .map(
        constructor_name
    )
)

grid_2026["constructorRef"] = (
    grid_2026["constructorId"]
    .map(
        constructor_ref
    )
)


grid_summary_2026 = (
    grid_2026
    .groupby(
        [
            "constructorId",
            "constructorRef",
            "constructor",
        ],
        as_index=False,
    )
    .agg(
        drivers=(
            "driver",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x
                        )
                    )
                ),
        ),
        races=(
            "raceId",
            "nunique",
        ),
    )
    .sort_values(
        "constructor"
    )
)


print(
    "\n========================================"
)
print(
    "2026 CONSTRUCTOR / DRIVER GRID"
)
print(
    "========================================"
)

display(
    grid_summary_2026
)


# ==========================================================
# 6. TEAM-RACE PAIR COVERAGE — EXACTLY WHAT IS MISSING?
# ==========================================================

expected_team_races = (
    results_2026[
        [
            "raceId",
            "constructorId",
        ]
    ]
    .drop_duplicates()
    .copy()
)

expected_team_races = expected_team_races.merge(
    race_meta[
        [
            col
            for col in [
                "raceId",
                "round",
                "name",
            ]
            if col in race_meta.columns
        ]
    ],
    on="raceId",
    how="left",
    validate="many_to_one",
)

expected_team_races[
    "constructor"
] = (
    expected_team_races[
        "constructorId"
    ]
    .map(
        constructor_name
    )
)


pc_keys = set(
    zip(
        pc_2026["raceId"].astype(int),
        pc_2026["constructorId"].astype(int),
    )
)

ql_keys = set(
    zip(
        ql_2026["raceId"].astype(int),
        ql_2026["constructorId"].astype(int),
    )
)


expected_team_races[
    "has_pace_pair"
] = [
    (
        int(
            r.raceId
        ),
        int(
            r.constructorId
        ),
    )
    in
    pc_keys
    for r
    in expected_team_races.itertuples(
        index=False
    )
]


expected_team_races[
    "has_q1_level"
] = [
    (
        int(
            r.raceId
        ),
        int(
            r.constructorId
        ),
    )
    in
    ql_keys
    for r
    in expected_team_races.itertuples(
        index=False
    )
]


missing_pair_rows = expected_team_races[
    (
        ~expected_team_races[
            "has_pace_pair"
        ]
    )
    |
    (
        ~expected_team_races[
            "has_q1_level"
        ]
    )
].copy()


# Count how many valid driver rows each missing team-race had.
pace_valid_counts = (
    pace_2026
    .groupby(
        [
            "raceId",
            "constructorId",
        ]
    )[
        "driverId"
    ]
    .nunique()
    .rename(
        "valid_pace_drivers"
    )
)


q1_valid_counts = (
    quali_2026[
        quali_2026[
            "session"
        ]
        ==
        "q1"
    ]
    .groupby(
        [
            "raceId",
            "constructorId",
        ]
    )[
        "driverId"
    ]
    .nunique()
    .rename(
        "valid_q1_drivers"
    )
)


missing_pair_rows = (
    missing_pair_rows
    .merge(
        pace_valid_counts,
        left_on=[
            "raceId",
            "constructorId",
        ],
        right_index=True,
        how="left",
    )
    .merge(
        q1_valid_counts,
        left_on=[
            "raceId",
            "constructorId",
        ],
        right_index=True,
        how="left",
    )
)


for col in [
    "valid_pace_drivers",
    "valid_q1_drivers",
]:
    missing_pair_rows[
        col
    ] = (
        missing_pair_rows[
            col
        ]
        .fillna(
            0
        )
        .astype(int)
    )


print(
    "\n========================================"
)
print(
    "MISSING 2026 TEAM-RACE PAIRS"
)
print(
    "========================================"
)

print(
    "Rows shown only when a result team lacks a pace pair "
    "or Q1 team-level observation."
)

if missing_pair_rows.empty:
    print(
        "None — every result constructor/race has both."
    )
else:
    display(
        missing_pair_rows.sort_values(
            [
                "round",
                "constructor",
            ]
        )
    )


# ==========================================================
# 7. DUPLICATE CHECKS
# ==========================================================

duplicate_checks = {
    "pace driver/race":
        int(
            pace_2026.duplicated(
                [
                    "raceId",
                    "driverId",
                ]
            ).sum()
        ),

    "quali driver/race/session":
        int(
            quali_2026.duplicated(
                [
                    "raceId",
                    "driverId",
                    "session",
                ]
            ).sum()
        ),

    "pace contrast team/race":
        int(
            pc_2026.duplicated(
                [
                    "raceId",
                    "constructorId",
                ]
            ).sum()
        ),

    "pace level team/race":
        int(
            pl_2026.duplicated(
                [
                    "raceId",
                    "constructorId",
                ]
            ).sum()
        ),

    "qualifying contrast team/race/session":
        int(
            qc_2026.duplicated(
                [
                    "raceId",
                    "constructorId",
                    "session",
                ]
            ).sum()
        ),

    "Q1 level team/race":
        int(
            ql_2026.duplicated(
                [
                    "raceId",
                    "constructorId",
                ]
            ).sum()
        ),
}


duplicate_table = pd.DataFrame(
    {
        "check":
            list(
                duplicate_checks.keys()
            ),
        "duplicates":
            list(
                duplicate_checks.values()
            ),
    }
)


print(
    "\n========================================"
)
print(
    "DUPLICATE CHECKS"
)
print(
    "========================================"
)

display(
    duplicate_table
)


# ==========================================================
# 8. QUALIFYING SESSION / WEIGHT AUDIT
# ==========================================================

quali_session_summary = (
    quali_2026
    .groupby(
        "session"
    )
    .agg(
        driver_session_rows=(
            "driverId",
            "size",
        ),
        races=(
            "raceId",
            "nunique",
        ),
        mean_y=(
            "y_quali",
            "mean",
        ),
        sd_y=(
            "y_quali",
            "std",
        ),
        min_y=(
            "y_quali",
            "min",
        ),
        max_y=(
            "y_quali",
            "max",
        ),
    )
    .reset_index()
)


print(
    "\n========================================"
)
print(
    "QUALIFYING SESSION SUMMARY"
)
print(
    "========================================"
)

display(
    quali_session_summary
)


if len(
    qc_2026
):

    common_session_count = (
        qc_2026
        .groupby(
            [
                "raceId",
                "constructorId",
            ]
        )[
            "session"
        ]
        .transform(
            "count"
        )
        .astype(float)
    )

    expected_q_weight = (
        1.0
        /
        common_session_count
    )

    q_weight_error = np.abs(
        qc_2026[
            "weight"
        ].to_numpy(
            dtype=float
        )
        -
        expected_q_weight.to_numpy(
            dtype=float
        )
    )

    max_q_weight_error = float(
        q_weight_error.max()
    )

else:
    max_q_weight_error = np.nan


max_q1_weight_error = (
    float(
        np.abs(
            ql_2026[
                "weight"
            ].to_numpy(
                dtype=float
            )
            -
            1.0
        ).max()
    )
    if len(
        ql_2026
    )
    else np.nan
)


q_common_sessions_distribution = (
    qc_2026
    .groupby(
        [
            "raceId",
            "constructorId",
        ]
    )
    .agg(
        n_common_sessions=(
            "session",
            "count",
        )
    )
    .reset_index()
    .groupby(
        "n_common_sessions"
    )
    .size()
    .rename(
        "team_races"
    )
    .reset_index()
)


print(
    "\nCommon qualifying sessions per team/race:"
)

display(
    q_common_sessions_distribution
)


print(
    "Max |q contrast weight - 1/n_common_sessions|:",
    max_q_weight_error,
)


print(
    "Max |Q1 level weight - 1|:",
    max_q1_weight_error,
)


# ==========================================================
# 9. PACE SCALE / CLIPPING AUDIT
# ==========================================================

pace_race_summary = (
    pace_2026
    .merge(
        race_meta[
            [
                col
                for col in [
                    "raceId",
                    "round",
                    "name",
                ]
                if col in race_meta.columns
            ]
        ],
        on="raceId",
        how="left",
        validate="many_to_one",
    )
    .groupby(
        [
            col
            for col in [
                "raceId",
                "round",
                "name",
            ]
            if col in (
                pace_2026.columns.tolist()
                +
                race_meta.columns.tolist()
            )
        ],
        as_index=False,
    )
    .agg(
        drivers=(
            "driverId",
            "nunique",
        ),
        mean_y=(
            "y_pace",
            "mean",
        ),
        median_y=(
            "y_pace",
            "median",
        ),
        sd_y=(
            "y_pace",
            "std",
        ),
        min_y=(
            "y_pace",
            "min",
        ),
        max_y=(
            "y_pace",
            "max",
        ),
        mean_comparable_laps=(
            "n_comparable_laps",
            "mean",
        ),
        min_comparable_laps=(
            "n_comparable_laps",
            "min",
        ),
        mean_weight=(
            "pace_weight",
            "mean",
        ),
    )
)


clip_mask_2026 = np.isclose(
    np.abs(
        pace_2026[
            "y_pace"
        ].to_numpy(
            dtype=float
        )
    ),
    4.0,
)


n_clipped_2026 = int(
    clip_mask_2026.sum()
)


clip_fraction_2026 = (
    float(
        clip_mask_2026.mean()
    )
    if len(
        clip_mask_2026
    )
    else np.nan
)


print(
    "\n========================================"
)
print(
    "PACE SCALE / CLIPPING AUDIT"
)
print(
    "========================================"
)

display(
    pace_race_summary
)


print(
    "Exact +/-4 clipped 2026 pace observations:",
    n_clipped_2026,
)

print(
    "2026 pace clipping fraction:",
    clip_fraction_2026,
)


if n_clipped_2026:

    clipped_pace_2026 = (
        pace_2026.loc[
            clip_mask_2026,
            [
                col
                for col in [
                    "raceId",
                    "driver",
                    "constructorId",
                    "y_pace",
                    "pace_weight",
                    "n_comparable_laps",
                ]
                if col in pace_2026.columns
            ]
        ]
        .merge(
            race_meta[
                [
                    col
                    for col in [
                        "raceId",
                        "round",
                        "name",
                    ]
                    if col in race_meta.columns
                ]
            ],
            on="raceId",
            how="left",
        )
    )

    clipped_pace_2026[
        "constructor"
    ] = (
        clipped_pace_2026[
            "constructorId"
        ]
        .map(
            constructor_name
        )
    )

    display(
        clipped_pace_2026.sort_values(
            [
                "round",
                "y_pace",
            ]
        )
    )


# ==========================================================
# 10. TENURE AUDIT
# ==========================================================
#
# We inspect the uncentred Stage-3 tenure at the FIRST 2026
# race for every driver/team, plus the centred values used in
# pace and qualifying.
#
# A brand-new constructor identity should normally have
# tenure_raw = 0 for its drivers at the first race.
# ==========================================================

first_2026_race_id = int(
    race_meta.sort_values(
        [
            col
            for col in [
                "round",
                "raceId",
            ]
            if col in race_meta.columns
        ]
    )
    .iloc[
        0
    ][
        "raceId"
    ]
)


first_race_results = (
    results_2026[
        pd.to_numeric(
            results_2026[
                "raceId"
            ],
            errors="coerce",
        )
        ==
        first_2026_race_id
    ][
        [
            "raceId",
            "driverId",
            "constructorId",
        ]
    ]
    .drop_duplicates()
    .copy()
)


tenure_first = (
    first_race_results
    .merge(
        driver_event_cov[
            [
                "raceId",
                "driverId",
                "constructorId",
                "tenure_raw",
            ]
        ],
        on=[
            "raceId",
            "driverId",
            "constructorId",
        ],
        how="left",
        validate="one_to_one",
    )
)


tenure_first[
    "driver"
] = (
    tenure_first[
        "driverId"
    ]
    .map(
        driver_name
    )
)


tenure_first[
    "constructor"
] = (
    tenure_first[
        "constructorId"
    ]
    .map(
        constructor_name
    )
)


tenure_first[
    "constructorRef"
] = (
    tenure_first[
        "constructorId"
    ]
    .map(
        constructor_ref
    )
)


# Recover the number of prior constructor starts implied by
# tenure_raw = tanh(s / tenure_scale_races).
tenure_scale = float(
    CFG.tenure_scale_races
)


def implied_prior_starts(
    x,
):
    if pd.isna(
        x
    ):
        return np.nan

    x = float(
        np.clip(
            x,
            -0.999999999,
            0.999999999,
        )
    )

    return (
        np.arctanh(
            x
        )
        *
        tenure_scale
    )


tenure_first[
    "implied_prior_constructor_starts"
] = (
    tenure_first[
        "tenure_raw"
    ]
    .apply(
        implied_prior_starts
    )
)


print(
    "\n========================================"
)
print(
    "FIRST-2026-RACE CONSTRUCTOR TENURE"
)
print(
    "========================================"
)

display(
    tenure_first[
        [
            "driver",
            "constructorRef",
            "constructor",
            "tenure_raw",
            "implied_prior_constructor_starts",
        ]
    ]
    .sort_values(
        [
            "constructor",
            "driver",
        ]
    )
)


# Centred tenure sanity.
tenure_missing = pd.DataFrame(
    {
        "table":
            [
                "pace driver rows",
                "quali driver/session rows",
                "pace contrasts",
                "pace levels",
                "quali contrasts",
                "Q1 levels",
            ],

        "missing_tenure":
            [
                int(
                    pace_2026[
                        "tenure_c"
                    ].isna().sum()
                ),

                int(
                    quali_2026[
                        "tenure_c"
                    ].isna().sum()
                ),

                int(
                    pc_2026[
                        "tenure_diff"
                    ].isna().sum()
                ),

                int(
                    pl_2026[
                        "tenure_mean"
                    ].isna().sum()
                ),

                int(
                    qc_2026[
                        "tenure_diff"
                    ].isna().sum()
                ),

                int(
                    ql_2026[
                        "tenure_mean"
                    ].isna().sum()
                ),
            ],
    }
)


print(
    "\nTenure missingness:"
)

display(
    tenure_missing
)


# ==========================================================
# 11. 2025 -> 2026 DRIVER / CONSTRUCTOR TRANSITIONS
# ==========================================================

races_2025 = raw.races[
    pd.to_numeric(
        raw.races[
            "year"
        ],
        errors="coerce",
    )
    ==
    2025
][
    [
        "raceId",
    ]
].copy()


race_ids_2025 = set(
    pd.to_numeric(
        races_2025[
            "raceId"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


results_2025 = raw.results[
    pd.to_numeric(
        raw.results[
            "raceId"
        ],
        errors="coerce",
    )
    .isin(
        race_ids_2025
    )
].copy()


last_team_2025 = (
    results_2025[
        [
            "driverId",
            "constructorId",
            "raceId",
        ]
    ]
    .sort_values(
        "raceId"
    )
    .drop_duplicates(
        subset=[
            "driverId",
        ],
        keep="last",
    )
    .rename(
        columns={
            "constructorId":
                "constructorId_2025",
        }
    )[
        [
            "driverId",
            "constructorId_2025",
        ]
    ]
)


first_team_2026 = (
    first_race_results[
        [
            "driverId",
            "constructorId",
        ]
    ]
    .rename(
        columns={
            "constructorId":
                "constructorId_2026",
        }
    )
)


transitions = (
    first_team_2026
    .merge(
        last_team_2025,
        on="driverId",
        how="left",
    )
)


transitions[
    "driver"
] = (
    transitions[
        "driverId"
    ]
    .map(
        driver_name
    )
)


transitions[
    "constructor_2025"
] = (
    transitions[
        "constructorId_2025"
    ]
    .map(
        constructor_name
    )
)


transitions[
    "constructor_2026"
] = (
    transitions[
        "constructorId_2026"
    ]
    .map(
        constructor_name
    )
)


transitions[
    "same_constructor_identity"
] = (
    transitions[
        "constructorId_2025"
    ]
    ==
    transitions[
        "constructorId_2026"
    ]
)


print(
    "\n========================================"
)
print(
    "2025 -> 2026 DRIVER / CONSTRUCTOR TRANSITIONS"
)
print(
    "========================================"
)

display(
    transitions[
        [
            "driver",
            "constructor_2025",
            "constructor_2026",
            "same_constructor_identity",
        ]
    ]
    .sort_values(
        [
            "constructor_2026",
            "driver",
        ]
    )
)


# ==========================================================
# 12. TEAMMATE PAIR SUMMARY ACROSS 2026 SO FAR
# ==========================================================

pace_pair_summary = (
    pc_2026
    .assign(
        pair=lambda x:
            x.apply(
                lambda r:
                    " / ".join(
                        sorted(
                            [
                                str(
                                    r[
                                        "driver_a"
                                    ]
                                ),
                                str(
                                    r[
                                        "driver_b"
                                    ]
                                ),
                            ]
                        )
                    ),
                axis=1,
            )
    )
    .groupby(
        [
            "constructorId",
            "pair",
        ],
        as_index=False,
    )
    .agg(
        pace_races=(
            "raceId",
            "nunique",
        ),
        mean_pace_delta_oriented_as_stored=(
            "y",
            "mean",
        ),
    )
)


pace_pair_summary[
    "constructor"
] = (
    pace_pair_summary[
        "constructorId"
    ]
    .map(
        constructor_name
    )
)


quali_pair_summary = (
    qc_2026
    .assign(
        pair=lambda x:
            x.apply(
                lambda r:
                    " / ".join(
                        sorted(
                            [
                                str(
                                    r[
                                        "driver_a"
                                    ]
                                ),
                                str(
                                    r[
                                        "driver_b"
                                    ]
                                ),
                            ]
                        )
                    ),
                axis=1,
            )
    )
    .groupby(
        [
            "constructorId",
            "pair",
        ],
        as_index=False,
    )
    .agg(
        qualifying_common_sessions=(
            "session",
            "size",
        ),
        qualifying_races=(
            "raceId",
            "nunique",
        ),
    )
)


quali_pair_summary[
    "constructor"
] = (
    quali_pair_summary[
        "constructorId"
    ]
    .map(
        constructor_name
    )
)


pair_summary = (
    pace_pair_summary
    .merge(
        quali_pair_summary[
            [
                "constructorId",
                "pair",
                "qualifying_common_sessions",
                "qualifying_races",
            ]
        ],
        on=[
            "constructorId",
            "pair",
        ],
        how="outer",
    )
)


pair_summary[
    "constructor"
] = (
    pair_summary[
        "constructorId"
    ]
    .map(
        constructor_name
    )
)


print(
    "\n========================================"
)
print(
    "2026 TEAMMATE PAIRS"
)
print(
    "========================================"
)

display(
    pair_summary.sort_values(
        [
            "constructor",
            "pair",
        ]
    )
)


# ==========================================================
# 13. EXTREME 2026 MODERN OBSERVATIONS
# ==========================================================

extreme_pace = (
    pace_2026[
        [
            col
            for col in [
                "raceId",
                "driver",
                "constructorId",
                "y_pace",
                "pace_weight",
                "n_comparable_laps",
            ]
            if col in pace_2026.columns
        ]
    ]
    .assign(
        abs_y=lambda x:
            x[
                "y_pace"
            ].abs()
    )
    .merge(
        race_meta[
            [
                col
                for col in [
                    "raceId",
                    "round",
                    "name",
                ]
                if col in race_meta.columns
            ]
        ],
        on="raceId",
        how="left",
    )
    .sort_values(
        "abs_y",
        ascending=False,
    )
    .head(
        25
    )
)


extreme_pace[
    "constructor"
] = (
    extreme_pace[
        "constructorId"
    ]
    .map(
        constructor_name
    )
)


extreme_quali = (
    quali_2026[
        [
            col
            for col in [
                "raceId",
                "driver",
                "constructorId",
                "session",
                "y_quali",
                "quali_weight",
            ]
            if col in quali_2026.columns
        ]
    ]
    .assign(
        abs_y=lambda x:
            x[
                "y_quali"
            ].abs()
    )
    .merge(
        race_meta[
            [
                col
                for col in [
                    "raceId",
                    "round",
                    "name",
                ]
                if col in race_meta.columns
            ]
        ],
        on="raceId",
        how="left",
    )
    .sort_values(
        "abs_y",
        ascending=False,
    )
    .head(
        25
    )
)


extreme_quali[
    "constructor"
] = (
    extreme_quali[
        "constructorId"
    ]
    .map(
        constructor_name
    )
)


print(
    "\n========================================"
)
print(
    "MOST EXTREME 2026 PACE OBSERVATIONS"
)
print(
    "========================================"
)

display(
    extreme_pace
)


print(
    "\n========================================"
)
print(
    "MOST EXTREME 2026 QUALIFYING OBSERVATIONS"
)
print(
    "========================================"
)

display(
    extreme_quali
)


# ==========================================================
# 14. SIMPLE VISUAL CHECKS
# ==========================================================

plt.figure(
    figsize=(
        11,
        5,
    )
)

pace_plot = (
    coverage_2026
    .sort_values(
        "round"
    )
)

plt.plot(
    pace_plot[
        "round"
    ],
    pace_plot[
        "pace_drivers"
    ],
    marker="o",
    label=
        "Valid pace drivers",
)

plt.plot(
    pace_plot[
        "round"
    ],
    pace_plot[
        "q1_drivers"
    ],
    marker="o",
    label=
        "Valid Q1 drivers",
)

plt.axhline(
    22,
    linestyle="--",
    linewidth=1,
    label=
        "Full 11-team field = 22",
)

plt.xlabel(
    "2026 round"
)

plt.ylabel(
    "Driver observations"
)

plt.title(
    "2026 modern evidence coverage by round"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


plt.figure(
    figsize=(
        11,
        5,
    )
)

plt.scatter(
    pace_2026[
        "n_comparable_laps"
    ],
    pace_2026[
        "y_pace"
    ],
    alpha=0.65,
)

plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1,
)

plt.xlabel(
    "Comparable race laps"
)

plt.ylabel(
    "Standardised race-pace observation"
)

plt.title(
    "2026 race pace vs amount of comparable-lap evidence"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ==========================================================
# 15. HARD / SOFT FLAGS
# ==========================================================

hard_flags = []
soft_flags = []


if int(
    duplicate_table[
        "duplicates"
    ].sum()
) != 0:
    hard_flags.append(
        "Duplicate modern observations exist."
    )


if (
    not np.isnan(
        max_q_weight_error
    )
    and
    max_q_weight_error
    >
    1e-12
):
    hard_flags.append(
        "Qualifying contrast weights are not exactly 1/n_common_sessions."
    )


if (
    not np.isnan(
        max_q1_weight_error
    )
    and
    max_q1_weight_error
    >
    1e-12
):
    hard_flags.append(
        "Q1 level weights are not exactly 1."
    )


if int(
    tenure_missing[
        "missing_tenure"
    ].sum()
) != 0:
    hard_flags.append(
        "Missing constructor-tenure values exist."
    )


if pace_2026.empty:
    hard_flags.append(
        "No 2026 pace evidence."
    )


if quali_2026.empty:
    hard_flags.append(
        "No 2026 qualifying evidence."
    )


if len(
    missing_pair_rows
):
    soft_flags.append(
        f"{len(missing_pair_rows)} team-race rows lack a complete "
        "pace pair and/or Q1 team-level pair. Inspect the table."
    )


if (
    not np.isnan(
        clip_fraction_2026
    )
    and
    clip_fraction_2026
    >
    0.05
):
    soft_flags.append(
        "More than 5% of 2026 pace observations are clipped at +/-4."
    )


# Check expected constructor count directly from results rather
# than hard-coding 11, but flag if it varies by race.
constructors_per_race = (
    results_2026
    .groupby(
        "raceId"
    )[
        "constructorId"
    ]
    .nunique()
)


if (
    constructors_per_race.nunique()
    >
    1
):
    soft_flags.append(
        "The number of constructors represented in results varies by race."
    )


# ==========================================================
# 16. SAVE AUDIT TABLES
# ==========================================================

audit_dir = (
    Path(
        CFG.output_dir
    )
    /
    "validation_live_2026_input_audit"
)

audit_dir.mkdir(
    parents=True,
    exist_ok=True,
)


coverage_2026.to_csv(
    audit_dir
    /
    "race_by_race_coverage_2026.csv",
    index=False,
)


grid_summary_2026.to_csv(
    audit_dir
    /
    "constructor_driver_grid_2026.csv",
    index=False,
)


missing_pair_rows.to_csv(
    audit_dir
    /
    "missing_team_race_pairs_2026.csv",
    index=False,
)


duplicate_table.to_csv(
    audit_dir
    /
    "duplicate_checks_2026.csv",
    index=False,
)


quali_session_summary.to_csv(
    audit_dir
    /
    "qualifying_session_summary_2026.csv",
    index=False,
)


q_common_sessions_distribution.to_csv(
    audit_dir
    /
    "qualifying_common_session_distribution_2026.csv",
    index=False,
)


pace_race_summary.to_csv(
    audit_dir
    /
    "pace_race_summary_2026.csv",
    index=False,
)


tenure_first.to_csv(
    audit_dir
    /
    "first_race_tenure_2026.csv",
    index=False,
)


transitions.to_csv(
    audit_dir
    /
    "driver_constructor_transitions_2025_to_2026.csv",
    index=False,
)


pair_summary.to_csv(
    audit_dir
    /
    "teammate_pairs_2026.csv",
    index=False,
)


extreme_pace.to_csv(
    audit_dir
    /
    "extreme_pace_2026.csv",
    index=False,
)


extreme_quali.to_csv(
    audit_dir
    /
    "extreme_qualifying_2026.csv",
    index=False,
)


# ==========================================================
# 17. TRIAGE SUMMARY
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 32B TRIAGE SUMMARY"
)
print(
    "========================================"
)


print(
    "Completed races represented:",
    races_2026[
        "raceId"
    ].nunique(),
)


print(
    "Constructors per race:",
    sorted(
        constructors_per_race.unique()
    ),
)


print(
    "2026 pace pairs:",
    len(
        pc_2026
    ),
)


print(
    "2026 qualifying contrasts:",
    len(
        qc_2026
    ),
)


print(
    "2026 Q1 team levels:",
    len(
        ql_2026
    ),
)


print(
    "Pace clipping fraction:",
    (
        f"{clip_fraction_2026:.3%}"
        if not np.isnan(
            clip_fraction_2026
        )
        else
        "n/a"
    ),
)


print(
    "Max qualifying-weight error:",
    max_q_weight_error,
)


print(
    "Max Q1-level-weight error:",
    max_q1_weight_error,
)


print(
    "Duplicate count:",
    int(
        duplicate_table[
            "duplicates"
        ].sum()
    ),
)


print(
    "Missing tenure count:",
    int(
        tenure_missing[
            "missing_tenure"
        ].sum()
    ),
)


print(
    "\nHARD FLAGS:"
)

if hard_flags:
    for flag in hard_flags:
        print(
            "  -",
            flag,
        )
else:
    print(
        "  none"
    )


print(
    "\nSOFT FLAGS:"
)

if soft_flags:
    for flag in soft_flags:
        print(
            "  -",
            flag,
        )
else:
    print(
        "  none"
    )


print(
    "\nDECISION RULE:"
)

print(
    "  If HARD FLAGS = none, qualifying weights are exact, "
    "tenure is complete, duplicates are zero, and any missing "
    "team-race pairs have an obvious data reason, the 2026 "
    "modern likelihood is structurally ready."
)

print(
    "  Do NOT sample yet if the first-race tenure table exposes "
    "a constructor-continuity definition we want to change "
    "(especially a rebrand/new identity), because that choice "
    "must be fixed before building the 2026 state space."
)


print(
    "\nSaved audit tables ->"
)

print(
    audit_dir.resolve()
)

print(
    "\nStage 32B complete."
)


In [ ]:
# ==========================================================
# STAGE 32C — FULL HISTORICAL TEAM-LINEAGE / TENURE AUDIT
# ==========================================================
#
# RUN THIS NOW with the ORIGINAL Stage 3 still in place.
#
# DO NOT replace Stage 3 yet.
# DO NOT rebuild / sample the model yet.
#
# PURPOSE
# -------
# The current tenure covariate counts prior starts by literal
# (driverId, constructorId). That means a constructor identity
# change can reset team familiarity even when the driver stayed
# with the same underlying organisation.
#
# This audit:
#
#   1. lists every constructor identity and active years;
#   2. lists every consecutive-season constructor switch made by
#      a continuing driver;
#   3. applies a PROVISIONAL, year-aware team-lineage map;
#   4. compares current constructorId-based tenure against
#      lineage-based tenure WITHOUT changing the model;
#   5. identifies every driver/race whose tenure would change;
#   6. quantifies exact changes to the modern pace/quali tenure
#      covariates already used in Stages 4–6;
#   7. quantifies historical 1982–95 tenure changes where possible;
#   8. highlights candidate constructor transitions not covered by
#      the provisional map so we can inspect them manually.
#
# IMPORTANT
# ---------
# This cell is READ ONLY.
# It does not overwrite:
#   raw
#   driver_event_cov
#   df_pace / df_quali
#   pace_contrast / pace_level
#   q_contrast / q_level
#
# The lineage rules below are deliberately explicit and YEAR-AWARE.
# That is necessary because some constructorRefs were reused by
# historically different teams (e.g. Renault / Alfa Romeo eras).
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==========================================================
# 0. REQUIRED OBJECTS
# ==========================================================

required = [
    "CFG",
    "raw",
    "driver_event_cov",
    "df_pace",
    "df_quali",
    "pace_contrast",
    "pace_level",
    "q_contrast",
    "q_level",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Stage 32C requires the original Stage 3 and Stages 4–6. "
        "Missing:\n  "
        + "\n  ".join(missing)
    )


# ==========================================================
# 1. PROVISIONAL YEAR-AWARE TEAM LINEAGES
# ==========================================================
#
# These rules are ONLY hypotheses for the audit.
#
# We are trying to represent continuity of the underlying team /
# entrant / operation for the TEAM-FAMILIARITY covariate.
#
# Car performance would STILL use the literal constructor-season
# identity if we later adopt a lineage correction.
#
# A rule applies only when:
#
#   constructorRef matches
#   AND start_year <= year <= end_year
#
# This avoids linking unrelated historical uses of the same brand.
#
# confidence:
#   "strong"  = clear operational/rebrand continuity
#   "review"  = useful to inspect, but we should decide explicitly
#               before using it in a production refit
# ==========================================================

LINEAGE_SEGMENTS = [

    # ------------------------------------------------------
    # Enstone:
    # Toleman -> Benetton -> Renault -> Lotus F1 ->
    # Renault -> Alpine
    #
    # The year windows deliberately exclude the original
    # 1977–85 Renault works team from the modern Enstone Renault.
    # ------------------------------------------------------
    dict(
        lineage="ENSTONE",
        constructor_refs=("toleman",),
        start_year=1982,
        end_year=1985,
        confidence="strong",
        note="Toleman",
    ),
    dict(
        lineage="ENSTONE",
        constructor_refs=("benetton",),
        start_year=1986,
        end_year=2001,
        confidence="strong",
        note="Benetton",
    ),
    dict(
        lineage="ENSTONE",
        constructor_refs=("renault",),
        start_year=2002,
        end_year=2011,
        confidence="strong",
        note="Renault / Lotus Renault GP era",
    ),
    dict(
        lineage="ENSTONE",
        constructor_refs=("lotus_f1",),
        start_year=2012,
        end_year=2015,
        confidence="strong",
        note="Lotus F1",
    ),
    dict(
        lineage="ENSTONE",
        constructor_refs=("renault",),
        start_year=2016,
        end_year=2020,
        confidence="strong",
        note="Renault return",
    ),
    dict(
        lineage="ENSTONE",
        constructor_refs=("alpine",),
        start_year=2021,
        end_year=2100,
        confidence="strong",
        note="Alpine",
    ),

    # ------------------------------------------------------
    # Brackley lineage:
    # Tyrrell -> BAR -> Honda -> Brawn -> Mercedes
    #
    # The Tyrrell -> BAR link is retained as REVIEW because
    # the entry/ownership lineage is clear but the operational
    # continuity is less direct than Honda -> Brawn -> Mercedes.
    # ------------------------------------------------------
    dict(
        lineage="BRACKLEY",
        constructor_refs=("tyrrell",),
        start_year=1982,
        end_year=1998,
        confidence="review",
        note="Tyrrell",
    ),
    dict(
        lineage="BRACKLEY",
        constructor_refs=("bar",),
        start_year=1999,
        end_year=2005,
        confidence="strong",
        note="BAR",
    ),
    dict(
        lineage="BRACKLEY",
        constructor_refs=("honda",),
        start_year=2006,
        end_year=2008,
        confidence="strong",
        note="Honda",
    ),
    dict(
        lineage="BRACKLEY",
        constructor_refs=("brawn",),
        start_year=2009,
        end_year=2009,
        confidence="strong",
        note="Brawn",
    ),
    dict(
        lineage="BRACKLEY",
        constructor_refs=("mercedes",),
        start_year=2010,
        end_year=2100,
        confidence="strong",
        note="Mercedes",
    ),

    # ------------------------------------------------------
    # Milton Keynes:
    # Stewart -> Jaguar -> Red Bull
    # ------------------------------------------------------
    dict(
        lineage="MILTON_KEYNES",
        constructor_refs=("stewart",),
        start_year=1997,
        end_year=1999,
        confidence="strong",
        note="Stewart",
    ),
    dict(
        lineage="MILTON_KEYNES",
        constructor_refs=("jaguar",),
        start_year=2000,
        end_year=2004,
        confidence="strong",
        note="Jaguar",
    ),
    dict(
        lineage="MILTON_KEYNES",
        constructor_refs=("red_bull",),
        start_year=2005,
        end_year=2100,
        confidence="strong",
        note="Red Bull",
    ),

    # ------------------------------------------------------
    # Faenza:
    # Minardi -> Toro Rosso -> AlphaTauri -> RB
    # ------------------------------------------------------
    dict(
        lineage="FAENZA",
        constructor_refs=("minardi",),
        start_year=1985,
        end_year=2005,
        confidence="strong",
        note="Minardi",
    ),
    dict(
        lineage="FAENZA",
        constructor_refs=("toro_rosso",),
        start_year=2006,
        end_year=2019,
        confidence="strong",
        note="Toro Rosso",
    ),
    dict(
        lineage="FAENZA",
        constructor_refs=("alphatauri",),
        start_year=2020,
        end_year=2023,
        confidence="strong",
        note="AlphaTauri",
    ),
    dict(
        lineage="FAENZA",
        constructor_refs=("rb",),
        start_year=2024,
        end_year=2100,
        confidence="strong",
        note="RB / Racing Bulls identity in source",
    ),

    # ------------------------------------------------------
    # Silverstone:
    # Jordan -> Midland -> Spyker -> Force India ->
    # Racing Point -> Aston Martin
    # ------------------------------------------------------
    dict(
        lineage="SILVERSTONE",
        constructor_refs=("jordan",),
        start_year=1991,
        end_year=2005,
        confidence="strong",
        note="Jordan",
    ),
    dict(
        lineage="SILVERSTONE",
        constructor_refs=("mf1", "midland"),
        start_year=2006,
        end_year=2006,
        confidence="strong",
        note="Midland / MF1",
    ),
    dict(
        lineage="SILVERSTONE",
        constructor_refs=("spyker",),
        start_year=2007,
        end_year=2007,
        confidence="strong",
        note="Spyker",
    ),
    dict(
        lineage="SILVERSTONE",
        constructor_refs=("force_india",),
        start_year=2008,
        end_year=2018,
        confidence="strong",
        note="Force India",
    ),
    dict(
        lineage="SILVERSTONE",
        constructor_refs=("racing_point",),
        start_year=2019,
        end_year=2020,
        confidence="strong",
        note="Racing Point",
    ),
    dict(
        lineage="SILVERSTONE",
        constructor_refs=("aston_martin",),
        start_year=2021,
        end_year=2100,
        confidence="strong",
        note="Aston Martin",
    ),

    # ------------------------------------------------------
    # Hinwil:
    # Sauber -> BMW Sauber -> Sauber -> Alfa Romeo ->
    # Sauber -> Audi
    #
    # The 2019–23 Alfa rule is year-limited so it cannot connect
    # the unrelated historic Alfa Romeo works team.
    # ------------------------------------------------------
    dict(
        lineage="HINWIL",
        constructor_refs=("sauber",),
        start_year=1993,
        end_year=2005,
        confidence="strong",
        note="Sauber",
    ),
    dict(
        lineage="HINWIL",
        constructor_refs=("bmw_sauber",),
        start_year=2006,
        end_year=2009,
        confidence="strong",
        note="BMW Sauber",
    ),
    dict(
        lineage="HINWIL",
        constructor_refs=("sauber",),
        start_year=2010,
        end_year=2018,
        confidence="strong",
        note="Sauber",
    ),
    dict(
        lineage="HINWIL",
        constructor_refs=("alfa", "alfa_romeo"),
        start_year=2019,
        end_year=2023,
        confidence="strong",
        note="Alfa Romeo-branded Sauber",
    ),
    dict(
        lineage="HINWIL",
        constructor_refs=("sauber",),
        start_year=2024,
        end_year=2025,
        confidence="strong",
        note="Sauber",
    ),
    dict(
        lineage="HINWIL",
        constructor_refs=("audi",),
        start_year=2026,
        end_year=2100,
        confidence="strong",
        note="Audi",
    ),

    # ------------------------------------------------------
    # Virgin / Marussia / Manor
    # ------------------------------------------------------
    dict(
        lineage="VIRGIN_MARUSSIA_MANOR",
        constructor_refs=("virgin",),
        start_year=2010,
        end_year=2011,
        confidence="strong",
        note="Virgin",
    ),
    dict(
        lineage="VIRGIN_MARUSSIA_MANOR",
        constructor_refs=("marussia",),
        start_year=2012,
        end_year=2015,
        confidence="strong",
        note="Marussia / Manor Marussia",
    ),
    dict(
        lineage="VIRGIN_MARUSSIA_MANOR",
        constructor_refs=("manor",),
        start_year=2016,
        end_year=2016,
        confidence="strong",
        note="Manor",
    ),

    # ------------------------------------------------------
    # Lotus Racing / Team Lotus / Caterham
    # ------------------------------------------------------
    dict(
        lineage="TEAM_LOTUS_CATERHAM",
        constructor_refs=("lotus_racing",),
        start_year=2010,
        end_year=2010,
        confidence="strong",
        note="Lotus Racing",
    ),
    dict(
        lineage="TEAM_LOTUS_CATERHAM",
        constructor_refs=("team_lotus",),
        start_year=2011,
        end_year=2011,
        confidence="strong",
        note="Team Lotus",
    ),
    dict(
        lineage="TEAM_LOTUS_CATERHAM",
        constructor_refs=("caterham",),
        start_year=2012,
        end_year=2014,
        confidence="strong",
        note="Caterham",
    ),

    # ------------------------------------------------------
    # Ligier -> Prost
    # ------------------------------------------------------
    dict(
        lineage="LIGIER_PROST",
        constructor_refs=("ligier",),
        start_year=1982,
        end_year=1996,
        confidence="strong",
        note="Ligier",
    ),
    dict(
        lineage="LIGIER_PROST",
        constructor_refs=("prost",),
        start_year=1997,
        end_year=2001,
        confidence="strong",
        note="Prost",
    ),

    # ------------------------------------------------------
    # Arrows -> Footwork -> Arrows
    # ------------------------------------------------------
    dict(
        lineage="ARROWS_FOOTWORK",
        constructor_refs=("arrows",),
        start_year=1982,
        end_year=1990,
        confidence="strong",
        note="Arrows",
    ),
    dict(
        lineage="ARROWS_FOOTWORK",
        constructor_refs=("footwork",),
        start_year=1991,
        end_year=1996,
        confidence="strong",
        note="Footwork",
    ),
    dict(
        lineage="ARROWS_FOOTWORK",
        constructor_refs=("arrows",),
        start_year=1997,
        end_year=2002,
        confidence="strong",
        note="Arrows",
    ),

    # ------------------------------------------------------
    # March / Leyton House / March
    #
    # Year-limited deliberately: it does NOT merge the separate
    # early-1980s March entry with the later 1987 operation.
    # ------------------------------------------------------
    dict(
        lineage="MARCH_LEYTON",
        constructor_refs=("march",),
        start_year=1987,
        end_year=1989,
        confidence="strong",
        note="March",
    ),
    dict(
        lineage="MARCH_LEYTON",
        constructor_refs=("leyton", "leyton_house"),
        start_year=1990,
        end_year=1991,
        confidence="strong",
        note="Leyton House",
    ),
    dict(
        lineage="MARCH_LEYTON",
        constructor_refs=("march",),
        start_year=1992,
        end_year=1992,
        confidence="strong",
        note="March return",
    ),

    # ------------------------------------------------------
    # Scuderia Italia:
    # Dallara chassis -> Lola chassis.
    #
    # This is included because constructor identity can differ
    # even though the entrant/team itself continued.
    # ------------------------------------------------------
    dict(
        lineage="SCUDERIA_ITALIA",
        constructor_refs=("dallara",),
        start_year=1988,
        end_year=1992,
        confidence="strong",
        note="Scuderia Italia / Dallara",
    ),
    dict(
        lineage="SCUDERIA_ITALIA",
        constructor_refs=("lola",),
        start_year=1993,
        end_year=1993,
        confidence="strong",
        note="Scuderia Italia / Lola",
    ),
]


# ==========================================================
# 2. LOOKUPS
# ==========================================================

drivers_lookup = (
    raw.drivers[
        [
            "driverId",
            "driverRef",
        ]
    ]
    .dropna()
    .copy()
)

drivers_lookup[
    "driverId"
] = pd.to_numeric(
    drivers_lookup[
        "driverId"
    ],
    errors="raise",
).astype(int)

driver_ref_map = dict(
    zip(
        drivers_lookup[
            "driverId"
        ],
        drivers_lookup[
            "driverRef"
        ].astype(str),
    )
)


constructors_lookup = (
    raw.constructors.copy()
)

constructors_lookup[
    "constructorId"
] = pd.to_numeric(
    constructors_lookup[
        "constructorId"
    ],
    errors="raise",
).astype(int)

if "constructorRef" not in constructors_lookup.columns:
    raise RuntimeError(
        "raw.constructors must contain constructorRef."
    )

constructor_ref_map = dict(
    zip(
        constructors_lookup[
            "constructorId"
        ],
        constructors_lookup[
            "constructorRef"
        ].astype(str),
    )
)

constructor_name_map = {}

if "name" in constructors_lookup.columns:
    constructor_name_map = dict(
        zip(
            constructors_lookup[
                "constructorId"
            ],
            constructors_lookup[
                "name"
            ].astype(str),
        )
    )


def dname(driver_id):
    return driver_ref_map.get(
        int(driver_id),
        str(driver_id),
    )


def cref(constructor_id):
    return constructor_ref_map.get(
        int(constructor_id),
        str(constructor_id),
    )


def cname(constructor_id):
    return constructor_name_map.get(
        int(constructor_id),
        cref(
            constructor_id
        ),
    )


# ==========================================================
# 3. BUILD ONE ROW PER ACTUAL DRIVER START
# ==========================================================

race_info = raw.races[
    [
        col
        for col in [
            "raceId",
            "year",
            "round",
            "date",
            "name",
        ]
        if col in raw.races.columns
    ]
].copy()

race_info[
    "raceId"
] = pd.to_numeric(
    race_info[
        "raceId"
    ],
    errors="raise",
).astype(int)

race_info[
    "year"
] = pd.to_numeric(
    race_info[
        "year"
    ],
    errors="coerce",
).astype("Int64")

if "round" in race_info.columns:
    race_info[
        "round"
    ] = pd.to_numeric(
        race_info[
            "round"
        ],
        errors="coerce",
    ).astype("Int64")

if "date" in race_info.columns:
    race_info[
        "date"
    ] = pd.to_datetime(
        race_info[
            "date"
        ],
        errors="coerce",
    )


starts = (
    raw.results[
        [
            "raceId",
            "driverId",
            "constructorId",
        ]
    ]
    .drop_duplicates(
        subset=[
            "raceId",
            "driverId",
            "constructorId",
        ]
    )
    .copy()
)

for col in [
    "raceId",
    "driverId",
    "constructorId",
]:
    starts[
        col
    ] = pd.to_numeric(
        starts[
            col
        ],
        errors="raise",
    ).astype(int)

starts = (
    starts
    .merge(
        race_info,
        on="raceId",
        how="left",
        validate="many_to_one",
    )
)

sort_cols = [
    col
    for col in [
        "date",
        "year",
        "round",
        "raceId",
        "driverId",
    ]
    if col in starts.columns
]

starts = (
    starts
    .sort_values(
        sort_cols
    )
    .reset_index(
        drop=True
    )
)

starts[
    "driver"
] = (
    starts[
        "driverId"
    ]
    .map(
        dname
    )
)

starts[
    "constructorRef"
] = (
    starts[
        "constructorId"
    ]
    .map(
        cref
    )
)

starts[
    "constructor"
] = (
    starts[
        "constructorId"
    ]
    .map(
        cname
    )
)


# Current Stage-3 tenure.
starts[
    "current_team_starts_before"
] = (
    starts
    .groupby(
        [
            "driverId",
            "constructorId",
        ]
    )
    .cumcount()
)

starts[
    "current_tenure_raw_rebuilt"
] = np.tanh(
    starts[
        "current_team_starts_before"
    ]
    /
    float(
        CFG.tenure_scale_races
    )
)


# Verify that the audit rebuild matches the currently loaded
# driver_event_cov exactly.
starts = starts.merge(
    driver_event_cov.rename(
        columns={
            "tenure_raw":
                "current_tenure_raw_loaded",
        }
    ),
    on=[
        "raceId",
        "driverId",
        "constructorId",
    ],
    how="left",
    validate="one_to_one",
)

max_current_rebuild_error = float(
    np.nanmax(
        np.abs(
            starts[
                "current_tenure_raw_rebuilt"
            ].to_numpy(
                dtype=float
            )
            -
            starts[
                "current_tenure_raw_loaded"
            ].to_numpy(
                dtype=float
            )
        )
    )
)

print(
    "========================================"
)
print(
    "STAGE 32C — FULL TEAM-LINEAGE / TENURE AUDIT"
)
print(
    "========================================"
)

print(
    "Driver-start rows:",
    f"{len(starts):,}",
)

print(
    "Max rebuilt-vs-loaded current tenure error:",
    max_current_rebuild_error,
)

if max_current_rebuild_error > 1e-12:
    raise RuntimeError(
        "The audit could not reproduce the currently loaded "
        "Stage-3 tenure covariate exactly."
    )


# ==========================================================
# 4. ASSIGN PROVISIONAL YEAR-AWARE LINEAGE KEYS
# ==========================================================

def provisional_lineage(
    constructor_ref,
    year,
):
    constructor_ref = str(
        constructor_ref
    )

    year = int(
        year
    )

    matches = []

    for rule in LINEAGE_SEGMENTS:

        if (
            constructor_ref
            in
            rule[
                "constructor_refs"
            ]
            and
            int(
                rule[
                    "start_year"
                ]
            )
            <=
            year
            <=
            int(
                rule[
                    "end_year"
                ]
            )
        ):
            matches.append(
                rule
            )

    if len(
        matches
    ) > 1:
        raise RuntimeError(
            "Overlapping provisional lineage rules for "
            f"{constructor_ref}, {year}: "
            f"{matches}"
        )

    if len(
        matches
    ) == 1:
        rule = matches[
            0
        ]

        return (
            "LINEAGE::"
            +
            rule[
                "lineage"
            ],
            rule[
                "lineage"
            ],
            rule[
                "confidence"
            ],
            rule[
                "note"
            ],
        )

    # Default: no lineage merging.
    return (
        "CONSTRUCTOR::"
        +
        constructor_ref,
        None,
        "literal",
        "No provisional lineage rule",
    )


lineage_fields = starts.apply(
    lambda r:
        provisional_lineage(
            r[
                "constructorRef"
            ],
            r[
                "year"
            ],
        ),
    axis=1,
    result_type="expand",
)

lineage_fields.columns = [
    "candidate_team_key",
    "candidate_lineage",
    "lineage_confidence",
    "lineage_note",
]

starts = pd.concat(
    [
        starts,
        lineage_fields,
    ],
    axis=1,
)


# Candidate lineage tenure.
starts[
    "candidate_team_starts_before"
] = (
    starts
    .groupby(
        [
            "driverId",
            "candidate_team_key",
        ]
    )
    .cumcount()
)

starts[
    "candidate_tenure_raw"
] = np.tanh(
    starts[
        "candidate_team_starts_before"
    ]
    /
    float(
        CFG.tenure_scale_races
    )
)

starts[
    "delta_prior_starts"
] = (
    starts[
        "candidate_team_starts_before"
    ]
    -
    starts[
        "current_team_starts_before"
    ]
)

starts[
    "delta_tenure_raw"
] = (
    starts[
        "candidate_tenure_raw"
    ]
    -
    starts[
        "current_tenure_raw_rebuilt"
    ]
)

starts[
    "tenure_changes"
] = (
    np.abs(
        starts[
            "delta_tenure_raw"
        ]
    )
    >
    1e-12
)


# ==========================================================
# 5. CONSTRUCTOR IDENTITY ACTIVE-YEAR AUDIT
# ==========================================================

constructor_active = (
    starts
    .groupby(
        [
            "constructorId",
            "constructorRef",
            "constructor",
        ],
        as_index=False,
    )
    .agg(
        first_year=(
            "year",
            "min",
        ),
        last_year=(
            "year",
            "max",
        ),
        starts=(
            "raceId",
            "size",
        ),
        drivers=(
            "driverId",
            "nunique",
        ),
    )
    .sort_values(
        [
            "first_year",
            "constructorRef",
        ]
    )
)


print(
    "\n========================================"
)
print(
    "CONSTRUCTOR IDENTITIES / ACTIVE YEARS"
)
print(
    "========================================"
)

display(
    constructor_active
)


# ==========================================================
# 6. ALL CONSECUTIVE-SEASON DRIVER CONSTRUCTOR SWITCHES
# ==========================================================
#
# This table is intentionally broad. It contains real transfers
# as well as rebrands. We use it to spot lineage changes that the
# provisional map may have missed.
# ==========================================================

driver_seasons = (
    starts
    .sort_values(
        sort_cols
    )
    .groupby(
        [
            "driverId",
            "year",
        ],
        as_index=False,
    )
    .agg(
        first_constructorId=(
            "constructorId",
            "first",
        ),
        last_constructorId=(
            "constructorId",
            "last",
        ),
        first_raceId=(
            "raceId",
            "first",
        ),
        last_raceId=(
            "raceId",
            "last",
        ),
    )
)

driver_seasons[
    "prev_year"
] = (
    driver_seasons
    .groupby(
        "driverId"
    )[
        "year"
    ]
    .shift(1)
)

driver_seasons[
    "prev_last_constructorId"
] = (
    driver_seasons
    .groupby(
        "driverId"
    )[
        "last_constructorId"
    ]
    .shift(1)
)

consecutive_switches = driver_seasons[
    (
        driver_seasons[
            "year"
        ]
        ==
        driver_seasons[
            "prev_year"
        ]
        +
        1
    )
    &
    (
        driver_seasons[
            "first_constructorId"
        ]
        !=
        driver_seasons[
            "prev_last_constructorId"
        ]
    )
].copy()

consecutive_switches[
    "from_constructorId"
] = (
    consecutive_switches[
        "prev_last_constructorId"
    ]
    .astype(int)
)

consecutive_switches[
    "to_constructorId"
] = (
    consecutive_switches[
        "first_constructorId"
    ]
    .astype(int)
)

consecutive_switches[
    "driver"
] = (
    consecutive_switches[
        "driverId"
    ]
    .map(
        dname
    )
)

consecutive_switches[
    "from_ref"
] = (
    consecutive_switches[
        "from_constructorId"
    ]
    .map(
        cref
    )
)

consecutive_switches[
    "to_ref"
] = (
    consecutive_switches[
        "to_constructorId"
    ]
    .map(
        cref
    )
)


def lineage_for_ref_year(
    ref,
    year,
):
    return provisional_lineage(
        ref,
        year,
    )[1]


consecutive_switches[
    "from_lineage"
] = [
    lineage_for_ref_year(
        ref,
        int(
            year
        )
        -
        1,
    )
    for ref, year
    in zip(
        consecutive_switches[
            "from_ref"
        ],
        consecutive_switches[
            "year"
        ],
    )
]

consecutive_switches[
    "to_lineage"
] = [
    lineage_for_ref_year(
        ref,
        int(
            year
        ),
    )
    for ref, year
    in zip(
        consecutive_switches[
            "to_ref"
        ],
        consecutive_switches[
            "year"
        ],
    )
]

consecutive_switches[
    "mapped_same_lineage"
] = (
    consecutive_switches[
        "from_lineage"
    ].notna()
    &
    (
        consecutive_switches[
            "from_lineage"
        ]
        ==
        consecutive_switches[
            "to_lineage"
        ]
    )
)


switch_summary = (
    consecutive_switches
    .groupby(
        [
            "year",
            "from_ref",
            "to_ref",
            "mapped_same_lineage",
            "from_lineage",
            "to_lineage",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        n_drivers=(
            "driverId",
            "nunique",
        ),
        drivers=(
            "driver",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            map(
                                str,
                                x,
                            )
                        )
                    )
                ),
        ),
    )
    .sort_values(
        [
            "year",
            "from_ref",
            "to_ref",
        ]
    )
)


print(
    "\n========================================"
)
print(
    "ALL CONSECUTIVE-SEASON CONSTRUCTOR SWITCHES"
)
print(
    "========================================"
)

print(
    "This includes genuine transfers. "
    "mapped_same_lineage=True marks provisional rebrand/team continuity."
)

display(
    switch_summary
)


# ==========================================================
# 7. PROVISIONAL SAME-LINEAGE TRANSITIONS
# ==========================================================

same_lineage_switches = (
    consecutive_switches[
        consecutive_switches[
            "mapped_same_lineage"
        ]
    ]
    .copy()
)


print(
    "\n========================================"
)
print(
    "PROVISIONAL SAME-TEAM / REBRAND TRANSITIONS"
)
print(
    "========================================"
)

if same_lineage_switches.empty:
    print(
        "None found."
    )
else:
    display(
        same_lineage_switches[
            [
                "year",
                "driver",
                "from_ref",
                "to_ref",
                "from_lineage",
            ]
        ]
        .sort_values(
            [
                "year",
                "from_ref",
                "driver",
            ]
        )
    )


# ==========================================================
# 8. FIRST AFFECTED START AFTER EACH LITERAL CONSTRUCTOR RESET
# ==========================================================

affected = starts[
    starts[
        "tenure_changes"
    ]
].copy()


first_affected = (
    affected
    .sort_values(
        sort_cols
    )
    .groupby(
        [
            "driverId",
            "constructorId",
            "candidate_team_key",
        ],
        as_index=False,
    )
    .first()
)


first_affected[
    "implied_current_prior_starts"
] = (
    first_affected[
        "current_team_starts_before"
    ]
)

first_affected[
    "implied_lineage_prior_starts"
] = (
    first_affected[
        "candidate_team_starts_before"
    ]
)


print(
    "\n========================================"
)
print(
    "FIRST AFFECTED START AFTER CONSTRUCTOR-ID RESET"
)
print(
    "========================================"
)

if first_affected.empty:
    print(
        "No tenure differences under the provisional map."
    )
else:
    display(
        first_affected[
            [
                "year",
                "driver",
                "constructorRef",
                "candidate_lineage",
                "lineage_confidence",
                "current_team_starts_before",
                "candidate_team_starts_before",
                "current_tenure_raw_rebuilt",
                "candidate_tenure_raw",
                "delta_tenure_raw",
            ]
        ]
        .sort_values(
            [
                "year",
                "candidate_lineage",
                "driver",
            ]
        )
    )


# ==========================================================
# 9. ALL AFFECTED DRIVERS — SUMMARY
# ==========================================================

affected_driver_summary = (
    affected
    .groupby(
        [
            "candidate_lineage",
            "lineage_confidence",
            "driverId",
            "driver",
        ],
        as_index=False,
    )
    .agg(
        first_affected_year=(
            "year",
            "min",
        ),
        last_affected_year=(
            "year",
            "max",
        ),
        affected_starts=(
            "raceId",
            "size",
        ),
        max_extra_prior_starts=(
            "delta_prior_starts",
            "max",
        ),
        max_abs_delta_tenure=(
            "delta_tenure_raw",
            lambda x:
                float(
                    np.max(
                        np.abs(
                            x
                        )
                    )
                ),
        ),
        mean_delta_tenure=(
            "delta_tenure_raw",
            "mean",
        ),
    )
    .sort_values(
        [
            "max_abs_delta_tenure",
            "affected_starts",
        ],
        ascending=[
            False,
            False,
        ],
    )
)


print(
    "\n========================================"
)
print(
    "AFFECTED DRIVER SUMMARY"
)
print(
    "========================================"
)

display(
    affected_driver_summary
)


# ==========================================================
# 10. LINEAGE-LEVEL IMPACT SUMMARY
# ==========================================================

lineage_impact = (
    affected
    .groupby(
        [
            "candidate_lineage",
            "lineage_confidence",
        ],
        as_index=False,
    )
    .agg(
        affected_driver_starts=(
            "raceId",
            "size",
        ),
        affected_drivers=(
            "driverId",
            "nunique",
        ),
        first_year=(
            "year",
            "min",
        ),
        last_year=(
            "year",
            "max",
        ),
        max_extra_prior_starts=(
            "delta_prior_starts",
            "max",
        ),
        max_abs_delta_tenure=(
            "delta_tenure_raw",
            lambda x:
                float(
                    np.max(
                        np.abs(
                            x
                        )
                    )
                ),
        ),
        mean_abs_delta_tenure=(
            "delta_tenure_raw",
            lambda x:
                float(
                    np.mean(
                        np.abs(
                            x
                        )
                    )
                ),
        ),
    )
    .sort_values(
        "max_abs_delta_tenure",
        ascending=False,
    )
)


print(
    "\n========================================"
)
print(
    "LINEAGE-LEVEL TENURE IMPACT"
)
print(
    "========================================"
)

display(
    lineage_impact
)


# ==========================================================
# 11. BUILD CANDIDATE TENURE LOOKUP
# ==========================================================

candidate_cov = starts[
    [
        "raceId",
        "driverId",
        "constructorId",
        "candidate_tenure_raw",
    ]
].copy()


# ==========================================================
# 12. EXACT MODERN PACE TENURE-COVARIATE IMPACT
# ==========================================================

pace_candidate = (
    df_pace
    .drop(
        columns=[
            "candidate_tenure_raw",
            "candidate_tenure_c",
        ],
        errors="ignore",
    )
    .merge(
        candidate_cov,
        on=[
            "raceId",
            "driverId",
            "constructorId",
        ],
        how="left",
        validate="many_to_one",
    )
)

pace_candidate[
    "candidate_tenure_raw"
] = (
    pace_candidate[
        "candidate_tenure_raw"
    ]
    .fillna(
        0.0
    )
)

pace_candidate[
    "candidate_tenure_c"
] = (
    pace_candidate[
        "candidate_tenure_raw"
    ]
    -
    pace_candidate
    .groupby(
        "raceId"
    )[
        "candidate_tenure_raw"
    ]
    .transform(
        "mean"
    )
)


pace_c_lookup = (
    pace_candidate[
        [
            "raceId",
            "driverId",
            "candidate_tenure_c",
        ]
    ]
    .drop_duplicates(
        [
            "raceId",
            "driverId",
        ]
    )
)


def attach_pair_candidate_tenure(
    pair_df,
    driver_lookup,
    *,
    session_col=None,
):
    out = pair_df.copy()

    key_cols = [
        "raceId",
        "driverId",
    ]

    if session_col is not None:
        key_cols.append(
            session_col
        )

    a_lookup = driver_lookup.rename(
        columns={
            "driverId":
                "driverId_a",
            "candidate_tenure_c":
                "candidate_tenure_c_a",
        }
    )

    b_lookup = driver_lookup.rename(
        columns={
            "driverId":
                "driverId_b",
            "candidate_tenure_c":
                "candidate_tenure_c_b",
        }
    )

    merge_a = [
        "raceId",
        "driverId_a",
    ]

    merge_b = [
        "raceId",
        "driverId_b",
    ]

    if session_col is not None:
        merge_a.append(
            session_col
        )
        merge_b.append(
            session_col
        )

    out = out.merge(
        a_lookup,
        on=merge_a,
        how="left",
        validate="many_to_one",
    )

    out = out.merge(
        b_lookup,
        on=merge_b,
        how="left",
        validate="many_to_one",
    )

    out[
        "candidate_tenure_diff"
    ] = (
        out[
            "candidate_tenure_c_a"
        ]
        -
        out[
            "candidate_tenure_c_b"
        ]
    )

    out[
        "candidate_tenure_mean"
    ] = (
        0.5
        *
        (
            out[
                "candidate_tenure_c_a"
            ]
            +
            out[
                "candidate_tenure_c_b"
            ]
        )
    )

    return out


pace_pairs_candidate = (
    attach_pair_candidate_tenure(
        pace_contrast,
        pace_c_lookup,
    )
)


# Pace contrast and level are the same team/race pairs.
# Merge candidate tenure to pace_level by exact pair orientation.
pace_level_candidate = (
    pace_level
    .merge(
        pace_pairs_candidate[
            [
                "raceId",
                "constructorId",
                "driverId_a",
                "driverId_b",
                "candidate_tenure_mean",
            ]
        ],
        on=[
            "raceId",
            "constructorId",
            "driverId_a",
            "driverId_b",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ==========================================================
# 13. EXACT MODERN QUALIFYING TENURE-COVARIATE IMPACT
# ==========================================================

quali_candidate = (
    df_quali
    .drop(
        columns=[
            "candidate_tenure_raw",
            "candidate_tenure_c",
        ],
        errors="ignore",
    )
    .merge(
        candidate_cov,
        on=[
            "raceId",
            "driverId",
            "constructorId",
        ],
        how="left",
        validate="many_to_one",
    )
)

quali_candidate[
    "candidate_tenure_raw"
] = (
    quali_candidate[
        "candidate_tenure_raw"
    ]
    .fillna(
        0.0
    )
)

quali_candidate[
    "candidate_tenure_c"
] = (
    quali_candidate[
        "candidate_tenure_raw"
    ]
    -
    quali_candidate
    .groupby(
        [
            "raceId",
            "session",
        ]
    )[
        "candidate_tenure_raw"
    ]
    .transform(
        "mean"
    )
)


quali_c_lookup = (
    quali_candidate[
        [
            "raceId",
            "session",
            "driverId",
            "candidate_tenure_c",
        ]
    ]
    .drop_duplicates(
        [
            "raceId",
            "session",
            "driverId",
        ]
    )
)


q_pairs_candidate = (
    attach_pair_candidate_tenure(
        q_contrast,
        quali_c_lookup,
        session_col=
            "session",
    )
)


# Q1 level rows: recover candidate tenure mean using Q1 driver values.
q1_lookup = (
    quali_c_lookup[
        quali_c_lookup[
            "session"
        ]
        ==
        "q1"
    ]
    .drop(
        columns=[
            "session",
        ]
    )
)


q1_pair_candidate = (
    attach_pair_candidate_tenure(
        q_level,
        q1_lookup,
    )
)


# ==========================================================
# 14. MODERN LIKELIHOOD IMPACT TABLE
# ==========================================================

modern_impact_rows = []


def impact_row(
    channel,
    current_values,
    candidate_values,
    years,
):
    current_values = np.asarray(
        current_values,
        dtype=float,
    )

    candidate_values = np.asarray(
        candidate_values,
        dtype=float,
    )

    years = np.asarray(
        years,
        dtype=int,
    )

    delta = (
        candidate_values
        -
        current_values
    )

    finite = (
        np.isfinite(
            current_values
        )
        &
        np.isfinite(
            candidate_values
        )
    )

    delta = delta[
        finite
    ]

    years_f = years[
        finite
    ]

    return {
        "channel":
            channel,

        "rows":
            int(
                finite.sum()
            ),

        "changed_rows":
            int(
                (
                    np.abs(
                        delta
                    )
                    >
                    1e-12
                ).sum()
            ),

        "changed_rows_1982_1995":
            int(
                (
                    (
                        years_f
                        <=
                        1995
                    )
                    &
                    (
                        np.abs(
                            delta
                        )
                        >
                        1e-12
                    )
                ).sum()
            ),

        "changed_rows_1996_plus":
            int(
                (
                    (
                        years_f
                        >=
                        1996
                    )
                    &
                    (
                        np.abs(
                            delta
                        )
                        >
                        1e-12
                    )
                ).sum()
            ),

        "mean_abs_covariate_change":
            (
                float(
                    np.mean(
                        np.abs(
                            delta
                        )
                    )
                )
                if len(
                    delta
                )
                else np.nan
            ),

        "median_abs_covariate_change":
            (
                float(
                    np.median(
                        np.abs(
                            delta
                        )
                    )
                )
                if len(
                    delta
                )
                else np.nan
            ),

        "max_abs_covariate_change":
            (
                float(
                    np.max(
                        np.abs(
                            delta
                        )
                    )
                )
                if len(
                    delta
                )
                else np.nan
            ),
    }


modern_impact_rows.append(
    impact_row(
        "pace_contrast tenure_diff",
        pace_pairs_candidate[
            "tenure_diff"
        ],
        pace_pairs_candidate[
            "candidate_tenure_diff"
        ],
        pace_pairs_candidate[
            "year"
        ],
    )
)

modern_impact_rows.append(
    impact_row(
        "pace_level tenure_mean",
        pace_level_candidate[
            "tenure_mean"
        ],
        pace_level_candidate[
            "candidate_tenure_mean"
        ],
        pace_level_candidate[
            "year"
        ],
    )
)

modern_impact_rows.append(
    impact_row(
        "q_contrast tenure_diff",
        q_pairs_candidate[
            "tenure_diff"
        ],
        q_pairs_candidate[
            "candidate_tenure_diff"
        ],
        q_pairs_candidate[
            "year"
        ],
    )
)

modern_impact_rows.append(
    impact_row(
        "q1_level tenure_mean",
        q1_pair_candidate[
            "tenure_mean"
        ],
        q1_pair_candidate[
            "candidate_tenure_mean"
        ],
        q1_pair_candidate[
            "year"
        ],
    )
)


modern_impact = pd.DataFrame(
    modern_impact_rows
)


print(
    "\n========================================"
)
print(
    "EXACT MODERN LIKELIHOOD TENURE-COVARIATE IMPACT"
)
print(
    "========================================"
)

display(
    modern_impact
)


# ==========================================================
# 15. TOP MODERN PAIR-ROW CHANGES
# ==========================================================

def top_pair_changes(
    df,
    current_col,
    candidate_col,
    label,
    n=30,
):
    out = df.copy()

    out[
        "abs_change"
    ] = (
        out[
            candidate_col
        ]
        -
        out[
            current_col
        ]
    ).abs()

    cols = [
        col
        for col in [
            "year",
            "raceId",
            "session",
            "constructorId",
            "driver_a",
            "driver_b",
            current_col,
            candidate_col,
            "abs_change",
        ]
        if col in out.columns
    ]

    print(
        "\n" + label
    )

    display(
        out[
            cols
        ]
        .sort_values(
            "abs_change",
            ascending=False,
        )
        .head(
            n
        )
    )


print(
    "\n========================================"
)
print(
    "LARGEST MODERN COVARIATE CHANGES"
)
print(
    "========================================"
)

top_pair_changes(
    pace_pairs_candidate,
    "tenure_diff",
    "candidate_tenure_diff",
    "PACE CONTRAST",
)

top_pair_changes(
    pace_level_candidate,
    "tenure_mean",
    "candidate_tenure_mean",
    "PACE LEVEL",
)

top_pair_changes(
    q_pairs_candidate,
    "tenure_diff",
    "candidate_tenure_diff",
    "QUALIFYING CONTRAST",
)

top_pair_changes(
    q1_pair_candidate,
    "tenure_mean",
    "candidate_tenure_mean",
    "Q1 LEVEL",
)


# ==========================================================
# 16. HISTORICAL 1982–95 RAW/CENTRED TENURE PROXY
# ==========================================================
#
# Exact historical PL rows are only available if the later
# historical-preparation cell has already been run.
#
# We therefore:
#
#   A. always report affected 1982–95 actual-start rows;
#   B. if hist_quali / hist_race already exist, compute the
#      exact centred tenure changes for those likelihood rows.
# ==========================================================

historical_starts = starts[
    (
        starts[
            "year"
        ]
        >=
        int(
            CFG.historical_floor_year
        )
    )
    &
    (
        starts[
            "year"
        ]
        <=
        int(
            CFG.historical_ceiling_year
        )
    )
].copy()


hist_start_impact = pd.DataFrame(
    {
        "metric":
            [
                "historical actual-start rows",
                "historical rows with changed raw tenure",
                "historical affected drivers",
                "historical affected constructor identities",
                "max |raw tenure change|",
            ],

        "value":
            [
                len(
                    historical_starts
                ),

                int(
                    historical_starts[
                        "tenure_changes"
                    ].sum()
                ),

                int(
                    historical_starts.loc[
                        historical_starts[
                            "tenure_changes"
                        ],
                        "driverId",
                    ].nunique()
                ),

                int(
                    historical_starts.loc[
                        historical_starts[
                            "tenure_changes"
                        ],
                        "constructorId",
                    ].nunique()
                ),

                (
                    float(
                        historical_starts[
                            "delta_tenure_raw"
                        ]
                        .abs()
                        .max()
                    )
                    if len(
                        historical_starts
                    )
                    else np.nan
                ),
            ],
    }
)


print(
    "\n========================================"
)
print(
    "HISTORICAL 1982–95 RAW-TENURE IMPACT"
)
print(
    "========================================"
)

display(
    hist_start_impact
)


def exact_historical_impact(
    hist_df,
    label,
):
    h = (
        hist_df
        .drop(
            columns=[
                "candidate_tenure_raw",
                "candidate_tenure_c",
            ],
            errors="ignore",
        )
        .merge(
            candidate_cov,
            on=[
                "raceId",
                "driverId",
                "constructorId",
            ],
            how="left",
            validate="many_to_one",
        )
    )

    h[
        "candidate_tenure_raw"
    ] = (
        h[
            "candidate_tenure_raw"
        ]
        .fillna(
            0.0
        )
    )

    h[
        "candidate_tenure_c"
    ] = (
        h[
            "candidate_tenure_raw"
        ]
        -
        h.groupby(
            "raceId"
        )[
            "candidate_tenure_raw"
        ]
        .transform(
            "mean"
        )
    )

    h[
        "delta_tenure_c"
    ] = (
        h[
            "candidate_tenure_c"
        ]
        -
        h[
            "tenure_c"
        ]
    )

    return {
        "historical_channel":
            label,

        "rows":
            len(
                h
            ),

        "changed_rows":
            int(
                (
                    h[
                        "delta_tenure_c"
                    ].abs()
                    >
                    1e-12
                ).sum()
            ),

        "affected_races":
            int(
                h.loc[
                    h[
                        "delta_tenure_c"
                    ].abs()
                    >
                    1e-12,
                    "raceId",
                ].nunique()
            ),

        "max_abs_centered_change":
            float(
                h[
                    "delta_tenure_c"
                ]
                .abs()
                .max()
            ),

        "mean_abs_centered_change":
            float(
                h[
                    "delta_tenure_c"
                ]
                .abs()
                .mean()
            ),
    }


historical_exact_rows = []


if (
    "hist_quali"
    in globals()
    and
    isinstance(
        hist_quali,
        pd.DataFrame,
    )
    and
    len(
        hist_quali
    )
):
    historical_exact_rows.append(
        exact_historical_impact(
            hist_quali,
            "historical qualifying PL",
        )
    )


if (
    "hist_race"
    in globals()
    and
    isinstance(
        hist_race,
        pd.DataFrame,
    )
    and
    len(
        hist_race
    )
):
    historical_exact_rows.append(
        exact_historical_impact(
            hist_race,
            "historical race PL",
        )
    )


historical_exact = pd.DataFrame(
    historical_exact_rows
)


if len(
    historical_exact
):
    print(
        "\nExact historical likelihood-row impact:"
    )

    display(
        historical_exact
    )
else:
    print(
        "\nExact historical PL rows are not loaded yet, "
        "so the table above is the raw-start audit only."
    )

    print(
        "That is enough to detect constructor-ID resets. "
        "We can compute exact PL-centred changes after the "
        "historical-preparation stage if needed."
    )


# ==========================================================
# 17. TRANSITIONS NOT COVERED BY PROVISIONAL LINEAGE MAP
# ==========================================================
#
# This is our "what did we miss?" table.
#
# Genuine transfers will dominate it. What we are looking for
# manually are repeated from_ref -> to_ref changes that look like
# a rebrand/chassis/ownership continuation rather than a transfer.
# ==========================================================

unmapped_switches = (
    switch_summary[
        ~switch_summary[
            "mapped_same_lineage"
        ]
    ]
    .copy()
)


# Aggregate the same transition across years.
unmapped_transition_summary = (
    unmapped_switches
    .groupby(
        [
            "from_ref",
            "to_ref",
        ],
        as_index=False,
    )
    .agg(
        first_transition_year=(
            "year",
            "min",
        ),
        last_transition_year=(
            "year",
            "max",
        ),
        total_driver_transitions=(
            "n_drivers",
            "sum",
        ),
        years_seen=(
            "year",
            lambda x:
                ", ".join(
                    map(
                        str,
                        sorted(
                            set(
                                map(
                                    int,
                                    x,
                                )
                            )
                        ),
                    )
                ),
        ),
        drivers=(
            "drivers",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            ", ".join(
                                map(
                                    str,
                                    x,
                                )
                            )
                            .split(
                                ", "
                            )
                        )
                    )
                ),
        ),
    )
    .sort_values(
        [
            "total_driver_transitions",
            "first_transition_year",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


print(
    "\n========================================"
)
print(
    "UNMAPPED CONSECUTIVE-SEASON SWITCHES — MANUAL REVIEW"
)
print(
    "========================================"
)

print(
    "Most rows here are ordinary driver transfers. "
    "We are looking for obvious team/rebrand continuity "
    "that the provisional map missed."
)

display(
    unmapped_transition_summary
)


# ==========================================================
# 18. SPECIFIC TRANSITIONS THE USER ASKED ABOUT
# ==========================================================

specific_pairs = [
    (
        "renault",
        "alpine",
    ),
    (
        "toro_rosso",
        "alphatauri",
    ),
    (
        "alphatauri",
        "rb",
    ),
    (
        "honda",
        "brawn",
    ),
    (
        "brawn",
        "mercedes",
    ),
    (
        "racing_point",
        "aston_martin",
    ),
    (
        "sauber",
        "audi",
    ),
]


specific_rows = []


for from_ref, to_ref in specific_pairs:

    g = switch_summary[
        (
            switch_summary[
                "from_ref"
            ]
            ==
            from_ref
        )
        &
        (
            switch_summary[
                "to_ref"
            ]
            ==
            to_ref
        )
    ]

    specific_rows.append(
        {
            "from_ref":
                from_ref,

            "to_ref":
                to_ref,

            "observed_consecutive_driver_crossings":
                int(
                    g[
                        "n_drivers"
                    ].sum()
                )
                if len(
                    g
                )
                else
                0,

            "years":
                (
                    ", ".join(
                        map(
                            str,
                            sorted(
                                set(
                                    g[
                                        "year"
                                    ].astype(int)
                                )
                            ),
                        )
                    )
                    if len(
                        g
                    )
                    else
                    ""
                ),

            "drivers":
                (
                    ", ".join(
                        sorted(
                            set(
                                ", ".join(
                                    g[
                                        "drivers"
                                    ].astype(str)
                                )
                                .split(
                                    ", "
                                )
                            )
                        )
                    )
                    if len(
                        g
                    )
                    else
                    ""
                ),

            "provisional_same_lineage":
                bool(
                    g[
                        "mapped_same_lineage"
                    ].all()
                )
                if len(
                    g
                )
                else
                True,
        }
    )


specific_audit = pd.DataFrame(
    specific_rows
)


print(
    "\n========================================"
)
print(
    "SPECIFIC REBRAND / CONTINUITY CHECKS"
)
print(
    "========================================"
)

display(
    specific_audit
)


# ==========================================================
# 19. OPTIONAL GAMMA-TENURE SCALE INTERPRETATION
# ==========================================================
#
# If the accepted posterior is already loaded, translate a
# covariate change into approximate latent-model units using the
# frozen posterior mean gamma_tenure.
# ==========================================================

gamma_tenure_hat = np.nan


if "posterior_final" in globals():
    try:
        gamma_tenure_hat = float(
            posterior_final[
                "gamma_tenure"
            ]
            .mean()
            .values
        )
    except Exception:
        pass


if np.isnan(
    gamma_tenure_hat
) and "final_trace_2025" in globals():
    try:
        gamma_tenure_hat = float(
            final_trace_2025[
                "posterior"
            ][
                "gamma_tenure"
            ]
            .mean()
            .values
        )
    except Exception:
        pass


print(
    "\n========================================"
)
print(
    "TENURE EFFECT SCALE"
)
print(
    "========================================"
)


if np.isfinite(
    gamma_tenure_hat
):
    print(
        "Loaded posterior mean gamma_tenure:",
        gamma_tenure_hat,
    )

    modern_impact_with_mu = (
        modern_impact.copy()
    )

    modern_impact_with_mu[
        "max_abs_mu_shift_at_gamma_mean"
    ] = (
        modern_impact_with_mu[
            "max_abs_covariate_change"
        ]
        *
        gamma_tenure_hat
    )

    display(
        modern_impact_with_mu
    )

else:
    print(
        "Frozen posterior not currently loaded."
    )

    print(
        "No problem: the covariate changes above are the "
        "important part of this audit."
    )


# ==========================================================
# 20. SIMPLE VISUALS
# ==========================================================

if len(
    lineage_impact
):

    plot_df = (
        lineage_impact
        .sort_values(
            "max_abs_delta_tenure",
            ascending=True,
        )
    )

    plt.figure(
        figsize=(
            10,
            max(
                4,
                0.45
                *
                len(
                    plot_df
                )
                +
                1.5,
            ),
        )
    )

    plt.barh(
        plot_df[
            "candidate_lineage"
        ],
        plot_df[
            "max_abs_delta_tenure"
        ],
    )

    plt.xlabel(
        "Maximum absolute change in raw tenure covariate"
    )

    plt.ylabel(
        "Provisional team lineage"
    )

    plt.title(
        "How much literal constructor IDs can reset team familiarity"
    )

    plt.grid(
        axis="x",
        alpha=0.25,
    )

    plt.tight_layout()
    plt.show()


if len(
    affected_driver_summary
):

    top_driver_plot = (
        affected_driver_summary
        .sort_values(
            "max_abs_delta_tenure",
            ascending=False,
        )
        .head(
            25
        )
        .sort_values(
            "max_abs_delta_tenure",
            ascending=True,
        )
    )

    labels = (
        top_driver_plot[
            "driver"
        ]
        +
        " — "
        +
        top_driver_plot[
            "candidate_lineage"
        ]
    )

    plt.figure(
        figsize=(
            11,
            8,
        )
    )

    plt.barh(
        labels,
        top_driver_plot[
            "max_abs_delta_tenure"
        ],
    )

    plt.xlabel(
        "Maximum absolute raw-tenure change"
    )

    plt.ylabel(
        "Driver / lineage"
    )

    plt.title(
        "Drivers most affected by constructor-ID tenure resets"
    )

    plt.grid(
        axis="x",
        alpha=0.25,
    )

    plt.tight_layout()
    plt.show()


# ==========================================================
# 21. SAVE OUTPUTS
# ==========================================================

audit_dir = (
    Path(
        CFG.output_dir
    )
    /
    "validation_team_lineage_tenure_audit"
)

audit_dir.mkdir(
    parents=True,
    exist_ok=True,
)


constructor_active.to_csv(
    audit_dir
    /
    "constructor_active_years.csv",
    index=False,
)

switch_summary.to_csv(
    audit_dir
    /
    "all_consecutive_season_constructor_switches.csv",
    index=False,
)

same_lineage_switches.to_csv(
    audit_dir
    /
    "provisional_same_lineage_driver_crossings.csv",
    index=False,
)

first_affected.to_csv(
    audit_dir
    /
    "first_affected_start_after_constructor_reset.csv",
    index=False,
)

affected_driver_summary.to_csv(
    audit_dir
    /
    "affected_driver_summary.csv",
    index=False,
)

lineage_impact.to_csv(
    audit_dir
    /
    "lineage_impact_summary.csv",
    index=False,
)

modern_impact.to_csv(
    audit_dir
    /
    "modern_likelihood_covariate_impact.csv",
    index=False,
)

hist_start_impact.to_csv(
    audit_dir
    /
    "historical_raw_tenure_impact.csv",
    index=False,
)

if len(
    historical_exact
):
    historical_exact.to_csv(
        audit_dir
        /
        "historical_exact_likelihood_tenure_impact.csv",
        index=False,
    )

unmapped_transition_summary.to_csv(
    audit_dir
    /
    "unmapped_transition_manual_review.csv",
    index=False,
)

specific_audit.to_csv(
    audit_dir
    /
    "specific_rebrand_checks.csv",
    index=False,
)

affected.to_csv(
    audit_dir
    /
    "all_affected_driver_start_rows.csv",
    index=False,
)


# ==========================================================
# 22. TRIAGE SUMMARY
# ==========================================================

print(
    "\n========================================"
)
print(
    "STAGE 32C TRIAGE SUMMARY"
)
print(
    "========================================"
)


print(
    "Provisional lineage rules:",
    len(
        LINEAGE_SEGMENTS
    ),
)

print(
    "Driver starts with changed tenure:",
    int(
        starts[
            "tenure_changes"
        ].sum()
    ),
)

print(
    "Drivers affected:",
    int(
        starts.loc[
            starts[
                "tenure_changes"
            ],
            "driverId",
        ].nunique()
    ),
)

print(
    "Lineages producing at least one tenure change:",
    int(
        affected[
            "candidate_lineage"
        ].nunique()
    ),
)


if len(
    affected
):
    print(
        "Maximum raw-tenure change:",
        float(
            affected[
                "delta_tenure_raw"
            ]
            .abs()
            .max()
        ),
    )

    print(
        "Maximum extra prior-team starts recovered:",
        int(
            affected[
                "delta_prior_starts"
            ]
            .max()
        ),
    )


print(
    "\nModern likelihood covariate changes:"
)

for row in modern_impact.itertuples(
    index=False
):
    print(
        f"  {row.channel}: "
        f"{row.changed_rows}/{row.rows} rows changed; "
        f"max |Δ tenure covariate| = "
        f"{row.max_abs_covariate_change:.4f}"
    )


print(
    "\nHistorical 1982–95 actual-start rows with changed raw tenure:",
    int(
        historical_starts[
            "tenure_changes"
        ].sum()
    ),
)


print(
    "\nWHAT TO SEND BACK:"
)

print(
    "  1. LINEAGE-LEVEL TENURE IMPACT"
)

print(
    "  2. FIRST AFFECTED START AFTER CONSTRUCTOR-ID RESET"
)

print(
    "  3. EXACT MODERN LIKELIHOOD TENURE-COVARIATE IMPACT"
)

print(
    "  4. SPECIFIC REBRAND / CONTINUITY CHECKS"
)

print(
    "  5. the top of UNMAPPED CONSECUTIVE-SEASON SWITCHES"
)

print(
    "  6. STAGE 32C TRIAGE SUMMARY"
)


print(
    "\nIMPORTANT:"
)

print(
    "  Do NOT change Stage 3 yet. This audit deliberately "
    "uses a broad provisional lineage map so we can first "
    "see which historical cases materially matter."
)


print(
    "\nSaved audit outputs ->"
)

print(
    audit_dir.resolve()
)

print(
    "\nStage 32C complete."
)


In [ ]:
# ============================================================
# STAGE 32D — CORRECTED CONTINUOUS-ORGANISATIONAL-STINT TENURE
# ============================================================
#
# PURPOSE
# -------
# Replace the deliberately broad Stage 32C lineage experiment with
# the tenure definition we actually want:
#
#   tenure = tanh(
#       starts in driver's CURRENT CONTINUOUS organisational stint / 18
#   )
#
# Rules:
#   * rebrand / ownership change within the same operational team:
#         KEEP tenure
#   * driver races for another organisational lineage:
#         RESET tenure
#   * driver leaves F1 temporarily but does not start for another team:
#         KEEP the existing stint if they return to the same lineage
#   * old constructor names reused by unrelated teams are handled with
#         YEAR-SENSITIVE lineage rules.
#
# This cell:
#   1. reconstructs the ORIGINAL constructor-ID tenure from raw starts;
#   2. verifies that reconstruction against the existing likelihood
#      tables;
#   3. constructs corrected continuous-stint tenure;
#   4. audits known carry/reset examples;
#   5. quantifies the exact modern likelihood covariate changes;
#   6. audits 1982–95 historical raw/within-race tenure behaviour;
#   7. saves all diagnostic tables.
#
# IT DOES NOT MODIFY:
#   pace_contrast_all
#   pace_level_all
#   q_contrast_all
#   q_level_all_model
#
# IT DOES NOT SAMPLE.
#
# ============================================================

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ============================================================
# 0. CONFIG
# ============================================================

TENURE_SCALE = 18.0
TENURE_TOL = 1e-8

# Frozen 1982–2025 posterior mean, used only to translate a tenure
# covariate change into an approximate linear-predictor change.
GAMMA_TENURE_REFERENCE = 0.283208

OUTDIR_32D = Path(
    "model_outputs/"
    "joint_driver_car_v2_1_1982_2026_LIVE/"
    "validation_continuous_team_stint_tenure"
)
OUTDIR_32D.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "raw",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Stage 32D requires the 2026 raw/evidence environment.\n"
        "Missing objects:\n  "
        + "\n  ".join(missing_objects)
    )


# ============================================================
# 2. ROBUST ACCESS TO RAW TABLES
# ============================================================

def get_raw_table(*candidate_names):
    for name in candidate_names:
        if hasattr(raw, name):
            obj = getattr(raw, name)
            if isinstance(obj, pd.DataFrame):
                return obj.copy()
    raise RuntimeError(
        "Could not find any of these raw tables: "
        + ", ".join(candidate_names)
    )


races = get_raw_table("races")
results = get_raw_table("results")
drivers = get_raw_table("drivers")
constructors = get_raw_table("constructors")

try:
    statuses = get_raw_table("status", "statuses")
except RuntimeError:
    statuses = None


# ============================================================
# 3. BASIC COLUMN CHECKS
# ============================================================

for table_name, df, cols in [
    ("races", races, ["raceId", "year"]),
    ("results", results, ["raceId", "driverId", "constructorId"]),
    ("drivers", drivers, ["driverId", "driverRef"]),
    ("constructors", constructors, ["constructorId", "constructorRef"]),
]:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"{table_name} is missing columns: {missing}"
        )


# Race chronology.
race_cols = ["raceId", "year"]

if "round" in races.columns:
    race_cols.append("round")

race_calendar = races[race_cols].copy()

race_calendar["year"] = pd.to_numeric(
    race_calendar["year"], errors="coerce"
).astype("Int64")

if "round" not in race_calendar.columns:
    race_calendar["round"] = (
        race_calendar
        .groupby("year")
        .cumcount()
        .add(1)
    )

race_calendar["round"] = pd.to_numeric(
    race_calendar["round"], errors="coerce"
)

race_calendar = race_calendar.drop_duplicates("raceId")


# Driver names.
driver_lookup = (
    drivers[["driverId", "driverRef"]]
    .drop_duplicates("driverId")
    .rename(columns={"driverRef": "driver"})
)


# Constructor names.
constructor_name_col = None

for c in ["name", "constructor", "constructorName"]:
    if c in constructors.columns:
        constructor_name_col = c
        break

constructor_keep = ["constructorId", "constructorRef"]

if constructor_name_col is not None:
    constructor_keep.append(constructor_name_col)

constructor_lookup = (
    constructors[constructor_keep]
    .drop_duplicates("constructorId")
)

if constructor_name_col is not None:
    constructor_lookup = constructor_lookup.rename(
        columns={constructor_name_col: "constructor"}
    )
else:
    constructor_lookup["constructor"] = constructor_lookup["constructorRef"]


# ============================================================
# 4. IDENTIFY ACTUAL RACE STARTS
# ============================================================
#
# Tenure is START-based.
#
# Non-starting qualifying appearances therefore receive the tenure
# accumulated BEFORE the event, but do not increment the counter.
#
# The ORIGINAL model tenure is reconstructed below and compared with
# the likelihood tables. If our actual-start definition differs from
# the production preprocessing, the reconstruction check will expose it.
# ============================================================

results_status = results.copy()

status_text_col = None

if "status" in results_status.columns:
    status_text_col = "status"

elif statuses is not None and "statusId" in results_status.columns:
    candidate_status_cols = [
        c for c in ["status", "description", "name"]
        if c in statuses.columns
    ]

    if not candidate_status_cols:
        raise RuntimeError(
            "Found raw status table but could not identify its "
            "status-text column."
        )

    status_text_col_raw = candidate_status_cols[0]

    results_status = results_status.merge(
        statuses[["statusId", status_text_col_raw]]
        .drop_duplicates("statusId"),
        on="statusId",
        how="left",
    )

    status_text_col = status_text_col_raw

else:
    raise RuntimeError(
        "Could not obtain race-result status text. "
        "Need raw.results['status'] or raw.status/raw.statuses."
    )


status_text = (
    results_status[status_text_col]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


# These are explicitly PRE-RACE/non-start outcomes.
non_start_regex = (
    r"did not start"
    r"|did not qualify"
    r"|did not prequalify"
    r"|did not pre-qualify"
    r"|withdrawn"
    r"|withdrew"
    r"|not started"
    r"|\bdns\b"
    r"|\bdnq\b"
    r"|\bdnpq\b"
)

results_status["actual_start"] = ~status_text.str.contains(
    non_start_regex,
    regex=True,
    na=False,
)


print("=" * 72)
print("STAGE 32D — CONTINUOUS ORGANISATIONAL STINT TENURE")
print("=" * 72)

print("\nResult rows:", len(results_status))
print(
    "Actual-start rows:",
    int(results_status["actual_start"].sum())
)
print(
    "Excluded non-start rows:",
    int((~results_status["actual_start"]).sum())
)

print("\nStatuses treated as NON-STARTS:")
nonstart_status_counts = (
    results_status.loc[
        ~results_status["actual_start"],
        status_text_col,
    ]
    .fillna("<missing>")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="rows")
)
display(nonstart_status_counts)


# ============================================================
# 5. YEAR-SENSITIVE ORGANISATIONAL LINEAGE RULES
# ============================================================
#
# IMPORTANT:
# These identify operational continuity, not simply brand names.
#
# In particular:
#
#   * old works Renault != later Enstone Renault
#   * old Alfa Romeo != modern Hinwil Alfa Romeo
#   * old Honda != Brackley Honda
#   * old Mercedes != modern Brackley Mercedes
#   * Tyrrell is NOT folded into BRACKLEY
#
# ============================================================

LINEAGE_RULES = [

    # --------------------------------------------------------
    # ENSTONE
    # Toleman -> Benetton -> Renault -> Lotus F1
    # -> Renault -> Alpine
    # --------------------------------------------------------
    ("toleman",       1981, 1985, "ENSTONE"),
    ("benetton",      1986, 2001, "ENSTONE"),
    ("renault",       2002, 2011, "ENSTONE"),
    ("lotus_f1",      2012, 2015, "ENSTONE"),
    ("renault",       2016, 2020, "ENSTONE"),
    ("alpine",        2021, 9999, "ENSTONE"),

    # --------------------------------------------------------
    # BRACKLEY
    # BAR -> Honda -> Brawn -> Mercedes
    # --------------------------------------------------------
    ("bar",           1999, 2005, "BRACKLEY"),
    ("honda",         2006, 2008, "BRACKLEY"),
    ("brawn",         2009, 2009, "BRACKLEY"),
    ("mercedes",      2010, 9999, "BRACKLEY"),

    # --------------------------------------------------------
    # SILVERSTONE
    # Jordan -> Midland/MF1 -> Spyker -> Force India
    # -> Racing Point -> Aston Martin
    # --------------------------------------------------------
    ("jordan",        1991, 2005, "SILVERSTONE"),
    ("mf1",           2006, 2006, "SILVERSTONE"),
    ("midland",       2006, 2006, "SILVERSTONE"),
    ("spyker_mf1",    2006, 2006, "SILVERSTONE"),
    ("spyker",        2007, 2007, "SILVERSTONE"),
    ("force_india",   2008, 2018, "SILVERSTONE"),
    ("racing_point",  2019, 2020, "SILVERSTONE"),
    ("aston_martin",  2021, 9999, "SILVERSTONE"),

    # --------------------------------------------------------
    # FAENZA
    # Minardi -> Toro Rosso -> AlphaTauri -> RB
    # --------------------------------------------------------
    ("minardi",       1985, 2005, "FAENZA"),
    ("toro_rosso",    2006, 2019, "FAENZA"),
    ("alphatauri",    2020, 2023, "FAENZA"),
    ("rb",            2024, 9999, "FAENZA"),

    # --------------------------------------------------------
    # HINWIL
    # Sauber -> BMW Sauber -> Sauber -> Alfa Romeo
    # -> Sauber -> Audi
    # --------------------------------------------------------
    ("sauber",        1993, 2005, "HINWIL"),
    ("bmw_sauber",    2006, 2009, "HINWIL"),
    ("sauber",        2010, 2018, "HINWIL"),
    ("alfa",          2019, 2023, "HINWIL"),
    ("sauber",        2024, 2025, "HINWIL"),
    ("audi",          2026, 9999, "HINWIL"),

    # --------------------------------------------------------
    # MILTON KEYNES
    # Stewart -> Jaguar -> Red Bull
    # --------------------------------------------------------
    ("stewart",       1997, 1999, "MILTON_KEYNES"),
    ("jaguar",        2000, 2004, "MILTON_KEYNES"),
    ("red_bull",      2005, 9999, "MILTON_KEYNES"),

    # --------------------------------------------------------
    # Ligier -> Prost
    # --------------------------------------------------------
    ("ligier",        1976, 1996, "LIGIER_PROST"),
    ("prost",         1997, 2001, "LIGIER_PROST"),

    # --------------------------------------------------------
    # Arrows <-> Footwork
    # --------------------------------------------------------
    ("arrows",        1978, 1990, "ARROWS_FOOTWORK"),
    ("footwork",      1991, 1996, "ARROWS_FOOTWORK"),
    ("arrows",        1997, 2002, "ARROWS_FOOTWORK"),

    # --------------------------------------------------------
    # March -> Leyton House -> March
    #
    # Restricted to the 1987–92 incarnation so earlier March
    # entries cannot accidentally become the same modern stint.
    # --------------------------------------------------------
    ("march",         1987, 1989, "MARCH_LEYTON"),
    ("leyton",        1990, 1991, "MARCH_LEYTON"),
    ("march",         1992, 1992, "MARCH_LEYTON"),

    # --------------------------------------------------------
    # Virgin -> Marussia -> Manor
    # --------------------------------------------------------
    ("virgin",        2010, 2011, "VIRGIN_MARUSSIA_MANOR"),
    ("marussia",      2012, 2015, "VIRGIN_MARUSSIA_MANOR"),
    ("manor",         2016, 2016, "VIRGIN_MARUSSIA_MANOR"),
    ("manor_marussia",2015, 2016, "VIRGIN_MARUSSIA_MANOR"),

    # --------------------------------------------------------
    # Lotus Racing -> Caterham
    #
    # Deliberately NOT historical Team Lotus.
    # --------------------------------------------------------
    ("lotus_racing",  2010, 2011, "TEAM_LOTUS_CATERHAM"),
    ("caterham",      2012, 2014, "TEAM_LOTUS_CATERHAM"),
]


lineage_rules_df = pd.DataFrame(
    LINEAGE_RULES,
    columns=[
        "constructorRef",
        "start_year",
        "end_year",
        "lineage",
    ],
)

print("\n" + "=" * 72)
print("CURATED YEAR-SENSITIVE LINEAGE RULES")
print("=" * 72)
display(lineage_rules_df)


def resolve_lineage(constructor_ref, year):
    """
    Return organisational-lineage key for one constructor/year.

    If no curated continuity rule applies, use a literal constructor
    identity so unrelated teams remain separate.
    """
    ref = str(constructor_ref)
    yr = int(year)

    matches = lineage_rules_df.loc[
        (lineage_rules_df["constructorRef"] == ref)
        & (lineage_rules_df["start_year"] <= yr)
        & (lineage_rules_df["end_year"] >= yr)
    ]

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple lineage rules match {ref=} {yr=}:\n"
            f"{matches}"
        )

    if len(matches) == 1:
        return matches.iloc[0]["lineage"]

    return f"LITERAL::{ref}"


# ============================================================
# 6. BUILD EVERY DRIVER/RACE/CONSTRUCTOR CONTEXT WE NEED
# ============================================================
#
# Results are included across the whole database so prior-start
# histories are correct.
#
# Modern evidence rows are added separately so a qualifying
# participant who did not actually start still receives a tenure
# value for that event.
# ============================================================

result_context = (
    results_status[
        ["raceId", "driverId", "constructorId", "actual_start"]
    ]
    .copy()
)

result_context = (
    result_context
    .groupby(
        ["raceId", "driverId", "constructorId"],
        as_index=False,
    )["actual_start"]
    .max()
)


def pair_context_from_table(df):
    needed = {
        "raceId",
        "constructorId",
        "driverId_a",
        "driverId_b",
    }

    if not needed.issubset(df.columns):
        raise RuntimeError(
            "Likelihood table missing one of required columns "
            f"{sorted(needed)}"
        )

    a = df[
        ["raceId", "constructorId", "driverId_a"]
    ].rename(columns={"driverId_a": "driverId"})

    b = df[
        ["raceId", "constructorId", "driverId_b"]
    ].rename(columns={"driverId_b": "driverId"})

    return pd.concat([a, b], ignore_index=True)


evidence_context = pd.concat(
    [
        pair_context_from_table(pace_contrast_all),
        pair_context_from_table(pace_level_all),
        pair_context_from_table(q_contrast_all),
        pair_context_from_table(q_level_all_model),
    ],
    ignore_index=True,
).drop_duplicates()


# Start with every result context.
context = result_context.copy()

# Add evidence-only contexts.
extra_context = evidence_context.merge(
    result_context,
    on=["raceId", "driverId", "constructorId"],
    how="left",
)

extra_context["actual_start"] = (
    extra_context["actual_start"]
    .fillna(False)
    .astype(bool)
)

context = pd.concat(
    [context, extra_context],
    ignore_index=True,
)

context = (
    context
    .groupby(
        ["raceId", "driverId", "constructorId"],
        as_index=False,
    )["actual_start"]
    .max()
)


# Add chronology / names.
context = context.merge(
    race_calendar,
    on="raceId",
    how="left",
)

context = context.merge(
    driver_lookup,
    on="driverId",
    how="left",
)

context = context.merge(
    constructor_lookup,
    on="constructorId",
    how="left",
)


if context["year"].isna().any():
    raise RuntimeError(
        "Some driver/race contexts could not be matched to a race year."
    )

if context["constructorRef"].isna().any():
    raise RuntimeError(
        "Some driver/race contexts could not be matched to a "
        "constructorRef."
    )

if context["driver"].isna().any():
    raise RuntimeError(
        "Some driver/race contexts could not be matched to driverRef."
    )


context["lineage"] = [
    resolve_lineage(ref, year)
    for ref, year in zip(
        context["constructorRef"],
        context["year"],
    )
]


# ============================================================
# 7. CHECK FOR IMPOSSIBLE CURATED-LINEAGE OVERLAPS
# ============================================================
#
# If two different constructor identities are simultaneously active
# in the same year under one of our curated operational lineages,
# inspect it before proceeding.
# ============================================================

active_lineage_overlap = (
    context.loc[
        context["actual_start"]
        & ~context["lineage"].str.startswith("LITERAL::")
    ]
    .groupby(["year", "lineage"])
    .agg(
        constructor_refs=(
            "constructorRef",
            lambda x: ", ".join(sorted(set(x))),
        ),
        n_constructor_refs=("constructorRef", "nunique"),
    )
    .reset_index()
)

active_lineage_overlap = active_lineage_overlap.loc[
    active_lineage_overlap["n_constructor_refs"] > 1
].copy()


print("\n" + "=" * 72)
print("SIMULTANEOUS CONSTRUCTOR IDENTITIES WITHIN ONE CURATED LINEAGE")
print("=" * 72)

if active_lineage_overlap.empty:
    print("None.")
else:
    display(active_lineage_overlap)


# ============================================================
# 8. RECONSTRUCT ORIGINAL + CORRECTED TENURE
# ============================================================
#
# ORIGINAL:
#   all previous actual starts with the exact constructorId.
#
# CORRECTED:
#   number of consecutive actual starts in the driver's CURRENT
#   organisational lineage.
#
# The current continuous stint is broken only when the driver
# ACTUALLY STARTS for another lineage.
#
# A DNS/DNQ/etc. does not increment or break a start-based stint.
# ============================================================

context = context.sort_values(
    ["driverId", "year", "round", "raceId", "constructorId"]
).reset_index(drop=True)


tenure_records = []

for driver_id, g in context.groupby("driverId", sort=False):

    # Original model:
    exact_constructor_start_counts = defaultdict(int)

    # Corrected model:
    last_actual_start_lineage = None
    continuous_stint_start_count = 0

    for row in g.itertuples(index=False):

        constructor_id = int(row.constructorId)
        lineage = row.lineage

        # -------------------------------
        # ORIGINAL exact-constructor count
        # -------------------------------
        original_starts_before = (
            exact_constructor_start_counts[constructor_id]
        )

        original_tenure = np.tanh(
            original_starts_before / TENURE_SCALE
        )

        # -------------------------------
        # CORRECTED continuous-lineage stint
        # -------------------------------
        if lineage == last_actual_start_lineage:
            corrected_starts_before = (
                continuous_stint_start_count
            )
        else:
            corrected_starts_before = 0

        corrected_tenure = np.tanh(
            corrected_starts_before / TENURE_SCALE
        )

        tenure_records.append(
            {
                "raceId": row.raceId,
                "driverId": row.driverId,
                "constructorId": row.constructorId,
                "year": int(row.year),
                "round": row.round,
                "driver": row.driver,
                "constructorRef": row.constructorRef,
                "constructor": row.constructor,
                "lineage": lineage,
                "actual_start": bool(row.actual_start),

                "original_constructor_starts_before":
                    original_starts_before,

                "corrected_stint_starts_before":
                    corrected_starts_before,

                "original_tenure_raw":
                    original_tenure,

                "corrected_tenure_raw":
                    corrected_tenure,
            }
        )

        # -----------------------------------
        # Update counts AFTER the race starts
        # -----------------------------------
        if bool(row.actual_start):

            exact_constructor_start_counts[
                constructor_id
            ] += 1

            if lineage == last_actual_start_lineage:
                continuous_stint_start_count += 1
            else:
                last_actual_start_lineage = lineage
                continuous_stint_start_count = 1


tenure_context_32d = pd.DataFrame(tenure_records)

tenure_context_32d["delta_tenure_raw"] = (
    tenure_context_32d["corrected_tenure_raw"]
    - tenure_context_32d["original_tenure_raw"]
)

tenure_context_32d["abs_delta_tenure_raw"] = (
    tenure_context_32d["delta_tenure_raw"].abs()
)


# ============================================================
# 9. IDENTIFY THE TWO IMPORTANT TYPES OF CHANGE
# ============================================================
#
# A. REBRAND RESET RECOVERED
#      old exact constructor tenure = 0
#      corrected continuous lineage tenure > 0
#
# B. STALE RETURN FAMILIARITY REMOVED
#      old exact constructor tenure > 0
#      corrected current-stint tenure = 0
#
# ============================================================

actual_starts_32d = tenure_context_32d.loc[
    tenure_context_32d["actual_start"]
].copy()


rebrand_recoveries = actual_starts_32d.loc[
    (actual_starts_32d["original_constructor_starts_before"] == 0)
    & (actual_starts_32d["corrected_stint_starts_before"] > 0)
].copy()

stale_return_resets = actual_starts_32d.loc[
    (actual_starts_32d["original_constructor_starts_before"] > 0)
    & (actual_starts_32d["corrected_stint_starts_before"] == 0)
].copy()


first_rebrand_recovery = (
    rebrand_recoveries
    .sort_values(
        ["driver", "year", "round", "raceId"]
    )
    .groupby(
        ["driver", "constructorRef", "lineage"],
        as_index=False,
    )
    .first()
)

first_stale_return_reset = (
    stale_return_resets
    .sort_values(
        ["driver", "year", "round", "raceId"]
    )
    .groupby(
        ["driver", "constructorRef", "lineage"],
        as_index=False,
    )
    .first()
)


print("\n" + "=" * 72)
print("FIRST REBRAND / ID RESET RECOVERED BY CONTINUOUS STINT")
print("=" * 72)

display(
    first_rebrand_recovery[
        [
            "year",
            "driver",
            "constructorRef",
            "lineage",
            "original_constructor_starts_before",
            "corrected_stint_starts_before",
            "original_tenure_raw",
            "corrected_tenure_raw",
            "delta_tenure_raw",
        ]
    ]
    .sort_values(
        "corrected_stint_starts_before",
        ascending=False,
    )
    .head(50)
)


print("\n" + "=" * 72)
print("STALE OLD-TEAM FAMILIARITY REMOVED AFTER DRIVER LEFT TEAM")
print("=" * 72)

display(
    first_stale_return_reset[
        [
            "year",
            "driver",
            "constructorRef",
            "lineage",
            "original_constructor_starts_before",
            "corrected_stint_starts_before",
            "original_tenure_raw",
            "corrected_tenure_raw",
            "delta_tenure_raw",
        ]
    ]
    .sort_values(
        "original_constructor_starts_before",
        ascending=False,
    )
    .head(50)
)


# ============================================================
# 10. EXPLICIT SANITY CHECKS
# ============================================================
#
# expected_starts:
#   exact expected corrected starts before first relevant event,
#   where we know the answer from the underlying start history.
#
# expected_mode:
#   "carry" -> must be > 0
#   "reset" -> must equal 0
#
# ============================================================

CONTINUITY_CHECKS = [

    # --------------------------------------------------------
    # Things that MUST carry
    # --------------------------------------------------------
    {
        "label": "Warwick — Toleman at 1982 model boundary",
        "driver": "warwick",
        "year": 1982,
        "constructorRef": "toleman",
        "expected_mode": "carry",
        "expected_starts": 12,
    },
    {
        "label": "Laffite — Ligier at 1982 model boundary",
        "driver": "laffite",
        "year": 1982,
        "constructorRef": "ligier",
        "expected_mode": "carry",
        "expected_starts": 93,
    },
    {
        "label": "Fabi — Toleman -> Benetton",
        "driver": "fabi",
        "year": 1986,
        "constructorRef": "benetton",
        "expected_mode": "carry",
        "expected_starts": 27,
    },
    {
        "label": "Capelli — March -> Leyton House",
        "driver": "capelli",
        "year": 1990,
        "constructorRef": "leyton",
        "expected_mode": "carry",
        "expected_starts": 48,
    },
    {
        "label": "Alboreto — Arrows -> Footwork",
        "driver": "alboreto",
        "year": 1991,
        "constructorRef": "footwork",
        "expected_mode": "carry",
        "expected_starts": 16,
    },
    {
        "label": "Button — Honda -> Brawn",
        "driver": "button",
        "year": 2009,
        "constructorRef": "brawn",
        "expected_mode": "carry",
        "expected_starts": 104,
    },
    {
        "label": "Barrichello — Honda -> Brawn",
        "driver": "barrichello",
        "year": 2009,
        "constructorRef": "brawn",
        "expected_mode": "carry",
        "expected_starts": 53,
    },
    {
        "label": "Pérez — Force India -> Racing Point",
        "driver": "perez",
        "year": 2019,
        "constructorRef": "racing_point",
        "expected_mode": "carry",
        "expected_starts": 100,
    },
    {
        "label": "Gasly — Toro Rosso -> AlphaTauri",
        "driver": "gasly",
        "year": 2020,
        "constructorRef": "alphatauri",
        "expected_mode": "carry",
        "expected_starts": 35,
    },
    {
        "label": "Kvyat — Toro Rosso -> AlphaTauri",
        "driver": "kvyat",
        "year": 2020,
        "constructorRef": "alphatauri",
        "expected_mode": "carry",
        "expected_starts": 72,
    },
    {
        "label": "Ocon — Renault -> Alpine",
        "driver": "ocon",
        "year": 2021,
        "constructorRef": "alpine",
        "expected_mode": "carry",
        "expected_starts": 17,
    },
    {
        "label": "Stroll — Racing Point -> Aston Martin",
        "driver": "stroll",
        "year": 2021,
        "constructorRef": "aston_martin",
        "expected_mode": "carry",
        "expected_starts": 37,
    },
    {
        "label": "Ricciardo — AlphaTauri -> RB",
        "driver": "ricciardo",
        "year": 2024,
        "constructorRef": "rb",
        "expected_mode": "carry",
        "expected_starts": 7,
    },
    {
        "label": "Tsunoda — AlphaTauri -> RB",
        "driver": "tsunoda",
        "year": 2024,
        "constructorRef": "rb",
        "expected_mode": "carry",
        "expected_starts": 66,
    },
    {
        "label": "Bottas — Alfa Romeo -> Sauber",
        "driver": "bottas",
        "year": 2024,
        "constructorRef": "sauber",
        "expected_mode": "carry",
        "expected_starts": 44,
    },
    {
        "label": "Zhou — Alfa Romeo -> Sauber",
        "driver": "zhou",
        "year": 2024,
        "constructorRef": "sauber",
        "expected_mode": "carry",
        "expected_starts": 44,
    },
    {
        "label": "Bortoleto — Sauber -> Audi",
        "driver": "bortoleto",
        "year": 2026,
        "constructorRef": "audi",
        "expected_mode": "carry",
        "expected_starts": 24,
    },
    {
        "label": "Hülkenberg — Sauber 2025 -> Audi",
        "driver": "hulkenberg",
        "year": 2026,
        "constructorRef": "audi",
        "expected_mode": "carry",
        "expected_starts": 24,
    },

    # --------------------------------------------------------
    # Things that MUST reset
    # --------------------------------------------------------
    {
        "label": "Warwick — Toleman -> unrelated works Renault",
        "driver": "warwick",
        "year": 1984,
        "constructorRef": "renault",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Fisichella — returns to Enstone after other teams",
        "driver": "fisichella",
        "year": 2005,
        "constructorRef": "renault",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Heidfeld — returns to Hinwil after Williams",
        "driver": "heidfeld",
        "year": 2006,
        "constructorRef": "bmw_sauber",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Webber — returns to Milton Keynes lineage after Williams",
        "driver": "webber",
        "year": 2007,
        "constructorRef": "red_bull",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Kovalainen — returns to Enstone after Caterham",
        "driver": "kovalainen",
        "year": 2013,
        "constructorRef": "lotus_f1",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Räikkönen — old Sauber history must not carry to Alfa",
        "driver": "raikkonen",
        "year": 2019,
        "constructorRef": "alfa",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Hülkenberg — Force India history must not carry after Renault",
        "driver": "hulkenberg",
        "year": 2020,
        "constructorRef": "racing_point",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Alonso — old Renault history must not carry to Alpine 2021",
        "driver": "alonso",
        "year": 2021,
        "constructorRef": "alpine",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Kubica — old BMW Sauber history must not carry to Alfa 2021",
        "driver": "kubica",
        "year": 2021,
        "constructorRef": "alfa",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Pérez — Cadillac is a genuinely new team",
        "driver": "perez",
        "year": 2026,
        "constructorRef": "cadillac",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Bottas — Cadillac is a genuinely new team",
        "driver": "bottas",
        "year": 2026,
        "constructorRef": "cadillac",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
    {
        "label": "Hadjar — RB -> Red Bull is NOT same organisation",
        "driver": "hadjar",
        "year": 2026,
        "constructorRef": "red_bull",
        "expected_mode": "reset",
        "expected_starts": 0,
    },
]


check_rows = []

for spec in CONTINUITY_CHECKS:

    sub = tenure_context_32d.loc[
        (tenure_context_32d["driver"] == spec["driver"])
        & (tenure_context_32d["year"] == spec["year"])
        & (
            tenure_context_32d["constructorRef"]
            == spec["constructorRef"]
        )
    ].sort_values(["round", "raceId"])

    if sub.empty:
        check_rows.append(
            {
                "check": spec["label"],
                "driver": spec["driver"],
                "year": spec["year"],
                "constructorRef": spec["constructorRef"],
                "expected_mode": spec["expected_mode"],
                "expected_starts": spec["expected_starts"],
                "observed_corrected_starts_before": np.nan,
                "original_constructor_starts_before": np.nan,
                "corrected_tenure_raw": np.nan,
                "pass": False,
                "note": "NO MATCHING ROW FOUND",
            }
        )
        continue

    row = sub.iloc[0]

    observed = int(row["corrected_stint_starts_before"])
    expected = spec["expected_starts"]

    exact_match = observed == expected

    if spec["expected_mode"] == "carry":
        mode_match = observed > 0
    else:
        mode_match = observed == 0

    passed = bool(exact_match and mode_match)

    check_rows.append(
        {
            "check": spec["label"],
            "driver": spec["driver"],
            "year": spec["year"],
            "constructorRef": spec["constructorRef"],
            "expected_mode": spec["expected_mode"],
            "expected_starts": expected,
            "observed_corrected_starts_before": observed,
            "original_constructor_starts_before":
                int(row["original_constructor_starts_before"]),
            "corrected_tenure_raw":
                row["corrected_tenure_raw"],
            "pass": passed,
            "note": "" if passed else "CHECK EXPECTED COUNT",
        }
    )


continuity_checks_32d = pd.DataFrame(check_rows)

print("\n" + "=" * 72)
print("SPECIFIC CONTINUITY / RESET SANITY CHECKS")
print("=" * 72)
display(continuity_checks_32d)


# ============================================================
# 11. ATTACH TENURE TO MODERN LIKELIHOOD TABLES
# ============================================================

tenure_lookup = tenure_context_32d[
    [
        "raceId",
        "driverId",
        "constructorId",
        "original_constructor_starts_before",
        "corrected_stint_starts_before",
        "original_tenure_raw",
        "corrected_tenure_raw",
    ]
].copy()


def audit_modern_channel(
    df,
    channel_name,
    current_covariate,
    mode,
):
    """
    mode:
      'contrast' -> tenure_a - tenure_b
      'level'    -> (tenure_a + tenure_b)/2
    """

    x = df.copy()

    lookup_a = tenure_lookup.rename(
        columns={
            "driverId": "driverId_a",
            "original_constructor_starts_before":
                "original_starts_a",
            "corrected_stint_starts_before":
                "corrected_starts_a",
            "original_tenure_raw":
                "original_tenure_a",
            "corrected_tenure_raw":
                "corrected_tenure_a",
        }
    )

    lookup_b = tenure_lookup.rename(
        columns={
            "driverId": "driverId_b",
            "original_constructor_starts_before":
                "original_starts_b",
            "corrected_stint_starts_before":
                "corrected_starts_b",
            "original_tenure_raw":
                "original_tenure_b",
            "corrected_tenure_raw":
                "corrected_tenure_b",
        }
    )

    join_keys_a = [
        "raceId",
        "constructorId",
        "driverId_a",
    ]

    join_keys_b = [
        "raceId",
        "constructorId",
        "driverId_b",
    ]

    x = x.merge(
        lookup_a[
            join_keys_a
            + [
                "original_starts_a",
                "corrected_starts_a",
                "original_tenure_a",
                "corrected_tenure_a",
            ]
        ],
        on=join_keys_a,
        how="left",
    )

    x = x.merge(
        lookup_b[
            join_keys_b
            + [
                "original_starts_b",
                "corrected_starts_b",
                "original_tenure_b",
                "corrected_tenure_b",
            ]
        ],
        on=join_keys_b,
        how="left",
    )

    missing_lookup = int(
        x[
            [
                "original_tenure_a",
                "original_tenure_b",
                "corrected_tenure_a",
                "corrected_tenure_b",
            ]
        ].isna().any(axis=1).sum()
    )

    if mode == "contrast":
        x["rebuilt_original_covariate"] = (
            x["original_tenure_a"]
            - x["original_tenure_b"]
        )

        x["corrected_covariate"] = (
            x["corrected_tenure_a"]
            - x["corrected_tenure_b"]
        )

    elif mode == "level":
        x["rebuilt_original_covariate"] = 0.5 * (
            x["original_tenure_a"]
            + x["original_tenure_b"]
        )

        x["corrected_covariate"] = 0.5 * (
            x["corrected_tenure_a"]
            + x["corrected_tenure_b"]
        )

    else:
        raise ValueError(mode)

    x["current_covariate"] = x[current_covariate]

    x["reconstruction_error"] = (
        x["rebuilt_original_covariate"]
        - x["current_covariate"]
    )

    x["delta_tenure_covariate"] = (
        x["corrected_covariate"]
        - x["current_covariate"]
    )

    x["abs_delta_tenure_covariate"] = (
        x["delta_tenure_covariate"].abs()
    )

    x["approx_mu_shift_at_gamma_reference"] = (
        GAMMA_TENURE_REFERENCE
        * x["delta_tenure_covariate"]
    )

    x["abs_approx_mu_shift_at_gamma_reference"] = (
        x["approx_mu_shift_at_gamma_reference"].abs()
    )

    finite_reconstruction = x[
        "reconstruction_error"
    ].dropna()

    max_reconstruction_error = (
        float(finite_reconstruction.abs().max())
        if len(finite_reconstruction)
        else np.nan
    )

    changed = (
        x["abs_delta_tenure_covariate"] > TENURE_TOL
    )

    summary = {
        "channel": channel_name,
        "rows": len(x),
        "missing_tenure_lookup": missing_lookup,
        "max_original_reconstruction_error":
            max_reconstruction_error,
        "changed_rows": int(changed.sum()),
        "changed_fraction": float(changed.mean()),
        "mean_abs_covariate_change":
            float(x["abs_delta_tenure_covariate"].mean()),
        "median_abs_covariate_change":
            float(x["abs_delta_tenure_covariate"].median()),
        "max_abs_covariate_change":
            float(x["abs_delta_tenure_covariate"].max()),
        "max_abs_mu_shift_at_gamma_reference":
            float(
                x[
                    "abs_approx_mu_shift_at_gamma_reference"
                ].max()
            ),
    }

    return x, summary


pace_contrast_32d, pace_contrast_summary = (
    audit_modern_channel(
        pace_contrast_all,
        "pace_contrast tenure_diff",
        "tenure_diff",
        "contrast",
    )
)

pace_level_32d, pace_level_summary = (
    audit_modern_channel(
        pace_level_all,
        "pace_level tenure_mean",
        "tenure_mean",
        "level",
    )
)

q_contrast_32d, q_contrast_summary = (
    audit_modern_channel(
        q_contrast_all,
        "q_contrast tenure_diff",
        "tenure_diff",
        "contrast",
    )
)

q1_level_32d, q1_level_summary = (
    audit_modern_channel(
        q_level_all_model,
        "q1_level tenure_mean",
        "tenure_mean",
        "level",
    )
)


modern_covariate_impact_32d = pd.DataFrame(
    [
        pace_contrast_summary,
        pace_level_summary,
        q_contrast_summary,
        q1_level_summary,
    ]
)


print("\n" + "=" * 72)
print("ORIGINAL TENURE RECONSTRUCTION + CORRECTED MODERN IMPACT")
print("=" * 72)

display(modern_covariate_impact_32d)


# ============================================================
# 12. SHOW LARGEST MODERN CONTRAST CHANGES
# ============================================================

def top_modern_changes(df, label, n=30):

    show_cols = [
        c for c in [
            "year",
            "raceId",
            "session",
            "constructorId",
            "driver_a",
            "driver_b",
            "current_covariate",
            "corrected_covariate",
            "delta_tenure_covariate",
            "abs_delta_tenure_covariate",
            "approx_mu_shift_at_gamma_reference",
        ]
        if c in df.columns
    ]

    print("\n" + "=" * 72)
    print(label)
    print("=" * 72)

    display(
        df.sort_values(
            "abs_delta_tenure_covariate",
            ascending=False,
        )[show_cols].head(n)
    )


top_modern_changes(
    pace_contrast_32d,
    "LARGEST PACE-CONTRAST TENURE CHANGES",
)

top_modern_changes(
    q_contrast_32d,
    "LARGEST QUALIFYING-CONTRAST TENURE CHANGES",
)


# ============================================================
# 13. LINEAGE-LEVEL IMPACT
# ============================================================

changed_actual = actual_starts_32d.loc[
    actual_starts_32d["abs_delta_tenure_raw"] > TENURE_TOL
].copy()


lineage_impact_32d = (
    changed_actual
    .groupby("lineage")
    .agg(
        affected_driver_starts=("raceId", "size"),
        affected_drivers=("driverId", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        mean_delta_tenure=("delta_tenure_raw", "mean"),
        mean_abs_delta_tenure=(
            "abs_delta_tenure_raw",
            "mean",
        ),
        max_abs_delta_tenure=(
            "abs_delta_tenure_raw",
            "max",
        ),
        positive_changes=(
            "delta_tenure_raw",
            lambda x: int((x > TENURE_TOL).sum()),
        ),
        negative_changes=(
            "delta_tenure_raw",
            lambda x: int((x < -TENURE_TOL).sum()),
        ),
    )
    .reset_index()
    .sort_values(
        "max_abs_delta_tenure",
        ascending=False,
    )
)


driver_impact_32d = (
    changed_actual
    .groupby(["driverId", "driver"])
    .agg(
        affected_starts=("raceId", "size"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        max_abs_delta_tenure=(
            "abs_delta_tenure_raw",
            "max",
        ),
        mean_abs_delta_tenure=(
            "abs_delta_tenure_raw",
            "mean",
        ),
        max_positive_change=(
            "delta_tenure_raw",
            "max",
        ),
        max_negative_change=(
            "delta_tenure_raw",
            "min",
        ),
    )
    .reset_index()
    .sort_values(
        "max_abs_delta_tenure",
        ascending=False,
    )
)


print("\n" + "=" * 72)
print("LINEAGE-LEVEL CONTINUOUS-STINT TENURE IMPACT")
print("=" * 72)
display(lineage_impact_32d)


print("\n" + "=" * 72)
print("DRIVERS MOST AFFECTED BY CORRECTED STINT DEFINITION")
print("=" * 72)
display(driver_impact_32d.head(50))


# ============================================================
# 14. HISTORICAL 1982–95 SANITY AUDIT
# ============================================================
#
# This is deliberately a RAW / WITHIN-RACE tenure audit.
#
# We are NOT modifying hist_q_pack or hist_race_pack here.
# The purpose is to prove that the Stage 32C artificial 1982
# boundary reset has disappeared before rebuilding the historical
# likelihood.
# ============================================================

hist_start_32d = actual_starts_32d.loc[
    actual_starts_32d["year"].between(1982, 1995)
].copy()


hist_start_32d["original_centered_tenure"] = (
    hist_start_32d["original_tenure_raw"]
    - hist_start_32d.groupby("raceId")[
        "original_tenure_raw"
    ].transform("mean")
)

hist_start_32d["corrected_centered_tenure"] = (
    hist_start_32d["corrected_tenure_raw"]
    - hist_start_32d.groupby("raceId")[
        "corrected_tenure_raw"
    ].transform("mean")
)

hist_start_32d["delta_centered_tenure"] = (
    hist_start_32d["corrected_centered_tenure"]
    - hist_start_32d["original_centered_tenure"]
)

hist_start_32d["abs_delta_centered_tenure"] = (
    hist_start_32d["delta_centered_tenure"].abs()
)


historical_summary_32d = pd.DataFrame(
    [
        {
            "metric": "historical actual-start rows",
            "value": len(hist_start_32d),
        },
        {
            "metric": "rows with changed raw tenure",
            "value": int(
                (
                    hist_start_32d["abs_delta_tenure_raw"]
                    > TENURE_TOL
                ).sum()
            ),
        },
        {
            "metric": "rows with changed within-race centred tenure",
            "value": int(
                (
                    hist_start_32d[
                        "abs_delta_centered_tenure"
                    ]
                    > TENURE_TOL
                ).sum()
            ),
        },
        {
            "metric": "affected drivers",
            "value": int(
                hist_start_32d.loc[
                    hist_start_32d[
                        "abs_delta_tenure_raw"
                    ] > TENURE_TOL,
                    "driverId",
                ].nunique()
            ),
        },
        {
            "metric": "max |raw tenure change|",
            "value": float(
                hist_start_32d[
                    "abs_delta_tenure_raw"
                ].max()
            ),
        },
        {
            "metric": "max |centred tenure change|",
            "value": float(
                hist_start_32d[
                    "abs_delta_centered_tenure"
                ].max()
            ),
        },
    ]
)


print("\n" + "=" * 72)
print("1982–95 HISTORICAL TENURE SANITY AUDIT")
print("=" * 72)
display(historical_summary_32d)


print("\n1982 rows with changed raw tenure:")
hist_1982_changes = hist_start_32d.loc[
    (hist_start_32d["year"] == 1982)
    & (
        hist_start_32d["abs_delta_tenure_raw"]
        > TENURE_TOL
    )
].copy()

if hist_1982_changes.empty:
    print("None.")
else:
    display(
        hist_1982_changes[
            [
                "driver",
                "constructorRef",
                "lineage",
                "original_constructor_starts_before",
                "corrected_stint_starts_before",
                "original_tenure_raw",
                "corrected_tenure_raw",
                "delta_tenure_raw",
            ]
        ]
    )


# ============================================================
# 15. FOCUS: FIRST 2026 STARTS
# ============================================================

first_2026 = (
    tenure_context_32d.loc[
        tenure_context_32d["year"] == 2026
    ]
    .sort_values(["driver", "round", "raceId"])
    .groupby("driver", as_index=False)
    .first()
)


print("\n" + "=" * 72)
print("FIRST 2026 TENURE VALUES UNDER CORRECTED DEFINITION")
print("=" * 72)

display(
    first_2026[
        [
            "driver",
            "constructor",
            "constructorRef",
            "lineage",
            "original_constructor_starts_before",
            "corrected_stint_starts_before",
            "original_tenure_raw",
            "corrected_tenure_raw",
            "delta_tenure_raw",
        ]
    ]
    .sort_values(["constructor", "driver"])
)


# ============================================================
# 16. PLOTS
# ============================================================

if not lineage_impact_32d.empty:

    plot_lineages = (
        lineage_impact_32d
        .head(20)
        .sort_values("max_abs_delta_tenure")
    )

    plt.figure(figsize=(11, 7))
    plt.barh(
        plot_lineages["lineage"],
        plot_lineages["max_abs_delta_tenure"],
    )
    plt.xlabel("Maximum absolute change in raw tenure")
    plt.ylabel("Organisational lineage")
    plt.title(
        "Impact of continuous organisational-stint tenure"
    )
    plt.tight_layout()
    plt.show()


if not driver_impact_32d.empty:

    plot_drivers = (
        driver_impact_32d
        .head(30)
        .sort_values("max_abs_delta_tenure")
    )

    plt.figure(figsize=(11, 9))
    plt.barh(
        plot_drivers["driver"],
        plot_drivers["max_abs_delta_tenure"],
    )
    plt.xlabel("Maximum absolute change in raw tenure")
    plt.ylabel("Driver")
    plt.title(
        "Drivers most affected by corrected team familiarity"
    )
    plt.tight_layout()
    plt.show()


# ============================================================
# 17. HARD-FLAG LOGIC
# ============================================================

hard_flags = []


# A. Existing production tenure MUST reconstruct correctly.
bad_reconstruction = modern_covariate_impact_32d.loc[
    (
        modern_covariate_impact_32d[
            "max_original_reconstruction_error"
        ] > TENURE_TOL
    )
    | (
        modern_covariate_impact_32d[
            "missing_tenure_lookup"
        ] > 0
    )
]

if len(bad_reconstruction):
    hard_flags.append(
        "Original production tenure did not reconstruct exactly "
        "in one or more modern channels."
    )


# B. Curated lineage must not create simultaneous teams.
if len(active_lineage_overlap):
    hard_flags.append(
        "At least one curated lineage contains multiple active "
        "constructor identities in the same year."
    )


# C. Specific carry/reset tests.
failed_checks = continuity_checks_32d.loc[
    ~continuity_checks_32d["pass"]
]

if len(failed_checks):
    hard_flags.append(
        f"{len(failed_checks)} explicit continuity/reset sanity "
        "checks failed."
    )


# D. Basic bounds.
if (
    (tenure_context_32d["corrected_tenure_raw"] < -TENURE_TOL).any()
    or
    (tenure_context_32d["corrected_tenure_raw"] > 1 + TENURE_TOL).any()
):
    hard_flags.append(
        "Corrected raw tenure fell outside [0, 1]."
    )


# ============================================================
# 18. SAVE OUTPUTS
# ============================================================

lineage_rules_df.to_csv(
    OUTDIR_32D / "lineage_rules.csv",
    index=False,
)

tenure_context_32d.to_csv(
    OUTDIR_32D / "driver_race_tenure_context.csv",
    index=False,
)

continuity_checks_32d.to_csv(
    OUTDIR_32D / "continuity_reset_checks.csv",
    index=False,
)

first_rebrand_recovery.to_csv(
    OUTDIR_32D / "first_rebrand_recoveries.csv",
    index=False,
)

first_stale_return_reset.to_csv(
    OUTDIR_32D / "first_stale_return_resets.csv",
    index=False,
)

modern_covariate_impact_32d.to_csv(
    OUTDIR_32D / "modern_covariate_impact_summary.csv",
    index=False,
)

pace_contrast_32d.to_csv(
    OUTDIR_32D / "pace_contrast_corrected_tenure_audit.csv",
    index=False,
)

pace_level_32d.to_csv(
    OUTDIR_32D / "pace_level_corrected_tenure_audit.csv",
    index=False,
)

q_contrast_32d.to_csv(
    OUTDIR_32D / "q_contrast_corrected_tenure_audit.csv",
    index=False,
)

q1_level_32d.to_csv(
    OUTDIR_32D / "q1_level_corrected_tenure_audit.csv",
    index=False,
)

lineage_impact_32d.to_csv(
    OUTDIR_32D / "lineage_impact.csv",
    index=False,
)

driver_impact_32d.to_csv(
    OUTDIR_32D / "driver_impact.csv",
    index=False,
)

hist_start_32d.to_csv(
    OUTDIR_32D / "historical_1982_1995_tenure_audit.csv",
    index=False,
)

historical_summary_32d.to_csv(
    OUTDIR_32D / "historical_summary.csv",
    index=False,
)

first_2026.to_csv(
    OUTDIR_32D / "first_2026_tenure.csv",
    index=False,
)


# ============================================================
# 19. TRIAGE SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("STAGE 32D TRIAGE SUMMARY")
print("=" * 72)

print(
    "Driver/race contexts:",
    f"{len(tenure_context_32d):,}",
)

print(
    "Actual race starts:",
    f"{len(actual_starts_32d):,}",
)

print(
    "Actual starts whose raw tenure changes:",
    f"{len(changed_actual):,}",
)

print(
    "Drivers affected:",
    changed_actual["driverId"].nunique(),
)

print(
    "Recovered constructor-ID/rebrand resets:",
    f"{len(rebrand_recoveries):,}",
)

print(
    "Stale old-team familiarity resets removed:",
    f"{len(stale_return_resets):,}",
)


print("\nModern likelihood reconstruction / impact:")

for _, r in modern_covariate_impact_32d.iterrows():
    print(
        f"  {r['channel']}: "
        f"{int(r['changed_rows'])}/{int(r['rows'])} changed; "
        f"max original rebuild error = "
        f"{r['max_original_reconstruction_error']:.3e}; "
        f"max |Δ tenure covariate| = "
        f"{r['max_abs_covariate_change']:.4f}; "
        f"max |Δμ| at gamma={GAMMA_TENURE_REFERENCE:.6f} = "
        f"{r['max_abs_mu_shift_at_gamma_reference']:.4f}"
    )


print("\nSpecific carry/reset checks:")
print(
    f"  Passed: "
    f"{int(continuity_checks_32d['pass'].sum())}/"
    f"{len(continuity_checks_32d)}"
)

if len(failed_checks):
    print("\nFAILED CHECKS:")
    display(failed_checks)


print("\nHistorical 1982–95:")
print(
    "  Raw tenure rows changed:",
    int(
        (
            hist_start_32d["abs_delta_tenure_raw"]
            > TENURE_TOL
        ).sum()
    ),
)
print(
    "  Within-race centred rows changed:",
    int(
        (
            hist_start_32d["abs_delta_centered_tenure"]
            > TENURE_TOL
        ).sum()
    ),
)


print("\nHARD FLAGS:")

if not hard_flags:
    print("  none")
else:
    for flag in hard_flags:
        print("  -", flag)


print("\nDECISION RULE:")
print(
    "  If HARD FLAGS = none, all explicit continuity/reset checks "
    "pass, and the original modern tenure reconstructs to numerical "
    "precision, this continuous-organisational-stint definition is "
    "ready to replace the Stage 3 tenure construction."
)

print(
    "\n  In particular, verify that:"
    "\n    • Bortoleto 2026 Audi carries 24 Sauber starts;"
    "\n    • Hülkenberg 2026 Audi carries ONLY his 2025 Sauber stint,"
    "\n      not his old 2013 Sauber history;"
    "\n    • Alonso 2021 Alpine resets to zero;"
    "\n    • Button/Barrichello 2009 Brawn retain Brackley tenure;"
    "\n    • Pérez 2019 Racing Point retains Force India tenure;"
    "\n    • Warwick/Laffite do NOT suffer an artificial 1982 reset;"
    "\n    • Warwick's later move from Toleman to the unrelated works"
    "\n      Renault team DOES reset."
)

print("\nSaved outputs ->")
print(OUTDIR_32D.resolve())

print("\nStage 32D complete.")

In [ ]:
# ============================================================
# STAGE 32E — FORENSIC AUDIT OF THE PRODUCTION TENURE MECHANICS
# ============================================================
#
# PURPOSE
# -------
# Stage 32D deliberately tried a new "continuous organisational stint"
# definition, but its comparison with the frozen production covariates
# was not apples-to-apples:
#
#   * it switched from production result appearances to actual starts;
#   * it compared raw pair means against centred level covariates;
#   * qualifying contrasts showed a larger discrepancy that needs
#     explicit diagnosis.
#
# This cell therefore DOES NOT propose a new tenure definition.
#
# Instead it asks:
#
#   1. What event count reproduces the frozen tenure_diff?
#        - previous RESULT APPEARANCES with exact constructor ID
#        - previous ACTUAL STARTS with exact constructor ID
#        - including-current variants
#
#   2. Do level covariates equal:
#
#          pair raw mean - one common race-specific centre
#
#      as expected?
#
#   3. Is the qualifying tenure_diff actually consistent with the
#      same production mechanics as pace?
#
#   4. Are any curated organisational lineages represented by two
#      constructor identities IN THE SAME RACE?
#
#   5. Do the Stage 32D continuity checks pass if we test the thing
#      we actually care about (carry vs reset) rather than brittle
#      hand-entered exact start counts?
#
# NO DATA ARE MODIFIED.
# NO MODEL IS SAMPLED.
#
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. CONFIG
# ============================================================

TENURE_SCALE_32E = 18.0
TOL_32E = 1e-10

OUTDIR_32E = Path(
    "model_outputs/"
    "joint_driver_car_v2_1_1982_2026_LIVE/"
    "validation_production_tenure_forensics"
)
OUTDIR_32E.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required = [
    "raw",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "resolve_lineage",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Stage 32E must be run after the 2026 evidence build and "
        "Stage 32D.\nMissing objects:\n  "
        + "\n  ".join(missing)
    )


def raw_table(*names):
    for name in names:
        if hasattr(raw, name):
            obj = getattr(raw, name)
            if isinstance(obj, pd.DataFrame):
                return obj.copy()

    raise RuntimeError(
        "Could not locate raw table from candidates: "
        + ", ".join(names)
    )


races_32e = raw_table("races")
results_32e = raw_table("results")
drivers_32e = raw_table("drivers")
constructors_32e = raw_table("constructors")

try:
    statuses_32e = raw_table("status", "statuses")
except RuntimeError:
    statuses_32e = None


# ============================================================
# 2. BASIC LOOKUPS
# ============================================================

race_cols = ["raceId", "year"]

if "round" in races_32e.columns:
    race_cols.append("round")

race_calendar_32e = (
    races_32e[race_cols]
    .drop_duplicates("raceId")
    .copy()
)

if "round" not in race_calendar_32e.columns:
    race_calendar_32e["round"] = (
        race_calendar_32e
        .sort_values(["year", "raceId"])
        .groupby("year")
        .cumcount()
        + 1
    )

race_calendar_32e["year"] = pd.to_numeric(
    race_calendar_32e["year"],
    errors="coerce",
)

race_calendar_32e["round"] = pd.to_numeric(
    race_calendar_32e["round"],
    errors="coerce",
)


driver_lookup_32e = (
    drivers_32e[
        ["driverId", "driverRef"]
    ]
    .drop_duplicates("driverId")
    .rename(columns={"driverRef": "driver"})
)


constructor_name_col = next(
    (
        c
        for c in ["name", "constructor", "constructorName"]
        if c in constructors_32e.columns
    ),
    None,
)

constructor_cols = [
    "constructorId",
    "constructorRef",
]

if constructor_name_col is not None:
    constructor_cols.append(constructor_name_col)

constructor_lookup_32e = (
    constructors_32e[constructor_cols]
    .drop_duplicates("constructorId")
    .copy()
)

if constructor_name_col is not None:
    constructor_lookup_32e = (
        constructor_lookup_32e.rename(
            columns={constructor_name_col: "constructor"}
        )
    )
else:
    constructor_lookup_32e["constructor"] = (
        constructor_lookup_32e["constructorRef"]
    )


# ============================================================
# 3. BUILD RESULT-PRESENCE / ACTUAL-START FLAGS
# ============================================================
#
# "result_presence" deliberately means:
#
#     there is a row in raw.results for this driver/race/team.
#
# We separately create "actual_start" using the same exclusion logic
# as Stage 32D.
#
# This lets the frozen data itself tell us which count its tenure
# covariate was based on.
# ============================================================

results_work_32e = results_32e.copy()


if "status" in results_work_32e.columns:

    status_col_32e = "status"

elif (
    statuses_32e is not None
    and "statusId" in results_work_32e.columns
):

    candidate_status_cols = [
        c
        for c in ["status", "description", "name"]
        if c in statuses_32e.columns
    ]

    if not candidate_status_cols:
        raise RuntimeError(
            "Could not identify status-text column."
        )

    status_source_col = candidate_status_cols[0]

    results_work_32e = results_work_32e.merge(
        statuses_32e[
            ["statusId", status_source_col]
        ].drop_duplicates("statusId"),
        on="statusId",
        how="left",
    )

    status_col_32e = status_source_col

else:
    raise RuntimeError(
        "Could not obtain race-result status text."
    )


status_text_32e = (
    results_work_32e[status_col_32e]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


non_start_regex_32e = (
    r"did not start"
    r"|did not qualify"
    r"|did not prequalify"
    r"|did not pre-qualify"
    r"|withdrawn"
    r"|withdrew"
    r"|not started"
    r"|\bdns\b"
    r"|\bdnq\b"
    r"|\bdnpq\b"
)


results_work_32e["actual_start"] = (
    ~status_text_32e.str.contains(
        non_start_regex_32e,
        regex=True,
        na=False,
    )
)

results_work_32e["result_presence"] = True


result_context_32e = (
    results_work_32e[
        [
            "raceId",
            "driverId",
            "constructorId",
            "result_presence",
            "actual_start",
        ]
    ]
    .groupby(
        ["raceId", "driverId", "constructorId"],
        as_index=False,
    )
    .agg(
        result_presence=("result_presence", "max"),
        actual_start=("actual_start", "max"),
    )
)


# ============================================================
# 4. ADD EVERY DRIVER/RACE PAIR USED IN THE MODERN LIKELIHOOD
# ============================================================

def pair_context_32e(df):

    required_cols = {
        "raceId",
        "constructorId",
        "driverId_a",
        "driverId_b",
    }

    if not required_cols.issubset(df.columns):
        raise RuntimeError(
            f"Likelihood dataframe lacks {required_cols}"
        )

    a = (
        df[
            ["raceId", "constructorId", "driverId_a"]
        ]
        .rename(columns={"driverId_a": "driverId"})
    )

    b = (
        df[
            ["raceId", "constructorId", "driverId_b"]
        ]
        .rename(columns={"driverId_b": "driverId"})
    )

    return pd.concat([a, b], ignore_index=True)


evidence_context_32e = pd.concat(
    [
        pair_context_32e(pace_contrast_all),
        pair_context_32e(pace_level_all),
        pair_context_32e(q_contrast_all),
        pair_context_32e(q_level_all_model),
    ],
    ignore_index=True,
).drop_duplicates()


context_32e = evidence_context_32e.merge(
    result_context_32e,
    on=["raceId", "driverId", "constructorId"],
    how="outer",
)

context_32e["result_presence"] = (
    context_32e["result_presence"]
    .fillna(False)
    .astype(bool)
)

context_32e["actual_start"] = (
    context_32e["actual_start"]
    .fillna(False)
    .astype(bool)
)


context_32e = context_32e.merge(
    race_calendar_32e,
    on="raceId",
    how="left",
)

context_32e = context_32e.merge(
    driver_lookup_32e,
    on="driverId",
    how="left",
)

context_32e = context_32e.merge(
    constructor_lookup_32e,
    on="constructorId",
    how="left",
)


if context_32e["year"].isna().any():
    raise RuntimeError(
        "Some contexts have no race year."
    )

if context_32e["constructorRef"].isna().any():
    raise RuntimeError(
        "Some contexts have no constructorRef."
    )


context_32e["lineage"] = [
    resolve_lineage(ref, yr)
    for ref, yr in zip(
        context_32e["constructorRef"],
        context_32e["year"],
    )
]


context_32e = (
    context_32e
    .sort_values(
        [
            "driverId",
            "year",
            "round",
            "raceId",
            "constructorId",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 5. CONSTRUCT COMPETING EXACT-CONSTRUCTOR TENURE DEFINITIONS
# ============================================================
#
# Candidate A:
#   previous raw.results appearances with exact constructor ID
#
# Candidate B:
#   previous ACTUAL STARTS with exact constructor ID
#
# Candidate C/D:
#   the corresponding INCLUDING-CURRENT-event versions.
#
# An evidence-only row that is absent from raw.results does not
# increment either result or start count.
# ============================================================

candidate_rows = []


for driver_id, g in context_32e.groupby(
    "driverId",
    sort=False,
):

    result_counts = {}
    start_counts = {}

    for row in g.itertuples(index=False):

        cid = int(row.constructorId)

        result_before = result_counts.get(cid, 0)
        start_before = start_counts.get(cid, 0)

        result_including = (
            result_before + int(row.result_presence)
        )

        start_including = (
            start_before + int(row.actual_start)
        )

        candidate_rows.append(
            {
                "raceId": row.raceId,
                "driverId": row.driverId,
                "constructorId": row.constructorId,

                "driver": row.driver,
                "year": int(row.year),
                "round": row.round,
                "constructorRef": row.constructorRef,
                "lineage": row.lineage,

                "result_presence": bool(row.result_presence),
                "actual_start": bool(row.actual_start),

                "n_result_before": result_before,
                "n_start_before": start_before,
                "n_result_including": result_including,
                "n_start_including": start_including,

                "tenure_result_before":
                    np.tanh(
                        result_before / TENURE_SCALE_32E
                    ),

                "tenure_start_before":
                    np.tanh(
                        start_before / TENURE_SCALE_32E
                    ),

                "tenure_result_including":
                    np.tanh(
                        result_including / TENURE_SCALE_32E
                    ),

                "tenure_start_including":
                    np.tanh(
                        start_including / TENURE_SCALE_32E
                    ),
            }
        )

        if row.result_presence:
            result_counts[cid] = result_before + 1

        if row.actual_start:
            start_counts[cid] = start_before + 1


tenure_candidates_32e = pd.DataFrame(candidate_rows)


candidate_cols_32e = [
    "tenure_result_before",
    "tenure_start_before",
    "tenure_result_including",
    "tenure_start_including",
]


print("=" * 76)
print("STAGE 32E — PRODUCTION TENURE FORENSICS")
print("=" * 76)

print(
    "Contexts:",
    f"{len(tenure_candidates_32e):,}",
)
print(
    "raw.results presences:",
    f"{int(tenure_candidates_32e['result_presence'].sum()):,}",
)
print(
    "actual-start presences:",
    f"{int(tenure_candidates_32e['actual_start'].sum()):,}",
)


# ============================================================
# 6. MERGE A/B TENURE CANDIDATES INTO A LIKELIHOOD TABLE
# ============================================================

lookup_base_cols = [
    "raceId",
    "driverId",
    "constructorId",
] + candidate_cols_32e


def merge_candidate_tenures_32e(df):

    x = df.copy()

    a = (
        tenure_candidates_32e[lookup_base_cols]
        .rename(
            columns={
                "driverId": "driverId_a",
                **{
                    c: f"{c}_a"
                    for c in candidate_cols_32e
                },
            }
        )
    )

    b = (
        tenure_candidates_32e[lookup_base_cols]
        .rename(
            columns={
                "driverId": "driverId_b",
                **{
                    c: f"{c}_b"
                    for c in candidate_cols_32e
                },
            }
        )
    )

    x = x.merge(
        a,
        on=[
            "raceId",
            "constructorId",
            "driverId_a",
        ],
        how="left",
    )

    x = x.merge(
        b,
        on=[
            "raceId",
            "constructorId",
            "driverId_b",
        ],
        how="left",
    )

    return x


# ============================================================
# 7. CONTRAST RECONSTRUCTION
# ============================================================

def contrast_forensics_32e(
    df,
    channel_name,
):

    x = merge_candidate_tenures_32e(df)

    if "tenure_diff" not in x.columns:
        raise RuntimeError(
            f"{channel_name} has no tenure_diff column."
        )

    metrics = []

    for c in candidate_cols_32e:

        pred_col = f"pred_{c}"

        x[pred_col] = (
            x[f"{c}_a"]
            - x[f"{c}_b"]
        )

        err_col = f"err_{c}"

        x[err_col] = (
            x[pred_col]
            - x["tenure_diff"]
        )

        valid = (
            x["tenure_diff"].notna()
            & x[pred_col].notna()
        )

        err = x.loc[valid, err_col]

        if valid.sum() >= 2:
            corr = np.corrcoef(
                x.loc[valid, pred_col],
                x.loc[valid, "tenure_diff"],
            )[0, 1]
        else:
            corr = np.nan

        metrics.append(
            {
                "channel": channel_name,
                "candidate": c,
                "rows": int(valid.sum()),
                "missing": int((~valid).sum()),
                "mean_abs_error": float(
                    err.abs().mean()
                ),
                "median_abs_error": float(
                    err.abs().median()
                ),
                "max_abs_error": float(
                    err.abs().max()
                ),
                "rmse": float(
                    np.sqrt(np.mean(err**2))
                ),
                "correlation": float(corr),
                "fraction_exact_1e-10": float(
                    (err.abs() < 1e-10).mean()
                ),
                "fraction_within_1e-6": float(
                    (err.abs() < 1e-6).mean()
                ),
            }
        )

    return x, pd.DataFrame(metrics)


pace_contrast_forensic_32e, pace_contrast_metrics_32e = (
    contrast_forensics_32e(
        pace_contrast_all,
        "pace_contrast",
    )
)

q_contrast_forensic_32e, q_contrast_metrics_32e = (
    contrast_forensics_32e(
        q_contrast_all,
        "q_contrast",
    )
)


contrast_metrics_32e = pd.concat(
    [
        pace_contrast_metrics_32e,
        q_contrast_metrics_32e,
    ],
    ignore_index=True,
)


print("\n" + "=" * 76)
print("CONTRAST RECONSTRUCTION — WHICH TENURE DEFINITION WINS?")
print("=" * 76)

display(
    contrast_metrics_32e.sort_values(
        ["channel", "mean_abs_error"]
    )
)


# ============================================================
# 8. LEVEL FORENSICS
# ============================================================
#
# If the loaded tenure_mean is:
#
#     0.5 * (tenure_a + tenure_b) - race_centre
#
# then, for the correct raw tenure candidate:
#
#     implied_centre
#       = raw_pair_mean - loaded_tenure_mean
#
# must be essentially IDENTICAL for every team in a race.
#
# This tests the raw tenure mechanics WITHOUT needing to know the
# centring population yet.
# ============================================================

def level_forensics_32e(
    df,
    channel_name,
):

    x = merge_candidate_tenures_32e(df)

    if "tenure_mean" not in x.columns:
        raise RuntimeError(
            f"{channel_name} has no tenure_mean column."
        )

    metric_rows = []
    centre_tables = []

    for c in candidate_cols_32e:

        pair_col = f"raw_pair_mean_{c}"
        centre_col = f"implied_centre_{c}"

        x[pair_col] = 0.5 * (
            x[f"{c}_a"]
            + x[f"{c}_b"]
        )

        x[centre_col] = (
            x[pair_col]
            - x["tenure_mean"]
        )

        race_stats = (
            x.groupby("raceId")[centre_col]
            .agg(
                n="count",
                centre_mean="mean",
                centre_median="median",
                centre_sd="std",
                centre_min="min",
                centre_max="max",
            )
            .reset_index()
        )

        race_stats["centre_range"] = (
            race_stats["centre_max"]
            - race_stats["centre_min"]
        )

        race_stats["candidate"] = c
        race_stats["channel"] = channel_name

        centre_tables.append(race_stats)

        metric_rows.append(
            {
                "channel": channel_name,
                "candidate": c,
                "races": int(
                    race_stats["raceId"].nunique()
                ),
                "median_within_race_centre_range":
                    float(
                        race_stats[
                            "centre_range"
                        ].median()
                    ),
                "mean_within_race_centre_range":
                    float(
                        race_stats[
                            "centre_range"
                        ].mean()
                    ),
                "max_within_race_centre_range":
                    float(
                        race_stats[
                            "centre_range"
                        ].max()
                    ),
                "median_within_race_centre_sd":
                    float(
                        race_stats[
                            "centre_sd"
                        ].fillna(0).median()
                    ),
            }
        )

    return (
        x,
        pd.DataFrame(metric_rows),
        pd.concat(
            centre_tables,
            ignore_index=True,
        ),
    )


pace_level_forensic_32e, pace_level_metrics_32e, pace_centres_32e = (
    level_forensics_32e(
        pace_level_all,
        "pace_level",
    )
)

q_level_forensic_32e, q_level_metrics_32e, q_centres_32e = (
    level_forensics_32e(
        q_level_all_model,
        "q1_level",
    )
)


level_metrics_32e = pd.concat(
    [
        pace_level_metrics_32e,
        q_level_metrics_32e,
    ],
    ignore_index=True,
)


print("\n" + "=" * 76)
print("LEVEL RECONSTRUCTION — COMMON RACE-CENTRE TEST")
print("=" * 76)

display(
    level_metrics_32e.sort_values(
        [
            "channel",
            "median_within_race_centre_range",
            "max_within_race_centre_range",
        ]
    )
)


# ============================================================
# 9. IDENTIFY THE BEST PRODUCTION CANDIDATE PER CHANNEL
# ============================================================

best_contrast_32e = (
    contrast_metrics_32e
    .sort_values(
        ["channel", "mean_abs_error"]
    )
    .groupby("channel", as_index=False)
    .first()
)


best_level_32e = (
    level_metrics_32e
    .sort_values(
        [
            "channel",
            "median_within_race_centre_range",
            "max_within_race_centre_range",
        ]
    )
    .groupby("channel", as_index=False)
    .first()
)


print("\n" + "=" * 76)
print("BEST CANDIDATE PER CONTRAST CHANNEL")
print("=" * 76)
display(best_contrast_32e)

print("\n" + "=" * 76)
print("BEST CANDIDATE PER LEVEL CHANNEL")
print("=" * 76)
display(best_level_32e)


# ============================================================
# 10. WORST CONTRAST MISMATCHES FOR THE BEST CANDIDATE
# ============================================================

def show_worst_best_candidate_32e(
    x,
    metric_table,
    channel_name,
    n=30,
):

    best = (
        metric_table
        .sort_values("mean_abs_error")
        .iloc[0]["candidate"]
    )

    pred_col = f"pred_{best}"
    err_col = f"err_{best}"

    cols = [
        c
        for c in [
            "year",
            "raceId",
            "session",
            "constructorId",
            "driver_a",
            "driver_b",
            "tenure_diff",
            pred_col,
            err_col,
        ]
        if c in x.columns
    ]

    print("\n" + "=" * 76)
    print(
        f"{channel_name.upper()} — WORST MISMATCHES "
        f"USING BEST CANDIDATE: {best}"
    )
    print("=" * 76)

    display(
        x.assign(
            abs_error=x[err_col].abs()
        )
        .sort_values(
            "abs_error",
            ascending=False,
        )[cols + ["abs_error"]]
        .head(n)
    )


show_worst_best_candidate_32e(
    pace_contrast_forensic_32e,
    pace_contrast_metrics_32e,
    "pace contrast",
)

show_worst_best_candidate_32e(
    q_contrast_forensic_32e,
    q_contrast_metrics_32e,
    "qualifying contrast",
)


# ============================================================
# 11. RACE-LEVEL LINEAGE COLLISION TEST
# ============================================================
#
# Stage 32D incorrectly flagged multiple constructor identities in
# the SAME YEAR. That is too strict for mid-season rebrands.
#
# The real invalid case is:
#
#      two constructorRefs from one proposed lineage competing in
#      the SAME RACE.
# ============================================================

race_lineage_collision_32e = (
    tenure_candidates_32e.loc[
        tenure_candidates_32e["result_presence"]
        & ~tenure_candidates_32e[
            "lineage"
        ].str.startswith("LITERAL::")
    ]
    .groupby(
        [
            "raceId",
            "year",
            "round",
            "lineage",
        ]
    )
    .agg(
        n_constructor_refs=(
            "constructorRef",
            "nunique",
        ),
        constructor_refs=(
            "constructorRef",
            lambda x:
                ", ".join(sorted(set(x))),
        ),
    )
    .reset_index()
)

race_lineage_collision_32e = (
    race_lineage_collision_32e.loc[
        race_lineage_collision_32e[
            "n_constructor_refs"
        ] > 1
    ]
    .copy()
)


print("\n" + "=" * 76)
print("TRUE SAME-RACE LINEAGE COLLISIONS")
print("=" * 76)

if race_lineage_collision_32e.empty:
    print("None.")
else:
    display(race_lineage_collision_32e)


# ============================================================
# 12. RE-EVALUATE STAGE 32D CHECKS USING MODE, NOT GUESSED COUNTS
# ============================================================
#
# The exact expected counts in Stage 32D were hand-entered and some
# were plainly entry counts rather than start counts.
#
# The structural question is:
#
#   carry -> was there positive familiarity?
#   reset -> was familiarity zero?
# ============================================================

if "continuity_checks_32d" in globals():

    mode_checks_32e = continuity_checks_32d.copy()

    mode_checks_32e["mode_pass"] = np.where(
        mode_checks_32e["expected_mode"] == "carry",
        (
            mode_checks_32e[
                "observed_corrected_starts_before"
            ] > 0
        ),
        (
            mode_checks_32e[
                "observed_corrected_starts_before"
            ] == 0
        ),
    )

    print("\n" + "=" * 76)
    print("STAGE 32D CHECKS — STRUCTURAL CARRY/RESET ONLY")
    print("=" * 76)

    display(
        mode_checks_32e[
            [
                "check",
                "driver",
                "year",
                "constructorRef",
                "expected_mode",
                "observed_corrected_starts_before",
                "mode_pass",
            ]
        ]
    )

else:
    mode_checks_32e = pd.DataFrame()

    print(
        "\ncontinuity_checks_32d not found; "
        "mode-only check skipped."
    )


# ============================================================
# 13. ENTRY VS START DIFFERENCES AT THE SPECIFIC FAILED CHECKS
# ============================================================

focus_driver_years = [
    ("warwick", 1982, "toleman"),
    ("fabi", 1986, "benetton"),
    ("capelli", 1990, "leyton"),
    ("alboreto", 1991, "footwork"),
    ("button", 2009, "brawn"),
    ("perez", 2019, "racing_point"),
    ("gasly", 2020, "alphatauri"),
    ("kvyat", 2020, "alphatauri"),
    ("hulkenberg", 2026, "audi"),
]


focus_rows = []

for driver, year, cref in focus_driver_years:

    sub = tenure_candidates_32e.loc[
        (tenure_candidates_32e["driver"] == driver)
        & (tenure_candidates_32e["year"] == year)
        & (
            tenure_candidates_32e[
                "constructorRef"
            ] == cref
        )
    ].sort_values(["round", "raceId"])

    if sub.empty:
        continue

    r = sub.iloc[0]

    focus_rows.append(
        {
            "driver": driver,
            "year": year,
            "constructorRef": cref,
            "result_appearances_before":
                int(r["n_result_before"]),
            "actual_starts_before":
                int(r["n_start_before"]),
            "tenure_result_before":
                r["tenure_result_before"],
            "tenure_start_before":
                r["tenure_start_before"],
        }
    )


entry_vs_start_focus_32e = pd.DataFrame(focus_rows)


print("\n" + "=" * 76)
print("FAILED 32D COUNT CHECKS — ENTRY/APPEARANCE VS START")
print("=" * 76)
display(entry_vs_start_focus_32e)


# ============================================================
# 14. SAVE
# ============================================================

contrast_metrics_32e.to_csv(
    OUTDIR_32E / "contrast_candidate_reconstruction.csv",
    index=False,
)

level_metrics_32e.to_csv(
    OUTDIR_32E / "level_candidate_common_centre_test.csv",
    index=False,
)

pace_centres_32e.to_csv(
    OUTDIR_32E / "pace_level_implied_centres.csv",
    index=False,
)

q_centres_32e.to_csv(
    OUTDIR_32E / "q1_level_implied_centres.csv",
    index=False,
)

race_lineage_collision_32e.to_csv(
    OUTDIR_32E / "same_race_lineage_collisions.csv",
    index=False,
)

entry_vs_start_focus_32e.to_csv(
    OUTDIR_32E / "entry_vs_start_focus_checks.csv",
    index=False,
)

if not mode_checks_32e.empty:
    mode_checks_32e.to_csv(
        OUTDIR_32E / "continuity_mode_only_checks.csv",
        index=False,
    )


# ============================================================
# 15. TRIAGE
# ============================================================

print("\n" + "=" * 76)
print("STAGE 32E TRIAGE SUMMARY")
print("=" * 76)


for channel in [
    "pace_contrast",
    "q_contrast",
]:

    row = (
        best_contrast_32e.loc[
            best_contrast_32e["channel"] == channel
        ]
        .iloc[0]
    )

    print(
        f"{channel}: best = {row['candidate']}; "
        f"MAE = {row['mean_abs_error']:.6g}; "
        f"max error = {row['max_abs_error']:.6g}; "
        f"exact fraction = "
        f"{row['fraction_exact_1e-10']:.3%}"
    )


for channel in [
    "pace_level",
    "q1_level",
]:

    row = (
        best_level_32e.loc[
            best_level_32e["channel"] == channel
        ]
        .iloc[0]
    )

    print(
        f"{channel}: best raw tenure = "
        f"{row['candidate']}; "
        f"median within-race centre range = "
        f"{row['median_within_race_centre_range']:.6g}; "
        f"max range = "
        f"{row['max_within_race_centre_range']:.6g}"
    )


print(
    "\nSame-race lineage collisions:",
    len(race_lineage_collision_32e),
)

if not mode_checks_32e.empty:
    print(
        "Structural carry/reset checks passed:",
        f"{int(mode_checks_32e['mode_pass'].sum())}/"
        f"{len(mode_checks_32e)}",
    )


print("\nINTERPRETATION:")

print(
    "  • If tenure_result_before wins both contrast channels, "
    "the production variable was based on prior raw.results "
    "appearances/weekends, not actual race starts."
)

print(
    "  • If tenure_start_before wins, Stage 32D's start-based "
    "history was conceptually right and the discrepancy lies "
    "elsewhere."
)

print(
    "  • For the level channels, the correct raw candidate should "
    "produce an essentially zero within-race implied-centre range."
)

print(
    "  • If pace reconstructs cleanly but qualifying contrast does "
    "not, we have isolated a qualifying-specific tenure construction "
    "or alignment issue and should inspect that BEFORE rebuilding "
    "the model."
)

print(
    "  • Same-year lineage overlap is irrelevant if the same-race "
    "collision table is empty."
)

print("\nSaved outputs ->")
print(OUTDIR_32E.resolve())

print("\nStage 32E complete.")

In [ ]:
# ============================================================
# STAGE 32F — ORGANISATIONAL FAMILIARITY MEMORY AUDIT
# ============================================================
#
# PURPOSE
# -------
# Stage 32E established that the frozen production tenure variable was:
#
#     tanh(previous raw.results appearances for literal constructor / 18)
#
# i.e. it used RESULT APPEARANCES / RACE WEEKENDS, not actual race starts.
#
# The remaining conceptual problem is that literal constructor IDs:
#   1. reset familiarity across genuine organisational rebrands;
#   2. cannot handle short departures / returns sensibly.
#
# This stage therefore constructs alternative familiarity stocks for the
# SAME UNDERLYING ORGANISATION.
#
# Key idea:
#
#     experience stock with organisation L
#
# accumulates by one for every raw.results appearance with L.
#
# While the driver is elsewhere, the stored stock decays according to
# MISSED F1 RACE WEEKENDS:
#
#     E_new = E_old * 0.5 ** (missed_GPs / half_life_GPs)
#
# The familiar production saturation transform is retained:
#
#     tenure = tanh(E / 18)
#
# Candidate definitions:
#
#   literal_production  : frozen-model literal constructor definition
#   strict_stint        : reset after any interrupted appearance streak
#   memory_10gp         : 10 missed-GP half-life  (~0.5 modern season)
#   memory_20gp         : 20 missed-GP half-life  (~1 modern season)
#   memory_40gp         : 40 missed-GP half-life  (~2 modern seasons)
#   memory_80gp         : 80 missed-GP half-life  (~4 modern seasons)
#   no_decay            : all prior experience with organisation retained
#
# IMPORTANT:
#   - Uses raw.results appearances, matching Stage 32E.
#   - Familiarity is measured BEFORE the current weekend.
#   - Rebrands carry familiarity.
#   - A return after a short spell elsewhere does NOT automatically reset.
#   - Long absences naturally decay.
#   - Cadillac remains genuinely new.
#   - RB/Faenza and Red Bull/Milton Keynes remain DIFFERENT organisations.
#
# THIS CELL DOES NOT:
#   - mutate the production tables;
#   - replace Stage 3;
#   - build a PyMC model;
#   - sample anything.
#
# ============================================================

from pathlib import Path
from collections import defaultdict
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

STAGE32F_OUT = Path(
    "model_outputs"
) / "joint_driver_car_v2_1_1982_2026_LIVE" / \
    "validation_organisational_familiarity_memory_audit"

STAGE32F_OUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. BASIC CHECKS / FIND MODERN TABLES
# ============================================================

if "raw" not in globals():
    raise RuntimeError(
        "Stage 32F requires the loaded raw F1 data object `raw`."
    )


def _pick_dataframe(label, candidate_names):
    """
    Find the first existing pandas DataFrame from candidate global names.
    """
    for name in candidate_names:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame):
            print(f"{label}: using `{name}` ({len(obj):,} rows)")
            return obj.copy(), name

    raise RuntimeError(
        f"Could not find {label}.\n"
        f"Tried: {candidate_names}"
    )


pace_contrast_32f, pace_contrast_name_32f = _pick_dataframe(
    "Pace contrast table",
    [
        "pace_contrast_all",
        "pace_contrasts_all",
        "pace_contrast",
        "pace_contrasts",
    ],
)

pace_level_32f, pace_level_name_32f = _pick_dataframe(
    "Pace level table",
    [
        "pace_level_all",
        "pace_levels_all",
        "pace_level",
        "pace_levels",
    ],
)

q_contrast_32f, q_contrast_name_32f = _pick_dataframe(
    "Qualifying contrast table",
    [
        "q_contrast_all",
        "q_contrasts_all",
        "quali_contrast_all",
        "qualifying_contrast_all",
        "q_contrast",
        "q_contrasts",
    ],
)

q1_level_32f, q1_level_name_32f = _pick_dataframe(
    "Q1 level table",
    [
        "q_level_all_model",
        "q1_level_all",
        "q_level_all",
        "q1_levels_all",
        "q_level",
        "q1_level",
    ],
)


required_pair_cols = {
    "raceId",
    "driver_a",
    "driver_b",
}

for label, df in [
    ("pace contrast", pace_contrast_32f),
    ("pace level", pace_level_32f),
    ("qualifying contrast", q_contrast_32f),
    ("Q1 level", q1_level_32f),
]:
    missing = required_pair_cols - set(df.columns)

    if missing:
        raise RuntimeError(
            f"{label} table is missing required columns: {missing}"
        )


# ============================================================
# 2. RAW TABLES
# ============================================================

races_32f = raw.races.copy()
results_32f = raw.results.copy()
drivers_32f = raw.drivers.copy()
constructors_32f = raw.constructors.copy()


for col in ["raceId"]:
    races_32f[col] = pd.to_numeric(
        races_32f[col],
        errors="coerce",
    )

for col in ["raceId", "driverId", "constructorId"]:
    results_32f[col] = pd.to_numeric(
        results_32f[col],
        errors="coerce",
    )

drivers_32f["driverId"] = pd.to_numeric(
    drivers_32f["driverId"],
    errors="coerce",
)

constructors_32f["constructorId"] = pd.to_numeric(
    constructors_32f["constructorId"],
    errors="coerce",
)


driver_ref_col = (
    "driverRef"
    if "driverRef" in drivers_32f.columns
    else "driver"
)

constructor_ref_col = (
    "constructorRef"
    if "constructorRef" in constructors_32f.columns
    else "constructor"
)

constructor_name_col = (
    "name"
    if "name" in constructors_32f.columns
    else constructor_ref_col
)


# ============================================================
# 3. GLOBAL RACE ORDER
# ============================================================

race_order_32f = races_32f[
    ["raceId", "year", "round", "name"]
].copy()

race_order_32f["year"] = pd.to_numeric(
    race_order_32f["year"],
    errors="coerce",
)

race_order_32f["round"] = pd.to_numeric(
    race_order_32f["round"],
    errors="coerce",
)

race_order_32f = (
    race_order_32f
    .dropna(subset=["raceId", "year", "round"])
    .sort_values(
        ["year", "round", "raceId"],
        kind="mergesort",
    )
    .drop_duplicates("raceId")
    .reset_index(drop=True)
)

race_order_32f["race_ord"] = np.arange(
    len(race_order_32f),
    dtype=int,
)


# ============================================================
# 4. CURATED ORGANISATIONAL LINEAGE RULES
# ============================================================
#
# These are deliberately year-bounded.
#
# This prevents e.g.
#   1980s works Renault
# from being confused with
#   2000s Enstone Renault,
#
# or historical Alfa Romeo from being confused with
#   2019–23 Hinwil Alfa Romeo.
#
# ============================================================

LINEAGE_RULES_32F = [

    # ---------------- ENSTONE ----------------
    ("toleman",       1981, 1985, "ENSTONE"),
    ("benetton",      1986, 2001, "ENSTONE"),
    ("renault",       2002, 2011, "ENSTONE"),
    ("lotus_f1",      2012, 2015, "ENSTONE"),
    ("renault",       2016, 2020, "ENSTONE"),
    ("alpine",        2021, 9999, "ENSTONE"),

    # ---------------- BRACKLEY ----------------
    ("bar",           1999, 2005, "BRACKLEY"),
    ("honda",         2006, 2008, "BRACKLEY"),
    ("brawn",         2009, 2009, "BRACKLEY"),
    ("mercedes",      2010, 9999, "BRACKLEY"),

    # ---------------- SILVERSTONE ----------------
    ("jordan",        1991, 2005, "SILVERSTONE"),
    ("mf1",           2006, 2006, "SILVERSTONE"),
    ("midland",       2006, 2006, "SILVERSTONE"),
    ("spyker_mf1",    2006, 2006, "SILVERSTONE"),
    ("spyker",        2007, 2007, "SILVERSTONE"),
    ("force_india",   2008, 2018, "SILVERSTONE"),
    ("racing_point",  2019, 2020, "SILVERSTONE"),
    ("aston_martin",  2021, 9999, "SILVERSTONE"),

    # ---------------- FAENZA ----------------
    ("minardi",       1985, 2005, "FAENZA"),
    ("toro_rosso",    2006, 2019, "FAENZA"),
    ("alphatauri",    2020, 2023, "FAENZA"),
    ("rb",            2024, 9999, "FAENZA"),

    # ---------------- HINWIL ----------------
    ("sauber",        1993, 2005, "HINWIL"),
    ("bmw_sauber",    2006, 2009, "HINWIL"),
    ("sauber",        2010, 2018, "HINWIL"),
    ("alfa",          2019, 2023, "HINWIL"),
    ("sauber",        2024, 2025, "HINWIL"),
    ("audi",          2026, 9999, "HINWIL"),

    # ---------------- MILTON KEYNES ----------------
    ("stewart",       1997, 1999, "MILTON_KEYNES"),
    ("jaguar",        2000, 2004, "MILTON_KEYNES"),
    ("red_bull",      2005, 9999, "MILTON_KEYNES"),

    # ---------------- LIGIER / PROST ----------------
    ("ligier",        1976, 1996, "LIGIER_PROST"),
    ("prost",         1997, 2001, "LIGIER_PROST"),

    # ---------------- ARROWS / FOOTWORK ----------------
    ("arrows",        1978, 1990, "ARROWS_FOOTWORK"),
    ("footwork",      1991, 1996, "ARROWS_FOOTWORK"),
    ("arrows",        1997, 2002, "ARROWS_FOOTWORK"),

    # ---------------- MARCH / LEYTON HOUSE ----------------
    ("march",         1987, 1989, "MARCH_LEYTON"),
    ("leyton",        1990, 1991, "MARCH_LEYTON"),
    ("march",         1992, 1992, "MARCH_LEYTON"),

    # ---------------- VIRGIN / MARUSSIA / MANOR ----------------
    ("virgin",        2010, 2011, "VIRGIN_MARUSSIA_MANOR"),
    ("marussia",      2012, 2015, "VIRGIN_MARUSSIA_MANOR"),
    ("manor_marussia",2015, 2016, "VIRGIN_MARUSSIA_MANOR"),
    ("manor",         2016, 2016, "VIRGIN_MARUSSIA_MANOR"),

    # ---------------- TEAM LOTUS / CATERHAM ----------------
    ("lotus_racing",  2010, 2011, "TEAM_LOTUS_CATERHAM"),
    ("caterham",      2012, 2014, "TEAM_LOTUS_CATERHAM"),
]


lineage_rules_df_32f = pd.DataFrame(
    LINEAGE_RULES_32F,
    columns=[
        "constructorRef",
        "start_year",
        "end_year",
        "lineage",
    ],
)

lineage_rules_df_32f.to_csv(
    STAGE32F_OUT / "lineage_rules.csv",
    index=False,
)


# ============================================================
# 5. RESULT-APPEARANCE CONTEXT TABLE
#    + HISTORICAL DUPLICATE-WEEKEND HANDLING
# ============================================================
#
# raw.results is not globally unique in raceId x driverId.
#
# In some older races the same driver can appear in multiple result
# records within the same Grand Prix. For team-familiarity purposes,
# however, a Grand Prix is ONE exposure weekend.
#
# Therefore:
#
#   • first construct every raw result context;
#   • assign organisational lineage;
#   • audit duplicate driver/race records;
#   • collapse them to ONE driver/race exposure;
#   • refuse to continue if a model-era duplicate is structurally
#     ambiguous across multiple organisational lineages.
#
# This is preferable to blindly counting raw result rows, because
# familiarity is intended to represent experience with the team/car
# environment, not database row multiplicity.
#
# ============================================================

contexts_32f = (
    results_32f[
        ["raceId", "driverId", "constructorId"]
    ]
    .merge(
        race_order_32f,
        on="raceId",
        how="left",
        validate="many_to_one",
    )
    .merge(
        drivers_32f[
            ["driverId", driver_ref_col]
        ],
        on="driverId",
        how="left",
        validate="many_to_one",
    )
    .merge(
        constructors_32f[
            [
                "constructorId",
                constructor_ref_col,
                constructor_name_col,
            ]
        ].drop_duplicates("constructorId"),
        on="constructorId",
        how="left",
        validate="many_to_one",
    )
)

contexts_32f = contexts_32f.rename(
    columns={
        driver_ref_col: "driver",
        constructor_ref_col: "constructorRef",
        constructor_name_col: "constructor",
    }
)

contexts_32f["driver"] = (
    contexts_32f["driver"]
    .astype(str)
)

contexts_32f["constructorRef"] = (
    contexts_32f["constructorRef"]
    .astype(str)
)

contexts_32f = contexts_32f.dropna(
    subset=[
        "raceId",
        "driverId",
        "constructorId",
        "year",
        "round",
        "race_ord",
    ]
).copy()

contexts_32f["year"] = (
    contexts_32f["year"]
    .astype(int)
)

contexts_32f["race_ord"] = (
    contexts_32f["race_ord"]
    .astype(int)
)


# ============================================================
# 6. ASSIGN ORGANISATIONAL LINEAGE BEFORE DEDUPLICATION
# ============================================================

contexts_32f["lineage"] = (
    "LITERAL::"
    + contexts_32f["constructorRef"]
)

for (
    ref,
    start_year,
    end_year,
    lineage,
) in LINEAGE_RULES_32F:

    mask = (
        contexts_32f["constructorRef"].eq(ref)
        & contexts_32f["year"].between(
            start_year,
            end_year,
        )
    )

    contexts_32f.loc[
        mask,
        "lineage",
    ] = lineage


# ============================================================
# 6A. AUDIT DUPLICATE DRIVER/RACE RESULT RECORDS
# ============================================================
#
# First attach the race name explicitly. Do not assume that the
# earlier merge happened to preserve a column called "name".
#
# ============================================================

# ------------------------------------------------------------
# Explicit race-name lookup
# ------------------------------------------------------------

race_name_lookup_32f = (
    raw.races[
        ["raceId", "name"]
    ]
    .drop_duplicates("raceId")
    .rename(
        columns={
            "name": "race_name"
        }
    )
)

# Remove any stale race_name column in case this section is rerun.
if "race_name" in contexts_32f.columns:
    contexts_32f = contexts_32f.drop(
        columns=["race_name"]
    )

contexts_32f = contexts_32f.merge(
    race_name_lookup_32f,
    on="raceId",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# Locate duplicate driver/race result records
# ------------------------------------------------------------

duplicate_mask_32f = (
    contexts_32f.duplicated(
        ["raceId", "driverId"],
        keep=False,
    )
)

duplicate_rows_32f = (
    contexts_32f.loc[
        duplicate_mask_32f
    ]
    .copy()
    .sort_values(
        [
            "year",
            "round",
            "driver",
            "constructorRef",
        ]
    )
)


# ------------------------------------------------------------
# Summarise duplicates
# ------------------------------------------------------------

if len(duplicate_rows_32f):

    duplicate_summary_32f = (
        duplicate_rows_32f
        .groupby(
            [
                "raceId",
                "driverId",
            ],
            as_index=False,
        )
        .agg(
            year=(
                "year",
                "first",
            ),

            round=(
                "round",
                "first",
            ),

            race=(
                "race_name",
                "first",
            ),

            driver=(
                "driver",
                "first",
            ),

            result_rows=(
                "constructorId",
                "size",
            ),

            n_constructor_ids=(
                "constructorId",
                "nunique",
            ),

            n_constructor_refs=(
                "constructorRef",
                "nunique",
            ),

            n_lineages=(
                "lineage",
                "nunique",
            ),

            constructor_refs=(
                "constructorRef",
                lambda s:
                    ", ".join(
                        sorted(
                            set(
                                map(str, s)
                            )
                        )
                    ),
            ),

            lineages=(
                "lineage",
                lambda s:
                    ", ".join(
                        sorted(
                            set(
                                map(str, s)
                            )
                        )
                    ),
            ),
        )
        .sort_values(
            [
                "year",
                "round",
                "driver",
            ]
        )
        .reset_index(drop=True)
    )

else:

    duplicate_summary_32f = pd.DataFrame(
        columns=[
            "raceId",
            "driverId",
            "year",
            "round",
            "race",
            "driver",
            "result_rows",
            "n_constructor_ids",
            "n_constructor_refs",
            "n_lineages",
            "constructor_refs",
            "lineages",
        ]
    )


# ------------------------------------------------------------
# Print audit
# ------------------------------------------------------------

print()
print("=" * 76)
print("RAW.RESULTS DUPLICATE DRIVER/RACE AUDIT")
print("=" * 76)

print(
    f"Duplicate result rows: "
    f"{len(duplicate_rows_32f):,}"
)

print(
    f"Duplicate driver/race combinations: "
    f"{len(duplicate_summary_32f):,}"
)

if len(duplicate_summary_32f):

    print(
        f"Year range: "
        f"{int(duplicate_summary_32f['year'].min())}–"
        f"{int(duplicate_summary_32f['year'].max())}"
    )

    print(
        "Duplicate driver/races from 1982 onward:",
        int(
            (
                duplicate_summary_32f[
                    "year"
                ] >= 1982
            ).sum()
        ),
    )

    print(
        "Duplicate driver/races spanning >1 constructor ID:",
        int(
            (
                duplicate_summary_32f[
                    "n_constructor_ids"
                ] > 1
            ).sum()
        ),
    )

    print(
        "Duplicate driver/races spanning >1 organisational lineage:",
        int(
            (
                duplicate_summary_32f[
                    "n_lineages"
                ] > 1
            ).sum()
        ),
    )

    display(
        duplicate_summary_32f
    )

else:

    print(
        "No duplicate driver/race result records found."
    )


# ============================================================
# 6B. STRUCTURAL SAFETY CHECK
# ============================================================
#
# A duplicate OLD historical result is fine: we collapse it to one
# weekend of familiarity.
#
# What would be concerning is a 1982+ driver/race represented under
# more than one organisational lineage, because then "which team was
# he familiar with that weekend?" would genuinely be ambiguous for
# the modelled period.
#
# ============================================================

ambiguous_model_era_duplicates_32f = (
    duplicate_summary_32f.loc[
        (
            duplicate_summary_32f[
                "year"
            ] >= 1982
        )
        & (
            duplicate_summary_32f[
                "n_lineages"
            ] > 1
        )
    ]
    .copy()
)


if len(
    ambiguous_model_era_duplicates_32f
):

    raise RuntimeError(
        "\nStage 32F found a 1982+ duplicate "
        "driver/race spanning multiple organisational "
        "lineages.\n\n"
        "This needs manual inspection before constructing "
        "familiarity.\n\n"
        + ambiguous_model_era_duplicates_32f.to_string(
            index=False
        )
    )


# ============================================================
# 6C. COLLAPSE TO ONE FAMILIARITY EXPOSURE PER DRIVER / GP
# ============================================================
#
# A GP weekend contributes at most ONE unit of familiarity.
#
# If duplicate rows belong to the same organisational lineage, the
# particular constructor record chosen does not affect the lineage
# familiarity calculation.
#
# We retain the first row only as a representative context.
#
# ============================================================

contexts_before_collapse_32f = len(
    contexts_32f
)

contexts_32f = (
    contexts_32f
    .sort_values(
        [
            "driverId",
            "race_ord",
            "constructorId",
        ],
        kind="mergesort",
    )
    .drop_duplicates(
        [
            "raceId",
            "driverId",
        ],
        keep="first",
    )
    .sort_values(
        [
            "driverId",
            "race_ord",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

contexts_removed_by_collapse_32f = (
    contexts_before_collapse_32f
    - len(contexts_32f)
)


# Final uniqueness assertion.
remaining_duplicate_count_32f = int(
    contexts_32f.duplicated(
        [
            "raceId",
            "driverId",
        ]
    ).sum()
)

if remaining_duplicate_count_32f:
    raise RuntimeError(
        "Duplicate driver/race rows remain after collapse."
    )


print()
print("=" * 76)
print("FAMILIARITY-WEEKEND COLLAPSE")
print("=" * 76)

print(
    f"Raw result contexts before collapse: "
    f"{contexts_before_collapse_32f:,}"
)

print(
    f"Duplicate rows removed: "
    f"{contexts_removed_by_collapse_32f:,}"
)

print(
    f"Unique driver/GP familiarity contexts: "
    f"{len(contexts_32f):,}"
)

print(
    f"Remaining duplicate driver/race contexts: "
    f"{remaining_duplicate_count_32f}"
)


# Save the duplicate audit now.
duplicate_summary_32f.to_csv(
    STAGE32F_OUT
    / "raw_results_duplicate_driver_race_audit.csv",
    index=False,
)

duplicate_rows_32f.to_csv(
    STAGE32F_OUT
    / "raw_results_duplicate_driver_race_rows.csv",
    index=False,
)

print(
    "\nHistorical duplicate audit saved."
)


# ============================================================
# 7. FAMILIARITY MEMORY DEFINITIONS
# ============================================================
#
# Half-life is expressed in MISSED GRAND PRIX WEEKENDS.
#
# In the modern calendar:
#
#     10 GP ~ half a season
#     20 GP ~ one season
#     40 GP ~ two seasons
#     80 GP ~ four seasons
#
# Importantly, normal consecutive weekends at the same team have:
#
#     missed_gps = 0
#
# so familiarity accumulates without decay.
#
# ============================================================

MEMORY_HALF_LIVES_32F = {
    "memory_10gp": 10.0,
    "memory_20gp": 20.0,
    "memory_40gp": 40.0,
    "memory_80gp": 80.0,
}

FAMILIARITY_SCALE_32F = 18.0


def _sat(stock):
    return np.tanh(
        np.asarray(stock, dtype=float)
        / FAMILIARITY_SCALE_32F
    )


# Initialise output columns.
contexts_32f["literal_appearances_before"] = 0
contexts_32f["lineage_appearances_before"] = 0

contexts_32f["fam_literal_production"] = np.nan
contexts_32f["fam_strict_stint"] = np.nan
contexts_32f["fam_no_decay"] = np.nan

for cand in MEMORY_HALF_LIVES_32F:
    contexts_32f[f"stock_{cand}"] = np.nan
    contexts_32f[f"fam_{cand}"] = np.nan

contexts_32f["previous_lineage"] = None
contexts_32f["last_same_lineage_race_ord"] = np.nan
contexts_32f["missed_gps_since_same_lineage"] = np.nan
contexts_32f["return_to_known_lineage"] = False


# ------------------------------------------------------------
# Driver-by-driver recursive construction
# ------------------------------------------------------------

for driver_id, idx in contexts_32f.groupby(
    "driverId",
    sort=False,
).groups.items():

    driver_idx = list(idx)

    # Frozen production-style literal-constructor counts.
    literal_counts = defaultdict(int)

    # No-decay organisational counts.
    lineage_counts = defaultdict(int)

    # Lazy-decay states:
    # candidate -> lineage -> (stock_after_last_exposure,
    #                          race_ord_of_last_exposure)
    decay_state = {
        cand: {}
        for cand in MEMORY_HALF_LIVES_32F
    }

    # Last exposure information.
    last_lineage_ord = {}
    previous_lineage = None
    previous_race_ord = None

    # Strict appearance-streak familiarity.
    strict_stint_count_after_previous = 0

    for row_i in driver_idx:

        row = contexts_32f.loc[row_i]

        ref = row["constructorRef"]
        lineage = row["lineage"]
        race_ord = int(row["race_ord"])

        # ----------------------------------------------------
        # Frozen literal-constructor production definition
        # ----------------------------------------------------

        n_literal_before = literal_counts[ref]

        contexts_32f.at[
            row_i,
            "literal_appearances_before",
        ] = n_literal_before

        contexts_32f.at[
            row_i,
            "fam_literal_production",
        ] = float(_sat(n_literal_before))


        # ----------------------------------------------------
        # No-decay organisational memory
        # ----------------------------------------------------

        n_lineage_before = lineage_counts[lineage]

        contexts_32f.at[
            row_i,
            "lineage_appearances_before",
        ] = n_lineage_before

        contexts_32f.at[
            row_i,
            "fam_no_decay",
        ] = float(_sat(n_lineage_before))


        # ----------------------------------------------------
        # Gap / return information
        # ----------------------------------------------------

        if lineage in last_lineage_ord:

            last_same = int(
                last_lineage_ord[lineage]
            )

            missed_same = max(
                0,
                race_ord - last_same - 1,
            )

            contexts_32f.at[
                row_i,
                "last_same_lineage_race_ord",
            ] = last_same

            contexts_32f.at[
                row_i,
                "missed_gps_since_same_lineage",
            ] = missed_same

            # A "return" here means there has been at least
            # one F1 race weekend since the driver's last
            # appearance for this organisation.
            if missed_same > 0:
                contexts_32f.at[
                    row_i,
                    "return_to_known_lineage",
                ] = True

        contexts_32f.at[
            row_i,
            "previous_lineage",
        ] = previous_lineage


        # ----------------------------------------------------
        # Strict continuous-appearance-streak baseline
        # ----------------------------------------------------
        #
        # This deliberately harsh baseline resets if:
        #
        #   - the driver was at another organisation; OR
        #   - the driver missed an intervening GP.
        #
        # It is NOT expected to be our preferred definition.
        # ----------------------------------------------------

        consecutive_same = (
            previous_lineage == lineage
            and previous_race_ord is not None
            and race_ord == previous_race_ord + 1
        )

        if consecutive_same:
            strict_before = (
                strict_stint_count_after_previous
            )
        else:
            strict_before = 0

        contexts_32f.at[
            row_i,
            "fam_strict_stint",
        ] = float(_sat(strict_before))

        strict_stint_count_after_previous = (
            strict_before + 1
        )


        # ----------------------------------------------------
        # Decaying organisational memory
        # ----------------------------------------------------

        for cand, half_life in (
            MEMORY_HALF_LIVES_32F.items()
        ):

            state = decay_state[cand]

            if lineage not in state:
                stock_before = 0.0

            else:
                stock_after_last, last_ord = (
                    state[lineage]
                )

                missed = max(
                    0,
                    race_ord - last_ord - 1,
                )

                stock_before = (
                    stock_after_last
                    * 0.5 ** (
                        missed / half_life
                    )
                )

            contexts_32f.at[
                row_i,
                f"stock_{cand}",
            ] = stock_before

            contexts_32f.at[
                row_i,
                f"fam_{cand}",
            ] = float(
                _sat(stock_before)
            )

            # Current weekend contributes one appearance
            # AFTER the pre-race familiarity has been recorded.
            state[lineage] = (
                stock_before + 1.0,
                race_ord,
            )


        # ----------------------------------------------------
        # Update non-decaying counters
        # ----------------------------------------------------

        literal_counts[ref] += 1
        lineage_counts[lineage] += 1

        last_lineage_ord[lineage] = race_ord

        previous_lineage = lineage
        previous_race_ord = race_ord


# ============================================================
# 8. CANDIDATE COLUMN DEFINITIONS
# ============================================================

FAM_CANDIDATES_32F = [
    "literal_production",
    "strict_stint",
    "memory_10gp",
    "memory_20gp",
    "memory_40gp",
    "memory_80gp",
    "no_decay",
]

FAM_COLS_32F = {
    cand: f"fam_{cand}"
    for cand in FAM_CANDIDATES_32F
}


# ============================================================
# 9. RETURN-TO-OLD-ORGANISATION EPISODES
# ============================================================

# ============================================================
# 9A. RESTORE HUMAN-READABLE RACE / CONSTRUCTOR LABELS
# ============================================================
#
# The familiarity calculations only require IDs / refs, so some
# earlier transformations do not preserve the human-readable
# columns "name" and "constructor".
#
# Reattach them explicitly here for the diagnostic tables.
# This is DISPLAY METADATA ONLY and does not alter any tenure
# calculation.
# ============================================================


# ------------------------------------------------------------
# Race-name lookup
# ------------------------------------------------------------

race_label_lookup_32f = (
    raw.races[
        [
            "raceId",
            "name",
        ]
    ]
    .drop_duplicates("raceId")
    .rename(
        columns={
            "name": "_race_name_32f",
        }
    )
)


# ------------------------------------------------------------
# Constructor-name lookup
# ------------------------------------------------------------

# Ergast/Jolpica-style constructor tables normally use "name".
# Make this robust in case the loaded table has a different
# display-name column.
constructor_name_candidates_32f = [
    "name",
    "constructor",
    "constructorName",
]

constructor_name_col_32f = next(
    (
        c
        for c in constructor_name_candidates_32f
        if c in raw.constructors.columns
    ),
    None,
)

if constructor_name_col_32f is None:

    # constructorRef is always sufficient as a readable fallback.
    constructor_label_lookup_32f = (
        raw.constructors[
            [
                "constructorId",
                "constructorRef",
            ]
        ]
        .drop_duplicates("constructorId")
        .rename(
            columns={
                "constructorRef": "_constructor_name_32f",
            }
        )
    )

else:

    constructor_label_lookup_32f = (
        raw.constructors[
            [
                "constructorId",
                constructor_name_col_32f,
            ]
        ]
        .drop_duplicates("constructorId")
        .rename(
            columns={
                constructor_name_col_32f:
                    "_constructor_name_32f",
            }
        )
    )


# ------------------------------------------------------------
# Remove stale helper columns if this section is rerun
# ------------------------------------------------------------

for col_32f in [
    "_race_name_32f",
    "_constructor_name_32f",
]:

    if col_32f in contexts_32f.columns:

        contexts_32f = contexts_32f.drop(
            columns=[col_32f]
        )


# ------------------------------------------------------------
# Merge labels back onto contexts
# ------------------------------------------------------------

contexts_32f = contexts_32f.merge(
    race_label_lookup_32f,
    on="raceId",
    how="left",
    validate="many_to_one",
)

contexts_32f = contexts_32f.merge(
    constructor_label_lookup_32f,
    on="constructorId",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# Restore the names expected by the remainder of Stage 32F
# ------------------------------------------------------------

contexts_32f["name"] = (
    contexts_32f[
        "_race_name_32f"
    ]
)

contexts_32f["constructor"] = (
    contexts_32f[
        "_constructor_name_32f"
    ]
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

missing_race_labels_32f = int(
    contexts_32f[
        "name"
    ].isna().sum()
)

missing_constructor_labels_32f = int(
    contexts_32f[
        "constructor"
    ].isna().sum()
)

print()
print("=" * 76)
print("DISPLAY-LABEL RESTORATION")
print("=" * 76)

print(
    "Missing race labels:",
    missing_race_labels_32f,
)

print(
    "Missing constructor labels:",
    missing_constructor_labels_32f,
)

if missing_race_labels_32f:
    raise RuntimeError(
        "Some race IDs could not be mapped back to race names."
    )

if missing_constructor_labels_32f:
    raise RuntimeError(
        "Some constructor IDs could not be mapped back to "
        "constructor names."
    )

return_cases_32f = contexts_32f.loc[
    contexts_32f["return_to_known_lineage"]
].copy()

return_keep_cols = [
    "raceId",
    "year",
    "round",
    "name",
    "driver",
    "constructorRef",
    "constructor",
    "lineage",
    "previous_lineage",
    "lineage_appearances_before",
    "missed_gps_since_same_lineage",
] + list(FAM_COLS_32F.values())

return_cases_32f = return_cases_32f[
    return_keep_cols
].sort_values(
    [
        "missed_gps_since_same_lineage",
        "year",
        "round",
    ]
)


# ============================================================
# 10. IMPORTANT RETURN CASES
# ============================================================
#
# These are the cases we especially care about conceptually.
#
# Gasly / Kvyat:
#   short-ish returns to Faenza should retain familiarity.
#
# Ricciardo / Alonso / Räikkönen / etc:
#   long returns should have much less retained familiarity.
#
# Hülkenberg:
#   old 2013 Hinwil experience should have largely decayed by
#   2025, but 2025 Sauber -> 2026 Audi should carry strongly.
#
# ============================================================

FOCUS_RETURN_SPECS_32F = [

    # driver, target year, target lineage, label

    (
        "gasly",
        2019,
        "FAENZA",
        "Gasly returns Red Bull -> Toro Rosso",
    ),

    (
        "kvyat",
        2019,
        "FAENZA",
        "Kvyat returns to Toro Rosso",
    ),

    (
        "ricciardo",
        2023,
        "FAENZA",
        "Ricciardo returns to Faenza after long absence",
    ),

    (
        "alonso",
        2021,
        "ENSTONE",
        "Alonso returns to Enstone",
    ),

    (
        "raikkonen",
        2019,
        "HINWIL",
        "Räikkönen returns to Hinwil",
    ),

    (
        "hulkenberg",
        2025,
        "HINWIL",
        "Hülkenberg returns to Hinwil",
    ),

    (
        "fisichella",
        2005,
        "ENSTONE",
        "Fisichella returns to Enstone",
    ),

    (
        "webber",
        2007,
        "MILTON_KEYNES",
        "Webber returns to Milton Keynes lineage",
    ),

    (
        "kubica",
        2021,
        "HINWIL",
        "Kubica returns to Hinwil",
    ),
]


focus_return_rows_32f = []

for (
    driver,
    target_year,
    lineage,
    label,
) in FOCUS_RETURN_SPECS_32F:

    sub = return_cases_32f.loc[
        return_cases_32f["driver"].eq(driver)
        & return_cases_32f["year"].eq(target_year)
        & return_cases_32f["lineage"].eq(lineage)
    ].copy()

    if sub.empty:
        continue

    # First appearance in that return episode.
    row = (
        sub
        .sort_values(
            ["round", "raceId"]
        )
        .iloc[0]
        .copy()
    )

    row["case"] = label
    focus_return_rows_32f.append(row)


if focus_return_rows_32f:
    focus_returns_32f = pd.DataFrame(
        focus_return_rows_32f
    )
else:
    focus_returns_32f = pd.DataFrame()


# ============================================================
# 11. LOOKUP HELPERS FOR MODERN TABLES
# ============================================================

lookup_cols_32f = (
    ["raceId", "driver"]
    + list(FAM_COLS_32F.values())
)

fam_lookup_32f = (
    contexts_32f[
        lookup_cols_32f
    ]
    .drop_duplicates(
        ["raceId", "driver"]
    )
    .set_index(
        ["raceId", "driver"]
    )
)


def attach_driver_familiarities_32f(df):
    """
    Attach candidate familiarity values for driver_a and driver_b.
    """

    out = df.copy()

    keys_a = pd.MultiIndex.from_arrays(
        [
            out["raceId"].to_numpy(),
            out["driver_a"].astype(str).to_numpy(),
        ],
        names=["raceId", "driver"],
    )

    keys_b = pd.MultiIndex.from_arrays(
        [
            out["raceId"].to_numpy(),
            out["driver_b"].astype(str).to_numpy(),
        ],
        names=["raceId", "driver"],
    )

    for cand, fam_col in FAM_COLS_32F.items():

        source = fam_lookup_32f[fam_col]

        out[f"{cand}_a"] = (
            source
            .reindex(keys_a)
            .to_numpy()
        )

        out[f"{cand}_b"] = (
            source
            .reindex(keys_b)
            .to_numpy()
        )

        out[f"{cand}_diff_raw"] = (
            out[f"{cand}_a"]
            - out[f"{cand}_b"]
        )

        out[f"{cand}_mean_raw"] = (
            0.5
            * (
                out[f"{cand}_a"]
                + out[f"{cand}_b"]
            )
        )

    return out


pace_contrast_fam_32f = (
    attach_driver_familiarities_32f(
        pace_contrast_32f
    )
)

pace_level_fam_32f = (
    attach_driver_familiarities_32f(
        pace_level_32f
    )
)

q_contrast_fam_32f = (
    attach_driver_familiarities_32f(
        q_contrast_32f
    )
)

q1_level_fam_32f = (
    attach_driver_familiarities_32f(
        q1_level_32f
    )
)


# ============================================================
# 12. INFER THE PRODUCTION LEVEL-CENTRING OPERATOR
# ============================================================
#
# The contrast covariates are just:
#
#     tenure_a - tenure_b
#
# The level channels were centred within race.
#
# Stage 32E showed this via the constant implied centre.
#
# Here we explicitly determine whether the production centre is
# best reconstructed by:
#
#     unweighted race mean
#
# or
#
#     weight-weighted race mean.
#
# We then apply the SAME operator to every new candidate.
#
# ============================================================


def _weighted_group_mean(
    df,
    value_col,
    weight_col,
):
    """
    Return one weighted mean per row, grouped by raceId.
    """

    temp = df[
        ["raceId", value_col, weight_col]
    ].copy()

    temp["_vw"] = (
        temp[value_col]
        * temp[weight_col]
    )

    numerator = (
        temp
        .groupby("raceId")["_vw"]
        .transform("sum")
    )

    denominator = (
        temp
        .groupby("raceId")[weight_col]
        .transform("sum")
    )

    return numerator / denominator


def infer_level_centre_method_32f(
    df,
    stored_col="tenure_mean",
):
    """
    Compare candidate centring rules using literal-production familiarity.
    """

    if stored_col not in df.columns:
        raise RuntimeError(
            f"`{stored_col}` not found in level table."
        )

    raw_col = "literal_production_mean_raw"

    # Unweighted mean.
    centre_unweighted = (
        df
        .groupby("raceId")[raw_col]
        .transform("mean")
    )

    pred_unweighted = (
        df[raw_col]
        - centre_unweighted
    )

    err_unweighted = (
        pred_unweighted
        - df[stored_col]
    )

    scores = {
        "unweighted": float(
            np.nanmean(
                np.abs(err_unweighted)
            )
        )
    }

    predictions = {
        "unweighted": pred_unweighted
    }

    # Weighted alternative if available.
    if (
        "weight" in df.columns
        and np.isfinite(
            pd.to_numeric(
                df["weight"],
                errors="coerce",
            )
        ).any()
    ):

        work = df.copy()

        work["weight"] = pd.to_numeric(
            work["weight"],
            errors="coerce",
        )

        centre_weighted = (
            _weighted_group_mean(
                work,
                raw_col,
                "weight",
            )
        )

        pred_weighted = (
            work[raw_col]
            - centre_weighted
        )

        err_weighted = (
            pred_weighted
            - work[stored_col]
        )

        scores["weighted"] = float(
            np.nanmean(
                np.abs(err_weighted)
            )
        )

        predictions["weighted"] = (
            pred_weighted
        )

    best = min(
        scores,
        key=scores.get,
    )

    return best, scores, predictions[best]


pace_level_centre_method_32f, \
pace_level_centre_scores_32f, \
pace_level_literal_rebuild_32f = (
    infer_level_centre_method_32f(
        pace_level_fam_32f,
        "tenure_mean",
    )
)

q1_level_centre_method_32f, \
q1_level_centre_scores_32f, \
q1_level_literal_rebuild_32f = (
    infer_level_centre_method_32f(
        q1_level_fam_32f,
        "tenure_mean",
    )
)


def apply_level_centring_32f(
    df,
    method,
):
    """
    Apply inferred production-style within-race centring to every
    familiarity candidate.
    """

    out = df.copy()

    for cand in FAM_CANDIDATES_32F:

        raw_col = f"{cand}_mean_raw"
        out_col = f"{cand}_tenure_mean"

        if method == "weighted":

            centre = (
                _weighted_group_mean(
                    out,
                    raw_col,
                    "weight",
                )
            )

        elif method == "unweighted":

            centre = (
                out
                .groupby("raceId")[raw_col]
                .transform("mean")
            )

        else:
            raise ValueError(
                f"Unknown centring method: {method}"
            )

        out[out_col] = (
            out[raw_col]
            - centre
        )

    return out


pace_level_fam_32f = (
    apply_level_centring_32f(
        pace_level_fam_32f,
        pace_level_centre_method_32f,
    )
)

q1_level_fam_32f = (
    apply_level_centring_32f(
        q1_level_fam_32f,
        q1_level_centre_method_32f,
    )
)


# ============================================================
# 13. VERIFY FROZEN LITERAL-CONSTRUCTOR RECONSTRUCTION
# ============================================================

reconstruction_rows_32f = []


def contrast_reconstruction_summary_32f(
    label,
    df,
):
    stored = pd.to_numeric(
        df["tenure_diff"],
        errors="coerce",
    )

    pred = pd.to_numeric(
        df["literal_production_diff_raw"],
        errors="coerce",
    )

    err = pred - stored
    abs_err = np.abs(err)

    reconstruction_rows_32f.append(
        {
            "channel": label,
            "rows": len(df),
            "mean_abs_error": float(
                np.nanmean(abs_err)
            ),
            "median_abs_error": float(
                np.nanmedian(abs_err)
            ),
            "max_abs_error": float(
                np.nanmax(abs_err)
            ),
            "fraction_exact_1e-10": float(
                np.nanmean(abs_err < 1e-10)
            ),
        }
    )

    out = df[
        [
            "raceId",
            "driver_a",
            "driver_b",
            "tenure_diff",
        ]
    ].copy()

    if "year" in df.columns:
        out["year"] = df["year"]

    if "session" in df.columns:
        out["session"] = df["session"]

    out["reconstructed"] = pred
    out["error"] = err
    out["abs_error"] = abs_err

    return (
        out
        .sort_values(
            "abs_error",
            ascending=False,
        )
    )


def level_reconstruction_summary_32f(
    label,
    df,
):
    stored = pd.to_numeric(
        df["tenure_mean"],
        errors="coerce",
    )

    pred = pd.to_numeric(
        df[
            "literal_production_tenure_mean"
        ],
        errors="coerce",
    )

    err = pred - stored
    abs_err = np.abs(err)

    reconstruction_rows_32f.append(
        {
            "channel": label,
            "rows": len(df),
            "mean_abs_error": float(
                np.nanmean(abs_err)
            ),
            "median_abs_error": float(
                np.nanmedian(abs_err)
            ),
            "max_abs_error": float(
                np.nanmax(abs_err)
            ),
            "fraction_exact_1e-10": float(
                np.nanmean(abs_err < 1e-10)
            ),
        }
    )

    out = df[
        [
            "raceId",
            "driver_a",
            "driver_b",
            "tenure_mean",
        ]
    ].copy()

    if "year" in df.columns:
        out["year"] = df["year"]

    if "session" in df.columns:
        out["session"] = df["session"]

    out["reconstructed"] = pred
    out["error"] = err
    out["abs_error"] = abs_err

    return (
        out
        .sort_values(
            "abs_error",
            ascending=False,
        )
    )


pace_contrast_recon_32f = (
    contrast_reconstruction_summary_32f(
        "pace_contrast",
        pace_contrast_fam_32f,
    )
)

q_contrast_recon_32f = (
    contrast_reconstruction_summary_32f(
        "q_contrast",
        q_contrast_fam_32f,
    )
)

pace_level_recon_32f = (
    level_reconstruction_summary_32f(
        "pace_level",
        pace_level_fam_32f,
    )
)

q1_level_recon_32f = (
    level_reconstruction_summary_32f(
        "q1_level",
        q1_level_fam_32f,
    )
)

reconstruction_summary_32f = pd.DataFrame(
    reconstruction_rows_32f
)


# ============================================================
# 14. MODERN LIKELIHOOD IMPACT OF EACH MEMORY DEFINITION
# ============================================================

GAMMA_TENURE_REFERENCE_32F = 0.283208


impact_rows_32f = []


def _candidate_impact_32f(
    label,
    df,
    stored_col,
    candidate_suffix,
):

    stored = pd.to_numeric(
        df[stored_col],
        errors="coerce",
    )

    years = (
        pd.to_numeric(
            df["year"],
            errors="coerce",
        )
        if "year" in df.columns
        else pd.Series(
            np.nan,
            index=df.index,
        )
    )

    for cand in FAM_CANDIDATES_32F:

        if cand == "literal_production":
            continue

        new_col = (
            f"{cand}_{candidate_suffix}"
        )

        new = pd.to_numeric(
            df[new_col],
            errors="coerce",
        )

        delta = new - stored
        abs_delta = np.abs(delta)

        valid = (
            np.isfinite(stored)
            & np.isfinite(new)
        )

        valid_2026 = (
            valid
            & years.eq(2026)
        )

        if valid.sum() == 0:
            continue

        impact_rows_32f.append(
            {
                "channel": label,
                "candidate": cand,
                "rows": int(valid.sum()),
                "changed_rows_1e-10": int(
                    np.sum(
                        abs_delta[valid]
                        > 1e-10
                    )
                ),
                "changed_fraction": float(
                    np.mean(
                        abs_delta[valid]
                        > 1e-10
                    )
                ),
                "mean_abs_change": float(
                    np.mean(
                        abs_delta[valid]
                    )
                ),
                "median_abs_change": float(
                    np.median(
                        abs_delta[valid]
                    )
                ),
                "p95_abs_change": float(
                    np.quantile(
                        abs_delta[valid],
                        0.95,
                    )
                ),
                "max_abs_change": float(
                    np.max(
                        abs_delta[valid]
                    )
                ),
                "corr_with_frozen_covariate": float(
                    np.corrcoef(
                        stored[valid],
                        new[valid],
                    )[0, 1]
                ),
                "max_abs_mu_shift_gamma_ref": float(
                    GAMMA_TENURE_REFERENCE_32F
                    * np.max(
                        abs_delta[valid]
                    )
                ),
                "rows_2026": int(
                    valid_2026.sum()
                ),
                "mean_abs_change_2026": (
                    float(
                        np.mean(
                            abs_delta[
                                valid_2026
                            ]
                        )
                    )
                    if valid_2026.sum()
                    else np.nan
                ),
                "max_abs_change_2026": (
                    float(
                        np.max(
                            abs_delta[
                                valid_2026
                            ]
                        )
                    )
                    if valid_2026.sum()
                    else np.nan
                ),
            }
        )


_candidate_impact_32f(
    "pace_contrast",
    pace_contrast_fam_32f,
    "tenure_diff",
    "diff_raw",
)

_candidate_impact_32f(
    "pace_level",
    pace_level_fam_32f,
    "tenure_mean",
    "tenure_mean",
)

_candidate_impact_32f(
    "q_contrast",
    q_contrast_fam_32f,
    "tenure_diff",
    "diff_raw",
)

_candidate_impact_32f(
    "q1_level",
    q1_level_fam_32f,
    "tenure_mean",
    "tenure_mean",
)

candidate_impact_32f = pd.DataFrame(
    impact_rows_32f
)


# ============================================================
# 15. FIRST 2026 APPEARANCE FOR EACH DRIVER
# ============================================================
#
# This is particularly useful for:
#
#   Audi:
#       Bortoleto / Hülkenberg should inherit recent Hinwil
#       familiarity.
#
#   Cadillac:
#       Bottas / Pérez should start from zero.
#
#   Red Bull vs RB:
#       must remain separate organisations.
#
# ============================================================

first_2026_32f = (
    contexts_32f.loc[
        contexts_32f["year"].eq(2026)
    ]
    .sort_values(
        ["driver", "round"]
    )
    .groupby(
        "driver",
        as_index=False,
    )
    .first()
)

first_2026_cols_32f = [
    "driver",
    "constructor",
    "constructorRef",
    "lineage",
    "round",
    "literal_appearances_before",
    "lineage_appearances_before",
    "fam_literal_production",
    "fam_strict_stint",
    "fam_memory_10gp",
    "fam_memory_20gp",
    "fam_memory_40gp",
    "fam_memory_80gp",
    "fam_no_decay",
]

first_2026_32f = (
    first_2026_32f[
        first_2026_cols_32f
    ]
    .sort_values(
        ["constructor", "driver"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 16. RETURN-GAP SUMMARY
# ============================================================

if len(return_cases_32f):

    return_gap_summary_32f = pd.DataFrame(
        {
            "metric": [
                "return episodes",
                "median missed GPs",
                "mean missed GPs",
                "90th percentile missed GPs",
                "maximum missed GPs",
            ],
            "value": [
                len(return_cases_32f),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].median()
                ),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].mean()
                ),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].quantile(0.90)
                ),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].max()
                ),
            ],
        }
    )

else:
    return_gap_summary_32f = pd.DataFrame()


# ============================================================
# 17. THEORETICAL MEMORY RETENTION TABLE
# ============================================================
#
# This gives an intuitive view of what each half-life means.
#
# We use a driver with 40 accumulated team appearances as an
# illustrative reasonably experienced team member.
#
# ============================================================

example_stock_32f = 40.0

gap_grid_32f = np.array(
    [0, 5, 10, 15, 20, 30, 40, 60, 80, 120],
    dtype=float,
)

retention_rows_32f = []

for gap in gap_grid_32f:

    row = {
        "missed_gps": int(gap),
        "pre_absence_stock": example_stock_32f,
        "pre_absence_familiarity": float(
            _sat(example_stock_32f)
        ),
    }

    for cand, half_life in (
        MEMORY_HALF_LIVES_32F.items()
    ):

        stock_after_gap = (
            example_stock_32f
            * 0.5 ** (
                gap / half_life
            )
        )

        row[f"{cand}_stock"] = (
            stock_after_gap
        )

        row[f"{cand}_familiarity"] = float(
            _sat(stock_after_gap)
        )

    row["no_decay_familiarity"] = float(
        _sat(example_stock_32f)
    )

    retention_rows_32f.append(row)


memory_retention_32f = pd.DataFrame(
    retention_rows_32f
)


# ============================================================
# 18. PLOT — MEMORY CURVES
# ============================================================

plot_gaps = np.arange(
    0,
    161,
    dtype=float,
)

plt.figure(
    figsize=(10, 6)
)

for cand, half_life in (
    MEMORY_HALF_LIVES_32F.items()
):

    stock = (
        example_stock_32f
        * 0.5 ** (
            plot_gaps / half_life
        )
    )

    fam = _sat(stock)

    plt.plot(
        plot_gaps,
        fam,
        label=cand,
    )

plt.axhline(
    float(_sat(example_stock_32f)),
    linestyle="--",
    label="no_decay",
)

plt.axhline(
    0.0,
    linestyle=":",
)

plt.xlabel(
    "Missed F1 race weekends since last exposure"
)

plt.ylabel(
    "Retained organisational familiarity"
)

plt.title(
    "Candidate familiarity-memory decay curves\n"
    "(example starting stock = 40 prior appearances)"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    STAGE32F_OUT
    / "familiarity_memory_decay_curves.png",
    dpi=180,
)

plt.show()


# ============================================================
# 19. PLOT — FOCUS RETURN CASE HEATMAP
# ============================================================

if not focus_returns_32f.empty:

    heat_candidates = [
        "fam_strict_stint",
        "fam_memory_10gp",
        "fam_memory_20gp",
        "fam_memory_40gp",
        "fam_memory_80gp",
        "fam_no_decay",
    ]

    heat_labels = [
        "Strict stint",
        "10 GP",
        "20 GP",
        "40 GP",
        "80 GP",
        "No decay",
    ]

    heat = (
        focus_returns_32f[
            heat_candidates
        ]
        .astype(float)
        .to_numpy()
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            max(
                5,
                0.55 * len(
                    focus_returns_32f
                ),
            ),
        )
    )

    im = ax.imshow(
        heat,
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )

    ax.set_xticks(
        np.arange(
            len(heat_labels)
        )
    )

    ax.set_xticklabels(
        heat_labels,
        rotation=30,
        ha="right",
    )

    ylabels = []

    for _, row in (
        focus_returns_32f.iterrows()
    ):

        ylabels.append(
            f"{row['driver']} "
            f"{int(row['year'])} "
            f"({int(row['missed_gps_since_same_lineage'])} GP gap)"
        )

    ax.set_yticks(
        np.arange(
            len(ylabels)
        )
    )

    ax.set_yticklabels(
        ylabels
    )

    ax.set_title(
        "Organisational familiarity retained on important returns"
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
    )

    cbar.set_label(
        "Familiarity entering return weekend"
    )

    # Add values into cells.
    for i in range(
        heat.shape[0]
    ):
        for j in range(
            heat.shape[1]
        ):
            ax.text(
                j,
                i,
                f"{heat[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
            )

    plt.tight_layout()

    plt.savefig(
        STAGE32F_OUT
        / "focus_return_familiarity_heatmap.png",
        dpi=180,
    )

    plt.show()


# ============================================================
# 20. SAVE TABLES
# ============================================================

contexts_32f.to_csv(
    STAGE32F_OUT
    / "organisational_familiarity_contexts.csv",
    index=False,
)

return_cases_32f.to_csv(
    STAGE32F_OUT
    / "all_return_episodes.csv",
    index=False,
)

if not focus_returns_32f.empty:
    focus_returns_32f.to_csv(
        STAGE32F_OUT
        / "focus_return_episodes.csv",
        index=False,
    )

first_2026_32f.to_csv(
    STAGE32F_OUT
    / "first_2026_appearance_familiarity.csv",
    index=False,
)

candidate_impact_32f.to_csv(
    STAGE32F_OUT
    / "candidate_modern_likelihood_covariate_impact.csv",
    index=False,
)

reconstruction_summary_32f.to_csv(
    STAGE32F_OUT
    / "literal_production_reconstruction_summary.csv",
    index=False,
)

pace_contrast_recon_32f.to_csv(
    STAGE32F_OUT
    / "pace_contrast_literal_reconstruction_rows.csv",
    index=False,
)

q_contrast_recon_32f.to_csv(
    STAGE32F_OUT
    / "q_contrast_literal_reconstruction_rows.csv",
    index=False,
)

pace_level_recon_32f.to_csv(
    STAGE32F_OUT
    / "pace_level_literal_reconstruction_rows.csv",
    index=False,
)

q1_level_recon_32f.to_csv(
    STAGE32F_OUT
    / "q1_level_literal_reconstruction_rows.csv",
    index=False,
)

memory_retention_32f.to_csv(
    STAGE32F_OUT
    / "theoretical_memory_retention.csv",
    index=False,
)

if not return_gap_summary_32f.empty:
    return_gap_summary_32f.to_csv(
        STAGE32F_OUT
        / "return_gap_summary.csv",
        index=False,
    )


# ============================================================
# 21. DISPLAY RESULTS
# ============================================================

print()
print("=" * 76)
print("STAGE 32F — ORGANISATIONAL FAMILIARITY MEMORY AUDIT")
print("=" * 76)

print(
    f"Result-appearance contexts: "
    f"{len(contexts_32f):,}"
)

print(
    f"Return-to-known-organisation episodes: "
    f"{len(return_cases_32f):,}"
)

print()
print("=" * 76)
print("PRODUCTION TENURE RECONSTRUCTION")
print("=" * 76)

display(
    reconstruction_summary_32f
)

print()
print(
    "Pace level centring method:",
    pace_level_centre_method_32f,
    pace_level_centre_scores_32f,
)

print(
    "Q1 level centring method:",
    q1_level_centre_method_32f,
    q1_level_centre_scores_32f,
)


# ------------------------------------------------------------
# Show any genuine reconstruction anomalies.
# ------------------------------------------------------------

print()
print("=" * 76)
print("LARGEST LITERAL-PRODUCTION RECONSTRUCTION ERRORS")
print("=" * 76)

print("\nPace contrast:")
display(
    pace_contrast_recon_32f.head(10)
)

print("\nQualifying contrast:")
display(
    q_contrast_recon_32f.head(10)
)

print("\nPace level:")
display(
    pace_level_recon_32f.head(10)
)

print("\nQ1 level:")
display(
    q1_level_recon_32f.head(10)
)


print()
print("=" * 76)
print("FOCUS RETURN EPISODES")
print("=" * 76)

if focus_returns_32f.empty:
    print("No focus return episodes found.")
else:

    focus_display_cols = [
        "case",
        "driver",
        "year",
        "round",
        "constructorRef",
        "lineage",
        "previous_lineage",
        "lineage_appearances_before",
        "missed_gps_since_same_lineage",
        "fam_literal_production",
        "fam_strict_stint",
        "fam_memory_10gp",
        "fam_memory_20gp",
        "fam_memory_40gp",
        "fam_memory_80gp",
        "fam_no_decay",
    ]

    display(
        focus_returns_32f[
            focus_display_cols
        ]
        .sort_values(
            "missed_gps_since_same_lineage"
        )
    )


print()
print("=" * 76)
print("2026 FIRST-APPEARANCE FAMILIARITY")
print("=" * 76)

display(
    first_2026_32f
)


print()
print("=" * 76)
print("MODERN LIKELIHOOD COVARIATE IMPACT")
print("=" * 76)

display(
    candidate_impact_32f.sort_values(
        [
            "channel",
            "candidate",
        ]
    )
)


print()
print("=" * 76)
print("MEMORY RETENTION — EXAMPLE STOCK = 40 APPEARANCES")
print("=" * 76)

retention_display_cols = [
    "missed_gps",
    "pre_absence_familiarity",
    "memory_10gp_familiarity",
    "memory_20gp_familiarity",
    "memory_40gp_familiarity",
    "memory_80gp_familiarity",
    "no_decay_familiarity",
]

display(
    memory_retention_32f[
        retention_display_cols
    ]
)


# ============================================================
# 22. AUTOMATIC TRIAGE
# ============================================================

pace_contrast_maxerr = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("pace_contrast"),
        "max_abs_error",
    ].iloc[0]
)

pace_level_maxerr = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("pace_level"),
        "max_abs_error",
    ].iloc[0]
)

q_contrast_exact = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("q_contrast"),
        "fraction_exact_1e-10",
    ].iloc[0]
)

q1_level_exact = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("q1_level"),
        "fraction_exact_1e-10",
    ].iloc[0]
)


print()
print("=" * 76)
print("STAGE 32F TRIAGE SUMMARY")
print("=" * 76)

print(
    f"Pace contrast literal reconstruction max error: "
    f"{pace_contrast_maxerr:.3e}"
)

print(
    f"Pace level literal reconstruction max error: "
    f"{pace_level_maxerr:.3e}"
)

print(
    f"Qualifying contrast exact fraction: "
    f"{q_contrast_exact:.3%}"
)

print(
    f"Q1 level exact fraction: "
    f"{q1_level_exact:.3%}"
)

print(
    f"\nReturn episodes detected: "
    f"{len(return_cases_32f):,}"
)

print(
    "\nCandidate memory half-lives "
    "(missed race weekends):"
)

for cand, half_life in (
    MEMORY_HALF_LIVES_32F.items()
):
    print(
        f"  {cand}: {half_life:g} GP"
    )


print()
print("INTERPRETATION GUIDE:")
print(
    """
  1. SHORT RETURNS
     Gasly/Kvyat-style short returns should retain substantial
     familiarity under the preferred candidate.

     If a definition drives these close to zero, it is probably
     forgetting too quickly.

  2. LONG RETURNS
     Alonso 2021, Räikkönen 2019, etc. should NOT retain anything
     close to full old-team familiarity.

     If a definition leaves them near 1, it is probably remembering
     for too long.

  3. HÜLKENBERG / HINWIL
     His old 2013 Sauber experience should have largely faded before
     his 2025 return.

     But once he has spent 2025 at Sauber, the 2026 transition to
     Audi should retain that RECENT Hinwil familiarity.

  4. AUDI
     Bortoleto and Hülkenberg in 2026 should no longer be treated as
     completely new to the team merely because constructorId changed.

  5. CADILLAC
     Pérez and Bottas should still begin at zero because Cadillac is
     genuinely a new organisation.

  6. RB vs RED BULL
     Faenza and Milton Keynes remain distinct organisations. A move
     between them is a genuine team change.

  7. DO NOT SELECT A HALF-LIFE JUST FROM THESE TABLES.
     This stage establishes behavioural plausibility and the magnitude
     of the resulting likelihood changes.

     Once we see which candidates behave sensibly, Stage 32G should
     compare the plausible survivors using a proper predictive
     sensitivity test rather than choosing the half-life by eye.
"""
)

print(
    "\nSaved outputs ->\n"
    f"{STAGE32F_OUT.resolve()}"
)

print("\nStage 32F complete.")

In [ ]:
# ============================================================
# STAGE 32F.1 — EXACT DRIVER-LEVEL FAMILIARITY CENTRING
#               + QUALIFYING CONSTRUCTOR-MISMATCH AUDIT
# ============================================================
#
# WHY THIS EXISTS
# ---------------
# Stage 32F correctly constructed the candidate raw familiarity
# histories, but its LEVEL-channel comparison centred the pair means.
#
# Production actually does:
#
#   1. compute driver-level raw tenure;
#   2. centre driver tenure over the WHOLE OBSERVED FIELD:
#
#        pace:       raceId
#        qualifying: raceId x session
#
#   3. construct teammate contrast / level observations:
#
#        contrast = c_a - c_b
#        level    = 0.5 * (c_a + c_b)
#
# This cell reproduces that exact order for every familiarity
# candidate.
#
# It also isolates the handful of qualifying rows for which the
# frozen literal-constructor tenure does not reconstruct.
#
# NO MODEL IS FIT.
# NO PRODUCTION TABLE IS MUTATED.
#
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. REQUIREMENTS
# ============================================================

required_32f1 = [
    "contexts_32f",
    "FAM_CANDIDATES_32F",
    "FAM_COLS_32F",
    "df_pace",
    "df_quali",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "raw",
]

missing_32f1 = [
    x for x in required_32f1
    if x not in globals()
]

if missing_32f1:
    raise RuntimeError(
        "Stage 32F.1 is missing required objects:\n  "
        + "\n  ".join(missing_32f1)
    )


OUTDIR_32F1 = (
    Path("model_outputs")
    / "joint_driver_car_v2_1_1982_2026_LIVE"
    / "validation_familiarity_exact_centring"
)

OUTDIR_32F1.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. UNIQUE DRIVER/RACE FAMILIARITY LOOKUP
# ============================================================

fam_lookup_cols_32f1 = [
    "raceId",
    "driverId",
    "driver",
    "constructorId",
    "constructorRef",
    "lineage",
] + list(FAM_COLS_32F.values())


fam_lookup_driver_32f1 = (
    contexts_32f[
        fam_lookup_cols_32f1
    ]
    .drop_duplicates(
        ["raceId", "driverId"]
    )
    .copy()
)


dup_lookup_32f1 = (
    fam_lookup_driver_32f1
    .duplicated(
        ["raceId", "driverId"]
    )
    .sum()
)

if dup_lookup_32f1:
    raise RuntimeError(
        "Familiarity lookup is not unique "
        "on raceId x driverId."
    )


# ============================================================
# 2. ATTACH RAW CANDIDATE FAMILIARITY TO DRIVER-LEVEL DATA
# ============================================================

candidate_raw_cols_32f1 = [
    FAM_COLS_32F[c]
    for c in FAM_CANDIDATES_32F
]


def attach_raw_familiarity_32f1(
    df,
    *,
    label,
):
    """
    Attach every Stage-32F raw familiarity definition to a
    driver-level pace or qualifying observation using raceId+driverId.

    We deliberately do NOT require constructorId to match here.

    This is important for the corrected organisational definition:
    familiarity belongs to the driver/team organisation at that
    weekend, and should not disappear because one source table uses
    a different literal constructor ID.
    """

    out = df.copy()

    before = len(out)

    lookup = fam_lookup_driver_32f1[
        [
            "raceId",
            "driverId",
        ]
        + candidate_raw_cols_32f1
    ].copy()

    out = out.merge(
        lookup,
        on=[
            "raceId",
            "driverId",
        ],
        how="left",
        validate="many_to_one",
    )

    if len(out) != before:
        raise RuntimeError(
            f"{label}: row count changed during "
            "familiarity merge."
        )

    missing_summary = {}

    for cand in FAM_CANDIDATES_32F:

        col = FAM_COLS_32F[cand]

        missing_summary[cand] = int(
            out[col].isna().sum()
        )

    return out, missing_summary


pace_driver_32f1, pace_missing_32f1 = (
    attach_raw_familiarity_32f1(
        df_pace,
        label="pace",
    )
)

quali_driver_32f1, quali_missing_32f1 = (
    attach_raw_familiarity_32f1(
        df_quali,
        label="qualifying",
    )
)


print("=" * 76)
print("STAGE 32F.1 — DRIVER-LEVEL FAMILIARITY ATTACHMENT")
print("=" * 76)

print(
    "Pace driver observations:",
    f"{len(pace_driver_32f1):,}",
)

print(
    "Qualifying driver/session observations:",
    f"{len(quali_driver_32f1):,}",
)

print(
    "\nMissing candidate familiarity — pace:"
)
print(pace_missing_32f1)

print(
    "\nMissing candidate familiarity — qualifying:"
)
print(quali_missing_32f1)


# ============================================================
# 3. EXACT PRODUCTION-ORDER CENTRING
# ============================================================
#
# Production Stage 3:
#
#   tenure_c =
#       tenure_raw
#       - mean(tenure_raw within group)
#
# pace group:
#       raceId
#
# qualifying group:
#       raceId, session
#
# ============================================================


def centre_driver_familiarity_32f1(
    df,
    *,
    group_cols,
):

    out = df.copy()

    for cand in FAM_CANDIDATES_32F:

        raw_col = FAM_COLS_32F[cand]

        centred_col = (
            f"{cand}_tenure_c"
        )

        out[centred_col] = (
            out[raw_col]
            - out.groupby(
                group_cols
            )[raw_col]
            .transform("mean")
        )

    return out


pace_driver_32f1 = (
    centre_driver_familiarity_32f1(
        pace_driver_32f1,
        group_cols=[
            "raceId",
        ],
    )
)

quali_driver_32f1 = (
    centre_driver_familiarity_32f1(
        quali_driver_32f1,
        group_cols=[
            "raceId",
            "session",
        ],
    )
)


# ============================================================
# 4. REBUILD PAIR COVARIATES FROM DRIVER-LEVEL CENTRED VALUES
# ============================================================


def attach_pair_candidate_covariates_32f1(
    pair_df,
    driver_df,
    *,
    include_session,
):

    out = pair_df.copy()

    key_base = [
        "raceId",
        "driverId",
    ]

    if include_session:
        key_base.append(
            "session"
        )

    candidate_centred_cols = [
        f"{cand}_tenure_c"
        for cand in FAM_CANDIDATES_32F
    ]


    # ---------------- driver A ----------------

    a_lookup = (
        driver_df[
            key_base
            + candidate_centred_cols
        ]
        .copy()
        .rename(
            columns={
                "driverId":
                    "driverId_a",

                **{
                    c: f"{c}_a"
                    for c
                    in candidate_centred_cols
                },
            }
        )
    )

    a_keys = [
        "raceId",
        "driverId_a",
    ]

    if include_session:
        a_keys.append(
            "session"
        )

    out = out.merge(
        a_lookup,
        on=a_keys,
        how="left",
        validate="many_to_one",
    )


    # ---------------- driver B ----------------

    b_lookup = (
        driver_df[
            key_base
            + candidate_centred_cols
        ]
        .copy()
        .rename(
            columns={
                "driverId":
                    "driverId_b",

                **{
                    c: f"{c}_b"
                    for c
                    in candidate_centred_cols
                },
            }
        )
    )

    b_keys = [
        "raceId",
        "driverId_b",
    ]

    if include_session:
        b_keys.append(
            "session"
        )

    out = out.merge(
        b_lookup,
        on=b_keys,
        how="left",
        validate="many_to_one",
    )


    # ---------------- pair quantities ----------------

    for cand in FAM_CANDIDATES_32F:

        c = (
            f"{cand}_tenure_c"
        )

        out[
            f"{cand}_tenure_diff"
        ] = (
            out[f"{c}_a"]
            - out[f"{c}_b"]
        )

        out[
            f"{cand}_tenure_mean"
        ] = (
            0.5
            * (
                out[f"{c}_a"]
                + out[f"{c}_b"]
            )
        )

    return out


pace_contrast_exact_32f1 = (
    attach_pair_candidate_covariates_32f1(
        pace_contrast_all,
        pace_driver_32f1,
        include_session=False,
    )
)

pace_level_exact_32f1 = (
    attach_pair_candidate_covariates_32f1(
        pace_level_all,
        pace_driver_32f1,
        include_session=False,
    )
)

q_contrast_exact_32f1 = (
    attach_pair_candidate_covariates_32f1(
        q_contrast_all,
        quali_driver_32f1,
        include_session=True,
    )
)

q1_level_exact_32f1 = (
    attach_pair_candidate_covariates_32f1(
        q_level_all_model,
        quali_driver_32f1,
        include_session=True,
    )
)


# ============================================================
# 5. FROZEN LITERAL-PRODUCTION RECONSTRUCTION
# ============================================================


recon_rows_32f1 = []


def summarise_reconstruction_32f1(
    *,
    channel,
    df,
    stored_col,
    rebuilt_col,
):

    stored = pd.to_numeric(
        df[stored_col],
        errors="coerce",
    )

    rebuilt = pd.to_numeric(
        df[rebuilt_col],
        errors="coerce",
    )

    valid = (
        stored.notna()
        & rebuilt.notna()
    )

    err = (
        rebuilt[valid]
        - stored[valid]
    )

    ae = err.abs()

    recon_rows_32f1.append(
        {
            "channel":
                channel,

            "rows":
                int(valid.sum()),

            "missing":
                int((~valid).sum()),

            "mean_abs_error":
                float(
                    ae.mean()
                ),

            "median_abs_error":
                float(
                    ae.median()
                ),

            "max_abs_error":
                float(
                    ae.max()
                ),

            "fraction_exact_1e-10":
                float(
                    (ae < 1e-10)
                    .mean()
                ),

            "fraction_within_1e-6":
                float(
                    (ae < 1e-6)
                    .mean()
                ),
        }
    )


summarise_reconstruction_32f1(
    channel="pace_contrast",
    df=pace_contrast_exact_32f1,
    stored_col="tenure_diff",
    rebuilt_col=(
        "literal_production_"
        "tenure_diff"
    ),
)

summarise_reconstruction_32f1(
    channel="pace_level",
    df=pace_level_exact_32f1,
    stored_col="tenure_mean",
    rebuilt_col=(
        "literal_production_"
        "tenure_mean"
    ),
)

summarise_reconstruction_32f1(
    channel="q_contrast",
    df=q_contrast_exact_32f1,
    stored_col="tenure_diff",
    rebuilt_col=(
        "literal_production_"
        "tenure_diff"
    ),
)

summarise_reconstruction_32f1(
    channel="q1_level",
    df=q1_level_exact_32f1,
    stored_col="tenure_mean",
    rebuilt_col=(
        "literal_production_"
        "tenure_mean"
    ),
)


reconstruction_summary_32f1 = (
    pd.DataFrame(
        recon_rows_32f1
    )
)


print()
print("=" * 76)
print("EXACT PRODUCTION-ORDER RECONSTRUCTION")
print("=" * 76)

display(
    reconstruction_summary_32f1
)


# ============================================================
# 6. CORRECT CANDIDATE IMPACT TABLE
# ============================================================

GAMMA_REFERENCE_32F1 = 0.283208

impact_rows_32f1 = []


def candidate_impact_32f1(
    *,
    channel,
    df,
    stored_col,
    candidate_suffix,
):

    stored = pd.to_numeric(
        df[stored_col],
        errors="coerce",
    )

    for cand in FAM_CANDIDATES_32F:

        if cand == "literal_production":
            continue

        new_col = (
            f"{cand}_"
            f"{candidate_suffix}"
        )

        new = pd.to_numeric(
            df[new_col],
            errors="coerce",
        )

        valid = (
            stored.notna()
            & new.notna()
        )

        delta = (
            new[valid]
            - stored[valid]
        )

        ae = delta.abs()

        if len(ae) == 0:
            continue

        impact_rows_32f1.append(
            {
                "channel":
                    channel,

                "candidate":
                    cand,

                "rows":
                    int(valid.sum()),

                "changed_rows_1e-10":
                    int(
                        (
                            ae > 1e-10
                        ).sum()
                    ),

                "changed_fraction":
                    float(
                        (
                            ae > 1e-10
                        ).mean()
                    ),

                "mean_abs_change":
                    float(
                        ae.mean()
                    ),

                "median_abs_change":
                    float(
                        ae.median()
                    ),

                "p95_abs_change":
                    float(
                        ae.quantile(
                            0.95
                        )
                    ),

                "max_abs_change":
                    float(
                        ae.max()
                    ),

                "corr_with_frozen":
                    float(
                        np.corrcoef(
                            stored[valid],
                            new[valid],
                        )[0, 1]
                    ),

                "max_abs_mu_shift_gamma_ref":
                    float(
                        GAMMA_REFERENCE_32F1
                        * ae.max()
                    ),
            }
        )


candidate_impact_32f1(
    channel="pace_contrast",
    df=pace_contrast_exact_32f1,
    stored_col="tenure_diff",
    candidate_suffix="tenure_diff",
)

candidate_impact_32f1(
    channel="pace_level",
    df=pace_level_exact_32f1,
    stored_col="tenure_mean",
    candidate_suffix="tenure_mean",
)

candidate_impact_32f1(
    channel="q_contrast",
    df=q_contrast_exact_32f1,
    stored_col="tenure_diff",
    candidate_suffix="tenure_diff",
)

candidate_impact_32f1(
    channel="q1_level",
    df=q1_level_exact_32f1,
    stored_col="tenure_mean",
    candidate_suffix="tenure_mean",
)


candidate_impact_exact_32f1 = (
    pd.DataFrame(
        impact_rows_32f1
    )
)


print()
print("=" * 76)
print("CORRECTED MODERN-LIKELIHOOD COVARIATE IMPACT")
print("=" * 76)

display(
    candidate_impact_exact_32f1
    .sort_values(
        [
            "channel",
            "candidate",
        ]
    )
)


# ============================================================
# 7. FIND EVERY NON-RECONSTRUCTING QUALIFYING PAIR
# ============================================================

q_contrast_audit_32f1 = (
    q_contrast_exact_32f1.copy()
)

q_contrast_audit_32f1[
    "literal_reconstruction_error"
] = (
    q_contrast_audit_32f1[
        "literal_production_tenure_diff"
    ]
    - q_contrast_audit_32f1[
        "tenure_diff"
    ]
)

q_contrast_audit_32f1[
    "abs_literal_reconstruction_error"
] = (
    q_contrast_audit_32f1[
        "literal_reconstruction_error"
    ].abs()
)


q_anomaly_pairs_32f1 = (
    q_contrast_audit_32f1.loc[
        q_contrast_audit_32f1[
            "abs_literal_reconstruction_error"
        ] > 1e-8
    ]
    .copy()
    .sort_values(
        "abs_literal_reconstruction_error",
        ascending=False,
    )
)


print()
print("=" * 76)
print("QUALIFYING LITERAL-TENURE ANOMALY PAIRS")
print("=" * 76)

print(
    "Non-reconstructing qualifying rows:",
    len(q_anomaly_pairs_32f1),
)

if len(q_anomaly_pairs_32f1):

    display_cols = [
        c
        for c in [
            "year",
            "raceId",
            "session",
            "constructorId",
            "driver_a",
            "driver_b",
            "tenure_diff",
            "literal_production_tenure_diff",
            "literal_reconstruction_error",
            "abs_literal_reconstruction_error",
        ]
        if c in q_anomaly_pairs_32f1.columns
    ]

    display(
        q_anomaly_pairs_32f1[
            display_cols
        ]
    )


# ============================================================
# 8. DRIVER-LEVEL QUALIFYING CONSTRUCTOR-ID MISMATCH AUDIT
# ============================================================
#
# Old production joins tenure on:
#
#     raceId + driverId + constructorId
#
# so if the qualifying constructor ID differs from the result-table
# constructor ID, frozen production tenure becomes zero.
#
# The corrected organisational familiarity should not do that.
#
# ============================================================

results_constructor_32f1 = (
    raw.results[
        [
            "raceId",
            "driverId",
            "constructorId",
        ]
    ]
    .drop_duplicates(
        [
            "raceId",
            "driverId",
        ]
    )
    .rename(
        columns={
            "constructorId":
                "results_constructorId"
        }
    )
)


quali_constructor_audit_32f1 = (
    df_quali[
        [
            "raceId",
            "year",
            "session",
            "driverId",
            "driver",
            "constructorId",
            "tenure_c",
        ]
    ]
    .merge(
        results_constructor_32f1,
        on=[
            "raceId",
            "driverId",
        ],
        how="left",
        validate="many_to_one",
    )
)


quali_constructor_audit_32f1[
    "constructor_id_mismatch"
] = (
    quali_constructor_audit_32f1[
        "results_constructorId"
    ].notna()
    &
    (
        quali_constructor_audit_32f1[
            "constructorId"
        ]
        !=
        quali_constructor_audit_32f1[
            "results_constructorId"
        ]
    )
)


q_mismatch_rows_32f1 = (
    quali_constructor_audit_32f1.loc[
        quali_constructor_audit_32f1[
            "constructor_id_mismatch"
        ]
    ]
    .copy()
)


print()
print("=" * 76)
print("QUALIFYING vs RESULTS CONSTRUCTOR-ID MISMATCHES")
print("=" * 76)

print(
    "Driver/session rows with mismatch:",
    len(q_mismatch_rows_32f1),
)

if len(q_mismatch_rows_32f1):

    display(
        q_mismatch_rows_32f1
        .sort_values(
            [
                "year",
                "raceId",
                "session",
                "driver",
            ]
        )
    )


# ============================================================
# 9. SHORTLIST-SPECIFIC IMPACT
# ============================================================
#
# Behavioural audit suggests:
#
#   10 GP = useful fast-forgetting lower bound
#   20 GP = plausible
#   40 GP = plausible
#
# Strict stint, 80 GP and no-decay remain in the full table but
# are not proposed as Stage-32G finalists.
#
# ============================================================

SHORTLIST_32F1 = [
    "memory_10gp",
    "memory_20gp",
    "memory_40gp",
]


shortlist_impact_32f1 = (
    candidate_impact_exact_32f1.loc[
        candidate_impact_exact_32f1[
            "candidate"
        ].isin(
            SHORTLIST_32F1
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("STAGE 32G SHORTLIST — CORRECTED COVARIATE IMPACT")
print("=" * 76)

display(
    shortlist_impact_32f1
    .sort_values(
        [
            "channel",
            "candidate",
        ]
    )
)


# ============================================================
# 10. SAVE
# ============================================================

reconstruction_summary_32f1.to_csv(
    OUTDIR_32F1
    / "exact_production_reconstruction.csv",
    index=False,
)

candidate_impact_exact_32f1.to_csv(
    OUTDIR_32F1
    / "candidate_covariate_impact_exact_centring.csv",
    index=False,
)

q_anomaly_pairs_32f1.to_csv(
    OUTDIR_32F1
    / "qualifying_literal_tenure_anomaly_pairs.csv",
    index=False,
)

q_mismatch_rows_32f1.to_csv(
    OUTDIR_32F1
    / "qualifying_results_constructor_id_mismatches.csv",
    index=False,
)

shortlist_impact_32f1.to_csv(
    OUTDIR_32F1
    / "stage32g_shortlist_covariate_impact.csv",
    index=False,
)


# ============================================================
# 11. TRIAGE
# ============================================================

print()
print("=" * 76)
print("STAGE 32F.1 TRIAGE SUMMARY")
print("=" * 76)

for _, r in (
    reconstruction_summary_32f1
    .iterrows()
):

    print(
        f"{r['channel']:15s}: "
        f"exact={r['fraction_exact_1e-10']:.3%}, "
        f"MAE={r['mean_abs_error']:.6g}, "
        f"max={r['max_abs_error']:.6g}"
    )


print(
    "\nQualifying anomaly pair rows:",
    len(q_anomaly_pairs_32f1),
)

print(
    "Qualifying driver/session constructor-ID mismatches:",
    len(q_mismatch_rows_32f1),
)


print(
    "\nProvisional Stage-32G predictive shortlist:"
)

for candidate in SHORTLIST_32F1:
    print(
        f"  • {candidate}"
    )


print(
    "\nCurrent interpretation:"
)

print(
    "  • strict_stint forgets too aggressively;"
)

print(
    "  • no_decay retains implausibly ancient familiarity;"
)

print(
    "  • memory_80gp appears too persistent on very long returns;"
)

print(
    "  • memory_10gp is retained as a fast-decay lower bound;"
)

print(
    "  • memory_20gp and memory_40gp are the principal plausible "
    "candidates;"
)

print(
    "  • Stage 32G should choose among the plausible candidates "
    "using predictive evidence, not subjective preference."
)


print(
    "\nSaved outputs ->"
)
print(
    OUTDIR_32F1.resolve()
)

print(
    "\nStage 32F.1 complete."
)

In [ ]:
# ============================================================
# STAGE 32F.2 — OBSERVATION-SAFE ORGANISATIONAL FAMILIARITY
# ============================================================
#
# PURPOSE
# -------
# Stage 32F/32F.1 constructed familiarity on raw.results contexts.
#
# That is sufficient for almost all modern observations, but a tiny
# number of qualifying rows have no matching same-race results context
# or use a different literal constructorId.
#
# Conceptually, that should NOT make familiarity missing:
#
#   familiarity entering race r
#
# depends only on the driver's PRIOR experience with the underlying
# organisation before race r.
#
# This cell therefore evaluates familiarity for ANY driver/race/team
# observation using the driver's stored pre-event organisational
# history.
#
# It:
#   • keeps result appearances as the exposure source;
#   • uses organisational lineage, not literal constructor ID;
#   • evaluates 10/20/40 GP memory at arbitrary modern observations;
#   • fixes the 2015 Manor constructor-ID mismatch naturally;
#   • centres at driver level exactly as production did;
#   • rebuilds the four modern likelihood tenure covariates;
#   • verifies zero missing values.
#
# NO MODEL IS FIT HERE.
#
# ============================================================

from pathlib import Path
import bisect

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 0. REQUIREMENTS
# ============================================================

required_32f2 = [
    "contexts_32f",
    "LINEAGE_RULES_32F",
    "MEMORY_HALF_LIVES_32F",
    "df_pace",
    "df_quali",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "raw",
]

missing_32f2 = [
    x for x in required_32f2
    if x not in globals()
]

if missing_32f2:
    raise RuntimeError(
        "Stage 32F.2 missing required objects:\n  "
        + "\n  ".join(missing_32f2)
    )


OUTDIR_32F2 = (
    Path("model_outputs")
    / "joint_driver_car_v2_1_1982_2026_LIVE"
    / "validation_observation_safe_familiarity"
)

OUTDIR_32F2.mkdir(
    parents=True,
    exist_ok=True,
)


SHORTLIST_32F2 = {
    "memory_10gp": 10.0,
    "memory_20gp": 20.0,
    "memory_40gp": 40.0,
}

FAMILIARITY_SCALE_32F2 = 18.0


def sat_32f2(x):
    return np.tanh(
        np.asarray(x, dtype=float)
        / FAMILIARITY_SCALE_32F2
    )


# ============================================================
# 1. RACE ORDER LOOKUP
# ============================================================

race_order_lookup_32f2 = (
    raw.races[
        [
            "raceId",
            "year",
            "round",
        ]
    ]
    .copy()
)

for col in [
    "raceId",
    "year",
    "round",
]:
    race_order_lookup_32f2[col] = (
        pd.to_numeric(
            race_order_lookup_32f2[col],
            errors="coerce",
        )
    )

race_order_lookup_32f2 = (
    race_order_lookup_32f2
    .dropna(
        subset=[
            "raceId",
            "year",
            "round",
        ]
    )
    .sort_values(
        [
            "year",
            "round",
            "raceId",
        ],
        kind="mergesort",
    )
    .drop_duplicates("raceId")
    .reset_index(drop=True)
)

race_order_lookup_32f2["race_ord"] = (
    np.arange(
        len(race_order_lookup_32f2),
        dtype=int,
    )
)


# ============================================================
# 2. CONSTRUCTOR-ID -> REF LOOKUP
# ============================================================

constructor_lookup_32f2 = (
    raw.constructors[
        [
            "constructorId",
            "constructorRef",
        ]
    ]
    .copy()
)

constructor_lookup_32f2[
    "constructorId"
] = pd.to_numeric(
    constructor_lookup_32f2[
        "constructorId"
    ],
    errors="coerce",
)

constructor_lookup_32f2 = (
    constructor_lookup_32f2
    .dropna(
        subset=[
            "constructorId",
            "constructorRef",
        ]
    )
    .drop_duplicates("constructorId")
)

constructor_lookup_32f2[
    "constructorId"
] = constructor_lookup_32f2[
    "constructorId"
].astype(int)


# ============================================================
# 3. LINEAGE RESOLVER
# ============================================================

def lineage_from_ref_year_32f2(
    constructor_ref,
    year,
):
    """
    Resolve one constructorRef/year to the curated underlying
    organisational lineage.
    """

    constructor_ref = str(
        constructor_ref
    )

    year = int(year)

    for (
        ref,
        start_year,
        end_year,
        lineage,
    ) in LINEAGE_RULES_32F:

        if (
            constructor_ref == ref
            and start_year <= year <= end_year
        ):
            return lineage

    return (
        "LITERAL::"
        + constructor_ref
    )


# ============================================================
# 4. BUILD EXPOSURE HISTORY
# ============================================================
#
# contexts_32f is one exposure per driver per GP after the historical
# duplicate collapse.
#
# For each driver-lineage pair we need:
#
#   race ordinals of exposures
#
# and, for each candidate half-life:
#
#   familiarity stock immediately AFTER each exposure.
#
# This lets us query familiarity at an arbitrary later observation.
#
# ============================================================

history_32f2 = {}


for (
    driver_id,
    lineage,
), grp in contexts_32f.groupby(
    [
        "driverId",
        "lineage",
    ],
    sort=False,
):

    g = (
        grp
        .sort_values(
            "race_ord"
        )
        .copy()
    )

    ords = (
        g["race_ord"]
        .astype(int)
        .to_numpy()
    )

    entry = {
        "race_ord":
            ords,
    }

    for (
        candidate,
        half_life,
    ) in SHORTLIST_32F2.items():

        stock_after = np.zeros(
            len(g),
            dtype=float,
        )

        previous_stock_after = 0.0
        previous_ord = None

        for j, race_ord in enumerate(
            ords
        ):

            if previous_ord is None:

                stock_before = 0.0

            else:

                missed = max(
                    0,
                    int(race_ord)
                    - int(previous_ord)
                    - 1,
                )

                stock_before = (
                    previous_stock_after
                    * 0.5 ** (
                        missed
                        / half_life
                    )
                )

            current_stock_after = (
                stock_before
                + 1.0
            )

            stock_after[j] = (
                current_stock_after
            )

            previous_stock_after = (
                current_stock_after
            )

            previous_ord = int(
                race_ord
            )

        entry[
            f"stock_after_{candidate}"
        ] = stock_after

    history_32f2[
        (
            int(driver_id),
            str(lineage),
        )
    ] = entry


# ============================================================
# 5. QUERY FAMILIARITY BEFORE AN ARBITRARY OBSERVATION
# ============================================================

def familiarity_before_observation_32f2(
    driver_id,
    lineage,
    race_ord,
    candidate,
):
    """
    Familiarity entering an arbitrary race/weekend.

    Uses the most recent PRIOR exposure to this organisation.

    Current-race result existence is irrelevant.
    """

    key = (
        int(driver_id),
        str(lineage),
    )

    if key not in history_32f2:
        return 0.0

    hist = history_32f2[key]

    ords = hist[
        "race_ord"
    ]

    # Index of final exposure STRICTLY before current race.
    pos = (
        np.searchsorted(
            ords,
            int(race_ord),
            side="left",
        )
        - 1
    )

    if pos < 0:
        return 0.0

    last_ord = int(
        ords[pos]
    )

    stock_after = float(
        hist[
            f"stock_after_{candidate}"
        ][pos]
    )

    half_life = (
        SHORTLIST_32F2[
            candidate
        ]
    )

    missed = max(
        0,
        int(race_ord)
        - last_ord
        - 1,
    )

    stock_before = (
        stock_after
        * 0.5 ** (
            missed
            / half_life
        )
    )

    return float(
        sat_32f2(
            stock_before
        )
    )


# ============================================================
# 6. PREPARE ARBITRARY DRIVER-LEVEL OBSERVATIONS
# ============================================================

def prepare_observations_32f2(
    df,
    *,
    label,
):
    """
    Add year/race_ord/constructorRef/lineage and evaluate each
    familiarity candidate.
    """

    out = df.copy()

    # Remove stale helper columns if this cell is rerun.
    stale = [
        "race_ord",
        "_year_lookup",
        "constructorRef_32f2",
        "lineage_32f2",
    ]

    out = out.drop(
        columns=[
            c for c in stale
            if c in out.columns
        ],
        errors="ignore",
    )

    # Race order / fallback year.
    race_meta = (
        race_order_lookup_32f2[
            [
                "raceId",
                "year",
                "race_ord",
            ]
        ]
        .rename(
            columns={
                "year":
                    "_year_lookup",
            }
        )
    )

    out = out.merge(
        race_meta,
        on="raceId",
        how="left",
        validate="many_to_one",
    )

    if "year" not in out.columns:
        out["year"] = (
            out["_year_lookup"]
        )
    else:
        out["year"] = (
            pd.to_numeric(
                out["year"],
                errors="coerce",
            )
            .fillna(
                out["_year_lookup"]
            )
        )

    # Constructor ref.
    out = out.merge(
        constructor_lookup_32f2
        .rename(
            columns={
                "constructorRef":
                    "constructorRef_32f2",
            }
        ),
        on="constructorId",
        how="left",
        validate="many_to_one",
    )

    if out[
        "race_ord"
    ].isna().any():

        raise RuntimeError(
            f"{label}: missing race_ord "
            "after race lookup."
        )

    if out[
        "constructorRef_32f2"
    ].isna().any():

        bad = out.loc[
            out[
                "constructorRef_32f2"
            ].isna(),
            [
                "raceId",
                "driverId",
                "constructorId",
            ],
        ]

        raise RuntimeError(
            f"{label}: constructorRef lookup failed:\n"
            + bad.head(20).to_string(
                index=False
            )
        )

    # Organisational lineage at this observation.
    out["lineage_32f2"] = [
        lineage_from_ref_year_32f2(
            ref,
            year,
        )
        for ref, year in zip(
            out[
                "constructorRef_32f2"
            ],
            out["year"],
        )
    ]

    # Candidate raw familiarity.
    for candidate in (
        SHORTLIST_32F2
    ):

        vals = []

        for (
            driver_id,
            lineage,
            race_ord,
        ) in zip(
            out["driverId"],
            out["lineage_32f2"],
            out["race_ord"],
        ):

            vals.append(
                familiarity_before_observation_32f2(
                    driver_id,
                    lineage,
                    race_ord,
                    candidate,
                )
            )

        out[
            f"{candidate}_raw"
        ] = np.asarray(
            vals,
            dtype=float,
        )

    return out


pace_obs_32f2 = (
    prepare_observations_32f2(
        df_pace,
        label="pace",
    )
)

quali_obs_32f2 = (
    prepare_observations_32f2(
        df_quali,
        label="qualifying",
    )
)


# ============================================================
# 7. CENTRE EXACTLY AS THE MODEL DATA PIPELINE DOES
# ============================================================

for candidate in (
    SHORTLIST_32F2
):

    # Pace: whole race field.
    pace_obs_32f2[
        f"{candidate}_tenure_c"
    ] = (
        pace_obs_32f2[
            f"{candidate}_raw"
        ]
        -
        pace_obs_32f2.groupby(
            "raceId"
        )[
            f"{candidate}_raw"
        ]
        .transform("mean")
    )

    # Qualifying: each timed session separately.
    quali_obs_32f2[
        f"{candidate}_tenure_c"
    ] = (
        quali_obs_32f2[
            f"{candidate}_raw"
        ]
        -
        quali_obs_32f2.groupby(
            [
                "raceId",
                "session",
            ]
        )[
            f"{candidate}_raw"
        ]
        .transform("mean")
    )


# ============================================================
# 8. ATTACH CENTRED DRIVER FAMILIARITIES TO PAIR TABLES
# ============================================================

def rebuild_pair_table_32f2(
    pair_df,
    driver_df,
    *,
    include_session,
):
    """
    Rebuild candidate tenure_diff and tenure_mean using the exact
    driver-level centred candidate familiarities.
    """

    out = pair_df.copy()

    candidate_cols = [
        f"{c}_tenure_c"
        for c in SHORTLIST_32F2
    ]

    base_cols = [
        "raceId",
        "driverId",
    ]

    if include_session:
        base_cols.append(
            "session"
        )

    # Driver A.
    a = (
        driver_df[
            base_cols
            + candidate_cols
        ]
        .copy()
        .rename(
            columns={
                "driverId":
                    "driverId_a",

                **{
                    c:
                        f"{c}_a"
                    for c
                    in candidate_cols
                },
            }
        )
    )

    a_keys = [
        "raceId",
        "driverId_a",
    ]

    if include_session:
        a_keys.append(
            "session"
        )

    out = out.merge(
        a,
        on=a_keys,
        how="left",
        validate="many_to_one",
    )

    # Driver B.
    b = (
        driver_df[
            base_cols
            + candidate_cols
        ]
        .copy()
        .rename(
            columns={
                "driverId":
                    "driverId_b",

                **{
                    c:
                        f"{c}_b"
                    for c
                    in candidate_cols
                },
            }
        )
    )

    b_keys = [
        "raceId",
        "driverId_b",
    ]

    if include_session:
        b_keys.append(
            "session"
        )

    out = out.merge(
        b,
        on=b_keys,
        how="left",
        validate="many_to_one",
    )

    for candidate in (
        SHORTLIST_32F2
    ):

        c = (
            f"{candidate}_tenure_c"
        )

        out[
            f"{candidate}_tenure_diff"
        ] = (
            out[f"{c}_a"]
            -
            out[f"{c}_b"]
        )

        out[
            f"{candidate}_tenure_mean"
        ] = (
            0.5
            * (
                out[f"{c}_a"]
                +
                out[f"{c}_b"]
            )
        )

    return out


pace_contrast_32f2 = (
    rebuild_pair_table_32f2(
        pace_contrast_all,
        pace_obs_32f2,
        include_session=False,
    )
)

pace_level_32f2 = (
    rebuild_pair_table_32f2(
        pace_level_all,
        pace_obs_32f2,
        include_session=False,
    )
)

q_contrast_32f2 = (
    rebuild_pair_table_32f2(
        q_contrast_all,
        quali_obs_32f2,
        include_session=True,
    )
)

q1_level_32f2 = (
    rebuild_pair_table_32f2(
        q_level_all_model,
        quali_obs_32f2,
        include_session=True,
    )
)


# ============================================================
# 9. ZERO-MISSINGNESS AUDIT
# ============================================================

missing_rows_32f2 = []


for channel, df, suffix in [
    (
        "pace_contrast",
        pace_contrast_32f2,
        "tenure_diff",
    ),
    (
        "pace_level",
        pace_level_32f2,
        "tenure_mean",
    ),
    (
        "q_contrast",
        q_contrast_32f2,
        "tenure_diff",
    ),
    (
        "q1_level",
        q1_level_32f2,
        "tenure_mean",
    ),
]:

    for candidate in (
        SHORTLIST_32F2
    ):

        col = (
            f"{candidate}_{suffix}"
        )

        n_missing = int(
            df[col]
            .isna()
            .sum()
        )

        missing_rows_32f2.append(
            {
                "channel":
                    channel,

                "candidate":
                    candidate,

                "rows":
                    len(df),

                "missing":
                    n_missing,

                "finite":
                    int(
                        np.isfinite(
                            df[col]
                        ).sum()
                    ),
            }
        )


missingness_32f2 = (
    pd.DataFrame(
        missing_rows_32f2
    )
)


print()
print("=" * 76)
print("STAGE 32F.2 — CANDIDATE MISSINGNESS")
print("=" * 76)

display(
    missingness_32f2
)


if (
    missingness_32f2[
        "missing"
    ].sum()
    != 0
):

    raise RuntimeError(
        "Stage 32F.2 still has missing "
        "candidate familiarity values."
    )


# ============================================================
# 10. IDENTIFY THE OBSERVATIONS THAT STAGE 32F.1 MISSED
# ============================================================

# Candidate used only as a convenient probe; all three should now
# resolve identically in terms of missingness.
probe_candidate_32f2 = (
    "memory_20gp"
)

previous_q_missing_32f2 = (
    q_contrast_exact_32f1.loc[
        q_contrast_exact_32f1[
            f"{probe_candidate_32f2}_tenure_diff"
        ].isna()
    ]
    .copy()
    if (
        "q_contrast_exact_32f1"
        in globals()
    )
    else pd.DataFrame()
)


print()
print("=" * 76)
print("PREVIOUSLY MISSING QUALIFYING PAIRS")
print("=" * 76)

if previous_q_missing_32f2.empty:

    print(
        "No previous missing qualifying pairs "
        "found in memory."
    )

else:

    keys = [
        c for c in [
            "year",
            "raceId",
            "session",
            "driver_a",
            "driver_b",
            "constructorId",
        ]
        if c in previous_q_missing_32f2.columns
    ]

    display(
        previous_q_missing_32f2[
            keys
        ]
    )


# ============================================================
# 11. CONFIRM 2015 MANOR IS NOW HANDLED ORGANISATIONALLY
# ============================================================

manor_2015_32f2 = (
    quali_obs_32f2.loc[
        (
            quali_obs_32f2[
                "year"
            ].eq(2015)
        )
        &
        (
            quali_obs_32f2[
                "constructorId"
            ].eq(206)
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("2015 MANOR QUALIFYING — CORRECTED ORGANISATIONAL FAMILIARITY")
print("=" * 76)

manor_cols_32f2 = [
    c for c in [
        "raceId",
        "year",
        "session",
        "driver",
        "driverId",
        "constructorId",
        "constructorRef_32f2",
        "lineage_32f2",
        "memory_10gp_raw",
        "memory_20gp_raw",
        "memory_40gp_raw",
        "memory_10gp_tenure_c",
        "memory_20gp_tenure_c",
        "memory_40gp_tenure_c",
    ]
    if c in manor_2015_32f2.columns
]

display(
    manor_2015_32f2[
        manor_cols_32f2
    ]
    .sort_values(
        [
            "raceId",
            "session",
            "driver",
        ]
    )
)


# ============================================================
# 12. CROSS-CANDIDATE SENSITIVITY
# ============================================================
#
# These are the rows that can actually discriminate between the
# 10/20/40-GP definitions in Stage 32G.
#
# ============================================================

sensitivity_rows_32f2 = []


for (
    channel,
    df,
    suffix,
) in [
    (
        "pace_contrast",
        pace_contrast_32f2,
        "tenure_diff",
    ),
    (
        "pace_level",
        pace_level_32f2,
        "tenure_mean",
    ),
    (
        "q_contrast",
        q_contrast_32f2,
        "tenure_diff",
    ),
    (
        "q1_level",
        q1_level_32f2,
        "tenure_mean",
    ),
]:

    vals = np.column_stack(
        [
            df[
                f"{candidate}_{suffix}"
            ].to_numpy(
                dtype=float
            )
            for candidate in SHORTLIST_32F2
        ]
    )

    spread = (
        np.max(
            vals,
            axis=1,
        )
        -
        np.min(
            vals,
            axis=1,
        )
    )

    temp = pd.DataFrame(
        {
            "channel":
                channel,

            "raceId":
                df[
                    "raceId"
                ].to_numpy(),

            "year":
                (
                    df["year"]
                    .to_numpy()
                    if "year" in df.columns
                    else np.nan
                ),

            "candidate_spread":
                spread,
        }
    )

    sensitivity_rows_32f2.append(
        temp
    )


sensitivity_32f2 = (
    pd.concat(
        sensitivity_rows_32f2,
        ignore_index=True,
    )
)


sensitivity_summary_32f2 = (
    sensitivity_32f2.groupby(
        "channel",
        as_index=False,
    )
    .agg(
        rows=(
            "candidate_spread",
            "size",
        ),
        rows_different_1e_6=(
            "candidate_spread",
            lambda x:
                int(
                    (
                        x > 1e-6
                    ).sum()
                ),
        ),
        rows_different_0_01=(
            "candidate_spread",
            lambda x:
                int(
                    (
                        x > 0.01
                    ).sum()
                ),
        ),
        median_spread=(
            "candidate_spread",
            "median",
        ),
        p90_spread=(
            "candidate_spread",
            lambda x:
                float(
                    np.quantile(
                        x,
                        0.90,
                    )
                ),
        ),
        p95_spread=(
            "candidate_spread",
            lambda x:
                float(
                    np.quantile(
                        x,
                        0.95,
                    )
                ),
        ),
        max_spread=(
            "candidate_spread",
            "max",
        ),
    )
)


print()
print("=" * 76)
print("10/20/40-GP DISCRIMINATING OBSERVATIONS")
print("=" * 76)

display(
    sensitivity_summary_32f2
)


# ============================================================
# 13. SAVE
# ============================================================

pace_obs_32f2.to_csv(
    OUTDIR_32F2
    / "pace_driver_candidate_familiarity.csv",
    index=False,
)

quali_obs_32f2.to_csv(
    OUTDIR_32F2
    / "quali_driver_candidate_familiarity.csv",
    index=False,
)

missingness_32f2.to_csv(
    OUTDIR_32F2
    / "candidate_missingness.csv",
    index=False,
)

sensitivity_32f2.to_csv(
    OUTDIR_32F2
    / "candidate_discrimination_rows.csv",
    index=False,
)

sensitivity_summary_32f2.to_csv(
    OUTDIR_32F2
    / "candidate_discrimination_summary.csv",
    index=False,
)


# ============================================================
# 14. FINAL TRIAGE
# ============================================================

print()
print("=" * 76)
print("STAGE 32F.2 TRIAGE SUMMARY")
print("=" * 76)

print(
    "Candidate familiarity missing values:",
    int(
        missingness_32f2[
            "missing"
        ].sum()
    ),
)

print(
    "Pace driver rows:",
    f"{len(pace_obs_32f2):,}",
)

print(
    "Qualifying driver/session rows:",
    f"{len(quali_obs_32f2):,}",
)

print(
    "\nStage-32G candidates:"
)

for candidate in (
    SHORTLIST_32F2
):
    print(
        f"  • {candidate}"
    )

print(
    "\nIf missing values = 0, the candidate covariates are now "
    "safe for predictive model comparison."
)

print(
    "\nSaved outputs ->"
)
print(
    OUTDIR_32F2.resolve()
)

print(
    "\nStage 32F.2 complete."
)

In [ ]:
# ============================================================
# STAGE 32F.2A — FIX 2015 MARUSSIA / MANOR LINEAGE ALIAS
# ============================================================
#
# The raw data use two constructor identities for the same
# underlying 2015 operation:
#
#   qualifying: marussia
#   results:    manor
#
# For TEAM FAMILIARITY these must belong to the same lineage.
#
# Car / constructor-season identity remains untouched.
#
# This patch changes ONLY the organisational-familiarity lineage.
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. SHOW CURRENT RELEVANT RULES
# ============================================================

print("=" * 76)
print("CURRENT MARUSSIA / MANOR LINEAGE RULES")
print("=" * 76)

for rule in LINEAGE_RULES_32F:

    ref, start_year, end_year, lineage = rule

    if (
        ref in {
            "virgin",
            "marussia",
            "manor",
        }
        or lineage
        == "VIRGIN_MARUSSIA_MANOR"
    ):
        print(rule)


# ============================================================
# 2. PATCH MANOR TO INCLUDE 2015
# ============================================================
#
# Desired mapping:
#
#   Virgin   2010–2011
#   Marussia 2012–2015
#   Manor    2015–2016
#
# The 2015 overlap is deliberate because the source tables use
# different constructorRefs for the same underlying operation.
#
# ============================================================

patched_rules_32f2a = []

manor_rule_found_32f2a = False


for (
    ref,
    start_year,
    end_year,
    lineage,
) in LINEAGE_RULES_32F:

    if (
        ref == "manor"
        and lineage
        == "VIRGIN_MARUSSIA_MANOR"
    ):

        patched_rules_32f2a.append(
            (
                ref,
                min(
                    int(start_year),
                    2015,
                ),
                int(end_year),
                lineage,
            )
        )

        manor_rule_found_32f2a = True

    else:

        patched_rules_32f2a.append(
            (
                ref,
                int(start_year),
                int(end_year),
                lineage,
            )
        )


if not manor_rule_found_32f2a:

    patched_rules_32f2a.append(
        (
            "manor",
            2015,
            2016,
            "VIRGIN_MARUSSIA_MANOR",
        )
    )


LINEAGE_RULES_32F = (
    patched_rules_32f2a
)


# ============================================================
# 3. LINEAGE RESOLVER
# ============================================================

def lineage_from_ref_year_32f2a(
    constructor_ref,
    year,
):

    constructor_ref = str(
        constructor_ref
    )

    year = int(
        year
    )

    for (
        ref,
        start_year,
        end_year,
        lineage,
    ) in LINEAGE_RULES_32F:

        if (
            constructor_ref == ref
            and
            start_year
            <= year
            <= end_year
        ):

            return lineage

    return (
        "LITERAL::"
        + constructor_ref
    )


# ============================================================
# 4. REASSIGN LINEAGES IN THE RESULT-EXPOSURE CONTEXT TABLE
# ============================================================
#
# Stage 32F.2 builds its memory history from contexts_32f.
# Therefore patching the rule alone is not enough: the already
# constructed contexts table must also be re-labelled.
#
# ============================================================

required_context_cols_32f2a = {
    "constructorRef",
    "year",
    "lineage",
}

missing_context_cols_32f2a = (
    required_context_cols_32f2a
    - set(
        contexts_32f.columns
    )
)

if missing_context_cols_32f2a:

    raise RuntimeError(
        "contexts_32f is missing required columns: "
        + ", ".join(
            sorted(
                missing_context_cols_32f2a
            )
        )
    )


contexts_32f[
    "lineage"
] = [

    lineage_from_ref_year_32f2a(
        ref,
        year,
    )

    for ref, year
    in zip(
        contexts_32f[
            "constructorRef"
        ],
        contexts_32f[
            "year"
        ],
    )
]


# ============================================================
# 5. VERIFY THE TWO 2015 SOURCE IDENTITIES NOW COINCIDE
# ============================================================

marussia_manor_2015_32f2a = (
    contexts_32f.loc[
        (
            contexts_32f[
                "year"
            ].eq(2015)
        )
        &
        (
            contexts_32f[
                "constructorRef"
            ].isin(
                [
                    "marussia",
                    "manor",
                ]
            )
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("2015 MARUSSIA / MANOR RESULT CONTEXTS AFTER PATCH")
print("=" * 76)


cols_32f2a = [
    c
    for c in [
        "raceId",
        "year",
        "round",
        "driver",
        "driverId",
        "constructorId",
        "constructorRef",
        "lineage",
        "race_ord",
    ]
    if c
    in marussia_manor_2015_32f2a.columns
]


display(
    marussia_manor_2015_32f2a[
        cols_32f2a
    ]
    .sort_values(
        [
            "race_ord",
            "driver",
        ]
    )
)


lineages_2015_32f2a = set(
    marussia_manor_2015_32f2a[
        "lineage"
    ]
    .dropna()
    .astype(str)
)


if (
    marussia_manor_2015_32f2a[
        "constructorRef"
    ]
    .nunique()
    >= 1
):

    assert (
        lineages_2015_32f2a
        ==
        {
            "VIRGIN_MARUSSIA_MANOR"
        }
    ), (
        "2015 Marussia/Manor identities are still not "
        "mapped to one organisational lineage."
    )


# ============================================================
# 6. SPECIFIC DRIVER EXPOSURE CHECK
# ============================================================

drivers_32f2a = [
    "stevens",
    "merhi",
    "rossi",
]


driver_exposure_32f2a = (
    contexts_32f.loc[
        (
            contexts_32f[
                "year"
            ].eq(2015)
        )
        &
        (
            contexts_32f[
                "driver"
            ].isin(
                drivers_32f2a
            )
        )
        &
        (
            contexts_32f[
                "lineage"
            ].eq(
                "VIRGIN_MARUSSIA_MANOR"
            )
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("2015 MANOR DRIVER EXPOSURE HISTORY")
print("=" * 76)


display(
    driver_exposure_32f2a[
        [
            c
            for c in [
                "raceId",
                "round",
                "driver",
                "constructorRef",
                "lineage",
                "race_ord",
            ]
            if c
            in driver_exposure_32f2a.columns
        ]
    ]
    .sort_values(
        [
            "race_ord",
            "driver",
        ]
    )
)


# Stevens and Merhi should have repeated exposure records.
counts_32f2a = (
    driver_exposure_32f2a
    .groupby(
        "driver"
    )
    .size()
    .rename(
        "2015_lineage_exposures"
    )
    .reset_index()
)


print()
print(
    "Exposure counts:"
)

display(
    counts_32f2a
)


for driver in [
    "stevens",
    "merhi",
]:

    n = int(
        (
            driver_exposure_32f2a[
                "driver"
            ]
            == driver
        ).sum()
    )

    if n <= 1:

        raise RuntimeError(
            f"{driver} still has only {n} "
            "2015 familiarity exposure(s); "
            "the lineage patch has not worked."
        )


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print()
print("=" * 76)
print("STAGE 32F.2A SUMMARY")
print("=" * 76)

print(
    "2015 Marussia/Manor familiarity lineage:",
    "VIRGIN_MARUSSIA_MANOR",
)

print(
    "Car constructor IDs remain unchanged."
)

print(
    "contexts_32f lineages have been rebuilt."
)

print()
print(
    "NOW RERUN THE FULL STAGE 32F.2 CELL."
)

print(
    "Do not run Stage 32G yet."
)

In [ ]:
# ============================================================
# STAGE 32G-A — FAMILIARITY HALF-LIFE PREDICTIVE-CV SETUP
# ============================================================
#
# PURPOSE
# -------
# Prepare a fair held-out predictive comparison of:
#
#     memory_10gp
#     memory_20gp
#     memory_40gp
#
# WITHOUT fitting anything yet.
#
# IMPORTANT DESIGN CHOICES
# ------------------------
#
# 1. 2026 is deliberately excluded from model selection.
#    We choose the familiarity rule using evidence through 2025,
#    then apply the selected rule prospectively to live 2026.
#
# 2. Holdout unit = ENTIRE RACE WEEKEND.
#    Pace and qualifying observations from the same GP always
#    belong to the same fold.
#
# 3. Three folds, stratified within season.
#    Each season contributes races to all three folds.
#
# 4. Candidate familiarity is inserted into BOTH:
#       - modern likelihoods
#       - historical 1982–95 PL likelihoods
#
# 5. This cell DOES NOT mutate the frozen production tables.
#
# 6. This cell DOES NOT sample the model.
#
# AFTER RUNNING:
#   paste the final triage output before Stage 32G-B.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. CONFIGURATION
# ============================================================

STAGE32G_CANDIDATES = [
    "memory_10gp",
    "memory_20gp",
    "memory_40gp",
]

STAGE32G_N_FOLDS = 3

STAGE32G_SEED = 3207

STAGE32G_MODERN_FIRST_YEAR = 1996
STAGE32G_SELECTION_LAST_YEAR = 2025

# Secondary diagnostic thresholds.
# These do NOT define the main CV score.
STAGE32G_SENSITIVE_THRESHOLDS = [
    0.01,
    0.02,
    0.05,
]


STAGE32G_OUTDIR = (
    Path(CFG.output_dir)
    /
    "validation_familiarity_predictive_cv"
)

STAGE32G_OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_32g = [
    "CFG",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "hist_quali",
    "hist_race",
    "contexts_32f",
    "pack_rankings",
    "idx_all",
    "build_joint_model_1982_2025",
]


missing_32g = [
    name
    for name in required_32g
    if name not in globals()
]


if missing_32g:

    raise RuntimeError(
        "Stage 32G-A is missing required objects:\n  "
        +
        "\n  ".join(
            missing_32g
        )
    )


# ============================================================
# 2. MAKE SURE WE ARE SELECTING USING PRE-2026 DATA ONLY
# ============================================================

modern_tables_32g = {
    "pace_contrast":
        pace_contrast_all,

    "pace_level":
        pace_level_all,

    "q_contrast":
        q_contrast_all,

    "q1_level":
        q_level_all_model,
}


for channel, df in modern_tables_32g.items():

    years = pd.to_numeric(
        df["year"],
        errors="coerce",
    )

    max_year = int(
        years.max()
    )

    min_year = int(
        years.min()
    )

    print(
        f"{channel:15s}: "
        f"{len(df):,} rows; "
        f"{min_year}–{max_year}"
    )

    if max_year > STAGE32G_SELECTION_LAST_YEAR:

        raise RuntimeError(
            f"{channel} contains {max_year} data. "
            "Stage 32G deliberately selects the half-life "
            "using data only through 2025."
        )


# ============================================================
# 3. LOCATE THE CANDIDATE MODERN COVARIATE COLUMNS
# ============================================================
#
# Stage 32F.2 created candidate-specific columns, but this helper
# does not assume one exact naming convention.
#
# Contrast tables need the candidate equivalent of tenure_diff.
# Level tables need the candidate equivalent of tenure_mean.
# ============================================================

def find_candidate_covariate_col_32g(
    df,
    candidate,
    kind,
):
    """
    Find a Stage-32F.2 candidate familiarity covariate.

    kind:
        "diff" -> teammate contrast
        "mean" -> team level
    """

    cols = list(
        df.columns
    )

    lower = {
        c:
            str(c).lower()
        for c in cols
    }


    # --------------------------------------------------------
    # Preferred exact / near-exact names
    # --------------------------------------------------------

    preferred = [
        f"{candidate}_tenure_{kind}",
        f"{candidate}_{kind}",
        f"tenure_{kind}_{candidate}",
        f"fam_{candidate}_{kind}",
        f"{candidate}_fam_{kind}",
    ]


    for wanted in preferred:

        if wanted in cols:

            return wanted


    # --------------------------------------------------------
    # General candidate + kind match
    # --------------------------------------------------------

    matches = [
        c
        for c in cols
        if (
            candidate.lower()
            in lower[c]
            and
            kind.lower()
            in lower[c]
        )
    ]


    if len(matches) == 1:

        return matches[0]


    # --------------------------------------------------------
    # Candidate + tenure + kind match
    # --------------------------------------------------------

    matches = [
        c
        for c in cols
        if (
            candidate.lower()
            in lower[c]
            and
            "tenure"
            in lower[c]
            and
            kind.lower()
            in lower[c]
        )
    ]


    if len(matches) == 1:

        return matches[0]


    # --------------------------------------------------------
    # Failure: show all candidate-like columns.
    # --------------------------------------------------------

    candidate_like = [
        c
        for c in cols
        if candidate.lower()
        in lower[c]
    ]


    raise RuntimeError(
        "\nCould not uniquely identify candidate covariate.\n"
        f"Candidate: {candidate}\n"
        f"Kind: {kind}\n"
        f"Candidate-like columns:\n"
        +
        "\n".join(
            f"  {c}"
            for c in candidate_like
        )
    )


modern_candidate_cols_32g = {}


for channel, df in modern_tables_32g.items():

    kind = (
        "diff"
        if channel
        in {
            "pace_contrast",
            "q_contrast",
        }
        else
        "mean"
    )

    modern_candidate_cols_32g[
        channel
    ] = {}

    for candidate in STAGE32G_CANDIDATES:

        modern_candidate_cols_32g[
            channel
        ][candidate] = (
            find_candidate_covariate_col_32g(
                df,
                candidate,
                kind,
            )
        )


print()
print("=" * 76)
print("CANDIDATE MODERN COVARIATE COLUMNS")
print("=" * 76)


candidate_column_rows_32g = []


for channel in modern_candidate_cols_32g:

    for candidate, col in (
        modern_candidate_cols_32g[
            channel
        ].items()
    ):

        candidate_column_rows_32g.append(
            {
                "channel":
                    channel,

                "candidate":
                    candidate,

                "column":
                    col,
            }
        )


candidate_column_table_32g = pd.DataFrame(
    candidate_column_rows_32g
)


display(
    candidate_column_table_32g
)


# ============================================================
# 4. VERIFY MODERN CANDIDATE VALUES
# ============================================================

modern_candidate_checks_32g = []


for channel, df in modern_tables_32g.items():

    for candidate in STAGE32G_CANDIDATES:

        col = (
            modern_candidate_cols_32g[
                channel
            ][candidate]
        )

        values = pd.to_numeric(
            df[col],
            errors="coerce",
        )

        modern_candidate_checks_32g.append(
            {
                "channel":
                    channel,

                "candidate":
                    candidate,

                "rows":
                    len(df),

                "missing":
                    int(
                        values.isna().sum()
                    ),

                "finite":
                    int(
                        np.isfinite(
                            values
                        ).sum()
                    ),

                "min":
                    float(
                        values.min()
                    ),

                "max":
                    float(
                        values.max()
                    ),
            }
        )


modern_candidate_checks_32g = (
    pd.DataFrame(
        modern_candidate_checks_32g
    )
)


print()
print("=" * 76)
print("MODERN CANDIDATE VALUE CHECK")
print("=" * 76)

display(
    modern_candidate_checks_32g
)


if (
    modern_candidate_checks_32g[
        "missing"
    ].sum()
    != 0
):

    raise RuntimeError(
        "At least one modern candidate covariate contains "
        "missing values."
    )


# ============================================================
# 5. BUILD HISTORICAL CANDIDATE FAMILIARITY
# ============================================================
#
# contexts_32f contains one driver/GP organisational-memory
# state based on prior appearances.
#
# Historical PL uses tenure_c, so for each candidate:
#
#     raw familiarity
#          ↓
#     merge by (raceId, driver)
#          ↓
#     subtract within-race mean
#
# This leaves the historical likelihood structure unchanged.
# ============================================================

historical_memory_cols_32g = {
    candidate:
        f"fam_{candidate}"
    for candidate
    in STAGE32G_CANDIDATES
}


missing_hist_memory_cols_32g = [
    col
    for col in historical_memory_cols_32g.values()
    if col
    not in contexts_32f.columns
]


if missing_hist_memory_cols_32g:

    raise RuntimeError(
        "contexts_32f is missing historical memory columns:\n  "
        +
        "\n  ".join(
            missing_hist_memory_cols_32g
        )
    )


hist_memory_lookup_32g = (
    contexts_32f[
        [
            "raceId",
            "driver",
        ]
        +
        list(
            historical_memory_cols_32g.values()
        )
    ]
    .drop_duplicates(
        [
            "raceId",
            "driver",
        ]
    )
    .copy()
)


dup_hist_lookup_32g = (
    hist_memory_lookup_32g
    .duplicated(
        [
            "raceId",
            "driver",
        ],
        keep=False,
    )
)


if dup_hist_lookup_32g.any():

    raise RuntimeError(
        "Historical familiarity lookup is not unique by "
        "(raceId, driver)."
    )


def build_candidate_historical_table_32g(
    base_df,
    candidate,
):

    raw_col = (
        historical_memory_cols_32g[
            candidate
        ]
    )

    out = (
        base_df
        .drop(
            columns=[
                raw_col,
            ],
            errors="ignore",
        )
        .merge(
            hist_memory_lookup_32g[
                [
                    "raceId",
                    "driver",
                    raw_col,
                ]
            ],
            on=[
                "raceId",
                "driver",
            ],
            how="left",
            validate="many_to_one",
        )
    )


    missing = int(
        out[
            raw_col
        ].isna().sum()
    )


    if missing:

        bad = (
            out.loc[
                out[
                    raw_col
                ].isna(),
                [
                    c
                    for c in [
                        "raceId",
                        "year",
                        "driver",
                        "constructorId",
                    ]
                    if c
                    in out.columns
                ]
            ]
            .head(30)
        )

        print(
            "\nMissing historical candidate rows:"
        )

        display(
            bad
        )

        raise RuntimeError(
            f"{candidate}: {missing} historical rows "
            "could not be matched to contexts_32f."
        )


    race_mean = (
        out
        .groupby(
            "raceId"
        )[
            raw_col
        ]
        .transform(
            "mean"
        )
    )


    out[
        "tenure_c"
    ] = (
        pd.to_numeric(
            out[
                raw_col
            ],
            errors="raise",
        )
        -
        pd.to_numeric(
            race_mean,
            errors="raise",
        )
    )


    return out


stage32g_hist_tables = {}
stage32g_hist_packs = {}

historical_check_rows_32g = []


for candidate in STAGE32G_CANDIDATES:

    hq = (
        build_candidate_historical_table_32g(
            hist_quali.copy(),
            candidate,
        )
    )

    hr = (
        build_candidate_historical_table_32g(
            hist_race.copy(),
            candidate,
        )
    )


    stage32g_hist_tables[
        candidate
    ] = {
        "quali":
            hq,

        "race":
            hr,
    }


    stage32g_hist_packs[
        candidate
    ] = {
        "quali":
            pack_rankings(
                hq
            ),

        "race":
            pack_rankings(
                hr
            ),
    }


    for label, table in [
        (
            "hist_quali",
            hq,
        ),
        (
            "hist_race",
            hr,
        ),
    ]:

        historical_check_rows_32g.append(
            {
                "candidate":
                    candidate,

                "table":
                    label,

                "rows":
                    len(table),

                "missing_tenure_c":
                    int(
                        table[
                            "tenure_c"
                        ].isna().sum()
                    ),

                "mean_tenure_c":
                    float(
                        table[
                            "tenure_c"
                        ].mean()
                    ),

                "max_abs_race_mean":
                    float(
                        table
                        .groupby(
                            "raceId"
                        )[
                            "tenure_c"
                        ]
                        .mean()
                        .abs()
                        .max()
                    ),
            }
        )


historical_checks_32g = pd.DataFrame(
    historical_check_rows_32g
)


print()
print("=" * 76)
print("HISTORICAL CANDIDATE TENURE CHECK")
print("=" * 76)

display(
    historical_checks_32g
)


# ============================================================
# 6. BUILD RACE-LEVEL THREE-FOLD CV
# ============================================================
#
# Each race weekend is assigned to one fold.
#
# Allocation is stratified by season:
#
#     shuffle races within year
#     assign 0,1,2,0,1,2,...
#
# Therefore no GP is split between pace/quali likelihoods.
# ============================================================

race_universe_parts_32g = []


for channel, df in modern_tables_32g.items():

    race_universe_parts_32g.append(
        df[
            [
                "raceId",
                "year",
            ]
        ]
        .copy()
    )


race_universe_32g = (
    pd.concat(
        race_universe_parts_32g,
        ignore_index=True,
    )
    .drop_duplicates(
        [
            "raceId",
            "year",
        ]
    )
)


race_universe_32g[
    "year"
] = pd.to_numeric(
    race_universe_32g[
        "year"
    ],
    errors="raise",
).astype(int)


race_universe_32g[
    "raceId"
] = pd.to_numeric(
    race_universe_32g[
        "raceId"
    ],
    errors="raise",
).astype(int)


race_universe_32g = (
    race_universe_32g.loc[
        (
            race_universe_32g[
                "year"
            ]
            >=
            STAGE32G_MODERN_FIRST_YEAR
        )
        &
        (
            race_universe_32g[
                "year"
            ]
            <=
            STAGE32G_SELECTION_LAST_YEAR
        )
    ]
    .copy()
)


fold_rows_32g = []


for year, grp in (
    race_universe_32g
    .groupby(
        "year",
        sort=True,
    )
):

    race_ids = (
        grp[
            "raceId"
        ]
        .drop_duplicates()
        .to_numpy(
            dtype=int
        )
    )


    rng = np.random.default_rng(
        STAGE32G_SEED
        +
        int(year)
    )


    shuffled = (
        race_ids[
            rng.permutation(
                len(
                    race_ids
                )
            )
        ]
    )


    for j, race_id in enumerate(
        shuffled
    ):

        fold_rows_32g.append(
            {
                "raceId":
                    int(
                        race_id
                    ),

                "year":
                    int(
                        year
                    ),

                "fold":
                    int(
                        j
                        %
                        STAGE32G_N_FOLDS
                    ),
            }
        )


stage32g_fold_map = pd.DataFrame(
    fold_rows_32g
)


if (
    stage32g_fold_map[
        "raceId"
    ].duplicated().any()
):

    raise RuntimeError(
        "A raceId has been assigned to multiple CV folds."
    )


# ============================================================
# 7. CHECK FOLD BALANCE BY SEASON / RACE COUNT
# ============================================================

fold_race_counts_32g = (
    stage32g_fold_map
    .groupby(
        "fold"
    )
    .agg(
        races=(
            "raceId",
            "nunique",
        ),
        seasons=(
            "year",
            "nunique",
        ),
        first_year=(
            "year",
            "min",
        ),
        last_year=(
            "year",
            "max",
        ),
    )
    .reset_index()
)


print()
print("=" * 76)
print("RACE-LEVEL CV FOLDS")
print("=" * 76)

display(
    fold_race_counts_32g
)


# Every sufficiently large modern season should touch all folds.

season_fold_counts_32g = (
    stage32g_fold_map
    .groupby(
        "year"
    )[
        "fold"
    ]
    .nunique()
)


bad_season_folds_32g = (
    season_fold_counts_32g[
        season_fold_counts_32g
        <
        STAGE32G_N_FOLDS
    ]
)


if len(
    bad_season_folds_32g
):

    print(
        "\nWARNING — seasons not represented "
        "in all folds:"
    )

    display(
        bad_season_folds_32g
    )


# ============================================================
# 8. ATTACH FOLD IDs AND MEASURE CANDIDATE DISCRIMINATION
# ============================================================

stage32g_base_tables = {}
stage32g_discrimination_rows = []


for channel, df0 in modern_tables_32g.items():

    df = (
        df0
        .copy()
        .merge(
            stage32g_fold_map[
                [
                    "raceId",
                    "fold",
                ]
            ],
            on="raceId",
            how="left",
            validate="many_to_one",
        )
    )


    if df[
        "fold"
    ].isna().any():

        bad = (
            df.loc[
                df[
                    "fold"
                ].isna(),
                [
                    "raceId",
                    "year",
                ]
            ]
            .drop_duplicates()
        )

        display(
            bad
        )

        raise RuntimeError(
            f"{channel}: observations exist outside "
            "the Stage-32G race-fold universe."
        )


    df[
        "fold"
    ] = df[
        "fold"
    ].astype(int)


    stage32g_base_tables[
        channel
    ] = df


    candidate_matrix = np.column_stack(
        [
            pd.to_numeric(
                df[
                    modern_candidate_cols_32g[
                        channel
                    ][candidate]
                ],
                errors="raise",
            )
            .to_numpy(
                dtype=float
            )
            for candidate
            in STAGE32G_CANDIDATES
        ]
    )


    spread = (
        np.max(
            candidate_matrix,
            axis=1,
        )
        -
        np.min(
            candidate_matrix,
            axis=1,
        )
    )


    df[
        "_candidate_spread"
    ] = spread


    for fold in range(
        STAGE32G_N_FOLDS
    ):

        held = (
            df[
                "fold"
            ]
            ==
            fold
        )

        row = {
            "channel":
                channel,

            "fold":
                fold,

            "heldout_rows":
                int(
                    held.sum()
                ),

            "mean_candidate_spread":
                float(
                    spread[
                        held
                    ].mean()
                ),

            "median_candidate_spread":
                float(
                    np.median(
                        spread[
                            held
                        ]
                    )
                ),

            "p95_candidate_spread":
                float(
                    np.quantile(
                        spread[
                            held
                        ],
                        0.95,
                    )
                ),

            "max_candidate_spread":
                float(
                    spread[
                        held
                    ].max()
                ),
        }


        for threshold in (
            STAGE32G_SENSITIVE_THRESHOLDS
        ):

            key = (
                "rows_spread_ge_"
                +
                str(
                    threshold
                )
                .replace(
                    ".",
                    "_",
                )
            )

            row[
                key
            ] = int(
                (
                    spread[
                        held
                    ]
                    >=
                    threshold
                )
                .sum()
            )


        stage32g_discrimination_rows.append(
            row
        )


stage32g_discrimination = pd.DataFrame(
    stage32g_discrimination_rows
)


print()
print("=" * 76)
print("HELD-OUT CANDIDATE DISCRIMINATION BY CHANNEL/FOLD")
print("=" * 76)

display(
    stage32g_discrimination
)


# ============================================================
# 9. CANDIDATE-SPECIFIC MODERN TABLE FACTORY
# ============================================================
#
# This is what Stage 32G-B will call.
#
# It changes ONLY the tenure variable in a COPY.
# ============================================================

def make_stage32g_candidate_modern_tables(
    candidate,
):

    if candidate not in (
        STAGE32G_CANDIDATES
    ):

        raise ValueError(
            f"Unknown Stage-32G candidate: {candidate}"
        )


    out = {}


    for channel, df0 in (
        stage32g_base_tables.items()
    ):

        df = (
            df0
            .copy()
        )


        source_col = (
            modern_candidate_cols_32g[
                channel
            ][candidate]
        )


        if channel in {
            "pace_contrast",
            "q_contrast",
        }:

            target_col = (
                "tenure_diff"
            )

        else:

            target_col = (
                "tenure_mean"
            )


        df[
            target_col
        ] = pd.to_numeric(
            df[
                source_col
            ],
            errors="raise",
        ).astype(float)


        out[
            channel
        ] = df


    return out


# Quick factory test.

for candidate in STAGE32G_CANDIDATES:

    test_tables = (
        make_stage32g_candidate_modern_tables(
            candidate
        )
    )

    for channel, df in (
        test_tables.items()
    ):

        target_col = (
            "tenure_diff"
            if channel
            in {
                "pace_contrast",
                "q_contrast",
            }
            else
            "tenure_mean"
        )

        if not np.isfinite(
            pd.to_numeric(
                df[
                    target_col
                ],
                errors="coerce",
            )
        ).all():

            raise RuntimeError(
                f"{candidate}/{channel}: "
                f"{target_col} is not fully finite."
            )


# ============================================================
# 10. SAVE THE FIXED CV SPLIT
# ============================================================

stage32g_fold_map.to_csv(
    STAGE32G_OUTDIR
    /
    "race_folds_1996_2025.csv",
    index=False,
)


candidate_column_table_32g.to_csv(
    STAGE32G_OUTDIR
    /
    "candidate_covariate_columns.csv",
    index=False,
)


stage32g_discrimination.to_csv(
    STAGE32G_OUTDIR
    /
    "heldout_candidate_discrimination.csv",
    index=False,
)


historical_checks_32g.to_csv(
    STAGE32G_OUTDIR
    /
    "historical_candidate_tenure_checks.csv",
    index=False,
)


# ============================================================
# 11. FINAL GATES
# ============================================================

hard_flags_32g = []


if (
    modern_candidate_checks_32g[
        "missing"
    ].sum()
    != 0
):

    hard_flags_32g.append(
        "Modern candidate familiarity contains missing values."
    )


if (
    historical_checks_32g[
        "missing_tenure_c"
    ].sum()
    != 0
):

    hard_flags_32g.append(
        "Historical candidate tenure_c contains missing values."
    )


if (
    historical_checks_32g[
        "max_abs_race_mean"
    ].max()
    >
    1e-10
):

    hard_flags_32g.append(
        "Historical candidate tenure_c is not centred "
        "within race."
    )


if (
    stage32g_fold_map[
        "fold"
    ]
    .nunique()
    !=
    STAGE32G_N_FOLDS
):

    hard_flags_32g.append(
        "Not all three CV folds were created."
    )


if (
    stage32g_discrimination[
        "heldout_rows"
    ].min()
    <= 0
):

    hard_flags_32g.append(
        "At least one channel/fold has zero held-out rows."
    )


# Require some genuinely discriminating held-out observations
# in every fold, pooled across channels.

sensitive_005_by_fold_32g = (
    stage32g_discrimination
    .groupby(
        "fold"
    )[
        "rows_spread_ge_0_05"
    ]
    .sum()
)


if (
    sensitive_005_by_fold_32g
    <= 0
).any():

    hard_flags_32g.append(
        "At least one CV fold has no rows where the "
        "candidate familiarity definitions differ by >= 0.05."
    )


print()
print("=" * 76)
print("STAGE 32G-A TRIAGE SUMMARY")
print("=" * 76)

print(
    "Candidate definitions:",
    ", ".join(
        STAGE32G_CANDIDATES
    ),
)

print(
    "Selection data:",
    f"{STAGE32G_MODERN_FIRST_YEAR}–"
    f"{STAGE32G_SELECTION_LAST_YEAR}",
)

print(
    "CV folds:",
    STAGE32G_N_FOLDS,
)

print(
    "Modern candidate missing values:",
    int(
        modern_candidate_checks_32g[
            "missing"
        ].sum()
    ),
)

print(
    "Historical candidate missing values:",
    int(
        historical_checks_32g[
            "missing_tenure_c"
        ].sum()
    ),
)

print(
    "Max historical within-race tenure mean:",
    f"{historical_checks_32g['max_abs_race_mean'].max():.3e}",
)

print(
    "\nRows with candidate spread >= 0.05 by held-out fold:"
)

for fold, n in (
    sensitive_005_by_fold_32g.items()
):

    print(
        f"  fold {int(fold)}: {int(n):,}"
    )


print()

if hard_flags_32g:

    print(
        "HARD FLAGS:"
    )

    for flag in hard_flags_32g:

        print(
            "  -",
            flag,
        )

else:

    print(
        "HARD FLAGS: none"
    )


print()
print(
    "Saved setup outputs ->"
)

print(
    STAGE32G_OUTDIR
)


if hard_flags_32g:

    raise RuntimeError(
        "Stage 32G-A failed its setup gate. "
        "Resolve the hard flags before fitting."
    )


print()
print(
    "Stage 32G-A complete."
)

print(
    "If this gate passes, Stage 32G-B can run the "
    "checkpointed 3-candidate × 3-fold predictive comparison."
)

In [ ]:
# ============================================================
# STAGE 32G-A0 — MATERIALISE CANDIDATE PAIR COVARIATES
# ============================================================
#
# RUN ONCE:
#   after successful Stage 32F.2
#   before Stage 32G-A
#
# WHY
# ---
# Stage 32F.2 stores the observation-safe familiarity candidates
# at DRIVER-OBSERVATION level:
#
#     memory_10gp_tenure_c
#     memory_20gp_tenure_c
#     memory_40gp_tenure_c
#
# but Stage 32G-A operates on the already-built pair likelihood
# tables.
#
# Reconstruct them exactly as the production pair builder does:
#
#   contrast = tenure_c(a) - tenure_c(b)
#   level    = 0.5 * (tenure_c(a) + tenure_c(b))
#
# This ADDS candidate columns only.
#
# Existing frozen:
#     tenure_diff
#     tenure_mean
#
# are NOT changed.
# ============================================================

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. CONFIGURATION
# ============================================================

CANDIDATES_32GA0 = [
    "memory_10gp",
    "memory_20gp",
    "memory_40gp",
]

TENURE_C_COLS_32GA0 = {
    candidate:
        f"{candidate}_tenure_c"
    for candidate
    in CANDIDATES_32GA0
}


# ============================================================
# 1. REQUIRED PAIR TABLES
# ============================================================

required_32ga0 = [
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
]

missing_32ga0 = [
    name
    for name in required_32ga0
    if name not in globals()
]

if missing_32ga0:

    raise RuntimeError(
        "Missing required likelihood tables:\n  "
        +
        "\n  ".join(
            missing_32ga0
        )
    )


# ============================================================
# 2. AUTOMATICALLY FIND THE FULL STAGE-32F.2 DRIVER TABLES
# ============================================================
#
# Rather than depending on whatever variable names Stage 32F.2
# used, identify them from their actual structure.
#
# Pace:
#     raceId + driver + candidate tenure_c
#
# Qualifying:
#     raceId + session + driver + candidate tenure_c
#
# We score candidates by coverage of the pair-table observations.
# ============================================================

candidate_value_cols_32ga0 = list(
    TENURE_C_COLS_32GA0.values()
)


def required_driver_keys_from_pairs_32ga0(
    pair_tables,
    *,
    session=False,
):

    pieces = []

    for df in pair_tables:

        if session:

            base_cols = [
                "raceId",
                "session",
            ]

        else:

            base_cols = [
                "raceId",
            ]


        a = df[
            base_cols
            +
            [
                "driver_a",
            ]
        ].copy()

        b = df[
            base_cols
            +
            [
                "driver_b",
            ]
        ].copy()


        a = a.rename(
            columns={
                "driver_a":
                    "driver",
            }
        )

        b = b.rename(
            columns={
                "driver_b":
                    "driver",
            }
        )


        pieces.extend(
            [
                a,
                b,
            ]
        )


    keys = (
        pd.concat(
            pieces,
            ignore_index=True,
        )
        .drop_duplicates()
    )

    return keys


pace_required_keys_32ga0 = (
    required_driver_keys_from_pairs_32ga0(
        [
            pace_contrast_all,
            pace_level_all,
        ],
        session=False,
    )
)


q_required_keys_32ga0 = (
    required_driver_keys_from_pairs_32ga0(
        [
            q_contrast_all,
            q_level_all_model,
        ],
        session=True,
    )
)


def find_driver_source_32ga0(
    required_keys,
    *,
    session,
):

    key_cols = (
        [
            "raceId",
            "session",
            "driver",
        ]
        if session
        else
        [
            "raceId",
            "driver",
        ]
    )


    required_columns = set(
        key_cols
        +
        candidate_value_cols_32ga0
    )


    candidates = []


    for name, obj in list(
        globals().items()
    ):

        if not isinstance(
            obj,
            pd.DataFrame,
        ):

            continue


        if not required_columns.issubset(
            obj.columns
        ):

            continue


        # Pace source should be race/driver level,
        # not qualifying session level.
        if (
            not session
            and
            "session"
            in obj.columns
        ):

            continue


        source = (
            obj[
                key_cols
                +
                candidate_value_cols_32ga0
            ]
            .copy()
        )


        duplicate_count = int(
            source.duplicated(
                key_cols
            ).sum()
        )


        unique_source = (
            source
            .drop_duplicates(
                key_cols
            )
        )


        coverage = (
            required_keys
            .merge(
                unique_source[
                    key_cols
                ],
                on=key_cols,
                how="left",
                indicator=True,
            )[
                "_merge"
            ]
            .eq(
                "both"
            )
            .mean()
        )


        missing_values = int(
            unique_source[
                candidate_value_cols_32ga0
            ]
            .isna()
            .sum()
            .sum()
        )


        candidates.append(
            {
                "name":
                    name,

                "rows":
                    len(obj),

                "unique_key_rows":
                    len(unique_source),

                "duplicate_keys":
                    duplicate_count,

                "coverage":
                    float(
                        coverage
                    ),

                "missing_candidate_values":
                    missing_values,

                "data":
                    obj,
            }
        )


    if not candidates:

        raise RuntimeError(
            "Could not find a Stage-32F.2 driver-level "
            "candidate familiarity table."
        )


    summary = (
        pd.DataFrame(
            [
                {
                    k:
                        v
                    for k, v in row.items()
                    if k != "data"
                }
                for row in candidates
            ]
        )
        .sort_values(
            [
                "coverage",
                "missing_candidate_values",
                "rows",
            ],
            ascending=[
                False,
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    best_name = summary.iloc[
        0
    ][
        "name"
    ]


    best = next(
        row[
            "data"
        ]
        for row in candidates
        if row[
            "name"
        ]
        ==
        best_name
    )


    best_coverage = float(
        summary.iloc[
            0
        ][
            "coverage"
        ]
    )


    print()
    print(
        "Candidate driver-level sources:"
    )

    display(
        summary
    )


    if best_coverage < 1.0:

        raise RuntimeError(
            f"Best source {best_name!r} covers only "
            f"{best_coverage:.3%} of required observations."
        )


    print(
        f"Selected source: {best_name!r}"
    )


    return (
        best.copy(),
        best_name,
    )


print(
    "=" * 76
)
print(
    "PACE DRIVER-LEVEL SOURCE DISCOVERY"
)
print(
    "=" * 76
)

(
    pace_driver_candidates_32ga0,
    pace_driver_source_name_32ga0,
) = find_driver_source_32ga0(
    pace_required_keys_32ga0,
    session=False,
)


print()
print(
    "=" * 76
)
print(
    "QUALIFYING DRIVER/SESSION SOURCE DISCOVERY"
)
print(
    "=" * 76
)

(
    q_driver_candidates_32ga0,
    q_driver_source_name_32ga0,
) = find_driver_source_32ga0(
    q_required_keys_32ga0,
    session=True,
)


# ============================================================
# 3. BUILD UNIQUE LOOKUPS
# ============================================================

pace_lookup_32ga0 = (
    pace_driver_candidates_32ga0[
        [
            "raceId",
            "driver",
        ]
        +
        candidate_value_cols_32ga0
    ]
    .drop_duplicates(
        [
            "raceId",
            "driver",
        ]
    )
    .copy()
)


q_lookup_32ga0 = (
    q_driver_candidates_32ga0[
        [
            "raceId",
            "session",
            "driver",
        ]
        +
        candidate_value_cols_32ga0
    ]
    .drop_duplicates(
        [
            "raceId",
            "session",
            "driver",
        ]
    )
    .copy()
)


if pace_lookup_32ga0.duplicated(
    [
        "raceId",
        "driver",
    ]
).any():

    raise RuntimeError(
        "Pace candidate lookup is not unique."
    )


if q_lookup_32ga0.duplicated(
    [
        "raceId",
        "session",
        "driver",
    ]
).any():

    raise RuntimeError(
        "Qualifying candidate lookup is not unique."
    )


# ============================================================
# 4. HELPER — ADD CANDIDATE COVARIATES TO A PAIR TABLE
# ============================================================

def add_candidate_pair_covariates_32ga0(
    pair_df,
    driver_lookup,
    *,
    session,
):

    out = pair_df.copy()


    if session:

        lookup_keys = [
            "raceId",
            "session",
            "driver",
        ]

        pair_base_keys = [
            "raceId",
            "session",
        ]

    else:

        lookup_keys = [
            "raceId",
            "driver",
        ]

        pair_base_keys = [
            "raceId",
        ]


    # --------------------------------------------------------
    # Driver A lookup
    # --------------------------------------------------------

    lookup_a = (
        driver_lookup[
            lookup_keys
            +
            candidate_value_cols_32ga0
        ]
        .rename(
            columns={
                "driver":
                    "driver_a",

                **{
                    col:
                        f"{col}_a"
                    for col
                    in candidate_value_cols_32ga0
                },
            }
        )
    )


    out[
        "_row_order_32ga0"
    ] = np.arange(
        len(out),
        dtype=int,
    )


    out = out.merge(
        lookup_a,
        on=
            pair_base_keys
            +
            [
                "driver_a",
            ],
        how="left",
        validate="many_to_one",
        sort=False,
    )


    # --------------------------------------------------------
    # Driver B lookup
    # --------------------------------------------------------

    lookup_b = (
        driver_lookup[
            lookup_keys
            +
            candidate_value_cols_32ga0
        ]
        .rename(
            columns={
                "driver":
                    "driver_b",

                **{
                    col:
                        f"{col}_b"
                    for col
                    in candidate_value_cols_32ga0
                },
            }
        )
    )


    out = out.merge(
        lookup_b,
        on=
            pair_base_keys
            +
            [
                "driver_b",
            ],
        how="left",
        validate="many_to_one",
        sort=False,
    )


    out = (
        out
        .sort_values(
            "_row_order_32ga0"
        )
        .drop(
            columns=[
                "_row_order_32ga0",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Reproduce production Stage-6 pair construction exactly:
    #
    # contrast = a - b
    # level    = 0.5 * (a + b)
    # --------------------------------------------------------

    for candidate in CANDIDATES_32GA0:

        source_col = (
            TENURE_C_COLS_32GA0[
                candidate
            ]
        )


        a_col = (
            f"{source_col}_a"
        )

        b_col = (
            f"{source_col}_b"
        )


        out[
            f"{candidate}_tenure_diff"
        ] = (
            pd.to_numeric(
                out[
                    a_col
                ],
                errors="coerce",
            )
            -
            pd.to_numeric(
                out[
                    b_col
                ],
                errors="coerce",
            )
        )


        out[
            f"{candidate}_tenure_mean"
        ] = (
            0.5
            *
            (
                pd.to_numeric(
                    out[
                        a_col
                    ],
                    errors="coerce",
                )
                +
                pd.to_numeric(
                    out[
                        b_col
                    ],
                    errors="coerce",
                )
            )
        )


    # Remove temporary a/b driver values.
    drop_cols = []

    for candidate in CANDIDATES_32GA0:

        source_col = (
            TENURE_C_COLS_32GA0[
                candidate
            ]
        )

        drop_cols.extend(
            [
                f"{source_col}_a",
                f"{source_col}_b",
            ]
        )


    out = out.drop(
        columns=drop_cols,
        errors="ignore",
    )


    return out


# ============================================================
# 5. MATERIALISE CANDIDATE COLUMNS
# ============================================================
#
# We deliberately keep the original tenure_diff / tenure_mean
# untouched.
# ============================================================

pace_contrast_all = (
    add_candidate_pair_covariates_32ga0(
        pace_contrast_all,
        pace_lookup_32ga0,
        session=False,
    )
)


pace_level_all = (
    add_candidate_pair_covariates_32ga0(
        pace_level_all,
        pace_lookup_32ga0,
        session=False,
    )
)


q_contrast_all = (
    add_candidate_pair_covariates_32ga0(
        q_contrast_all,
        q_lookup_32ga0,
        session=True,
    )
)


q_level_all_model = (
    add_candidate_pair_covariates_32ga0(
        q_level_all_model,
        q_lookup_32ga0,
        session=True,
    )
)


# ============================================================
# 6. FINAL AUDIT
# ============================================================

audit_rows_32ga0 = []


tables_32ga0 = {
    "pace_contrast":
        (
            pace_contrast_all,
            "tenure_diff",
        ),

    "pace_level":
        (
            pace_level_all,
            "tenure_mean",
        ),

    "q_contrast":
        (
            q_contrast_all,
            "tenure_diff",
        ),

    "q1_level":
        (
            q_level_all_model,
            "tenure_mean",
        ),
}


for channel, (
    df,
    suffix,
) in tables_32ga0.items():

    for candidate in CANDIDATES_32GA0:

        col = (
            f"{candidate}_{suffix}"
        )

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce",
        )


        audit_rows_32ga0.append(
            {
                "channel":
                    channel,

                "candidate":
                    candidate,

                "column":
                    col,

                "rows":
                    len(df),

                "missing":
                    int(
                        values.isna().sum()
                    ),

                "finite":
                    int(
                        np.isfinite(
                            values
                        ).sum()
                    ),

                "min":
                    float(
                        values.min()
                    ),

                "max":
                    float(
                        values.max()
                    ),
            }
        )


audit_32ga0 = pd.DataFrame(
    audit_rows_32ga0
)


print()
print(
    "=" * 76
)
print(
    "STAGE 32G-A0 — MATERIALISED PAIR COVARIATES"
)
print(
    "=" * 76
)

display(
    audit_32ga0
)


total_missing_32ga0 = int(
    audit_32ga0[
        "missing"
    ].sum()
)


print()
print(
    "Pace source:",
    pace_driver_source_name_32ga0,
)

print(
    "Qualifying source:",
    q_driver_source_name_32ga0,
)

print(
    "Candidate pair-covariate missing values:",
    total_missing_32ga0,
)


if total_missing_32ga0 != 0:

    raise RuntimeError(
        "Candidate pair covariates still contain missing values."
    )


# Make sure frozen production covariates themselves were not lost.

for channel, (
    df,
    original_col,
) in tables_32ga0.items():

    if original_col not in df.columns:

        raise RuntimeError(
            f"{channel}: original {original_col!r} "
            "was accidentally lost."
        )


print()
print(
    "Original production tenure_diff / tenure_mean "
    "columns remain untouched."
)

print()
print(
    "Stage 32G-A0 complete."
)

print(
    "NOW RERUN THE EXISTING STAGE 32G-A CELL FROM THE TOP."
)

In [ ]:
# ============================================================
# STAGE 32G-B — CHECKPOINTED PREDICTIVE CV FOR
#               ORGANISATIONAL-FAMILIARITY MEMORY
# ============================================================
#
# PURPOSE
# -------
# Choose among:
#
#     memory_10gp
#     memory_20gp
#     memory_40gp
#
# using genuinely held-out MODERN race weekends.
#
# For each candidate and each of the three Stage-32G-A folds:
#
#   1. remove ALL four modern likelihood channels for the
#      held-out race weekends;
#   2. replace the production tenure covariate with that
#      candidate's familiarity covariate;
#   3. use the SAME candidate definition in the historical
#      Plackett-Luce likelihood;
#   4. refit the complete latent driver/car model;
#   5. score the held-out modern observations under the
#      posterior predictive density.
#
# We then compare candidates using:
#
#   - total held-out ELPD;
#   - ELPD by likelihood channel;
#   - ELPD by fold;
#   - paired race-clustered ELPD differences;
#   - posterior/sampler health.
#
# IMPORTANT
# ---------
# Selection is based on 1996–2025 only.
#
# 2026 is deliberately excluded from candidate selection because
# it is an incomplete live season.
#
# Nothing here overwrites the frozen production trace or the
# eventual 1982–2026 LIVE trace.
# ============================================================

from pathlib import Path
import gc
import time

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import scipy.stats as st
from scipy.special import logsumexp
import xarray as xr

from IPython.display import display


# ============================================================
# 0. CONFIGURATION
# ============================================================

STAGE32G_CANDIDATES = [
    "memory_10gp",
    "memory_20gp",
    "memory_40gp",
]

STAGE32G_FOLDS = 3

STAGE32G_DRAWS = 300
STAGE32G_TUNE = 300
STAGE32G_CHAINS = 2
STAGE32G_TARGET_ACCEPT = 0.92

STAGE32G_BASE_SEED = 320700

STAGE32G_CHECKPOINT_ENGINE = "h5netcdf"

STAGE32G_DIR = (
    Path(CFG.output_dir)
    / "validation_familiarity_predictive_cv"
)

STAGE32G_FIT_DIR = (
    STAGE32G_DIR
    / "fits"
)

STAGE32G_FIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_objects_32gb = [
    "build_joint_model_1982_2025",
    "pack_rankings",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "hist_quali",
    "hist_race",
    "idx_all",
    "CFG",
]

missing_objects_32gb = [
    name
    for name in required_objects_32gb
    if name not in globals()
]

if missing_objects_32gb:

    raise RuntimeError(
        "Stage 32G-B is missing required objects:\n  "
        +
        "\n  ".join(
            missing_objects_32gb
        )
    )


# ============================================================
# 2. WORK ONLY WITH COMPLETED 1996–2025 MODERN DATA
# ============================================================

def modern_selection_copy_32gb(df):

    out = df[
        (df["year"] >= 1996)
        &
        (df["year"] <= 2025)
    ].copy()

    return out.reset_index(
        drop=False
    ).rename(
        columns={
            "index":
                "_source_row",
        }
    )


pc_32gb = modern_selection_copy_32gb(
    pace_contrast_all
)

pl_32gb = modern_selection_copy_32gb(
    pace_level_all
)

qc_32gb = modern_selection_copy_32gb(
    q_contrast_all
)

ql_32gb = modern_selection_copy_32gb(
    q_level_all_model
)


modern_tables_32gb = {
    "pace_contrast":
        pc_32gb,

    "pace_level":
        pl_32gb,

    "q_contrast":
        qc_32gb,

    "q1_level":
        ql_32gb,
}


print(
    "=" * 76
)

print(
    "STAGE 32G-B — MODERN SELECTION TABLES"
)

print(
    "=" * 76
)


for channel, df in modern_tables_32gb.items():

    print(
        f"{channel:15s}: "
        f"{len(df):,} rows; "
        f"{df['year'].min()}–{df['year'].max()}"
    )


# ============================================================
# 3. VERIFY CANDIDATE COLUMNS
# ============================================================

candidate_column_map_32gb = {
    "pace_contrast":
        {
            candidate:
                f"{candidate}_tenure_diff"
            for candidate
            in STAGE32G_CANDIDATES
        },

    "pace_level":
        {
            candidate:
                f"{candidate}_tenure_mean"
            for candidate
            in STAGE32G_CANDIDATES
        },

    "q_contrast":
        {
            candidate:
                f"{candidate}_tenure_diff"
            for candidate
            in STAGE32G_CANDIDATES
        },

    "q1_level":
        {
            candidate:
                f"{candidate}_tenure_mean"
            for candidate
            in STAGE32G_CANDIDATES
        },
}


candidate_audit_rows_32gb = []


for channel, df in modern_tables_32gb.items():

    for candidate in STAGE32G_CANDIDATES:

        col = (
            candidate_column_map_32gb[
                channel
            ][
                candidate
            ]
        )

        if col not in df.columns:

            raise RuntimeError(
                f"{channel}: missing candidate column "
                f"{col!r}."
            )

        values = pd.to_numeric(
            df[col],
            errors="coerce",
        )

        candidate_audit_rows_32gb.append(
            {
                "channel":
                    channel,

                "candidate":
                    candidate,

                "column":
                    col,

                "rows":
                    len(df),

                "missing":
                    int(
                        values.isna().sum()
                    ),

                "finite":
                    int(
                        np.isfinite(
                            values
                        ).sum()
                    ),
            }
        )


candidate_audit_32gb = pd.DataFrame(
    candidate_audit_rows_32gb
)


print()
print(
    "Candidate modern covariate audit:"
)

display(
    candidate_audit_32gb
)


if candidate_audit_32gb[
    "missing"
].sum() != 0:

    raise RuntimeError(
        "Missing candidate modern covariates."
    )


# ============================================================
# 4. RECOVER EXACT STAGE-32G-A RACE/FOLD ASSIGNMENT
# ============================================================
#
# We deliberately discover it from the objects created by 32G-A
# rather than silently generate a new partition.
# ============================================================

required_modern_races_32gb = set(
    pd.concat(
        [
            df[
                [
                    "raceId",
                ]
            ]
            for df
            in modern_tables_32gb.values()
        ],
        ignore_index=True,
    )[
        "raceId"
    ]
    .astype(int)
    .unique()
)


def find_fold_table_32gb():

    candidates = []


    for name, obj in list(
        globals().items()
    ):

        if not isinstance(
            obj,
            pd.DataFrame,
        ):

            continue

        if not {
            "raceId",
            "fold",
        }.issubset(
            obj.columns
        ):

            continue


        temp = obj[
            [
                "raceId",
                "fold",
            ]
            +
            (
                ["year"]
                if "year" in obj.columns
                else []
            )
        ].copy()


        temp[
            "raceId"
        ] = pd.to_numeric(
            temp[
                "raceId"
            ],
            errors="coerce",
        )

        temp[
            "fold"
        ] = pd.to_numeric(
            temp[
                "fold"
            ],
            errors="coerce",
        )


        temp = temp.dropna(
            subset=[
                "raceId",
                "fold",
            ]
        )


        if len(temp) == 0:

            continue


        temp[
            "raceId"
        ] = temp[
            "raceId"
        ].astype(int)

        temp[
            "fold"
        ] = temp[
            "fold"
        ].astype(int)


        fold_values = set(
            temp[
                "fold"
            ].unique()
        )


        if fold_values != set(
            range(
                STAGE32G_FOLDS
            )
        ):

            continue


        # Need one fold assignment per race.
        race_fold_nunique = (
            temp
            .groupby(
                "raceId"
            )[
                "fold"
            ]
            .nunique()
        )


        if (
            race_fold_nunique
            >
            1
        ).any():

            continue


        unique = (
            temp[
                [
                    "raceId",
                    "fold",
                ]
            ]
            .drop_duplicates()
        )


        found_races = set(
            unique[
                "raceId"
            ]
        )


        overlap = len(
            required_modern_races_32gb
            &
            found_races
        )

        coverage = (
            overlap
            /
            len(
                required_modern_races_32gb
            )
        )


        fold_counts = (
            unique[
                unique[
                    "raceId"
                ].isin(
                    required_modern_races_32gb
                )
            ]
            .groupby(
                "fold"
            )[
                "raceId"
            ]
            .nunique()
            .to_dict()
        )


        candidates.append(
            {
                "name":
                    name,

                "coverage":
                    coverage,

                "unique_races":
                    len(
                        found_races
                    ),

                "fold_counts":
                    fold_counts,

                "data":
                    unique,
            }
        )


    if not candidates:

        raise RuntimeError(
            "Could not find the Stage-32G-A "
            "race/fold assignment DataFrame."
        )


    candidates = sorted(
        candidates,
        key=lambda x: (
            x[
                "coverage"
            ],
            -abs(
                x[
                    "unique_races"
                ]
                -
                len(
                    required_modern_races_32gb
                )
            ),
        ),
        reverse=True,
    )


    best = candidates[
        0
    ]


    if best[
        "coverage"
    ] < 1.0:

        raise RuntimeError(
            "Best discovered fold table covers only "
            f"{best['coverage']:.3%} of modern races."
        )


    print()
    print(
        "Stage-32G-A fold object:",
        best[
            "name"
        ],
    )

    print(
        "Fold race counts:",
        best[
            "fold_counts"
        ],
    )


    return (
        best[
            "data"
        ][
            best[
                "data"
            ][
                "raceId"
            ].isin(
                required_modern_races_32gb
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


fold_table_32gb = find_fold_table_32gb()


if (
    fold_table_32gb[
        "raceId"
    ].nunique()
    !=
    len(
        required_modern_races_32gb
    )
):

    raise RuntimeError(
        "Fold table does not contain exactly one "
        "assignment for every modern race."
    )


# ============================================================
# 5. BUILD HISTORICAL CANDIDATE TENURE DIRECTLY FROM
#    THE STAGE-32F MASTER DRIVER/RACE FAMILIARITY HISTORY
# ============================================================
#
# Do NOT try to rediscover a Stage-32G-A intermediate object.
#
# contexts_32f is the master one-row-per-driver/GP familiarity
# history created in Stage 32F.  It contains the candidate raw
# organisational familiarity before the current race weekend.
#
# Historical Plackett-Luce tenure is then constructed exactly
# analogously to the production historical tenure:
#
#     raw familiarity
#       -> merge onto the historical observation rows
#       -> centre within that historical race field
#
# This is deliberately done separately for hist_quali and
# hist_race because their race fields are different.
# ============================================================

if "contexts_32f" not in globals():
    raise RuntimeError(
        "Stage 32G-B requires contexts_32f from Stage 32F."
    )


def candidate_raw_column_32gb(
    df,
    candidate,
):
    """
    Find the raw Stage-32F organisational-familiarity column
    corresponding to e.g. memory_10gp.

    Stage 32F / 32F.2 have used both naming styles during the
    audit, so accept the known exact alternatives explicitly.
    """

    possible = [
        f"fam_{candidate}",
        f"{candidate}_raw",
        candidate,
        f"{candidate}_familiarity",
    ]

    found = [
        c
        for c in possible
        if c in df.columns
    ]

    if len(found) == 0:
        raise RuntimeError(
            f"Could not find raw historical familiarity column "
            f"for {candidate!r} in contexts_32f.\n"
            f"Looked for: {possible}\n"
            f"Available candidate-like columns:\n"
            + "\n".join(
                "  " + c
                for c in df.columns
                if (
                    "memory_" in c.lower()
                    or "fam_" in c.lower()
                )
            )
        )

    # Prefer the first name in the explicit priority list.
    return found[0]


# ------------------------------------------------------------
# One source row per driver/race
# ------------------------------------------------------------
#
# Stage 32F already collapsed the historical duplicate-result
# oddities.  From 1982 onward the duplicate audit found no
# duplicate driver/race contexts, but assert it again here.
# ------------------------------------------------------------

historical_context_source_32gb = (
    contexts_32f[
        pd.to_numeric(
            contexts_32f["year"],
            errors="coerce",
        ).between(
            CFG.historical_floor_year,
            CFG.historical_ceiling_year,
        )
    ]
    .copy()
)


key_cols_32gb = [
    "raceId",
    "driverId",
]


if historical_context_source_32gb.duplicated(
    key_cols_32gb
).any():

    dup = (
        historical_context_source_32gb.loc[
            historical_context_source_32gb.duplicated(
                key_cols_32gb,
                keep=False,
            )
        ]
        .sort_values(
            key_cols_32gb
        )
    )

    raise RuntimeError(
        "Historical Stage-32F familiarity source still contains "
        "duplicate driver/race contexts.\n\n"
        f"{dup.head(30)}"
    )


# ------------------------------------------------------------
# Audit the raw candidate source columns
# ------------------------------------------------------------

historical_raw_col_32gb = {}

raw_audit_rows_32gb = []


for candidate in STAGE32G_CANDIDATES:

    col = candidate_raw_column_32gb(
        historical_context_source_32gb,
        candidate,
    )

    historical_raw_col_32gb[
        candidate
    ] = col

    vals = pd.to_numeric(
        historical_context_source_32gb[col],
        errors="coerce",
    )

    raw_audit_rows_32gb.append(
        {
            "candidate": candidate,
            "source_column": col,
            "rows": len(vals),
            "missing": int(vals.isna().sum()),
            "finite": int(
                np.isfinite(vals).sum()
            ),
            "min": float(vals.min()),
            "max": float(vals.max()),
        }
    )


historical_raw_audit_32gb = pd.DataFrame(
    raw_audit_rows_32gb
)


print()
print("=" * 76)
print("HISTORICAL RAW FAMILIARITY SOURCE")
print("=" * 76)

display(
    historical_raw_audit_32gb
)


if (
    historical_raw_audit_32gb["missing"].sum()
    != 0
):

    raise RuntimeError(
        "Historical Stage-32F familiarity source contains "
        "missing candidate values."
    )


# ============================================================
# 6. CONSTRUCT CANDIDATE HISTORICAL TABLES + PACKS
# ============================================================

def build_candidate_historical_table_32gb(
    base,
    *,
    candidate,
    label,
):
    """
    Attach the candidate raw familiarity to exactly the rows in
    `base`, then centre it within each race.

    This matches the original historical tenure construction:

        tenure_c =
            tenure_raw
            - mean(tenure_raw within race)

    but substitutes the selected organisational-memory candidate
    for literal-constructor tenure.
    """

    raw_col = (
        historical_raw_col_32gb[
            candidate
        ]
    )

    source = (
        historical_context_source_32gb[
            [
                "raceId",
                "driverId",
                raw_col,
            ]
        ]
        .copy()
        .rename(
            columns={
                raw_col:
                    "_candidate_familiarity_raw",
            }
        )
    )


    if source.duplicated(
        [
            "raceId",
            "driverId",
        ]
    ).any():

        raise RuntimeError(
            f"{label}/{candidate}: familiarity source is not "
            "one row per driver/race."
        )


    out = (
        base
        .drop(
            columns=[
                "_candidate_familiarity_raw",
            ],
            errors="ignore",
        )
        .merge(
            source,
            on=[
                "raceId",
                "driverId",
            ],
            how="left",
            validate="many_to_one",
        )
    )


    missing = int(
        out[
            "_candidate_familiarity_raw"
        ].isna().sum()
    )


    if missing != 0:

        missing_rows = (
            out.loc[
                out[
                    "_candidate_familiarity_raw"
                ].isna(),
                [
                    c
                    for c in [
                        "raceId",
                        "year",
                        "round",
                        "driverId",
                        "driver",
                        "constructorId",
                    ]
                    if c in out.columns
                ],
            ]
            .head(30)
        )

        raise RuntimeError(
            f"{label}/{candidate}: "
            f"{missing:,}/{len(out):,} historical rows "
            "did not match contexts_32f.\n\n"
            f"{missing_rows}"
        )


    out[
        "tenure_raw"
    ] = pd.to_numeric(
        out[
            "_candidate_familiarity_raw"
        ],
        errors="raise",
    ).astype(float)


    out[
        "tenure_c"
    ] = (
        out[
            "tenure_raw"
        ]
        -
        out.groupby(
            "raceId"
        )[
            "tenure_raw"
        ].transform(
            "mean"
        )
    )


    # --------------------------------------------------------
    # Exact centring audit
    # --------------------------------------------------------

    race_means = (
        out
        .groupby(
            "raceId"
        )[
            "tenure_c"
        ]
        .mean()
    )


    max_abs_race_mean = float(
        race_means.abs().max()
    )


    if max_abs_race_mean > 1e-10:

        raise RuntimeError(
            f"{label}/{candidate}: historical familiarity "
            "did not centre to numerical zero. "
            f"max |race mean| = {max_abs_race_mean:.3e}"
        )


    if not np.isfinite(
        out[
            "tenure_c"
        ].to_numpy(
            dtype=float
        )
    ).all():

        raise RuntimeError(
            f"{label}/{candidate}: non-finite tenure_c."
        )


    return (
        out,
        max_abs_race_mean,
    )


historical_packs_32gb = {}

historical_source_rows_32gb = []


for candidate in STAGE32G_CANDIDATES:

    (
        hist_q_candidate,
        q_max_mean,
    ) = build_candidate_historical_table_32gb(
        hist_quali,
        candidate=candidate,
        label="hist_quali",
    )

    (
        hist_r_candidate,
        r_max_mean,
    ) = build_candidate_historical_table_32gb(
        hist_race,
        candidate=candidate,
        label="hist_race",
    )


    historical_packs_32gb[
        candidate
    ] = {
        "quali":
            pack_rankings(
                hist_q_candidate
            ),

        "race":
            pack_rankings(
                hist_r_candidate
            ),
    }


    historical_source_rows_32gb.extend(
        [
            {
                "candidate":
                    candidate,

                "table":
                    "hist_quali",

                "source":
                    (
                        "contexts_32f."
                        +
                        historical_raw_col_32gb[
                            candidate
                        ]
                    ),

                "rows":
                    len(
                        hist_q_candidate
                    ),

                "missing":
                    int(
                        hist_q_candidate[
                            "tenure_c"
                        ].isna().sum()
                    ),

                "max_abs_race_mean":
                    q_max_mean,
            },

            {
                "candidate":
                    candidate,

                "table":
                    "hist_race",

                "source":
                    (
                        "contexts_32f."
                        +
                        historical_raw_col_32gb[
                            candidate
                        ]
                    ),

                "rows":
                    len(
                        hist_r_candidate
                    ),

                "missing":
                    int(
                        hist_r_candidate[
                            "tenure_c"
                        ].isna().sum()
                    ),

                "max_abs_race_mean":
                    r_max_mean,
            },
        ]
    )


historical_sources_32gb = pd.DataFrame(
    historical_source_rows_32gb
)


print()
print("=" * 76)
print("HISTORICAL CANDIDATE PACK SOURCES")
print("=" * 76)

display(
    historical_sources_32gb
)


if (
    historical_sources_32gb[
        "missing"
    ].sum()
    != 0
):

    raise RuntimeError(
        "Historical candidate packs contain missing tenure."
    )


if (
    historical_sources_32gb[
        "max_abs_race_mean"
    ].max()
    >
    1e-10
):

    raise RuntimeError(
        "Historical candidate tenure is not centred "
        "within race to numerical precision."
    )


print()
print(
    "Historical candidate tenure successfully rebuilt "
    "directly from contexts_32f."
)


# ============================================================
# 7. HELPERS — CANDIDATE TRAIN/TEST TABLES
# ============================================================

def candidate_modern_split_32gb(
    df,
    *,
    candidate,
    channel,
    heldout_races,
):

    heldout_mask = (
        df[
            "raceId"
        ]
        .astype(int)
        .isin(
            heldout_races
        )
    )


    train = (
        df.loc[
            ~heldout_mask
        ]
        .copy()
    )

    test = (
        df.loc[
            heldout_mask
        ]
        .copy()
    )


    candidate_col = (
        candidate_column_map_32gb[
            channel
        ][
            candidate
        ]
    )


    if channel in {
        "pace_contrast",
        "q_contrast",
    }:

        production_col = (
            "tenure_diff"
        )

    else:

        production_col = (
            "tenure_mean"
        )


    train[
        production_col
    ] = pd.to_numeric(
        train[
            candidate_col
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )


    test[
        production_col
    ] = pd.to_numeric(
        test[
            candidate_col
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )


    return (
        train,
        test,
    )


# ============================================================
# 8. TRACE HELPERS
# ============================================================

def posterior_dataset_32gb(
    trace,
):

    if hasattr(
        trace,
        "children",
    ):

        return (
            trace[
                "posterior"
            ]
            .to_dataset()
        )


    if hasattr(
        trace,
        "posterior",
    ):

        return trace.posterior


    raise TypeError(
        "Unrecognised trace type."
    )


def sample_stats_dataset_32gb(
    trace,
):

    if hasattr(
        trace,
        "children",
    ):

        if (
            "sample_stats"
            in trace.children
        ):

            return (
                trace[
                    "sample_stats"
                ]
                .to_dataset()
            )

        return None


    if hasattr(
        trace,
        "sample_stats",
    ):

        return trace.sample_stats


    return None


def posterior_scalar_32gb(
    posterior,
    name,
):

    arr = np.asarray(
        posterior[
            name
        ]
    )

    return arr.reshape(
        -1
    ).astype(
        float
    )


def posterior_vector_32gb(
    posterior,
    name,
):

    arr = np.asarray(
        posterior[
            name
        ]
    )

    return arr.reshape(
        -1,
        arr.shape[
            -1
        ],
    ).astype(
        float
    )


# ============================================================
# 9. HELD-OUT STUDENT-T POSTERIOR-PREDICTIVE SCORER
# ============================================================

SIGMA_VAR_32GB = {
    "pace_contrast":
        "sigma_pace_contrast",

    "pace_level":
        "sigma_pace_level",

    "q_contrast":
        "sigma_q_contrast",

    "q1_level":
        "sigma_q_level",
}


def heldout_log_predictive_density_32gb(
    trace,
    test_df,
    *,
    channel,
    candidate,
    fold,
    chunk_size=512,
):

    posterior = (
        posterior_dataset_32gb(
            trace
        )
    )


    alpha = (
        posterior_vector_32gb(
            posterior,
            "alpha_dy",
        )
    )

    car = (
        posterior_vector_32gb(
            posterior,
            "car_csy",
        )
    )

    gamma = (
        posterior_scalar_32gb(
            posterior,
            "gamma_tenure",
        )
    )

    sigma = (
        posterior_scalar_32gb(
            posterior,
            SIGMA_VAR_32GB[
                channel
            ],
        )
    )


    n_samples = alpha.shape[
        0
    ]


    if not (
        car.shape[
            0
        ]
        ==
        gamma.shape[
            0
        ]
        ==
        sigma.shape[
            0
        ]
        ==
        n_samples
    ):

        raise RuntimeError(
            "Posterior sample dimensions do not agree."
        )


    dy_a = test_df[
        "dy_a"
    ].to_numpy(
        dtype=np.int64
    )

    dy_b = test_df[
        "dy_b"
    ].to_numpy(
        dtype=np.int64
    )

    y = test_df[
        "y"
    ].to_numpy(
        dtype=float
    )

    weight = test_df[
        "weight"
    ].to_numpy(
        dtype=float
    )


    if channel in {
        "pace_contrast",
        "q_contrast",
    }:

        tenure = test_df[
            "tenure_diff"
        ].to_numpy(
            dtype=float
        )

        csy = None

    else:

        tenure = test_df[
            "tenure_mean"
        ].to_numpy(
            dtype=float
        )

        csy = test_df[
            "csy"
        ].to_numpy(
            dtype=np.int64
        )


    if (
        np.any(
            weight
            <=
            0
        )
        or
        not np.isfinite(
            weight
        ).all()
    ):

        raise RuntimeError(
            f"{channel}: invalid held-out reliability weights."
        )


    lpd = np.empty(
        len(
            test_df
        ),
        dtype=float,
    )


    nu = float(
        CFG.student_t_nu
    )


    for start in range(
        0,
        len(
            test_df
        ),
        chunk_size,
    ):

        stop = min(
            start
            +
            chunk_size,
            len(
                test_df
            ),
        )


        a = dy_a[
            start:stop
        ]

        b = dy_b[
            start:stop
        ]

        x = tenure[
            start:stop
        ]


        if channel in {
            "pace_contrast",
            "q_contrast",
        }:

            mu = (
                alpha[
                    :,
                    a,
                ]
                -
                alpha[
                    :,
                    b,
                ]
                +
                gamma[
                    :,
                    None
                ]
                *
                x[
                    None,
                    :
                ]
            )

        else:

            c = csy[
                start:stop
            ]

            mu = (
                0.5
                *
                (
                    alpha[
                        :,
                        a,
                    ]
                    +
                    alpha[
                        :,
                        b,
                    ]
                )
                +
                car[
                    :,
                    c,
                ]
                +
                gamma[
                    :,
                    None
                ]
                *
                x[
                    None,
                    :
                ]
            )


        scale = (
            sigma[
                :,
                None
            ]
            /
            np.sqrt(
                weight[
                    start:stop
                ][
                    None,
                    :
                ]
            )
        )


        logp_draws = (
            st.t.logpdf(
                y[
                    start:stop
                ][
                    None,
                    :
                ],
                df=nu,
                loc=mu,
                scale=scale,
            )
        )


        lpd[
            start:stop
        ] = (
            logsumexp(
                logp_draws,
                axis=0,
            )
            -
            np.log(
                n_samples
            )
        )


    result = pd.DataFrame(
        {
            "candidate":
                candidate,

            "fold":
                int(
                    fold
                ),

            "channel":
                channel,

            "raceId":
                test_df[
                    "raceId"
                ]
                .to_numpy(
                    dtype=int
                ),

            "year":
                test_df[
                    "year"
                ]
                .to_numpy(
                    dtype=int
                ),

            "source_row":
                test_df[
                    "_source_row"
                ]
                .to_numpy(
                    dtype=int
                ),

            "lpd":
                lpd,
        }
    )


    result[
        "obs_id"
    ] = (
        result[
            "channel"
        ]
        .astype(str)
        +
        "|"
        +
        result[
            "source_row"
        ]
        .astype(str)
    )


    return result


# ============================================================
# 10. FIT HEALTH HELPER
# ============================================================

HEALTH_VARS_32GB = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "sigma_pace_contrast",
    "sigma_pace_level",
    "sigma_q_contrast",
    "sigma_q_level",
    "tau_hist_quali",
    "tau_hist_race",
]


def health_summary_32gb(
    trace,
    *,
    candidate,
    fold,
):

    posterior = (
        posterior_dataset_32gb(
            trace
        )
    )


    rhats = []
    esses = []


    for var in HEALTH_VARS_32GB:

        if var not in posterior:

            continue


        rhat_obj = az.rhat(
            posterior[
                var
            ]
        )

        ess_obj = az.ess(
            posterior[
                var
            ],
            method="bulk",
        )


        rhat_values = np.asarray(
            rhat_obj
        ).reshape(
            -1
        )

        ess_values = np.asarray(
            ess_obj
        ).reshape(
            -1
        )


        rhats.extend(
            rhat_values[
                np.isfinite(
                    rhat_values
                )
            ].tolist()
        )

        esses.extend(
            ess_values[
                np.isfinite(
                    ess_values
                )
            ].tolist()
        )


    stats = (
        sample_stats_dataset_32gb(
            trace
        )
    )


    divergences = 0


    if (
        stats is not None
        and
        "diverging"
        in stats
    ):

        divergences = int(
            np.asarray(
                stats[
                    "diverging"
                ]
            ).sum()
        )


    return {
        "candidate":
            candidate,

        "fold":
            int(
                fold
            ),

        "max_rhat":
            (
                float(
                    np.max(
                        rhats
                    )
                )
                if rhats
                else np.nan
            ),

        "min_ess_bulk":
            (
                float(
                    np.min(
                        esses
                    )
                )
                if esses
                else np.nan
            ),

        "divergences":
            divergences,
    }


# ============================================================
# 11. RUN / LOAD ALL 3 × 3 FITS
# ============================================================

prediction_frames_32gb = []
health_rows_32gb = []
fit_runtime_rows_32gb = []


for candidate_index, candidate in enumerate(
    STAGE32G_CANDIDATES
):

    print()
    print(
        "#" * 76
    )

    print(
        f"CANDIDATE: {candidate}"
    )

    print(
        "#" * 76
    )


    hist_q_pack_candidate = (
        historical_packs_32gb[
            candidate
        ][
            "quali"
        ]
    )

    hist_r_pack_candidate = (
        historical_packs_32gb[
            candidate
        ][
            "race"
        ]
    )


    for fold in range(
        STAGE32G_FOLDS
    ):

        print()
        print(
            "=" * 76
        )

        print(
            f"{candidate} — FOLD "
            f"{fold + 1}/{STAGE32G_FOLDS}"
        )

        print(
            "=" * 76
        )


        heldout_races = set(
            fold_table_32gb.loc[
                fold_table_32gb[
                    "fold"
                ]
                ==
                fold,
                "raceId",
            ]
            .astype(int)
        )


        (
            train_pc,
            test_pc,
        ) = candidate_modern_split_32gb(
            pc_32gb,
            candidate=candidate,
            channel="pace_contrast",
            heldout_races=heldout_races,
        )


        (
            train_pl,
            test_pl,
        ) = candidate_modern_split_32gb(
            pl_32gb,
            candidate=candidate,
            channel="pace_level",
            heldout_races=heldout_races,
        )


        (
            train_qc,
            test_qc,
        ) = candidate_modern_split_32gb(
            qc_32gb,
            candidate=candidate,
            channel="q_contrast",
            heldout_races=heldout_races,
        )


        (
            train_ql,
            test_ql,
        ) = candidate_modern_split_32gb(
            ql_32gb,
            candidate=candidate,
            channel="q1_level",
            heldout_races=heldout_races,
        )


        print(
            "Held-out races:",
            len(
                heldout_races
            ),
        )

        print(
            "Pace contrast:",
            f"{len(train_pc):,} train / "
            f"{len(test_pc):,} test"
        )

        print(
            "Pace level:",
            f"{len(train_pl):,} train / "
            f"{len(test_pl):,} test"
        )

        print(
            "Qualifying contrast:",
            f"{len(train_qc):,} train / "
            f"{len(test_qc):,} test"
        )

        print(
            "Q1 level:",
            f"{len(train_ql):,} train / "
            f"{len(test_ql):,} test"
        )


        trace_path = (
            STAGE32G_FIT_DIR
            /
            (
                f"{candidate}_"
                f"fold{fold}_"
                f"{STAGE32G_TUNE}tune_"
                f"{STAGE32G_DRAWS}draw_"
                f"{STAGE32G_CHAINS}chain_"
                "nutpie_jax.nc"
            )
        )


        if trace_path.exists():

            print()
            print(
                "Existing checkpoint found:"
            )

            print(
                trace_path
            )


            try:

                trace_32gb = (
                    xr.open_datatree(
                        trace_path,
                        engine=
                            STAGE32G_CHECKPOINT_ENGINE,
                    )
                )

            except Exception as exc:

                raise RuntimeError(
                    "Checkpoint exists but could not be read.\n"
                    "It will NOT be overwritten automatically.\n\n"
                    f"{trace_path}\n\n"
                    f"{repr(exc)}"
                ) from exc


            print(
                "Loaded checkpoint; no resampling."
            )

            elapsed_minutes = np.nan


        else:

            print()
            print(
                "Building model..."
            )


            model_32gb = (
                build_joint_model_1982_2025(
                    train_pc,
                    train_pl,
                    train_qc,
                    train_ql,
                    hist_q_pack_candidate,
                    hist_r_pack_candidate,
                    idx_all,
                    cfg=CFG,
                )
            )


            seed_32gb = (
                STAGE32G_BASE_SEED
                +
                100
                *
                candidate_index
                +
                fold
            )


            print(
                f"Sampling {STAGE32G_CHAINS} chains × "
                f"({STAGE32G_TUNE} tune + "
                f"{STAGE32G_DRAWS} posterior)"
            )

            print(
                "Sampler: nutpie / JAX"
            )

            print(
                "Seed:",
                seed_32gb,
            )


            t0_32gb = (
                time.perf_counter()
            )


            with model_32gb:

                trace_32gb = pm.sample(
                    draws=
                        STAGE32G_DRAWS,

                    tune=
                        STAGE32G_TUNE,

                    chains=
                        STAGE32G_CHAINS,

                    random_seed=
                        seed_32gb,

                    target_accept=
                        STAGE32G_TARGET_ACCEPT,

                    nuts_sampler=
                        "nutpie",

                    nuts_sampler_kwargs={
                        "backend":
                            "jax",
                    },

                    return_inferencedata=
                        True,

                    progressbar=
                        True,
                )


            elapsed_minutes = (
                time.perf_counter()
                -
                t0_32gb
            ) / 60.0


            print()
            print(
                "Sampling wall time:",
                f"{elapsed_minutes:.2f} min"
            )


            trace_32gb.to_netcdf(
                trace_path,
                engine=
                    STAGE32G_CHECKPOINT_ENGINE,
                mode="w",
            )


            print(
                "Saved ->"
            )

            print(
                trace_path
            )


        # ----------------------------------------------------
        # Health
        # ----------------------------------------------------

        health_row = (
            health_summary_32gb(
                trace_32gb,
                candidate=candidate,
                fold=fold,
            )
        )

        health_rows_32gb.append(
            health_row
        )


        print()
        print(
            "Fit health:"
        )

        print(
            "  max R-hat =",
            f"{health_row['max_rhat']:.4f}"
        )

        print(
            "  min ESS bulk =",
            f"{health_row['min_ess_bulk']:.1f}"
        )

        print(
            "  divergences =",
            health_row[
                "divergences"
            ],
        )


        fit_runtime_rows_32gb.append(
            {
                "candidate":
                    candidate,

                "fold":
                    fold,

                "minutes":
                    elapsed_minutes,

                "checkpoint":
                    str(
                        trace_path
                    ),
            }
        )


        # ----------------------------------------------------
        # Held-out scoring
        # ----------------------------------------------------

        print()
        print(
            "Scoring held-out observations..."
        )


        prediction_frames_32gb.append(
            heldout_log_predictive_density_32gb(
                trace_32gb,
                test_pc,
                channel=
                    "pace_contrast",
                candidate=
                    candidate,
                fold=
                    fold,
            )
        )


        prediction_frames_32gb.append(
            heldout_log_predictive_density_32gb(
                trace_32gb,
                test_pl,
                channel=
                    "pace_level",
                candidate=
                    candidate,
                fold=
                    fold,
            )
        )


        prediction_frames_32gb.append(
            heldout_log_predictive_density_32gb(
                trace_32gb,
                test_qc,
                channel=
                    "q_contrast",
                candidate=
                    candidate,
                fold=
                    fold,
            )
        )


        prediction_frames_32gb.append(
            heldout_log_predictive_density_32gb(
                trace_32gb,
                test_ql,
                channel=
                    "q1_level",
                candidate=
                    candidate,
                fold=
                    fold,
            )
        )


        # Free the large model/trace objects before next fit.
        try:
            del model_32gb
        except NameError:
            pass

        del trace_32gb

        gc.collect()


# ============================================================
# 12. COMBINE HELD-OUT SCORES
# ============================================================

scores_32gb = pd.concat(
    prediction_frames_32gb,
    ignore_index=True,
)


health_32gb = pd.DataFrame(
    health_rows_32gb
)


fit_runtime_32gb = pd.DataFrame(
    fit_runtime_rows_32gb
)


# ============================================================
# 13. COMPLETENESS / MATCHING AUDIT
# ============================================================

obs_counts_32gb = (
    scores_32gb
    .groupby(
        [
            "candidate",
            "channel",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "obs_id",
            "size",
        ),

        unique_rows=(
            "obs_id",
            "nunique",
        ),
    )
)


print()
print(
    "=" * 76
)

print(
    "HELD-OUT SCORE COMPLETENESS"
)

print(
    "=" * 76
)

display(
    obs_counts_32gb
)


pivot_presence_32gb = (
    scores_32gb[
        [
            "candidate",
            "obs_id",
        ]
    ]
    .drop_duplicates()
    .assign(
        present=1
    )
    .pivot(
        index="obs_id",
        columns="candidate",
        values="present",
    )
)


if pivot_presence_32gb.isna().any().any():

    raise RuntimeError(
        "Candidates were not scored on identical held-out "
        "observations."
    )


# ============================================================
# 14. TOTAL ELPD BY CANDIDATE
# ============================================================

total_elpd_32gb = (
    scores_32gb
    .groupby(
        "candidate",
        as_index=False,
    )
    .agg(
        heldout_rows=(
            "lpd",
            "size",
        ),

        total_elpd=(
            "lpd",
            "sum",
        ),

        mean_lpd=(
            "lpd",
            "mean",
        ),
    )
    .sort_values(
        "total_elpd",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


best_total_elpd_32gb = float(
    total_elpd_32gb.iloc[
        0
    ][
        "total_elpd"
    ]
)


total_elpd_32gb[
    "delta_elpd_from_best"
] = (
    total_elpd_32gb[
        "total_elpd"
    ]
    -
    best_total_elpd_32gb
)


print()
print(
    "=" * 76
)

print(
    "TOTAL HELD-OUT ELPD"
)

print(
    "=" * 76
)

display(
    total_elpd_32gb
)


# ============================================================
# 15. ELPD BY CHANNEL
# ============================================================

channel_elpd_32gb = (
    scores_32gb
    .groupby(
        [
            "candidate",
            "channel",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "lpd",
            "size",
        ),

        elpd=(
            "lpd",
            "sum",
        ),

        mean_lpd=(
            "lpd",
            "mean",
        ),
    )
    .sort_values(
        [
            "channel",
            "elpd",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


print()
print(
    "=" * 76
)

print(
    "HELD-OUT ELPD BY CHANNEL"
)

print(
    "=" * 76
)

display(
    channel_elpd_32gb
)


# ============================================================
# 16. ELPD BY FOLD
# ============================================================

fold_elpd_32gb = (
    scores_32gb
    .groupby(
        [
            "candidate",
            "fold",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "lpd",
            "size",
        ),

        elpd=(
            "lpd",
            "sum",
        ),

        mean_lpd=(
            "lpd",
            "mean",
        ),
    )
    .sort_values(
        [
            "fold",
            "elpd",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


print()
print(
    "=" * 76
)

print(
    "HELD-OUT ELPD BY FOLD"
)

print(
    "=" * 76
)

display(
    fold_elpd_32gb
)


# ============================================================
# 17. PAIRED RACE-CLUSTERED CANDIDATE COMPARISONS
# ============================================================
#
# Individual observations from the same race are correlated.
# Therefore the uncertainty calculation is based on total
# log-score difference PER RACE, not treating thousands of
# likelihood rows as independent.
# ============================================================

race_candidate_scores_32gb = (
    scores_32gb
    .groupby(
        [
            "candidate",
            "raceId",
        ],
        as_index=False,
    )
    .agg(
        race_elpd=(
            "lpd",
            "sum",
        )
    )
)


race_wide_32gb = (
    race_candidate_scores_32gb
    .pivot(
        index="raceId",
        columns="candidate",
        values="race_elpd",
    )
    .sort_index()
)


if race_wide_32gb.isna().any().any():

    raise RuntimeError(
        "Race-level candidate scores do not align."
    )


comparison_rows_32gb = []


for i, candidate_a in enumerate(
    STAGE32G_CANDIDATES
):

    for candidate_b in (
        STAGE32G_CANDIDATES[
            i
            +
            1:
        ]
    ):

        delta_by_race = (
            race_wide_32gb[
                candidate_a
            ]
            -
            race_wide_32gb[
                candidate_b
            ]
        ).to_numpy(
            dtype=float
        )


        n_races = len(
            delta_by_race
        )


        delta_total = float(
            delta_by_race.sum()
        )


        if n_races > 1:

            se_total = float(
                np.sqrt(
                    n_races
                    *
                    np.var(
                        delta_by_race,
                        ddof=1,
                    )
                )
            )

        else:

            se_total = np.nan


        z_like = (
            delta_total
            /
            se_total
            if (
                np.isfinite(
                    se_total
                )
                and
                se_total
                >
                0
            )
            else np.nan
        )


        comparison_rows_32gb.append(
            {
                "candidate_a":
                    candidate_a,

                "candidate_b":
                    candidate_b,

                "n_races":
                    n_races,

                "delta_elpd_A_minus_B":
                    delta_total,

                "race_clustered_se":
                    se_total,

                "delta_over_se":
                    z_like,

                "mean_delta_per_race":
                    float(
                        np.mean(
                            delta_by_race
                        )
                    ),

                "median_delta_per_race":
                    float(
                        np.median(
                            delta_by_race
                        )
                    ),

                "fraction_races_A_better":
                    float(
                        np.mean(
                            delta_by_race
                            >
                            0
                        )
                    ),
            }
        )


pairwise_elpd_32gb = pd.DataFrame(
    comparison_rows_32gb
)


print()
print(
    "=" * 76
)

print(
    "PAIRED RACE-CLUSTERED ELPD DIFFERENCES"
)

print(
    "=" * 76
)

display(
    pairwise_elpd_32gb
)


# ============================================================
# 18. SAMPLER HEALTH
# ============================================================

print()
print(
    "=" * 76
)

print(
    "SAMPLER HEALTH"
)

print(
    "=" * 76
)

display(
    health_32gb.sort_values(
        [
            "candidate",
            "fold",
        ]
    )
)


max_rhat_32gb = float(
    health_32gb[
        "max_rhat"
    ].max()
)

min_ess_32gb = float(
    health_32gb[
        "min_ess_bulk"
    ].min()
)

total_div_32gb = int(
    health_32gb[
        "divergences"
    ].sum()
)


# ============================================================
# 19. SAVE ALL COMPARISON OUTPUTS
# ============================================================

scores_path_32gb = (
    STAGE32G_DIR
    /
    "stage32g_b_heldout_row_scores.csv"
)

total_path_32gb = (
    STAGE32G_DIR
    /
    "stage32g_b_total_elpd.csv"
)

channel_path_32gb = (
    STAGE32G_DIR
    /
    "stage32g_b_channel_elpd.csv"
)

fold_path_32gb = (
    STAGE32G_DIR
    /
    "stage32g_b_fold_elpd.csv"
)

pairwise_path_32gb = (
    STAGE32G_DIR
    /
    "stage32g_b_pairwise_race_clustered_elpd.csv"
)

health_path_32gb = (
    STAGE32G_DIR
    /
    "stage32g_b_sampler_health.csv"
)

runtime_path_32gb = (
    STAGE32G_DIR
    /
    "stage32g_b_fit_runtimes.csv"
)


scores_32gb.to_csv(
    scores_path_32gb,
    index=False,
)

total_elpd_32gb.to_csv(
    total_path_32gb,
    index=False,
)

channel_elpd_32gb.to_csv(
    channel_path_32gb,
    index=False,
)

fold_elpd_32gb.to_csv(
    fold_path_32gb,
    index=False,
)

pairwise_elpd_32gb.to_csv(
    pairwise_path_32gb,
    index=False,
)

health_32gb.to_csv(
    health_path_32gb,
    index=False,
)

fit_runtime_32gb.to_csv(
    runtime_path_32gb,
    index=False,
)


# ============================================================
# 20. FINAL TRIAGE
# ============================================================

print()
print(
    "=" * 76
)

print(
    "STAGE 32G-B TRIAGE SUMMARY"
)

print(
    "=" * 76
)


print(
    "Candidates:",
    ", ".join(
        STAGE32G_CANDIDATES
    ),
)

print(
    "CV folds:",
    STAGE32G_FOLDS,
)

print(
    "Held-out races:",
    race_wide_32gb.shape[
        0
    ],
)

print(
    "Held-out likelihood rows per candidate:",
    int(
        scores_32gb.groupby(
            "candidate"
        ).size().iloc[
            0
        ]
    ),
)

print()

print(
    "Sampler:"
)

print(
    "  max R-hat:",
    f"{max_rhat_32gb:.4f}",
)

print(
    "  min ESS bulk:",
    f"{min_ess_32gb:.1f}",
)

print(
    "  total divergences:",
    total_div_32gb,
)


print()
print(
    "Total held-out ELPD:"
)

for _, row in total_elpd_32gb.iterrows():

    print(
        f"  {row['candidate']:12s}"
        f"  ELPD = {row['total_elpd']:.3f}"
        f"  Δbest = "
        f"{row['delta_elpd_from_best']:.3f}"
    )


print()
print(
    "PAIRWISE RACE-CLUSTERED DIFFERENCES:"
)

for _, row in pairwise_elpd_32gb.iterrows():

    print(
        f"  {row['candidate_a']} - "
        f"{row['candidate_b']}: "
        f"ΔELPD = "
        f"{row['delta_elpd_A_minus_B']:.3f} "
        f"± {row['race_clustered_se']:.3f} "
        f"(Δ/SE = {row['delta_over_se']:.2f})"
    )


hard_flags_32gb = []


if max_rhat_32gb > 1.02:

    hard_flags_32gb.append(
        "At least one validation fit has R-hat > 1.02."
    )


if total_div_32gb > 0:

    hard_flags_32gb.append(
        "At least one validation fit contains divergences."
    )


if not np.isfinite(
    scores_32gb[
        "lpd"
    ]
).all():

    hard_flags_32gb.append(
        "Non-finite held-out log predictive densities."
    )


print()


if hard_flags_32gb:

    print(
        "HARD FLAGS:"
    )

    for flag in hard_flags_32gb:

        print(
            "  -",
            flag,
        )

else:

    print(
        "HARD FLAGS: none"
    )


print()
print(
    "DECISION:"
)

print(
    "  Do NOT choose the memory half-life from the raw "
    "ELPD ordering alone."
)

print(
    "  Inspect the paired race-clustered ΔELPD ± SE table."
)

print(
    "  A clear candidate should improve predictive density "
    "consistently enough that the gain is large relative to "
    "its race-clustered uncertainty."
)

print(
    "  If 20 GP and 40 GP are effectively tied, we should "
    "treat the data as unable to distinguish them rather than "
    "pretending the tiny numerical winner is decisive."
)


print()
print(
    "Saved outputs ->"
)

print(
    STAGE32G_DIR
)

print()
print(
    "Stage 32G-B complete."
)

## Interrupted-fit recovery utilities


In [ ]:
# ============================================================
# STAGE 32H-B.2 — REPAIR INTERRUPTED TRACE IN MEMORY
#                  + REAL SALVAGE DIAGNOSTICS
# ============================================================
#
# PURPOSE
# -------
# The interrupted nutpie run may have produced slightly
# different completed draw counts across the four chains.
#
# NetCDF/xarray must represent all chains on one rectangular
# (chain, draw) grid, so unfinished tail positions can appear
# as NaN padding.
#
# We:
#
#   1. inspect the number of genuinely finite draws per chain;
#   2. verify missing values occur ONLY as trailing padding;
#   3. trim IN MEMORY to the common complete prefix;
#   4. verify the retained posterior contains no NaNs;
#   5. calculate proper R-hat / ESS / BFMI / divergences;
#   6. inspect global and latent-state chain agreement.
#
# IMPORTANT
# ---------
# This cell is READ-ONLY.
#
# It does NOT alter or overwrite the partial checkpoint.
# ============================================================

from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd
import xarray as xr

from IPython.display import display


# ============================================================
# 0. LOCATE PARTIAL CHECKPOINT
# ============================================================

if "LIVE_TRACE_PATH_32HB" not in globals():

    raise RuntimeError(
        "LIVE_TRACE_PATH_32HB is not defined. "
        "Run Stage 32H-B far enough to construct the checkpoint "
        "path first."
    )


partial_trace_path_32hb2 = Path(
    LIVE_TRACE_PATH_32HB
)


if not partial_trace_path_32hb2.exists():

    raise RuntimeError(
        "Partial LIVE checkpoint does not exist:\n"
        f"{partial_trace_path_32hb2}"
    )


print(
    "=" * 76
)

print(
    "STAGE 32H-B.2 — INTERRUPTED TRACE RECOVERY DIAGNOSTIC"
)

print(
    "=" * 76
)

print(
    "Checkpoint:"
)

print(
    partial_trace_path_32hb2
)


# ============================================================
# 1. OPEN THE DATATREE DIRECTLY
# ============================================================

partial_tree_32hb2 = xr.open_datatree(

    partial_trace_path_32hb2,

    engine="h5netcdf",
)


if "posterior" not in partial_tree_32hb2:

    raise RuntimeError(
        "Checkpoint has no posterior group."
    )


if "sample_stats" not in partial_tree_32hb2:

    raise RuntimeError(
        "Checkpoint has no sample_stats group."
    )


posterior_raw_32hb2 = (
    partial_tree_32hb2[
        "posterior"
    ]
    .to_dataset()
)


stats_raw_32hb2 = (
    partial_tree_32hb2[
        "sample_stats"
    ]
    .to_dataset()
)


n_chains_raw_32hb2 = int(
    posterior_raw_32hb2.sizes[
        "chain"
    ]
)


n_draw_slots_raw_32hb2 = int(
    posterior_raw_32hb2.sizes[
        "draw"
    ]
)


print()
print(
    "Stored chain dimension:",
    n_chains_raw_32hb2,
)

print(
    "Stored draw dimension:",
    n_draw_slots_raw_32hb2,
)


if n_chains_raw_32hb2 != 4:

    raise RuntimeError(
        f"Expected four chains, found {n_chains_raw_32hb2}."
    )


# ============================================================
# 2. FIND GENUINE COMPLETED DRAWS PER CHAIN
# ============================================================
#
# Use a scalar posterior variable as the completion marker.
#
# Every legitimate posterior draw must contain a finite
# sigma_driver_base value.
# ============================================================

completion_var_32hb2 = (
    "sigma_driver_base"
)


if completion_var_32hb2 not in posterior_raw_32hb2:

    raise RuntimeError(
        f"{completion_var_32hb2!r} is missing from posterior."
    )


completion_values_32hb2 = np.asarray(

    posterior_raw_32hb2[
        completion_var_32hb2
    ].values,

    dtype=float,
)


if completion_values_32hb2.ndim != 2:

    raise RuntimeError(
        "Completion marker is unexpectedly non-scalar."
    )


def contiguous_finite_prefix_32hb2(
    values,
):

    finite = np.isfinite(
        values
    )


    bad = np.flatnonzero(
        ~finite
    )


    if len(
        bad
    ) == 0:

        prefix = len(
            finite
        )

    else:

        prefix = int(
            bad[
                0
            ]
        )


    # If anything becomes finite again AFTER the first NaN,
    # this is not simple interruption padding.

    internal_gap = bool(
        finite[
            prefix:
        ].any()
    )


    return (
        prefix,
        int(
            finite.sum()
        ),
        internal_gap,
    )


completion_rows_32hb2 = []


for chain in range(
    n_chains_raw_32hb2
):

    (
        prefix,
        n_finite,
        internal_gap,
    ) = contiguous_finite_prefix_32hb2(

        completion_values_32hb2[
            chain
        ]
    )


    completion_rows_32hb2.append(
        {
            "chain":
                chain,

            "stored_draw_slots":
                n_draw_slots_raw_32hb2,

            "contiguous_complete_draws":
                prefix,

            "total_finite_draws":
                n_finite,

            "internal_gap":
                internal_gap,

            "trailing_padding":
                (
                    n_draw_slots_raw_32hb2
                    -
                    prefix
                ),
        }
    )


completion_table_32hb2 = pd.DataFrame(
    completion_rows_32hb2
)


print()
print(
    "=" * 76
)

print(
    "ACTUAL COMPLETED DRAWS BY CHAIN"
)

print(
    "=" * 76
)


display(
    completion_table_32hb2
)


if completion_table_32hb2[
    "internal_gap"
].any():

    raise RuntimeError(
        "At least one chain contains an internal missing-value "
        "gap rather than simple trailing interruption padding.\n"
        "Do not salvage automatically."
    )


COMMON_DRAWS_32HB2 = int(

    completion_table_32hb2[
        "contiguous_complete_draws"
    ].min()
)


MAX_COMPLETED_DRAWS_32HB2 = int(

    completion_table_32hb2[
        "contiguous_complete_draws"
    ].max()
)


print(
    "Common complete prefix:",
    COMMON_DRAWS_32HB2,
)


print(
    "Most-complete chain:",
    MAX_COMPLETED_DRAWS_32HB2,
)


print(
    "Draws lost by equal-length trimming:",
    int(
        completion_table_32hb2[
            "contiguous_complete_draws"
        ].sum()
        -
        (
            n_chains_raw_32hb2
            *
            COMMON_DRAWS_32HB2
        )
    ),
)


if COMMON_DRAWS_32HB2 < 50:

    raise RuntimeError(
        "Too few common completed draws for useful diagnostics."
    )


# ============================================================
# 3. TRIM ONLY IN MEMORY
# ============================================================

posterior_clean_32hb2 = (
    posterior_raw_32hb2
    .isel(
        draw=slice(
            0,
            COMMON_DRAWS_32HB2,
        )
    )
    .load()
)


stats_clean_32hb2 = (
    stats_raw_32hb2
    .isel(
        draw=slice(
            0,
            COMMON_DRAWS_32HB2,
        )
    )
    .load()
)


print()
print(
    "Diagnostic posterior dimensions:",
    dict(
        posterior_clean_32hb2.sizes
    ),
)


# ============================================================
# 4. VERIFY IMPORTANT POSTERIOR VARIABLES ARE FULLY FINITE
# ============================================================

key_vars_32hb2 = [

    "sigma_driver_base",
    "sigma_driver_drift",

    "sigma_car",

    "gamma_tenure",

    "sigma_pace_contrast",
    "sigma_pace_level",

    "sigma_q_contrast",
    "sigma_q_level",

    "tau_hist_quali",
    "tau_hist_race",

    "alpha_dy",
    "car_csy",

    "theta_z",
    "drift_z",
    "car_z",
]


key_vars_32hb2 = [
    var
    for var
    in key_vars_32hb2
    if var
    in posterior_clean_32hb2
]


finite_rows_32hb2 = []


for variable in key_vars_32hb2:

    values = np.asarray(
        posterior_clean_32hb2[
            variable
        ].values
    )


    finite_rows_32hb2.append(
        {
            "variable":
                variable,

            "n_values":
                values.size,

            "missing":
                int(
                    np.isnan(
                        values
                    ).sum()
                ),

            "nonfinite":
                int(
                    (
                        ~np.isfinite(
                            values
                        )
                    ).sum()
                ),
        }
    )


finite_audit_32hb2 = pd.DataFrame(
    finite_rows_32hb2
)


print()
print(
    "=" * 76
)

print(
    "POST-TRIM FINITE-VALUE AUDIT"
)

print(
    "=" * 76
)


display(
    finite_audit_32hb2
)


if finite_audit_32hb2[
    "nonfinite"
].sum() != 0:

    raise RuntimeError(
        "Non-finite posterior values remain inside the common "
        "completed prefix."
    )


# ============================================================
# 5. BUILD A CLEAN INFERENCEDATA OBJECT IN MEMORY
# ============================================================

partial_clean_idata_32hb2 = az.InferenceData(

    posterior=
        posterior_clean_32hb2,

    sample_stats=
        stats_clean_32hb2,
)


# ============================================================
# 6. DIVERGENCES
# ============================================================

print()
print(
    "=" * 76
)

print(
    "DIVERGENCES"
)

print(
    "=" * 76
)


if "diverging" in stats_clean_32hb2:

    divergence_values_32hb2 = np.asarray(

        stats_clean_32hb2[
            "diverging"
        ].values
    )


    divergence_by_chain_32hb2 = (

        divergence_values_32hb2
        .astype(
            bool
        )
        .sum(
            axis=1
        )
        .astype(int)
    )


    divergence_table_32hb2 = pd.DataFrame(
        {
            "chain":
                np.arange(
                    n_chains_raw_32hb2
                ),

            "divergences":
                divergence_by_chain_32hb2,
        }
    )


    display(
        divergence_table_32hb2
    )


    total_divergences_32hb2 = int(
        divergence_by_chain_32hb2.sum()
    )


else:

    total_divergences_32hb2 = np.nan

    print(
        "'diverging' not present in sample_stats."
    )


print(
    "Total divergences:",
    total_divergences_32hb2,
)


# ============================================================
# 7. BFMI — USE ENERGY ARRAY, NOT THE DATATREE
# ============================================================

print()
print(
    "=" * 76
)

print(
    "BFMI"
)

print(
    "=" * 76
)


if "energy" in stats_clean_32hb2:

    energy_32hb2 = np.asarray(

        stats_clean_32hb2[
            "energy"
        ].values,

        dtype=float,
    )


    if not np.isfinite(
        energy_32hb2
    ).all():

        raise RuntimeError(
            "Energy contains non-finite values inside the "
            "common completed prefix."
        )


    bfmi_32hb2 = np.asarray(
        az.bfmi(
            energy_32hb2
        ),
        dtype=float,
    )


    bfmi_table_32hb2 = pd.DataFrame(
        {
            "chain":
                np.arange(
                    len(
                        bfmi_32hb2
                    )
                ),

            "BFMI":
                bfmi_32hb2,
        }
    )


    display(
        bfmi_table_32hb2
    )


    min_bfmi_32hb2 = float(
        np.min(
            bfmi_32hb2
        )
    )


    print(
        "Minimum BFMI:",
        f"{min_bfmi_32hb2:.4f}",
    )


else:

    bfmi_32hb2 = np.array(
        [
            np.nan
        ]
    )


    min_bfmi_32hb2 = np.nan


    print(
        "'energy' not present in sample_stats."
    )


# ============================================================
# 8. NUTS STEP INFORMATION
# ============================================================

print()
print(
    "=" * 76
)

print(
    "NUTS STEP INFORMATION"
)

print(
    "=" * 76
)


print(
    "sample_stats variables:"
)

print(
    sorted(
        list(
            stats_clean_32hb2.data_vars
        )
    )
)


if "n_steps" in stats_clean_32hb2:

    n_steps_32hb2 = np.asarray(

        stats_clean_32hb2[
            "n_steps"
        ].values,

        dtype=float,
    )


    print()
    print(
        "n_steps minimum:",
        int(
            np.min(
                n_steps_32hb2
            )
        ),
    )


    print(
        "n_steps median:",
        float(
            np.median(
                n_steps_32hb2
            )
        ),
    )


    print(
        "n_steps 90th percentile:",
        float(
            np.quantile(
                n_steps_32hb2,
                0.90,
            )
        ),
    )


    print(
        "n_steps 99th percentile:",
        float(
            np.quantile(
                n_steps_32hb2,
                0.99,
            )
        ),
    )


    print(
        "n_steps maximum:",
        int(
            np.max(
                n_steps_32hb2
            )
        ),
    )


# Look for an explicit maximum-tree-depth indicator if
# this nutpie version stored one.

max_depth_candidates_32hb2 = [

    "reached_max_treedepth",
    "reached_max_tree_depth",
    "max_depth",
    "maxdepth_reached",
]


for field in max_depth_candidates_32hb2:

    if field in stats_clean_32hb2:

        values = np.asarray(
            stats_clean_32hb2[
                field
            ].values
        )


        print()
        print(
            f"{field}:",
            int(
                np.asarray(
                    values,
                    dtype=bool,
                ).sum()
            ),
        )


# ============================================================
# 9. R-HAT / ESS
# ============================================================

print()
print(
    "=" * 76
)

print(
    "R-HAT / ESS"
)

print(
    "=" * 76
)


rhat_ds_32hb2 = az.rhat(

    posterior_clean_32hb2,

    method="rank",
)


ess_bulk_ds_32hb2 = az.ess(

    posterior_clean_32hb2,

    method="bulk",
)


ess_tail_ds_32hb2 = az.ess(

    posterior_clean_32hb2,

    method="tail",
)


def diagnostic_values_32hb2(
    dataset,
    variable,
):

    if variable not in dataset:

        return np.array(
            [],
            dtype=float,
        )


    values = np.asarray(

        dataset[
            variable
        ].values,

        dtype=float,
    ).ravel()


    return values[
        np.isfinite(
            values
        )
    ]


def diagnostic_row_32hb2(
    variable,
):

    rh = diagnostic_values_32hb2(
        rhat_ds_32hb2,
        variable,
    )


    eb = diagnostic_values_32hb2(
        ess_bulk_ds_32hb2,
        variable,
    )


    et = diagnostic_values_32hb2(
        ess_tail_ds_32hb2,
        variable,
    )


    return {
        "variable":
            variable,

        "n_elements":
            len(
                rh
            ),

        "max_rhat":
            (
                float(
                    np.max(
                        rh
                    )
                )
                if len(
                    rh
                )
                else np.nan
            ),

        "p95_rhat":
            (
                float(
                    np.quantile(
                        rh,
                        0.95,
                    )
                )
                if len(
                    rh
                )
                else np.nan
            ),

        "n_rhat_gt_1.01":
            int(
                (
                    rh
                    >
                    1.01
                ).sum()
            ),

        "n_rhat_gt_1.03":
            int(
                (
                    rh
                    >
                    1.03
                ).sum()
            ),

        "n_rhat_gt_1.05":
            int(
                (
                    rh
                    >
                    1.05
                ).sum()
            ),

        "min_ess_bulk":
            (
                float(
                    np.min(
                        eb
                    )
                )
                if len(
                    eb
                )
                else np.nan
            ),

        "p05_ess_bulk":
            (
                float(
                    np.quantile(
                        eb,
                        0.05,
                    )
                )
                if len(
                    eb
                )
                else np.nan
            ),

        "min_ess_tail":
            (
                float(
                    np.min(
                        et
                    )
                )
                if len(
                    et
                )
                else np.nan
            ),

        "p05_ess_tail":
            (
                float(
                    np.quantile(
                        et,
                        0.05,
                    )
                )
                if len(
                    et
                )
                else np.nan
            ),
    }


# ============================================================
# 10. GLOBAL PARAMETERS
# ============================================================

global_vars_32hb2 = [

    "sigma_driver_base",
    "sigma_driver_drift",

    "sigma_car",

    "gamma_tenure",

    "sigma_pace_contrast",
    "sigma_pace_level",

    "sigma_q_contrast",
    "sigma_q_level",

    "tau_hist_quali",
    "tau_hist_race",
]


global_vars_32hb2 = [
    var
    for var
    in global_vars_32hb2
    if var
    in posterior_clean_32hb2
]


global_diag_32hb2 = pd.DataFrame(
    [
        diagnostic_row_32hb2(
            variable
        )
        for variable
        in global_vars_32hb2
    ]
)


print()
print(
    "Global diagnostic metrics:"
)


display(
    global_diag_32hb2
    .sort_values(
        "max_rhat",
        ascending=False,
    )
)


print()
print(
    "Global posterior summary:"
)


global_summary_32hb2 = az.summary(

    partial_clean_idata_32hb2,

    var_names=
        global_vars_32hb2,

    hdi_prob=
        0.89,

    round_to=
        4,
)


display(
    global_summary_32hb2
)


# ============================================================
# 11. LATENT-STATE DIAGNOSTICS
# ============================================================

latent_vars_32hb2 = [

    "alpha_dy",
    "car_csy",

    "theta_z",
    "drift_z",
    "car_z",
]


latent_vars_32hb2 = [
    var
    for var
    in latent_vars_32hb2
    if var
    in posterior_clean_32hb2
]


latent_diag_32hb2 = pd.DataFrame(
    [
        diagnostic_row_32hb2(
            variable
        )
        for variable
        in latent_vars_32hb2
    ]
)


print()
print(
    "=" * 76
)

print(
    "LATENT-STATE DIAGNOSTICS"
)

print(
    "=" * 76
)


display(
    latent_diag_32hb2
)


# ============================================================
# 12. OVERALL R-HAT / ESS DISTRIBUTION
# ============================================================

all_rhat_32hb2 = np.concatenate(
    [
        diagnostic_values_32hb2(
            rhat_ds_32hb2,
            variable,
        )
        for variable
        in rhat_ds_32hb2.data_vars
        if len(
            diagnostic_values_32hb2(
                rhat_ds_32hb2,
                variable,
            )
        )
        >
        0
    ]
)


all_bulk_ess_32hb2 = np.concatenate(
    [
        diagnostic_values_32hb2(
            ess_bulk_ds_32hb2,
            variable,
        )
        for variable
        in ess_bulk_ds_32hb2.data_vars
        if len(
            diagnostic_values_32hb2(
                ess_bulk_ds_32hb2,
                variable,
            )
        )
        >
        0
    ]
)


print()
print(
    "=" * 76
)

print(
    "OVERALL PARTIAL-TRACE HEALTH"
)

print(
    "=" * 76
)


print(
    "Diagnostic draws / chain:",
    COMMON_DRAWS_32HB2,
)


print(
    "Total diagnostic draws:",
    (
        n_chains_raw_32hb2
        *
        COMMON_DRAWS_32HB2
    ),
)


print(
    "R-hat elements:",
    len(
        all_rhat_32hb2
    ),
)


print(
    "Max R-hat:",
    f"{np.max(all_rhat_32hb2):.5f}",
)


print(
    "95th percentile R-hat:",
    f"{np.quantile(all_rhat_32hb2, 0.95):.5f}",
)


print(
    "R-hat > 1.01:",
    int(
        (
            all_rhat_32hb2
            >
            1.01
        ).sum()
    ),
)


print(
    "R-hat > 1.03:",
    int(
        (
            all_rhat_32hb2
            >
            1.03
        ).sum()
    ),
)


print(
    "R-hat > 1.05:",
    int(
        (
            all_rhat_32hb2
            >
            1.05
        ).sum()
    ),
)


print(
    "Minimum bulk ESS:",
    f"{np.min(all_bulk_ess_32hb2):.1f}",
)


print(
    "5th percentile bulk ESS:",
    f"{np.quantile(all_bulk_ess_32hb2, 0.05):.1f}",
)


print(
    "Median bulk ESS:",
    f"{np.median(all_bulk_ess_32hb2):.1f}",
)


# ============================================================
# 13. BETWEEN-CHAIN AGREEMENT — GLOBAL PARAMETERS
# ============================================================

chain_agreement_rows_32hb2 = []


for variable in global_vars_32hb2:

    values = np.asarray(

        posterior_clean_32hb2[
            variable
        ].values,

        dtype=float,
    )


    chain_means = np.mean(
        values,
        axis=1,
    )


    pooled_sd = float(
        np.std(
            values.reshape(
                -1
            ),
            ddof=1,
        )
    )


    chain_range = float(
        np.max(
            chain_means
        )
        -
        np.min(
            chain_means
        )
    )


    chain_agreement_rows_32hb2.append(
        {
            "variable":
                variable,

            "chain0_mean":
                chain_means[
                    0
                ],

            "chain1_mean":
                chain_means[
                    1
                ],

            "chain2_mean":
                chain_means[
                    2
                ],

            "chain3_mean":
                chain_means[
                    3
                ],

            "pooled_sd":
                pooled_sd,

            "chain_mean_range_sd":
                (
                    chain_range
                    /
                    pooled_sd
                    if pooled_sd
                    >
                    0
                    else np.nan
                ),
        }
    )


chain_agreement_32hb2 = pd.DataFrame(
    chain_agreement_rows_32hb2
)


print()
print(
    "=" * 76
)

print(
    "GLOBAL BETWEEN-CHAIN AGREEMENT"
)

print(
    "=" * 76
)


display(
    chain_agreement_32hb2
    .sort_values(
        "chain_mean_range_sd",
        ascending=False,
    )
)


# ============================================================
# 14. FIRST-HALF VS SECOND-HALF DRIFT
# ============================================================

half_point_32hb2 = (
    COMMON_DRAWS_32HB2
    //
    2
)


half_shift_rows_32hb2 = []


for variable in global_vars_32hb2:

    values = np.asarray(

        posterior_clean_32hb2[
            variable
        ].values,

        dtype=float,
    )


    pooled_sd = float(
        np.std(
            values.reshape(
                -1
            ),
            ddof=1,
        )
    )


    for chain in range(
        n_chains_raw_32hb2
    ):

        first_mean = float(
            np.mean(
                values[
                    chain,
                    :half_point_32hb2,
                ]
            )
        )


        second_mean = float(
            np.mean(
                values[
                    chain,
                    half_point_32hb2:,
                ]
            )
        )


        shift = (
            second_mean
            -
            first_mean
        )


        half_shift_rows_32hb2.append(
            {
                "variable":
                    variable,

                "chain":
                    chain,

                "first_half_mean":
                    first_mean,

                "second_half_mean":
                    second_mean,

                "shift_in_posterior_sd":
                    (
                        shift
                        /
                        pooled_sd
                        if pooled_sd
                        >
                        0
                        else np.nan
                    ),
            }
        )


half_shift_32hb2 = pd.DataFrame(
    half_shift_rows_32hb2
)


half_shift_32hb2[
    "abs_shift_sd"
] = np.abs(
    half_shift_32hb2[
        "shift_in_posterior_sd"
    ]
)


half_shift_summary_32hb2 = (
    half_shift_32hb2
    .groupby(
        "variable",
        as_index=False,
    )
    .agg(
        max_abs_half_shift_sd=(
            "abs_shift_sd",
            "max",
        ),

        mean_abs_half_shift_sd=(
            "abs_shift_sd",
            "mean",
        ),
    )
    .sort_values(
        "max_abs_half_shift_sd",
        ascending=False,
    )
)


print()
print(
    "=" * 76
)

print(
    "FIRST-HALF → SECOND-HALF GLOBAL DRIFT"
)

print(
    "=" * 76
)


display(
    half_shift_summary_32hb2
)


# ============================================================
# 15. PRELIMINARY SALVAGE TRIAGE
# ============================================================

max_global_rhat_32hb2 = float(
    global_diag_32hb2[
        "max_rhat"
    ].max()
)


max_all_rhat_32hb2 = float(
    np.max(
        all_rhat_32hb2
    )
)


n_all_rhat_gt_103_32hb2 = int(
    (
        all_rhat_32hb2
        >
        1.03
    ).sum()
)


n_all_rhat_gt_105_32hb2 = int(
    (
        all_rhat_32hb2
        >
        1.05
    ).sum()
)


max_chain_range_sd_32hb2 = float(

    chain_agreement_32hb2[
        "chain_mean_range_sd"
    ].max()
)


max_half_shift_sd_32hb2 = float(

    half_shift_summary_32hb2[
        "max_abs_half_shift_sd"
    ].max()
)


print()
print(
    "=" * 76
)

print(
    "STAGE 32H-B.2 SALVAGE TRIAGE SUMMARY"
)

print(
    "=" * 76
)


print(
    "Common genuine draws / chain:",
    COMMON_DRAWS_32HB2,
)


print(
    "Total divergences:",
    total_divergences_32hb2,
)


print(
    "Minimum BFMI:",
    (
        f"{min_bfmi_32hb2:.4f}"
        if np.isfinite(
            min_bfmi_32hb2
        )
        else "NA"
    ),
)


print(
    "Max GLOBAL R-hat:",
    f"{max_global_rhat_32hb2:.5f}",
)


print(
    "Max ALL-PARAMETER R-hat:",
    f"{max_all_rhat_32hb2:.5f}",
)


print(
    "All-parameter R-hat > 1.03:",
    n_all_rhat_gt_103_32hb2,
)


print(
    "All-parameter R-hat > 1.05:",
    n_all_rhat_gt_105_32hb2,
)


print(
    "Minimum bulk ESS:",
    f"{np.min(all_bulk_ess_32hb2):.1f}",
)


print(
    "Max global chain-mean range / posterior SD:",
    f"{max_chain_range_sd_32hb2:.3f}",
)


print(
    "Max global half-run shift / posterior SD:",
    f"{max_half_shift_sd_32hb2:.3f}",
)


print()
print(
    "Do NOT sample or delete anything yet."
)

print(
    "Use this output to decide whether the interrupted segment "
    "is worth salvaging."
)

In [ ]:
# ============================================================
# STAGE 32H-B.2a — INSPECT INTERRUPTED CHECKPOINT STRUCTURE
# ============================================================

import numpy as np
import pandas as pd
import xarray as xr

from IPython.display import display


tree = xr.open_datatree(
    LIVE_TRACE_PATH_32HB,
    engine="h5netcdf",
)

posterior = tree["posterior"].to_dataset()
stats = tree["sample_stats"].to_dataset()


marker = np.asarray(
    posterior["sigma_driver_base"].values,
    dtype=float,
)


rows = []


for chain in range(marker.shape[0]):

    x = marker[chain]

    finite = np.isfinite(x)

    finite_idx = np.flatnonzero(finite)


    if len(finite_idx) == 0:

        first_finite = None
        last_finite = None
        n_finite = 0

    else:

        first_finite = int(finite_idx[0])
        last_finite = int(finite_idx[-1])
        n_finite = int(len(finite_idx))


    # Longest finite run anywhere in the stored chain
    longest_run = 0
    current_run = 0

    for ok in finite:

        if ok:
            current_run += 1
            longest_run = max(
                longest_run,
                current_run,
            )

        else:
            current_run = 0


    rows.append(
        {
            "chain": chain,
            "stored_slots": len(x),
            "n_finite": n_finite,
            "first_finite_index": first_finite,
            "last_finite_index": last_finite,
            "longest_contiguous_finite_run": longest_run,
            "n_nan": int((~finite).sum()),
        }
    )


structure = pd.DataFrame(rows)


print("=" * 76)
print("POSTERIOR STORAGE STRUCTURE")
print("=" * 76)

display(structure)


# ------------------------------------------------------------
# Show first 20 and last 20 finite masks explicitly
# ------------------------------------------------------------

for chain in range(marker.shape[0]):

    finite = np.isfinite(
        marker[chain]
    )

    print()
    print(f"Chain {chain}")

    print(
        "first 20 finite flags:",
        finite[:20].astype(int).tolist(),
    )

    print(
        "last 20 finite flags:",
        finite[-20:].astype(int).tolist(),
    )


# ------------------------------------------------------------
# Check whether this pattern occurs across all key variables
# ------------------------------------------------------------

key_vars = [
    "sigma_driver_base",
    "sigma_driver_drift",
    "sigma_car",
    "gamma_tenure",
    "alpha_dy",
    "car_csy",
]


audit_rows = []


for var in key_vars:

    if var not in posterior:
        continue

    x = np.asarray(
        posterior[var].values
    )

    # collapse every dimension except chain/draw:
    finite = np.isfinite(x)

    if x.ndim > 2:

        finite = finite.all(
            axis=tuple(
                range(
                    2,
                    x.ndim,
                )
            )
        )


    for chain in range(
        finite.shape[0]
    ):

        audit_rows.append(
            {
                "variable": var,
                "chain": chain,
                "finite_draws": int(
                    finite[chain].sum()
                ),
                "first_draw_finite": bool(
                    finite[chain, 0]
                ),
                "last_draw_finite": bool(
                    finite[chain, -1]
                ),
            }
        )


print()
print("=" * 76)
print("FINITE DRAW COUNTS ACROSS KEY VARIABLES")
print("=" * 76)

display(
    pd.DataFrame(
        audit_rows
    )
)

In [ ]:
# ============================================================
# STAGE 32H-B RECOVERY — ARCHIVE INTERRUPTED FIT
# ============================================================
#
# Preserve the 318 / 281 / 216 / 0 interrupted posterior
# for later validation, but move it out of the production
# checkpoint path so Stage 32H-B can start a clean fit.
# ============================================================

from pathlib import Path
import shutil


trace_path = Path(
    LIVE_TRACE_PATH_32HB
)

manifest_path = Path(
    LIVE_MANIFEST_PATH_32HB
)


partial_trace_path = trace_path.with_name(
    trace_path.stem
    +
    "_INTERRUPTED_318_281_216_0"
    +
    trace_path.suffix
)


partial_manifest_path = manifest_path.with_name(
    manifest_path.stem
    +
    "_INTERRUPTED_318_281_216_0"
    +
    manifest_path.suffix
)


print("=" * 76)
print("ARCHIVING INTERRUPTED LIVE FIT")
print("=" * 76)

print()
print("Current production checkpoint:")
print(trace_path)

print()
print("Interrupted archive:")
print(partial_trace_path)


# ------------------------------------------------------------
# Guard against accidentally overwriting an existing archive
# ------------------------------------------------------------

if partial_trace_path.exists():

    raise RuntimeError(
        "Interrupted archive already exists:\n"
        f"{partial_trace_path}\n\n"
        "Nothing has been changed."
    )


if not trace_path.exists():

    raise RuntimeError(
        "The interrupted production checkpoint no longer exists:\n"
        f"{trace_path}"
    )


# ------------------------------------------------------------
# Move the NetCDF rather than delete it
# ------------------------------------------------------------

trace_path.rename(
    partial_trace_path
)


print()
print("Archived posterior successfully.")


# ------------------------------------------------------------
# Archive its manifest too, if Stage 32H-B had already written
# one before reaching the failed dimension check.
# ------------------------------------------------------------

if manifest_path.exists():

    if partial_manifest_path.exists():

        raise RuntimeError(
            "Posterior was archived, but interrupted manifest "
            "archive already exists:\n"
            f"{partial_manifest_path}\n\n"
            "Resolve that filename manually before continuing."
        )

    manifest_path.rename(
        partial_manifest_path
    )

    print(
        "Archived manifest successfully."
    )

else:

    print(
        "No manifest existed for the interrupted run."
    )


# ------------------------------------------------------------
# Final check
# ------------------------------------------------------------

print()
print("=" * 76)
print("RECOVERY STATUS")
print("=" * 76)

print(
    "Interrupted posterior preserved:",
    partial_trace_path.exists(),
)

print(
    "Production checkpoint path now free:",
    not trace_path.exists(),
)

print()
print(
    "You can now rerun Stage 32H-B from the top."
)

In [ ]:
# ============================================================
# CLOSE OPEN HANDLES TO THE INTERRUPTED NETCDF
# ============================================================

import gc
import time


possible_open_objects = [
    "live_trace_2026",

    "partial_tree_32hb2",
    "partial_tree_32hb1",

    "partial_idata_32hb1",

    "tree",

    "posterior_raw_32hb2",
    "stats_raw_32hb2",

    "partial_trace_32hb1",
]


print("=" * 76)
print("CLOSING POSSIBLE OPEN NETCDF HANDLES")
print("=" * 76)


for name in possible_open_objects:

    if name not in globals():
        continue

    obj = globals()[name]

    close_method = getattr(
        obj,
        "close",
        None,
    )

    if callable(
        close_method
    ):

        try:

            close_method()

            print(
                f"Closed: {name}"
            )

        except Exception as exc:

            print(
                f"Could not close {name}: "
                f"{type(exc).__name__}: {exc}"
            )


# Remove references as well, so xarray's file managers can be
# garbage-collected.

for name in possible_open_objects:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()

# Give Windows a moment to release the file handle.
time.sleep(
    1.0
)


print()
print(
    "Cleanup complete."
)

print(
    "Now rerun the archive cell."
)

## LIVE posterior interrogation / stability audits


In [ ]:
# ============================================================
# STAGE 32H-D — LIVE STATE INTERROGATION
# ============================================================
#
# PURPOSE
# -------
# Now that Stage 32H-C has passed, interrogate what the LIVE
# posterior is actually doing before constructing any LIVE JAWS.
#
# This stage examines:
#
#   A. 2025 -> 2026 driver transitions
#   B. direct 2026 teammate alpha differences
#   C. 2026 driver/car allocation ridge by constructor
#   D. LIVE vs frozen 1982–2025 driver-season states
#   E. LIVE vs frozen 1982–2025 constructor-season states
#   F. where historical shifts are concentrated by year
#
# IMPORTANT
# ---------
# The frozen-vs-LIVE comparison does NOT isolate the familiarity
# redesign alone.
#
# Two things changed simultaneously:
#
#   1. familiarity was changed to the locked memory_20gp model;
#   2. 2026 evidence was added.
#
# Therefore this stage measures the NET effect on historical
# states. A pure familiarity-only effect would require a separate
# 1982–2025 refit under memory_20gp.
#
# NO JAWS IS CALCULATED HERE.
# ============================================================

from pathlib import Path
from itertools import combinations

import arviz as az
import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. PREREQUISITES
# ============================================================

required_32hd = [
    "posterior_32hc",
    "idx_all",
    "raw",
]


missing_32hd = [
    name
    for name in required_32hd
    if name not in globals()
]


if missing_32hd:

    raise RuntimeError(
        "Stage 32H-D requires:\n  "
        +
        "\n  ".join(
            missing_32hd
        )
        +
        "\n\nRun Stage 32H-C first."
    )


print("=" * 80)
print("STAGE 32H-D — LIVE STATE INTERROGATION")
print("=" * 80)

print()
print("Stage 32H-C status: PASS")
print("NO JAWS WILL BE CALCULATED.")


# ============================================================
# 1. POSTERIOR MATRICES
# ============================================================

def flatten_state_draws_32hd(
    variable,
):

    values = np.asarray(
        posterior_32hc[
            variable
        ].values,
        dtype=float,
    )


    if values.ndim != 3:

        raise RuntimeError(
            f"{variable} expected "
            "(chain, draw, state), "
            f"found {values.shape}."
        )


    return values.reshape(
        -1,
        values.shape[
            -1
        ],
    )


alpha_live_32hd = flatten_state_draws_32hd(
    "alpha_dy"
)


car_live_32hd = flatten_state_draws_32hd(
    "car_csy"
)


driver_states_32hd = list(
    idx_all.active_driver_years
)


constructor_states_32hd = list(
    idx_all.active_constructor_years
)


if alpha_live_32hd.shape[1] != len(
    driver_states_32hd
):

    raise RuntimeError(
        "alpha_dy state count mismatch."
    )


if car_live_32hd.shape[1] != len(
    constructor_states_32hd
):

    raise RuntimeError(
        "car_csy state count mismatch."
    )


driver_state_to_idx_32hd = {
    state: i
    for i, state
    in enumerate(
        driver_states_32hd
    )
}


constructor_state_to_idx_32hd = {
    state: i
    for i, state
    in enumerate(
        constructor_states_32hd
    )
}


# ============================================================
# 2. 2025 -> 2026 DRIVER TRANSITIONS
# ============================================================

transition_rows_32hd = []


drivers_2026_32hd = [
    driver
    for driver, year
    in driver_states_32hd
    if int(year) == 2026
]


for driver in drivers_2026_32hd:

    state_2026 = (
        driver,
        2026,
    )


    state_2025 = (
        driver,
        2025,
    )


    if state_2025 not in driver_state_to_idx_32hd:

        continue


    idx_2025 = driver_state_to_idx_32hd[
        state_2025
    ]


    idx_2026 = driver_state_to_idx_32hd[
        state_2026
    ]


    a25 = alpha_live_32hd[
        :,
        idx_2025,
    ]


    a26 = alpha_live_32hd[
        :,
        idx_2026,
    ]


    delta = (
        a26
        -
        a25
    )


    transition_rows_32hd.append(
        {
            "driver":
                driver,

            "alpha_2025":
                float(
                    np.mean(
                        a25
                    )
                ),

            "alpha_2026":
                float(
                    np.mean(
                        a26
                    )
                ),

            "delta_mean":
                float(
                    np.mean(
                        delta
                    )
                ),

            "delta_sd":
                float(
                    np.std(
                        delta,
                        ddof=1,
                    )
                ),

            "delta_89_lb":
                float(
                    np.quantile(
                        delta,
                        0.055,
                    )
                ),

            "delta_89_ub":
                float(
                    np.quantile(
                        delta,
                        0.945,
                    )
                ),

            "p_2026_gt_2025":
                float(
                    np.mean(
                        delta
                        >
                        0
                    )
                ),

            "corr_2025_2026":
                float(
                    np.corrcoef(
                        a25,
                        a26,
                    )[
                        0,
                        1,
                    ]
                ),
        }
    )


transition_2025_2026_32hd = (
    pd.DataFrame(
        transition_rows_32hd
    )
    .sort_values(
        "delta_mean",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("A. 2025 -> 2026 DRIVER-ABILITY TRANSITIONS")
print("=" * 80)


display(
    transition_2025_2026_32hd
)


new_2026_drivers_32hd = sorted(
    [
        driver
        for driver in drivers_2026_32hd
        if (
            driver,
            2025,
        )
        not in driver_state_to_idx_32hd
    ]
)


print()
print(
    "2026 drivers without a 2025 state:"
)

print(
    new_2026_drivers_32hd
)


# ============================================================
# 3. BUILD 2026 RESULT CONTEXT
# ============================================================

results_32hd = raw.results.copy()
races_32hd = raw.races.copy()


if "year" not in results_32hd.columns:

    results_32hd = results_32hd.merge(
        races_32hd[
            [
                "raceId",
                "year",
            ]
        ],
        on="raceId",
        how="left",
        validate="many_to_one",
    )


results_2026_32hd = (
    results_32hd.loc[
        results_32hd[
            "year"
        ].eq(
            2026
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# driverId -> driverRef
# ------------------------------------------------------------

if not hasattr(
    raw,
    "drivers",
):

    raise RuntimeError(
        "raw.drivers is required."
    )


drivers_meta_32hd = (
    raw.drivers.copy()
)


required_driver_cols_32hd = [
    "driverId",
    "driverRef",
]


missing_driver_cols_32hd = [
    col
    for col
    in required_driver_cols_32hd
    if col not in drivers_meta_32hd.columns
]


if missing_driver_cols_32hd:

    raise RuntimeError(
        "Missing driver metadata columns: "
        +
        str(
            missing_driver_cols_32hd
        )
    )


driver_ref_map_32hd = dict(
    zip(
        drivers_meta_32hd[
            "driverId"
        ],
        drivers_meta_32hd[
            "driverRef"
        ],
    )
)


results_2026_32hd[
    "driver"
] = (
    results_2026_32hd[
        "driverId"
    ]
    .map(
        driver_ref_map_32hd
    )
)


if results_2026_32hd[
    "driver"
].isna().any():

    bad = (
        results_2026_32hd.loc[
            results_2026_32hd[
                "driver"
            ].isna(),
            "driverId",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise RuntimeError(
        "Missing driverRef mappings for: "
        +
        str(
            bad
        )
    )


# ------------------------------------------------------------
# Constructor labels
# ------------------------------------------------------------

constructor_name_map_32hd = {}


if hasattr(
    raw,
    "constructors"
):

    constructors_meta_32hd = (
        raw.constructors.copy()
    )


    if (
        "constructorId"
        in constructors_meta_32hd.columns
        and
        "name"
        in constructors_meta_32hd.columns
    ):

        constructor_name_map_32hd = dict(
            zip(
                constructors_meta_32hd[
                    "constructorId"
                ],
                constructors_meta_32hd[
                    "name"
                ],
            )
        )


# ============================================================
# 4. DIRECT 2026 TEAMMATE PAIRS
# ============================================================
#
# A direct teammate pair means both drivers have a result row
# for the same constructor in the same 2026 GP.
#
# This therefore avoids treating sequential occupants who never
# overlapped as though they had direct head-to-head evidence.
# ============================================================

pair_base_32hd = (
    results_2026_32hd[
        [
            "raceId",
            "constructorId",
            "driver",
        ]
    ]
    .drop_duplicates()
)


pair_merge_32hd = pair_base_32hd.merge(
    pair_base_32hd,
    on=[
        "raceId",
        "constructorId",
    ],
    suffixes=(
        "_a",
        "_b",
    ),
)


pair_merge_32hd = (
    pair_merge_32hd.loc[
        pair_merge_32hd[
            "driver_a"
        ]
        <
        pair_merge_32hd[
            "driver_b"
        ]
    ]
    .copy()
)


pair_counts_32hd = (
    pair_merge_32hd
    .groupby(
        [
            "constructorId",
            "driver_a",
            "driver_b",
        ],
        as_index=False,
    )
    .agg(
        common_races=(
            "raceId",
            "nunique",
        )
    )
)


teammate_rows_32hd = []


for row in pair_counts_32hd.itertuples(
    index=False
):

    state_a = (
        row.driver_a,
        2026,
    )


    state_b = (
        row.driver_b,
        2026,
    )


    if (
        state_a
        not in driver_state_to_idx_32hd
        or
        state_b
        not in driver_state_to_idx_32hd
    ):

        continue


    a = alpha_live_32hd[
        :,
        driver_state_to_idx_32hd[
            state_a
        ],
    ]


    b = alpha_live_32hd[
        :,
        driver_state_to_idx_32hd[
            state_b
        ],
    ]


    delta = (
        a
        -
        b
    )


    teammate_rows_32hd.append(
        {
            "constructor_id":
                row.constructorId,

            "constructor":
                constructor_name_map_32hd.get(
                    row.constructorId,
                    str(
                        row.constructorId
                    ),
                ),

            "driver_a":
                row.driver_a,

            "driver_b":
                row.driver_b,

            "common_races":
                row.common_races,

            "delta_a_minus_b_mean":
                float(
                    np.mean(
                        delta
                    )
                ),

            "delta_89_lb":
                float(
                    np.quantile(
                        delta,
                        0.055,
                    )
                ),

            "delta_89_ub":
                float(
                    np.quantile(
                        delta,
                        0.945,
                    )
                ),

            "p_a_gt_b":
                float(
                    np.mean(
                        delta
                        >
                        0
                    )
                ),

            "alpha_corr":
                float(
                    np.corrcoef(
                        a,
                        b,
                    )[
                        0,
                        1,
                    ]
                ),
        }
    )


teammate_2026_32hd = (
    pd.DataFrame(
        teammate_rows_32hd
    )
    .sort_values(
        [
            "constructor",
            "common_races",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("B. DIRECT 2026 TEAMMATE ALPHA DIFFERENCES")
print("=" * 80)


display(
    teammate_2026_32hd
)


# ============================================================
# 5. 2026 DRIVER/CAR ALLOCATION RIDGE
# ============================================================
#
# For each constructor:
#
#   common_driver =
#       appearance-weighted mean alpha of its 2026 drivers
#
#   beta =
#       constructor-season latent effect
#
# Then inspect:
#
#   corr(common_driver, beta)
#
# and compare uncertainty in:
#
#   common_driver
#   beta
#   common_driver + beta
#
# Strong negative correlation with a more tightly identified
# sum is the structural allocation ridge already seen in the
# frozen model.
# ============================================================

appearance_counts_32hd = (
    results_2026_32hd
    .groupby(
        [
            "constructorId",
            "driver",
        ],
        as_index=False,
    )
    .agg(
        appearances=(
            "raceId",
            "nunique",
        )
    )
)


ridge_rows_32hd = []


for constructor_id, group in (
    appearance_counts_32hd
    .groupby(
        "constructorId"
    )
):

    constructor_state = (
        constructor_id,
        2026,
    )


    if (
        constructor_state
        not in constructor_state_to_idx_32hd
    ):

        continue


    group = group.loc[
        group[
            "driver"
        ].map(
            lambda d:
                (
                    d,
                    2026,
                )
                in driver_state_to_idx_32hd
        )
    ].copy()


    if len(
        group
    ) == 0:

        continue


    weights = np.asarray(
        group[
            "appearances"
        ],
        dtype=float,
    )


    weights = (
        weights
        /
        weights.sum()
    )


    driver_matrix = np.column_stack(
        [
            alpha_live_32hd[
                :,
                driver_state_to_idx_32hd[
                    (
                        driver,
                        2026,
                    )
                ],
            ]
            for driver
            in group[
                "driver"
            ]
        ]
    )


    common_driver = (
        driver_matrix
        @
        weights
    )


    beta = car_live_32hd[
        :,
        constructor_state_to_idx_32hd[
            constructor_state
        ],
    ]


    total_team = (
        common_driver
        +
        beta
    )


    ridge_rows_32hd.append(
        {
            "constructor_id":
                constructor_id,

            "constructor":
                constructor_name_map_32hd.get(
                    constructor_id,
                    str(
                        constructor_id
                    ),
                ),

            "drivers":
                ", ".join(
                    group[
                        "driver"
                    ].tolist()
                ),

            "corr_beta_common_driver":
                float(
                    np.corrcoef(
                        beta,
                        common_driver,
                    )[
                        0,
                        1,
                    ]
                ),

            "sd_beta":
                float(
                    np.std(
                        beta,
                        ddof=1,
                    )
                ),

            "sd_common_driver":
                float(
                    np.std(
                        common_driver,
                        ddof=1,
                    )
                ),

            "sd_beta_plus_driver":
                float(
                    np.std(
                        total_team,
                        ddof=1,
                    )
                ),

            "mean_beta":
                float(
                    np.mean(
                        beta
                    )
                ),

            "mean_common_driver":
                float(
                    np.mean(
                        common_driver
                    )
                ),

            "mean_total_team":
                float(
                    np.mean(
                        total_team
                    )
                ),
        }
    )


ridge_2026_32hd = (
    pd.DataFrame(
        ridge_rows_32hd
    )
    .sort_values(
        "corr_beta_common_driver"
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("C. 2026 DRIVER/CAR ALLOCATION RIDGE")
print("=" * 80)


display(
    ridge_2026_32hd
)


# ============================================================
# 6. LOAD FROZEN 1982–2025 POSTERIOR
# ============================================================

frozen_trace_path_32hd = (
    Path(
        "model_outputs"
    )
    /
    "joint_driver_car_v2_1_1982_2025"
    /
    "joint_driver_car_v2_1_1982_2025_final_nutpie_jax_1000x1000_4chain.nc"
)


print()
print("=" * 80)
print("D. FROZEN 1982–2025 COMPARISON")
print("=" * 80)

print(
    "Frozen checkpoint:"
)

print(
    frozen_trace_path_32hd
)


if not frozen_trace_path_32hd.exists():

    raise RuntimeError(
        "Frozen 1982–2025 checkpoint was not found:\n"
        f"{frozen_trace_path_32hd}"
    )


frozen_idata_32hd = az.from_netcdf(
    frozen_trace_path_32hd
)


alpha_frozen_32hd = np.asarray(
    frozen_idata_32hd.posterior[
        "alpha_dy"
    ].values,
    dtype=float,
)


alpha_frozen_32hd = (
    alpha_frozen_32hd.reshape(
        -1,
        alpha_frozen_32hd.shape[
            -1
        ],
    )
)


car_frozen_32hd = np.asarray(
    frozen_idata_32hd.posterior[
        "car_csy"
    ].values,
    dtype=float,
)


car_frozen_32hd = (
    car_frozen_32hd.reshape(
        -1,
        car_frozen_32hd.shape[
            -1
        ],
    )
)


# ============================================================
# 7. ALIGN PRE-2026 DRIVER STATES
# ============================================================

old_driver_states_32hd = [
    state
    for state
    in driver_states_32hd
    if int(
        state[
            1
        ]
    )
    <=
    2025
]


if len(
    old_driver_states_32hd
) != alpha_frozen_32hd.shape[
    1
]:

    raise RuntimeError(
        "Cannot safely align frozen driver states.\n"
        f"Current <=2025 states: {len(old_driver_states_32hd)}\n"
        f"Frozen alpha states: {alpha_frozen_32hd.shape[1]}"
    )


live_old_driver_indices_32hd = np.array(
    [
        driver_state_to_idx_32hd[
            state
        ]
        for state
        in old_driver_states_32hd
    ],
    dtype=int,
)


alpha_live_old_32hd = (
    alpha_live_32hd[
        :,
        live_old_driver_indices_32hd,
    ]
)


live_alpha_mean_old_32hd = np.mean(
    alpha_live_old_32hd,
    axis=0,
)


frozen_alpha_mean_32hd = np.mean(
    alpha_frozen_32hd,
    axis=0,
)


live_alpha_sd_old_32hd = np.std(
    alpha_live_old_32hd,
    axis=0,
    ddof=1,
)


frozen_alpha_sd_32hd = np.std(
    alpha_frozen_32hd,
    axis=0,
    ddof=1,
)


driver_shift_rows_32hd = []


for i, state in enumerate(
    old_driver_states_32hd
):

    driver, year = state


    shift = (
        live_alpha_mean_old_32hd[
            i
        ]
        -
        frozen_alpha_mean_32hd[
            i
        ]
    )


    combined_sd = np.sqrt(
        live_alpha_sd_old_32hd[
            i
        ]
        ** 2
        +
        frozen_alpha_sd_32hd[
            i
        ]
        ** 2
    )


    driver_shift_rows_32hd.append(
        {
            "driver":
                driver,

            "year":
                int(
                    year
                ),

            "frozen_mean":
                frozen_alpha_mean_32hd[
                    i
                ],

            "live_mean":
                live_alpha_mean_old_32hd[
                    i
                ],

            "shift":
                shift,

            "abs_shift":
                abs(
                    shift
                ),

            "frozen_sd":
                frozen_alpha_sd_32hd[
                    i
                ],

            "live_sd":
                live_alpha_sd_old_32hd[
                    i
                ],

            "shift_over_combined_sd":
                (
                    shift
                    /
                    combined_sd
                    if combined_sd
                    >
                    0
                    else np.nan
                ),
        }
    )


driver_shift_32hd = pd.DataFrame(
    driver_shift_rows_32hd
)


driver_pearson_32hd = float(
    driver_shift_32hd[
        [
            "frozen_mean",
            "live_mean",
        ]
    ]
    .corr(
        method="pearson"
    )
    .iloc[
        0,
        1,
    ]
)


driver_spearman_32hd = float(
    driver_shift_32hd[
        [
            "frozen_mean",
            "live_mean",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1,
    ]
)


driver_mae_32hd = float(
    driver_shift_32hd[
        "abs_shift"
    ].mean()
)


driver_rmse_32hd = float(
    np.sqrt(
        np.mean(
            driver_shift_32hd[
                "shift"
            ]
            ** 2
        )
    )
)


driver_max_shift_32hd = float(
    driver_shift_32hd[
        "abs_shift"
    ].max()
)


print()
print(
    "Driver-season frozen vs LIVE:"
)

print(
    "  n states:",
    len(
        driver_shift_32hd
    ),
)

print(
    "  Pearson:",
    f"{driver_pearson_32hd:.6f}",
)

print(
    "  Spearman:",
    f"{driver_spearman_32hd:.6f}",
)

print(
    "  MAE:",
    f"{driver_mae_32hd:.6f}",
)

print(
    "  RMSE:",
    f"{driver_rmse_32hd:.6f}",
)

print(
    "  max |shift|:",
    f"{driver_max_shift_32hd:.6f}",
)


print()
print(
    "Largest absolute driver-season shifts:"
)


display(
    driver_shift_32hd
    .sort_values(
        "abs_shift",
        ascending=False,
    )
    .head(
        25
    )
)


# ============================================================
# 8. 2025 STATES SPECIFICALLY
# ============================================================

driver_shift_2025_32hd = (
    driver_shift_32hd.loc[
        driver_shift_32hd[
            "year"
        ].eq(
            2025
        )
    ]
    .sort_values(
        "abs_shift",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("FROZEN -> LIVE SHIFTS FOR 2025 DRIVER STATES")
print("=" * 80)


display(
    driver_shift_2025_32hd
)


# ============================================================
# 9. HISTORICAL SHIFT BY YEAR
# ============================================================
#
# This helps distinguish:
#
#   - concentrated movement near 2025, which could naturally
#     arise from adding 2026 through temporal smoothing;
#
# versus
#
#   - broader changes deep in history, which would be more
#     suggestive of the global familiarity redesign.
#
# It still does NOT formally isolate causation.
# ============================================================

driver_shift_by_year_32hd = (
    driver_shift_32hd
    .groupby(
        "year",
        as_index=False,
    )
    .agg(
        n_states=(
            "driver",
            "size",
        ),

        mean_shift=(
            "shift",
            "mean",
        ),

        mean_abs_shift=(
            "abs_shift",
            "mean",
        ),

        max_abs_shift=(
            "abs_shift",
            "max",
        ),
    )
)


print()
print("=" * 80)
print("DRIVER-STATE SHIFT BY YEAR")
print("=" * 80)


display(
    driver_shift_by_year_32hd.tail(
        15
    )
)


# ============================================================
# 10. ALIGN PRE-2026 CONSTRUCTOR STATES
# ============================================================

old_constructor_states_32hd = [
    state
    for state
    in constructor_states_32hd
    if int(
        state[
            1
        ]
    )
    <=
    2025
]


if len(
    old_constructor_states_32hd
) != car_frozen_32hd.shape[
    1
]:

    raise RuntimeError(
        "Cannot safely align frozen constructor states.\n"
        f"Current <=2025 states: {len(old_constructor_states_32hd)}\n"
        f"Frozen car states: {car_frozen_32hd.shape[1]}"
    )


live_old_car_indices_32hd = np.array(
    [
        constructor_state_to_idx_32hd[
            state
        ]
        for state
        in old_constructor_states_32hd
    ],
    dtype=int,
)


car_live_old_32hd = (
    car_live_32hd[
        :,
        live_old_car_indices_32hd,
    ]
)


live_car_mean_old_32hd = np.mean(
    car_live_old_32hd,
    axis=0,
)


frozen_car_mean_32hd = np.mean(
    car_frozen_32hd,
    axis=0,
)


live_car_sd_old_32hd = np.std(
    car_live_old_32hd,
    axis=0,
    ddof=1,
)


frozen_car_sd_32hd = np.std(
    car_frozen_32hd,
    axis=0,
    ddof=1,
)


car_shift_rows_32hd = []


for i, state in enumerate(
    old_constructor_states_32hd
):

    constructor_id, year = state


    shift = (
        live_car_mean_old_32hd[
            i
        ]
        -
        frozen_car_mean_32hd[
            i
        ]
    )


    combined_sd = np.sqrt(
        live_car_sd_old_32hd[
            i
        ]
        ** 2
        +
        frozen_car_sd_32hd[
            i
        ]
        ** 2
    )


    car_shift_rows_32hd.append(
        {
            "constructor_id":
                constructor_id,

            "constructor":
                constructor_name_map_32hd.get(
                    constructor_id,
                    str(
                        constructor_id
                    ),
                ),

            "year":
                int(
                    year
                ),

            "frozen_mean":
                frozen_car_mean_32hd[
                    i
                ],

            "live_mean":
                live_car_mean_old_32hd[
                    i
                ],

            "shift":
                shift,

            "abs_shift":
                abs(
                    shift
                ),

            "frozen_sd":
                frozen_car_sd_32hd[
                    i
                ],

            "live_sd":
                live_car_sd_old_32hd[
                    i
                ],

            "shift_over_combined_sd":
                (
                    shift
                    /
                    combined_sd
                    if combined_sd
                    >
                    0
                    else np.nan
                ),
        }
    )


car_shift_32hd = pd.DataFrame(
    car_shift_rows_32hd
)


car_pearson_32hd = float(
    car_shift_32hd[
        [
            "frozen_mean",
            "live_mean",
        ]
    ]
    .corr(
        method="pearson"
    )
    .iloc[
        0,
        1,
    ]
)


car_spearman_32hd = float(
    car_shift_32hd[
        [
            "frozen_mean",
            "live_mean",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1,
    ]
)


car_mae_32hd = float(
    car_shift_32hd[
        "abs_shift"
    ].mean()
)


car_rmse_32hd = float(
    np.sqrt(
        np.mean(
            car_shift_32hd[
                "shift"
            ]
            ** 2
        )
    )
)


car_max_shift_32hd = float(
    car_shift_32hd[
        "abs_shift"
    ].max()
)


print()
print("=" * 80)
print("E. CONSTRUCTOR-SEASON FROZEN VS LIVE")
print("=" * 80)


print(
    "n states:",
    len(
        car_shift_32hd
    ),
)

print(
    "Pearson:",
    f"{car_pearson_32hd:.6f}",
)

print(
    "Spearman:",
    f"{car_spearman_32hd:.6f}",
)

print(
    "MAE:",
    f"{car_mae_32hd:.6f}",
)

print(
    "RMSE:",
    f"{car_rmse_32hd:.6f}",
)

print(
    "max |shift|:",
    f"{car_max_shift_32hd:.6f}",
)


print()
print(
    "Largest constructor-season shifts:"
)


display(
    car_shift_32hd
    .sort_values(
        "abs_shift",
        ascending=False,
    )
    .head(
        20
    )
)


# ============================================================
# 11. 2025 CONSTRUCTOR STATES
# ============================================================

car_shift_2025_32hd = (
    car_shift_32hd.loc[
        car_shift_32hd[
            "year"
        ].eq(
            2025
        )
    ]
    .sort_values(
        "abs_shift",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("FROZEN -> LIVE SHIFTS FOR 2025 CONSTRUCTOR STATES")
print("=" * 80)


display(
    car_shift_2025_32hd
)


# ============================================================
# 12. COMPACT SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-D SUMMARY")
print("=" * 80)


print()
print("2026 posterior:")
print(
    "  driver states:",
    len(
        drivers_2026_32hd
    ),
)

print(
    "  direct teammate pairs:",
    len(
        teammate_2026_32hd
    ),
)

print(
    "  constructor allocation audits:",
    len(
        ridge_2026_32hd
    ),
)


print()
print("Frozen -> LIVE driver-season stability:")
print(
    "  Pearson:",
    f"{driver_pearson_32hd:.6f}",
)

print(
    "  Spearman:",
    f"{driver_spearman_32hd:.6f}",
)

print(
    "  MAE:",
    f"{driver_mae_32hd:.6f}",
)

print(
    "  RMSE:",
    f"{driver_rmse_32hd:.6f}",
)

print(
    "  max |shift|:",
    f"{driver_max_shift_32hd:.6f}",
)


print()
print("Frozen -> LIVE constructor-season stability:")
print(
    "  Pearson:",
    f"{car_pearson_32hd:.6f}",
)

print(
    "  Spearman:",
    f"{car_spearman_32hd:.6f}",
)

print(
    "  MAE:",
    f"{car_mae_32hd:.6f}",
)

print(
    "  RMSE:",
    f"{car_rmse_32hd:.6f}",
)

print(
    "  max |shift|:",
    f"{car_max_shift_32hd:.6f}",
)


print()
print(
    "IMPORTANT:"
)

print(
    "Frozen -> LIVE shifts combine the familiarity redesign "
    "AND the addition of 2026 evidence."
)

print(
    "They are not a pure causal estimate of the familiarity "
    "change."
)

print()
print(
    "NO LIVE JAWS HAS BEEN CALCULATED."
)

In [ ]:
# ============================================================
# STAGE 32H-E — FROZEN-PANTHEON SCORE STABILITY AUDIT
# ============================================================
#
# PURPOSE
# -------
# Apply the EXACT locked 1982–2025 Pantheon / JAWS definition
# to the NEW LIVE posterior, whilst deliberately excluding
# 2026 from:
#
#   - modelled seasons
#   - peak
#   - career surplus / longevity
#   - actual-start eligibility
#   - teammate eligibility
#
# Then compare that counterfactual 1982–2025 leaderboard with
# the frozen official 1982–2025 leaderboard.
#
# This answers:
#
#   "Did the familiarity redesign + backward influence of
#    2026 evidence materially change the historical Pantheon?"
#
# IMPORTANT
# ---------
# This is NOT the LIVE 1982–2026 leaderboard.
# 2026 contributes zero score here.
#
# Nothing is overwritten in the frozen model directory.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. LOCKED PANTHEON CONSTANTS
# ============================================================

AUDIT_START_YEAR_32HE = 1982
AUDIT_END_YEAR_32HE = 2025

AUDIT_MIN_STARTS_32HE = 35
AUDIT_MIN_TEAMMATES_32HE = 3
AUDIT_MIN_SEASONS_32HE = 5

AUDIT_PEAK_WEIGHT_32HE = 0.60
AUDIT_LONGEVITY_WEIGHT_32HE = 0.40


print("=" * 80)
print("STAGE 32H-E — HISTORICAL PANTHEON STABILITY AUDIT")
print("=" * 80)

print()
print("Scoring window: 1982–2025 ONLY")
print("2026 contribution: NONE")


# ============================================================
# 1. LIVE POSTERIOR ALPHA MATRIX
# ============================================================

if "posterior_32hc" not in globals():

    raise RuntimeError(
        "posterior_32hc is missing. Run Stage 32H-C first."
    )


alpha_live_32he = np.asarray(
    posterior_32hc[
        "alpha_dy"
    ].values,
    dtype=float,
)


alpha_live_32he = alpha_live_32he.reshape(
    -1,
    alpha_live_32he.shape[-1],
)


n_draws_32he = int(
    alpha_live_32he.shape[0]
)


driver_states_32he = list(
    idx_all.active_driver_years
)


if alpha_live_32he.shape[1] != len(
    driver_states_32he
):

    raise RuntimeError(
        "alpha_dy state count does not match "
        "idx_all.active_driver_years."
    )


if n_draws_32he != 4000:

    raise RuntimeError(
        f"Expected 4000 posterior draws, found {n_draws_32he}."
    )


driver_state_to_idx_32he = {
    (
        driver,
        int(year),
    ): i

    for i, (
        driver,
        year,
    )
    in enumerate(
        driver_states_32he
    )
}


all_driver_ids_32he = sorted(
    {
        driver
        for driver, year
        in driver_states_32he
    }
)


print()
print(
    "LIVE posterior draws:",
    n_draws_32he,
)

print(
    "Total LIVE driver-season states:",
    len(
        driver_states_32he
    ),
)


# ============================================================
# 2. DRIVER -> MODELLED SEASONS, TRUNCATED AT 2025
# ============================================================

driver_years_2025_32he = {}


for driver, year in driver_states_32he:

    year = int(
        year
    )


    if not (
        AUDIT_START_YEAR_32HE
        <= year
        <= AUDIT_END_YEAR_32HE
    ):

        continue


    driver_years_2025_32he.setdefault(
        driver,
        [],
    ).append(
        year
    )


for driver in driver_years_2025_32he:

    driver_years_2025_32he[
        driver
    ] = sorted(
        set(
            driver_years_2025_32he[
                driver
            ]
        )
    )


scored_drivers_32he = [
    driver

    for driver
    in all_driver_ids_32he

    if len(
        driver_years_2025_32he.get(
            driver,
            [],
        )
    )
    >= AUDIT_MIN_SEASONS_32HE
]


print()
print(
    "Drivers in >=5-season normalisation pool:",
    len(
        scored_drivers_32he
    ),
)


# ============================================================
# 3. EXACT FROZEN ACTUAL-START DEFINITION
# ============================================================

career_results_32he = (
    raw.results.copy()
)


if "year" not in career_results_32he.columns:

    career_results_32he = career_results_32he.merge(

        raw.races[
            [
                "raceId",
                "year",
            ]
        ],

        on="raceId",
        how="left",
        validate="many_to_one",
    )


career_results_32he = (
    career_results_32he.loc[
        career_results_32he[
            "year"
        ].between(
            AUDIT_START_YEAR_32HE,
            AUDIT_END_YEAR_32HE,
        )
    ]
    .copy()
)


career_results_32he[
    "driver"
] = (
    career_results_32he[
        "driverId"
    ]
    .map(
        raw.driver_ref
    )
)


career_results_32he[
    "status_str"
] = (
    career_results_32he[
        "statusId"
    ]
    .map(
        raw.status_map
    )
    .fillna("")
    .astype(str)
    .str.lower()
)


NON_START_PATTERN_32HE = (
    r"did not start"
    r"|did not qualify"
    r"|did not prequalify"
    r"|withdrawn"
    r"|withdrew"
    r"|injured"
    r"|illness"
    r"|unwell"
    r"|not permitted to start"
)


career_results_32he[
    "is_actual_start"
] = ~(
    career_results_32he[
        "status_str"
    ]
    .str.contains(
        NON_START_PATTERN_32HE,
        regex=True,
        na=False,
    )
)


actual_starts_32he = (
    career_results_32he.loc[
        career_results_32he[
            "is_actual_start"
        ],
        [
            "raceId",
            "year",
            "driverId",
            "driver",
            "constructorId",
            "status_str",
        ],
    ]
    .drop_duplicates(
        subset=[
            "raceId",
            "driverId",
            "constructorId",
        ]
    )
    .copy()
)


starts_lookup_32he = (
    actual_starts_32he
    .groupby(
        "driver"
    )[
        "raceId"
    ]
    .nunique()
    .to_dict()
)


print()
print(
    "1982–2025 actual-start rows:",
    len(
        actual_starts_32he
    ),
)


# ============================================================
# 4. EXACT FROZEN UNIQUE-TEAMMATE DEFINITION
# ============================================================
#
# A teammate is another driver who actually started for the
# same constructor in the same GP.
# ============================================================

teammate_sets_32he = {
    driver: set()
    for driver
    in all_driver_ids_32he
}


for (
    race_id,
    constructor_id,
), grp in actual_starts_32he.groupby(
    [
        "raceId",
        "constructorId",
    ]
):

    drivers_here = (
        grp[
            "driver"
        ]
        .dropna()
        .unique()
        .tolist()
    )


    for driver in drivers_here:

        teammate_sets_32he.setdefault(
            driver,
            set(),
        )


        teammate_sets_32he[
            driver
        ].update(
            other
            for other
            in drivers_here
            if other != driver
        )


teammates_lookup_32he = {
    driver:
        len(
            teammate_sets_32he.get(
                driver,
                set(),
            )
        )

    for driver
    in all_driver_ids_32he
}


# ============================================================
# 5. LOCKED 1982–2025 ELIGIBILITY
# ============================================================

eligible_drivers_32he = [
    driver

    for driver
    in scored_drivers_32he

    if (
        starts_lookup_32he.get(
            driver,
            0,
        )
        >= AUDIT_MIN_STARTS_32HE
    )

    and

    (
        teammates_lookup_32he.get(
            driver,
            0,
        )
        >= AUDIT_MIN_TEAMMATES_32HE
    )
]


print()
print(
    "Pantheon-eligible through 2025:",
    len(
        eligible_drivers_32he
    ),
)


# ============================================================
# 6. LOAD OFFICIAL FROZEN 1982–2025 LEADERBOARD
# ============================================================

official_csv_32he = (
    Path(
        "model_outputs"
    )
    /
    "joint_driver_car_v2_1_1982_2025"
    /
    "pantheon_jaws_1982_2025_FINAL.csv"
)


if not official_csv_32he.exists():

    raise RuntimeError(
        "Official frozen Pantheon CSV not found:\n"
        f"{official_csv_32he}"
    )


official_32he = pd.read_csv(
    official_csv_32he
)


if "driver" not in official_32he.columns:

    raise RuntimeError(
        "Official Pantheon CSV has no 'driver' column."
    )


official_driver_set_32he = set(
    official_32he[
        "driver"
    ].astype(str)
)


audit_driver_set_32he = set(
    eligible_drivers_32he
)


missing_from_audit_32he = sorted(
    official_driver_set_32he
    -
    audit_driver_set_32he
)


extra_in_audit_32he = sorted(
    audit_driver_set_32he
    -
    official_driver_set_32he
)


print()
print(
    "Official frozen eligible drivers:",
    len(
        official_driver_set_32he
    ),
)

print(
    "Eligibility missing from audit:",
    missing_from_audit_32he,
)

print(
    "Eligibility extra in audit:",
    extra_in_audit_32he,
)


if (
    missing_from_audit_32he
    or
    extra_in_audit_32he
):

    raise RuntimeError(
        "1982–2025 eligibility does not reproduce the "
        "official frozen set. Stop before scoring."
    )


print()
print(
    "Eligibility reproduction: EXACT"
)


# ============================================================
# 7. RAW JAWS COMPONENTS — LIVE POSTERIOR, <=2025 ONLY
# ============================================================

n_scored_32he = len(
    scored_drivers_32he
)


surplus_mat_32he = np.zeros(
    (
        n_draws_32he,
        n_scored_32he,
    ),
    dtype=float,
)


top5_mat_32he = np.zeros_like(
    surplus_mat_32he
)


season_count_lookup_32he = {}


for j, driver in enumerate(
    scored_drivers_32he
):

    years = (
        driver_years_2025_32he[
            driver
        ]
    )


    season_count_lookup_32he[
        driver
    ] = len(
        years
    )


    dy_idx = np.array(
        [
            driver_state_to_idx_32he[
                (
                    driver,
                    year,
                )
            ]

            for year
            in years
        ],
        dtype=int,
    )


    states = (
        alpha_live_32he[
            :,
            dy_idx,
        ]
    )


    # --------------------------------------------------------
    # Career positive surplus
    # --------------------------------------------------------

    surplus_mat_32he[
        :,
        j,
    ] = (
        np.maximum(
            states,
            0.0,
        )
        .sum(
            axis=1
        )
    )


    # --------------------------------------------------------
    # Literal best-five-season peak, selected per draw
    # --------------------------------------------------------

    top5_mat_32he[
        :,
        j,
    ] = (
        np.sort(
            states,
            axis=1,
        )[
            :,
            -5:,
        ]
        .mean(
            axis=1
        )
    )


# ============================================================
# 8. EXACT DRAW-WISE NORMALISATION
# ============================================================

damped_surplus_32he = np.sqrt(
    surplus_mat_32he
)


peak_denominator_32he = (
    top5_mat_32he.max(
        axis=1,
        keepdims=True,
    )
)


longevity_denominator_32he = (
    damped_surplus_32he.max(
        axis=1,
        keepdims=True,
    )
)


if np.any(
    peak_denominator_32he
    <= 0
):

    raise RuntimeError(
        "Non-positive peak normalisation denominator."
    )


if np.any(
    longevity_denominator_32he
    <= 0
):

    raise RuntimeError(
        "Non-positive longevity normalisation denominator."
    )


peak_score_mat_32he = (
    top5_mat_32he
    /
    peak_denominator_32he
    *
    100.0
)


longevity_score_mat_32he = (
    damped_surplus_32he
    /
    longevity_denominator_32he
    *
    100.0
)


jaws_mat_32he = (
    AUDIT_PEAK_WEIGHT_32HE
    *
    peak_score_mat_32he

    +

    AUDIT_LONGEVITY_WEIGHT_32HE
    *
    longevity_score_mat_32he
)


# ============================================================
# 9. APPLY LOCKED ELIGIBILITY
# ============================================================

eligible_indices_32he = np.array(
    [
        scored_drivers_32he.index(
            driver
        )

        for driver
        in eligible_drivers_32he
    ],
    dtype=int,
)


eligible_jaws_32he = (
    jaws_mat_32he[
        :,
        eligible_indices_32he,
    ]
)


# ============================================================
# 10. POSTERIOR RANKS
# ============================================================

order_32he = np.argsort(
    -eligible_jaws_32he,
    axis=1,
)


rank_mat_32he = np.empty_like(
    order_32he,
    dtype=int,
)


row_ids_32he = np.arange(
    n_draws_32he
)[
    :,
    None,
]


rank_mat_32he[
    row_ids_32he,
    order_32he,
] = np.arange(
    1,
    len(
        eligible_drivers_32he
    )
    +
    1,
)[
    None,
    :
]


# ============================================================
# 11. BUILD COUNTERFACTUAL <=2025 LIVE-POSTERIOR LEADERBOARD
# ============================================================

audit_rows_32he = []


for eligible_j, driver in enumerate(
    eligible_drivers_32he
):

    scored_j = eligible_indices_32he[
        eligible_j
    ]


    jaws = eligible_jaws_32he[
        :,
        eligible_j,
    ]


    ranks = rank_mat_32he[
        :,
        eligible_j,
    ]


    audit_rows_32he.append(
        {
            "driver":
                driver,

            "jaws_mean_livefit_2025":
                float(
                    np.mean(
                        jaws
                    )
                ),

            "jaws_median_livefit_2025":
                float(
                    np.median(
                        jaws
                    )
                ),

            "jaws_89_lb_livefit_2025":
                float(
                    np.quantile(
                        jaws,
                        0.055,
                    )
                ),

            "jaws_89_ub_livefit_2025":
                float(
                    np.quantile(
                        jaws,
                        0.945,
                    )
                ),

            "peak_raw_livefit_2025":
                float(
                    top5_mat_32he[
                        :,
                        scored_j,
                    ].mean()
                ),

            "surplus_livefit_2025":
                float(
                    surplus_mat_32he[
                        :,
                        scored_j,
                    ].mean()
                ),

            "peak_score_livefit_2025":
                float(
                    peak_score_mat_32he[
                        :,
                        scored_j,
                    ].mean()
                ),

            "longevity_score_livefit_2025":
                float(
                    longevity_score_mat_32he[
                        :,
                        scored_j,
                    ].mean()
                ),

            "rank_mean_livefit_2025":
                float(
                    ranks.mean()
                ),

            "rank_median_livefit_2025":
                float(
                    np.median(
                        ranks
                    )
                ),

            "rank_89_lb_livefit_2025":
                float(
                    np.quantile(
                        ranks,
                        0.055,
                    )
                ),

            "rank_89_ub_livefit_2025":
                float(
                    np.quantile(
                        ranks,
                        0.945,
                    )
                ),

            "seasons":
                int(
                    season_count_lookup_32he[
                        driver
                    ]
                ),

            "starts":
                int(
                    starts_lookup_32he.get(
                        driver,
                        0,
                    )
                ),

            "unique_teammates":
                int(
                    teammates_lookup_32he.get(
                        driver,
                        0,
                    )
                ),
        }
    )


audit_board_32he = pd.DataFrame(
    audit_rows_32he
)


audit_board_32he = (
    audit_board_32he
    .sort_values(
        "jaws_mean_livefit_2025",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


audit_board_32he[
    "livefit_point_rank"
] = (
    np.arange(
        1,
        len(
            audit_board_32he
        )
        +
        1,
    )
)


# ============================================================
# 12. PREPARE OFFICIAL FROZEN POINT RANKS
# ============================================================

required_official_cols_32he = [
    "driver",
    "jaws_mean",
]


missing_cols_32he = [
    col
    for col
    in required_official_cols_32he
    if col not in official_32he.columns
]


if missing_cols_32he:

    raise RuntimeError(
        "Official CSV missing required columns: "
        +
        str(
            missing_cols_32he
        )
    )


official_ranked_32he = (
    official_32he
    .sort_values(
        "jaws_mean",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
    .copy()
)


official_ranked_32he[
    "official_point_rank"
] = np.arange(
    1,
    len(
        official_ranked_32he
    )
    +
    1,
)


official_keep_32he = [
    "driver",
    "official_point_rank",
    "jaws_mean",
]


optional_official_cols_32he = [
    "top5_peak_alpha_mean",
    "career_surplus_mean",
    "peak_score_mean",
    "longevity_score_mean",
    "rank_mean",
]


for col in optional_official_cols_32he:

    if col in official_ranked_32he.columns:

        official_keep_32he.append(
            col
        )


official_compare_32he = (
    official_ranked_32he[
        official_keep_32he
    ]
    .copy()
)


rename_map_32he = {
    "jaws_mean":
        "jaws_mean_frozen",

    "top5_peak_alpha_mean":
        "peak_raw_frozen",

    "career_surplus_mean":
        "surplus_frozen",

    "peak_score_mean":
        "peak_score_frozen",

    "longevity_score_mean":
        "longevity_score_frozen",

    "rank_mean":
        "rank_mean_frozen",
}


official_compare_32he = (
    official_compare_32he.rename(
        columns=rename_map_32he
    )
)


# ============================================================
# 13. MERGE + STABILITY METRICS
# ============================================================

score_stability_32he = (
    audit_board_32he
    .merge(
        official_compare_32he,
        on="driver",
        how="inner",
        validate="one_to_one",
    )
)


score_stability_32he[
    "jaws_shift"
] = (
    score_stability_32he[
        "jaws_mean_livefit_2025"
    ]
    -
    score_stability_32he[
        "jaws_mean_frozen"
    ]
)


score_stability_32he[
    "abs_jaws_shift"
] = np.abs(
    score_stability_32he[
        "jaws_shift"
    ]
)


# Positive = moved UP the leaderboard in the LIVE refit.

score_stability_32he[
    "rank_change"
] = (
    score_stability_32he[
        "official_point_rank"
    ]
    -
    score_stability_32he[
        "livefit_point_rank"
    ]
)


score_stability_32he[
    "abs_rank_change"
] = np.abs(
    score_stability_32he[
        "rank_change"
    ]
)


if (
    "peak_score_frozen"
    in score_stability_32he.columns
):

    score_stability_32he[
        "peak_score_shift"
    ] = (
        score_stability_32he[
            "peak_score_livefit_2025"
        ]
        -
        score_stability_32he[
            "peak_score_frozen"
        ]
    )


if (
    "longevity_score_frozen"
    in score_stability_32he.columns
):

    score_stability_32he[
        "longevity_score_shift"
    ] = (
        score_stability_32he[
            "longevity_score_livefit_2025"
        ]
        -
        score_stability_32he[
            "longevity_score_frozen"
        ]
    )


pearson_jaws_32he = float(
    score_stability_32he[
        [
            "jaws_mean_frozen",
            "jaws_mean_livefit_2025",
        ]
    ]
    .corr(
        method="pearson"
    )
    .iloc[
        0,
        1,
    ]
)


spearman_jaws_32he = float(
    score_stability_32he[
        [
            "jaws_mean_frozen",
            "jaws_mean_livefit_2025",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1,
    ]
)


mae_jaws_32he = float(
    score_stability_32he[
        "abs_jaws_shift"
    ].mean()
)


rmse_jaws_32he = float(
    np.sqrt(
        np.mean(
            score_stability_32he[
                "jaws_shift"
            ]
            ** 2
        )
    )
)


max_score_shift_32he = float(
    score_stability_32he[
        "abs_jaws_shift"
    ].max()
)


max_rank_move_32he = int(
    score_stability_32he[
        "abs_rank_change"
    ].max()
)


# ============================================================
# 14. TOP-N STABILITY
# ============================================================

official_top10_32he = (
    official_ranked_32he[
        "driver"
    ]
    .head(
        10
    )
    .tolist()
)


audit_top10_32he = (
    audit_board_32he[
        "driver"
    ]
    .head(
        10
    )
    .tolist()
)


official_top30_32he = (
    official_ranked_32he[
        "driver"
    ]
    .head(
        30
    )
    .tolist()
)


audit_top30_32he = (
    audit_board_32he[
        "driver"
    ]
    .head(
        30
    )
    .tolist()
)


top10_set_same_32he = (
    set(
        official_top10_32he
    )
    ==
    set(
        audit_top10_32he
    )
)


top10_order_same_32he = (
    official_top10_32he
    ==
    audit_top10_32he
)


top30_set_same_32he = (
    set(
        official_top30_32he
    )
    ==
    set(
        audit_top30_32he
    )
)


top30_order_same_32he = (
    official_top30_32he
    ==
    audit_top30_32he
)


# ============================================================
# 15. OUTPUT
# ============================================================

print()
print("=" * 80)
print("FROZEN PANTHEON SCORE STABILITY")
print("=" * 80)


print(
    "Eligibility set identical:",
    True,
)


print(
    "Normalisation pool:",
    len(
        scored_drivers_32he
    ),
)


print(
    "Pantheon eligible:",
    len(
        eligible_drivers_32he
    ),
)


print()
print(
    "JAWS Pearson:",
    f"{pearson_jaws_32he:.6f}",
)

print(
    "JAWS Spearman:",
    f"{spearman_jaws_32he:.6f}",
)

print(
    "Mean absolute score shift:",
    f"{mae_jaws_32he:.6f}",
)

print(
    "RMSE score shift:",
    f"{rmse_jaws_32he:.6f}",
)

print(
    "Maximum absolute score shift:",
    f"{max_score_shift_32he:.6f}",
)

print(
    "Maximum point-rank movement:",
    max_rank_move_32he,
)


print()
print(
    "Top-10 set identical:",
    top10_set_same_32he,
)

print(
    "Top-10 exact order identical:",
    top10_order_same_32he,
)

print(
    "Top-30 set identical:",
    top30_set_same_32he,
)

print(
    "Top-30 exact order identical:",
    top30_order_same_32he,
)


# ------------------------------------------------------------
# Top 30 comparison
# ------------------------------------------------------------

print()
print("=" * 80)
print("TOP 30 — FROZEN VS LIVE-REFIT, BOTH SCORED THROUGH 2025")
print("=" * 80)


top30_compare_32he = (
    score_stability_32he
    .sort_values(
        "livefit_point_rank"
    )
    .head(
        30
    )
    [
        [
            "driver",
            "official_point_rank",
            "livefit_point_rank",
            "rank_change",
            "jaws_mean_frozen",
            "jaws_mean_livefit_2025",
            "jaws_shift",
        ]
    ]
)


display(
    top30_compare_32he
)


# ------------------------------------------------------------
# Biggest movers anywhere
# ------------------------------------------------------------

print()
print("=" * 80)
print("LARGEST SCORE SHIFTS")
print("=" * 80)


shift_cols_32he = [
    "driver",
    "official_point_rank",
    "livefit_point_rank",
    "rank_change",
    "jaws_mean_frozen",
    "jaws_mean_livefit_2025",
    "jaws_shift",
    "abs_jaws_shift",
]


if "peak_score_shift" in score_stability_32he.columns:

    shift_cols_32he.append(
        "peak_score_shift"
    )


if "longevity_score_shift" in score_stability_32he.columns:

    shift_cols_32he.append(
        "longevity_score_shift"
    )


display(
    score_stability_32he
    .sort_values(
        "abs_jaws_shift",
        ascending=False,
    )
    .head(
        25
    )[
        shift_cols_32he
    ]
)


print()
print("=" * 80)
print("LARGEST POINT-RANK MOVES")
print("=" * 80)


display(
    score_stability_32he
    .sort_values(
        [
            "abs_rank_change",
            "abs_jaws_shift",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(
        25
    )[
        shift_cols_32he
    ]
)


# ============================================================
# 16. SAVE AUDIT ONLY — NEVER OVERWRITE OFFICIAL FILES
# ============================================================

audit_output_dir_32he = Path(
    CFG.output_dir
)


audit_output_dir_32he.mkdir(
    parents=True,
    exist_ok=True,
)


audit_path_32he = (
    audit_output_dir_32he
    /
    "historical_pantheon_stability_livefit_scored_through_2025.csv"
)


score_stability_32he.to_csv(
    audit_path_32he,
    index=False,
)


print()
print(
    "Saved audit ->"
)

print(
    audit_path_32he
)


print()
print("=" * 80)
print("STAGE 32H-E COMPLETE")
print("=" * 80)

print()
print(
    "2026 has NOT been included in peak, longevity, "
    "normalisation eligibility, starts or teammate counts."
)

print(
    "This is a stability audit, not the LIVE 2026 Pantheon."
)

## Superseded Stage 32H-K validation recovery


In [ ]:
# ============================================================
# STAGE 32H-K — VALIDATION FIX / RECOVERY
# ============================================================
#
# The dashboard was already successfully generated.
#
# The original validation incorrectly searched for:
#
#     "Driver Explorer"
#
# whilst the dashboard uses:
#
#     "Driver explorer"
#
# This reruns ONLY the final HTML validation and summary.
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# 1. CONFIRM GENERATED DASHBOARD EXISTS
# ============================================================

if "generated_path_32hk" not in globals():

    raise RuntimeError(
        "generated_path_32hk is not in memory. "
        "Stage 32H-K must have reached the dashboard-build "
        "stage before running this recovery cell."
    )


generated_path_32hk = Path(
    generated_path_32hk
)


if not generated_path_32hk.exists():

    raise RuntimeError(
        "Generated dashboard does not exist:\n"
        f"{generated_path_32hk}"
    )


html_32hk = generated_path_32hk.read_text(
    encoding="utf-8"
)


# ============================================================
# 2. CORRECTED HTML VALIDATION
# ============================================================

html_checks_32hk = {

    # --------------------------------------------------------
    # 95% intervals
    # --------------------------------------------------------

    "95% JAWS interval":
        (
            "95% JAWS interval"
            in html_32hk
        ),

    "95% current driver alpha":
        (
            "95% α interval"
            in html_32hk
        ),

    "95% current team beta":
        (
            "95% β interval"
            in html_32hk
        ),

    "95% posterior labels":
        (
            "95%"
            in html_32hk
        ),

    "no visible 89% labels":
        (
            "89%"
            not in html_32hk
        ),

    "2.5th percentile":
        (
            "2.5th percentile"
            in html_32hk
        ),

    "97.5th percentile":
        (
            "97.5th percentile"
            in html_32hk
        ),


    # --------------------------------------------------------
    # Career value terminology
    # --------------------------------------------------------

    "Career value leaderboard":
        (
            "Career value"
            in html_32hk
        ),

    "Career value score option":
        (
            "Career value score"
            in html_32hk
        ),

    "Peak vs career value":
        (
            "Peak vs career value"
            in html_32hk
        ),

    "40% career value":
        (
            "40% career value"
            in html_32hk
        ),

    "career value plot axis":
        (
            "Career value score"
            in html_32hk
        ),


    # --------------------------------------------------------
    # Existing features preserved
    # --------------------------------------------------------

    "Driver current tab":
        (
            'id="current-driver-tab"'
            in html_32hk
        ),

    "Team current tab":
        (
            'id="current-team-tab"'
            in html_32hk
        ),

    # FIX:
    # make this case-insensitive so display capitalisation
    # cannot trigger a false failure.
    "Driver explorer":
        (
            "driver explorer"
            in html_32hk.lower()
        ),

    "Teammate network":
        (
            'id="teammate-network"'
            in html_32hk
        ),

    "Teammate Chain":
        (
            "Teammate Chain"
            in html_32hk
        ),
}


validation_32hk = pd.DataFrame(
    [
        {
            "check": check,
            "passed": passed,
        }

        for check, passed
        in html_checks_32hk.items()
    ]
)


print("=" * 80)
print("STAGE 32H-K — CORRECTED HTML VALIDATION")
print("=" * 80)

display(
    validation_32hk
)


if not all(
    html_checks_32hk.values()
):

    failed_32hk = [

        check

        for check, passed
        in html_checks_32hk.items()

        if not passed
    ]

    raise RuntimeError(
        "95%-CI / Career-value dashboard failed "
        "validation:\n  "
        +
        "\n  ".join(
            failed_32hk
        )
    )


# ============================================================
# 3. VERIFY PAYLOAD STILL AVAILABLE
# ============================================================

if "dashboard_namespace_32hk" not in globals():

    raise RuntimeError(
        "dashboard_namespace_32hk is not available."
    )


payload_32hk = (
    dashboard_namespace_32hk[
        "payload"
    ]
)


# ============================================================
# 4. 95% INTERVAL SPOT CHECK
# ============================================================

max_row_32hk = next(

    row

    for row
    in payload_32hk[
        "current_rankings"
    ]

    if row[
        "driver"
    ]
    ==
    "max_verstappen"
)


print()
print("=" * 80)
print("95% INTERVAL SPOT CHECK")
print("=" * 80)


print()
print(
    "Max Verstappen 2026 alpha:"
)

print(
    f"  mean = {max_row_32hk['alpha_mean']:.3f}"
)

print(
    "  95% CrI = "
    f"[{max_row_32hk['alpha_lb']:.3f}, "
    f"{max_row_32hk['alpha_ub']:.3f}]"
)

print(
    "  95% rank interval = "
    f"#{max_row_32hk['rank_lb']}"
    "–"
    f"#{max_row_32hk['rank_ub']}"
)


# ============================================================
# 5. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-K COMPLETE")
print("=" * 80)


print()
print(
    "Final dashboard:"
)

print(
    generated_path_32hk.resolve()
)


print()
print(
    "Public uncertainty:"
)

print(
    "  central 95% Bayesian credible intervals"
)

print(
    "  posterior quantiles 2.5% – 97.5%"
)


print()
print(
    "Public score terminology:"
)

print(
    "  Peak         = 60%"
)

print(
    "  Career value = 40%"
)

print(
    "  Career value formula unchanged:"
)

print(
    "  sqrt(cumulative positive career alpha surplus)"
)


print()
print(
    "No model, posterior, JAWS score or ranking was altered."
)

print()
print(
    "Stage 32H-K validation: PASS."
)